# Notebook 11 — Deep-Learning Forecasting Models

## Purpose

This notebook extends the controlled SOLETE forecasting experiment from
the classical machine-learning models to neural sequence models.

The models evaluated in this notebook are:

1. **LSTM**
2. **GRU**
3. **Bidirectional LSTM (BiLSTM)**
4. **Temporal Convolutional Network (TCN)**

The scientific question remains unchanged:

> **Do Vedic/lunisolar calendar features improve short-term solar-PV
> forecasting beyond historical measurements, conventional Gregorian
> time features, and continuous astronomical features?**

The central comparison therefore remains:

$$
\boxed{F3 \rightarrow F4}
$$

where:

- **F3** contains historical measurements, Gregorian time features,
  solar geometry, and continuous lunar/Sun-Moon astronomical variables.
- **F4** adds the categorical Vedic/Panchang representation
  (Tithi, Paksha, Karana, Nakshatra, and Yoga).

The deep-learning experiment must preserve the same controlled
ablation logic used in Notebooks 09 and 10.

---

## Forecasting Task

### Target

$$
P_{\mathrm{Solar}} \; [\mathrm{kW}]
$$

from the SOLETE photovoltaic system.

### Forecast Horizons

The same three direct forecasting horizons are retained:

- **15 minutes**
- **30 minutes**
- **60 minutes**

Each horizon is treated as a separate direct forecasting problem.

No recursive multi-step forecasting is used.

---

## Chronological Data Split

The exact same valid-time split used in the earlier experiments is
retained:

- **Training:** 2018-06-01 to 2019-04-01
- **Validation:** 2019-04-01 to 2019-06-01
- **Test:** 2019-06-01 to 2019-09-01

The split remains strictly chronological.

The validation period is used for:

- neural-network architecture/hyperparameter selection,
- training-duration selection through early stopping,
- any other model-selection decision.

The held-out test period must not influence model selection.

The test set will be evaluated only after the corresponding
configuration has been frozen.

---

## Leakage-Safe Sequence Formulation

Unlike the classical models, the neural models will receive the recent
history as a true temporal sequence rather than as a flattened vector.

For an issue time $t$, the historical sequence contains observations:

$$
t-60,\; t-55,\; \ldots,\; t-5,\; t
$$

at 5-minute resolution.

Therefore:

$$
\text{sequence length} = 13
$$

and every sequence uses information available **at or before the issue
time $t$**.

No observation later than the forecast issue time may enter the
historical sequence.

This is especially important for the BiLSTM experiment. A BiLSTM is
permitted only because both directional passes operate over the
already-observed historical window:

$$
[t-60,\ldots,t]
$$

It must **not** receive future observations between $t$ and the
forecast-valid time.

Thus, bidirectional processing of the historical context does not by
itself constitute forecasting leakage.

---

## Historical Sequence Channels

The sequential branch will use the same historical variables underlying
the F0 baseline:

1. `P_Solar[kW]`
2. `GHI[kW1m2]`
3. `POA Irr[kW1m2]`
4. `TEMPERATURE[degC]`
5. `Humidity_clean`
6. `Pressure_clean[mbar]`
7. `WIND_SPEED[m1s]`
8. `wind_direction_sin`
9. `wind_direction_cos`

Therefore the core history tensor for one sample has shape:

$$
(13,\;9)
$$

corresponding to:

$$
13 \text{ timesteps} \times 9 \text{ channels}
$$

The existing 24-hour PV lag used in the classical F0 baseline will be
retained as an additional scalar input rather than being inserted into
the short 60-minute sequence.

---

## Nested Feature-Family Design

To preserve comparability with the earlier controlled ablation, the
deep-learning models will use the same nested feature families.

### F0 — Historical Baseline

F0 contains:

- the 13-step $\times$ 9-channel observed historical sequence,
- `P_Solar` lag at $t-24$ hours.

No future observed weather or irradiance is permitted.

### F1 — Gregorian Calendar Augmentation

$$
F1 = F0 + \text{Gregorian/calendar features}
$$

The 15 deterministic Gregorian features are evaluated at the
**forecast-valid time**.

### F2 — Solar-Astronomy Augmentation

$$
F2 = F1 + \text{solar geometry}
$$

The five independently computed solar features are evaluated at the
forecast-valid time:

- solar elevation,
- cosine of solar zenith,
- solar azimuth sine,
- solar azimuth cosine,
- daylight indicator.

### F3 — Continuous Lunar/Sun-Moon Astronomy

$$
F3 = F2 + \text{continuous lunar/Sun-Moon astronomy}
$$

The nine continuous astronomical variables are evaluated at the
forecast-valid time.

### F4 — Vedic/Panchang Categorical Representation

$$
F4 = F3 + \text{Panchang one-hot features}
$$

The 97 fixed-vocabulary one-hot Panchang indicators are evaluated at
the forecast-valid time.

This preserves the key interpretation:

> **F3 controls for the continuous astronomy from which much of the
> Panchang representation is derived; F4 tests whether the categorical
> Vedic representation adds useful forecasting information beyond those
> continuous astronomical controls.**

---

## Neural-Network Input Structure

The deep models will use two conceptual input components.

### 1. Sequential Historical Branch

Input shape:

$$
(13,\;9)
$$

This branch is processed by the sequence model:

- LSTM,
- GRU,
- BiLSTM, or
- TCN.

### 2. Auxiliary Deterministic/Static Branch

This contains the non-sequential predictors appropriate to F0-F4,
including:

- 24-hour PV lag,
- Gregorian valid-time features,
- solar astronomy,
- continuous lunar astronomy,
- Panchang one-hot indicators.

The encoded sequence representation and auxiliary feature vector will
be concatenated before the final regression layers.

This design preserves the temporal structure of recent observations
while retaining the same nested F0-F4 scientific ablation.

---

## Preprocessing and Scaling Rules

All learned preprocessing parameters must be estimated from the
**training population only**.

In particular:

- sequence-feature scaling must be fitted using training samples only;
- auxiliary continuous-feature scaling must use training samples only;
- validation and test data must be transformed using those frozen
  training-derived parameters;
- one-hot categorical indicators will remain binary;
- no scaler may be fitted using validation or test observations.

The target may be normalized for neural-network optimization, but any
normalization rule must be frozen before test evaluation and all final
reported metrics must be returned to the original **kW** scale.

---

## Model-Selection Protocol

For each model family and forecasting horizon:

1. define the candidate architecture/hyperparameter search space
   **before examining its validation results**;
2. fit candidate models using the training period only;
3. use chronological validation MAE as the primary selection criterion;
4. use validation RMSE only as a secondary/tie-breaking metric;
5. freeze the selected configuration;
6. refit the selected configuration using training + validation;
7. evaluate the held-out test set exactly once.

The test set must never be used to:

- choose hidden units,
- choose number of layers,
- choose dropout,
- choose learning rate,
- choose batch size,
- choose number of epochs,
- choose early-stopping settings,
- choose feature family,
- or otherwise modify the network.

---

## Early Stopping

Deep networks require control of training duration.

Early stopping will therefore operate on the chronological validation
population during model selection.

The exact settings, including:

- maximum epochs,
- patience,
- minimum improvement threshold,
- weight restoration,

will be frozen before model-selection experiments begin.

When the selected architecture is finally refitted on
training + validation, the final-training procedure will be defined
explicitly so that the held-out test set remains completely untouched.

---

## Reproducibility

The experiment will use fixed random seeds wherever supported,
including:

- NumPy,
- TensorFlow/Keras,
- weight initialization,
- data-order operations.

The software environment for this notebook is:

- Python 3.12.14
- TensorFlow 2.18.1
- Keras 3.15.1
- tensorflow-metal 1.2.0
- Apple M2 GPU via Metal

The environment has been verified with an actual TensorFlow matrix
multiplication executed on `/GPU:0`.

Exact bitwise reproducibility is not guaranteed for all GPU kernels,
but the experiment will use deterministic settings wherever practical
and all seeds/configurations will be recorded.

---

## Evaluation Metrics

The same final metrics used in the earlier experiments will be retained:

- MAE
- RMSE
- normalized MAE
- normalized RMSE
- $R^2$

Evaluation will be reported for:

1. **all test rows**, and
2. **daylight-only test rows**.

MAPE will not be used because PV output contains many zero and
near-zero observations.

The main optimization and comparison metric remains **MAE**.

---

## Statistical Comparison of F3 and F4

After final held-out predictions are obtained, the primary F3-vs-F4
uncertainty analysis will use the same paired bootstrap protocol as the
earlier models.

The comparison is:

$$
\mathrm{MAE}_{F3} - \mathrm{MAE}_{F4}
$$

Therefore:

$$
\mathrm{MAE}_{F3} - \mathrm{MAE}_{F4} > 0
$$

means that **F4 has lower MAE than F3**.

The bootstrap protocol is:

- forecast-valid UTC calendar day as the resampling block;
- 92 test-day blocks;
- 10,000 bootstrap replicates;
- seed 42;
- 95% percentile confidence interval;
- no model retraining during bootstrap;
- pooled row-weighted MAE;
- separate all-time and daylight analyses.

Using the same uncertainty protocol across model families allows the
deep-learning findings to be compared directly with the classical-model
results.

---

## Notebook Execution Order

The deep-learning study will proceed sequentially:

$$
\boxed{
\text{LSTM}
\rightarrow
\text{GRU}
\rightarrow
\text{BiLSTM}
\rightarrow
\text{TCN}
}
$$

For each model family, the sequence will be:

1. construct and verify leakage-safe datasets;
2. freeze the candidate hyperparameter protocol;
3. perform validation-only model selection;
4. freeze the selected configurations;
5. evaluate the held-out test set;
6. perform the paired F3-vs-F4 daily-block bootstrap;
7. interpret results;
8. persist artifacts before proceeding to the next model family.

No conclusion regarding a Panchang forecasting benefit will be drawn
from isolated point estimates alone.

In [1]:
# ============================================================
# Step 1: Deep-learning environment and reproducibility setup
# ============================================================
#
# This cell establishes the software/runtime environment used
# throughout Notebook 11.
#
# Goals:
#
#   1. import the core scientific-computing libraries;
#   2. verify the TensorFlow / Keras versions;
#   3. confirm that the Apple Metal GPU is visible;
#   4. establish a single experiment-wide random seed;
#   5. request deterministic TensorFlow operations where the
#      installed backend supports them.
#
# No dataset is loaded and no model is trained in this cell.
# ============================================================


# ------------------------------------------------------------
# 1. Standard-library imports
# ------------------------------------------------------------

from pathlib import Path

import gc
import json
import os
import random
import time


# ------------------------------------------------------------
# 2. Scientific-computing imports
# ------------------------------------------------------------

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3. TensorFlow / Keras
# ------------------------------------------------------------

import tensorflow as tf


# ------------------------------------------------------------
# 4. Global experiment seed
#
# We use one documented seed throughout the deep-learning
# experiments unless a later protocol explicitly states
# otherwise.
# ------------------------------------------------------------

DEEP_LEARNING_SEED = 42


# ------------------------------------------------------------
# 5. Seed Python, NumPy, and TensorFlow
#
# tf.keras.utils.set_random_seed() sets:
#
#   - Python's random module;
#   - NumPy's global random state;
#   - TensorFlow's random state.
#
# We explicitly seed Python and NumPy as well so the notebook
# remains self-documenting.
# ------------------------------------------------------------

os.environ[
    "PYTHONHASHSEED"
] = str(
    DEEP_LEARNING_SEED
)


random.seed(
    DEEP_LEARNING_SEED
)


np.random.seed(
    DEEP_LEARNING_SEED
)


tf.keras.utils.set_random_seed(
    DEEP_LEARNING_SEED
)


# ------------------------------------------------------------
# 6. Request deterministic TensorFlow operations
#
# Some accelerator kernels may still have platform-specific
# reproducibility limitations. Therefore we record whether the
# deterministic-operation request succeeds rather than silently
# assuming full bitwise determinism.
# ------------------------------------------------------------

determinism_enabled = False

determinism_message = None


try:

    tf.config.experimental.enable_op_determinism()

    determinism_enabled = True

    determinism_message = (
        "TensorFlow deterministic operations requested successfully."
    )

except Exception as exc:

    determinism_message = (
        "TensorFlow deterministic-operation request was not enabled: "
        f"{type(exc).__name__}: {exc}"
    )


# ------------------------------------------------------------
# 7. Detect physical devices
# ------------------------------------------------------------

physical_gpus = (
    tf.config.list_physical_devices(
        "GPU"
    )
)


physical_cpus = (
    tf.config.list_physical_devices(
        "CPU"
    )
)


# ------------------------------------------------------------
# 8. Basic environment assertions
# ------------------------------------------------------------

assert (
    tf.__version__
    == "2.18.1"
)


assert (
    len(
        physical_cpus
    )
    >= 1
)


assert (
    len(
        physical_gpus
    )
    >= 1
), (
    "No TensorFlow GPU device detected. "
    "Apple Metal acceleration is expected for this notebook."
)


# ------------------------------------------------------------
# 9. Report environment
# ------------------------------------------------------------

print(
    "Deep-learning environment"
)

print(
    "-------------------------"
)

print(
    "Python-backed TensorFlow:",
    tf.__version__,
)

print(
    "Keras:",
    tf.keras.__version__,
)

print(
    "NumPy:",
    np.__version__,
)

print(
    "Pandas:",
    pd.__version__,
)

print(
    "Experiment seed:",
    DEEP_LEARNING_SEED,
)

print(
    "Physical GPUs:",
    physical_gpus,
)

print(
    "Physical CPUs:",
    physical_cpus,
)

print(
    "Deterministic operations requested:",
    determinism_enabled,
)

print(
    determinism_message
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nNotebook 11 environment setup: PASSED"
)

Deep-learning environment
-------------------------
Python-backed TensorFlow: 2.18.1
Keras: 3.15.1
NumPy: 2.0.2
Pandas: 3.0.5
Experiment seed: 42
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Physical CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Deterministic operations requested: True
TensorFlow deterministic operations requested successfully.

Notebook 11 environment setup: PASSED


In [2]:
# ============================================================
# Step 2: Project paths and reusable project-module imports
# ============================================================
#
# Notebook 11 is located inside:
#
#       Project/notebooks/
#
# Therefore the project root should be one directory above the
# notebook working directory.
#
# In this cell we:
#
#   1. determine the project root;
#   2. make the project root importable;
#   3. verify the expected source/data directories;
#   4. import the reusable SOLETE project modules;
#   5. confirm exactly which module files Python loaded.
#
# No dataset is loaded yet.
# ============================================================


# ------------------------------------------------------------
# 1. Additional standard-library import
# ------------------------------------------------------------

import sys


# ------------------------------------------------------------
# 2. Determine notebook working directory
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd().resolve()


print(
    "Notebook working directory:"
)

print(
    NOTEBOOK_DIR
)


# ------------------------------------------------------------
# 3. Determine project root
#
# Expected:
#
#       .../Project/notebooks   <- cwd
#       .../Project             <- project root
# ------------------------------------------------------------

if NOTEBOOK_DIR.name == "notebooks":

    PROJECT_ROOT = (
        NOTEBOOK_DIR.parent
        .resolve()
    )

else:

    raise RuntimeError(
        "Notebook is expected to run from the Project/notebooks "
        f"directory, but cwd is: {NOTEBOOK_DIR}"
    )


print(
    "\nProject root:"
)

print(
    PROJECT_ROOT
)


# ------------------------------------------------------------
# 4. Important project paths
# ------------------------------------------------------------

SRC_DIR = (
    PROJECT_ROOT
    / "src"
)

DATA_DIR = (
    PROJECT_ROOT
    / "solete_dataset"
)

EPHEMERIS_DIR = (
    PROJECT_ROOT
    / "ephemeris"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook11_deep_learning_models"
)


# ------------------------------------------------------------
# 5. Structural checks
# ------------------------------------------------------------

assert PROJECT_ROOT.exists()

assert SRC_DIR.exists()

assert DATA_DIR.exists()

assert EPHEMERIS_DIR.exists()


# ------------------------------------------------------------
# 6. Make project root importable
#
# We import using:
#
#       from src import ...
#
# rather than inserting src/ itself into sys.path.
# ------------------------------------------------------------

project_root_string = str(
    PROJECT_ROOT
)


if project_root_string not in sys.path:

    sys.path.insert(
        0,
        project_root_string,
    )


# ------------------------------------------------------------
# 7. Import reusable project modules
#
# Importing the modules themselves first lets us verify that
# Python is loading our local project files before using any
# individual helper functions.
# ------------------------------------------------------------

from src import data_loader
from src import preprocessing
from src import astronomy_features
from src import vedic_calendar


# ------------------------------------------------------------
# 8. Verify module locations
# ------------------------------------------------------------

loaded_modules = {
    "data_loader":
        Path(
            data_loader.__file__
        ).resolve(),

    "preprocessing":
        Path(
            preprocessing.__file__
        ).resolve(),

    "astronomy_features":
        Path(
            astronomy_features.__file__
        ).resolve(),

    "vedic_calendar":
        Path(
            vedic_calendar.__file__
        ).resolve(),
}


for (
    module_name,
    module_path,
) in loaded_modules.items():

    assert (
        SRC_DIR
        in module_path.parents
    ), (
        f"{module_name} was not loaded from the project's src/ "
        f"directory: {module_path}"
    )


# ------------------------------------------------------------
# 9. Report paths and imported modules
# ------------------------------------------------------------

print(
    "\nProject directories"
)

print(
    "-------------------"
)

print(
    "src:",
    SRC_DIR,
)

print(
    "data:",
    DATA_DIR,
)

print(
    "ephemeris:",
    EPHEMERIS_DIR,
)

print(
    "Notebook 11 results:",
    RESULTS_DIR,
)


print(
    "\nReusable modules"
)

print(
    "----------------"
)


for (
    module_name,
    module_path,
) in loaded_modules.items():

    print(
        f"{module_name}:"
    )

    print(
        f"  {module_path}"
    )


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nNotebook 11 project/module setup: PASSED"
)

Notebook working directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/notebooks

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Project directories
-------------------
src: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/src
data: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset
ephemeris: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/ephemeris
Notebook 11 results: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models

Reusable modules
----------------
data_loader:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/src/data_loader.py
preprocessing:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Proj

In [3]:
# ============================================================
# Step 3: Inspect reusable project-module APIs
# ============================================================
#
# Before reconstructing the modelling dataset, we inspect the
# public functions already defined in:
#
#   - data_loader.py
#   - preprocessing.py
#   - astronomy_features.py
#   - vedic_calendar.py
#
# This prevents Notebook 11 from:
#
#   - guessing function names;
#   - duplicating preprocessing logic;
#   - accidentally using a different feature definition from
#     Notebooks 06-10.
#
# No data is loaded in this cell.
# ============================================================


# ------------------------------------------------------------
# 1. Standard-library introspection helper
# ------------------------------------------------------------

import inspect


# ------------------------------------------------------------
# 2. Helper to list public functions defined by a module
#
# We exclude:
#
#   - names beginning with "_";
#   - imported functions/classes from external libraries.
#
# Only functions whose __module__ matches the project module
# itself are displayed.
# ------------------------------------------------------------

def show_public_module_functions(
    module,
):

    print(
        f"\n{module.__name__}"
    )

    print(
        "-" * len(
            module.__name__
        )
    )


    public_functions = []


    for name in dir(
        module
    ):

        if name.startswith(
            "_"
        ):
            continue


        obj = getattr(
            module,
            name,
        )


        if not inspect.isfunction(
            obj
        ):
            continue


        if obj.__module__ != module.__name__:
            continue


        public_functions.append(
            (
                name,
                inspect.signature(
                    obj
                ),
            )
        )


    if not public_functions:

        print(
            "  No public module-defined functions found."
        )

        return


    for (
        function_name,
        signature,
    ) in public_functions:

        print(
            f"  {function_name}{signature}"
        )


# ------------------------------------------------------------
# 3. Inspect all four reusable project modules
# ------------------------------------------------------------

for module in [
    data_loader,
    preprocessing,
    astronomy_features,
    vedic_calendar,
]:

    show_public_module_functions(
        module
    )


# ------------------------------------------------------------
# 4. Final checkpoint
# ------------------------------------------------------------

print(
    "\nReusable module API inspection: PASSED"
)


src.data_loader
---------------
  load_solete(file_path: str | pathlib.Path) -> pandas.DataFrame

src.preprocessing
-----------------
  build_cleaned_solete_features(df: 'pd.DataFrame') -> 'pd.DataFrame'
  build_gregorian_calendar_features(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'
  build_gregorian_cyclic_features(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'
  build_gregorian_features(timestamps: 'pd.DatetimeIndex') -> 'pd.DataFrame'
  build_preprocessed_solete_features(df: 'pd.DataFrame') -> 'pd.DataFrame'
  clean_humidity_fraction(humidity: 'pd.Series') -> 'pd.Series'
  clean_pressure_mbar(pressure_mbar: 'pd.Series') -> 'pd.Series'
  clean_wind_direction_deg(wind_direction_deg: 'pd.Series') -> 'pd.Series'
  gaia_quality_flag(reconstruction_error_kw: 'pd.Series', threshold_kw: 'float' = 3.0) -> 'pd.Series'
  wind_direction_cyclic_features(wind_direction_deg: 'pd.Series') -> 'tuple[pd.Series, pd.Series]'

src.astronomy_features
----------------------
  apparent_tropical_e

In [4]:
# ============================================================
# Step 4: Locate the canonical 5-minute SOLETE HDF5 file
# ============================================================
#
# Notebook 11 must use exactly the same 5-minute source data
# resolution used in Notebooks 06-10.
#
# Rather than guessing a filename, this cell inventories the
# HDF5 files currently present under:
#
#       Project/solete_dataset/
#
# We will inspect the candidates first and select the exact
# 5-minute file in the next step.
#
# No dataset is loaded yet.
# ============================================================


# ------------------------------------------------------------
# 1. Find HDF5 files recursively
# ------------------------------------------------------------

hdf5_candidates = sorted(
    list(
        DATA_DIR.rglob(
            "*.h5"
        )
    )
    +
    list(
        DATA_DIR.rglob(
            "*.hdf5"
        )
    )
)


# ------------------------------------------------------------
# 2. Structural check
# ------------------------------------------------------------

assert len(
    hdf5_candidates
) > 0, (
    f"No HDF5 files were found under: {DATA_DIR}"
)


# ------------------------------------------------------------
# 3. Display candidate files
# ------------------------------------------------------------

print(
    "SOLETE HDF5 candidates"
)

print(
    "----------------------"
)


for i, file_path in enumerate(
    hdf5_candidates,
    start=1,
):

    relative_path = (
        file_path
        .relative_to(
            PROJECT_ROOT
        )
    )

    size_mb = (
        file_path
        .stat()
        .st_size
        / 1024**2
    )


    print(
        f"{i:2d}. "
        f"{relative_path}"
    )

    print(
        f"    size: "
        f"{size_mb:,.2f} MB"
    )


# ------------------------------------------------------------
# 4. Highlight filenames containing likely 5-minute markers
# ------------------------------------------------------------

likely_5min_candidates = [
    path
    for path in hdf5_candidates
    if any(
        marker
        in path.name.lower()
        for marker in [
            "5min",
            "5_min",
            "5m",
        ]
    )
]


print(
    "\nLikely 5-minute filename matches:"
)


if likely_5min_candidates:

    for path in likely_5min_candidates:

        print(
            " ",
            path.relative_to(
                PROJECT_ROOT
            ),
        )

else:

    print(
        "  No filename contains an obvious 5-minute marker."
    )

    print(
        "  We will identify the correct file from its contents."
    )


# ------------------------------------------------------------
# 5. Final checkpoint
# ------------------------------------------------------------

print(
    "\nSOLETE source-file inventory: PASSED"
)

SOLETE HDF5 candidates
----------------------
 1. solete_dataset/SOLETE_Pombo_1min.h5
    size: 60.26 MB
 2. solete_dataset/SOLETE_Pombo_1sec.h5
    size: 3,614.95 MB
 3. solete_dataset/SOLETE_Pombo_5min.h5
    size: 12.06 MB
 4. solete_dataset/SOLETE_Pombo_60min.h5
    size: 1.01 MB
 5. solete_dataset/SOLETE_short.h5
    size: 0.01 MB

Likely 5-minute filename matches:
  solete_dataset/SOLETE_Pombo_5min.h5

SOLETE source-file inventory: PASSED


In [5]:
# ============================================================
# Step 5: Load and verify the canonical 5-minute SOLETE data
# ============================================================
#
# This is the modelling resolution used throughout the
# controlled experiments in Notebooks 06-10.
#
# In this cell we:
#
#   1. load the raw 5-minute HDF5 file using src.data_loader;
#   2. verify the expected row count;
#   3. verify the UTC timestamp index;
#   4. verify chronological ordering and uniqueness;
#   5. verify the exact 5-minute spacing;
#   6. inspect the original source columns.
#
# No cleaning or feature engineering is performed yet.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical source file
# ------------------------------------------------------------

SOLETE_5MIN_PATH = (
    DATA_DIR
    / "SOLETE_Pombo_5min.h5"
)


assert SOLETE_5MIN_PATH.exists()


print(
    "SOLETE 5-minute source:"
)

print(
    SOLETE_5MIN_PATH
)


# ------------------------------------------------------------
# 2. Load raw dataset
# ------------------------------------------------------------

df_raw_5min = (
    data_loader.load_solete(
        SOLETE_5MIN_PATH
    )
)


# ------------------------------------------------------------
# 3. Basic structure
# ------------------------------------------------------------

print(
    "\nRaw dataset shape:"
)

print(
    df_raw_5min.shape
)


print(
    "\nColumns:"
)


for i, column_name in enumerate(
    df_raw_5min.columns,
    start=1,
):

    print(
        f"  {i:2d}. {column_name}"
    )


# ------------------------------------------------------------
# 4. Timestamp/index checks
# ------------------------------------------------------------

assert isinstance(
    df_raw_5min.index,
    pd.DatetimeIndex,
)


assert (
    str(
        df_raw_5min.index.tz
    )
    == "UTC"
)


assert df_raw_5min.index.is_monotonic_increasing


assert df_raw_5min.index.is_unique


# ------------------------------------------------------------
# 5. Canonical row-count check
# ------------------------------------------------------------

assert (
    len(
        df_raw_5min
    )
    == 131_617
)


# ------------------------------------------------------------
# 6. Verify exact dataset coverage
# ------------------------------------------------------------

expected_start = pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC",
)


expected_end = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC",
)


assert (
    df_raw_5min.index[0]
    == expected_start
)


assert (
    df_raw_5min.index[-1]
    == expected_end
)


# ------------------------------------------------------------
# 7. Verify exact 5-minute grid
#
# There should be no gaps or irregular intervals.
# ------------------------------------------------------------

timestamp_differences = (
    df_raw_5min.index
    .to_series()
    .diff()
    .dropna()
)


unique_time_steps = (
    timestamp_differences
    .value_counts()
)


assert (
    len(
        unique_time_steps
    )
    == 1
)


assert (
    unique_time_steps.index[0]
    == pd.Timedelta(
        minutes=5
    )
)


# ------------------------------------------------------------
# 8. Missing-value / duplicate summary
# ------------------------------------------------------------

n_duplicate_timestamps = int(
    df_raw_5min.index.duplicated().sum()
)


n_missing_values = int(
    df_raw_5min.isna().sum().sum()
)


# ------------------------------------------------------------
# 9. Report canonical-grid verification
# ------------------------------------------------------------

print(
    "\nTimestamp coverage:"
)

print(
    "  Start:",
    df_raw_5min.index[0],
)

print(
    "  End:",
    df_raw_5min.index[-1],
)

print(
    "  Rows:",
    len(
        df_raw_5min
    ),
)

print(
    "  Duplicate timestamps:",
    n_duplicate_timestamps,
)

print(
    "  Missing values:",
    n_missing_values,
)

print(
    "  Unique sampling interval:",
    unique_time_steps.index[0],
)


# ------------------------------------------------------------
# 10. Small data preview
# ------------------------------------------------------------

print(
    "\nFirst 3 rows:"
)


display(
    df_raw_5min.head(
        3
    )
)


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nCanonical raw 5-minute SOLETE dataset: PASSED"
)

SOLETE 5-minute source:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_5min.h5

Raw dataset shape:
(131617, 11)

Columns:
   1. TEMPERATURE[degC]
   2. HUMIDITY[%]
   3. WIND_SPEED[m1s]
   4. WIND_DIR[deg]
   5. GHI[kW1m2]
   6. POA Irr[kW1m2]
   7. P_Gaia[kW]
   8. P_Solar[kW]
   9. Pressure[mbar]
  10. Azimuth[deg]
  11. Elevation[deg]

Timestamp coverage:
  Start: 2018-06-01 00:00:00+00:00
  End: 2019-09-01 00:00:00+00:00
  Rows: 131617
  Duplicate timestamps: 0
  Missing values: 0
  Unique sampling interval: 0 days 00:05:00

First 3 rows:


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.561204,0.7,1.293645,111.635452,0.0,0.0,0.046917,0.0,1017.559985,0.0,0.0
2018-06-01 00:05:00+00:00,14.487667,0.7,1.442333,91.236667,0.0,0.0,0.000267,0.0,1017.539990,0.0,0.0
2018-06-01 00:10:00+00:00,14.400000,0.7,1.408667,104.423333,0.0,0.0,0.000000,0.0,1017.493668,0.0,0.0



Canonical raw 5-minute SOLETE dataset: PASSED


In [6]:
# ============================================================
# Step 6: Reconstruct the canonical cleaned 5-minute dataset
# ============================================================
#
# We now reuse the cleaning logic implemented in:
#
#       src/preprocessing.py
#
# using:
#
#       build_cleaned_solete_features()
#
# This is important because Notebook 11 must NOT implement a
# second version of pressure, humidity, or wind-direction
# cleaning.
#
# The reusable cleaning pipeline should:
#
#   - preserve all 11 original SOLETE columns;
#   - correct the known pressure artifact;
#   - create the causal cleaned humidity series;
#   - wrap wind direction correctly;
#   - create wind-direction sine/cosine features;
#   - introduce no missing values;
#   - preserve the exact timestamp grid.
#
# No Gregorian, solar, lunar, or Panchang features are added
# in this cell.
# ============================================================


# ------------------------------------------------------------
# 1. Apply reusable cleaning pipeline
# ------------------------------------------------------------

df_clean_5min = (
    preprocessing.build_cleaned_solete_features(
        df_raw_5min
    )
)


# ------------------------------------------------------------
# 2. Basic structural checks
# ------------------------------------------------------------

assert isinstance(
    df_clean_5min.index,
    pd.DatetimeIndex,
)


assert df_clean_5min.index.equals(
    df_raw_5min.index
)


assert (
    len(
        df_clean_5min
    )
    == 131_617
)


assert (
    df_clean_5min.shape[1]
    == 16
)


# ------------------------------------------------------------
# 3. Original source columns must all be preserved
# ------------------------------------------------------------

original_columns = list(
    df_raw_5min.columns
)


assert all(
    column_name
    in df_clean_5min.columns
    for column_name
    in original_columns
)


# ------------------------------------------------------------
# 4. Identify columns added by the cleaning pipeline
# ------------------------------------------------------------

cleaning_added_columns = [
    column_name
    for column_name in df_clean_5min.columns
    if column_name not in df_raw_5min.columns
]


assert (
    len(
        cleaning_added_columns
    )
    == 5
)


# ------------------------------------------------------------
# 5. Verify the cleaned features required by Notebook 11
# ------------------------------------------------------------

required_clean_features = [
    "Humidity_clean",
    "Pressure_clean[mbar]",
    "wind_direction_sin",
    "wind_direction_cos",
]


for column_name in required_clean_features:

    assert (
        column_name
        in df_clean_5min.columns
    ), (
        f"Expected cleaned feature is missing: "
        f"{column_name}"
    )


# ------------------------------------------------------------
# 6. Missing-value / finite-value checks
# ------------------------------------------------------------

assert (
    df_clean_5min.isna()
    .sum()
    .sum()
    == 0
)


assert np.isfinite(
    df_clean_5min.select_dtypes(
        include=np.number
    ).to_numpy()
).all()


# ------------------------------------------------------------
# 7. Verify known cleaned-variable ranges
# ------------------------------------------------------------

# Humidity was treated as a fraction in the existing pipeline.
assert (
    df_clean_5min[
        "Humidity_clean"
    ].min()
    >= 0.0
)


assert (
    df_clean_5min[
        "Humidity_clean"
    ].max()
    <= 1.0
)


# Corrected pressure should no longer contain the +1000 mbar
# midnight artifact.
assert (
    df_clean_5min[
        "Pressure_clean[mbar]"
    ].max()
    < 1100.0
)


# Circular wind-direction encoding must remain bounded.
assert (
    df_clean_5min[
        "wind_direction_sin"
    ].between(
        -1.0,
        1.0,
    ).all()
)


assert (
    df_clean_5min[
        "wind_direction_cos"
    ].between(
        -1.0,
        1.0,
    ).all()
)


# ------------------------------------------------------------
# 8. Report reconstructed structure
# ------------------------------------------------------------

print(
    "Canonical cleaned dataset"
)

print(
    "-------------------------"
)

print(
    "Shape:",
    df_clean_5min.shape,
)

print(
    "Rows:",
    len(
        df_clean_5min
    ),
)

print(
    "Columns added by cleaning:",
)


for column_name in cleaning_added_columns:

    print(
        " ",
        column_name,
    )


print(
    "\nCleaned-variable ranges:"
)

print(
    "  Humidity_clean:",
    (
        float(
            df_clean_5min[
                "Humidity_clean"
            ].min()
        ),
        float(
            df_clean_5min[
                "Humidity_clean"
            ].max()
        ),
    ),
)

print(
    "  Pressure_clean[mbar]:",
    (
        float(
            df_clean_5min[
                "Pressure_clean[mbar]"
            ].min()
        ),
        float(
            df_clean_5min[
                "Pressure_clean[mbar]"
            ].max()
        ),
    ),
)

print(
    "  wind_direction_sin:",
    (
        float(
            df_clean_5min[
                "wind_direction_sin"
            ].min()
        ),
        float(
            df_clean_5min[
                "wind_direction_sin"
            ].max()
        ),
    ),
)

print(
    "  wind_direction_cos:",
    (
        float(
            df_clean_5min[
                "wind_direction_cos"
            ].min()
        ),
        float(
            df_clean_5min[
                "wind_direction_cos"
            ].max()
        ),
    ),
)


# ------------------------------------------------------------
# 9. Small preview of the cleaned variables
# ------------------------------------------------------------

display(
    df_clean_5min[
        cleaning_added_columns
    ].head(
        5
    )
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nCanonical cleaned 5-minute dataset: PASSED"
)

Canonical cleaned dataset
-------------------------
Shape: (131617, 16)
Rows: 131617
Columns added by cleaning:
  Pressure_clean[mbar]
  Humidity_clean
  Wind_dir_clean[deg]
  wind_direction_sin
  wind_direction_cos

Cleaned-variable ranges:
  Humidity_clean: (0.0, 1.0)
  Pressure_clean[mbar]: (974.2629986572266, 1042.3197102864617)
  wind_direction_sin: (-1.0, 1.0)
  wind_direction_cos: (-0.999999998307681, 1.0)


,Pressure_clean[mbar],Humidity_clean,Wind_dir_clean[deg],wind_direction_sin,wind_direction_cos
Timestamp,,,,,
2018-06-01 00:00:00+00:00,1017.559985,0.7,111.635452,0.929549,-0.368700
2018-06-01 00:05:00+00:00,1017.539990,0.7,91.236667,0.999767,-0.021582
2018-06-01 00:10:00+00:00,1017.493668,0.7,104.423333,0.968482,-0.249084
2018-06-01 00:15:00+00:00,1017.466342,0.7,103.976667,0.970394,-0.241527
2018-06-01 00:20:00+00:00,1017.533658,0.7,96.606667,0.993359,-0.115053



Canonical cleaned 5-minute dataset: PASSED


In [7]:
# ============================================================
# Step 7: Construct the leakage-safe deep-learning F0 data
# ============================================================
#
# Deep-learning F0 must represent exactly the same information
# available to the classical F0 baseline, but preserve the
# temporal structure of the recent observations.
#
# Classical F0 contained:
#
#   13 historical timestamps × 9 observed variables = 117
#   + P_Solar lagged by 24 hours                     =   1
#                                                       ----
#                                                        118
#
# For Notebook 11 we represent this as:
#
#   sequence input:
#       shape = (13, 9)
#
#   auxiliary scalar:
#       P_Solar at t - 24 hours
#
# Critically, every sequence ends at issue time t.
# No observation after t is allowed into the historical branch.
#
# We also reconstruct the exact valid-time populations used in
# Notebook 10 for the 15-, 30-, and 60-minute horizons.
# ============================================================


# ------------------------------------------------------------
# 1. Historical sequence definition
# ------------------------------------------------------------

DEEP_SEQUENCE_LENGTH = 13

DEEP_SEQUENCE_STEP_MINUTES = 5

DEEP_HISTORY_MINUTES = (
    (DEEP_SEQUENCE_LENGTH - 1)
    * DEEP_SEQUENCE_STEP_MINUTES
)


assert DEEP_HISTORY_MINUTES == 60


DEEP_SEQUENCE_CHANNELS = [
    "P_Solar[kW]",
    "GHI[kW1m2]",
    "POA Irr[kW1m2]",
    "TEMPERATURE[degC]",
    "Humidity_clean",
    "Pressure_clean[mbar]",
    "WIND_SPEED[m1s]",
    "wind_direction_sin",
    "wind_direction_cos",
]


assert (
    len(
        DEEP_SEQUENCE_CHANNELS
    )
    == 9
)


# ------------------------------------------------------------
# 2. Confirm all sequence channels exist
# ------------------------------------------------------------

for column_name in DEEP_SEQUENCE_CHANNELS:

    assert (
        column_name
        in df_clean_5min.columns
    ), (
        f"Missing sequence channel: {column_name}"
    )


# ------------------------------------------------------------
# 3. Convert observed sequential variables to float32
#
# float32 is the natural TensorFlow training dtype and uses
# half the memory of float64.
# ------------------------------------------------------------

sequence_source_array = (
    df_clean_5min[
        DEEP_SEQUENCE_CHANNELS
    ]
    .to_numpy(
        dtype=np.float32
    )
)


assert sequence_source_array.shape == (
    131_617,
    9,
)


assert np.isfinite(
    sequence_source_array
).all()


# ------------------------------------------------------------
# 4. Build all 13-step rolling windows
#
# sliding_window_view over axis=0 initially returns:
#
#       (n_windows, n_channels, sequence_length)
#
# We swap the final two dimensions to obtain the TensorFlow
# convention:
#
#       (samples, timesteps, channels)
# ------------------------------------------------------------

all_history_windows = (
    np.lib.stride_tricks
    .sliding_window_view(
        sequence_source_array,
        window_shape=DEEP_SEQUENCE_LENGTH,
        axis=0,
    )
)


all_history_windows = (
    np.swapaxes(
        all_history_windows,
        1,
        2,
    )
)


assert all_history_windows.shape == (
    131_605,
    13,
    9,
)


# ------------------------------------------------------------
# 5. The 24-hour PV lag determines the earliest usable issue
#    time for F0.
#
# 24 hours / 5 minutes = 288 samples.
# ------------------------------------------------------------

PV_LAG_24H_STEPS = (
    24
    * 60
    // DEEP_SEQUENCE_STEP_MINUTES
)


assert PV_LAG_24H_STEPS == 288


first_deep_issue_position = (
    PV_LAG_24H_STEPS
)


first_deep_issue_time = (
    df_clean_5min.index[
        first_deep_issue_position
    ]
)


assert first_deep_issue_time == pd.Timestamp(
    "2018-06-02 00:00:00",
    tz="UTC",
)


# ------------------------------------------------------------
# 6. Align rolling windows with issue times
#
# all_history_windows[0] corresponds to an issue time at raw
# row 12 because the first 13-step window spans rows 0...12.
#
# Therefore, to obtain the first issue at raw row 288:
#
#       window position = 288 - 12 = 276
# ------------------------------------------------------------

first_required_window_position = (
    first_deep_issue_position
    - (
        DEEP_SEQUENCE_LENGTH
        - 1
    )
)


assert first_required_window_position == 276


deep_history_tensor = (
    np.ascontiguousarray(
        all_history_windows[
            first_required_window_position:
        ],
        dtype=np.float32,
    )
)


deep_history_issue_index = (
    df_clean_5min.index[
        first_deep_issue_position:
    ]
)


assert deep_history_tensor.shape == (
    len(
        deep_history_issue_index
    ),
    13,
    9,
)


assert deep_history_tensor.shape == (
    131_329,
    13,
    9,
)


# ------------------------------------------------------------
# 7. Verify temporal alignment of the sequence tensor
#
# The final timestep of every sequence must equal the observed
# variables at issue time t.
#
# The first timestep must equal the observations at t - 60 min.
# ------------------------------------------------------------

expected_sequence_end = (
    df_clean_5min.loc[
        deep_history_issue_index,
        DEEP_SEQUENCE_CHANNELS,
    ]
    .to_numpy(
        dtype=np.float32
    )
)


expected_sequence_start = (
    df_clean_5min[
        DEEP_SEQUENCE_CHANNELS
    ]
    .iloc[
        (
            first_deep_issue_position
            - (
                DEEP_SEQUENCE_LENGTH
                - 1
            )
        ):
        (
            len(
                df_clean_5min
            )
            - (
                DEEP_SEQUENCE_LENGTH
                - 1
            )
        )
    ]
    .to_numpy(
        dtype=np.float32
    )
)


assert np.array_equal(
    deep_history_tensor[
        :,
        -1,
        :,
    ],
    expected_sequence_end,
)


assert np.array_equal(
    deep_history_tensor[
        :,
        0,
        :,
    ],
    expected_sequence_start,
)


# ------------------------------------------------------------
# 8. Strong first-window leakage check
# ------------------------------------------------------------

assert (
    deep_history_issue_index[0]
    - pd.Timedelta(
        minutes=60
    )
    == pd.Timestamp(
        "2018-06-01 23:00:00",
        tz="UTC",
    )
)


print(
    "First deep-learning issue time:",
    deep_history_issue_index[0],
)

print(
    "First sequence starts at:",
    (
        deep_history_issue_index[0]
        - pd.Timedelta(
            minutes=60
        )
    ),
)

print(
    "First sequence ends at:",
    deep_history_issue_index[0],
)


# ------------------------------------------------------------
# 9. Construct the 24-hour PV-lag scalar
# ------------------------------------------------------------

pv_solar_24h_lag_full = (
    df_clean_5min[
        "P_Solar[kW]"
    ]
    .shift(
        PV_LAG_24H_STEPS
    )
)


deep_pv_lag_24h = (
    pv_solar_24h_lag_full
    .loc[
        deep_history_issue_index
    ]
    .to_numpy(
        dtype=np.float32
    )
)


assert deep_pv_lag_24h.shape == (
    len(
        deep_history_issue_index
    ),
)


assert np.isfinite(
    deep_pv_lag_24h
).all()


# ------------------------------------------------------------
# 10. Valid-time chronological split boundaries
#
# These are identical to Notebooks 09 and 10.
# ------------------------------------------------------------

TRAIN_VALIDATION_CUTOFF = pd.Timestamp(
    "2019-04-01 00:00:00",
    tz="UTC",
)


VALIDATION_TEST_CUTOFF = pd.Timestamp(
    "2019-06-01 00:00:00",
    tz="UTC",
)


TEST_END_EXCLUSIVE = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC",
)


# ------------------------------------------------------------
# 11. Forecast horizons
# ------------------------------------------------------------

DEEP_FORECAST_HORIZONS = [
    15,
    30,
    60,
]


# ------------------------------------------------------------
# 12. Containers
#
# deep_base_modelling_tables[h]:
#
#   indexed by issue time and containing:
#
#       valid_time
#       target
#       pv_lag_24h
#       split
#
# deep_sequence_positions[h]:
#
#   integer positions selecting the corresponding rows from
#   deep_history_tensor without storing three duplicate tensors.
# ------------------------------------------------------------

deep_base_modelling_tables = {}

deep_sequence_positions = {}


# ------------------------------------------------------------
# 13. Build one direct forecasting population per horizon
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    horizon_delta = pd.Timedelta(
        minutes=horizon_minutes
    )


    # --------------------------------------------------------
    # Forecast-valid time for every potential issue time
    # --------------------------------------------------------

    valid_times_all = (
        deep_history_issue_index
        + horizon_delta
    )


    # --------------------------------------------------------
    # The held-out experiment ends strictly before:
    #
    #       2019-09-01 00:00 UTC
    #
    # The source file contains that endpoint, but it is not
    # part of the experimental test interval.
    # --------------------------------------------------------

    population_mask = (
        valid_times_all
        < TEST_END_EXCLUSIVE
    )


    sequence_positions = (
        np.flatnonzero(
            population_mask
        )
    )


    issue_times = (
        deep_history_issue_index[
            population_mask
        ]
    )


    valid_times = (
        valid_times_all[
            population_mask
        ]
    )


    # --------------------------------------------------------
    # Target is PV output at the forecast-valid time.
    #
    # Because the source grid is exact and complete, direct
    # timestamp lookup is unambiguous.
    # --------------------------------------------------------

    target_values = (
        df_clean_5min[
            "P_Solar[kW]"
        ]
        .reindex(
            valid_times
        )
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # Corresponding 24-hour lag scalar
    # --------------------------------------------------------

    pv_lag_values = (
        deep_pv_lag_24h[
            population_mask
        ]
    )


    # --------------------------------------------------------
    # Chronological split is determined by VALID TIME,
    # not by issue time.
    # --------------------------------------------------------

    split_values = np.empty(
        len(
            issue_times
        ),
        dtype=object,
    )


    train_mask = (
        valid_times
        < TRAIN_VALIDATION_CUTOFF
    )


    validation_mask = (
        (
            valid_times
            >= TRAIN_VALIDATION_CUTOFF
        )
        &
        (
            valid_times
            < VALIDATION_TEST_CUTOFF
        )
    )


    test_mask = (
        (
            valid_times
            >= VALIDATION_TEST_CUTOFF
        )
        &
        (
            valid_times
            < TEST_END_EXCLUSIVE
        )
    )


    split_values[
        train_mask
    ] = "train"


    split_values[
        validation_mask
    ] = "validation"


    split_values[
        test_mask
    ] = "test"


    # --------------------------------------------------------
    # Every row must belong to exactly one split.
    # --------------------------------------------------------

    assert np.all(
        train_mask.astype(
            np.int8
        )
        +
        validation_mask.astype(
            np.int8
        )
        +
        test_mask.astype(
            np.int8
        )
        == 1
    )


    # --------------------------------------------------------
    # Assemble horizon-specific base table
    # --------------------------------------------------------

    modelling_table = pd.DataFrame(
        {
            "valid_time":
                valid_times,

            "target_P_Solar_kW":
                target_values,

            "pv_lag_24h":
                pv_lag_values,

            "split":
                split_values,
        },
        index=issue_times,
    )


    modelling_table.index.name = (
        "issue_time"
    )


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert modelling_table.index.is_monotonic_increasing

    assert modelling_table.index.is_unique

    assert modelling_table[
        "valid_time"
    ].is_monotonic_increasing

    assert (
        modelling_table.isna()
        .sum()
        .sum()
        == 0
    )


    assert np.isfinite(
        modelling_table[
            [
                "target_P_Solar_kW",
                "pv_lag_24h",
            ]
        ]
        .to_numpy()
    ).all()


    assert (
        len(
            sequence_positions
        )
        == len(
            modelling_table
        )
    )


    # --------------------------------------------------------
    # Store outputs
    # --------------------------------------------------------

    deep_base_modelling_tables[
        horizon_minutes
    ] = modelling_table


    deep_sequence_positions[
        horizon_minutes
    ] = sequence_positions


# ------------------------------------------------------------
# 14. Verify exact populations from Notebook 10
# ------------------------------------------------------------

expected_deep_population_counts = {
    15: {
        "total": 131_325,
        "train": 87_261,
        "validation": 17_568,
        "test": 26_496,
    },

    30: {
        "total": 131_322,
        "train": 87_258,
        "validation": 17_568,
        "test": 26_496,
    },

    60: {
        "total": 131_316,
        "train": 87_252,
        "validation": 17_568,
        "test": 26_496,
    },
}


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    observed_counts = (
        table[
            "split"
        ]
        .value_counts()
    )


    expected_counts = (
        expected_deep_population_counts[
            horizon_minutes
        ]
    )


    assert (
        len(
            table
        )
        == expected_counts[
            "total"
        ]
    )


    assert (
        int(
            observed_counts[
                "train"
            ]
        )
        == expected_counts[
            "train"
        ]
    )


    assert (
        int(
            observed_counts[
                "validation"
            ]
        )
        == expected_counts[
            "validation"
        ]
    )


    assert (
        int(
            observed_counts[
                "test"
            ]
        )
        == expected_counts[
            "test"
        ]
    )


# ------------------------------------------------------------
# 15. Verify TensorFlow-ready F0 dimensions
#
# Equivalent flattened F0 width:
#
#       13 × 9 + 1 = 118
# ------------------------------------------------------------

DEEP_F0_EQUIVALENT_FEATURE_COUNT = (
    DEEP_SEQUENCE_LENGTH
    * len(
        DEEP_SEQUENCE_CHANNELS
    )
    + 1
)


assert DEEP_F0_EQUIVALENT_FEATURE_COUNT == 118


# ------------------------------------------------------------
# 16. Report memory footprint
# ------------------------------------------------------------

deep_history_tensor_mb = (
    deep_history_tensor.nbytes
    / 1024**2
)


# ------------------------------------------------------------
# 17. Report final structures
# ------------------------------------------------------------

print(
    "\nDeep-learning historical representation"
)

print(
    "---------------------------------------"
)

print(
    "History tensor shape:",
    deep_history_tensor.shape,
)

print(
    "Tensor dtype:",
    deep_history_tensor.dtype,
)

print(
    "Tensor memory:",
    f"{deep_history_tensor_mb:.2f} MB",
)

print(
    "Sequence length:",
    DEEP_SEQUENCE_LENGTH,
)

print(
    "Sequence channels:",
    len(
        DEEP_SEQUENCE_CHANNELS
    ),
)

print(
    "History span:",
    f"{DEEP_HISTORY_MINUTES} minutes",
)

print(
    "24-hour PV lag steps:",
    PV_LAG_24H_STEPS,
)

print(
    "Equivalent flattened F0 width:",
    DEEP_F0_EQUIVALENT_FEATURE_COUNT,
)


print(
    "\nForecasting populations"
)

print(
    "-----------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    counts = (
        table[
            "split"
        ]
        .value_counts()
    )


    print(
        f"\n{horizon_minutes}-minute horizon"
    )

    print(
        "  Total:",
        len(
            table
        ),
    )

    print(
        "  Train:",
        int(
            counts[
                "train"
            ]
        ),
    )

    print(
        "  Validation:",
        int(
            counts[
                "validation"
            ]
        ),
    )

    print(
        "  Test:",
        int(
            counts[
                "test"
            ]
        ),
    )

    print(
        "  First issue time:",
        table.index[0],
    )

    print(
        "  First valid time:",
        table[
            "valid_time"
        ].iloc[0],
    )

    print(
        "  Last valid time:",
        table[
            "valid_time"
        ].iloc[-1],
    )


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLeakage-safe deep-learning F0 construction: PASSED"
)

First deep-learning issue time: 2018-06-02 00:00:00+00:00
First sequence starts at: 2018-06-01 23:00:00+00:00
First sequence ends at: 2018-06-02 00:00:00+00:00

Deep-learning historical representation
---------------------------------------
History tensor shape: (131329, 13, 9)
Tensor dtype: float32
Tensor memory: 58.61 MB
Sequence length: 13
Sequence channels: 9
History span: 60 minutes
24-hour PV lag steps: 288
Equivalent flattened F0 width: 118

Forecasting populations
-----------------------

15-minute horizon
  Total: 131325
  Train: 87261
  Validation: 17568
  Test: 26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:15:00+00:00
  Last valid time: 2019-08-31 23:55:00+00:00

30-minute horizon
  Total: 131322
  Train: 87258
  Validation: 17568
  Test: 26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:30:00+00:00
  Last valid time: 2019-08-31 23:55:00+00:00

60-minute horizon
  Total: 131316
  Train: 87252
  Valid

In [8]:
# ============================================================
# Step 8: Load the canonical frozen Panchang vocabulary
# ============================================================
#
# F4 must use exactly the same categorical vocabulary that was
# frozen in Notebook 09 and reused in Notebook 10.
#
# We must NOT infer a new vocabulary from the deep-learning
# train/validation/test populations because that could:
#
#   - change the F4 dimensionality;
#   - change column ordering;
#   - make Notebook 11 incomparable with earlier experiments.
#
# Expected total one-hot width:
#
#       97 columns
#
# corresponding to the fixed vocabularies for:
#
#       Tithi
#       Paksha
#       Karana
#       Nakshatra
#       Yoga
# ============================================================


# ------------------------------------------------------------
# 1. Canonical vocabulary artifact from Notebook 09
# ------------------------------------------------------------

PANCHANG_VOCABULARY_PATH = (
    PROJECT_ROOT
    / "results"
    / "notebook09_primary_experiment"
    / "panchang_vocabulary.json"
)


assert PANCHANG_VOCABULARY_PATH.exists(), (
    "Canonical Panchang vocabulary file not found: "
    f"{PANCHANG_VOCABULARY_PATH}"
)


# ------------------------------------------------------------
# 2. Load JSON
# ------------------------------------------------------------

with open(
    PANCHANG_VOCABULARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    panchang_vocabulary = json.load(
        f
    )


# ------------------------------------------------------------
# 3. Inspect top-level structure
# ------------------------------------------------------------

print(
    "Panchang vocabulary source:"
)

print(
    PANCHANG_VOCABULARY_PATH
)


print(
    "\nTop-level keys:"
)

for key in panchang_vocabulary:

    print(
        " ",
        key,
    )


# ------------------------------------------------------------
# 4. Display each vocabulary and its size
# ------------------------------------------------------------

print(
    "\nFrozen Panchang vocabularies"
)

print(
    "----------------------------"
)


total_categories = 0


for (
    feature_name,
    categories,
) in panchang_vocabulary.items():

    print(
        f"{feature_name}: "
        f"{len(categories)} categories"
    )

    print(
        f"  first: {categories[0]}"
    )

    print(
        f"  last:  {categories[-1]}"
    )


    total_categories += len(
        categories
    )


# ------------------------------------------------------------
# 5. Canonical total width check
# ------------------------------------------------------------

print(
    "\nTotal one-hot categories:",
    total_categories,
)


assert (
    total_categories
    == 97
), (
    "Expected the canonical F4 Panchang vocabulary to contain "
    f"97 one-hot categories, but found {total_categories}."
)


# ------------------------------------------------------------
# 6. Required Panchang components
# ------------------------------------------------------------

expected_panchang_components = {
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
}


assert set(
    panchang_vocabulary.keys()
) == expected_panchang_components


# ------------------------------------------------------------
# 7. Final checkpoint
# ------------------------------------------------------------

print(
    "\nCanonical Panchang vocabulary: PASSED"
)

Panchang vocabulary source:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook09_primary_experiment/panchang_vocabulary.json

Top-level keys:
  tithi
  paksha
  karana
  nakshatra
  yoga

Frozen Panchang vocabularies
----------------------------
tithi: 30 categories
  first: 1
  last:  30
paksha: 2 categories
  first: Shukla
  last:  Krishna
karana: 11 categories
  first: Kimstughna
  last:  Naga
nakshatra: 27 categories
  first: Ashwini
  last:  Revati
yoga: 27 categories
  first: Vishkambha
  last:  Vaidhriti

Total one-hot categories: 97

Canonical Panchang vocabulary: PASSED


In [9]:
# ============================================================
# Step 9: Load the canonical JPL ephemeris and smoke-test
#         valid-time deterministic feature builders
# ============================================================
#
# Before generating the full F1-F4 auxiliary matrices, we use
# only a few forecast-valid timestamps to verify:
#
#   - Gregorian feature output and width;
#   - solar F2 feature output and width;
#   - lunar F3 feature output and width;
#   - tropical Sun/Moon longitude output required for F4;
#   - exact column names returned by the reusable helpers.
#
# This avoids guessing any API details.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical JPL ephemeris
# ------------------------------------------------------------

JPL_EPHEMERIS_PATH = (
    EPHEMERIS_DIR
    / "de440s.bsp"
)


assert JPL_EPHEMERIS_PATH.exists(), (
    f"JPL ephemeris not found: {JPL_EPHEMERIS_PATH}"
)


print(
    "JPL ephemeris:"
)

print(
    JPL_EPHEMERIS_PATH
)


# ------------------------------------------------------------
# 2. Load ephemeris through the reusable project helper
# ------------------------------------------------------------

jpl_ephemeris = (
    astronomy_features.load_jpl_ephemeris(
        JPL_EPHEMERIS_PATH
    )
)


# ------------------------------------------------------------
# 3. Select a tiny representative valid-time sample
#
# Use timestamps from the already-constructed 60-minute
# forecasting population.
# ------------------------------------------------------------

sample_valid_times = pd.DatetimeIndex(
    deep_base_modelling_tables[
        60
    ][
        "valid_time"
    ]
    .iloc[
        [
            0,
            1,
            1000,
        ]
    ]
)


print(
    "\nSample forecast-valid timestamps:"
)


for timestamp in sample_valid_times:

    print(
        " ",
        timestamp,
    )


# ------------------------------------------------------------
# 4. F1 Gregorian features
# ------------------------------------------------------------

sample_f1 = (
    preprocessing.build_gregorian_features(
        sample_valid_times
    )
)


print(
    "\nF1 Gregorian features"
)

print(
    "---------------------"
)

print(
    "Shape:",
    sample_f1.shape,
)

print(
    "Columns:"
)


for column_name in sample_f1.columns:

    print(
        " ",
        column_name,
    )


assert sample_f1.shape == (
    3,
    15,
)


# ------------------------------------------------------------
# 5. F2 solar-astronomy features
# ------------------------------------------------------------

sample_f2 = (
    astronomy_features.build_solar_f2_features(
        sample_valid_times
    )
)


print(
    "\nF2 solar features"
)

print(
    "-----------------"
)

print(
    "Shape:",
    sample_f2.shape,
)

print(
    "Columns:"
)


for column_name in sample_f2.columns:

    print(
        " ",
        column_name,
    )


assert sample_f2.shape == (
    3,
    5,
)


# ------------------------------------------------------------
# 6. F3 lunar / Sun-Moon continuous astronomy
# ------------------------------------------------------------

sample_f3 = (
    astronomy_features.build_lunar_f3_features(
        sample_valid_times,
        jpl_ephemeris,
    )
)


print(
    "\nF3 lunar/Sun-Moon features"
)

print(
    "--------------------------"
)

print(
    "Shape:",
    sample_f3.shape,
)

print(
    "Columns:"
)


for column_name in sample_f3.columns:

    print(
        " ",
        column_name,
    )


assert sample_f3.shape == (
    3,
    9,
)


# ------------------------------------------------------------
# 7. Tropical longitudes required for canonical F4 Panchang
# ------------------------------------------------------------

sample_tropical_longitudes = (
    astronomy_features.tropical_longitudes_for_solete(
        sample_valid_times,
        jpl_ephemeris,
    )
)


print(
    "\nTropical longitudes for Panchang"
)

print(
    "-------------------------------"
)

print(
    "Shape:",
    sample_tropical_longitudes.shape,
)

print(
    "Columns:"
)


for column_name in sample_tropical_longitudes.columns:

    print(
        " ",
        column_name,
    )


# ------------------------------------------------------------
# 8. Inspect small outputs
# ------------------------------------------------------------

print(
    "\nSample F1:"
)

display(
    sample_f1
)


print(
    "\nSample F2:"
)

display(
    sample_f2
)


print(
    "\nSample F3:"
)

display(
    sample_f3
)


print(
    "\nSample tropical longitudes:"
)

display(
    sample_tropical_longitudes
)


# ------------------------------------------------------------
# 9. Generic integrity checks
# ------------------------------------------------------------

for feature_table in [
    sample_f1,
    sample_f2,
    sample_f3,
    sample_tropical_longitudes,
]:

    assert feature_table.index.equals(
        sample_valid_times
    )

    assert (
        feature_table.isna()
        .sum()
        .sum()
        == 0
    )

    assert np.isfinite(
        feature_table
        .select_dtypes(
            include=np.number
        )
        .to_numpy()
    ).all()


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nDeterministic feature-builder smoke test: PASSED"
)

JPL ephemeris:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/ephemeris/de440s.bsp

Sample forecast-valid timestamps:
  2018-06-02 01:00:00+00:00
  2018-06-02 01:05:00+00:00
  2018-06-05 12:20:00+00:00

F1 Gregorian features
---------------------
Shape: (3, 15)
Columns:
  utc_year
  utc_month
  utc_day
  utc_day_of_week
  utc_day_of_year
  utc_hour
  utc_minute
  utc_minute_of_day
  utc_is_weekend
  utc_time_sin
  utc_time_cos
  utc_weekday_sin
  utc_weekday_cos
  utc_year_sin
  utc_year_cos

F2 solar features
-----------------
Shape: (3, 5)
Columns:
  solar_elevation_deg
  solar_cos_zenith
  solar_azimuth_sin
  solar_azimuth_cos
  solar_daylight

F3 lunar/Sun-Moon features
--------------------------
Shape: (3, 9)
Columns:
  moon_phase_sin
  moon_phase_cos
  moon_longitude_sin
  moon_longitude_cos
  sun_ecliptic_longitude_sin
  sun_ecliptic_longitude_cos
  moon_ecliptic_latitude_deg
  moon_illumination_fraction
  earth_moon_distance_km

Tropica

,utc_year,utc_month,utc_day,utc_day_of_week,utc_day_of_year,utc_hour,utc_minute,utc_minute_of_day,utc_is_weekend,utc_time_sin,utc_time_cos,utc_weekday_sin,utc_weekday_cos,utc_year_sin,utc_year_cos
valid_time,,,,,,,,,,,,,,,
2018-06-02 01:00:00+00:00,2018,6,2,5,153,1,0,60,1,0.258819,0.965926,-0.974928,-0.222521,0.500621,-0.865667
2018-06-02 01:05:00+00:00,2018,6,2,5,153,1,5,65,1,0.279829,0.960050,-0.974928,-0.222521,0.500569,-0.865696
2018-06-05 12:20:00+00:00,2018,6,5,1,156,12,20,740,0,-0.087156,-0.996195,0.781831,0.623490,0.448016,-0.894026



Sample F2:


,solar_elevation_deg,solar_cos_zenith,solar_azimuth_sin,solar_azimuth_cos,solar_daylight
valid_time,,,,,
2018-06-02 01:00:00+00:00,-8.571247,-0.149039,0.441480,0.897271,0
2018-06-02 01:05:00+00:00,-8.253567,-0.143554,0.459015,0.888428,0
2018-06-05 12:20:00+00:00,54.303441,0.812119,-0.488452,-0.872591,1



Sample F3:


,moon_phase_sin,moon_phase_cos,moon_longitude_sin,moon_longitude_cos,sun_ecliptic_longitude_sin,sun_ecliptic_longitude_cos,moon_ecliptic_latitude_deg,moon_illumination_fraction,earth_moon_distance_km
valid_time,,,,,,,,,
2018-06-02 01:00:00+00:00,-0.617980,-0.786194,-0.941826,0.336101,0.948162,0.317789,1.544368,0.893493,405091.945014
2018-06-02 01:05:00+00:00,-0.618498,-0.785787,-0.941585,0.336776,0.948180,0.317734,1.540866,0.893291,405094.028839
2018-06-05 12:20:00+00:00,-0.970051,-0.242903,-0.488805,0.872393,0.964997,0.262259,-2.062665,0.622662,400975.691666



Sample tropical longitudes:


,sun_tropical_longitude_deg,moon_tropical_longitude_deg
valid_time,,
2018-06-02 01:00:00+00:00,71.470755,289.639528
2018-06-02 01:05:00+00:00,71.474082,289.680583
2018-06-05 12:20:00+00:00,74.795827,330.737905



Deterministic feature-builder smoke test: PASSED


In [10]:
# ============================================================
# Step 10: Verify the F2 and F3 smoke-test objects
# ============================================================
#
# The previous textual output correctly reported:
#
#   F2 -> 5 solar features
#   F3 -> 9 lunar / Sun-Moon features
#
# However, the pasted rendering of the "Sample F2" table
# appeared to show the F3 columns.
#
# This cell verifies the actual in-memory objects before we
# construct the full valid-time feature matrices.
# ============================================================


# ------------------------------------------------------------
# 1. Expected canonical column definitions
# ------------------------------------------------------------

EXPECTED_F2_COLUMNS = [
    "solar_elevation_deg",
    "solar_cos_zenith",
    "solar_azimuth_sin",
    "solar_azimuth_cos",
    "solar_daylight",
]


EXPECTED_F3_COLUMNS = [
    "moon_phase_sin",
    "moon_phase_cos",
    "moon_longitude_sin",
    "moon_longitude_cos",
    "sun_ecliptic_longitude_sin",
    "sun_ecliptic_longitude_cos",
    "moon_ecliptic_latitude_deg",
    "moon_illumination_fraction",
    "earth_moon_distance_km",
]


# ------------------------------------------------------------
# 2. Strong object checks
# ------------------------------------------------------------

assert sample_f2.shape == (
    3,
    5,
)


assert sample_f3.shape == (
    3,
    9,
)


assert list(
    sample_f2.columns
) == EXPECTED_F2_COLUMNS


assert list(
    sample_f3.columns
) == EXPECTED_F3_COLUMNS


assert sample_f2.index.equals(
    sample_valid_times
)


assert sample_f3.index.equals(
    sample_valid_times
)


# ------------------------------------------------------------
# 3. Explicitly display F2 only
# ------------------------------------------------------------

print(
    "Verified F2 object"
)

print(
    "------------------"
)

print(
    "Shape:",
    sample_f2.shape,
)

print(
    "Columns:",
    list(
        sample_f2.columns
    ),
)


display(
    sample_f2
)


# ------------------------------------------------------------
# 4. Explicitly display F3 column identity
# ------------------------------------------------------------

print(
    "\nVerified F3 object"
)

print(
    "------------------"
)

print(
    "Shape:",
    sample_f3.shape,
)

print(
    "Columns:",
    list(
        sample_f3.columns
    ),
)


# ------------------------------------------------------------
# 5. Ensure F2 and F3 are genuinely different feature sets
# ------------------------------------------------------------

assert set(
    sample_f2.columns
).isdisjoint(
    set(
        sample_f3.columns
    )
)


print(
    "\nF2/F3 object identity verification: PASSED"
)

Verified F2 object
------------------
Shape: (3, 5)
Columns: ['solar_elevation_deg', 'solar_cos_zenith', 'solar_azimuth_sin', 'solar_azimuth_cos', 'solar_daylight']


,solar_elevation_deg,solar_cos_zenith,solar_azimuth_sin,solar_azimuth_cos,solar_daylight
valid_time,,,,,
2018-06-02 01:00:00+00:00,-8.571247,-0.149039,0.441480,0.897271,0
2018-06-02 01:05:00+00:00,-8.253567,-0.143554,0.459015,0.888428,0
2018-06-05 12:20:00+00:00,54.303441,0.812119,-0.488452,-0.872591,1



Verified F3 object
------------------
Shape: (3, 9)
Columns: ['moon_phase_sin', 'moon_phase_cos', 'moon_longitude_sin', 'moon_longitude_cos', 'sun_ecliptic_longitude_sin', 'sun_ecliptic_longitude_cos', 'moon_ecliptic_latitude_deg', 'moon_illumination_fraction', 'earth_moon_distance_km']

F2/F3 object identity verification: PASSED


In [11]:
# ============================================================
# Step 11: Smoke-test the canonical F4 Panchang construction
# ============================================================
#
# Notebook 08 established an important SOLETE timing detail:
#
#     astronomical reference time
#         = SOLETE timestamp + 2 minutes
#
# The reusable tropical-longitude helper already follows the
# SOLETE astronomy convention.
#
# For the Lahiri ayanamsha used by the Panchang calculation,
# we therefore explicitly evaluate the ayanamsha at the same
# +2-minute astronomical reference times.
#
# This cell verifies:
#
#   1. the +2-minute representative timestamps;
#   2. canonical Lahiri ayanamsha values;
#   3. raw Panchang F4 categories;
#   4. fixed-vocabulary 97-column one-hot encoding;
#   5. exactly five active Panchang indicators per row.
#
# Only the three smoke-test timestamps are used here.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical SOLETE astronomical reference times
# ------------------------------------------------------------

sample_astronomy_reference_times = (
    astronomy_features.solete_astronomy_reference_times(
        sample_valid_times
    )
)


assert sample_astronomy_reference_times.equals(
    sample_valid_times
    + pd.Timedelta(
        minutes=2
    )
)


print(
    "SOLETE valid times:"
)

for timestamp in sample_valid_times:

    print(
        " ",
        timestamp,
    )


print(
    "\nAstronomical reference times (+2 min):"
)

for timestamp in sample_astronomy_reference_times:

    print(
        " ",
        timestamp,
    )


# ------------------------------------------------------------
# 2. Canonical Lahiri ayanamsha
#
# IMPORTANT:
# evaluate at the same representative (+2 min) times used by
# the Sun/Moon astronomical geometry.
# ------------------------------------------------------------

sample_ayanamsha_values = (
    vedic_calendar.lahiri_ayanamsha_for_utc_times(
        sample_astronomy_reference_times
    )
)


sample_ayanamsha = pd.Series(
    sample_ayanamsha_values,
    index=sample_valid_times,
    name="lahiri_ayanamsha_deg",
    dtype=float,
)


assert sample_ayanamsha.shape == (
    3,
)


assert np.isfinite(
    sample_ayanamsha.to_numpy()
).all()


# ------------------------------------------------------------
# 3. Reuse the tropical Sun/Moon longitudes already computed
#    for the SOLETE valid timestamps.
# ------------------------------------------------------------

assert sample_tropical_longitudes.index.equals(
    sample_valid_times
)


# ------------------------------------------------------------
# 4. Construct raw canonical Panchang F4 categories
# ------------------------------------------------------------

sample_f4_raw = (
    vedic_calendar.build_panchang_f4_raw(
        sun_tropical_longitude_deg=(
            sample_tropical_longitudes[
                "sun_tropical_longitude_deg"
            ]
        ),
        moon_tropical_longitude_deg=(
            sample_tropical_longitudes[
                "moon_tropical_longitude_deg"
            ]
        ),
        ayanamsha_deg=sample_ayanamsha,
    )
)


EXPECTED_F4_RAW_COLUMNS = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]


assert list(
    sample_f4_raw.columns
) == EXPECTED_F4_RAW_COLUMNS


assert sample_f4_raw.index.equals(
    sample_valid_times
)


assert sample_f4_raw.shape == (
    3,
    5,
)


assert (
    sample_f4_raw.isna()
    .sum()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 5. Verify every raw category belongs to the frozen
#    Notebook-09 vocabulary.
# ------------------------------------------------------------

for column_name in EXPECTED_F4_RAW_COLUMNS:

    allowed_categories = set(
        panchang_vocabulary[
            column_name
        ]
    )


    observed_categories = set(
        sample_f4_raw[
            column_name
        ].tolist()
    )


    assert observed_categories.issubset(
        allowed_categories
    ), (
        f"Unexpected {column_name} category: "
        f"{observed_categories - allowed_categories}"
    )


# ------------------------------------------------------------
# 6. Build the canonical fixed-vocabulary one-hot matrix
#
# We do this explicitly rather than allowing pandas to infer
# categories from the current sample.
#
# This guarantees:
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 columns
# ------------------------------------------------------------

sample_f4_encoded_parts = []


for column_name in EXPECTED_F4_RAW_COLUMNS:

    categories = (
        panchang_vocabulary[
            column_name
        ]
    )


    categorical_series = pd.Categorical(
        sample_f4_raw[
            column_name
        ],
        categories=categories,
        ordered=True,
    )


    encoded_part = pd.DataFrame(
        {
            (
                f"{column_name}__"
                f"{category}"
            ):
                (
                    categorical_series
                    == category
                ).astype(
                    np.float32
                )
            for category in categories
        },
        index=sample_valid_times,
    )


    sample_f4_encoded_parts.append(
        encoded_part
    )


sample_f4_encoded = pd.concat(
    sample_f4_encoded_parts,
    axis=1,
)


# ------------------------------------------------------------
# 7. Strong F4 encoding checks
# ------------------------------------------------------------

assert sample_f4_encoded.shape == (
    3,
    97,
)


assert sample_f4_encoded.index.equals(
    sample_valid_times
)


assert (
    sample_f4_encoded.isna()
    .sum()
    .sum()
    == 0
)


assert np.isfinite(
    sample_f4_encoded.to_numpy()
).all()


# Every timestamp must activate exactly:
#
#   1 Tithi
#   1 Paksha
#   1 Karana
#   1 Nakshatra
#   1 Yoga
#
# = 5 active indicators.
assert np.array_equal(
    sample_f4_encoded.sum(
        axis=1
    ).to_numpy(),
    np.full(
        3,
        5.0,
    ),
)


# ------------------------------------------------------------
# 8. Report smoke-test outputs
# ------------------------------------------------------------

print(
    "\nCanonical Lahiri ayanamsha:"
)

display(
    sample_ayanamsha.to_frame()
)


print(
    "\nRaw F4 Panchang categories:"
)

display(
    sample_f4_raw
)


print(
    "\nEncoded F4 shape:"
)

print(
    sample_f4_encoded.shape
)


print(
    "Active one-hot indicators per row:"
)

print(
    sample_f4_encoded.sum(
        axis=1
    ).to_list()
)


print(
    "\nFirst 10 encoded F4 column names:"
)

for column_name in (
    sample_f4_encoded.columns[
        :10
    ]
):

    print(
        " ",
        column_name,
    )


print(
    "\nLast 10 encoded F4 column names:"
)

for column_name in (
    sample_f4_encoded.columns[
        -10:
    ]
):

    print(
        " ",
        column_name,
    )


# ------------------------------------------------------------
# 9. Equivalent nested feature widths
#
# Sequence branch contributes:
#
#       13 × 9 = 117
#
# Auxiliary widths are:
#
#       F0:  1
#       F1:  1 + 15
#       F2:  1 + 15 + 5
#       F3:  1 + 15 + 5 + 9
#       F4:  1 + 15 + 5 + 9 + 97
#
# These must reproduce the classical feature-family widths.
# ------------------------------------------------------------

deep_equivalent_feature_counts = {
    "F0":
        117 + 1,

    "F1":
        117 + 1 + 15,

    "F2":
        117 + 1 + 15 + 5,

    "F3":
        117 + 1 + 15 + 5 + 9,

    "F4":
        117 + 1 + 15 + 5 + 9 + 97,
}


assert deep_equivalent_feature_counts == {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}


print(
    "\nEquivalent feature-family widths:"
)

print(
    deep_equivalent_feature_counts
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nCanonical F4 Panchang smoke test: PASSED"
)

SOLETE valid times:
  2018-06-02 01:00:00+00:00
  2018-06-02 01:05:00+00:00
  2018-06-05 12:20:00+00:00

Astronomical reference times (+2 min):
  2018-06-02 01:02:00+00:00
  2018-06-02 01:07:00+00:00
  2018-06-05 12:22:00+00:00

Canonical Lahiri ayanamsha:


,lahiri_ayanamsha_deg
valid_time,
2018-06-02 01:00:00+00:00,24.110408
2018-06-02 01:05:00+00:00,24.110409
2018-06-05 12:20:00+00:00,24.110576



Raw F4 Panchang categories:


,tithi,paksha,karana,nakshatra,yoga
valid_time,,,,,
2018-06-02 01:00:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 01:05:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-05 12:20:00+00:00,22,Krishna,Vishti,Dhanishta,Vaidhriti



Encoded F4 shape:
(3, 97)
Active one-hot indicators per row:
[5.0, 5.0, 5.0]

First 10 encoded F4 column names:
  tithi__1
  tithi__2
  tithi__3
  tithi__4
  tithi__5
  tithi__6
  tithi__7
  tithi__8
  tithi__9
  tithi__10

Last 10 encoded F4 column names:
  yoga__Variyana
  yoga__Parigha
  yoga__Shiva
  yoga__Siddha
  yoga__Sadhya
  yoga__Shubha
  yoga__Shukla
  yoga__Brahma
  yoga__Indra
  yoga__Vaidhriti

Equivalent feature-family widths:
{'F0': 118, 'F1': 133, 'F2': 138, 'F3': 147, 'F4': 244}

Canonical F4 Panchang smoke test: PASSED


In [12]:
# ============================================================
# Step 12: Build the full canonical valid-time feature bank
#          for F1 -> F4
# ============================================================
#
# All deterministic features are functions of forecast-valid
# time only.
#
# Therefore we compute them ONCE over the union of valid times
# required by all three horizons and later reindex them for:
#
#       15 min
#       30 min
#       60 min
#
# This avoids repeating the expensive astronomical/Panchang
# calculations separately for each horizon.
#
# Canonical feature groups:
#
#   F1 additions = 15 Gregorian features
#   F2 additions =  5 solar features
#   F3 additions =  9 lunar/Sun-Moon features
#   F4 additions = 97 fixed-vocabulary Panchang one-hot columns
#
# The raw F4 categories are retained separately for auditing.
# ============================================================


# ------------------------------------------------------------
# 1. Construct the union of all required forecast-valid times
# ------------------------------------------------------------

all_required_valid_times = pd.DatetimeIndex(
    sorted(
        set().union(
            *[
                set(
                    deep_base_modelling_tables[
                        horizon_minutes
                    ][
                        "valid_time"
                    ]
                )
                for horizon_minutes
                in DEEP_FORECAST_HORIZONS
            ]
        )
    )
)


# ------------------------------------------------------------
# 2. The 15-minute population should already span the complete
#    required valid-time grid.
# ------------------------------------------------------------

valid_times_15 = pd.DatetimeIndex(
    deep_base_modelling_tables[
        15
    ][
        "valid_time"
    ]
)


assert all_required_valid_times.equals(
    valid_times_15
)


assert (
    len(
        all_required_valid_times
    )
    == 131_325
)


assert (
    all_required_valid_times[0]
    == pd.Timestamp(
        "2018-06-02 00:15:00",
        tz="UTC",
    )
)


assert (
    all_required_valid_times[-1]
    == pd.Timestamp(
        "2019-08-31 23:55:00",
        tz="UTC",
    )
)


print(
    "Full valid-time feature bank"
)

print(
    "----------------------------"
)

print(
    "Rows:",
    len(
        all_required_valid_times
    ),
)

print(
    "Start:",
    all_required_valid_times[0],
)

print(
    "End:",
    all_required_valid_times[-1],
)


# ------------------------------------------------------------
# 3. F1 — Gregorian valid-time features
# ------------------------------------------------------------

f1_start_time = time.perf_counter()


deep_f1_valid_features = (
    preprocessing.build_gregorian_features(
        all_required_valid_times
    )
)


f1_seconds = (
    time.perf_counter()
    - f1_start_time
)


assert deep_f1_valid_features.shape == (
    131_325,
    15,
)


assert list(
    deep_f1_valid_features.columns
) == list(
    sample_f1.columns
)


assert deep_f1_valid_features.index.equals(
    all_required_valid_times
)


print(
    "\nF1 Gregorian features:"
)

print(
    "  Shape:",
    deep_f1_valid_features.shape,
)

print(
    f"  Runtime: {f1_seconds:.2f} s"
)


# ------------------------------------------------------------
# 4. F2 — Solar geometry
# ------------------------------------------------------------

f2_start_time = time.perf_counter()


deep_f2_valid_features = (
    astronomy_features.build_solar_f2_features(
        all_required_valid_times
    )
)


f2_seconds = (
    time.perf_counter()
    - f2_start_time
)


assert deep_f2_valid_features.shape == (
    131_325,
    5,
)


assert list(
    deep_f2_valid_features.columns
) == EXPECTED_F2_COLUMNS


assert deep_f2_valid_features.index.equals(
    all_required_valid_times
)


print(
    "\nF2 solar features:"
)

print(
    "  Shape:",
    deep_f2_valid_features.shape,
)

print(
    f"  Runtime: {f2_seconds:.2f} s"
)


# ------------------------------------------------------------
# 5. F3 — Continuous lunar / Sun-Moon astronomy
# ------------------------------------------------------------

f3_start_time = time.perf_counter()


deep_f3_valid_features = (
    astronomy_features.build_lunar_f3_features(
        all_required_valid_times,
        jpl_ephemeris,
    )
)


f3_seconds = (
    time.perf_counter()
    - f3_start_time
)


assert deep_f3_valid_features.shape == (
    131_325,
    9,
)


assert list(
    deep_f3_valid_features.columns
) == EXPECTED_F3_COLUMNS


assert deep_f3_valid_features.index.equals(
    all_required_valid_times
)


print(
    "\nF3 lunar/Sun-Moon features:"
)

print(
    "  Shape:",
    deep_f3_valid_features.shape,
)

print(
    f"  Runtime: {f3_seconds:.2f} s"
)


# ------------------------------------------------------------
# 6. Tropical Sun/Moon longitudes needed for canonical F4
# ------------------------------------------------------------

longitude_start_time = time.perf_counter()


deep_tropical_longitudes = (
    astronomy_features.tropical_longitudes_for_solete(
        all_required_valid_times,
        jpl_ephemeris,
    )
)


longitude_seconds = (
    time.perf_counter()
    - longitude_start_time
)


assert deep_tropical_longitudes.shape == (
    131_325,
    2,
)


assert list(
    deep_tropical_longitudes.columns
) == [
    "sun_tropical_longitude_deg",
    "moon_tropical_longitude_deg",
]


assert deep_tropical_longitudes.index.equals(
    all_required_valid_times
)


print(
    "\nTropical longitudes:"
)

print(
    "  Shape:",
    deep_tropical_longitudes.shape,
)

print(
    f"  Runtime: {longitude_seconds:.2f} s"
)


# ------------------------------------------------------------
# 7. Canonical SOLETE astronomical reference timestamps
#
# Panchang ayanamsha must use the same +2-minute reference
# convention as the Sun/Moon geometry.
# ------------------------------------------------------------

deep_astronomy_reference_times = (
    astronomy_features.solete_astronomy_reference_times(
        all_required_valid_times
    )
)


assert deep_astronomy_reference_times.equals(
    all_required_valid_times
    + pd.Timedelta(
        minutes=2
    )
)


# ------------------------------------------------------------
# 8. Canonical extended Lahiri ayanamsha
# ------------------------------------------------------------

ayanamsha_start_time = time.perf_counter()


deep_ayanamsha_values = (
    vedic_calendar.lahiri_ayanamsha_for_utc_times(
        deep_astronomy_reference_times
    )
)


ayanamsha_seconds = (
    time.perf_counter()
    - ayanamsha_start_time
)


deep_ayanamsha = pd.Series(
    deep_ayanamsha_values,
    index=all_required_valid_times,
    name="lahiri_ayanamsha_deg",
    dtype=float,
)


assert deep_ayanamsha.shape == (
    131_325,
)


assert np.isfinite(
    deep_ayanamsha.to_numpy()
).all()


print(
    "\nCanonical Lahiri ayanamsha:"
)

print(
    "  Rows:",
    len(
        deep_ayanamsha
    ),
)

print(
    "  Range:",
    (
        float(
            deep_ayanamsha.min()
        ),
        float(
            deep_ayanamsha.max()
        ),
    ),
)

print(
    f"  Runtime: {ayanamsha_seconds:.2f} s"
)


# ------------------------------------------------------------
# 9. Raw canonical F4 Panchang categories
# ------------------------------------------------------------

f4_raw_start_time = time.perf_counter()


deep_f4_raw = (
    vedic_calendar.build_panchang_f4_raw(
        sun_tropical_longitude_deg=(
            deep_tropical_longitudes[
                "sun_tropical_longitude_deg"
            ]
        ),
        moon_tropical_longitude_deg=(
            deep_tropical_longitudes[
                "moon_tropical_longitude_deg"
            ]
        ),
        ayanamsha_deg=deep_ayanamsha,
    )
)


f4_raw_seconds = (
    time.perf_counter()
    - f4_raw_start_time
)


assert deep_f4_raw.shape == (
    131_325,
    5,
)


assert list(
    deep_f4_raw.columns
) == EXPECTED_F4_RAW_COLUMNS


assert deep_f4_raw.index.equals(
    all_required_valid_times
)


assert (
    deep_f4_raw.isna()
    .sum()
    .sum()
    == 0
)


print(
    "\nRaw F4 Panchang categories:"
)

print(
    "  Shape:",
    deep_f4_raw.shape,
)

print(
    f"  Runtime: {f4_raw_seconds:.2f} s"
)


# ------------------------------------------------------------
# 10. Verify every observed F4 category belongs to the
#     Notebook-09 frozen vocabulary.
# ------------------------------------------------------------

for column_name in EXPECTED_F4_RAW_COLUMNS:

    allowed_categories = set(
        panchang_vocabulary[
            column_name
        ]
    )


    observed_categories = set(
        deep_f4_raw[
            column_name
        ].unique()
    )


    unexpected_categories = (
        observed_categories
        - allowed_categories
    )


    assert not unexpected_categories, (
        f"Unexpected {column_name} categories found: "
        f"{unexpected_categories}"
    )


# ------------------------------------------------------------
# 11. Fixed-vocabulary F4 one-hot encoding
#
# We explicitly encode every category in the frozen vocabulary
# regardless of whether it appears in a particular time span.
# ------------------------------------------------------------

f4_encode_start_time = time.perf_counter()


deep_f4_encoded_parts = []


for column_name in EXPECTED_F4_RAW_COLUMNS:

    categories = (
        panchang_vocabulary[
            column_name
        ]
    )


    categorical_values = pd.Categorical(
        deep_f4_raw[
            column_name
        ],
        categories=categories,
        ordered=True,
    )


    encoded_part = pd.DataFrame(
        {
            (
                f"{column_name}__"
                f"{category}"
            ):
                (
                    categorical_values
                    == category
                ).astype(
                    np.float32
                )
            for category in categories
        },
        index=all_required_valid_times,
    )


    deep_f4_encoded_parts.append(
        encoded_part
    )


deep_f4_valid_features = pd.concat(
    deep_f4_encoded_parts,
    axis=1,
)


f4_encode_seconds = (
    time.perf_counter()
    - f4_encode_start_time
)


assert deep_f4_valid_features.shape == (
    131_325,
    97,
)


assert deep_f4_valid_features.index.equals(
    all_required_valid_times
)


assert (
    deep_f4_valid_features.isna()
    .sum()
    .sum()
    == 0
)


assert np.isfinite(
    deep_f4_valid_features.to_numpy()
).all()


# ------------------------------------------------------------
# 12. Every row must activate exactly five one-hot indicators
# ------------------------------------------------------------

f4_active_counts = (
    deep_f4_valid_features
    .sum(
        axis=1
    )
    .to_numpy()
)


assert np.array_equal(
    f4_active_counts,
    np.full(
        len(
            deep_f4_valid_features
        ),
        5.0,
    ),
)


print(
    "\nEncoded F4 Panchang features:"
)

print(
    "  Shape:",
    deep_f4_valid_features.shape,
)

print(
    "  Active indicators per row:",
    np.unique(
        f4_active_counts
    ).tolist(),
)

print(
    f"  Runtime: {f4_encode_seconds:.2f} s"
)


# ------------------------------------------------------------
# 13. Global missing / finite-value checks
# ------------------------------------------------------------

for (
    feature_name,
    feature_table,
) in {
    "F1":
        deep_f1_valid_features,

    "F2":
        deep_f2_valid_features,

    "F3":
        deep_f3_valid_features,

    "F4":
        deep_f4_valid_features,
}.items():

    assert (
        feature_table.isna()
        .sum()
        .sum()
        == 0
    ), (
        f"{feature_name} contains missing values."
    )


    assert np.isfinite(
        feature_table
        .to_numpy(
            dtype=np.float64
        )
    ).all(), (
        f"{feature_name} contains non-finite values."
    )


# ------------------------------------------------------------
# 14. Confirm canonical nested auxiliary widths
#
# The short-history sequence itself contains 117 values.
#
# Auxiliary widths:
#
#       F0 = 1
#       F1 = 16
#       F2 = 21
#       F3 = 30
#       F4 = 127
# ------------------------------------------------------------

deep_auxiliary_feature_counts = {
    "F0": 1,
    "F1": 1 + 15,
    "F2": 1 + 15 + 5,
    "F3": 1 + 15 + 5 + 9,
    "F4": 1 + 15 + 5 + 9 + 97,
}


assert deep_auxiliary_feature_counts == {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    assert (
        117
        + deep_auxiliary_feature_counts[
            feature_family
        ]
        == deep_equivalent_feature_counts[
            feature_family
        ]
    )


# ------------------------------------------------------------
# 15. Report memory footprint
# ------------------------------------------------------------

feature_bank_memory_mb = {
    "F1":
        (
            deep_f1_valid_features
            .memory_usage(
                deep=True
            )
            .sum()
            / 1024**2
        ),

    "F2":
        (
            deep_f2_valid_features
            .memory_usage(
                deep=True
            )
            .sum()
            / 1024**2
        ),

    "F3":
        (
            deep_f3_valid_features
            .memory_usage(
                deep=True
            )
            .sum()
            / 1024**2
        ),

    "F4":
        (
            deep_f4_valid_features
            .memory_usage(
                deep=True
            )
            .sum()
            / 1024**2
        ),
}


# ------------------------------------------------------------
# 16. Total deterministic-feature runtime
# ------------------------------------------------------------

total_feature_bank_seconds = (
    f1_seconds
    + f2_seconds
    + f3_seconds
    + longitude_seconds
    + ayanamsha_seconds
    + f4_raw_seconds
    + f4_encode_seconds
)


# ------------------------------------------------------------
# 17. Final report
# ------------------------------------------------------------

print(
    "\nCanonical deterministic feature bank"
)

print(
    "------------------------------------"
)


print(
    "F1 shape:",
    deep_f1_valid_features.shape,
)

print(
    "F2 shape:",
    deep_f2_valid_features.shape,
)

print(
    "F3 shape:",
    deep_f3_valid_features.shape,
)

print(
    "F4 shape:",
    deep_f4_valid_features.shape,
)


print(
    "\nAuxiliary feature counts:"
)

print(
    deep_auxiliary_feature_counts
)


print(
    "\nEquivalent total F0-F4 widths:"
)

print(
    deep_equivalent_feature_counts
)


print(
    "\nApproximate feature-bank memory:"
)

for (
    feature_name,
    memory_mb,
) in feature_bank_memory_mb.items():

    print(
        f"  {feature_name}: "
        f"{memory_mb:.2f} MB"
    )


print(
    "\nTotal deterministic feature-bank runtime:"
)

print(
    f"  {total_feature_bank_seconds:.2f} seconds"
)

print(
    f"  {total_feature_bank_seconds / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\nCanonical F1-F4 valid-time feature bank: PASSED"
)

Full valid-time feature bank
----------------------------
Rows: 131325
Start: 2018-06-02 00:15:00+00:00
End: 2019-08-31 23:55:00+00:00

F1 Gregorian features:
  Shape: (131325, 15)
  Runtime: 0.03 s

F2 solar features:
  Shape: (131325, 5)
  Runtime: 0.24 s

F3 lunar/Sun-Moon features:
  Shape: (131325, 9)
  Runtime: 5.08 s

Tropical longitudes:
  Shape: (131325, 2)
  Runtime: 3.76 s

Canonical Lahiri ayanamsha:
  Rows: 131325
  Range: (24.110406482629656, 24.12737225615406)
  Runtime: 0.40 s

Raw F4 Panchang categories:
  Shape: (131325, 5)
  Runtime: 0.04 s

Encoded F4 Panchang features:
  Shape: (131325, 97)
  Active indicators per row: [5.0]
  Runtime: 0.01 s

Canonical deterministic feature bank
------------------------------------
F1 shape: (131325, 15)
F2 shape: (131325, 5)
F3 shape: (131325, 9)
F4 shape: (131325, 97)

Auxiliary feature counts:
{'F0': 1, 'F1': 16, 'F2': 21, 'F3': 30, 'F4': 127}

Equivalent total F0-F4 widths:
{'F0': 118, 'F1': 133, 'F2': 138, 'F3': 147, 'F4': 24

In [13]:
# ============================================================
# Step 13: Assemble horizon-specific raw auxiliary matrices
# ============================================================
#
# We now align the deterministic valid-time feature bank with
# the issue-time forecasting populations for:
#
#       15 min
#       30 min
#       60 min
#
# For each horizon we create ONE raw auxiliary table containing
# the complete F4 information:
#
#       1   PV 24-hour lag
#      15   Gregorian features
#       5   solar features
#       9   lunar / Sun-Moon features
#      97   Panchang one-hot features
#     ----
#     127   total auxiliary columns
#
# F0-F4 are then represented as nested ordered column subsets:
#
#       F0 =   1 auxiliary column
#       F1 =  16
#       F2 =  21
#       F3 =  30
#       F4 = 127
#
# The historical sequence branch remains separate:
#
#       (13 timesteps, 9 channels)
#
# No scaling is performed yet.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical ordered auxiliary column groups
# ------------------------------------------------------------

DEEP_AUX_F0_COLUMNS = [
    "pv_lag_24h",
]


DEEP_AUX_F1_ADDITIONS = list(
    deep_f1_valid_features.columns
)


DEEP_AUX_F2_ADDITIONS = list(
    deep_f2_valid_features.columns
)


DEEP_AUX_F3_ADDITIONS = list(
    deep_f3_valid_features.columns
)


DEEP_AUX_F4_ADDITIONS = list(
    deep_f4_valid_features.columns
)


# ------------------------------------------------------------
# 2. Nested F0-F4 column definitions
# ------------------------------------------------------------

DEEP_AUX_FAMILY_COLUMNS = {
    "F0":
        DEEP_AUX_F0_COLUMNS,

    "F1":
        (
            DEEP_AUX_F0_COLUMNS
            + DEEP_AUX_F1_ADDITIONS
        ),

    "F2":
        (
            DEEP_AUX_F0_COLUMNS
            + DEEP_AUX_F1_ADDITIONS
            + DEEP_AUX_F2_ADDITIONS
        ),

    "F3":
        (
            DEEP_AUX_F0_COLUMNS
            + DEEP_AUX_F1_ADDITIONS
            + DEEP_AUX_F2_ADDITIONS
            + DEEP_AUX_F3_ADDITIONS
        ),

    "F4":
        (
            DEEP_AUX_F0_COLUMNS
            + DEEP_AUX_F1_ADDITIONS
            + DEEP_AUX_F2_ADDITIONS
            + DEEP_AUX_F3_ADDITIONS
            + DEEP_AUX_F4_ADDITIONS
        ),
}


assert {
    family: len(columns)
    for family, columns
    in DEEP_AUX_FAMILY_COLUMNS.items()
} == {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


# ------------------------------------------------------------
# 3. Ensure the nested feature-family property holds exactly
# ------------------------------------------------------------

assert (
    DEEP_AUX_FAMILY_COLUMNS["F1"][
        :len(
            DEEP_AUX_FAMILY_COLUMNS["F0"]
        )
    ]
    == DEEP_AUX_FAMILY_COLUMNS["F0"]
)


assert (
    DEEP_AUX_FAMILY_COLUMNS["F2"][
        :len(
            DEEP_AUX_FAMILY_COLUMNS["F1"]
        )
    ]
    == DEEP_AUX_FAMILY_COLUMNS["F1"]
)


assert (
    DEEP_AUX_FAMILY_COLUMNS["F3"][
        :len(
            DEEP_AUX_FAMILY_COLUMNS["F2"]
        )
    ]
    == DEEP_AUX_FAMILY_COLUMNS["F2"]
)


assert (
    DEEP_AUX_FAMILY_COLUMNS["F4"][
        :len(
            DEEP_AUX_FAMILY_COLUMNS["F3"]
        )
    ]
    == DEEP_AUX_FAMILY_COLUMNS["F3"]
)


# ------------------------------------------------------------
# 4. Verify no duplicate auxiliary column names exist
# ------------------------------------------------------------

assert (
    len(
        DEEP_AUX_FAMILY_COLUMNS["F4"]
    )
    == len(
        set(
            DEEP_AUX_FAMILY_COLUMNS["F4"]
        )
    )
)


# ------------------------------------------------------------
# 5. Build one complete raw auxiliary table per horizon
#
# Index:
#
#       issue_time
#
# Feature values:
#
#       deterministic features evaluated at valid_time
#
# This makes the alignment with the sequence tensor explicit.
# ------------------------------------------------------------

deep_auxiliary_raw = {}


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    valid_times = pd.DatetimeIndex(
        base_table[
            "valid_time"
        ]
    )


    # --------------------------------------------------------
    # Reindex each deterministic feature group by valid time.
    # --------------------------------------------------------

    f1_aligned = (
        deep_f1_valid_features
        .reindex(
            valid_times
        )
    )


    f2_aligned = (
        deep_f2_valid_features
        .reindex(
            valid_times
        )
    )


    f3_aligned = (
        deep_f3_valid_features
        .reindex(
            valid_times
        )
    )


    f4_aligned = (
        deep_f4_valid_features
        .reindex(
            valid_times
        )
    )


    # --------------------------------------------------------
    # Strong valid-time alignment checks
    # --------------------------------------------------------

    assert f1_aligned.index.equals(
        valid_times
    )

    assert f2_aligned.index.equals(
        valid_times
    )

    assert f3_aligned.index.equals(
        valid_times
    )

    assert f4_aligned.index.equals(
        valid_times
    )


    # --------------------------------------------------------
    # Construct issue-time-indexed auxiliary table.
    #
    # All numerical values are stored as float32 for later
    # TensorFlow use.
    # --------------------------------------------------------

    auxiliary_table = pd.DataFrame(
        index=base_table.index
    )


    auxiliary_table.index.name = (
        "issue_time"
    )


    auxiliary_table[
        "pv_lag_24h"
    ] = (
        base_table[
            "pv_lag_24h"
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    for (
        source_table,
        column_names,
    ) in [
        (
            f1_aligned,
            DEEP_AUX_F1_ADDITIONS,
        ),
        (
            f2_aligned,
            DEEP_AUX_F2_ADDITIONS,
        ),
        (
            f3_aligned,
            DEEP_AUX_F3_ADDITIONS,
        ),
        (
            f4_aligned,
            DEEP_AUX_F4_ADDITIONS,
        ),
    ]:

        auxiliary_table[
            column_names
        ] = (
            source_table[
                column_names
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


    # --------------------------------------------------------
    # Enforce canonical full-F4 ordering
    # --------------------------------------------------------

    auxiliary_table = (
        auxiliary_table[
            DEEP_AUX_FAMILY_COLUMNS[
                "F4"
            ]
        ]
    )


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert auxiliary_table.index.equals(
        base_table.index
    )


    assert auxiliary_table.shape == (
        len(
            base_table
        ),
        127,
    )


    assert (
        auxiliary_table.isna()
        .sum()
        .sum()
        == 0
    )


    assert np.isfinite(
        auxiliary_table.to_numpy()
    ).all()


    assert all(
        dtype == np.float32
        for dtype in auxiliary_table.dtypes
    )


    # --------------------------------------------------------
    # F4 one-hot portion must still contain exactly five active
    # Panchang indicators per sample.
    # --------------------------------------------------------

    panchang_active_counts = (
        auxiliary_table[
            DEEP_AUX_F4_ADDITIONS
        ]
        .sum(
            axis=1
        )
        .to_numpy()
    )


    assert np.array_equal(
        panchang_active_counts,
        np.full(
            len(
                auxiliary_table
            ),
            5.0,
            dtype=np.float32,
        ),
    )


    deep_auxiliary_raw[
        horizon_minutes
    ] = auxiliary_table


# ------------------------------------------------------------
# 6. Verify sequence / auxiliary / target row alignment
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    sequence_positions = (
        deep_sequence_positions[
            horizon_minutes
        ]
    )


    assert (
        len(
            sequence_positions
        )
        == len(
            auxiliary_table
        )
        == len(
            base_table
        )
    )


    # Sequence endpoint must correspond exactly to issue time.
    issue_times_from_sequence = (
        deep_history_issue_index[
            sequence_positions
        ]
    )


    assert issue_times_from_sequence.equals(
        base_table.index
    )


    assert auxiliary_table.index.equals(
        base_table.index
    )


# ------------------------------------------------------------
# 7. Report dimensions for all horizons / families
# ------------------------------------------------------------

print(
    "Horizon-specific auxiliary matrices"
)

print(
    "-----------------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    print(
        f"\n{horizon_minutes}-minute horizon"
    )

    print(
        "  Full F4 auxiliary shape:",
        auxiliary_table.shape,
    )


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        n_aux = len(
            DEEP_AUX_FAMILY_COLUMNS[
                feature_family
            ]
        )


        equivalent_total_width = (
            117
            + n_aux
        )


        print(
            f"  {feature_family}: "
            f"aux={n_aux:3d}, "
            f"equivalent total={equivalent_total_width:3d}"
        )


# ------------------------------------------------------------
# 8. Memory report
# ------------------------------------------------------------

print(
    "\nRaw auxiliary-table memory"
)

print(
    "--------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    memory_mb = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
        .memory_usage(
            deep=True
        )
        .sum()
        / 1024**2
    )


    print(
        f"  {horizon_minutes:2d} min: "
        f"{memory_mb:.2f} MB"
    )


# ------------------------------------------------------------
# 9. Display canonical F4 auxiliary column boundaries
# ------------------------------------------------------------

print(
    "\nFirst 10 F4 auxiliary columns:"
)

for column_name in (
    DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ][
        :10
    ]
):

    print(
        " ",
        column_name,
    )


print(
    "\nLast 10 F4 auxiliary columns:"
)

for column_name in (
    DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ][
        -10:
    ]
):

    print(
        " ",
        column_name,
    )


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nRaw horizon-specific auxiliary assembly: PASSED"
)

Horizon-specific auxiliary matrices
-----------------------------------

15-minute horizon
  Full F4 auxiliary shape: (131325, 127)
  F0: aux=  1, equivalent total=118
  F1: aux= 16, equivalent total=133
  F2: aux= 21, equivalent total=138
  F3: aux= 30, equivalent total=147
  F4: aux=127, equivalent total=244

30-minute horizon
  Full F4 auxiliary shape: (131322, 127)
  F0: aux=  1, equivalent total=118
  F1: aux= 16, equivalent total=133
  F2: aux= 21, equivalent total=138
  F3: aux= 30, equivalent total=147
  F4: aux=127, equivalent total=244

60-minute horizon
  Full F4 auxiliary shape: (131316, 127)
  F0: aux=  1, equivalent total=118
  F1: aux= 16, equivalent total=133
  F2: aux= 21, equivalent total=138
  F3: aux= 30, equivalent total=147
  F4: aux=127, equivalent total=244

Raw auxiliary-table memory
--------------------------
  15 min: 64.62 MB
  30 min: 64.62 MB
  60 min: 64.62 MB

First 10 F4 auxiliary columns:
  pv_lag_24h
  utc_year
  utc_month
  utc_day
  utc_day_of_week


/var/folders/c2/r4zn6_kd1cx7xc1k98qrt5qw0000gn/T/ipykernel_30919/1131060046.py:322: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  auxiliary_table[
/var/folders/c2/r4zn6_kd1cx7xc1k98qrt5qw0000gn/T/ipykernel_30919/1131060046.py:322: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  auxiliary_table[
/var/folders/c2/r4zn6_kd1cx7xc1k98qrt5qw0000gn/T/ipykernel_30919/1131060046.py:322: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

In [14]:
# ============================================================
# Step 14: Defragment horizon-specific auxiliary DataFrames
# ============================================================
#
# The previous construction produced the correct matrices, but
# pandas emitted a PerformanceWarning because many columns were
# inserted sequentially.
#
# This affects internal memory layout / performance only.
# It does NOT indicate incorrect feature values.
#
# A DataFrame.copy() rebuilds each table into a compact internal
# representation. We do this once before scaling and TensorFlow
# conversion.
# ============================================================


# ------------------------------------------------------------
# 1. Defragment each horizon-specific auxiliary table
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    deep_auxiliary_raw[
        horizon_minutes
    ] = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
        .copy()
    )


# ------------------------------------------------------------
# 2. Re-verify canonical dimensions and ordering
# ------------------------------------------------------------

expected_auxiliary_shapes = {
    15: (
        131_325,
        127,
    ),

    30: (
        131_322,
        127,
    ),

    60: (
        131_316,
        127,
    ),
}


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    assert auxiliary_table.shape == (
        expected_auxiliary_shapes[
            horizon_minutes
        ]
    )


    assert list(
        auxiliary_table.columns
    ) == DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ]


    assert auxiliary_table.index.equals(
        deep_base_modelling_tables[
            horizon_minutes
        ].index
    )


    assert (
        auxiliary_table.isna()
        .sum()
        .sum()
        == 0
    )


    assert np.isfinite(
        auxiliary_table.to_numpy()
    ).all()


# ------------------------------------------------------------
# 3. Confirm all columns remain float32
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    assert all(
        dtype == np.float32
        for dtype in (
            deep_auxiliary_raw[
                horizon_minutes
            ].dtypes
        )
    )


# ------------------------------------------------------------
# 4. Report
# ------------------------------------------------------------

print(
    "Defragmented auxiliary matrices"
)

print(
    "-------------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    print(
        f"{horizon_minutes:2d} min:",
        auxiliary_table.shape,
        "| dtype:",
        str(
            auxiliary_table.dtypes.iloc[0]
        ),
    )


print(
    "\nAuxiliary DataFrame defragmentation: PASSED"
)

Defragmented auxiliary matrices
-------------------------------
15 min: (131325, 127) | dtype: float32
30 min: (131322, 127) | dtype: float32
60 min: (131316, 127) | dtype: float32

Auxiliary DataFrame defragmentation: PASSED


In [15]:
# ============================================================
# Step 15: Freeze the neural-network preprocessing protocol
# ============================================================
#
# IMPORTANT:
#
# This cell DEFINES the scaling rules before any neural model
# is trained or any validation result is inspected.
#
# All learned scaling statistics will be fitted on TRAINING
# rows only for each forecast horizon.
#
#
# Sequence branch
# ---------------
#
# Shape:
#
#       (samples, 13 timesteps, 9 channels)
#
# Each of the 9 channels will receive its own training-derived
# mean and standard deviation.
#
# Statistics are computed by pooling:
#
#       training samples × 13 timesteps
#
# separately for each channel.
#
#
# Auxiliary branch
# ----------------
#
# Continuous/numeric variables:
#
#       standardized using train-only mean/std
#
# Binary variables:
#
#       kept unchanged as 0 / 1
#
# Binary variables include:
#
#       utc_is_weekend
#       solar_daylight
#       all 97 Panchang one-hot indicators
#
#
# Target
# ------
#
# P_Solar[kW] will also be standardized using train-only
# statistics for neural-network optimization.
#
# All predictions MUST be transformed back to kW before:
#
#       MAE
#       RMSE
#       R²
#       normalized metrics
#       paired F3-vs-F4 bootstrap analysis
#
#
# Final-model rule
# ----------------
#
# Even when the selected neural configuration is later refit
# using train + validation observations, the preprocessing
# statistics defined from the ORIGINAL TRAINING population
# remain frozen.
#
# This preserves the preprocessing protocol established before
# validation/test evaluation.
# ============================================================


# ------------------------------------------------------------
# 1. Binary deterministic auxiliary features that must remain
#    as literal 0/1 values.
# ------------------------------------------------------------

DEEP_BINARY_BASE_COLUMNS = [
    "utc_is_weekend",
    "solar_daylight",
]


DEEP_PANCHANG_ONEHOT_COLUMNS = list(
    DEEP_AUX_F4_ADDITIONS
)


DEEP_BINARY_AUX_COLUMNS = (
    DEEP_BINARY_BASE_COLUMNS
    + DEEP_PANCHANG_ONEHOT_COLUMNS
)


# ------------------------------------------------------------
# 2. Structural binary-column checks
# ------------------------------------------------------------

assert (
    len(
        DEEP_PANCHANG_ONEHOT_COLUMNS
    )
    == 97
)


assert (
    len(
        DEEP_BINARY_AUX_COLUMNS
    )
    == 99
)


assert (
    len(
        DEEP_BINARY_AUX_COLUMNS
    )
    == len(
        set(
            DEEP_BINARY_AUX_COLUMNS
        )
    )
)


# ------------------------------------------------------------
# 3. Every designated binary column must exist in full F4.
# ------------------------------------------------------------

for column_name in DEEP_BINARY_AUX_COLUMNS:

    assert (
        column_name
        in DEEP_AUX_FAMILY_COLUMNS[
            "F4"
        ]
    ), (
        f"Binary auxiliary column missing from F4: "
        f"{column_name}"
    )


# ------------------------------------------------------------
# 4. Continuous auxiliary columns are defined as:
#
#       all F4 auxiliary columns
#       minus designated binary columns
#
# This includes:
#
#       pv_lag_24h
#       raw Gregorian numeric fields
#       Gregorian cyclic variables
#       solar continuous variables
#       lunar / Sun-Moon continuous variables
# ------------------------------------------------------------

DEEP_CONTINUOUS_AUX_COLUMNS = [
    column_name
    for column_name
    in DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ]
    if column_name
    not in DEEP_BINARY_AUX_COLUMNS
]


assert (
    len(
        DEEP_CONTINUOUS_AUX_COLUMNS
    )
    == 28
)


# ------------------------------------------------------------
# 5. Verify exact partition of the 127-column F4 auxiliary
#    representation.
# ------------------------------------------------------------

assert (
    len(
        DEEP_CONTINUOUS_AUX_COLUMNS
    )
    + len(
        DEEP_BINARY_AUX_COLUMNS
    )
    == 127
)


assert set(
    DEEP_CONTINUOUS_AUX_COLUMNS
).isdisjoint(
    DEEP_BINARY_AUX_COLUMNS
)


assert set(
    DEEP_CONTINUOUS_AUX_COLUMNS
).union(
    DEEP_BINARY_AUX_COLUMNS
) == set(
    DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ]
)


# ------------------------------------------------------------
# 6. Validate that designated binary columns really contain
#    only 0 and 1 in every horizon.
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    for column_name in DEEP_BINARY_AUX_COLUMNS:

        observed_values = np.unique(
            auxiliary_table[
                column_name
            ].to_numpy()
        )


        assert np.all(
            np.isin(
                observed_values,
                [
                    0.0,
                    1.0,
                ],
            )
        ), (
            f"{column_name} is not binary at "
            f"{horizon_minutes} min: "
            f"{observed_values}"
        )


# ------------------------------------------------------------
# 7. Define per-family scaling behavior.
#
# Only continuous columns PRESENT in a given feature family
# are standardized.
#
# Binary columns PRESENT in that family remain unchanged.
# ------------------------------------------------------------

DEEP_AUX_CONTINUOUS_BY_FAMILY = {}

DEEP_AUX_BINARY_BY_FAMILY = {}


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    family_columns = (
        DEEP_AUX_FAMILY_COLUMNS[
            feature_family
        ]
    )


    DEEP_AUX_CONTINUOUS_BY_FAMILY[
        feature_family
    ] = [
        column_name
        for column_name
        in family_columns
        if column_name
        in DEEP_CONTINUOUS_AUX_COLUMNS
    ]


    DEEP_AUX_BINARY_BY_FAMILY[
        feature_family
    ] = [
        column_name
        for column_name
        in family_columns
        if column_name
        in DEEP_BINARY_AUX_COLUMNS
    ]


    assert (
        len(
            DEEP_AUX_CONTINUOUS_BY_FAMILY[
                feature_family
            ]
        )
        +
        len(
            DEEP_AUX_BINARY_BY_FAMILY[
                feature_family
            ]
        )
        == len(
            family_columns
        )
    )


# ------------------------------------------------------------
# 8. Freeze the mathematical scaling convention
#
# Population standard deviation:
#
#       ddof = 0
#
# For a value x:
#
#       z = (x - training_mean) / training_std
#
# Any zero-variance continuous variable would be treated as an
# error rather than silently assigned an arbitrary scale.
# ------------------------------------------------------------

DEEP_SCALING_DDOF = 0

DEEP_ALLOW_ZERO_VARIANCE_CONTINUOUS = False


# ------------------------------------------------------------
# 9. Freeze target-scaling convention
# ------------------------------------------------------------

DEEP_SCALE_TARGET = True

DEEP_TARGET_COLUMN = (
    "target_P_Solar_kW"
)


# ------------------------------------------------------------
# 10. Human-readable protocol summary
# ------------------------------------------------------------

print(
    "Frozen neural-network scaling protocol"
)

print(
    "--------------------------------------"
)


print(
    "Sequence channels:",
    len(
        DEEP_SEQUENCE_CHANNELS
    ),
)

print(
    "  scaling:"
    " train-only mean/std per channel,"
    " pooled across samples and timesteps"
)


print(
    "\nFull F4 auxiliary width:",
    len(
        DEEP_AUX_FAMILY_COLUMNS[
            "F4"
        ]
    ),
)

print(
    "Continuous auxiliary columns:",
    len(
        DEEP_CONTINUOUS_AUX_COLUMNS
    ),
)

print(
    "Binary auxiliary columns:",
    len(
        DEEP_BINARY_AUX_COLUMNS
    ),
)

print(
    "  Panchang one-hot:",
    len(
        DEEP_PANCHANG_ONEHOT_COLUMNS
    ),
)


print(
    "\nTarget scaling:",
    DEEP_SCALE_TARGET,
)

print(
    "Scaling ddof:",
    DEEP_SCALING_DDOF,
)

print(
    "Scaling statistics source:"
    " training population only"
)

print(
    "Validation/test fitting:"
    " prohibited"
)

print(
    "Final metrics:"
    " always computed after inverse transform to kW"
)


# ------------------------------------------------------------
# 11. Per-family scaling summary
# ------------------------------------------------------------

print(
    "\nPer-family auxiliary scaling"
)

print(
    "----------------------------"
)


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    n_total = len(
        DEEP_AUX_FAMILY_COLUMNS[
            feature_family
        ]
    )

    n_continuous = len(
        DEEP_AUX_CONTINUOUS_BY_FAMILY[
            feature_family
        ]
    )

    n_binary = len(
        DEEP_AUX_BINARY_BY_FAMILY[
            feature_family
        ]
    )


    print(
        f"{feature_family}: "
        f"total={n_total:3d}, "
        f"standardized={n_continuous:3d}, "
        f"unchanged_binary={n_binary:3d}"
    )


# ------------------------------------------------------------
# 12. Display the non-Panchang binary columns explicitly
# ------------------------------------------------------------

print(
    "\nNon-Panchang binary columns:"
)

for column_name in DEEP_BINARY_BASE_COLUMNS:

    print(
        " ",
        column_name,
    )


# ------------------------------------------------------------
# 13. Final checkpoint
# ------------------------------------------------------------

print(
    "\nNeural-network scaling protocol frozen: PASSED"
)

Frozen neural-network scaling protocol
--------------------------------------
Sequence channels: 9
  scaling: train-only mean/std per channel, pooled across samples and timesteps

Full F4 auxiliary width: 127
Continuous auxiliary columns: 28
Binary auxiliary columns: 99
  Panchang one-hot: 97

Target scaling: True
Scaling ddof: 0
Scaling statistics source: training population only
Validation/test fitting: prohibited
Final metrics: always computed after inverse transform to kW

Per-family auxiliary scaling
----------------------------
F0: total=  1, standardized=  1, unchanged_binary=  0
F1: total= 16, standardized= 15, unchanged_binary=  1
F2: total= 21, standardized= 19, unchanged_binary=  2
F3: total= 30, standardized= 28, unchanged_binary=  2
F4: total=127, standardized= 28, unchanged_binary= 99

Non-Panchang binary columns:
  utc_is_weekend
  solar_daylight

Neural-network scaling protocol frozen: PASSED


In [17]:
# ============================================================
# Step 16: Fit train-only neural-network scaling statistics
# ============================================================
#
# For each forecast horizon, this cell fits:
#
#   1. sequence-channel mean/std;
#   2. continuous auxiliary-feature mean/std;
#   3. target P_Solar mean/std.
#
# CRITICAL RULE:
#
#       ONLY TRAINING rows contribute to these statistics.
#
# Validation and test observations are never used for fitting
# preprocessing statistics.
#
# No data transformation and no neural-network training occurs
# in this cell.
# ============================================================


# ------------------------------------------------------------
# 1. Container for frozen horizon-specific statistics
# ------------------------------------------------------------

deep_scaling_statistics = {}


# ------------------------------------------------------------
# 2. Fit statistics independently for each forecast horizon
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    # --------------------------------------------------------
    # Horizon-specific canonical structures
    # --------------------------------------------------------

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )

    auxiliary_table = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )

    sequence_positions = (
        deep_sequence_positions[
            horizon_minutes
        ]
    )


    # --------------------------------------------------------
    # Split membership
    #
    # IMPORTANT:
    # These splits were already defined using FORECAST-VALID
    # time, not issue time.
    # --------------------------------------------------------

    train_mask = (
        base_table[
            "split"
        ]
        .eq(
            "train"
        )
        .to_numpy()
    )

    validation_mask = (
        base_table[
            "split"
        ]
        .eq(
            "validation"
        )
        .to_numpy()
    )

    test_mask = (
        base_table[
            "split"
        ]
        .eq(
            "test"
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Every sample must belong to exactly one split
    # --------------------------------------------------------

    split_membership_count = (
        train_mask.astype(
            np.int8
        )
        +
        validation_mask.astype(
            np.int8
        )
        +
        test_mask.astype(
            np.int8
        )
    )

    assert np.all(
        split_membership_count
        == 1
    )


    n_train = int(
        train_mask.sum()
    )

    n_validation = int(
        validation_mask.sum()
    )

    n_test = int(
        test_mask.sum()
    )


    # ========================================================
    # SEQUENCE BRANCH
    # ========================================================


    # --------------------------------------------------------
    # 3. Select only training sequence samples
    # --------------------------------------------------------

    train_sequence_positions = (
        sequence_positions[
            train_mask
        ]
    )

    train_sequence = (
        deep_history_tensor[
            train_sequence_positions
        ]
    )


    assert train_sequence.shape == (
        n_train,
        DEEP_SEQUENCE_LENGTH,
        len(
            DEEP_SEQUENCE_CHANNELS
        ),
    )


    # --------------------------------------------------------
    # 4. Convert temporarily to float64 for stable statistic
    #    computation.
    #
    # The final TensorFlow tensors will remain float32.
    # --------------------------------------------------------

    train_sequence_float64 = (
        train_sequence.astype(
            np.float64,
            copy=False,
        )
    )


    # --------------------------------------------------------
    # 5. Channel-wise training mean/std
    #
    # Pool over:
    #
    #       sample axis
    #       timestep axis
    #
    # Preserve:
    #
    #       channel axis
    # --------------------------------------------------------

    sequence_mean = (
        train_sequence_float64
        .mean(
            axis=(
                0,
                1,
            )
        )
    )

    sequence_std = (
        train_sequence_float64
        .std(
            axis=(
                0,
                1,
            ),
            ddof=DEEP_SCALING_DDOF,
        )
    )


    assert sequence_mean.shape == (
        len(
            DEEP_SEQUENCE_CHANNELS
        ),
    )

    assert sequence_std.shape == (
        len(
            DEEP_SEQUENCE_CHANNELS
        ),
    )

    assert np.isfinite(
        sequence_mean
    ).all()

    assert np.isfinite(
        sequence_std
    ).all()

    assert np.all(
        sequence_std
        > 0.0
    ), (
        f"Zero-variance sequence channel found at "
        f"{horizon_minutes}-minute horizon."
    )


    # ========================================================
    # AUXILIARY BRANCH
    # ========================================================


    # --------------------------------------------------------
    # 6. Select ONLY training rows for the 28 continuous
    #    auxiliary variables.
    #
    # Binary features are intentionally excluded because they
    # remain unchanged as 0/1.
    # --------------------------------------------------------

    train_auxiliary_continuous = (
        auxiliary_table.loc[
            train_mask,
            DEEP_CONTINUOUS_AUX_COLUMNS,
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    assert train_auxiliary_continuous.shape == (
        n_train,
        len(
            DEEP_CONTINUOUS_AUX_COLUMNS
        ),
    )


    # --------------------------------------------------------
    # 7. Continuous auxiliary training mean/std
    # --------------------------------------------------------

    auxiliary_mean = (
        train_auxiliary_continuous
        .mean(
            axis=0
        )
    )

    auxiliary_std = (
        train_auxiliary_continuous
        .std(
            axis=0,
            ddof=DEEP_SCALING_DDOF,
        )
    )


    assert auxiliary_mean.shape == (
        len(
            DEEP_CONTINUOUS_AUX_COLUMNS
        ),
    )

    assert auxiliary_std.shape == (
        len(
            DEEP_CONTINUOUS_AUX_COLUMNS
        ),
    )

    assert np.isfinite(
        auxiliary_mean
    ).all()

    assert np.isfinite(
        auxiliary_std
    ).all()


    if not DEEP_ALLOW_ZERO_VARIANCE_CONTINUOUS:

        assert np.all(
            auxiliary_std
            > 0.0
        ), (
            f"Zero-variance continuous auxiliary feature "
            f"found at {horizon_minutes}-minute horizon."
        )


    # ========================================================
    # TARGET
    # ========================================================


    # --------------------------------------------------------
    # 8. Select ONLY training targets
    # --------------------------------------------------------

    train_target = (
        base_table.loc[
            train_mask,
            DEEP_TARGET_COLUMN,
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    assert train_target.shape == (
        n_train,
    )

    assert np.isfinite(
        train_target
    ).all()


    # --------------------------------------------------------
    # 9. Training-only target mean/std
    # --------------------------------------------------------

    target_mean = float(
        train_target.mean()
    )

    target_std = float(
        train_target.std(
            ddof=DEEP_SCALING_DDOF
        )
    )


    assert np.isfinite(
        target_mean
    )

    assert np.isfinite(
        target_std
    )

    assert (
        target_std
        > 0.0
    )


    # ========================================================
    # TEMPORAL LEAKAGE AUDIT
    # ========================================================


    # --------------------------------------------------------
    # 10. Extract valid-time ranges for each population
    # --------------------------------------------------------

    train_valid_times = pd.DatetimeIndex(
        base_table.loc[
            train_mask,
            "valid_time",
        ]
    )

    validation_valid_times = pd.DatetimeIndex(
        base_table.loc[
            validation_mask,
            "valid_time",
        ]
    )

    test_valid_times = pd.DatetimeIndex(
        base_table.loc[
            test_mask,
            "valid_time",
        ]
    )


    # --------------------------------------------------------
    # 11. Strong chronological-boundary checks
    # --------------------------------------------------------

    assert (
        train_valid_times.max()
        < TRAIN_VALIDATION_CUTOFF
    )

    assert (
        validation_valid_times.min()
        >= TRAIN_VALIDATION_CUTOFF
    )

    assert (
        validation_valid_times.max()
        < VALIDATION_TEST_CUTOFF
    )

    assert (
        test_valid_times.min()
        >= VALIDATION_TEST_CUTOFF
    )

    assert (
        test_valid_times.max()
        < TEST_END_EXCLUSIVE
    )


    # --------------------------------------------------------
    # 12. Store frozen copies
    # --------------------------------------------------------

    deep_scaling_statistics[
        horizon_minutes
    ] = {

        "n_train":
            n_train,

        "n_validation":
            n_validation,

        "n_test":
            n_test,

        "sequence_channels":
            tuple(
                DEEP_SEQUENCE_CHANNELS
            ),

        "sequence_mean":
            sequence_mean.copy(),

        "sequence_std":
            sequence_std.copy(),

        "continuous_aux_columns":
            tuple(
                DEEP_CONTINUOUS_AUX_COLUMNS
            ),

        "auxiliary_mean":
            auxiliary_mean.copy(),

        "auxiliary_std":
            auxiliary_std.copy(),

        "target_mean":
            target_mean,

        "target_std":
            target_std,

        "train_valid_time_start":
            train_valid_times.min(),

        "train_valid_time_end":
            train_valid_times.max(),

        "validation_valid_time_start":
            validation_valid_times.min(),

        "validation_valid_time_end":
            validation_valid_times.max(),

        "test_valid_time_start":
            test_valid_times.min(),

        "test_valid_time_end":
            test_valid_times.max(),

        "ddof":
            DEEP_SCALING_DDOF,
    }


# ============================================================
# GLOBAL VERIFICATION
# ============================================================


# ------------------------------------------------------------
# 13. Verify exact split counts against Notebook 10
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )

    expected_counts = (
        expected_deep_population_counts[
            horizon_minutes
        ]
    )


    assert (
        stats[
            "n_train"
        ]
        == expected_counts[
            "train"
        ]
    )

    assert (
        stats[
            "n_validation"
        ]
        == expected_counts[
            "validation"
        ]
    )

    assert (
        stats[
            "n_test"
        ]
        == expected_counts[
            "test"
        ]
    )


# ============================================================
# READABLE AUDIT TABLES
# ============================================================


# ------------------------------------------------------------
# 14. Sequence-scaling audit table
# ------------------------------------------------------------

sequence_scaling_columns = {}


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    sequence_scaling_columns[
        f"{horizon_minutes}min_mean"
    ] = (
        deep_scaling_statistics[
            horizon_minutes
        ][
            "sequence_mean"
        ]
    )

    sequence_scaling_columns[
        f"{horizon_minutes}min_std"
    ] = (
        deep_scaling_statistics[
            horizon_minutes
        ][
            "sequence_std"
        ]
    )


sequence_scaling_audit = pd.DataFrame(
    sequence_scaling_columns,
    index=DEEP_SEQUENCE_CHANNELS,
)


sequence_scaling_audit.index.name = (
    "sequence_channel"
)


# ------------------------------------------------------------
# 15. Target / temporal audit table
# ------------------------------------------------------------

target_scaling_rows = []


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )


    target_scaling_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "n_train":
                stats[
                    "n_train"
                ],

            "n_validation":
                stats[
                    "n_validation"
                ],

            "n_test":
                stats[
                    "n_test"
                ],

            "target_mean_kW":
                stats[
                    "target_mean"
                ],

            "target_std_kW":
                stats[
                    "target_std"
                ],

            "train_valid_start":
                stats[
                    "train_valid_time_start"
                ],

            "train_valid_end":
                stats[
                    "train_valid_time_end"
                ],

            "validation_valid_start":
                stats[
                    "validation_valid_time_start"
                ],

            "validation_valid_end":
                stats[
                    "validation_valid_time_end"
                ],

            "test_valid_start":
                stats[
                    "test_valid_time_start"
                ],

            "test_valid_end":
                stats[
                    "test_valid_time_end"
                ],
        }
    )


target_scaling_audit = pd.DataFrame(
    target_scaling_rows
)


# ------------------------------------------------------------
# 16. Display target / split audit
# ------------------------------------------------------------

print(
    "Train-only scaling statistics fitted"
)

print(
    "------------------------------------"
)


display(
    target_scaling_audit
)


# ------------------------------------------------------------
# 17. Display sequence-channel statistics
# ------------------------------------------------------------

print(
    "\nSequence-channel scaling statistics"
)

print(
    "-----------------------------------"
)


display(
    sequence_scaling_audit
)


# ------------------------------------------------------------
# 18. Continuous auxiliary variability audit
# ------------------------------------------------------------

print(
    "\nContinuous auxiliary standard-deviation ranges"
)

print(
    "----------------------------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    auxiliary_std = (
        deep_scaling_statistics[
            horizon_minutes
        ][
            "auxiliary_std"
        ]
    )


    min_position = int(
        np.argmin(
            auxiliary_std
        )
    )

    max_position = int(
        np.argmax(
            auxiliary_std
        )
    )


    print(
        f"{horizon_minutes:2d} min:"
    )

    print(
        "  smallest std:",
        (
            DEEP_CONTINUOUS_AUX_COLUMNS[
                min_position
            ],
            float(
                auxiliary_std[
                    min_position
                ]
            ),
        ),
    )

    print(
        "  largest std:",
        (
            DEEP_CONTINUOUS_AUX_COLUMNS[
                max_position
            ],
            float(
                auxiliary_std[
                    max_position
                ]
            ),
        ),
    )


# ------------------------------------------------------------
# 19. Explicit leakage-prevention report
# ------------------------------------------------------------

print(
    "\nScaler fitting populations"
)

print(
    "--------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )


    print(
        f"\n{horizon_minutes}-minute horizon"
    )

    print(
        "  Train valid time:",
        stats[
            "train_valid_time_start"
        ],
        "->",
        stats[
            "train_valid_time_end"
        ],
    )

    print(
        "  Validation starts:",
        stats[
            "validation_valid_time_start"
        ],
    )

    print(
        "  Test starts:",
        stats[
            "test_valid_time_start"
        ],
    )


# ------------------------------------------------------------
# 20. Final checkpoint
# ------------------------------------------------------------

print(
    "\nTrain-only scaling-statistic fitting: PASSED"
)

Train-only scaling statistics fitted
------------------------------------


,horizon_minutes,n_train,n_validation,n_test,target_mean_kW,target_std_kW,train_valid_start,train_valid_end,validation_valid_start,validation_valid_end,test_valid_start,test_valid_end
0,15,87261,17568,26496,0.838792,1.587854,2018-06-02 00:15:00+00:00,2019-03-31 23:55:00+00:00,2019-04-01 00:00:00+00:00,2019-05-31 23:55:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00
1,30,87258,17568,26496,0.838821,1.587874,2018-06-02 00:30:00+00:00,2019-03-31 23:55:00+00:00,2019-04-01 00:00:00+00:00,2019-05-31 23:55:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00
2,60,87252,17568,26496,0.838879,1.587913,2018-06-02 01:00:00+00:00,2019-03-31 23:55:00+00:00,2019-04-01 00:00:00+00:00,2019-05-31 23:55:00+00:00,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00



Sequence-channel scaling statistics
-----------------------------------


,15min_mean,15min_std,30min_mean,30min_std,60min_mean,60min_std
sequence_channel,,,,,,
P_Solar[kW],0.838792,1.587854,0.838821,1.587874,0.838879,1.587913
GHI[kW1m2],0.102499,0.182448,0.102502,0.182451,0.102510,0.182455
POA Irr[kW1m2],0.119581,0.228349,0.119586,0.228352,0.119594,0.228358
TEMPERATURE[degC],10.488167,7.479538,10.488419,7.479542,10.488891,7.479582
Humidity_clean,0.794004,0.146618,0.794010,0.146616,0.794023,0.146613
Pressure_clean[mbar],1008.837890,10.188391,1008.837319,10.188100,1008.836182,10.187528
WIND_SPEED[m1s],3.773081,2.179392,3.773157,2.179386,3.773284,2.179404
wind_direction_sin,-0.293086,0.707084,-0.293076,0.707087,-0.293050,0.707097
wind_direction_cos,-0.312429,0.562602,-0.312452,0.562590,-0.312508,0.562560



Continuous auxiliary standard-deviation ranges
----------------------------------------------
15 min:
  smallest std: ('moon_illumination_fraction', 0.348298521355572)
  largest std: ('earth_moon_distance_km', 15701.977032695497)
30 min:
  smallest std: ('moon_illumination_fraction', 0.34829686915417174)
  largest std: ('earth_moon_distance_km', 15701.804163288529)
60 min:
  smallest std: ('moon_illumination_fraction', 0.34829363296764254)
  largest std: ('earth_moon_distance_km', 15701.457440019216)

Scaler fitting populations
--------------------------

15-minute horizon
  Train valid time: 2018-06-02 00:15:00+00:00 -> 2019-03-31 23:55:00+00:00
  Validation starts: 2019-04-01 00:00:00+00:00
  Test starts: 2019-06-01 00:00:00+00:00

30-minute horizon
  Train valid time: 2018-06-02 00:30:00+00:00 -> 2019-03-31 23:55:00+00:00
  Validation starts: 2019-04-01 00:00:00+00:00
  Test starts: 2019-06-01 00:00:00+00:00

60-minute horizon
  Train valid time: 2018-06-02 01:00:00+00:00 -> 2019-0

In [20]:
# ============================================================
# Step 17: Apply frozen train-only scaling statistics
#          using numerically stable float64 arithmetic
# ============================================================
#
# The previous diagnostic established that converting scaler
# parameters to float32 BEFORE subtraction introduced a small
# residual for large-offset variables such as:
#
#       utc_year ~ 2018
#
# Therefore the frozen protocol is implemented as:
#
#   1. raw values -> float64 temporarily;
#   2. subtract ORIGINAL float64 training mean;
#   3. divide by ORIGINAL float64 training std;
#   4. cast the FINISHED transformed tensor to float32.
#
# This preserves:
#
#   - train-only preprocessing;
#   - high numerical accuracy;
#   - TensorFlow-ready float32 tensors.
#
# Binary columns are never standardized and remain exactly 0/1.
#
# No neural-network training occurs in this cell.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical full-F4 auxiliary column ordering
# ------------------------------------------------------------

full_auxiliary_columns = (
    DEEP_AUX_FAMILY_COLUMNS[
        "F4"
    ]
)


assert (
    len(
        full_auxiliary_columns
    )
    == 127
)


# ------------------------------------------------------------
# 2. Map auxiliary column names -> integer positions
# ------------------------------------------------------------

auxiliary_column_to_position = {
    column_name: position
    for position, column_name
    in enumerate(
        full_auxiliary_columns
    )
}


# ------------------------------------------------------------
# 3. Continuous and binary positions within the complete
#    127-column F4 auxiliary representation
# ------------------------------------------------------------

continuous_auxiliary_positions = np.array(
    [
        auxiliary_column_to_position[
            column_name
        ]
        for column_name
        in DEEP_CONTINUOUS_AUX_COLUMNS
    ],
    dtype=np.int64,
)


binary_auxiliary_positions = np.array(
    [
        auxiliary_column_to_position[
            column_name
        ]
        for column_name
        in DEEP_BINARY_AUX_COLUMNS
    ],
    dtype=np.int64,
)


assert len(
    continuous_auxiliary_positions
) == 28


assert len(
    binary_auxiliary_positions
) == 99


assert set(
    continuous_auxiliary_positions.tolist()
).isdisjoint(
    binary_auxiliary_positions.tolist()
)


assert (
    len(
        continuous_auxiliary_positions
    )
    +
    len(
        binary_auxiliary_positions
    )
    == 127
)


# ------------------------------------------------------------
# 4. Precompute nested F0-F4 auxiliary column positions
#
# Later we can obtain:
#
#       F0 -> first relevant subset
#       F1 -> nested larger subset
#       ...
#       F4 -> complete 127-column matrix
#
# without rebuilding or rescaling the data.
# ------------------------------------------------------------

DEEP_AUX_FAMILY_POSITIONS = {}


for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    DEEP_AUX_FAMILY_POSITIONS[
        feature_family
    ] = np.array(
        [
            auxiliary_column_to_position[
                column_name
            ]
            for column_name
            in DEEP_AUX_FAMILY_COLUMNS[
                feature_family
            ]
        ],
        dtype=np.int64,
    )


assert {
    family: len(
        positions
    )
    for family, positions
    in DEEP_AUX_FAMILY_POSITIONS.items()
} == {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


# ------------------------------------------------------------
# 5. Start from a clean transformed-data container
#
# Cell 18 previously stopped during execution, so we explicitly
# rebuild this dictionary from scratch.
# ------------------------------------------------------------

deep_scaled_data = {}


# ------------------------------------------------------------
# 6. Transform each horizon independently
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )


    sequence_positions = (
        deep_sequence_positions[
            horizon_minutes
        ]
    )


    # ========================================================
    # SPLIT MASKS
    # ========================================================

    train_mask = (
        base_table[
            "split"
        ]
        .eq(
            "train"
        )
        .to_numpy()
    )


    validation_mask = (
        base_table[
            "split"
        ]
        .eq(
            "validation"
        )
        .to_numpy()
    )


    test_mask = (
        base_table[
            "split"
        ]
        .eq(
            "test"
        )
        .to_numpy()
    )


    assert np.all(
        (
            train_mask.astype(
                np.int8
            )
            +
            validation_mask.astype(
                np.int8
            )
            +
            test_mask.astype(
                np.int8
            )
        )
        == 1
    )


    # ========================================================
    # SEQUENCE BRANCH
    # ========================================================


    # --------------------------------------------------------
    # 7. Obtain raw horizon-specific sequence population
    # --------------------------------------------------------

    raw_sequence = (
        deep_history_tensor[
            sequence_positions
        ]
    )


    assert raw_sequence.shape == (
        len(
            base_table
        ),
        DEEP_SEQUENCE_LENGTH,
        len(
            DEEP_SEQUENCE_CHANNELS
        ),
    )


    # --------------------------------------------------------
    # 8. Scale in FLOAT64 using the ORIGINAL train statistics
    #
    # Broadcasting:
    #
    #   raw sequence:
    #       (samples, 13, 9)
    #
    #   mean/std:
    #       (9,)
    # --------------------------------------------------------

    scaled_sequence = (
        (
            raw_sequence.astype(
                np.float64
            )
            - stats[
                "sequence_mean"
            ]
        )
        /
        stats[
            "sequence_std"
        ]
    ).astype(
        np.float32
    )


    # ========================================================
    # AUXILIARY BRANCH
    # ========================================================


    # --------------------------------------------------------
    # 9. Complete raw F4 auxiliary matrix
    # --------------------------------------------------------

    raw_auxiliary = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
        .to_numpy(
            dtype=np.float32,
            copy=True,
        )
    )


    assert raw_auxiliary.shape == (
        len(
            base_table
        ),
        127,
    )


    # --------------------------------------------------------
    # 10. Preserve raw binary columns for exact equality audit
    # --------------------------------------------------------

    raw_binary_auxiliary = (
        raw_auxiliary[
            :,
            binary_auxiliary_positions,
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 11. Start output as a float32 copy
    #
    # Binary columns therefore already have their correct
    # unchanged 0/1 values.
    # --------------------------------------------------------

    scaled_auxiliary = (
        raw_auxiliary.copy()
    )


    # --------------------------------------------------------
    # 12. Standardize ONLY the 28 continuous columns.
    #
    # Arithmetic is performed in float64.
    # Finished standardized values are cast back to float32.
    # --------------------------------------------------------

    raw_continuous_float64 = (
        raw_auxiliary[
            :,
            continuous_auxiliary_positions,
        ]
        .astype(
            np.float64
        )
    )


    scaled_continuous_float32 = (
        (
            raw_continuous_float64
            - stats[
                "auxiliary_mean"
            ]
        )
        /
        stats[
            "auxiliary_std"
        ]
    ).astype(
        np.float32
    )


    scaled_auxiliary[
        :,
        continuous_auxiliary_positions,
    ] = (
        scaled_continuous_float32
    )


    # --------------------------------------------------------
    # 13. Binary columns must remain EXACTLY unchanged
    # --------------------------------------------------------

    assert np.array_equal(
        scaled_auxiliary[
            :,
            binary_auxiliary_positions,
        ],
        raw_binary_auxiliary,
    )


    # ========================================================
    # TARGET
    # ========================================================


    # --------------------------------------------------------
    # 14. Raw target in physical units
    # --------------------------------------------------------

    raw_target = (
        base_table[
            DEEP_TARGET_COLUMN
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    # --------------------------------------------------------
    # 15. Scale target in float64, then cast to float32
    # --------------------------------------------------------

    scaled_target = (
        (
            raw_target.astype(
                np.float64
            )
            - stats[
                "target_mean"
            ]
        )
        /
        stats[
            "target_std"
        ]
    ).astype(
        np.float32
    )


    # ========================================================
    # GENERAL INTEGRITY CHECKS
    # ========================================================


    assert scaled_sequence.dtype == np.float32

    assert scaled_auxiliary.dtype == np.float32

    assert scaled_target.dtype == np.float32


    assert np.isfinite(
        scaled_sequence
    ).all()

    assert np.isfinite(
        scaled_auxiliary
    ).all()

    assert np.isfinite(
        scaled_target
    ).all()


    # ========================================================
    # TRAINING-POPULATION SCALING AUDIT
    # ========================================================


    # --------------------------------------------------------
    # 16. Sequence training mean/std after transformation
    # --------------------------------------------------------

    training_scaled_sequence = (
        scaled_sequence[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    training_sequence_mean = (
        training_scaled_sequence
        .mean(
            axis=(
                0,
                1,
            )
        )
    )


    training_sequence_std = (
        training_scaled_sequence
        .std(
            axis=(
                0,
                1,
            ),
            ddof=0,
        )
    )


    assert np.allclose(
        training_sequence_mean,
        0.0,
        atol=1e-6,
        rtol=0.0,
    )


    assert np.allclose(
        training_sequence_std,
        1.0,
        atol=1e-6,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # 17. Continuous auxiliary training mean/std
    # --------------------------------------------------------

    training_scaled_auxiliary_continuous = (
        scaled_auxiliary[
            train_mask
        ][
            :,
            continuous_auxiliary_positions,
        ]
        .astype(
            np.float64
        )
    )


    training_auxiliary_mean = (
        training_scaled_auxiliary_continuous
        .mean(
            axis=0
        )
    )


    training_auxiliary_std = (
        training_scaled_auxiliary_continuous
        .std(
            axis=0,
            ddof=0,
        )
    )


    assert np.allclose(
        training_auxiliary_mean,
        0.0,
        atol=1e-6,
        rtol=0.0,
    )


    assert np.allclose(
        training_auxiliary_std,
        1.0,
        atol=1e-6,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # 18. Target training mean/std
    # --------------------------------------------------------

    training_scaled_target = (
        scaled_target[
            train_mask
        ]
        .astype(
            np.float64
        )
    )


    training_target_mean = float(
        training_scaled_target.mean()
    )


    training_target_std = float(
        training_scaled_target.std(
            ddof=0
        )
    )


    assert np.isclose(
        training_target_mean,
        0.0,
        atol=1e-6,
        rtol=0.0,
    )


    assert np.isclose(
        training_target_std,
        1.0,
        atol=1e-6,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # 19. Every binary auxiliary value must remain {0, 1}
    # --------------------------------------------------------

    binary_unique_values = np.unique(
        scaled_auxiliary[
            :,
            binary_auxiliary_positions,
        ]
    )


    assert np.array_equal(
        binary_unique_values,
        np.array(
            [
                0.0,
                1.0,
            ],
            dtype=np.float32,
        ),
    )


    # ========================================================
    # STORE FINAL TRANSFORMED POPULATION
    # ========================================================

    deep_scaled_data[
        horizon_minutes
    ] = {

        "sequence":
            np.ascontiguousarray(
                scaled_sequence,
                dtype=np.float32,
            ),

        "auxiliary_f4":
            np.ascontiguousarray(
                scaled_auxiliary,
                dtype=np.float32,
            ),

        "target_scaled":
            np.ascontiguousarray(
                scaled_target,
                dtype=np.float32,
            ),

        "target_kw":
            np.ascontiguousarray(
                raw_target,
                dtype=np.float32,
            ),

        "train_mask":
            train_mask.copy(),

        "validation_mask":
            validation_mask.copy(),

        "test_mask":
            test_mask.copy(),

        "training_sequence_mean_after_scaling":
            training_sequence_mean.copy(),

        "training_sequence_std_after_scaling":
            training_sequence_std.copy(),

        "training_auxiliary_mean_after_scaling":
            training_auxiliary_mean.copy(),

        "training_auxiliary_std_after_scaling":
            training_auxiliary_std.copy(),

        "training_target_mean_after_scaling":
            training_target_mean,

        "training_target_std_after_scaling":
            training_target_std,
    }


# ============================================================
# INVERSE-TRANSFORM AUDIT
# ============================================================


# ------------------------------------------------------------
# 20. Verify that scaled targets can be transformed back to
#     the original physical kW values.
# ------------------------------------------------------------

for horizon_minutes in DEEP_FORECAST_HORIZONS:

    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )


    transformed = (
        deep_scaled_data[
            horizon_minutes
        ]
    )


    reconstructed_target_kw = (
        (
            transformed[
                "target_scaled"
            ].astype(
                np.float64
            )
            * stats[
                "target_std"
            ]
        )
        + stats[
            "target_mean"
        ]
    )


    assert np.allclose(
        reconstructed_target_kw,
        transformed[
            "target_kw"
        ].astype(
            np.float64
        ),
        atol=1e-6,
        rtol=1e-6,
    )


# ============================================================
# REPORT
# ============================================================


# ------------------------------------------------------------
# 21. Population shapes and memory footprint
# ------------------------------------------------------------

print(
    "Scaled deep-learning populations"
)

print(
    "--------------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    transformed = (
        deep_scaled_data[
            horizon_minutes
        ]
    )


    sequence_mb = (
        transformed[
            "sequence"
        ].nbytes
        / 1024**2
    )


    auxiliary_mb = (
        transformed[
            "auxiliary_f4"
        ].nbytes
        / 1024**2
    )


    target_mb = (
        transformed[
            "target_scaled"
        ].nbytes
        / 1024**2
    )


    print(
        f"\n{horizon_minutes}-minute horizon"
    )

    print(
        "  Sequence shape:",
        transformed[
            "sequence"
        ].shape,
    )

    print(
        "  Auxiliary F4 shape:",
        transformed[
            "auxiliary_f4"
        ].shape,
    )

    print(
        "  Target shape:",
        transformed[
            "target_scaled"
        ].shape,
    )

    print(
        "  Sequence memory:",
        f"{sequence_mb:.2f} MB",
    )

    print(
        "  Auxiliary memory:",
        f"{auxiliary_mb:.2f} MB",
    )

    print(
        "  Target memory:",
        f"{target_mb:.2f} MB",
    )


# ------------------------------------------------------------
# 22. Numerical scaling audit
# ------------------------------------------------------------

print(
    "\nTraining-population scaling audit"
)

print(
    "---------------------------------"
)


for horizon_minutes in DEEP_FORECAST_HORIZONS:

    transformed = (
        deep_scaled_data[
            horizon_minutes
        ]
    )


    print(
        f"\n{horizon_minutes}-minute horizon"
    )


    print(
        "  max |sequence train mean|:",
        float(
            np.max(
                np.abs(
                    transformed[
                        "training_sequence_mean_after_scaling"
                    ]
                )
            )
        ),
    )


    print(
        "  max |sequence train std - 1|:",
        float(
            np.max(
                np.abs(
                    transformed[
                        "training_sequence_std_after_scaling"
                    ]
                    - 1.0
                )
            )
        ),
    )


    print(
        "  max |continuous aux train mean|:",
        float(
            np.max(
                np.abs(
                    transformed[
                        "training_auxiliary_mean_after_scaling"
                    ]
                )
            )
        ),
    )


    print(
        "  max |continuous aux train std - 1|:",
        float(
            np.max(
                np.abs(
                    transformed[
                        "training_auxiliary_std_after_scaling"
                    ]
                    - 1.0
                )
            )
        ),
    )


    print(
        "  target train mean:",
        transformed[
            "training_target_mean_after_scaling"
        ],
    )


    print(
        "  target train std:",
        transformed[
            "training_target_std_after_scaling"
        ],
    )


    print(
        "  binary values:",
        np.unique(
            transformed[
                "auxiliary_f4"
            ][
                :,
                binary_auxiliary_positions,
            ]
        ).tolist(),
    )


# ------------------------------------------------------------
# 23. Final checkpoint
# ------------------------------------------------------------

print(
    "\nDeep-learning scaling transformation: PASSED"
)

Scaled deep-learning populations
--------------------------------

15-minute horizon
  Sequence shape: (131325, 13, 9)
  Auxiliary F4 shape: (131325, 127)
  Target shape: (131325,)
  Sequence memory: 58.61 MB
  Auxiliary memory: 63.62 MB
  Target memory: 0.50 MB

30-minute horizon
  Sequence shape: (131322, 13, 9)
  Auxiliary F4 shape: (131322, 127)
  Target shape: (131322,)
  Sequence memory: 58.61 MB
  Auxiliary memory: 63.62 MB
  Target memory: 0.50 MB

60-minute horizon
  Sequence shape: (131316, 13, 9)
  Auxiliary F4 shape: (131316, 127)
  Target shape: (131316,)
  Sequence memory: 58.61 MB
  Auxiliary memory: 63.62 MB
  Target memory: 0.50 MB

Training-population scaling audit
---------------------------------

15-minute horizon
  max |sequence train mean|: 1.2589107141478587e-08
  max |sequence train std - 1|: 8.210283342080515e-09
  max |continuous aux train mean|: 1.2098388378103834e-08
  max |continuous aux train std - 1|: 3.8701798432683177e-08
  target train mean: 6.0103915

In [21]:
# ============================================================
# Step 18: Freeze neural-network model-selection protocol
# ============================================================
#
# This cell freezes the experimental rules BEFORE any neural
# model is trained.
#
# Models:
#
#       1. LSTM
#       2. GRU
#       3. BiLSTM
#       4. TCN
#
#
# ============================================================
# CORE MODEL-SELECTION PRINCIPLE
# ============================================================
#
# Architecture selection is performed ONLY on:
#
#       primary horizon = 60 minutes
#       feature family  = F0
#
# Why?
#
# The main scientific question concerns incremental feature
# information, especially:
#
#       F3 -> F4
#
# If a different architecture were separately optimized for
# every feature family, then differences between F3 and F4
# could partly reflect hyperparameter-selection differences.
#
# Therefore:
#
#   - each MODEL CLASS gets one selected architecture;
#   - architecture is selected using 60-min F0 validation only;
#   - the selected architecture is then frozen;
#   - that same architecture is used for ALL horizons and
#     ALL F0-F4 feature families.
#
#
# ============================================================
# EARLY-STOPPING / FINAL-REFIT PRINCIPLE
# ============================================================
#
# After architecture selection:
#
# For every horizon × feature-family combination:
#
#   Stage A
#   -------
#   train on TRAIN only;
#   monitor VALIDATION loss;
#   determine the best epoch.
#
#   Stage B
#   -------
#   reinitialize the network from scratch;
#   train on TRAIN + VALIDATION;
#   use exactly the previously selected number of epochs;
#   no early stopping and no test feedback.
#
#   Stage C
#   -------
#   evaluate the held-out TEST population once.
#
#
# Scaling remains the ORIGINAL train-only scaling already
# frozen earlier. It is NOT refitted using validation data.
#
#
# ============================================================
# REPRODUCIBILITY
# ============================================================
#
# Each fresh fit will:
#
#   - clear the Keras session;
#   - reset Python / NumPy / TensorFlow seed to 42;
#   - use deterministic TensorFlow operations where supported.
#
# Training samples may be shuffled because the networks are
# stateless and each sample already contains its complete
# leakage-safe historical sequence.
#
# The order INSIDE each 13-step sequence is never shuffled.
# ============================================================


# ------------------------------------------------------------
# 1. Fixed model execution order
# ------------------------------------------------------------

DEEP_MODEL_ORDER = (
    "LSTM",
    "GRU",
    "BiLSTM",
    "TCN",
)


# ------------------------------------------------------------
# 2. Architecture-selection population
#
# 60 minutes is the primary forecasting horizon established
# in the earlier experiment protocol.
# ------------------------------------------------------------

DEEP_ARCHITECTURE_SELECTION_HORIZON = 60

DEEP_ARCHITECTURE_SELECTION_FAMILY = "F0"


assert (
    DEEP_ARCHITECTURE_SELECTION_HORIZON
    in DEEP_FORECAST_HORIZONS
)


assert (
    DEEP_ARCHITECTURE_SELECTION_FAMILY
    in DEEP_AUX_FAMILY_COLUMNS
)


# ------------------------------------------------------------
# 3. Common optimizer / training protocol
# ------------------------------------------------------------

DEEP_BATCH_SIZE = 256

DEEP_MAX_EPOCHS = 80

DEEP_INITIAL_LEARNING_RATE = 1e-3

DEEP_ADAM_CLIPNORM = 1.0


# ------------------------------------------------------------
# 4. Early stopping
#
# Loss is MAE on the standardized target.
#
# Because:
#
#       y_scaled = (y - mean) / std
#
# and std is constant within a horizon, minimizing scaled MAE
# is mathematically equivalent to minimizing MAE in kW.
#
# Reported metrics will still always be computed in kW.
# ------------------------------------------------------------

DEEP_LOSS = "mae"

DEEP_EARLY_STOP_MONITOR = "val_loss"

DEEP_EARLY_STOP_MODE = "min"

DEEP_EARLY_STOP_PATIENCE = 8

DEEP_EARLY_STOP_MIN_DELTA = 1e-4

DEEP_RESTORE_BEST_WEIGHTS = True


# ------------------------------------------------------------
# 5. Training-sample shuffling
#
# Safe here because models are NOT stateful across samples.
#
# Each sample independently contains:
#
#       t-60, t-55, ..., t
#
# in its correct internal temporal order.
# ------------------------------------------------------------

DEEP_SHUFFLE_TRAINING_SAMPLES = True


# ------------------------------------------------------------
# 6. Final prediction policy
#
# Primary evaluation uses raw network predictions after inverse
# transformation to kW.
#
# We do NOT clip predictions to [0, 10] or another physical
# range before computing the primary metrics.
# ------------------------------------------------------------

DEEP_CLIP_PRIMARY_PREDICTIONS = False


# ============================================================
# RECURRENT ARCHITECTURE SEARCH SPACE
# ============================================================
#
# The same four candidate structures are used for:
#
#       LSTM
#       GRU
#       BiLSTM
#
# This keeps the comparison between recurrent model classes
# structurally consistent.
#
# recurrent_dropout is intentionally kept at zero:
#
#   - standard dropout remains available;
#   - recurrent_dropout can substantially reduce accelerated
#     recurrent-kernel performance;
#   - the search remains computationally tractable.
# ============================================================


RECURRENT_ARCHITECTURE_CANDIDATES = (
    {
        "candidate_id":
            "RNN_C1",

        "units":
            32,

        "n_recurrent_layers":
            1,

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "RNN_C2",

        "units":
            64,

        "n_recurrent_layers":
            1,

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "RNN_C3",

        "units":
            64,

        "n_recurrent_layers":
            1,

        "dropout":
            0.20,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "RNN_C4",

        "units":
            64,

        "n_recurrent_layers":
            2,

        "dropout":
            0.20,

        "fusion_dense_units":
            32,
    },
)


# ------------------------------------------------------------
# 7. Assign the common recurrent grid
# ------------------------------------------------------------

DEEP_ARCHITECTURE_CANDIDATES = {
    "LSTM":
        RECURRENT_ARCHITECTURE_CANDIDATES,

    "GRU":
        RECURRENT_ARCHITECTURE_CANDIDATES,

    "BiLSTM":
        RECURRENT_ARCHITECTURE_CANDIDATES,
}


# ============================================================
# TCN ARCHITECTURE SEARCH SPACE
# ============================================================
#
# TCN implementation later will use:
#
#   - causal Conv1D operations;
#   - residual blocks;
#   - exponentially increasing dilation;
#   - only historical sequence values through issue time t.
#
# Therefore TCN remains leakage-safe.
#
# No external keras-tcn package is required.
# ============================================================

TCN_ARCHITECTURE_CANDIDATES = (
    {
        "candidate_id":
            "TCN_C1",

        "filters":
            32,

        "kernel_size":
            2,

        "dilations":
            (1, 2, 4),

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "TCN_C2",

        "filters":
            64,

        "kernel_size":
            2,

        "dilations":
            (1, 2, 4),

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "TCN_C3",

        "filters":
            64,

        "kernel_size":
            3,

        "dilations":
            (1, 2, 4),

        "dropout":
            0.20,

        "fusion_dense_units":
            32,
    },

    {
        "candidate_id":
            "TCN_C4",

        "filters":
            64,

        "kernel_size":
            3,

        "dilations":
            (1, 2, 4),

        "dropout":
            0.20,

        "fusion_dense_units":
            64,
    },
)


DEEP_ARCHITECTURE_CANDIDATES[
    "TCN"
] = TCN_ARCHITECTURE_CANDIDATES


# ============================================================
# ARCHITECTURE-SELECTION RULE
# ============================================================
#
# Candidate ranking:
#
#       1. lowest validation MAE in kW
#       2. lowest validation RMSE in kW
#       3. fewer trainable parameters
#
# The test set is prohibited from architecture selection.
# ============================================================

DEEP_ARCHITECTURE_SELECTION_PRIMARY_METRIC = (
    "validation_MAE_kW"
)

DEEP_ARCHITECTURE_SELECTION_TIEBREAK_METRIC = (
    "validation_RMSE_kW"
)

DEEP_ARCHITECTURE_SELECTION_FINAL_TIEBREAK = (
    "trainable_parameter_count"
)


# ============================================================
# FEATURE-FAMILY EPOCH-SELECTION RULE
# ============================================================
#
# Once an architecture has been selected for a model class:
#
# each of the 15 combinations
#
#       3 horizons × 5 feature families
#
# receives its own train-only / validation early-stopping run.
#
# This chooses only the number of training epochs.
#
# Architecture and optimizer settings remain frozen.
# ============================================================

DEEP_FEATURE_FAMILIES = (
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
)


assert len(
    DEEP_FORECAST_HORIZONS
) * len(
    DEEP_FEATURE_FAMILIES
) == 15


# ============================================================
# FINAL REFIT RULE
# ============================================================
#
# For each frozen horizon × feature-family configuration:
#
#   1. reset model weights;
#   2. combine train + validation samples;
#   3. train for exactly best_epoch epochs;
#   4. keep the original TRAIN-ONLY scalers;
#   5. do not use validation or test callbacks;
#   6. evaluate test once after training.
# ============================================================

DEEP_FINAL_REFIT_USE_TRAIN_PLUS_VALIDATION = True

DEEP_FINAL_REFIT_REUSE_TRAIN_ONLY_SCALERS = True

DEEP_FINAL_REFIT_USES_EARLY_STOPPING = False


# ------------------------------------------------------------
# 8. Fixed random seed for every fresh model fit
# ------------------------------------------------------------

DEEP_MODEL_FIT_SEED = DEEP_LEARNING_SEED


assert DEEP_MODEL_FIT_SEED == 42


# ------------------------------------------------------------
# 9. Validate candidate-grid structure
# ------------------------------------------------------------

for model_name in DEEP_MODEL_ORDER:

    candidates = (
        DEEP_ARCHITECTURE_CANDIDATES[
            model_name
        ]
    )


    assert len(
        candidates
    ) == 4


    candidate_ids = [
        candidate[
            "candidate_id"
        ]
        for candidate in candidates
    ]


    assert len(
        candidate_ids
    ) == len(
        set(
            candidate_ids
        )
    )


# ------------------------------------------------------------
# 10. Build human-readable candidate summary
# ------------------------------------------------------------

candidate_summary_rows = []


for model_name in DEEP_MODEL_ORDER:

    for candidate in (
        DEEP_ARCHITECTURE_CANDIDATES[
            model_name
        ]
    ):

        row = {
            "model":
                model_name,

            "candidate_id":
                candidate[
                    "candidate_id"
                ],
        }


        for key, value in candidate.items():

            if key == "candidate_id":
                continue


            row[
                key
            ] = (
                str(
                    value
                )
                if isinstance(
                    value,
                    tuple,
                )
                else value
            )


        candidate_summary_rows.append(
            row
        )


deep_architecture_candidate_summary = pd.DataFrame(
    candidate_summary_rows
)


# ------------------------------------------------------------
# 11. Report frozen protocol
# ------------------------------------------------------------

print(
    "Frozen neural-network experiment protocol"
)

print(
    "-----------------------------------------"
)


print(
    "Architecture-selection horizon:",
    f"{DEEP_ARCHITECTURE_SELECTION_HORIZON} min",
)

print(
    "Architecture-selection family:",
    DEEP_ARCHITECTURE_SELECTION_FAMILY,
)

print(
    "Candidate architectures per model:",
    4,
)


print(
    "\nTraining"
)

print(
    "  Batch size:",
    DEEP_BATCH_SIZE,
)

print(
    "  Maximum epochs:",
    DEEP_MAX_EPOCHS,
)

print(
    "  Optimizer:",
    "Adam",
)

print(
    "  Initial learning rate:",
    DEEP_INITIAL_LEARNING_RATE,
)

print(
    "  Adam clipnorm:",
    DEEP_ADAM_CLIPNORM,
)

print(
    "  Loss:",
    DEEP_LOSS,
)

print(
    "  Shuffle training samples:",
    DEEP_SHUFFLE_TRAINING_SAMPLES,
)


print(
    "\nEarly stopping"
)

print(
    "  Monitor:",
    DEEP_EARLY_STOP_MONITOR,
)

print(
    "  Patience:",
    DEEP_EARLY_STOP_PATIENCE,
)

print(
    "  Min delta:",
    DEEP_EARLY_STOP_MIN_DELTA,
)

print(
    "  Restore best weights:",
    DEEP_RESTORE_BEST_WEIGHTS,
)


print(
    "\nArchitecture ranking"
)

print(
    "  1.",
    DEEP_ARCHITECTURE_SELECTION_PRIMARY_METRIC,
)

print(
    "  2.",
    DEEP_ARCHITECTURE_SELECTION_TIEBREAK_METRIC,
)

print(
    "  3.",
    DEEP_ARCHITECTURE_SELECTION_FINAL_TIEBREAK,
)


print(
    "\nFinal refit"
)

print(
    "  Data:",
    "train + validation",
)

print(
    "  Epochs:",
    "best epoch selected before test",
)

print(
    "  Scalers:",
    "original train-only statistics",
)

print(
    "  Early stopping:",
    DEEP_FINAL_REFIT_USES_EARLY_STOPPING,
)

print(
    "  Primary prediction clipping:",
    DEEP_CLIP_PRIMARY_PREDICTIONS,
)


print(
    "\nFrozen candidate architectures"
)

display(
    deep_architecture_candidate_summary
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nNeural architecture/training protocol frozen: PASSED"
)

Frozen neural-network experiment protocol
-----------------------------------------
Architecture-selection horizon: 60 min
Architecture-selection family: F0
Candidate architectures per model: 4

Training
  Batch size: 256
  Maximum epochs: 80
  Optimizer: Adam
  Initial learning rate: 0.001
  Adam clipnorm: 1.0
  Loss: mae
  Shuffle training samples: True

Early stopping
  Monitor: val_loss
  Patience: 8
  Min delta: 0.0001
  Restore best weights: True

Architecture ranking
  1. validation_MAE_kW
  2. validation_RMSE_kW
  3. trainable_parameter_count

Final refit
  Data: train + validation
  Epochs: best epoch selected before test
  Scalers: original train-only statistics
  Early stopping: False
  Primary prediction clipping: False

Frozen candidate architectures


,model,candidate_id,units,n_recurrent_layers,dropout,fusion_dense_units,filters,kernel_size,dilations
0,LSTM,RNN_C1,32.0,1.0,0.0,32,NaN,NaN,NaN
1,LSTM,RNN_C2,64.0,1.0,0.0,32,NaN,NaN,NaN
2,LSTM,RNN_C3,64.0,1.0,0.2,32,NaN,NaN,NaN
3,LSTM,RNN_C4,64.0,2.0,0.2,32,NaN,NaN,NaN
4,GRU,RNN_C1,32.0,1.0,0.0,32,NaN,NaN,NaN
5,GRU,RNN_C2,64.0,1.0,0.0,32,NaN,NaN,NaN
6,GRU,RNN_C3,64.0,1.0,0.2,32,NaN,NaN,NaN
7,GRU,RNN_C4,64.0,2.0,0.2,32,NaN,NaN,NaN
8,BiLSTM,RNN_C1,32.0,1.0,0.0,32,NaN,NaN,NaN
9,BiLSTM,RNN_C2,64.0,1.0,0.0,32,NaN,NaN,NaN



Neural architecture/training protocol frozen: PASSED


In [22]:
# ============================================================
# Step 19: Define the generic two-input neural-model factory
# ============================================================
#
# Every Notebook-11 neural model has the same conceptual form:
#
#
#        historical sequence branch
#        --------------------------
#
#        (13 timesteps, 9 channels)
#                    |
#                    v
#          LSTM / GRU / BiLSTM / TCN
#                    |
#                    v
#          learned sequence embedding
#
#
#        auxiliary deterministic branch
#        ------------------------------
#
#        F0:   1 feature
#        F1:  16 features
#        F2:  21 features
#        F3:  30 features
#        F4: 127 features
#
#
#                    sequence embedding
#                           |
#                           |
#                           +---------+
#                                     |
#                                     v
#                            concatenate
#                                     |
#                                     v
#                              Dense(ReLU)
#                                     |
#                                     v
#                              scalar output
#
#
# IMPORTANT:
#
#   - the auxiliary features are already scaled appropriately;
#   - no preprocessing is learned inside the network;
#   - F4 one-hot variables remain literal 0/1;
#   - the output predicts the STANDARDIZED P_Solar target;
#   - predictions will later be inverse-transformed to kW.
#
# This cell defines architectures only.
# NO training occurs here.
# ============================================================


# ------------------------------------------------------------
# 1. Convenience aliases
# ------------------------------------------------------------

keras = tf.keras

layers = tf.keras.layers


# ------------------------------------------------------------
# 2. Reproducible fresh-model reset
#
# Every independent model fit will begin from the same frozen
# seed so initialization and stochastic training operations are
# reproducible under the deterministic TensorFlow protocol.
# ------------------------------------------------------------

def reset_deep_model_state(
    seed: int = DEEP_MODEL_FIT_SEED,
):

    # Remove the previous Keras graph / layer-name state.
    tf.keras.backend.clear_session()


    # Reset Python, NumPy, and TensorFlow RNG state.
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    tf.keras.utils.set_random_seed(
        seed
    )


# ============================================================
# 3. RECURRENT SEQUENCE BRANCH
# ============================================================

def build_recurrent_sequence_branch(
    sequence_input,
    model_name: str,
    candidate_config: dict,
):
    """
    Build the historical-sequence encoder for:

        LSTM
        GRU
        BiLSTM

    Parameters
    ----------
    sequence_input
        Keras input tensor with shape (13, 9).

    model_name
        One of:
            "LSTM"
            "GRU"
            "BiLSTM"

    candidate_config
        Frozen architecture candidate dictionary.

    Returns
    -------
    Keras tensor
        Final fixed-length sequence representation.
    """

    assert model_name in {
        "LSTM",
        "GRU",
        "BiLSTM",
    }


    units = int(
        candidate_config[
            "units"
        ]
    )


    n_recurrent_layers = int(
        candidate_config[
            "n_recurrent_layers"
        ]
    )


    dropout = float(
        candidate_config[
            "dropout"
        ]
    )


    assert n_recurrent_layers in {
        1,
        2,
    }


    x = sequence_input


    # --------------------------------------------------------
    # Build one or two recurrent layers.
    #
    # If two recurrent layers are used:
    #
    #   first layer  -> returns the complete sequence
    #   final layer  -> returns one fixed-length representation
    # --------------------------------------------------------

    for layer_index in range(
        n_recurrent_layers
    ):

        is_last_layer = (
            layer_index
            == n_recurrent_layers - 1
        )


        return_sequences = (
            not is_last_layer
        )


        recurrent_layer_name = (
            f"{model_name.lower()}_"
            f"{layer_index + 1}"
        )


        # ----------------------------------------------------
        # LSTM
        # ----------------------------------------------------

        if model_name == "LSTM":

            x = layers.LSTM(
                units=units,
                return_sequences=return_sequences,
                dropout=dropout,
                recurrent_dropout=0.0,
                name=recurrent_layer_name,
            )(
                x
            )


        # ----------------------------------------------------
        # GRU
        # ----------------------------------------------------

        elif model_name == "GRU":

            x = layers.GRU(
                units=units,
                return_sequences=return_sequences,
                dropout=dropout,
                recurrent_dropout=0.0,
                name=recurrent_layer_name,
            )(
                x
            )


        # ----------------------------------------------------
        # Bidirectional LSTM
        #
        # Bidirectionality operates ONLY within the historical
        # window:
        #
        #       t-60 ... t
        #
        # It never receives observations after issue time t.
        # Therefore it does not introduce forecast leakage.
        # ----------------------------------------------------

        elif model_name == "BiLSTM":

            forward_lstm = layers.LSTM(
                units=units,
                return_sequences=return_sequences,
                dropout=dropout,
                recurrent_dropout=0.0,
                name=(
                    f"bilstm_forward_"
                    f"{layer_index + 1}"
                ),
            )


            x = layers.Bidirectional(
                forward_lstm,
                merge_mode="concat",
                name=recurrent_layer_name,
            )(
                x
            )


    return x


# ============================================================
# 4. TCN RESIDUAL BLOCK
# ============================================================

def tcn_residual_block(
    x,
    filters: int,
    kernel_size: int,
    dilation_rate: int,
    dropout: float,
    block_index: int,
):
    """
    One causal residual TCN block.

    Each block contains:

        causal Conv1D
        ReLU
        Dropout
        causal Conv1D
        ReLU
        Dropout
        residual connection

    A 1x1 projection is used when the residual channel count
    differs from the requested filter count.
    """

    residual = x


    # --------------------------------------------------------
    # First causal convolution
    # --------------------------------------------------------

    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        activation="relu",
        name=(
            f"tcn_block_{block_index}_"
            f"conv1_d{dilation_rate}"
        ),
    )(
        x
    )


    if dropout > 0.0:

        x = layers.Dropout(
            rate=dropout,
            name=(
                f"tcn_block_{block_index}_"
                f"dropout1"
            ),
        )(
            x
        )


    # --------------------------------------------------------
    # Second causal convolution
    # --------------------------------------------------------

    x = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="causal",
        activation="relu",
        name=(
            f"tcn_block_{block_index}_"
            f"conv2_d{dilation_rate}"
        ),
    )(
        x
    )


    if dropout > 0.0:

        x = layers.Dropout(
            rate=dropout,
            name=(
                f"tcn_block_{block_index}_"
                f"dropout2"
            ),
        )(
            x
        )


    # --------------------------------------------------------
    # Residual projection when channel dimensions differ
    # --------------------------------------------------------

    residual_channels = (
        residual.shape[-1]
    )


    if (
        residual_channels is None
        or int(
            residual_channels
        )
        != filters
    ):

        residual = layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            name=(
                f"tcn_block_{block_index}_"
                f"residual_projection"
            ),
        )(
            residual
        )


    # --------------------------------------------------------
    # Residual addition + activation
    # --------------------------------------------------------

    x = layers.Add(
        name=(
            f"tcn_block_{block_index}_"
            f"residual_add"
        )
    )(
        [
            x,
            residual,
        ]
    )


    x = layers.Activation(
        "relu",
        name=(
            f"tcn_block_{block_index}_"
            f"output_relu"
        ),
    )(
        x
    )


    return x


# ============================================================
# 5. TCN SEQUENCE BRANCH
# ============================================================

def build_tcn_sequence_branch(
    sequence_input,
    candidate_config: dict,
):
    """
    Build the causal TCN historical-sequence encoder.
    """

    filters = int(
        candidate_config[
            "filters"
        ]
    )


    kernel_size = int(
        candidate_config[
            "kernel_size"
        ]
    )


    dilations = tuple(
        int(
            dilation
        )
        for dilation
        in candidate_config[
            "dilations"
        ]
    )


    dropout = float(
        candidate_config[
            "dropout"
        ]
    )


    x = sequence_input


    # --------------------------------------------------------
    # Residual causal-convolution blocks
    # --------------------------------------------------------

    for block_index, dilation_rate in enumerate(
        dilations,
        start=1,
    ):

        x = tcn_residual_block(
            x=x,
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            dropout=dropout,
            block_index=block_index,
        )


    # --------------------------------------------------------
    # Convert temporal representation into a fixed-length vector
    #
    # Global average pooling summarizes the encoded historical
    # sequence without introducing any future information.
    # --------------------------------------------------------

    x = layers.GlobalAveragePooling1D(
        name="tcn_global_average_pool"
    )(
        x
    )


    return x


# ============================================================
# 6. GENERIC TWO-INPUT MODEL FACTORY
# ============================================================

def build_deep_forecasting_model(
    model_name: str,
    auxiliary_dim: int,
    candidate_config: dict,
    seed: int = DEEP_MODEL_FIT_SEED,
):
    """
    Build and compile one frozen-protocol neural forecasting
    architecture.

    Parameters
    ----------
    model_name
        "LSTM", "GRU", "BiLSTM", or "TCN".

    auxiliary_dim
        Number of auxiliary inputs for the selected family:
            F0 ->   1
            F1 ->  16
            F2 ->  21
            F3 ->  30
            F4 -> 127

    candidate_config
        One frozen architecture candidate.

    seed
        Reproducible initialization seed.

    Returns
    -------
    tf.keras.Model
        Compiled two-input regression model.
    """

    assert model_name in DEEP_MODEL_ORDER


    assert auxiliary_dim in {
        1,
        16,
        21,
        30,
        127,
    }


    # --------------------------------------------------------
    # Reproducible clean initialization
    # --------------------------------------------------------

    reset_deep_model_state(
        seed=seed
    )


    # ========================================================
    # INPUT 1: HISTORICAL SEQUENCE
    # ========================================================

    sequence_input = layers.Input(
        shape=(
            DEEP_SEQUENCE_LENGTH,
            len(
                DEEP_SEQUENCE_CHANNELS
            ),
        ),
        dtype=tf.float32,
        name="sequence_input",
    )


    # --------------------------------------------------------
    # Architecture-specific sequence encoder
    # --------------------------------------------------------

    if model_name in {
        "LSTM",
        "GRU",
        "BiLSTM",
    }:

        sequence_embedding = (
            build_recurrent_sequence_branch(
                sequence_input=sequence_input,
                model_name=model_name,
                candidate_config=candidate_config,
            )
        )


    elif model_name == "TCN":

        sequence_embedding = (
            build_tcn_sequence_branch(
                sequence_input=sequence_input,
                candidate_config=candidate_config,
            )
        )


    # ========================================================
    # INPUT 2: AUXILIARY FEATURES
    # ========================================================

    auxiliary_input = layers.Input(
        shape=(
            auxiliary_dim,
        ),
        dtype=tf.float32,
        name="auxiliary_input",
    )


    # ========================================================
    # FUSION
    # ========================================================

    fused = layers.Concatenate(
        name="sequence_auxiliary_fusion"
    )(
        [
            sequence_embedding,
            auxiliary_input,
        ]
    )


    fusion_dense_units = int(
        candidate_config[
            "fusion_dense_units"
        ]
    )


    fused = layers.Dense(
        units=fusion_dense_units,
        activation="relu",
        name="fusion_dense",
    )(
        fused
    )


    # --------------------------------------------------------
    # Scalar standardized-target prediction
    # --------------------------------------------------------

    output = layers.Dense(
        units=1,
        activation="linear",
        name="scaled_p_solar_output",
    )(
        fused
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = keras.Model(
        inputs=[
            sequence_input,
            auxiliary_input,
        ],
        outputs=output,
        name=(
            f"{model_name.lower()}_"
            f"{candidate_config['candidate_id'].lower()}"
        ),
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = keras.optimizers.Adam(
        learning_rate=(
            DEEP_INITIAL_LEARNING_RATE
        ),
        clipnorm=(
            DEEP_ADAM_CLIPNORM
        ),
    )


    # ========================================================
    # COMPILE
    # ========================================================

    model.compile(
        optimizer=optimizer,
        loss=DEEP_LOSS,
    )


    return model


# ============================================================
# 7. Structural smoke test:
#    LSTM candidate RNN_C1 + F0
# ============================================================


# ------------------------------------------------------------
# Candidate selected ONLY for architecture-factory validation.
#
# This is NOT an experimentally selected architecture yet.
# ------------------------------------------------------------

lstm_c1_config = (
    DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ][
        0
    ]
)


assert (
    lstm_c1_config[
        "candidate_id"
    ]
    == "RNN_C1"
)


f0_auxiliary_dim = len(
    DEEP_AUX_FAMILY_COLUMNS[
        "F0"
    ]
)


assert f0_auxiliary_dim == 1


lstm_c1_smoke_model = (
    build_deep_forecasting_model(
        model_name="LSTM",
        auxiliary_dim=f0_auxiliary_dim,
        candidate_config=lstm_c1_config,
    )
)


# ============================================================
# 8. Strong architecture checks
# ============================================================

assert len(
    lstm_c1_smoke_model.inputs
) == 2


assert tuple(
    lstm_c1_smoke_model.inputs[
        0
    ].shape
) == (
    None,
    13,
    9,
)


assert tuple(
    lstm_c1_smoke_model.inputs[
        1
    ].shape
) == (
    None,
    1,
)


assert tuple(
    lstm_c1_smoke_model.output.shape
) == (
    None,
    1,
)


assert (
    lstm_c1_smoke_model.loss
    == "mae"
)


# ------------------------------------------------------------
# Expected parameter count for LSTM RNN_C1 + F0:
#
# LSTM(9 -> 32):
#
#   4 × units × (input_dim + units + 1)
#
#   = 4 × 32 × (9 + 32 + 1)
#   = 5,376
#
# Fusion Dense:
#
#   (32 sequence + 1 auxiliary) -> 32
#
#   = 33 × 32 + 32
#   = 1,088
#
# Output:
#
#   32 -> 1
#
#   = 32 + 1
#   = 33
#
# Total:
#
#   5,376 + 1,088 + 33
#   = 6,497
# ------------------------------------------------------------

EXPECTED_LSTM_C1_F0_PARAMETERS = 6_497


assert (
    lstm_c1_smoke_model.count_params()
    == EXPECTED_LSTM_C1_F0_PARAMETERS
)


# ============================================================
# 9. Report
# ============================================================

print(
    "Generic deep-learning architecture factory"
)

print(
    "------------------------------------------"
)

print(
    "Supported models:",
    DEEP_MODEL_ORDER,
)

print(
    "Sequence input shape:",
    tuple(
        lstm_c1_smoke_model.inputs[
            0
        ].shape
    ),
)

print(
    "F0 auxiliary input shape:",
    tuple(
        lstm_c1_smoke_model.inputs[
            1
        ].shape
    ),
)

print(
    "Output shape:",
    tuple(
        lstm_c1_smoke_model.output.shape
    ),
)

print(
    "Smoke-test model:",
    lstm_c1_smoke_model.name,
)

print(
    "Trainable parameters:",
    lstm_c1_smoke_model.count_params(),
)


print(
    "\nLSTM RNN_C1 + F0 model summary"
)

print(
    "------------------------------"
)


lstm_c1_smoke_model.summary()


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nGeneric neural architecture factory: PASSED"
)

2026-09-21 12:42:45.057503: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-21 12:42:45.057672: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-21 12:42:45.057694: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1789974765.057904 48306307 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1789974765.058249 48306307 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Generic deep-learning architecture factory
------------------------------------------
Supported models: ('LSTM', 'GRU', 'BiLSTM', 'TCN')
Sequence input shape: (None, 13, 9)
F0 auxiliary input shape: (None, 1)
Output shape: (None, 1)
Smoke-test model: lstm_rnn_c1
Trainable parameters: 6497

LSTM RNN_C1 + F0 model summary
------------------------------


Model: "lstm_rnn_c1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 13, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │      5,376 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ auxiliary_input     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence_auxiliary… │ (None, 33)        │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ auxiliary_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_dense        │ (None, 32)        │      1,088 │ sequence_auxilia… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scaled_p_solar_out… │ (None, 1)         │         33 │ fusion_dense[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,497 (25.38 KB)

 Trainable params: 6,497 (25.38 KB)

 Non-trainable params: 0 (0.00 B)


Generic neural architecture factory: PASSED


In [23]:
# ============================================================
# Step 20: Real-data forward-pass smoke test
# ============================================================
#
# We now pass a very small batch of REAL, already-scaled
# 60-minute F0 samples through the untrained LSTM model.
#
# Purpose:
#
#   - verify sequence tensor compatibility;
#   - verify F0 auxiliary slicing;
#   - verify TensorFlow accepts both inputs together;
#   - verify output shape and numerical finiteness;
#   - verify inverse transformation from standardized output
#     back to physical kW units.
#
# IMPORTANT:
#
# The model is still UNTRAINED.
#
# Therefore the predictions shown here have absolutely no
# forecasting meaning and must not be interpreted as model
# performance.
# ============================================================


# ------------------------------------------------------------
# 1. Use the primary 60-minute population
# ------------------------------------------------------------

FORWARD_TEST_HORIZON = 60

FORWARD_TEST_FAMILY = "F0"

FORWARD_TEST_BATCH_SIZE = 8


forward_data = (
    deep_scaled_data[
        FORWARD_TEST_HORIZON
    ]
)


forward_base_table = (
    deep_base_modelling_tables[
        FORWARD_TEST_HORIZON
    ]
)


# ------------------------------------------------------------
# 2. Obtain the canonical F0 auxiliary positions
#
# F0 contains only:
#
#       P_Solar(t - 24 h)
# ------------------------------------------------------------

f0_positions = (
    DEEP_AUX_FAMILY_POSITIONS[
        FORWARD_TEST_FAMILY
    ]
)


assert f0_positions.shape == (
    1,
)


assert (
    DEEP_AUX_FAMILY_COLUMNS[
        "F0"
    ]
    == [
        "pv_lag_24h"
    ]
)


# ------------------------------------------------------------
# 3. Select a tiny real-data batch
# ------------------------------------------------------------

forward_sequence_batch = (
    forward_data[
        "sequence"
    ][
        :FORWARD_TEST_BATCH_SIZE
    ]
)


forward_auxiliary_batch = (
    forward_data[
        "auxiliary_f4"
    ][
        :FORWARD_TEST_BATCH_SIZE,
        :
    ][
        :,
        f0_positions
    ]
)


forward_target_scaled = (
    forward_data[
        "target_scaled"
    ][
        :FORWARD_TEST_BATCH_SIZE
    ]
)


forward_target_kw = (
    forward_data[
        "target_kw"
    ][
        :FORWARD_TEST_BATCH_SIZE
    ]
)


# ------------------------------------------------------------
# 4. Strong input-shape checks
# ------------------------------------------------------------

assert forward_sequence_batch.shape == (
    FORWARD_TEST_BATCH_SIZE,
    13,
    9,
)


assert forward_auxiliary_batch.shape == (
    FORWARD_TEST_BATCH_SIZE,
    1,
)


assert forward_target_scaled.shape == (
    FORWARD_TEST_BATCH_SIZE,
)


assert forward_sequence_batch.dtype == np.float32

assert forward_auxiliary_batch.dtype == np.float32


assert np.isfinite(
    forward_sequence_batch
).all()


assert np.isfinite(
    forward_auxiliary_batch
).all()


# ------------------------------------------------------------
# 5. Perform inference with the UNTRAINED model
#
# training=False guarantees inference behavior.
# ------------------------------------------------------------

forward_prediction_tensor = (
    lstm_c1_smoke_model(
        [
            forward_sequence_batch,
            forward_auxiliary_batch,
        ],
        training=False,
    )
)


# ------------------------------------------------------------
# 6. Convert TensorFlow output -> NumPy
# ------------------------------------------------------------

forward_prediction_scaled = (
    forward_prediction_tensor
    .numpy()
    .reshape(
        -1
    )
)


assert forward_prediction_scaled.shape == (
    FORWARD_TEST_BATCH_SIZE,
)


assert np.isfinite(
    forward_prediction_scaled
).all()


# ------------------------------------------------------------
# 7. Inverse-transform untrained predictions to kW
#
# These values are displayed ONLY to validate the inverse
# transformation pipeline.
# ------------------------------------------------------------

forward_stats = (
    deep_scaling_statistics[
        FORWARD_TEST_HORIZON
    ]
)


forward_prediction_kw = (
    forward_prediction_scaled.astype(
        np.float64
    )
    * forward_stats[
        "target_std"
    ]
    + forward_stats[
        "target_mean"
    ]
)


assert np.isfinite(
    forward_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Construct an audit table
#
# Again: these are UNTRAINED predictions.
# ------------------------------------------------------------

forward_audit = pd.DataFrame(
    {
        "issue_time":
            forward_base_table.index[
                :FORWARD_TEST_BATCH_SIZE
            ],

        "valid_time":
            forward_base_table[
                "valid_time"
            ]
            .iloc[
                :FORWARD_TEST_BATCH_SIZE
            ]
            .to_numpy(),

        "F0_pv_lag_24h_scaled":
            forward_auxiliary_batch[
                :,
                0
            ],

        "untrained_prediction_scaled":
            forward_prediction_scaled,

        "untrained_prediction_kW":
            forward_prediction_kw,

        "actual_target_kW":
            forward_target_kw,
    }
)


# ------------------------------------------------------------
# 9. Report structural results
# ------------------------------------------------------------

print(
    "Real-data LSTM forward-pass smoke test"
)

print(
    "--------------------------------------"
)


print(
    "Model:",
    lstm_c1_smoke_model.name,
)

print(
    "Horizon:",
    f"{FORWARD_TEST_HORIZON} min",
)

print(
    "Feature family:",
    FORWARD_TEST_FAMILY,
)

print(
    "Batch size:",
    FORWARD_TEST_BATCH_SIZE,
)


print(
    "\nSequence input:"
)

print(
    "  Shape:",
    forward_sequence_batch.shape,
)

print(
    "  dtype:",
    forward_sequence_batch.dtype,
)


print(
    "\nAuxiliary input:"
)

print(
    "  Shape:",
    forward_auxiliary_batch.shape,
)

print(
    "  dtype:",
    forward_auxiliary_batch.dtype,
)


print(
    "\nModel output:"
)

print(
    "  Tensor shape:",
    tuple(
        forward_prediction_tensor.shape
    ),
)

print(
    "  NumPy shape:",
    forward_prediction_scaled.shape,
)

print(
    "  All finite:",
    bool(
        np.isfinite(
            forward_prediction_scaled
        ).all()
    ),
)


print(
    "\nUNTRAINED prediction audit:"
)

display(
    forward_audit
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nReal-data neural forward pass: PASSED"
)

print(
    "Reminder: predictions above are from an UNTRAINED model."
)

Real-data LSTM forward-pass smoke test
--------------------------------------
Model: lstm_rnn_c1
Horizon: 60 min
Feature family: F0
Batch size: 8

Sequence input:
  Shape: (8, 13, 9)
  dtype: float32

Auxiliary input:
  Shape: (8, 1)
  dtype: float32

Model output:
  Tensor shape: (8, 1)
  NumPy shape: (8,)
  All finite: True

UNTRAINED prediction audit:


,issue_time,valid_time,F0_pv_lag_24h_scaled,untrained_prediction_scaled,untrained_prediction_kW,actual_target_kW
0,2018-06-02 00:00:00+00:00,2018-06-02 01:00:00+00:00,-0.528914,0.204789,1.164065,0.0
1,2018-06-02 00:05:00+00:00,2018-06-02 01:05:00+00:00,-0.528914,0.144086,1.067675,0.0
2,2018-06-02 00:10:00+00:00,2018-06-02 01:10:00+00:00,-0.528914,0.136339,1.055374,0.0
3,2018-06-02 00:15:00+00:00,2018-06-02 01:15:00+00:00,-0.528914,0.145609,1.070093,0.0
4,2018-06-02 00:20:00+00:00,2018-06-02 01:20:00+00:00,-0.528914,0.160862,1.094314,0.0
5,2018-06-02 00:25:00+00:00,2018-06-02 01:25:00+00:00,-0.528914,0.168671,1.106713,0.0
6,2018-06-02 00:30:00+00:00,2018-06-02 01:30:00+00:00,-0.528914,0.173225,1.113945,0.0
7,2018-06-02 00:35:00+00:00,2018-06-02 01:35:00+00:00,-0.528914,0.176850,1.119701,0.0



Real-data neural forward pass: PASSED
Reminder: predictions above are from an UNTRAINED model.


In [24]:
# ============================================================
# Step 21: Define reusable neural training / validation helpers
# ============================================================
#
# This cell defines the machinery used for architecture
# selection and later feature-family epoch selection.
#
# IMPORTANT:
#
#   - NO test data are accessed by the training function.
#   - NO test metric is computed here.
#   - validation is used only for model/epoch selection.
#   - all reported validation errors are transformed back to kW.
#
#
# Candidate-training workflow
# ---------------------------
#
# For one:
#
#       model class
#       architecture candidate
#       forecast horizon
#       feature family
#
# the function will:
#
#   1. extract TRAIN and VALIDATION data only;
#   2. build a fresh seeded model;
#   3. train with EarlyStopping;
#   4. restore callback-selected best weights;
#   5. predict validation samples;
#   6. inverse-transform predictions to kW;
#   7. compute validation MAE / RMSE / R²;
#   8. record the callback-selected best epoch;
#   9. discard the fitted model to conserve memory.
#
# The held-out TEST population is not touched.
# ============================================================


# ------------------------------------------------------------
# 1. Inverse-transform standardized target values to kW
# ------------------------------------------------------------

def inverse_transform_target_to_kw(
    scaled_values,
    horizon_minutes: int,
):
    """
    Convert standardized target values back to physical kW.
    """

    assert horizon_minutes in DEEP_FORECAST_HORIZONS


    stats = (
        deep_scaling_statistics[
            horizon_minutes
        ]
    )


    scaled_values = np.asarray(
        scaled_values,
        dtype=np.float64,
    )


    values_kw = (
        scaled_values
        * stats[
            "target_std"
        ]
        + stats[
            "target_mean"
        ]
    )


    assert np.isfinite(
        values_kw
    ).all()


    return values_kw


# ------------------------------------------------------------
# 2. Regression metrics in physical kW units
# ------------------------------------------------------------

def compute_regression_metrics_kw(
    y_true_kw,
    y_pred_kw,
):
    """
    Compute validation/test regression metrics in kW.

    Returns
    -------
    dict
        MAE, RMSE, and R².
    """

    y_true_kw = np.asarray(
        y_true_kw,
        dtype=np.float64,
    ).reshape(
        -1
    )


    y_pred_kw = np.asarray(
        y_pred_kw,
        dtype=np.float64,
    ).reshape(
        -1
    )


    assert y_true_kw.shape == y_pred_kw.shape

    assert len(
        y_true_kw
    ) > 0

    assert np.isfinite(
        y_true_kw
    ).all()

    assert np.isfinite(
        y_pred_kw
    ).all()


    errors = (
        y_pred_kw
        - y_true_kw
    )


    absolute_errors = np.abs(
        errors
    )


    squared_errors = np.square(
        errors
    )


    mae_kw = float(
        absolute_errors.mean()
    )


    rmse_kw = float(
        np.sqrt(
            squared_errors.mean()
        )
    )


    y_true_mean = float(
        y_true_kw.mean()
    )


    total_sum_of_squares = float(
        np.square(
            y_true_kw
            - y_true_mean
        ).sum()
    )


    residual_sum_of_squares = float(
        squared_errors.sum()
    )


    if total_sum_of_squares > 0.0:

        r2 = float(
            1.0
            - (
                residual_sum_of_squares
                / total_sum_of_squares
            )
        )

    else:

        r2 = np.nan


    return {
        "MAE_kW":
            mae_kw,

        "RMSE_kW":
            rmse_kw,

        "R2":
            r2,
    }


# ------------------------------------------------------------
# 3. Extract one scaled feature-family auxiliary matrix
# ------------------------------------------------------------

def get_scaled_auxiliary_family(
    horizon_minutes: int,
    feature_family: str,
):
    """
    Slice the full scaled F4 auxiliary matrix to one nested
    feature family without rebuilding or rescaling anything.
    """

    assert horizon_minutes in DEEP_FORECAST_HORIZONS

    assert feature_family in DEEP_FEATURE_FAMILIES


    full_auxiliary = (
        deep_scaled_data[
            horizon_minutes
        ][
            "auxiliary_f4"
        ]
    )


    positions = (
        DEEP_AUX_FAMILY_POSITIONS[
            feature_family
        ]
    )


    family_auxiliary = (
        full_auxiliary[
            :,
            positions,
        ]
    )


    expected_width = len(
        DEEP_AUX_FAMILY_COLUMNS[
            feature_family
        ]
    )


    assert family_auxiliary.shape == (
        full_auxiliary.shape[0],
        expected_width,
    )


    assert family_auxiliary.dtype == np.float32

    assert np.isfinite(
        family_auxiliary
    ).all()


    return np.ascontiguousarray(
        family_auxiliary,
        dtype=np.float32,
    )


# ------------------------------------------------------------
# 4. Extract one chronological split for model input
# ------------------------------------------------------------

def get_deep_split_data(
    horizon_minutes: int,
    feature_family: str,
    split_name: str,
):
    """
    Return scaled sequence, scaled auxiliary, scaled target,
    and physical target for exactly one frozen split.

    split_name:
        "train"
        "validation"
        "test"

    NOTE:
    The architecture-selection training function below will call
    this helper ONLY for train and validation.
    """

    assert split_name in {
        "train",
        "validation",
        "test",
    }


    transformed = (
        deep_scaled_data[
            horizon_minutes
        ]
    )


    mask_key = (
        f"{split_name}_mask"
    )


    split_mask = (
        transformed[
            mask_key
        ]
    )


    auxiliary_family = (
        get_scaled_auxiliary_family(
            horizon_minutes=horizon_minutes,
            feature_family=feature_family,
        )
    )


    sequence = (
        transformed[
            "sequence"
        ][
            split_mask
        ]
    )


    auxiliary = (
        auxiliary_family[
            split_mask
        ]
    )


    target_scaled = (
        transformed[
            "target_scaled"
        ][
            split_mask
        ]
    )


    target_kw = (
        transformed[
            "target_kw"
        ][
            split_mask
        ]
    )


    assert sequence.shape[0] == auxiliary.shape[0]

    assert sequence.shape[0] == target_scaled.shape[0]

    assert sequence.shape[0] == target_kw.shape[0]


    assert sequence.shape[1:] == (
        13,
        9,
    )


    assert auxiliary.shape[1] == len(
        DEEP_AUX_FAMILY_COLUMNS[
            feature_family
        ]
    )


    return {
        "sequence":
            np.ascontiguousarray(
                sequence,
                dtype=np.float32,
            ),

        "auxiliary":
            np.ascontiguousarray(
                auxiliary,
                dtype=np.float32,
            ),

        "target_scaled":
            np.ascontiguousarray(
                target_scaled,
                dtype=np.float32,
            ),

        "target_kw":
            np.ascontiguousarray(
                target_kw,
                dtype=np.float32,
            ),

        "n_samples":
            int(
                sequence.shape[0]
            ),
    }


# ------------------------------------------------------------
# 5. Candidate validation-training function
# ------------------------------------------------------------

def run_validation_candidate(
    model_name: str,
    candidate_config: dict,
    horizon_minutes: int,
    feature_family: str,
    verbose: int = 0,
):
    """
    Train one neural candidate on TRAIN only and evaluate it on
    VALIDATION only.

    The TEST split is never accessed.

    Returns
    -------
    result : dict
        Candidate-level validation summary.

    history_df : pandas.DataFrame
        Per-epoch training/validation loss history.
    """

    # --------------------------------------------------------
    # Structural protocol checks
    # --------------------------------------------------------

    assert model_name in DEEP_MODEL_ORDER

    assert horizon_minutes in DEEP_FORECAST_HORIZONS

    assert feature_family in DEEP_FEATURE_FAMILIES


    valid_candidate_ids = {
        candidate[
            "candidate_id"
        ]
        for candidate
        in DEEP_ARCHITECTURE_CANDIDATES[
            model_name
        ]
    }


    assert (
        candidate_config[
            "candidate_id"
        ]
        in valid_candidate_ids
    )


    # --------------------------------------------------------
    # TRAIN and VALIDATION only
    # --------------------------------------------------------

    train_data = (
        get_deep_split_data(
            horizon_minutes=horizon_minutes,
            feature_family=feature_family,
            split_name="train",
        )
    )


    validation_data = (
        get_deep_split_data(
            horizon_minutes=horizon_minutes,
            feature_family=feature_family,
            split_name="validation",
        )
    )


    auxiliary_dim = int(
        train_data[
            "auxiliary"
        ].shape[1]
    )


    assert auxiliary_dim == len(
        DEEP_AUX_FAMILY_COLUMNS[
            feature_family
        ]
    )


    # --------------------------------------------------------
    # Fresh, reproducibly seeded network
    # --------------------------------------------------------

    model = (
        build_deep_forecasting_model(
            model_name=model_name,
            auxiliary_dim=auxiliary_dim,
            candidate_config=candidate_config,
            seed=DEEP_MODEL_FIT_SEED,
        )
    )


    trainable_parameter_count = int(
        model.count_params()
    )


    # --------------------------------------------------------
    # Early stopping
    #
    # restore_best_weights=True means validation prediction
    # below uses the callback-selected best weights.
    # --------------------------------------------------------

    early_stopping = (
        keras.callbacks.EarlyStopping(
            monitor=DEEP_EARLY_STOP_MONITOR,
            mode=DEEP_EARLY_STOP_MODE,
            patience=DEEP_EARLY_STOP_PATIENCE,
            min_delta=DEEP_EARLY_STOP_MIN_DELTA,
            restore_best_weights=(
                DEEP_RESTORE_BEST_WEIGHTS
            ),
            verbose=0,
        )
    )


    # --------------------------------------------------------
    # Training timer
    # --------------------------------------------------------

    fit_start_time = (
        time.perf_counter()
    )


    history = model.fit(
        x=[
            train_data[
                "sequence"
            ],
            train_data[
                "auxiliary"
            ],
        ],

        y=(
            train_data[
                "target_scaled"
            ]
        ),

        validation_data=(
            [
                validation_data[
                    "sequence"
                ],
                validation_data[
                    "auxiliary"
                ],
            ],
            validation_data[
                "target_scaled"
            ],
        ),

        epochs=DEEP_MAX_EPOCHS,

        batch_size=DEEP_BATCH_SIZE,

        shuffle=(
            DEEP_SHUFFLE_TRAINING_SAMPLES
        ),

        callbacks=[
            early_stopping
        ],

        verbose=verbose,
    )


    fit_seconds = float(
        time.perf_counter()
        - fit_start_time
    )


    # --------------------------------------------------------
    # Number of epochs actually executed
    # --------------------------------------------------------

    epochs_ran = int(
        len(
            history.history[
                "loss"
            ]
        )
    )


    assert (
        1
        <= epochs_ran
        <= DEEP_MAX_EPOCHS
    )


    # --------------------------------------------------------
    # Callback-selected best epoch
    #
    # Keras uses zero-based epoch indexing internally.
    # We store one-based epoch counts because final refitting
    # requires:
    #
    #       model.fit(..., epochs=best_epoch)
    #
    # This epoch is intentionally taken from the SAME callback
    # whose weights were restored.
    # --------------------------------------------------------

    if hasattr(
        early_stopping,
        "best_epoch",
    ):

        best_epoch = int(
            early_stopping.best_epoch
            + 1
        )

    else:

        # Compatibility fallback.
        #
        # Only used if the installed Keras does not expose
        # best_epoch directly.
        best_epoch = int(
            np.argmin(
                np.asarray(
                    history.history[
                        DEEP_EARLY_STOP_MONITOR
                    ],
                    dtype=np.float64,
                )
            )
            + 1
        )


    assert (
        1
        <= best_epoch
        <= epochs_ran
    )


    # --------------------------------------------------------
    # Callback-best validation loss
    # --------------------------------------------------------

    callback_best_val_loss = float(
        early_stopping.best
    )


    assert np.isfinite(
        callback_best_val_loss
    )


    # --------------------------------------------------------
    # Validation prediction using RESTORED BEST WEIGHTS
    # --------------------------------------------------------

    validation_prediction_scaled = (
        model.predict(
            [
                validation_data[
                    "sequence"
                ],
                validation_data[
                    "auxiliary"
                ],
            ],
            batch_size=DEEP_BATCH_SIZE,
            verbose=0,
        )
        .reshape(
            -1
        )
    )


    assert validation_prediction_scaled.shape == (
        validation_data[
            "target_scaled"
        ].shape
    )


    assert np.isfinite(
        validation_prediction_scaled
    ).all()


    # --------------------------------------------------------
    # Convert validation predictions back to physical kW
    # --------------------------------------------------------

    validation_prediction_kw = (
        inverse_transform_target_to_kw(
            scaled_values=(
                validation_prediction_scaled
            ),
            horizon_minutes=horizon_minutes,
        )
    )


    validation_target_kw = (
        validation_data[
            "target_kw"
        ]
        .astype(
            np.float64
        )
    )


    # --------------------------------------------------------
    # Physical-unit validation metrics
    # --------------------------------------------------------

    validation_metrics = (
        compute_regression_metrics_kw(
            y_true_kw=validation_target_kw,
            y_pred_kw=validation_prediction_kw,
        )
    )


    # --------------------------------------------------------
    # Store per-epoch history
    # --------------------------------------------------------

    history_df = pd.DataFrame(
        history.history
    )


    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                history_df
            )
            + 1,
            dtype=int,
        ),
    )


    history_df[
        "model"
    ] = model_name


    history_df[
        "candidate_id"
    ] = candidate_config[
        "candidate_id"
    ]


    history_df[
        "horizon_minutes"
    ] = horizon_minutes


    history_df[
        "feature_family"
    ] = feature_family


    history_df[
        "is_callback_best_epoch"
    ] = (
        history_df[
            "epoch"
        ]
        == best_epoch
    )


    assert (
        history_df[
            "is_callback_best_epoch"
        ].sum()
        == 1
    )


    # --------------------------------------------------------
    # Candidate summary
    #
    # Architecture-selection ranking later uses:
    #
    #   validation_MAE_kW
    #   validation_RMSE_kW
    #   trainable_parameter_count
    # --------------------------------------------------------

    result = {
        "model":
            model_name,

        "candidate_id":
            candidate_config[
                "candidate_id"
            ],

        "horizon_minutes":
            int(
                horizon_minutes
            ),

        "feature_family":
            feature_family,

        "n_train":
            train_data[
                "n_samples"
            ],

        "n_validation":
            validation_data[
                "n_samples"
            ],

        "auxiliary_dim":
            auxiliary_dim,

        "trainable_parameter_count":
            trainable_parameter_count,

        "epochs_ran":
            epochs_ran,

        "best_epoch":
            best_epoch,

        "callback_best_val_loss_scaled":
            callback_best_val_loss,

        "validation_MAE_kW":
            validation_metrics[
                "MAE_kW"
            ],

        "validation_RMSE_kW":
            validation_metrics[
                "RMSE_kW"
            ],

        "validation_R2":
            validation_metrics[
                "R2"
            ],

        "fit_seconds":
            fit_seconds,
    }


    # --------------------------------------------------------
    # Preserve frozen architecture fields in the summary
    # --------------------------------------------------------

    for key, value in candidate_config.items():

        if key == "candidate_id":
            continue


        result[
            f"config_{key}"
        ] = value


    # --------------------------------------------------------
    # Explicitly release fitted network resources.
    #
    # Architecture selection needs the metrics/history, not
    # four simultaneously resident fitted models.
    # --------------------------------------------------------

    del model

    del history

    del early_stopping


    tf.keras.backend.clear_session()

    gc.collect()


    return (
        result,
        history_df,
    )


# ============================================================
# NON-TRAINING VALIDATION OF THE HELPERS
# ============================================================


# ------------------------------------------------------------
# 6. Verify split extraction on primary 60-min F0 population
# ------------------------------------------------------------

helper_train_f0_60 = (
    get_deep_split_data(
        horizon_minutes=60,
        feature_family="F0",
        split_name="train",
    )
)


helper_validation_f0_60 = (
    get_deep_split_data(
        horizon_minutes=60,
        feature_family="F0",
        split_name="validation",
    )
)


assert helper_train_f0_60[
    "sequence"
].shape == (
    87_252,
    13,
    9,
)


assert helper_train_f0_60[
    "auxiliary"
].shape == (
    87_252,
    1,
)


assert helper_validation_f0_60[
    "sequence"
].shape == (
    17_568,
    13,
    9,
)


assert helper_validation_f0_60[
    "auxiliary"
].shape == (
    17_568,
    1,
)


# ------------------------------------------------------------
# 7. Test inverse target scaling on real values
# ------------------------------------------------------------

reconstructed_validation_target = (
    inverse_transform_target_to_kw(
        scaled_values=(
            helper_validation_f0_60[
                "target_scaled"
            ]
        ),
        horizon_minutes=60,
    )
)


assert np.allclose(
    reconstructed_validation_target,
    helper_validation_f0_60[
        "target_kw"
    ].astype(
        np.float64
    ),
    atol=1e-6,
    rtol=1e-6,
)


# ------------------------------------------------------------
# 8. Test metric helper on a simple exact example
#
# True:
#       [0, 1, 2]
#
# Prediction:
#       [0, 1, 2]
#
# therefore:
#
#       MAE  = 0
#       RMSE = 0
#       R²   = 1
# ------------------------------------------------------------

metric_smoke_test = (
    compute_regression_metrics_kw(
        y_true_kw=np.array(
            [
                0.0,
                1.0,
                2.0,
            ]
        ),
        y_pred_kw=np.array(
            [
                0.0,
                1.0,
                2.0,
            ]
        ),
    )
)


assert np.isclose(
    metric_smoke_test[
        "MAE_kW"
    ],
    0.0,
)


assert np.isclose(
    metric_smoke_test[
        "RMSE_kW"
    ],
    0.0,
)


assert np.isclose(
    metric_smoke_test[
        "R2"
    ],
    1.0,
)


# ------------------------------------------------------------
# 9. Report helper validation
# ------------------------------------------------------------

print(
    "Neural training/validation helper setup"
)

print(
    "---------------------------------------"
)


print(
    "Primary architecture-selection population:"
)

print(
    "  Horizon:",
    "60 min",
)

print(
    "  Family:",
    "F0",
)


print(
    "\nTraining split:"
)

print(
    "  Sequence:",
    helper_train_f0_60[
        "sequence"
    ].shape,
)

print(
    "  Auxiliary:",
    helper_train_f0_60[
        "auxiliary"
    ].shape,
)

print(
    "  Target:",
    helper_train_f0_60[
        "target_scaled"
    ].shape,
)


print(
    "\nValidation split:"
)

print(
    "  Sequence:",
    helper_validation_f0_60[
        "sequence"
    ].shape,
)

print(
    "  Auxiliary:",
    helper_validation_f0_60[
        "auxiliary"
    ].shape,
)

print(
    "  Target:",
    helper_validation_f0_60[
        "target_scaled"
    ].shape,
)


print(
    "\nMetric helper smoke test:"
)

print(
    metric_smoke_test
)


print(
    "\nCandidate-training function:"
)

print(
    "  train input:      allowed"
)

print(
    "  validation input: allowed"
)

print(
    "  test input:       NOT USED"
)


# ------------------------------------------------------------
# 10. Clean up temporary helper arrays
# ------------------------------------------------------------

del helper_train_f0_60

del helper_validation_f0_60

del reconstructed_validation_target


gc.collect()


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nNeural training/validation helpers: PASSED"
)

Neural training/validation helper setup
---------------------------------------
Primary architecture-selection population:
  Horizon: 60 min
  Family: F0

Training split:
  Sequence: (87252, 13, 9)
  Auxiliary: (87252, 1)
  Target: (87252,)

Validation split:
  Sequence: (17568, 13, 9)
  Auxiliary: (17568, 1)
  Target: (17568,)

Metric helper smoke test:
{'MAE_kW': 0.0, 'RMSE_kW': 0.0, 'R2': 1.0}

Candidate-training function:
  train input:      allowed
  validation input: allowed
  test input:       NOT USED

Neural training/validation helpers: PASSED


In [25]:
# ============================================================
# Step 22: First actual neural architecture-selection run
#
# Model:
#       LSTM
#
# Candidate:
#       RNN_C1
#
# Architecture:
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# Population:
#       60-minute forecast horizon
#       F0 feature family
#
# Data usage:
#       TRAIN      -> model fitting
#       VALIDATION -> early stopping + architecture evaluation
#       TEST       -> NOT TOUCHED
#
# This is architecture-selection run 1 of 4 for LSTM.
# ============================================================


# ------------------------------------------------------------
# 1. Select frozen LSTM candidate RNN_C1
# ------------------------------------------------------------

lstm_c1_config = (
    DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ][
        0
    ]
)


assert (
    lstm_c1_config[
        "candidate_id"
    ]
    == "RNN_C1"
)


# ------------------------------------------------------------
# 2. Run train-only fitting + validation evaluation
#
# verbose=2 prints one concise line per epoch.
# ------------------------------------------------------------

(
    lstm_c1_60_f0_result,
    lstm_c1_60_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_c1_config
    ),

    horizon_minutes=(
        DEEP_ARCHITECTURE_SELECTION_HORIZON
    ),

    feature_family=(
        DEEP_ARCHITECTURE_SELECTION_FAMILY
    ),

    verbose=2,
)


# ------------------------------------------------------------
# 3. Convert result dictionary to a readable table
# ------------------------------------------------------------

lstm_c1_60_f0_result_df = pd.DataFrame(
    [
        lstm_c1_60_f0_result
    ]
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_c1_60_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_c1_60_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_c1_60_f0_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_c1_60_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_c1_60_f0_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_c1_60_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_c1_60_f0_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


assert (
    1
    <= lstm_c1_60_f0_result[
        "best_epoch"
    ]
    <= lstm_c1_60_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


assert np.isfinite(
    lstm_c1_60_f0_result[
        "validation_MAE_kW"
    ]
)


assert np.isfinite(
    lstm_c1_60_f0_result[
        "validation_RMSE_kW"
    ]
)


assert np.isfinite(
    lstm_c1_60_f0_result[
        "validation_R2"
    ]
)


# ------------------------------------------------------------
# 5. Report candidate-level result
# ------------------------------------------------------------

print(
    "\nLSTM architecture-selection run 1/4"
)

print(
    "-----------------------------------"
)


display(
    lstm_c1_60_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Show training history around the selected epoch
# ------------------------------------------------------------

best_epoch = (
    lstm_c1_60_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch - 4,
)


history_window_end = min(
    len(
        lstm_c1_60_f0_history
    ),
    best_epoch + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_c1_60_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Human-readable runtime
# ------------------------------------------------------------

fit_seconds = (
    lstm_c1_60_f0_result[
        "fit_seconds"
    ]
)


print(
    "\nRuntime:"
)

print(
    f"  {fit_seconds:.2f} seconds"
)

print(
    f"  {fit_seconds / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 8. Explicit held-out-test reminder
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 9. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM RNN_C1 60-min F0 validation run: PASSED"
)

Epoch 1/80


2026-09-21 13:36:58.423995: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:36:58.424537: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 8s - 22ms/step - loss: 0.2457 - val_loss: 0.2979
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.2032 - val_loss: 0.2949
Epoch 3/80
341/341 - 4s - 10ms/step - loss: 0.1976 - val_loss: 0.3031
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1963 - val_loss: 0.2942
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1996 - val_loss: 0.3262
Epoch 6/80
341/341 - 4s - 10ms/step - loss: 0.2162 - val_loss: 0.3000
Epoch 7/80
341/341 - 4s - 10ms/step - loss: 0.2121 - val_loss: 0.3005
Epoch 8/80
341/341 - 4s - 10ms/step - loss: 0.2093 - val_loss: 0.3038
Epoch 9/80
341/341 - 4s - 10ms/step - loss: 0.2015 - val_loss: 0.2968
Epoch 10/80
341/341 - 4s - 10ms/step - loss: 0.1979 - val_loss: 0.2931
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.2004 - val_loss: 0.3056
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.2250 - val_loss: 0.3304
Epoch 13/80
341/341 - 4s - 11ms/step - loss: 0.2174 - val_loss: 0.3269
Epoch 14/80
341/341 - 4s - 10ms/step - loss: 0.2210 - val_loss: 0.3239
Epoch 15/80
341/341 - 4s -

2026-09-21 13:38:07.602708: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:38:07.602948: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM architecture-selection run 1/4
-----------------------------------


,model,candidate_id,horizon_minutes,feature_family,trainable_parameter_count,epochs_ran,best_epoch,callback_best_val_loss_scaled,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds
0,LSTM,RNN_C1,60,F0,6497,18,10,0.293078,0.465383,0.869118,0.799001,69.252062



Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
6,7,0.212101,0.300505,False
7,8,0.209312,0.303763,False
8,9,0.201480,0.296767,False
9,10,0.197894,0.293078,True
10,11,0.200434,0.305635,False
11,12,0.225000,0.330366,False
12,13,0.217436,0.326902,False



Runtime:
  69.25 seconds
  1.15 minutes

Held-out test evaluation:
  NOT PERFORMED

LSTM RNN_C1 60-min F0 validation run: PASSED


In [26]:
# ============================================================
# Step 23: LSTM architecture-selection run 2 of 4
#
# Candidate:
#       RNN_C2
#
# Architecture:
#       64 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# Population:
#       60-minute horizon
#       F0 feature family
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       early stopping + architecture evaluation
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Select frozen candidate RNN_C2
# ------------------------------------------------------------

lstm_c2_config = (
    DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ][
        1
    ]
)


assert (
    lstm_c2_config[
        "candidate_id"
    ]
    == "RNN_C2"
)


# ------------------------------------------------------------
# 2. Train candidate on TRAIN and evaluate VALIDATION
# ------------------------------------------------------------

(
    lstm_c2_60_f0_result,
    lstm_c2_60_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_c2_config
    ),

    horizon_minutes=(
        DEEP_ARCHITECTURE_SELECTION_HORIZON
    ),

    feature_family=(
        DEEP_ARCHITECTURE_SELECTION_FAMILY
    ),

    verbose=2,
)


# ------------------------------------------------------------
# 3. Expected parameter count
#
# LSTM:
#
#   4 × 64 × (9 + 64 + 1)
#   = 18,944
#
# Fusion Dense:
#
#   (64 + 1) -> 32
#   = 65 × 32 + 32
#   = 2,112
#
# Output:
#
#   32 -> 1
#   = 33
#
# Total:
#
#   18,944 + 2,112 + 33
#   = 21,089
# ------------------------------------------------------------

EXPECTED_LSTM_C2_F0_PARAMETERS = (
    21_089
)


assert (
    lstm_c2_60_f0_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_C2_F0_PARAMETERS
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_c2_60_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_c2_60_f0_result[
        "candidate_id"
    ]
    == "RNN_C2"
)


assert (
    lstm_c2_60_f0_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_c2_60_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_c2_60_f0_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_c2_60_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_c2_60_f0_result[
        "best_epoch"
    ]
    <= lstm_c2_60_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_c2_60_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 5. Result table
# ------------------------------------------------------------

lstm_c2_60_f0_result_df = pd.DataFrame(
    [
        lstm_c2_60_f0_result
    ]
)


print(
    "\nLSTM architecture-selection run 2/4"
)

print(
    "-----------------------------------"
)


display(
    lstm_c2_60_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Show epochs around the callback-selected best epoch
# ------------------------------------------------------------

best_epoch_c2 = (
    lstm_c2_60_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_c2 - 4,
)


history_window_end = min(
    len(
        lstm_c2_60_f0_history
    ),
    best_epoch_c2 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_c2_60_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Temporary C1-vs-C2 validation comparison
#
# This is NOT the final architecture choice because RNN_C3
# and RNN_C4 have not yet been evaluated.
# ------------------------------------------------------------

lstm_partial_architecture_comparison = pd.DataFrame(
    [
        lstm_c1_60_f0_result,
        lstm_c2_60_f0_result,
    ]
)[
    [
        "candidate_id",
        "trainable_parameter_count",
        "epochs_ran",
        "best_epoch",
        "validation_MAE_kW",
        "validation_RMSE_kW",
        "validation_R2",
        "fit_seconds",
    ]
]


print(
    "\nPartial LSTM architecture comparison"
)

print(
    "------------------------------------"
)

display(
    lstm_partial_architecture_comparison
)


# ------------------------------------------------------------
# 8. Runtime
# ------------------------------------------------------------

fit_seconds_c2 = (
    lstm_c2_60_f0_result[
        "fit_seconds"
    ]
)


print(
    "\nRNN_C2 runtime:"
)

print(
    f"  {fit_seconds_c2:.2f} seconds"
)

print(
    f"  {fit_seconds_c2 / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 9. Test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM RNN_C2 60-min F0 validation run: PASSED"
)

Epoch 1/80


2026-09-21 13:44:03.228138: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:44:03.228563: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.2242 - val_loss: 0.3047
Epoch 2/80
341/341 - 4s - 13ms/step - loss: 0.2054 - val_loss: 0.2934
Epoch 3/80
341/341 - 4s - 12ms/step - loss: 0.2063 - val_loss: 0.3271
Epoch 4/80
341/341 - 4s - 12ms/step - loss: 0.2096 - val_loss: 0.3113
Epoch 5/80
341/341 - 4s - 13ms/step - loss: 0.2081 - val_loss: 0.2987
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.2080 - val_loss: 0.3079
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2028 - val_loss: 0.3021
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.2088 - val_loss: 0.3101
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.2125 - val_loss: 0.2974
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.2055 - val_loss: 0.3012


2026-09-21 13:44:43.806109: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:44:43.806365: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM architecture-selection run 2/4
-----------------------------------


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.224175,0.304733,False
1,2,0.205396,0.293412,True
2,3,0.206287,0.327106,False
3,4,0.209576,0.311294,False
4,5,0.208131,0.298683,False



Partial LSTM architecture comparison
------------------------------------


,candidate_id,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds
0,RNN_C1,6497,18,10,0.465383,0.869118,0.799001,69.252062
1,RNN_C2,21089,10,2,0.465913,0.882267,0.792872,40.599672



RNN_C2 runtime:
  40.60 seconds
  0.68 minutes

Held-out test evaluation:
  NOT PERFORMED

LSTM RNN_C2 60-min F0 validation run: PASSED


In [27]:
# ============================================================
# Step 24: LSTM architecture-selection run 3 of 4
#
# Candidate:
#       RNN_C3
#
# Architecture:
#       64 LSTM units
#       1 recurrent layer
#       dropout = 0.20
#       fusion dense = 32
#
# Population:
#       60-minute horizon
#       F0 feature family
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       early stopping + architecture evaluation
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Select frozen candidate RNN_C3
# ------------------------------------------------------------

lstm_c3_config = (
    DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ][
        2
    ]
)


assert (
    lstm_c3_config[
        "candidate_id"
    ]
    == "RNN_C3"
)


assert np.isclose(
    lstm_c3_config[
        "dropout"
    ],
    0.20,
)


# ------------------------------------------------------------
# 2. Train candidate on TRAIN and evaluate VALIDATION
# ------------------------------------------------------------

(
    lstm_c3_60_f0_result,
    lstm_c3_60_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_c3_config
    ),

    horizon_minutes=(
        DEEP_ARCHITECTURE_SELECTION_HORIZON
    ),

    feature_family=(
        DEEP_ARCHITECTURE_SELECTION_FAMILY
    ),

    verbose=2,
)


# ------------------------------------------------------------
# 3. Parameter-count check
#
# Dropout introduces no trainable parameters.
#
# Therefore RNN_C3 has the same parameter count as RNN_C2:
#
#       21,089
# ------------------------------------------------------------

EXPECTED_LSTM_C3_F0_PARAMETERS = (
    21_089
)


assert (
    lstm_c3_60_f0_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_C3_F0_PARAMETERS
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_c3_60_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_c3_60_f0_result[
        "candidate_id"
    ]
    == "RNN_C3"
)


assert (
    lstm_c3_60_f0_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_c3_60_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_c3_60_f0_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_c3_60_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_c3_60_f0_result[
        "best_epoch"
    ]
    <= lstm_c3_60_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_c3_60_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 5. Candidate-level result
# ------------------------------------------------------------

lstm_c3_60_f0_result_df = pd.DataFrame(
    [
        lstm_c3_60_f0_result
    ]
)


print(
    "\nLSTM architecture-selection run 3/4"
)

print(
    "-----------------------------------"
)


display(
    lstm_c3_60_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Show training history around the callback-selected
#    best epoch
# ------------------------------------------------------------

best_epoch_c3 = (
    lstm_c3_60_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_c3 - 4,
)


history_window_end = min(
    len(
        lstm_c3_60_f0_history
    ),
    best_epoch_c3 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_c3_60_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Provisional C1-C3 comparison
#
# Still NOT the final architecture decision because RNN_C4
# has not yet been evaluated.
# ------------------------------------------------------------

lstm_partial_architecture_comparison = pd.DataFrame(
    [
        lstm_c1_60_f0_result,
        lstm_c2_60_f0_result,
        lstm_c3_60_f0_result,
    ]
)[
    [
        "candidate_id",
        "trainable_parameter_count",
        "epochs_ran",
        "best_epoch",
        "validation_MAE_kW",
        "validation_RMSE_kW",
        "validation_R2",
        "fit_seconds",
    ]
]


# ------------------------------------------------------------
# 8. Sort provisionally using the already-frozen ranking rule
#
#       1. validation MAE
#       2. validation RMSE
#       3. parameter count
#
# This is informational only until C4 is complete.
# ------------------------------------------------------------

lstm_partial_architecture_comparison = (
    lstm_partial_architecture_comparison
    .sort_values(
        by=[
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "trainable_parameter_count",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nProvisional LSTM architecture comparison"
)

print(
    "----------------------------------------"
)


display(
    lstm_partial_architecture_comparison
)


# ------------------------------------------------------------
# 9. Runtime
# ------------------------------------------------------------

fit_seconds_c3 = (
    lstm_c3_60_f0_result[
        "fit_seconds"
    ]
)


print(
    "\nRNN_C3 runtime:"
)

print(
    f"  {fit_seconds_c3:.2f} seconds"
)

print(
    f"  {fit_seconds_c3 / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM RNN_C3 60-min F0 validation run: PASSED"
)

Epoch 1/80


2026-09-21 13:46:07.160484: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:46:07.160749: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 6s - 17ms/step - loss: 0.2607 - val_loss: 0.3758
Epoch 2/80
341/341 - 5s - 14ms/step - loss: 0.2270 - val_loss: 0.3241
Epoch 3/80
341/341 - 4s - 12ms/step - loss: 0.2231 - val_loss: 0.3087
Epoch 4/80
341/341 - 4s - 13ms/step - loss: 0.2274 - val_loss: 0.3344
Epoch 5/80
341/341 - 4s - 13ms/step - loss: 0.2517 - val_loss: 0.3088
Epoch 6/80
341/341 - 4s - 13ms/step - loss: 0.2462 - val_loss: 0.3114
Epoch 7/80
341/341 - 5s - 14ms/step - loss: 0.2413 - val_loss: 0.3221
Epoch 8/80
341/341 - 5s - 14ms/step - loss: 0.2385 - val_loss: 0.3303
Epoch 9/80
341/341 - 4s - 12ms/step - loss: 0.2411 - val_loss: 0.3358
Epoch 10/80
341/341 - 4s - 12ms/step - loss: 0.2343 - val_loss: 0.3130
Epoch 11/80
341/341 - 4s - 12ms/step - loss: 0.2396 - val_loss: 0.3288


2026-09-21 13:46:56.954561: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:46:56.954808: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM architecture-selection run 3/4
-----------------------------------


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.260708,0.375761,False
1,2,0.226964,0.324089,False
2,3,0.223118,0.308659,True
3,4,0.227418,0.334448,False
4,5,0.251720,0.308805,False
5,6,0.246206,0.311352,False



Provisional LSTM architecture comparison
----------------------------------------


,candidate_id,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds
0,RNN_C1,6497,18,10,0.465383,0.869118,0.799001,69.252062
1,RNN_C2,21089,10,2,0.465913,0.882267,0.792872,40.599672
2,RNN_C3,21089,11,3,0.490124,0.903015,0.783016,49.816826



RNN_C3 runtime:
  49.82 seconds
  0.83 minutes

Held-out test evaluation:
  NOT PERFORMED

LSTM RNN_C3 60-min F0 validation run: PASSED


In [28]:
# ============================================================
# Step 25: LSTM architecture-selection run 4 of 4
#
# Candidate:
#       RNN_C4
#
# Architecture:
#       64 LSTM units
#       2 recurrent layers
#       dropout = 0.20
#       fusion dense = 32
#
# Population:
#       60-minute horizon
#       F0 feature family
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       early stopping + architecture evaluation
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Select frozen candidate RNN_C4
# ------------------------------------------------------------

lstm_c4_config = (
    DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ][
        3
    ]
)


assert (
    lstm_c4_config[
        "candidate_id"
    ]
    == "RNN_C4"
)


assert (
    lstm_c4_config[
        "n_recurrent_layers"
    ]
    == 2
)


assert np.isclose(
    lstm_c4_config[
        "dropout"
    ],
    0.20,
)


# ------------------------------------------------------------
# 2. Train candidate on TRAIN and evaluate VALIDATION
# ------------------------------------------------------------

(
    lstm_c4_60_f0_result,
    lstm_c4_60_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_c4_config
    ),

    horizon_minutes=(
        DEEP_ARCHITECTURE_SELECTION_HORIZON
    ),

    feature_family=(
        DEEP_ARCHITECTURE_SELECTION_FAMILY
    ),

    verbose=2,
)


# ------------------------------------------------------------
# 3. Expected parameter count
#
# First LSTM:
#
#   input = 9
#   units = 64
#
#   4 × 64 × (9 + 64 + 1)
#   = 18,944
#
#
# Second LSTM:
#
#   input = 64
#   units = 64
#
#   4 × 64 × (64 + 64 + 1)
#   = 33,024
#
#
# Fusion Dense:
#
#   (64 sequence + 1 auxiliary) -> 32
#
#   = 65 × 32 + 32
#   = 2,112
#
#
# Output:
#
#   32 -> 1
#   = 33
#
#
# Total:
#
#   18,944
# + 33,024
# +  2,112
# +     33
# ----------
#   54,113
# ------------------------------------------------------------

EXPECTED_LSTM_C4_F0_PARAMETERS = (
    54_113
)


assert (
    lstm_c4_60_f0_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_C4_F0_PARAMETERS
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_c4_60_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_c4_60_f0_result[
        "candidate_id"
    ]
    == "RNN_C4"
)


assert (
    lstm_c4_60_f0_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_c4_60_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_c4_60_f0_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_c4_60_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_c4_60_f0_result[
        "best_epoch"
    ]
    <= lstm_c4_60_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_c4_60_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 5. Candidate-level result table
# ------------------------------------------------------------

lstm_c4_60_f0_result_df = pd.DataFrame(
    [
        lstm_c4_60_f0_result
    ]
)


print(
    "\nLSTM architecture-selection run 4/4"
)

print(
    "-----------------------------------"
)


display(
    lstm_c4_60_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_c4 = (
    lstm_c4_60_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_c4 - 4,
)


history_window_end = min(
    len(
        lstm_c4_60_f0_history
    ),
    best_epoch_c4 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_c4_60_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Complete four-candidate LSTM comparison
#
# We display the candidates according to the already-frozen
# architecture-selection rule:
#
#       1. lowest validation MAE
#       2. lowest validation RMSE
#       3. fewer trainable parameters
#
# We are not yet creating the final frozen-selection object in
# this cell; that will be done explicitly in the next step.
# ------------------------------------------------------------

lstm_architecture_comparison = pd.DataFrame(
    [
        lstm_c1_60_f0_result,
        lstm_c2_60_f0_result,
        lstm_c3_60_f0_result,
        lstm_c4_60_f0_result,
    ]
)[
    [
        "candidate_id",
        "trainable_parameter_count",
        "epochs_ran",
        "best_epoch",
        "validation_MAE_kW",
        "validation_RMSE_kW",
        "validation_R2",
        "fit_seconds",
    ]
]


lstm_architecture_comparison = (
    lstm_architecture_comparison
    .sort_values(
        by=[
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "trainable_parameter_count",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nComplete LSTM architecture comparison"
)

print(
    "-------------------------------------"
)


display(
    lstm_architecture_comparison
)


# ------------------------------------------------------------
# 8. Runtime
# ------------------------------------------------------------

fit_seconds_c4 = (
    lstm_c4_60_f0_result[
        "fit_seconds"
    ]
)


print(
    "\nRNN_C4 runtime:"
)

print(
    f"  {fit_seconds_c4:.2f} seconds"
)

print(
    f"  {fit_seconds_c4 / 60:.2f} minutes"
)


# ------------------------------------------------------------
# 9. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 10. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM RNN_C4 60-min F0 validation run: PASSED"
)

Epoch 1/80


2026-09-21 13:48:09.186448: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:48:09.186746: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 8s - 25ms/step - loss: 0.2783 - val_loss: 0.3764
Epoch 2/80
341/341 - 6s - 18ms/step - loss: 0.2425 - val_loss: 0.3085
Epoch 3/80
341/341 - 6s - 19ms/step - loss: 0.2398 - val_loss: 0.3112
Epoch 4/80
341/341 - 7s - 19ms/step - loss: 0.2357 - val_loss: 0.3128
Epoch 5/80
341/341 - 6s - 18ms/step - loss: 0.2308 - val_loss: 0.3074
Epoch 6/80
341/341 - 6s - 18ms/step - loss: 0.2211 - val_loss: 0.3166
Epoch 7/80
341/341 - 6s - 17ms/step - loss: 0.2183 - val_loss: 0.3285
Epoch 8/80
341/341 - 6s - 17ms/step - loss: 0.2385 - val_loss: 0.3156
Epoch 9/80
341/341 - 7s - 20ms/step - loss: 0.2422 - val_loss: 0.3526
Epoch 10/80
341/341 - 6s - 18ms/step - loss: 0.2435 - val_loss: 0.3229
Epoch 11/80
341/341 - 6s - 17ms/step - loss: 0.2393 - val_loss: 0.3239
Epoch 12/80
341/341 - 6s - 17ms/step - loss: 0.2678 - val_loss: 0.3333
Epoch 13/80
341/341 - 6s - 17ms/step - loss: 0.2590 - val_loss: 0.3466


2026-09-21 13:49:30.659589: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:49:30.659862: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM architecture-selection run 4/4
-----------------------------------


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
1,2,0.242517,0.308466,False
2,3,0.239833,0.311214,False
3,4,0.235745,0.312839,False
4,5,0.230757,0.307383,True
5,6,0.221100,0.316589,False
6,7,0.218279,0.328480,False
7,8,0.238483,0.315556,False



Complete LSTM architecture comparison
-------------------------------------


,candidate_id,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds
0,RNN_C1,6497,18,10,0.465383,0.869118,0.799001,69.252062
1,RNN_C2,21089,10,2,0.465913,0.882267,0.792872,40.599672
2,RNN_C4,54113,13,5,0.488097,0.869817,0.798677,81.483918
3,RNN_C3,21089,11,3,0.490124,0.903015,0.783016,49.816826



RNN_C4 runtime:
  81.48 seconds
  1.36 minutes

Held-out test evaluation:
  NOT PERFORMED

LSTM RNN_C4 60-min F0 validation run: PASSED


In [29]:
# ============================================================
# Step 26: Formally freeze the selected LSTM architecture
# ============================================================
#
# All four predeclared LSTM candidates have now been evaluated
# using ONLY:
#
#       horizon = 60 minutes
#       family  = F0
#
# and ONLY:
#
#       TRAIN      -> fitting
#       VALIDATION -> architecture selection
#
# The TEST population has not been accessed.
#
#
# Frozen architecture-selection rule
# ----------------------------------
#
# Sort by:
#
#   1. lowest validation MAE in kW
#   2. lowest validation RMSE in kW
#   3. fewer trainable parameters
#
# No manual override is permitted.
# ============================================================


# ------------------------------------------------------------
# 1. Collect the four completed candidate results
# ------------------------------------------------------------

lstm_architecture_results = pd.DataFrame(
    [
        lstm_c1_60_f0_result,
        lstm_c2_60_f0_result,
        lstm_c3_60_f0_result,
        lstm_c4_60_f0_result,
    ]
)


# ------------------------------------------------------------
# 2. Strong completeness checks
# ------------------------------------------------------------

assert len(
    lstm_architecture_results
) == 4


assert set(
    lstm_architecture_results[
        "candidate_id"
    ]
) == {
    "RNN_C1",
    "RNN_C2",
    "RNN_C3",
    "RNN_C4",
}


assert set(
    lstm_architecture_results[
        "model"
    ]
) == {
    "LSTM"
}


assert set(
    lstm_architecture_results[
        "horizon_minutes"
    ]
) == {
    60
}


assert set(
    lstm_architecture_results[
        "feature_family"
    ]
) == {
    "F0"
}


# ------------------------------------------------------------
# 3. Apply the PREDECLARED ranking rule exactly
# ------------------------------------------------------------

lstm_architecture_results_ranked = (
    lstm_architecture_results
    .sort_values(
        by=[
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "trainable_parameter_count",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


lstm_architecture_results_ranked.insert(
    0,
    "selection_rank",
    np.arange(
        1,
        len(
            lstm_architecture_results_ranked
        )
        + 1,
        dtype=int,
    ),
)


# ------------------------------------------------------------
# 4. Identify the mechanically selected candidate
# ------------------------------------------------------------

lstm_selected_candidate_id = (
    lstm_architecture_results_ranked
    .iloc[
        0
    ][
        "candidate_id"
    ]
)


assert (
    lstm_selected_candidate_id
    == "RNN_C1"
)


# ------------------------------------------------------------
# 5. Recover the immutable frozen candidate configuration
#    directly from the ORIGINAL candidate grid.
# ------------------------------------------------------------

lstm_selected_candidate_config = next(
    candidate.copy()
    for candidate
    in DEEP_ARCHITECTURE_CANDIDATES[
        "LSTM"
    ]
    if candidate[
        "candidate_id"
    ]
    == lstm_selected_candidate_id
)


# ------------------------------------------------------------
# 6. Verify the exact selected architecture
# ------------------------------------------------------------

assert lstm_selected_candidate_config == {
    "candidate_id":
        "RNN_C1",

    "units":
        32,

    "n_recurrent_layers":
        1,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,
}


# ------------------------------------------------------------
# 7. Extract the selected validation result
# ------------------------------------------------------------

lstm_selected_architecture_result = (
    lstm_architecture_results_ranked
    .iloc[
        0
    ]
    .to_dict()
)


assert np.isclose(
    lstm_selected_architecture_result[
        "validation_MAE_kW"
    ],
    lstm_c1_60_f0_result[
        "validation_MAE_kW"
    ],
)


assert (
    lstm_selected_architecture_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


# ------------------------------------------------------------
# 8. Preserve all four architecture-selection histories
#
# These are useful later for:
#
#   - experiment auditing;
#   - plotting learning curves;
#   - persistence to results/.
# ------------------------------------------------------------

lstm_architecture_selection_history = pd.concat(
    [
        lstm_c1_60_f0_history,
        lstm_c2_60_f0_history,
        lstm_c3_60_f0_history,
        lstm_c4_60_f0_history,
    ],
    axis=0,
    ignore_index=True,
)


assert set(
    lstm_architecture_selection_history[
        "candidate_id"
    ]
) == {
    "RNN_C1",
    "RNN_C2",
    "RNN_C3",
    "RNN_C4",
}


# ------------------------------------------------------------
# 9. Mark selection explicitly
# ------------------------------------------------------------

lstm_architecture_results_ranked[
    "selected_architecture"
] = (
    lstm_architecture_results_ranked[
        "candidate_id"
    ]
    == lstm_selected_candidate_id
)


assert (
    lstm_architecture_results_ranked[
        "selected_architecture"
    ].sum()
    == 1
)


# ------------------------------------------------------------
# 10. Compute total architecture-search runtime
# ------------------------------------------------------------

lstm_architecture_search_seconds = float(
    lstm_architecture_results[
        "fit_seconds"
    ].sum()
)


# ------------------------------------------------------------
# 11. Report complete frozen selection
# ------------------------------------------------------------

print(
    "Frozen LSTM architecture selection"
)

print(
    "----------------------------------"
)


display(
    lstm_architecture_results_ranked[
        [
            "selection_rank",
            "candidate_id",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "selected_architecture",
        ]
    ]
)


print(
    "\nSelected LSTM architecture:"
)

print(
    "  Candidate:",
    lstm_selected_candidate_id,
)

print(
    "  Units:",
    lstm_selected_candidate_config[
        "units"
    ],
)

print(
    "  Recurrent layers:",
    lstm_selected_candidate_config[
        "n_recurrent_layers"
    ],
)

print(
    "  Dropout:",
    lstm_selected_candidate_config[
        "dropout"
    ],
)

print(
    "  Fusion dense units:",
    lstm_selected_candidate_config[
        "fusion_dense_units"
    ],
)

print(
    "  Parameters:",
    int(
        lstm_selected_architecture_result[
            "trainable_parameter_count"
        ]
    ),
)


print(
    "\nSelection-population validation metrics:"
)

print(
    "  MAE:",
    f"{lstm_selected_architecture_result['validation_MAE_kW']:.6f} kW",
)

print(
    "  RMSE:",
    f"{lstm_selected_architecture_result['validation_RMSE_kW']:.6f} kW",
)

print(
    "  R²:",
    f"{lstm_selected_architecture_result['validation_R2']:.6f}",
)


print(
    "\nTotal four-candidate search runtime:"
)

print(
    f"  {lstm_architecture_search_seconds:.2f} seconds"
)

print(
    f"  {lstm_architecture_search_seconds / 60:.2f} minutes"
)


print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM architecture frozen: PASSED"
)

Frozen LSTM architecture selection
----------------------------------


,selection_rank,candidate_id,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds,selected_architecture
0,1,RNN_C1,6497,18,10,0.465383,0.869118,0.799001,69.252062,True
1,2,RNN_C2,21089,10,2,0.465913,0.882267,0.792872,40.599672,False
2,3,RNN_C4,54113,13,5,0.488097,0.869817,0.798677,81.483918,False
3,4,RNN_C3,21089,11,3,0.490124,0.903015,0.783016,49.816826,False



Selected LSTM architecture:
  Candidate: RNN_C1
  Units: 32
  Recurrent layers: 1
  Dropout: 0.0
  Fusion dense units: 32
  Parameters: 6497

Selection-population validation metrics:
  MAE: 0.465383 kW
  RMSE: 0.869118 kW
  R²: 0.799001

Total four-candidate search runtime:
  241.15 seconds
  4.02 minutes

Held-out test evaluation:
  NOT PERFORMED

LSTM architecture frozen: PASSED


In [30]:
# ============================================================
# Step 27: Begin LSTM horizon × feature-family epoch selection
# ============================================================
#
# The LSTM architecture is now permanently frozen as:
#
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# Architecture hyperparameters will NOT change for:
#
#       horizons: 15, 30, 60 minutes
#       families: F0, F1, F2, F3, F4
#
#
# Stage-2 purpose
# ---------------
#
# For each horizon × family combination:
#
#       TRAIN      -> fit frozen architecture
#       VALIDATION -> select best epoch
#       TEST       -> untouched
#
# Later, after all 15 best epochs are frozen, each model will
# be reinitialized and refitted on TRAIN + VALIDATION for
# exactly its selected epoch count.
#
#
# IMPORTANT:
#
# The already-completed 60-min F0 RNN_C1 run is identical to
# one required Stage-2 experiment, so we reuse it rather than
# training it again.
# ============================================================


# ------------------------------------------------------------
# 1. Initialize Stage-2 result/history containers
#
# Keys:
#
#       (horizon_minutes, feature_family)
# ------------------------------------------------------------

lstm_epoch_selection_results = {
    (
        60,
        "F0",
    ):
        lstm_c1_60_f0_result.copy(),
}


lstm_epoch_selection_histories = {
    (
        60,
        "F0",
    ):
        lstm_c1_60_f0_history.copy(),
}


# ------------------------------------------------------------
# 2. Mark the reused 60-min F0 result for experiment auditing
# ------------------------------------------------------------

lstm_epoch_selection_results[
    (
        60,
        "F0",
    )
][
    "epoch_selection_source"
] = (
    "reused_architecture_selection_run"
)


# ------------------------------------------------------------
# 3. Confirm frozen architecture
# ------------------------------------------------------------

assert (
    lstm_selected_candidate_id
    == "RNN_C1"
)


assert lstm_selected_candidate_config == {
    "candidate_id":
        "RNN_C1",

    "units":
        32,

    "n_recurrent_layers":
        1,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,
}


# ------------------------------------------------------------
# 4. Confirm the reused 60-min F0 run is valid
# ------------------------------------------------------------

reused_60_f0 = (
    lstm_epoch_selection_results[
        (
            60,
            "F0",
        )
    ]
)


assert (
    reused_60_f0[
        "candidate_id"
    ]
    == lstm_selected_candidate_id
)


assert (
    reused_60_f0[
        "horizon_minutes"
    ]
    == 60
)


assert (
    reused_60_f0[
        "feature_family"
    ]
    == "F0"
)


assert (
    reused_60_f0[
        "best_epoch"
    ]
    == 10
)


# ============================================================
# 5. RUN STAGE-2 COMBINATION:
#
#       horizon = 15 minutes
#       family  = F0
# ============================================================

(
    lstm_15_f0_result,
    lstm_15_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=15,

    feature_family="F0",

    verbose=2,
)


# ------------------------------------------------------------
# 6. Strong frozen-architecture checks
# ------------------------------------------------------------

assert (
    lstm_15_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_15_f0_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_15_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_15_f0_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


assert (
    lstm_15_f0_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_15_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_15_f0_result[
        "best_epoch"
    ]
    <= lstm_15_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


# ------------------------------------------------------------
# 7. Label source and store permanently in Stage-2 containers
# ------------------------------------------------------------

lstm_15_f0_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        15,
        "F0",
    )
] = (
    lstm_15_f0_result.copy()
)


lstm_epoch_selection_histories[
    (
        15,
        "F0",
    )
] = (
    lstm_15_f0_history.copy()
)


# ------------------------------------------------------------
# 8. Verify that we now have exactly two of the 15 required
#    Stage-2 combinations:
#
#       15-min F0 -> newly trained
#       60-min F0 -> reused
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (
        15,
        "F0",
    ),
    (
        60,
        "F0",
    ),
}


# ------------------------------------------------------------
# 9. Display the new 15-min F0 result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 15 min"
)

print(
    "  Family:  F0"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_15_f0_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Show history around selected epoch
# ------------------------------------------------------------

best_epoch_15_f0 = (
    lstm_15_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_15_f0 - 4,
)


history_window_end = min(
    len(
        lstm_15_f0_history
    ),
    best_epoch_15_f0 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_15_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Current Stage-2 progress table
# ------------------------------------------------------------

lstm_epoch_selection_progress = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "validation_MAE_kW":
                result[
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                result[
                    "validation_RMSE_kW"
                ],

            "epoch_selection_source":
                result[
                    "epoch_selection_source"
                ],
        }

        for (
            horizon_minutes,
            feature_family,
        ), result
        in lstm_epoch_selection_results.items()
    ]
)


lstm_epoch_selection_progress = (
    lstm_epoch_selection_progress
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent LSTM Stage-2 progress:"
)

display(
    lstm_epoch_selection_progress
)


print(
    "\nCompleted combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 12. Test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 13. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 15-min F0 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 13:56:33.776447: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:56:33.776934: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 6s - 18ms/step - loss: 0.1671 - val_loss: 0.1906
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.1290 - val_loss: 0.1894
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1339 - val_loss: 0.2012
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1270 - val_loss: 0.2024
Epoch 5/80
341/341 - 4s - 12ms/step - loss: 0.1237 - val_loss: 0.1859
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1219 - val_loss: 0.1844
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1213 - val_loss: 0.1822
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1213 - val_loss: 0.1838
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1244 - val_loss: 0.2022
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.1271 - val_loss: 0.1913
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.1302 - val_loss: 0.2020
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.1299 - val_loss: 0.1985
Epoch 13/80
341/341 - 4s - 11ms/step - loss: 0.1331 - val_loss: 0.2063
Epoch 14/80
341/341 - 4s - 11ms/step - loss: 0.1369 - val_loss: 0.2045
Epoch 15/80
341/341 - 4s -

2026-09-21 13:57:32.243884: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:57:32.244138: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 15 min
  Family:  F0
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
3,4,0.126986,0.202366,False
4,5,0.123704,0.185911,False
5,6,0.121931,0.184391,False
6,7,0.121272,0.182242,True
7,8,0.121338,0.183806,False
8,9,0.124356,0.202232,False
9,10,0.127127,0.191317,False



Current LSTM Stage-2 progress:


,horizon_minutes,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,epoch_selection_source
0,15,F0,7,0.289374,0.650396,stage2_validation_run
1,60,F0,10,0.465383,0.869118,reused_architecture_selection_run



Completed combinations: 2 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 15-min F0 epoch-selection run: PASSED


In [31]:
# ============================================================
# Step 28: LSTM Stage-2 epoch selection
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F1
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# Only the auxiliary feature family changes:
#
#       F0 auxiliary width =  1
#       F1 auxiliary width = 16
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen architecture on 15-minute F1
# ------------------------------------------------------------

(
    lstm_15_f1_result,
    lstm_15_f1_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=15,

    feature_family="F1",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376 parameters
#
# Fusion Dense:
#
#       sequence embedding = 32
#       F1 auxiliary       = 16
#
#       total fusion input = 48
#
#       48 × 32 + 32
#       = 1,568
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 1,568 + 33
#       = 6,977
# ------------------------------------------------------------

EXPECTED_LSTM_F1_PARAMETERS = (
    6_977
)


assert (
    lstm_15_f1_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F1_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_15_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_15_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_15_f1_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_15_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_15_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_15_f1_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_15_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_15_f1_result[
        "best_epoch"
    ]
    <= lstm_15_f1_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_15_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_15_f1_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        15,
        "F1",
    )
] = (
    lstm_15_f1_result.copy()
)


lstm_epoch_selection_histories[
    (
        15,
        "F1",
    )
] = (
    lstm_15_f1_history.copy()
)


# ------------------------------------------------------------
# 5. Verify expected Stage-2 progress
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (
        15,
        "F0",
    ),
    (
        15,
        "F1",
    ),
    (
        60,
        "F0",
    ),
}


# ------------------------------------------------------------
# 6. Display new F1 result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 15 min"
)

print(
    "  Family:  F1"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_15_f1_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected epoch
# ------------------------------------------------------------

best_epoch_15_f1 = (
    lstm_15_f1_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_15_f1 - 4,
)


history_window_end = min(
    len(
        lstm_15_f1_history
    ),
    best_epoch_15_f1 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_15_f1_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F0 -> F1 incremental comparison
#
# Positive percentage means F1 has lower validation MAE.
#
# This is NOT a test-set conclusion.
# ------------------------------------------------------------

f0_validation_mae = (
    lstm_epoch_selection_results[
        (
            15,
            "F0",
        )
    ][
        "validation_MAE_kW"
    ]
)


f1_validation_mae = (
    lstm_15_f1_result[
        "validation_MAE_kW"
    ]
)


f0_to_f1_validation_improvement_pct = (
    (
        f0_validation_mae
        - f1_validation_mae
    )
    /
    f0_validation_mae
    * 100.0
)


print(
    "\n15-min validation F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_validation_mae:.6f} kW"
)

print(
    f"  F1 MAE: {f1_validation_mae:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_validation_improvement_pct:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current Stage-2 progress table
# ------------------------------------------------------------

lstm_epoch_selection_progress = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "validation_MAE_kW":
                result[
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                result[
                    "validation_RMSE_kW"
                ],

            "epoch_selection_source":
                result[
                    "epoch_selection_source"
                ],
        }

        for (
            horizon_minutes,
            feature_family,
        ), result
        in lstm_epoch_selection_results.items()
    ]
)


lstm_epoch_selection_progress = (
    lstm_epoch_selection_progress
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent LSTM Stage-2 progress:"
)


display(
    lstm_epoch_selection_progress
)


print(
    "\nCompleted combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 15-min F1 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 13:58:42.010702: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 13:58:42.010982: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.1609 - val_loss: 0.1833
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.1244 - val_loss: 0.1853
Epoch 3/80
341/341 - 3s - 10ms/step - loss: 0.1284 - val_loss: 0.2018
Epoch 4/80
341/341 - 4s - 12ms/step - loss: 0.1360 - val_loss: 0.2051
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1352 - val_loss: 0.1948
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1356 - val_loss: 0.2218
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1379 - val_loss: 0.2104
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1453 - val_loss: 0.2089
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1634 - val_loss: 0.2811


2026-09-21 13:59:16.540537: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 13:59:16.540780: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 15 min
  Family:  F1
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.160905,0.183347,True
1,2,0.124424,0.185330,False
2,3,0.128410,0.201834,False
3,4,0.136039,0.205079,False



15-min validation F0 -> F1 comparison:
  F0 MAE: 0.289374 kW
  F1 MAE: 0.291128 kW
  Relative MAE change: -0.6064%
  Validation only — not a test-set result.

Current LSTM Stage-2 progress:


,horizon_minutes,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,epoch_selection_source
0,15,F0,7,0.289374,0.650396,stage2_validation_run
1,15,F1,1,0.291128,0.638846,stage2_validation_run
2,60,F0,10,0.465383,0.869118,reused_architecture_selection_run



Completed combinations: 3 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 15-min F1 epoch-selection run: PASSED


In [32]:
# ============================================================
# Step 29: LSTM Stage-2 epoch selection
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F2
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# F2 adds the five independently computed solar features:
#
#       solar_elevation_deg
#       solar_cos_zenith
#       solar_azimuth_sin
#       solar_azimuth_cos
#       solar_daylight
#
# Auxiliary width:
#
#       F1 = 16
#       F2 = 21
#
# TRAIN:
#       fitting
#
# VALIDATION:
#       epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen architecture on 15-minute F2
# ------------------------------------------------------------

(
    lstm_15_f2_result,
    lstm_15_f2_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=15,

    feature_family="F2",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F2 auxiliary       = 21
#
#       total              = 53
#
# Fusion Dense:
#
#       53 × 32 + 32
#       = 1,728
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 1,728 + 33
#       = 7,137
# ------------------------------------------------------------

EXPECTED_LSTM_F2_PARAMETERS = (
    7_137
)


assert (
    lstm_15_f2_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F2_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_15_f2_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_15_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_15_f2_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_15_f2_result[
        "feature_family"
    ]
    == "F2"
)


assert (
    lstm_15_f2_result[
        "auxiliary_dim"
    ]
    == 21
)


assert (
    lstm_15_f2_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_15_f2_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_15_f2_result[
        "best_epoch"
    ]
    <= lstm_15_f2_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_15_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_15_f2_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        15,
        "F2",
    )
] = (
    lstm_15_f2_result.copy()
)


lstm_epoch_selection_histories[
    (
        15,
        "F2",
    )
] = (
    lstm_15_f2_history.copy()
)


# ------------------------------------------------------------
# 5. Verify expected Stage-2 progress
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 15 min"
)

print(
    "  Family:  F2"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_15_f2_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_15_f2 = (
    lstm_15_f2_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_15_f2 - 4,
)


history_window_end = min(
    len(
        lstm_15_f2_history
    ),
    best_epoch_15_f2 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_15_f2_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F1 -> F2 incremental comparison
#
# Positive percentage means F2 has lower validation MAE.
# ------------------------------------------------------------

f1_validation_mae = (
    lstm_epoch_selection_results[
        (
            15,
            "F1",
        )
    ][
        "validation_MAE_kW"
    ]
)


f2_validation_mae = (
    lstm_15_f2_result[
        "validation_MAE_kW"
    ]
)


f1_to_f2_validation_improvement_pct = (
    (
        f1_validation_mae
        - f2_validation_mae
    )
    /
    f1_validation_mae
    * 100.0
)


print(
    "\n15-min validation F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_validation_mae:.6f} kW"
)

print(
    f"  F2 MAE: {f2_validation_mae:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_validation_improvement_pct:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current Stage-2 progress table
# ------------------------------------------------------------

lstm_epoch_selection_progress = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "validation_MAE_kW":
                result[
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                result[
                    "validation_RMSE_kW"
                ],

            "epoch_selection_source":
                result[
                    "epoch_selection_source"
                ],
        }

        for (
            horizon_minutes,
            feature_family,
        ), result
        in lstm_epoch_selection_results.items()
    ]
)


lstm_epoch_selection_progress = (
    lstm_epoch_selection_progress
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent LSTM Stage-2 progress:"
)


display(
    lstm_epoch_selection_progress
)


print(
    "\nCompleted combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 15-min F2 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 14:00:40.937532: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 14:00:40.937880: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.1846 - val_loss: 0.1880
Epoch 2/80
341/341 - 4s - 12ms/step - loss: 0.1291 - val_loss: 0.1871
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1293 - val_loss: 0.1885
Epoch 4/80
341/341 - 4s - 12ms/step - loss: 0.1293 - val_loss: 0.1956
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1291 - val_loss: 0.2026
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1362 - val_loss: 0.2109
Epoch 7/80
341/341 - 3s - 10ms/step - loss: 0.1386 - val_loss: 0.2035
Epoch 8/80
341/341 - 3s - 10ms/step - loss: 0.1360 - val_loss: 0.1969
Epoch 9/80
341/341 - 3s - 10ms/step - loss: 0.1308 - val_loss: 0.1888
Epoch 10/80
341/341 - 3s - 10ms/step - loss: 0.1334 - val_loss: 0.1913


2026-09-21 14:01:18.907555: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 14:01:18.907793: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 15 min
  Family:  F2
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.184603,0.187954,False
1,2,0.129117,0.187102,True
2,3,0.129302,0.188460,False
3,4,0.129302,0.195603,False
4,5,0.129059,0.202590,False



15-min validation F1 -> F2 comparison:
  F1 MAE: 0.291128 kW
  F2 MAE: 0.297091 kW
  Relative MAE change: -2.0483%
  Validation only — not a test-set result.

Current LSTM Stage-2 progress:


,horizon_minutes,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,epoch_selection_source
0,15,F0,7,0.289374,0.650396,stage2_validation_run
1,15,F1,1,0.291128,0.638846,stage2_validation_run
2,15,F2,2,0.297091,0.649842,stage2_validation_run
3,60,F0,10,0.465383,0.869118,reused_architecture_selection_run



Completed combinations: 4 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 15-min F2 epoch-selection run: PASSED


In [33]:
# ============================================================
# Step 30: LSTM Stage-2 epoch selection
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F3
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F3 extends F2 with nine continuous astronomical controls:
#
#       moon_phase_sin
#       moon_phase_cos
#       moon_longitude_sin
#       moon_longitude_cos
#       sun_ecliptic_longitude_sin
#       sun_ecliptic_longitude_cos
#       moon_ecliptic_latitude_deg
#       moon_illumination_fraction
#       earth_moon_distance_km
#
#
# Auxiliary width:
#
#       F2 = 21
#       F3 = 30
#
#
# This is especially important scientifically because the later
# F3 -> F4 comparison asks whether categorical Panchang
# representation adds predictive information beyond the
# underlying continuous Sun-Moon astronomical quantities.
#
#
# TRAIN:
#       fitting
#
# VALIDATION:
#       epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 15-minute F3
# ------------------------------------------------------------

(
    lstm_15_f3_result,
    lstm_15_f3_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=15,

    feature_family="F3",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F3 auxiliary       = 30
#
#       total              = 62
#
# Fusion Dense:
#
#       62 × 32 + 32
#       = 2,016
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 2,016 + 33
#       = 7,425
# ------------------------------------------------------------

EXPECTED_LSTM_F3_PARAMETERS = (
    7_425
)


assert (
    lstm_15_f3_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F3_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_15_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_15_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_15_f3_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_15_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_15_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_15_f3_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_15_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_15_f3_result[
        "best_epoch"
    ]
    <= lstm_15_f3_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_15_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_15_f3_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        15,
        "F3",
    )
] = (
    lstm_15_f3_result.copy()
)


lstm_epoch_selection_histories[
    (
        15,
        "F3",
    )
] = (
    lstm_15_f3_history.copy()
)


# ------------------------------------------------------------
# 5. Verify exact current Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 15 min"
)

print(
    "  Family:  F3"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_15_f3_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_15_f3 = (
    lstm_15_f3_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_15_f3 - 4,
)


history_window_end = min(
    len(
        lstm_15_f3_history
    ),
    best_epoch_15_f3 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_15_f3_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F2 -> F3 incremental comparison
#
# Positive percentage:
#       F3 has lower validation MAE.
#
# Negative percentage:
#       F3 has higher validation MAE.
#
# This remains validation-only evidence.
# ------------------------------------------------------------

f2_validation_mae = (
    lstm_epoch_selection_results[
        (
            15,
            "F2",
        )
    ][
        "validation_MAE_kW"
    ]
)


f3_validation_mae = (
    lstm_15_f3_result[
        "validation_MAE_kW"
    ]
)


f2_to_f3_validation_improvement_pct = (
    (
        f2_validation_mae
        - f3_validation_mae
    )
    /
    f2_validation_mae
    * 100.0
)


print(
    "\n15-min validation F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_validation_mae:.6f} kW"
)

print(
    f"  F3 MAE: {f3_validation_mae:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_validation_improvement_pct:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Show complete 15-minute validation progression so far
# ------------------------------------------------------------

lstm_15min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\n15-minute LSTM validation progression:"
)


display(
    lstm_15min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 15-min F3 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 14:19:07.465544: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 14:19:07.465879: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 6s - 16ms/step - loss: 0.1915 - val_loss: 0.1953
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.1306 - val_loss: 0.1841
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1300 - val_loss: 0.1906
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1309 - val_loss: 0.1893
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1339 - val_loss: 0.2074
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1432 - val_loss: 0.2098
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1389 - val_loss: 0.2004
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1404 - val_loss: 0.2096
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1359 - val_loss: 0.1982
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.1354 - val_loss: 0.1970


2026-09-21 14:19:45.624118: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 14:19:45.624362: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 15 min
  Family:  F3
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.191466,0.195330,False
1,2,0.130582,0.184094,True
2,3,0.129977,0.190636,False
3,4,0.130878,0.189254,False
4,5,0.133889,0.207413,False



15-min validation F2 -> F3 comparison:
  F2 MAE: 0.297091 kW
  F3 MAE: 0.292315 kW
  Relative MAE change: +1.6078%
  Validation only — not a test-set result.

15-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,7,0.289374,0.650396,0.887437
1,F1,1,0.291128,0.638846,0.891400
2,F2,2,0.297091,0.649842,0.887629
3,F3,2,0.292315,0.641045,0.890651



Completed Stage-2 combinations: 5 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 15-min F3 epoch-selection run: PASSED


In [34]:
# ============================================================
# Step 31: LSTM Stage-2 epoch selection
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F4
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F4 extends F3 with 97 canonical Panchang one-hot features:
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 total
#
#
# Auxiliary width:
#
#       F3 =  30
#       F4 = 127
#
#
# Scientific question:
#
# Does the Panchang representation add predictive information
# beyond the continuous astronomical controls already present
# in F3?
#
#
# TRAIN:
#       fitting
#
# VALIDATION:
#       epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 15-minute F4
# ------------------------------------------------------------

(
    lstm_15_f4_result,
    lstm_15_f4_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=15,

    feature_family="F4",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding =  32
#       F4 auxiliary       = 127
#
#       total              = 159
#
# Fusion Dense:
#
#       159 × 32 + 32
#       = 5,120
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 5,120 + 33
#       = 10,529
# ------------------------------------------------------------

EXPECTED_LSTM_F4_PARAMETERS = (
    10_529
)


assert (
    lstm_15_f4_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F4_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_15_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_15_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_15_f4_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_15_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_15_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_15_f4_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_15_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_15_f4_result[
        "best_epoch"
    ]
    <= lstm_15_f4_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_15_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_15_f4_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        15,
        "F4",
    )
] = (
    lstm_15_f4_result.copy()
)


lstm_epoch_selection_histories[
    (
        15,
        "F4",
    )
] = (
    lstm_15_f4_history.copy()
)


# ------------------------------------------------------------
# 5. Verify that the complete 15-minute F0-F4 family is now
#    available, together with the reused 60-minute F0 result.
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display the new F4 result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 15 min"
)

print(
    "  Family:  F4"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_15_f4_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_15_f4 = (
    lstm_15_f4_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_15_f4 - 4,
)


history_window_end = min(
    len(
        lstm_15_f4_history
    ),
    best_epoch_15_f4 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_15_f4_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. KEY VALIDATION COMPARISON:
#
#       F3 -> F4
#
# Positive percentage:
#       F4 has lower validation MAE.
#
# Negative percentage:
#       F4 has higher validation MAE.
#
# This is still VALIDATION ONLY.
# ------------------------------------------------------------

f3_validation_mae = (
    lstm_epoch_selection_results[
        (
            15,
            "F3",
        )
    ][
        "validation_MAE_kW"
    ]
)


f4_validation_mae = (
    lstm_15_f4_result[
        "validation_MAE_kW"
    ]
)


f3_to_f4_validation_improvement_pct = (
    (
        f3_validation_mae
        - f4_validation_mae
    )
    /
    f3_validation_mae
    * 100.0
)


print(
    "\n15-min validation F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_validation_mae:.6f} kW"
)

print(
    f"  F4 MAE: {f4_validation_mae:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_validation_improvement_pct:+.4f}%",
)

print(
    "  Validation only — not a held-out test conclusion."
)


# ------------------------------------------------------------
# 9. Complete 15-minute F0-F4 validation progression
# ------------------------------------------------------------

lstm_15min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "auxiliary_dim":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "auxiliary_dim"
                ],

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 15-minute LSTM validation progression:"
)


display(
    lstm_15min_validation_progress
)


# ------------------------------------------------------------
# 10. Compute nested validation-MAE changes for audit
# ------------------------------------------------------------

lstm_15min_incremental_rows = []


for previous_family, current_family in [
    ("F0", "F1"),
    ("F1", "F2"),
    ("F2", "F3"),
    ("F3", "F4"),
]:

    previous_mae = (
        lstm_epoch_selection_results[
            (
                15,
                previous_family,
            )
        ][
            "validation_MAE_kW"
        ]
    )


    current_mae = (
        lstm_epoch_selection_results[
            (
                15,
                current_family,
            )
        ][
            "validation_MAE_kW"
        ]
    )


    relative_change_pct = (
        (
            previous_mae
            - current_mae
        )
        /
        previous_mae
        * 100.0
    )


    lstm_15min_incremental_rows.append(
        {
            "comparison":
                (
                    f"{previous_family}"
                    f"_to_"
                    f"{current_family}"
                ),

            "previous_MAE_kW":
                previous_mae,

            "current_MAE_kW":
                current_mae,

            "relative_MAE_improvement_pct":
                relative_change_pct,
        }
    )


lstm_15min_incremental_validation = (
    pd.DataFrame(
        lstm_15min_incremental_rows
    )
)


print(
    "\n15-minute incremental validation changes:"
)


display(
    lstm_15min_incremental_validation
)


# ------------------------------------------------------------
# 11. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 12. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 13. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 15-min F4 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 14:20:58.525957: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 14:20:58.526299: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 16ms/step - loss: 0.1809 - val_loss: 0.1919
Epoch 2/80
341/341 - 3s - 10ms/step - loss: 0.1296 - val_loss: 0.1932
Epoch 3/80
341/341 - 3s - 10ms/step - loss: 0.1300 - val_loss: 0.1913
Epoch 4/80
341/341 - 4s - 10ms/step - loss: 0.1345 - val_loss: 0.1983
Epoch 5/80
341/341 - 4s - 12ms/step - loss: 0.1330 - val_loss: 0.1950
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1319 - val_loss: 0.1961
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1338 - val_loss: 0.1850
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1321 - val_loss: 0.1900
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1304 - val_loss: 0.1858
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.1313 - val_loss: 0.2064
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.1329 - val_loss: 0.2030
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.1335 - val_loss: 0.1924
Epoch 13/80
341/341 - 4s - 11ms/step - loss: 0.1360 - val_loss: 0.2074
Epoch 14/80
341/341 - 4s - 11ms/step - loss: 0.1407 - val_loss: 0.2247
Epoch 15/80
341/341 - 4s -

2026-09-21 14:21:55.247492: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 14:21:55.247742: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 15 min
  Family:  F4
  Architecture: RNN_C1 (frozen)


...


15-min validation F3 -> F4 comparison:
  F3 MAE: 0.292315 kW
  F4 MAE: 0.293711 kW
  Relative MAE change: -0.4778%
  Validation only — not a held-out test conclusion.

Complete 15-minute LSTM validation progression:


,feature_family,auxiliary_dim,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,1,7,0.289374,0.650396,0.887437
1,F1,16,1,0.291128,0.638846,0.891400
2,F2,21,2,0.297091,0.649842,0.887629
3,F3,30,2,0.292315,0.641045,0.890651
4,F4,127,7,0.293711,0.638679,0.891457



15-minute incremental validation changes:


,comparison,previous_MAE_kW,current_MAE_kW,relative_MAE_improvement_pct
0,F0_to_F1,0.289374,0.291128,-0.606359
1,F1_to_F2,0.291128,0.297091,-2.048284
2,F2_to_F3,0.297091,0.292315,1.607779
3,F3_to_F4,0.292315,0.293711,-0.477763



Completed Stage-2 combinations: 6 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 15-min F4 epoch-selection run: PASSED


In [35]:
# ============================================================
# Step 32: LSTM Stage-2 epoch selection
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F0
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F0 inputs:
#
#   Sequence branch:
#       13 × 9 historical tensor
#
#   Auxiliary branch:
#       P_Solar(t - 24 h)
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 30-minute F0
# ------------------------------------------------------------

(
    lstm_30_f0_result,
    lstm_30_f0_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=30,

    feature_family="F0",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# Same architecture and same F0 auxiliary width as:
#
#       15-min F0
#       60-min F0
#
# therefore:
#
#       6,497 parameters
# ------------------------------------------------------------

EXPECTED_LSTM_F0_PARAMETERS = (
    6_497
)


assert (
    lstm_30_f0_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F0_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_30_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_30_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_30_f0_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_30_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_30_f0_result[
        "auxiliary_dim"
    ]
    == 1
)


assert (
    lstm_30_f0_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_30_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_30_f0_result[
        "best_epoch"
    ]
    <= lstm_30_f0_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_30_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_30_f0_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        30,
        "F0",
    )
] = (
    lstm_30_f0_result.copy()
)


lstm_epoch_selection_histories[
    (
        30,
        "F0",
    )
] = (
    lstm_30_f0_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
#
# Complete:
#
#       15-min F0-F4
#       30-min F0
#       60-min F0
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
    (30, "F0"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 30 min"
)

print(
    "  Family:  F0"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_30_f0_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_30_f0 = (
    lstm_30_f0_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_30_f0 - 4,
)


history_window_end = min(
    len(
        lstm_30_f0_history
    ),
    best_epoch_30_f0 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_30_f0_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. F0 validation across forecast horizons
#
# This is useful only as a descriptive validation-stage audit.
# We do NOT use it to alter the frozen architecture.
# ------------------------------------------------------------

lstm_f0_horizon_validation = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "validation_R2"
                ],
        }

        for horizon_minutes in [
            15,
            30,
            60,
        ]
    ]
)


print(
    "\nLSTM F0 validation across horizons:"
)


display(
    lstm_f0_horizon_validation
)


# ------------------------------------------------------------
# 9. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 10. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 11. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 30-min F0 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 14:23:23.547955: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 14:23:23.549308: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.1966 - val_loss: 0.2439
Epoch 2/80
341/341 - 4s - 12ms/step - loss: 0.1563 - val_loss: 0.2355
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1603 - val_loss: 0.2442
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1666 - val_loss: 0.2372
Epoch 5/80
341/341 - 4s - 12ms/step - loss: 0.1730 - val_loss: 0.2414
Epoch 6/80
341/341 - 4s - 10ms/step - loss: 0.1704 - val_loss: 0.2492
Epoch 7/80
341/341 - 4s - 10ms/step - loss: 0.1716 - val_loss: 0.2466
Epoch 8/80
341/341 - 4s - 10ms/step - loss: 0.1680 - val_loss: 0.2542
Epoch 9/80
341/341 - 4s - 10ms/step - loss: 0.1694 - val_loss: 0.2397
Epoch 10/80
341/341 - 4s - 10ms/step - loss: 0.1616 - val_loss: 0.2311
Epoch 11/80
341/341 - 4s - 10ms/step - loss: 0.1660 - val_loss: 0.2588
Epoch 12/80
341/341 - 4s - 10ms/step - loss: 0.1663 - val_loss: 0.2359
Epoch 13/80
341/341 - 4s - 12ms/step - loss: 0.1672 - val_loss: 0.2516
Epoch 14/80
341/341 - 4s - 11ms/step - loss: 0.1646 - val_loss: 0.2516
Epoch 15/80
341/341 - 4s -

2026-09-21 14:24:31.031224: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 14:24:31.031480: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 30 min
  Family:  F0
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
6,7,0.171585,0.246556,False
7,8,0.168026,0.254207,False
8,9,0.169391,0.239718,False
9,10,0.161570,0.231120,True
10,11,0.166009,0.258848,False
11,12,0.166342,0.235936,False
12,13,0.167231,0.251595,False



LSTM F0 validation across horizons:


,horizon_minutes,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,15,7,0.289374,0.650396,0.887437
1,30,10,0.366990,0.741797,0.853577
2,60,10,0.465383,0.869118,0.799001



Completed Stage-2 combinations: 7 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 30-min F0 epoch-selection run: PASSED


In [36]:
# ============================================================
# Step 33: LSTM Stage-2 epoch selection
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F1
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
# F1 auxiliary features:
#
#       1 historical 24-hour PV lag
#       +
#       15 Gregorian/time features
#       =
#       16 auxiliary features
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 30-minute F1
# ------------------------------------------------------------

(
    lstm_30_f1_result,
    lstm_30_f1_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=30,

    feature_family="F1",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# Same feature width as 15-minute F1:
#
#       LSTM sequence branch = 5,376
#
#       Fusion input:
#           32 sequence embedding
#           16 auxiliary
#           = 48
#
#       Dense:
#           48 × 32 + 32
#           = 1,568
#
#       Output:
#           32 × 1 + 1
#           = 33
#
#       Total:
#           5,376 + 1,568 + 33
#           = 6,977
# ------------------------------------------------------------

EXPECTED_LSTM_F1_PARAMETERS = (
    6_977
)


assert (
    lstm_30_f1_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F1_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_30_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_30_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_30_f1_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_30_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_30_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_30_f1_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_30_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_30_f1_result[
        "best_epoch"
    ]
    <= lstm_30_f1_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_30_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_30_f1_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        30,
        "F1",
    )
] = (
    lstm_30_f1_result.copy()
)


lstm_epoch_selection_histories[
    (
        30,
        "F1",
    )
] = (
    lstm_30_f1_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
    (30, "F0"),
    (30, "F1"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 30 min"
)

print(
    "  Family:  F1"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_30_f1_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show training history around selected epoch
# ------------------------------------------------------------

best_epoch_30_f1 = (
    lstm_30_f1_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_30_f1 - 4,
)


history_window_end = min(
    len(
        lstm_30_f1_history
    ),
    best_epoch_30_f1 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_30_f1_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F0 -> F1 comparison
#
# Positive percentage:
#       F1 has lower validation MAE.
#
# Negative percentage:
#       F1 has higher validation MAE.
# ------------------------------------------------------------

f0_validation_mae_30 = (
    lstm_epoch_selection_results[
        (
            30,
            "F0",
        )
    ][
        "validation_MAE_kW"
    ]
)


f1_validation_mae_30 = (
    lstm_30_f1_result[
        "validation_MAE_kW"
    ]
)


f0_to_f1_validation_improvement_pct_30 = (
    (
        f0_validation_mae_30
        - f1_validation_mae_30
    )
    /
    f0_validation_mae_30
    * 100.0
)


print(
    "\n30-min validation F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_validation_mae_30:.6f} kW"
)

print(
    f"  F1 MAE: {f1_validation_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_validation_improvement_pct_30:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 30-minute validation progression
# ------------------------------------------------------------

lstm_30min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)


print(
    "\n30-minute LSTM validation progression:"
)


display(
    lstm_30min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 30-min F1 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 16:49:32.571042: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 16:49:32.571371: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 14ms/step - loss: 0.1900 - val_loss: 0.2282
Epoch 2/80
341/341 - 3s - 10ms/step - loss: 0.1580 - val_loss: 0.2327
Epoch 3/80
341/341 - 3s - 10ms/step - loss: 0.1608 - val_loss: 0.2304
Epoch 4/80
341/341 - 3s - 10ms/step - loss: 0.1588 - val_loss: 0.2337
Epoch 5/80
341/341 - 3s - 10ms/step - loss: 0.1593 - val_loss: 0.2446
Epoch 6/80
341/341 - 3s - 10ms/step - loss: 0.1660 - val_loss: 0.2426
Epoch 7/80
341/341 - 3s - 10ms/step - loss: 0.1660 - val_loss: 0.2445
Epoch 8/80
341/341 - 4s - 10ms/step - loss: 0.1662 - val_loss: 0.2605
Epoch 9/80
341/341 - 3s - 10ms/step - loss: 0.1671 - val_loss: 0.2552


2026-09-21 16:50:05.054775: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 16:50:05.055006: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 30 min
  Family:  F1
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.189978,0.228236,True
1,2,0.158020,0.232666,False
2,3,0.160796,0.230375,False
3,4,0.158845,0.233687,False



30-min validation F0 -> F1 comparison:
  F0 MAE: 0.366990 kW
  F1 MAE: 0.362409 kW
  Relative MAE change: +1.2481%
  Validation only — not a test-set result.

30-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.366990,0.741797,0.853577
1,F1,1,0.362409,0.735007,0.856246



Completed Stage-2 combinations: 8 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 30-min F1 epoch-selection run: PASSED


In [37]:
# ============================================================
# Step 34: LSTM Stage-2 epoch selection
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F2
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F2 = F1 + five independent solar-astronomy features:
#
#       solar_elevation_deg
#       solar_cos_zenith
#       solar_azimuth_sin
#       solar_azimuth_cos
#       solar_daylight
#
#
# Auxiliary width:
#
#       F1 = 16
#       F2 = 21
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 30-minute F2
# ------------------------------------------------------------

(
    lstm_30_f2_result,
    lstm_30_f2_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=30,

    feature_family="F2",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F2 auxiliary       = 21
#
#       total              = 53
#
# Fusion Dense:
#
#       53 × 32 + 32
#       = 1,728
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 1,728 + 33
#       = 7,137
# ------------------------------------------------------------

EXPECTED_LSTM_F2_PARAMETERS = (
    7_137
)


assert (
    lstm_30_f2_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F2_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_30_f2_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_30_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_30_f2_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_30_f2_result[
        "feature_family"
    ]
    == "F2"
)


assert (
    lstm_30_f2_result[
        "auxiliary_dim"
    ]
    == 21
)


assert (
    lstm_30_f2_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_30_f2_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_30_f2_result[
        "best_epoch"
    ]
    <= lstm_30_f2_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_30_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_30_f2_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        30,
        "F2",
    )
] = (
    lstm_30_f2_result.copy()
)


lstm_epoch_selection_histories[
    (
        30,
        "F2",
    )
] = (
    lstm_30_f2_history.copy()
)


# ------------------------------------------------------------
# 5. Verify expected Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 30 min"
)

print(
    "  Family:  F2"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_30_f2_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_30_f2 = (
    lstm_30_f2_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_30_f2 - 4,
)


history_window_end = min(
    len(
        lstm_30_f2_history
    ),
    best_epoch_30_f2 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_30_f2_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F1 -> F2 comparison
#
# Positive percentage:
#       F2 has lower validation MAE.
#
# Negative percentage:
#       F2 has higher validation MAE.
# ------------------------------------------------------------

f1_validation_mae_30 = (
    lstm_epoch_selection_results[
        (
            30,
            "F1",
        )
    ][
        "validation_MAE_kW"
    ]
)


f2_validation_mae_30 = (
    lstm_30_f2_result[
        "validation_MAE_kW"
    ]
)


f1_to_f2_validation_improvement_pct_30 = (
    (
        f1_validation_mae_30
        - f2_validation_mae_30
    )
    /
    f1_validation_mae_30
    * 100.0
)


print(
    "\n30-min validation F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_validation_mae_30:.6f} kW"
)

print(
    f"  F2 MAE: {f2_validation_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_validation_improvement_pct_30:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 30-minute validation progression
# ------------------------------------------------------------

lstm_30min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)


print(
    "\n30-minute LSTM validation progression:"
)


display(
    lstm_30min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 30-min F2 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 16:52:12.492740: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 16:52:12.493526: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 14ms/step - loss: 0.2119 - val_loss: 0.2331
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.1624 - val_loss: 0.2394
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1610 - val_loss: 0.2507
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1655 - val_loss: 0.2477
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1674 - val_loss: 0.2455
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1656 - val_loss: 0.2309
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1631 - val_loss: 0.2453
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1648 - val_loss: 0.2429
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1629 - val_loss: 0.2376
Epoch 10/80
341/341 - 4s - 12ms/step - loss: 0.1639 - val_loss: 0.2653
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.1652 - val_loss: 0.2351
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.1650 - val_loss: 0.2426
Epoch 13/80
341/341 - 4s - 11ms/step - loss: 0.1650 - val_loss: 0.2352
Epoch 14/80
341/341 - 4s - 11ms/step - loss: 0.1683 - val_loss: 0.2559


2026-09-21 16:53:06.317085: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 16:53:06.317328: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 30 min
  Family:  F2
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
2,3,0.160962,0.250705,False
3,4,0.165548,0.247741,False
4,5,0.167375,0.245514,False
5,6,0.165562,0.230935,True
6,7,0.163109,0.245309,False
7,8,0.164784,0.242878,False
8,9,0.162930,0.237643,False



30-min validation F1 -> F2 comparison:
  F1 MAE: 0.362409 kW
  F2 MAE: 0.366695 kW
  Relative MAE change: -1.1826%
  Validation only — not a test-set result.

30-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.366990,0.741797,0.853577
1,F1,1,0.362409,0.735007,0.856246
2,F2,6,0.366695,0.749496,0.850522



Completed Stage-2 combinations: 9 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 30-min F2 epoch-selection run: PASSED


In [38]:
# ============================================================
# Step 35: LSTM Stage-2 epoch selection
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F3
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F3 = F2 + nine continuous lunar / Sun-Moon features:
#
#       moon_phase_sin
#       moon_phase_cos
#       moon_longitude_sin
#       moon_longitude_cos
#       sun_ecliptic_longitude_sin
#       sun_ecliptic_longitude_cos
#       moon_ecliptic_latitude_deg
#       moon_illumination_fraction
#       earth_moon_distance_km
#
#
# Auxiliary width:
#
#       F2 = 21
#       F3 = 30
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 30-minute F3
# ------------------------------------------------------------

(
    lstm_30_f3_result,
    lstm_30_f3_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=30,

    feature_family="F3",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F3 auxiliary       = 30
#
#       total              = 62
#
# Fusion Dense:
#
#       62 × 32 + 32
#       = 2,016
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 2,016 + 33
#       = 7,425
# ------------------------------------------------------------

EXPECTED_LSTM_F3_PARAMETERS = (
    7_425
)


assert (
    lstm_30_f3_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F3_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_30_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_30_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_30_f3_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_30_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_30_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_30_f3_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_30_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_30_f3_result[
        "best_epoch"
    ]
    <= lstm_30_f3_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_30_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_30_f3_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        30,
        "F3",
    )
] = (
    lstm_30_f3_result.copy()
)


lstm_epoch_selection_histories[
    (
        30,
        "F3",
    )
] = (
    lstm_30_f3_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 30 min"
)

print(
    "  Family:  F3"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_30_f3_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show training history around selected epoch
# ------------------------------------------------------------

best_epoch_30_f3 = (
    lstm_30_f3_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_30_f3 - 4,
)


history_window_end = min(
    len(
        lstm_30_f3_history
    ),
    best_epoch_30_f3 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_30_f3_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F2 -> F3 comparison
#
# Positive percentage:
#       F3 has lower validation MAE.
#
# Negative percentage:
#       F3 has higher validation MAE.
#
# This is still validation-only evidence.
# ------------------------------------------------------------

f2_validation_mae_30 = (
    lstm_epoch_selection_results[
        (
            30,
            "F2",
        )
    ][
        "validation_MAE_kW"
    ]
)


f3_validation_mae_30 = (
    lstm_30_f3_result[
        "validation_MAE_kW"
    ]
)


f2_to_f3_validation_improvement_pct_30 = (
    (
        f2_validation_mae_30
        - f3_validation_mae_30
    )
    /
    f2_validation_mae_30
    * 100.0
)


print(
    "\n30-min validation F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_validation_mae_30:.6f} kW"
)

print(
    f"  F3 MAE: {f3_validation_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_validation_improvement_pct_30:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 30-minute validation progression
# ------------------------------------------------------------

lstm_30min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\n30-minute LSTM validation progression:"
)


display(
    lstm_30min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 30-min F3 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 16:55:29.753896: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 16:55:29.754160: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 16ms/step - loss: 0.2193 - val_loss: 0.2457
Epoch 2/80
341/341 - 4s - 12ms/step - loss: 0.1627 - val_loss: 0.2431
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1644 - val_loss: 0.2450
Epoch 4/80
341/341 - 4s - 12ms/step - loss: 0.1655 - val_loss: 0.2457
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1643 - val_loss: 0.2395
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1699 - val_loss: 0.2503
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1678 - val_loss: 0.2412
Epoch 8/80
341/341 - 4s - 12ms/step - loss: 0.1717 - val_loss: 0.2519
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1698 - val_loss: 0.2519
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.1716 - val_loss: 0.2692
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.1835 - val_loss: 0.2595
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.2200 - val_loss: 0.2934
Epoch 13/80
341/341 - 4s - 11ms/step - loss: 0.2017 - val_loss: 0.2768


2026-09-21 16:56:21.027229: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 16:56:21.027494: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 30 min
  Family:  F3
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
1,2,0.162703,0.243076,False
2,3,0.164449,0.245012,False
3,4,0.165499,0.245678,False
4,5,0.164311,0.239513,True
5,6,0.169869,0.250272,False
6,7,0.167819,0.241239,False
7,8,0.171685,0.251925,False



30-min validation F2 -> F3 comparison:
  F2 MAE: 0.366695 kW
  F3 MAE: 0.380316 kW
  Relative MAE change: -3.7145%
  Validation only — not a test-set result.

30-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.366990,0.741797,0.853577
1,F1,1,0.362409,0.735007,0.856246
2,F2,6,0.366695,0.749496,0.850522
3,F3,5,0.380316,0.757902,0.847150



Completed Stage-2 combinations: 10 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 30-min F3 epoch-selection run: PASSED


In [39]:
# ============================================================
# Step 36: LSTM Stage-2 epoch selection
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F4
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F4 = F3 + 97 canonical Panchang one-hot features:
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 categorical indicators
#
#
# Auxiliary width:
#
#       F3 =  30
#       F4 = 127
#
#
# Main scientific comparison:
#
#       F3 -> F4
#
# asks whether categorical Panchang information improves
# forecasting beyond the continuous astronomical variables
# already contained in F3.
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 30-minute F4
# ------------------------------------------------------------

(
    lstm_30_f4_result,
    lstm_30_f4_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=30,

    feature_family="F4",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding =  32
#       F4 auxiliary       = 127
#
#       total              = 159
#
# Fusion Dense:
#
#       159 × 32 + 32
#       = 5,120
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 5,120 + 33
#       = 10,529
# ------------------------------------------------------------

EXPECTED_LSTM_F4_PARAMETERS = (
    10_529
)


assert (
    lstm_30_f4_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F4_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_30_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_30_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_30_f4_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_30_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_30_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_30_f4_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_30_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_30_f4_result[
        "best_epoch"
    ]
    <= lstm_30_f4_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_30_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_30_f4_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        30,
        "F4",
    )
] = (
    lstm_30_f4_result.copy()
)


lstm_epoch_selection_histories[
    (
        30,
        "F4",
    )
] = (
    lstm_30_f4_history.copy()
)


# ------------------------------------------------------------
# 5. Verify that the complete 15-min and 30-min F0-F4
#    sequences are now available, plus reused 60-min F0.
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
}


# ------------------------------------------------------------
# 6. Display new F4 result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 30 min"
)

print(
    "  Family:  F4"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_30_f4_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show training history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_30_f4 = (
    lstm_30_f4_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_30_f4 - 4,
)


history_window_end = min(
    len(
        lstm_30_f4_history
    ),
    best_epoch_30_f4 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_30_f4_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. KEY validation comparison:
#
#       F3 -> F4
#
# Positive percentage:
#       F4 has lower validation MAE.
#
# Negative percentage:
#       F4 has higher validation MAE.
#
# This remains validation-only evidence.
# ------------------------------------------------------------

f3_validation_mae_30 = (
    lstm_epoch_selection_results[
        (
            30,
            "F3",
        )
    ][
        "validation_MAE_kW"
    ]
)


f4_validation_mae_30 = (
    lstm_30_f4_result[
        "validation_MAE_kW"
    ]
)


f3_to_f4_validation_improvement_pct_30 = (
    (
        f3_validation_mae_30
        - f4_validation_mae_30
    )
    /
    f3_validation_mae_30
    * 100.0
)


print(
    "\n30-min validation F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_validation_mae_30:.6f} kW"
)

print(
    f"  F4 MAE: {f4_validation_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_validation_improvement_pct_30:+.4f}%",
)

print(
    "  Validation only — not a held-out test conclusion."
)


# ------------------------------------------------------------
# 9. Complete 30-minute validation progression
# ------------------------------------------------------------

lstm_30min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "auxiliary_dim":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "auxiliary_dim"
                ],

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 30-minute LSTM validation progression:"
)


display(
    lstm_30min_validation_progress
)


# ------------------------------------------------------------
# 10. Compute complete nested 30-minute validation changes
# ------------------------------------------------------------

lstm_30min_incremental_rows = []


for previous_family, current_family in [
    ("F0", "F1"),
    ("F1", "F2"),
    ("F2", "F3"),
    ("F3", "F4"),
]:

    previous_mae = (
        lstm_epoch_selection_results[
            (
                30,
                previous_family,
            )
        ][
            "validation_MAE_kW"
        ]
    )


    current_mae = (
        lstm_epoch_selection_results[
            (
                30,
                current_family,
            )
        ][
            "validation_MAE_kW"
        ]
    )


    relative_change_pct = (
        (
            previous_mae
            - current_mae
        )
        /
        previous_mae
        * 100.0
    )


    lstm_30min_incremental_rows.append(
        {
            "comparison":
                (
                    f"{previous_family}"
                    f"_to_"
                    f"{current_family}"
                ),

            "previous_MAE_kW":
                previous_mae,

            "current_MAE_kW":
                current_mae,

            "relative_MAE_improvement_pct":
                relative_change_pct,
        }
    )


lstm_30min_incremental_validation = pd.DataFrame(
    lstm_30min_incremental_rows
)


print(
    "\n30-minute incremental validation changes:"
)


display(
    lstm_30min_incremental_validation
)


# ------------------------------------------------------------
# 11. Compare the two completed F3 -> F4 validation tests
#
# We now have:
#
#       15-minute F3 -> F4
#       30-minute F3 -> F4
#
# 60-minute F3 -> F4 is still pending.
# ------------------------------------------------------------

lstm_f3_f4_validation_summary_partial = pd.DataFrame(
    [
        {
            "horizon_minutes":
                15,

            "F3_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        "F3",
                    )
                ][
                    "validation_MAE_kW"
                ],

            "F4_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        15,
                        "F4",
                    )
                ][
                    "validation_MAE_kW"
                ],
        },

        {
            "horizon_minutes":
                30,

            "F3_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        "F3",
                    )
                ][
                    "validation_MAE_kW"
                ],

            "F4_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        30,
                        "F4",
                    )
                ][
                    "validation_MAE_kW"
                ],
        },
    ]
)


lstm_f3_f4_validation_summary_partial[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        lstm_f3_f4_validation_summary_partial[
            "F3_MAE_kW"
        ]
        -
        lstm_f3_f4_validation_summary_partial[
            "F4_MAE_kW"
        ]
    )
    /
    lstm_f3_f4_validation_summary_partial[
        "F3_MAE_kW"
    ]
    * 100.0
)


print(
    "\nPartial LSTM F3 -> F4 validation summary:"
)


display(
    lstm_f3_f4_validation_summary_partial
)


# ------------------------------------------------------------
# 12. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 13. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 14. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 30-min F4 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 16:57:40.268230: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 16:57:40.268656: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.2093 - val_loss: 0.2411
Epoch 2/80
341/341 - 4s - 10ms/step - loss: 0.1625 - val_loss: 0.2490
Epoch 3/80
341/341 - 3s - 10ms/step - loss: 0.1697 - val_loss: 0.2442
Epoch 4/80
341/341 - 3s - 10ms/step - loss: 0.1700 - val_loss: 0.2457
Epoch 5/80
341/341 - 3s - 10ms/step - loss: 0.1633 - val_loss: 0.2440
Epoch 6/80
341/341 - 3s - 10ms/step - loss: 0.1644 - val_loss: 0.2715
Epoch 7/80
341/341 - 3s - 10ms/step - loss: 0.1756 - val_loss: 0.2774
Epoch 8/80
341/341 - 3s - 10ms/step - loss: 0.1797 - val_loss: 0.2475
Epoch 9/80
341/341 - 3s - 10ms/step - loss: 0.1788 - val_loss: 0.2634


2026-09-21 16:58:13.304634: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 16:58:13.304918: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 30 min
  Family:  F4
  Architecture: RNN_C1 (frozen)


...


30-minute incremental validation changes:


,comparison,previous_MAE_kW,current_MAE_kW,relative_MAE_improvement_pct
0,F0_to_F1,0.366990,0.362409,1.248130
1,F1_to_F2,0.362409,0.366695,-1.182576
2,F2_to_F3,0.366695,0.380316,-3.714462
3,F3_to_F4,0.380316,0.382804,-0.654289



Partial LSTM F3 -> F4 validation summary:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.292315,0.293711,-0.477763
1,30,0.380316,0.382804,-0.654289



Completed Stage-2 combinations: 11 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 30-min F4 epoch-selection run: PASSED


In [40]:
# ============================================================
# Step 37: LSTM Stage-2 epoch selection
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F1
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F1 auxiliary branch:
#
#       1 historical 24-hour PV lag
#       +
#       15 Gregorian/time features
#       =
#       16 auxiliary features
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 60-minute F1
# ------------------------------------------------------------

(
    lstm_60_f1_result,
    lstm_60_f1_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=60,

    feature_family="F1",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F1 auxiliary       = 16
#
#       total              = 48
#
# Fusion Dense:
#
#       48 × 32 + 32
#       = 1,568
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 1,568 + 33
#       = 6,977
# ------------------------------------------------------------

EXPECTED_LSTM_F1_PARAMETERS = (
    6_977
)


assert (
    lstm_60_f1_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F1_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_60_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_60_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_60_f1_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_60_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_60_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_60_f1_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_60_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_60_f1_result[
        "best_epoch"
    ]
    <= lstm_60_f1_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_60_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_60_f1_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        60,
        "F1",
    )
] = (
    lstm_60_f1_result.copy()
)


lstm_epoch_selection_histories[
    (
        60,
        "F1",
    )
] = (
    lstm_60_f1_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
#
# Complete:
#
#       15-min F0-F4
#       30-min F0-F4
#       60-min F0-F1
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 60 min"
)

print(
    "  Family:  F1"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_60_f1_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_60_f1 = (
    lstm_60_f1_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_60_f1 - 4,
)


history_window_end = min(
    len(
        lstm_60_f1_history
    ),
    best_epoch_60_f1 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_60_f1_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F0 -> F1 comparison
#
# Positive percentage:
#       F1 has lower validation MAE.
#
# Negative percentage:
#       F1 has higher validation MAE.
# ------------------------------------------------------------

f0_validation_mae_60 = (
    lstm_epoch_selection_results[
        (
            60,
            "F0",
        )
    ][
        "validation_MAE_kW"
    ]
)


f1_validation_mae_60 = (
    lstm_60_f1_result[
        "validation_MAE_kW"
    ]
)


f0_to_f1_validation_improvement_pct_60 = (
    (
        f0_validation_mae_60
        - f1_validation_mae_60
    )
    /
    f0_validation_mae_60
    * 100.0
)


print(
    "\n60-min validation F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_validation_mae_60:.6f} kW"
)

print(
    f"  F1 MAE: {f1_validation_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_validation_improvement_pct_60:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 60-minute validation progression
# ------------------------------------------------------------

lstm_60min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)


print(
    "\n60-minute LSTM validation progression:"
)


display(
    lstm_60min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 60-min F1 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 17:02:12.636086: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:02:12.636628: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 6s - 16ms/step - loss: 0.2412 - val_loss: 0.3052
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.2089 - val_loss: 0.2971
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.2136 - val_loss: 0.2985
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.2025 - val_loss: 0.2908
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.2142 - val_loss: 0.2995
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.2101 - val_loss: 0.3041
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2098 - val_loss: 0.3155
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.2278 - val_loss: 0.3215
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.2340 - val_loss: 0.3414
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.2204 - val_loss: 0.3206
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.2211 - val_loss: 0.3223
Epoch 12/80
341/341 - 4s - 11ms/step - loss: 0.2271 - val_loss: 0.3204


2026-09-21 17:02:59.217524: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:02:59.217883: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 60 min
  Family:  F1
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.241238,0.305237,False
1,2,0.208943,0.297053,False
2,3,0.213600,0.298536,False
3,4,0.202487,0.290847,True
4,5,0.214198,0.299465,False
5,6,0.210061,0.304071,False
6,7,0.209845,0.315460,False



60-min validation F0 -> F1 comparison:
  F0 MAE: 0.465383 kW
  F1 MAE: 0.461840 kW
  Relative MAE change: +0.7614%
  Validation only — not a test-set result.

60-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.465383,0.869118,0.799001
1,F1,4,0.461840,0.894092,0.787283



Completed Stage-2 combinations: 12 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 60-min F1 epoch-selection run: PASSED


In [41]:
# ============================================================
# Step 38: LSTM Stage-2 epoch selection
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F2
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F2 = F1 + five independent solar-astronomy features:
#
#       solar_elevation_deg
#       solar_cos_zenith
#       solar_azimuth_sin
#       solar_azimuth_cos
#       solar_daylight
#
#
# Auxiliary width:
#
#       F1 = 16
#       F2 = 21
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 60-minute F2
# ------------------------------------------------------------

(
    lstm_60_f2_result,
    lstm_60_f2_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=60,

    feature_family="F2",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# Sequence branch:
#       5,376
#
# Fusion input:
#       32 sequence embedding
#       21 auxiliary
#       = 53
#
# Fusion Dense:
#       53 × 32 + 32
#       = 1,728
#
# Output:
#       32 × 1 + 1
#       = 33
#
# Total:
#       5,376 + 1,728 + 33
#       = 7,137
# ------------------------------------------------------------

EXPECTED_LSTM_F2_PARAMETERS = 7_137


assert (
    lstm_60_f2_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F2_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert lstm_60_f2_result["model"] == "LSTM"

assert (
    lstm_60_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)

assert (
    lstm_60_f2_result[
        "horizon_minutes"
    ]
    == 60
)

assert (
    lstm_60_f2_result[
        "feature_family"
    ]
    == "F2"
)

assert (
    lstm_60_f2_result[
        "auxiliary_dim"
    ]
    == 21
)

assert (
    lstm_60_f2_result[
        "n_train"
    ]
    == 87_252
)

assert (
    lstm_60_f2_result[
        "n_validation"
    ]
    == 17_568
)

assert (
    1
    <= lstm_60_f2_result["best_epoch"]
    <= lstm_60_f2_result["epochs_ran"]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_60_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_60_f2_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        60,
        "F2",
    )
] = (
    lstm_60_f2_result.copy()
)


lstm_epoch_selection_histories[
    (
        60,
        "F2",
    )
] = (
    lstm_60_f2_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
    (60, "F2"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 60 min"
)

print(
    "  Family:  F2"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_60_f2_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_60_f2 = (
    lstm_60_f2_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_60_f2 - 4,
)


history_window_end = min(
    len(
        lstm_60_f2_history
    ),
    best_epoch_60_f2 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_60_f2_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F1 -> F2 comparison
# ------------------------------------------------------------

f1_validation_mae_60 = (
    lstm_epoch_selection_results[
        (
            60,
            "F1",
        )
    ][
        "validation_MAE_kW"
    ]
)


f2_validation_mae_60 = (
    lstm_60_f2_result[
        "validation_MAE_kW"
    ]
)


f1_to_f2_validation_improvement_pct_60 = (
    (
        f1_validation_mae_60
        - f2_validation_mae_60
    )
    /
    f1_validation_mae_60
    * 100.0
)


print(
    "\n60-min validation F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_validation_mae_60:.6f} kW"
)

print(
    f"  F2 MAE: {f2_validation_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_validation_improvement_pct_60:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 60-minute validation progression
# ------------------------------------------------------------

lstm_60min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)


print(
    "\n60-minute LSTM validation progression:"
)


display(
    lstm_60min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 60-min F2 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 17:05:13.987012: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:05:13.987741: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.2596 - val_loss: 0.2941
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.2081 - val_loss: 0.2992
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.2107 - val_loss: 0.3174
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.2344 - val_loss: 0.3099
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.2171 - val_loss: 0.2977
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.2147 - val_loss: 0.3127
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2146 - val_loss: 0.3196
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.2196 - val_loss: 0.3181
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.2256 - val_loss: 0.3070


2026-09-21 17:05:48.983160: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:05:48.983415: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 60 min
  Family:  F2
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.259560,0.294143,True
1,2,0.208122,0.299187,False
2,3,0.210656,0.317394,False
3,4,0.234430,0.309875,False



60-min validation F1 -> F2 comparison:
  F1 MAE: 0.461840 kW
  F2 MAE: 0.467074 kW
  Relative MAE change: -1.1333%
  Validation only — not a test-set result.

60-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.465383,0.869118,0.799001
1,F1,4,0.461840,0.894092,0.787283
2,F2,1,0.467074,0.855104,0.805430



Completed Stage-2 combinations: 13 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 60-min F2 epoch-selection run: PASSED


In [42]:
# ============================================================
# Step 39: LSTM Stage-2 epoch selection
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F3
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F3 = F2 + nine continuous lunar / Sun-Moon features:
#
#       moon_phase_sin
#       moon_phase_cos
#       moon_longitude_sin
#       moon_longitude_cos
#       sun_ecliptic_longitude_sin
#       sun_ecliptic_longitude_cos
#       moon_ecliptic_latitude_deg
#       moon_illumination_fraction
#       earth_moon_distance_km
#
#
# Auxiliary width:
#
#       F2 = 21
#       F3 = 30
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 60-minute F3
# ------------------------------------------------------------

(
    lstm_60_f3_result,
    lstm_60_f3_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=60,

    feature_family="F3",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding = 32
#       F3 auxiliary       = 30
#
#       total              = 62
#
# Fusion Dense:
#
#       62 × 32 + 32
#       = 2,016
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 2,016 + 33
#       = 7,425
# ------------------------------------------------------------

EXPECTED_LSTM_F3_PARAMETERS = (
    7_425
)


assert (
    lstm_60_f3_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F3_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_60_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_60_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_60_f3_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_60_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_60_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_60_f3_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_60_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_60_f3_result[
        "best_epoch"
    ]
    <= lstm_60_f3_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_60_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_60_f3_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        60,
        "F3",
    )
] = (
    lstm_60_f3_result.copy()
)


lstm_epoch_selection_histories[
    (
        60,
        "F3",
    )
] = (
    lstm_60_f3_history.copy()
)


# ------------------------------------------------------------
# 5. Verify current Stage-2 contents
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
    (60, "F2"),
    (60, "F3"),
}


# ------------------------------------------------------------
# 6. Display new result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 60 min"
)

print(
    "  Family:  F3"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_60_f3_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show training history around selected epoch
# ------------------------------------------------------------

best_epoch_60_f3 = (
    lstm_60_f3_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_60_f3 - 4,
)


history_window_end = min(
    len(
        lstm_60_f3_history
    ),
    best_epoch_60_f3 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_60_f3_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Validation-only F2 -> F3 comparison
#
# Positive percentage:
#       F3 has lower validation MAE.
#
# Negative percentage:
#       F3 has higher validation MAE.
# ------------------------------------------------------------

f2_validation_mae_60 = (
    lstm_epoch_selection_results[
        (
            60,
            "F2",
        )
    ][
        "validation_MAE_kW"
    ]
)


f3_validation_mae_60 = (
    lstm_60_f3_result[
        "validation_MAE_kW"
    ]
)


f2_to_f3_validation_improvement_pct_60 = (
    (
        f2_validation_mae_60
        - f3_validation_mae_60
    )
    /
    f2_validation_mae_60
    * 100.0
)


print(
    "\n60-min validation F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_validation_mae_60:.6f} kW"
)

print(
    f"  F3 MAE: {f3_validation_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_validation_improvement_pct_60:+.4f}%",
)

print(
    "  Validation only — not a test-set result."
)


# ------------------------------------------------------------
# 9. Current 60-minute validation progression
# ------------------------------------------------------------

lstm_60min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\n60-minute LSTM validation progression:"
)


display(
    lstm_60min_validation_progress
)


# ------------------------------------------------------------
# 10. Overall Stage-2 progress
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15",
)


# ------------------------------------------------------------
# 11. Explicit test-set protection
# ------------------------------------------------------------

print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM 60-min F3 epoch-selection run: PASSED"
)

Epoch 1/80


2026-09-21 17:07:20.082619: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:07:20.082913: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 15ms/step - loss: 0.2693 - val_loss: 0.2966
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.2111 - val_loss: 0.3064
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.2116 - val_loss: 0.2932
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.2114 - val_loss: 0.2986
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.2184 - val_loss: 0.3061
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.2146 - val_loss: 0.3040
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2158 - val_loss: 0.3255
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.2200 - val_loss: 0.3304
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.2227 - val_loss: 0.3185
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.2439 - val_loss: 0.4197
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.2617 - val_loss: 0.3450


2026-09-21 17:08:02.753165: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:08:02.753414: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 60 min
  Family:  F3
  Architecture: RNN_C1 (frozen)


...


Training history around callback-selected best epoch
----------------------------------------------------


,epoch,loss,val_loss,is_callback_best_epoch
0,1,0.269268,0.296586,False
1,2,0.211063,0.306362,False
2,3,0.211551,0.293245,True
3,4,0.211410,0.298624,False
4,5,0.218440,0.306117,False
5,6,0.214559,0.303968,False



60-min validation F2 -> F3 comparison:
  F2 MAE: 0.467074 kW
  F3 MAE: 0.465647 kW
  Relative MAE change: +0.3054%
  Validation only — not a test-set result.

60-minute LSTM validation progression:


,feature_family,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,F0,10,0.465383,0.869118,0.799001
1,F1,4,0.461840,0.894092,0.787283
2,F2,1,0.467074,0.855104,0.805430
3,F3,3,0.465647,0.867028,0.799966



Completed Stage-2 combinations: 14 / 15

Held-out test evaluation:
  NOT PERFORMED

LSTM 60-min F3 epoch-selection run: PASSED


In [43]:
# ============================================================
# Step 40: FINAL LSTM Stage-2 epoch-selection run
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F4
#
# Frozen architecture:
#       RNN_C1
#       32 LSTM units
#       1 recurrent layer
#       dropout = 0
#       fusion dense = 32
#
#
# F4 = F3 + 97 canonical Panchang one-hot indicators:
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 categorical indicators
#
#
# Auxiliary width:
#
#       F3 =  30
#       F4 = 127
#
#
# This completes the third and final validation-stage
# F3 -> F4 Panchang comparison.
#
#
# TRAIN:
#       model fitting
#
# VALIDATION:
#       best-epoch selection
#
# TEST:
#       NOT ACCESSED
# ============================================================


# ------------------------------------------------------------
# 1. Run frozen LSTM architecture on 60-minute F4
# ------------------------------------------------------------

(
    lstm_60_f4_result,
    lstm_60_f4_history,
) = run_validation_candidate(

    model_name="LSTM",

    candidate_config=(
        lstm_selected_candidate_config
    ),

    horizon_minutes=60,

    feature_family="F4",

    verbose=2,
)


# ------------------------------------------------------------
# 2. Expected parameter count
#
# LSTM sequence branch:
#
#       5,376
#
# Fusion input:
#
#       sequence embedding =  32
#       F4 auxiliary       = 127
#
#       total              = 159
#
# Fusion Dense:
#
#       159 × 32 + 32
#       = 5,120
#
# Output:
#
#       32 × 1 + 1
#       = 33
#
# Total:
#
#       5,376 + 5,120 + 33
#       = 10,529
# ------------------------------------------------------------

EXPECTED_LSTM_F4_PARAMETERS = (
    10_529
)


assert (
    lstm_60_f4_result[
        "trainable_parameter_count"
    ]
    == EXPECTED_LSTM_F4_PARAMETERS
)


# ------------------------------------------------------------
# 3. Strong frozen-protocol checks
# ------------------------------------------------------------

assert (
    lstm_60_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_60_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_60_f4_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_60_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_60_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_60_f4_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_60_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    1
    <= lstm_60_f4_result[
        "best_epoch"
    ]
    <= lstm_60_f4_result[
        "epochs_ran"
    ]
    <= DEEP_MAX_EPOCHS
)


for metric_name in [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "fit_seconds",
]:

    assert np.isfinite(
        lstm_60_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 4. Label and store Stage-2 result
# ------------------------------------------------------------

lstm_60_f4_result[
    "epoch_selection_source"
] = (
    "stage2_validation_run"
)


lstm_epoch_selection_results[
    (
        60,
        "F4",
    )
] = (
    lstm_60_f4_result.copy()
)


lstm_epoch_selection_histories[
    (
        60,
        "F4",
    )
] = (
    lstm_60_f4_history.copy()
)


# ------------------------------------------------------------
# 5. Verify COMPLETE 3 × 5 Stage-2 grid
#
# Horizons:
#       15, 30, 60
#
# Families:
#       F0, F1, F2, F3, F4
#
# Total:
#       15 combinations
# ------------------------------------------------------------

expected_lstm_stage2_keys = {
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes in [
        15,
        30,
        60,
    ]

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]
}


assert set(
    lstm_epoch_selection_results.keys()
) == expected_lstm_stage2_keys


assert len(
    lstm_epoch_selection_results
) == 15


assert len(
    lstm_epoch_selection_histories
) == 15


# ------------------------------------------------------------
# 6. Display new F4 result
# ------------------------------------------------------------

print(
    "\nLSTM Stage-2 epoch selection"
)

print(
    "----------------------------"
)

print(
    "Completed combination:"
)

print(
    "  Horizon: 60 min"
)

print(
    "  Family:  F4"
)

print(
    "  Architecture: RNN_C1 (frozen)"
)


display(
    pd.DataFrame(
        [
            lstm_60_f4_result
        ]
    )[
        [
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "callback_best_val_loss_scaled",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
            "epoch_selection_source",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Show history around callback-selected best epoch
# ------------------------------------------------------------

best_epoch_60_f4 = (
    lstm_60_f4_result[
        "best_epoch"
    ]
)


history_window_start = max(
    0,
    best_epoch_60_f4 - 4,
)


history_window_end = min(
    len(
        lstm_60_f4_history
    ),
    best_epoch_60_f4 + 3,
)


print(
    "\nTraining history around callback-selected best epoch"
)

print(
    "----------------------------------------------------"
)


display(
    lstm_60_f4_history.iloc[
        history_window_start:
        history_window_end
    ][
        [
            "epoch",
            "loss",
            "val_loss",
            "is_callback_best_epoch",
        ]
    ]
)


# ------------------------------------------------------------
# 8. KEY 60-minute validation comparison:
#
#       F3 -> F4
#
# Positive:
#       F4 has lower MAE.
#
# Negative:
#       F4 has higher MAE.
#
# Still VALIDATION ONLY.
# ------------------------------------------------------------

f3_validation_mae_60 = (
    lstm_epoch_selection_results[
        (
            60,
            "F3",
        )
    ][
        "validation_MAE_kW"
    ]
)


f4_validation_mae_60 = (
    lstm_60_f4_result[
        "validation_MAE_kW"
    ]
)


f3_to_f4_validation_improvement_pct_60 = (
    (
        f3_validation_mae_60
        - f4_validation_mae_60
    )
    /
    f3_validation_mae_60
    * 100.0
)


print(
    "\n60-min validation F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_validation_mae_60:.6f} kW"
)

print(
    f"  F4 MAE: {f4_validation_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_validation_improvement_pct_60:+.4f}%",
)

print(
    "  Validation only — not a held-out test conclusion."
)


# ------------------------------------------------------------
# 9. Complete 60-minute validation progression
# ------------------------------------------------------------

lstm_60min_validation_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "auxiliary_dim":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "auxiliary_dim"
                ],

            "best_epoch":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "best_epoch"
                ],

            "validation_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                lstm_epoch_selection_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "validation_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 60-minute LSTM validation progression:"
)


display(
    lstm_60min_validation_progress
)


# ------------------------------------------------------------
# 10. Complete cross-horizon F3 -> F4 validation summary
# ------------------------------------------------------------

lstm_f3_f4_validation_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "F3_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F3",
                    )
                ][
                    "validation_MAE_kW"
                ],

            "F4_MAE_kW":
                lstm_epoch_selection_results[
                    (
                        horizon_minutes,
                        "F4",
                    )
                ][
                    "validation_MAE_kW"
                ],
        }

        for horizon_minutes in [
            15,
            30,
            60,
        ]
    ]
)


lstm_f3_f4_validation_summary[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        lstm_f3_f4_validation_summary[
            "F3_MAE_kW"
        ]
        -
        lstm_f3_f4_validation_summary[
            "F4_MAE_kW"
        ]
    )
    /
    lstm_f3_f4_validation_summary[
        "F3_MAE_kW"
    ]
    * 100.0
)


print(
    "\nComplete LSTM F3 -> F4 validation summary:"
)


display(
    lstm_f3_f4_validation_summary
)


# ------------------------------------------------------------
# 11. Create complete Stage-2 audit table
# ------------------------------------------------------------

lstm_epoch_selection_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "auxiliary_dim":
                result[
                    "auxiliary_dim"
                ],

            "trainable_parameter_count":
                result[
                    "trainable_parameter_count"
                ],

            "epochs_ran":
                result[
                    "epochs_ran"
                ],

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "validation_MAE_kW":
                result[
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                result[
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                result[
                    "validation_R2"
                ],

            "epoch_selection_source":
                result[
                    "epoch_selection_source"
                ],
        }

        for (
            horizon_minutes,
            feature_family,
        ), result
        in lstm_epoch_selection_results.items()
    ]
)


lstm_epoch_selection_summary = (
    lstm_epoch_selection_summary
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    lstm_epoch_selection_summary
) == 15


print(
    "\nComplete LSTM Stage-2 epoch-selection table:"
)


display(
    lstm_epoch_selection_summary
)


# ------------------------------------------------------------
# 12. Explicitly confirm that every selected epoch is valid
# ------------------------------------------------------------

assert (
    lstm_epoch_selection_summary[
        "best_epoch"
    ]
    >= 1
).all()


assert (
    lstm_epoch_selection_summary[
        "best_epoch"
    ]
    <= DEEP_MAX_EPOCHS
).all()


# ------------------------------------------------------------
# 13. Final progress report
# ------------------------------------------------------------

print(
    "\nCompleted Stage-2 combinations:",
    len(
        lstm_epoch_selection_results
    ),
    "/ 15"
)


print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


print(
    "\nAll LSTM Stage-2 validation runs complete: PASSED"
)

Epoch 1/80


2026-09-21 17:09:16.668264: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:09:16.668537: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 6s - 17ms/step - loss: 0.2585 - val_loss: 0.3027
Epoch 2/80
341/341 - 4s - 11ms/step - loss: 0.2079 - val_loss: 0.2925
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.2071 - val_loss: 0.3069
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.2097 - val_loss: 0.3137
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.2077 - val_loss: 0.3130
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.2075 - val_loss: 0.3094
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2071 - val_loss: 0.3359
Epoch 8/80
341/341 - 4s - 12ms/step - loss: 0.2093 - val_loss: 0.3193
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.2111 - val_loss: 0.3188
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.2219 - val_loss: 0.3429


2026-09-21 17:09:56.336289: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:09:56.336551: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


LSTM Stage-2 epoch selection
----------------------------
Completed combination:
  Horizon: 60 min
  Family:  F4
  Architecture: RNN_C1 (frozen)


...


Complete LSTM F3 -> F4 validation summary:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.292315,0.293711,-0.477763
1,30,0.380316,0.382804,-0.654289
2,60,0.465647,0.464525,0.241009



Complete LSTM Stage-2 epoch-selection table:


,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,epoch_selection_source
0,15,F0,1,6497,15,7,0.289374,0.650396,0.887437,stage2_validation_run
1,15,F1,16,6977,9,1,0.291128,0.638846,0.891400,stage2_validation_run
2,15,F2,21,7137,10,2,0.297091,0.649842,0.887629,stage2_validation_run
3,15,F3,30,7425,10,2,0.292315,0.641045,0.890651,stage2_validation_run
4,15,F4,127,10529,15,7,0.293711,0.638679,0.891457,stage2_validation_run
5,30,F0,1,6497,18,10,0.366990,0.741797,0.853577,stage2_validation_run
6,30,F1,16,6977,9,1,0.362409,0.735007,0.856246,stage2_validation_run
7,30,F2,21,7137,14,6,0.366695,0.749496,0.850522,stage2_validation_run
8,30,F3,30,7425,13,5,0.380316,0.757902,0.847150,stage2_validation_run
9,30,F4,127,10529,9,1,0.382804,0.756728,0.847624,stage2_validation_run



Completed Stage-2 combinations: 15 / 15

Held-out test evaluation:
  NOT PERFORMED

All LSTM Stage-2 validation runs complete: PASSED


In [44]:
# ============================================================
# Step 41: Freeze the complete LSTM model-selection protocol
# ============================================================
#
# At this point:
#
#   Architecture selection:
#       COMPLETE
#
#   Epoch selection:
#       COMPLETE for all 15 horizon × family combinations
#
#   Held-out TEST evaluation:
#       NOT STARTED
#
#
# This cell creates a permanent methodological checkpoint
# BEFORE any TRAIN + VALIDATION refitting or TEST prediction.
#
#
# Frozen architecture:
#
#       model                  = LSTM
#       candidate              = RNN_C1
#       units                  = 32
#       recurrent layers       = 1
#       dropout                = 0.0
#       fusion dense units     = 32
#
#
# Frozen epoch grid:
#
#              F0   F1   F2   F3   F4
#
#       15m     7    1    2    2    7
#       30m    10    1    6    5    1
#       60m    10    4    1    3    2
#
#
# IMPORTANT:
#
# After this checkpoint, none of these values should be altered
# in response to held-out test performance.
# ============================================================


from types import MappingProxyType
import json


# ------------------------------------------------------------
# 1. Define the exact expected experimental grid
# ------------------------------------------------------------

LSTM_FROZEN_HORIZONS = (
    15,
    30,
    60,
)


LSTM_FROZEN_FEATURE_FAMILIES = (
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
)


expected_lstm_grid = {
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes
    in LSTM_FROZEN_HORIZONS

    for feature_family
    in LSTM_FROZEN_FEATURE_FAMILIES
}


assert len(
    expected_lstm_grid
) == 15


# ------------------------------------------------------------
# 2. Verify that Stage-2 contains the complete exact grid
# ------------------------------------------------------------

assert set(
    lstm_epoch_selection_results.keys()
) == expected_lstm_grid


assert len(
    lstm_epoch_selection_results
) == 15


assert len(
    lstm_epoch_selection_histories
) == 15


# ------------------------------------------------------------
# 3. Freeze the selected architecture
#
# MappingProxyType provides a read-only mapping so accidental
# assignment such as:
#
#       LSTM_FROZEN_ARCHITECTURE["units"] = 64
#
# will fail.
# ------------------------------------------------------------

LSTM_FROZEN_ARCHITECTURE = MappingProxyType(
    {
        "model":
            "LSTM",

        "candidate_id":
            str(
                lstm_selected_candidate_id
            ),

        "units":
            int(
                lstm_selected_candidate_config[
                    "units"
                ]
            ),

        "n_recurrent_layers":
            int(
                lstm_selected_candidate_config[
                    "n_recurrent_layers"
                ]
            ),

        "dropout":
            float(
                lstm_selected_candidate_config[
                    "dropout"
                ]
            ),

        "fusion_dense_units":
            int(
                lstm_selected_candidate_config[
                    "fusion_dense_units"
                ]
            ),
    }
)


# ------------------------------------------------------------
# 4. Verify exact architecture before freezing epochs
# ------------------------------------------------------------

assert dict(
    LSTM_FROZEN_ARCHITECTURE
) == {
    "model":
        "LSTM",

    "candidate_id":
        "RNN_C1",

    "units":
        32,

    "n_recurrent_layers":
        1,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,
}


# ------------------------------------------------------------
# 5. Extract the 15 validation-selected epoch counts
# ------------------------------------------------------------

lstm_best_epochs_mutable = {
    (
        horizon_minutes,
        feature_family,
    ):
        int(
            lstm_epoch_selection_results[
                (
                    horizon_minutes,
                    feature_family,
                )
            ][
                "best_epoch"
            ]
        )

    for horizon_minutes
    in LSTM_FROZEN_HORIZONS

    for feature_family
    in LSTM_FROZEN_FEATURE_FAMILIES
}


# ------------------------------------------------------------
# 6. Strong exact-value audit
#
# These values come directly from the completed Stage-2 runs.
# ------------------------------------------------------------

expected_lstm_best_epochs = {
    (15, "F0"): 7,
    (15, "F1"): 1,
    (15, "F2"): 2,
    (15, "F3"): 2,
    (15, "F4"): 7,

    (30, "F0"): 10,
    (30, "F1"): 1,
    (30, "F2"): 6,
    (30, "F3"): 5,
    (30, "F4"): 1,

    (60, "F0"): 10,
    (60, "F1"): 4,
    (60, "F2"): 1,
    (60, "F3"): 3,
    (60, "F4"): 2,
}


assert (
    lstm_best_epochs_mutable
    == expected_lstm_best_epochs
)


# ------------------------------------------------------------
# 7. Freeze the epoch mapping
# ------------------------------------------------------------

LSTM_FROZEN_BEST_EPOCHS = MappingProxyType(
    lstm_best_epochs_mutable.copy()
)


assert len(
    LSTM_FROZEN_BEST_EPOCHS
) == 15


# ------------------------------------------------------------
# 8. Create a clean frozen epoch table
# ------------------------------------------------------------

lstm_frozen_epoch_table = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "feature_family":
                feature_family,

            "best_epoch":
                LSTM_FROZEN_BEST_EPOCHS[
                    (
                        horizon_minutes,
                        feature_family,
                    )
                ],

            "auxiliary_dim":
                int(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "auxiliary_dim"
                    ]
                ),

            "trainable_parameter_count":
                int(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "trainable_parameter_count"
                    ]
                ),

            "validation_MAE_kW":
                float(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "validation_MAE_kW"
                    ]
                ),

            "validation_RMSE_kW":
                float(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "validation_RMSE_kW"
                    ]
                ),

            "validation_R2":
                float(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "validation_R2"
                    ]
                ),

            "epoch_selection_source":
                str(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            feature_family,
                        )
                    ][
                        "epoch_selection_source"
                    ]
                ),
        }

        for horizon_minutes
        in LSTM_FROZEN_HORIZONS

        for feature_family
        in LSTM_FROZEN_FEATURE_FAMILIES
    ]
)


assert len(
    lstm_frozen_epoch_table
) == 15


# ------------------------------------------------------------
# 9. Create a compact epoch matrix for visual verification
# ------------------------------------------------------------

lstm_frozen_epoch_matrix = (
    lstm_frozen_epoch_table
    .pivot(
        index="horizon_minutes",
        columns="feature_family",
        values="best_epoch",
    )
    .loc[
        list(
            LSTM_FROZEN_HORIZONS
        ),
        list(
            LSTM_FROZEN_FEATURE_FAMILIES
        ),
    ]
)


expected_epoch_matrix = pd.DataFrame(
    {
        "F0": [7, 10, 10],
        "F1": [1, 1, 4],
        "F2": [2, 6, 1],
        "F3": [2, 5, 3],
        "F4": [7, 1, 2],
    },
    index=[
        15,
        30,
        60,
    ],
)


expected_epoch_matrix.index.name = (
    "horizon_minutes"
)


expected_epoch_matrix.columns.name = (
    "feature_family"
)


pd.testing.assert_frame_equal(
    lstm_frozen_epoch_matrix,
    expected_epoch_matrix,
)


# ------------------------------------------------------------
# 10. Recompute the frozen validation F3 -> F4 summary
# ------------------------------------------------------------

lstm_frozen_f3_f4_validation_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "F3_MAE_kW":
                float(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            "F3",
                        )
                    ][
                        "validation_MAE_kW"
                    ]
                ),

            "F4_MAE_kW":
                float(
                    lstm_epoch_selection_results[
                        (
                            horizon_minutes,
                            "F4",
                        )
                    ][
                        "validation_MAE_kW"
                    ]
                ),
        }

        for horizon_minutes
        in LSTM_FROZEN_HORIZONS
    ]
)


lstm_frozen_f3_f4_validation_summary[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        lstm_frozen_f3_f4_validation_summary[
            "F3_MAE_kW"
        ]
        -
        lstm_frozen_f3_f4_validation_summary[
            "F4_MAE_kW"
        ]
    )
    /
    lstm_frozen_f3_f4_validation_summary[
        "F3_MAE_kW"
    ]
    * 100.0
)


# ------------------------------------------------------------
# 11. Build serializable frozen-protocol metadata
#
# Tuple dictionary keys are converted to records because JSON
# requires string-compatible object keys.
# ------------------------------------------------------------

lstm_frozen_protocol_metadata = {
    "model":
        "LSTM",

    "architecture_selection_horizon_minutes":
        60,

    "architecture_selection_feature_family":
        "F0",

    "architecture":
        dict(
            LSTM_FROZEN_ARCHITECTURE
        ),

    "training_protocol":
        {
            "batch_size":
                int(
                    DEEP_BATCH_SIZE
                ),

            "max_validation_epochs":
                int(
                    DEEP_MAX_EPOCHS
                ),

            "optimizer":
                "Adam",

            "learning_rate":
                1e-3,

            "clipnorm":
                1.0,

            "loss":
                "MAE_on_standardized_target",

            "early_stopping_monitor":
                "val_loss",

            "early_stopping_patience":
                8,

            "early_stopping_min_delta":
                1e-4,

            "restore_best_weights":
                True,

            "shuffle_training_samples":
                True,

            "random_seed":
                42,
        },

    "final_refit_policy":
        {
            "training_population":
                "train_plus_validation",

            "epochs":
                "frozen_validation_selected_best_epoch",

            "early_stopping":
                False,

            "scalers":
                "original_train_only_scalers",

            "test_used_during_refit":
                False,
        },

    "best_epochs":
        [
            {
                "horizon_minutes":
                    int(
                        horizon_minutes
                    ),

                "feature_family":
                    feature_family,

                "best_epoch":
                    int(
                        LSTM_FROZEN_BEST_EPOCHS[
                            (
                                horizon_minutes,
                                feature_family,
                            )
                        ]
                    ),
            }

            for horizon_minutes
            in LSTM_FROZEN_HORIZONS

            for feature_family
            in LSTM_FROZEN_FEATURE_FAMILIES
        ],
}


# ------------------------------------------------------------
# 12. Persist the methodological checkpoint
# ------------------------------------------------------------

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


lstm_frozen_epoch_csv_path = (
    RESULTS_DIR
    /
    "lstm_frozen_epoch_selection.csv"
)


lstm_frozen_protocol_json_path = (
    RESULTS_DIR
    /
    "lstm_frozen_protocol.json"
)


lstm_frozen_f3_f4_csv_path = (
    RESULTS_DIR
    /
    "lstm_validation_f3_f4_summary.csv"
)


lstm_frozen_epoch_table.to_csv(
    lstm_frozen_epoch_csv_path,
    index=False,
)


lstm_frozen_f3_f4_validation_summary.to_csv(
    lstm_frozen_f3_f4_csv_path,
    index=False,
)


with open(
    lstm_frozen_protocol_json_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        lstm_frozen_protocol_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 13. Confirm checkpoint files exist and are non-empty
# ------------------------------------------------------------

for checkpoint_path in [
    lstm_frozen_epoch_csv_path,
    lstm_frozen_protocol_json_path,
    lstm_frozen_f3_f4_csv_path,
]:

    assert (
        checkpoint_path.exists()
    )

    assert (
        checkpoint_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 14. Report frozen architecture
# ------------------------------------------------------------

print(
    "Frozen LSTM architecture"
)

print(
    "------------------------"
)


for key, value in (
    LSTM_FROZEN_ARCHITECTURE.items()
):

    print(
        f"  {key}: {value}"
    )


# ------------------------------------------------------------
# 15. Report frozen epoch matrix
# ------------------------------------------------------------

print(
    "\nFrozen epoch matrix"
)

print(
    "-------------------"
)


display(
    lstm_frozen_epoch_matrix
)


# ------------------------------------------------------------
# 16. Report complete audit table
# ------------------------------------------------------------

print(
    "\nFrozen LSTM epoch-selection audit"
)

print(
    "---------------------------------"
)


display(
    lstm_frozen_epoch_table
)


# ------------------------------------------------------------
# 17. Report F3 -> F4 validation evidence
# ------------------------------------------------------------

print(
    "\nFrozen validation F3 -> F4 summary"
)

print(
    "----------------------------------"
)


display(
    lstm_frozen_f3_f4_validation_summary
)


# ------------------------------------------------------------
# 18. Report persisted files
# ------------------------------------------------------------

print(
    "\nSaved methodological checkpoint:"
)


print(
    " ",
    lstm_frozen_epoch_csv_path,
)


print(
    " ",
    lstm_frozen_protocol_json_path,
)


print(
    " ",
    lstm_frozen_f3_f4_csv_path,
)


# ------------------------------------------------------------
# 19. Explicit methodological boundary
# ------------------------------------------------------------

print(
    "\nModel-selection status:"
)

print(
    "  Architecture: FROZEN"
)

print(
    "  Epoch counts: FROZEN"
)

print(
    "  Feature definitions: FROZEN"
)

print(
    "  Scaling protocol: FROZEN"
)


print(
    "\nHeld-out test evaluation:"
)

print(
    "  NOT PERFORMED"
)


# ------------------------------------------------------------
# 20. Final checkpoint
# ------------------------------------------------------------

print(
    "\nLSTM selection protocol frozen: PASSED"
)

Frozen LSTM architecture
------------------------
  model: LSTM
  candidate_id: RNN_C1
  units: 32
  n_recurrent_layers: 1
  dropout: 0.0
  fusion_dense_units: 32

Frozen epoch matrix
-------------------


feature_family,F0,F1,F2,F3,F4
horizon_minutes,,,,,
15,7,1,2,2,7
30,10,1,6,5,1
60,10,4,1,3,2



Frozen LSTM epoch-selection audit
---------------------------------


,horizon_minutes,feature_family,best_epoch,auxiliary_dim,trainable_parameter_count,validation_MAE_kW,validation_RMSE_kW,validation_R2,epoch_selection_source
0,15,F0,7,1,6497,0.289374,0.650396,0.887437,stage2_validation_run
1,15,F1,1,16,6977,0.291128,0.638846,0.891400,stage2_validation_run
2,15,F2,2,21,7137,0.297091,0.649842,0.887629,stage2_validation_run
3,15,F3,2,30,7425,0.292315,0.641045,0.890651,stage2_validation_run
4,15,F4,7,127,10529,0.293711,0.638679,0.891457,stage2_validation_run
5,30,F0,10,1,6497,0.366990,0.741797,0.853577,stage2_validation_run
6,30,F1,1,16,6977,0.362409,0.735007,0.856246,stage2_validation_run
7,30,F2,6,21,7137,0.366695,0.749496,0.850522,stage2_validation_run
8,30,F3,5,30,7425,0.380316,0.757902,0.847150,stage2_validation_run
9,30,F4,1,127,10529,0.382804,0.756728,0.847624,stage2_validation_run



Frozen validation F3 -> F4 summary
----------------------------------


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.292315,0.293711,-0.477763
1,30,0.380316,0.382804,-0.654289
2,60,0.465647,0.464525,0.241009



Saved methodological checkpoint:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_frozen_epoch_selection.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_frozen_protocol.json
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_validation_f3_f4_summary.csv

Model-selection status:
  Architecture: FROZEN
  Epoch counts: FROZEN
  Feature definitions: FROZEN
  Scaling protocol: FROZEN

Held-out test evaluation:
  NOT PERFORMED

LSTM selection protocol frozen: PASSED


In [53]:
# ============================================================
# Step 42B: Inspect the existing model-factory API
# ============================================================
#
# Purpose
# -------
#
# Before defining the final TRAIN + VALIDATION refit helper,
# inspect the exact API of the model factory already defined
# and successfully used earlier in this notebook.
#
# This prevents us from guessing argument names.
#
#
# IMPORTANT
# ---------
#
# No model training occurs here.
#
# No held-out TEST data are accessed here.
# ============================================================


import inspect


# ------------------------------------------------------------
# 1. Display the exact model-factory signature
# ------------------------------------------------------------

print(
    "build_deep_forecasting_model signature:"
)

print(
    inspect.signature(
        build_deep_forecasting_model
    )
)


# ------------------------------------------------------------
# 2. Display each parameter explicitly
# ------------------------------------------------------------

print(
    "\nParameter names:"
)


for parameter_name, parameter in (
    inspect.signature(
        build_deep_forecasting_model
    ).parameters.items()
):

    print(
        f"  {parameter_name}:"
        f" default={parameter.default}"
    )


# ------------------------------------------------------------
# 3. Methodological status
# ------------------------------------------------------------

print(
    "\nHeld-out TEST evaluation:"
)

print(
    "  NOT PERFORMED"
)


print(
    "\nStep 42B API inspection: PASSED"
)

build_deep_forecasting_model signature:
(model_name: str, auxiliary_dim: int, candidate_config: dict, seed: int = 42)

Parameter names:
  model_name: default=<class 'inspect._empty'>
  auxiliary_dim: default=<class 'inspect._empty'>
  candidate_config: default=<class 'inspect._empty'>
  seed: default=42

Held-out TEST evaluation:
  NOT PERFORMED

Step 42B API inspection: PASSED


In [54]:
# ============================================================
# Step 42C: Inspect the regression-metric helper API
# ============================================================
#
# Purpose
# -------
#
# Before defining the final TRAIN + VALIDATION refit / TEST
# helper, inspect the exact API of:
#
#       compute_regression_metrics_kw(...)
#
# This prevents us from guessing its parameter names.
#
#
# IMPORTANT
# ---------
#
# No model training occurs here.
#
# No held-out TEST data are accessed here.
# ============================================================


import inspect


# ------------------------------------------------------------
# 1. Display the exact metric-helper signature
# ------------------------------------------------------------

print(
    "compute_regression_metrics_kw signature:"
)

print(
    inspect.signature(
        compute_regression_metrics_kw
    )
)


# ------------------------------------------------------------
# 2. Display each parameter explicitly
# ------------------------------------------------------------

print(
    "\nParameter names:"
)


for parameter_name, parameter in (
    inspect.signature(
        compute_regression_metrics_kw
    ).parameters.items()
):

    print(
        f"  {parameter_name}:"
        f" default={parameter.default}"
    )


# ------------------------------------------------------------
# 3. Methodological status
# ------------------------------------------------------------

print(
    "\nHeld-out TEST evaluation:"
)

print(
    "  NOT PERFORMED"
)


print(
    "\nStep 42C metric API inspection: PASSED"
)

compute_regression_metrics_kw signature:
(y_true_kw, y_pred_kw)

Parameter names:
  y_true_kw: default=<class 'inspect._empty'>
  y_pred_kw: default=<class 'inspect._empty'>

Held-out TEST evaluation:
  NOT PERFORMED

Step 42C metric API inspection: PASSED


In [55]:
# ============================================================
# Step 42D: Inspect remaining final-evaluation helper APIs
# ============================================================
#
# We have already confirmed:
#
#   build_deep_forecasting_model(
#       model_name,
#       auxiliary_dim,
#       candidate_config,
#       seed=42,
#   )
#
#   compute_regression_metrics_kw(
#       y_true_kw,
#       y_pred_kw,
#   )
#
#
# Before defining the final refit / TEST helper, verify:
#
#   1. get_deep_split_data(...) signature
#   2. inverse_transform_target_to_kw(...) signature
#   3. exact keys returned by get_deep_split_data(...)
#
#
# IMPORTANT
# ---------
#
# We inspect ONLY the TRAIN split below.
#
# No model training occurs.
# No VALIDATION data are needed.
# No held-out TEST data are accessed.
# ============================================================


import inspect


# ------------------------------------------------------------
# 1. Inspect get_deep_split_data(...)
# ------------------------------------------------------------

print(
    "get_deep_split_data signature:"
)

print(
    inspect.signature(
        get_deep_split_data
    )
)


print(
    "\nget_deep_split_data parameters:"
)


for parameter_name, parameter in (
    inspect.signature(
        get_deep_split_data
    ).parameters.items()
):

    print(
        f"  {parameter_name}:"
        f" default={parameter.default}"
    )


# ------------------------------------------------------------
# 2. Inspect inverse_transform_target_to_kw(...)
# ------------------------------------------------------------

print(
    "\ninverse_transform_target_to_kw signature:"
)

print(
    inspect.signature(
        inverse_transform_target_to_kw
    )
)


print(
    "\ninverse_transform_target_to_kw parameters:"
)


for parameter_name, parameter in (
    inspect.signature(
        inverse_transform_target_to_kw
    ).parameters.items()
):

    print(
        f"  {parameter_name}:"
        f" default={parameter.default}"
    )


# ------------------------------------------------------------
# 3. Inspect an existing TRAIN split only
#
# Use the simplest canonical combination:
#
#       horizon = 15 minutes
#       family  = F0
#
# This does NOT touch the held-out TEST split.
# ------------------------------------------------------------

train_structure_probe = get_deep_split_data(
    horizon_minutes=15,
    feature_family="F0",
    split_name="train",
)


print(
    "\nTRAIN split returned keys:"
)

print(
    sorted(
        train_structure_probe.keys()
    )
)


# ------------------------------------------------------------
# 4. Display type / shape / dtype for every returned item
# ------------------------------------------------------------

print(
    "\nTRAIN split structure:"
)


for key, value in (
    train_structure_probe.items()
):

    value_type = type(
        value
    ).__name__


    value_shape = getattr(
        value,
        "shape",
        None,
    )


    value_dtype = getattr(
        value,
        "dtype",
        None,
    )


    print(
        f"  {key}:"
        f" type={value_type},"
        f" shape={value_shape},"
        f" dtype={value_dtype}"
    )


# ------------------------------------------------------------
# 5. Verify the core arrays we require actually exist
# ------------------------------------------------------------

required_train_keys = {
    "sequence",
    "auxiliary",
    "target_scaled",
    "target_kw",
}


assert required_train_keys.issubset(
    train_structure_probe.keys()
)


# ------------------------------------------------------------
# 6. Verify canonical 15-minute F0 TRAIN shapes
# ------------------------------------------------------------

assert (
    train_structure_probe[
        "sequence"
    ].shape
    ==
    (
        87_261,
        13,
        9,
    )
)


assert (
    train_structure_probe[
        "auxiliary"
    ].shape
    ==
    (
        87_261,
        1,
    )
)


assert (
    train_structure_probe[
        "target_scaled"
    ].shape[
        0
    ]
    == 87_261
)


assert (
    train_structure_probe[
        "target_kw"
    ].shape[
        0
    ]
    == 87_261
)


# ------------------------------------------------------------
# 7. Clean temporary reference
# ------------------------------------------------------------

del train_structure_probe


# ------------------------------------------------------------
# 8. Methodological status
# ------------------------------------------------------------

print(
    "\nHeld-out TEST evaluation:"
)

print(
    "  NOT PERFORMED"
)


print(
    "\nStep 42D remaining API / structure inspection: PASSED"
)

get_deep_split_data signature:
(horizon_minutes: int, feature_family: str, split_name: str)

get_deep_split_data parameters:
  horizon_minutes: default=<class 'inspect._empty'>
  feature_family: default=<class 'inspect._empty'>
  split_name: default=<class 'inspect._empty'>

inverse_transform_target_to_kw signature:
(scaled_values, horizon_minutes: int)

inverse_transform_target_to_kw parameters:
  scaled_values: default=<class 'inspect._empty'>
  horizon_minutes: default=<class 'inspect._empty'>

TRAIN split returned keys:
['auxiliary', 'n_samples', 'sequence', 'target_kw', 'target_scaled']

TRAIN split structure:
  sequence: type=ndarray, shape=(87261, 13, 9), dtype=float32
  auxiliary: type=ndarray, shape=(87261, 1), dtype=float32
  target_scaled: type=ndarray, shape=(87261,), dtype=float32
  target_kw: type=ndarray, shape=(87261,), dtype=float32
  n_samples: type=int, shape=None, dtype=None

Held-out TEST evaluation:
  NOT PERFORMED

Step 42D remaining API / structure inspection: P

In [56]:
# ============================================================
# Step 42E: Define the final LSTM refit / TEST machinery
# ============================================================
#
# All relevant APIs have now been inspected explicitly.
#
# Confirmed APIs
# --------------
#
# build_deep_forecasting_model(
#     model_name,
#     auxiliary_dim,
#     candidate_config,
#     seed=42,
# )
#
#
# get_deep_split_data(
#     horizon_minutes,
#     feature_family,
#     split_name,
# )
#
#
# inverse_transform_target_to_kw(
#     scaled_values,
#     horizon_minutes,
# )
#
#
# compute_regression_metrics_kw(
#     y_true_kw,
#     y_pred_kw,
# )
#
#
# Confirmed split-data keys
# -------------------------
#
#     sequence
#     auxiliary
#     target_scaled
#     target_kw
#     n_samples
#
#
# IMPORTANT
# ---------
#
# This cell:
#
#   1. defines TRAIN + VALIDATION assembly;
#   2. defines the final refit / held-out TEST function;
#   3. validates TRAIN + VALIDATION population sizes;
#   4. validates the frozen LSTM model factory;
#
# but:
#
#   NO final model is trained.
#   NO held-out TEST data are accessed.
# ============================================================


import time
import gc


# ------------------------------------------------------------
# 1. Explicit frozen auxiliary widths
#
# These are determined entirely by the already-frozen
# feature-family definitions.
# ------------------------------------------------------------

LSTM_EXPECTED_AUX_DIMS = {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


assert set(
    LSTM_EXPECTED_AUX_DIMS.keys()
) == set(
    LSTM_FROZEN_FEATURE_FAMILIES
)


# ------------------------------------------------------------
# 2. Reconstruct the frozen candidate configuration
#
# We construct this from the read-only frozen architecture
# rather than depending on an earlier mutable dictionary.
# ------------------------------------------------------------

LSTM_FINAL_CANDIDATE_CONFIG = {
    "candidate_id":
        str(
            LSTM_FROZEN_ARCHITECTURE[
                "candidate_id"
            ]
        ),

    "units":
        int(
            LSTM_FROZEN_ARCHITECTURE[
                "units"
            ]
        ),

    "n_recurrent_layers":
        int(
            LSTM_FROZEN_ARCHITECTURE[
                "n_recurrent_layers"
            ]
        ),

    "dropout":
        float(
            LSTM_FROZEN_ARCHITECTURE[
                "dropout"
            ]
        ),

    "fusion_dense_units":
        int(
            LSTM_FROZEN_ARCHITECTURE[
                "fusion_dense_units"
            ]
        ),
}


assert LSTM_FINAL_CANDIDATE_CONFIG == {
    "candidate_id": "RNN_C1",
    "units": 32,
    "n_recurrent_layers": 1,
    "dropout": 0.0,
    "fusion_dense_units": 32,
}


# ============================================================
# 3. Helper: assemble final TRAIN + VALIDATION population
# ============================================================

def get_final_refit_training_data(
    horizon_minutes: int,
    feature_family: str,
):
    """
    Assemble the final TRAIN + VALIDATION population.

    Important
    ---------
    The arrays returned by get_deep_split_data(...) have
    already been transformed using the ORIGINAL TRAIN-ONLY
    scaling statistics.

    Therefore this function only concatenates existing scaled
    arrays. It does NOT refit any scaler.

    TEST data are never accessed here.
    """

    # --------------------------------------------------------
    # Normalize inputs
    # --------------------------------------------------------

    horizon_minutes = int(
        horizon_minutes
    )

    feature_family = str(
        feature_family
    )


    # --------------------------------------------------------
    # Validate against the frozen experimental grid
    # --------------------------------------------------------

    assert horizon_minutes in (
        LSTM_FROZEN_HORIZONS
    )


    assert feature_family in (
        LSTM_FROZEN_FEATURE_FAMILIES
    )


    expected_auxiliary_dim = int(
        LSTM_EXPECTED_AUX_DIMS[
            feature_family
        ]
    )


    # --------------------------------------------------------
    # Retrieve TRAIN split
    # --------------------------------------------------------

    train_data = get_deep_split_data(
        horizon_minutes=
            horizon_minutes,

        feature_family=
            feature_family,

        split_name=
            "train",
    )


    # --------------------------------------------------------
    # Retrieve VALIDATION split
    # --------------------------------------------------------

    validation_data = get_deep_split_data(
        horizon_minutes=
            horizon_minutes,

        feature_family=
            feature_family,

        split_name=
            "validation",
    )


    # --------------------------------------------------------
    # Confirm the exact expected dictionary structure
    # --------------------------------------------------------

    required_keys = {
        "sequence",
        "auxiliary",
        "target_scaled",
        "target_kw",
        "n_samples",
    }


    assert required_keys.issubset(
        train_data.keys()
    )


    assert required_keys.issubset(
        validation_data.keys()
    )


    # --------------------------------------------------------
    # Read authoritative split counts
    # --------------------------------------------------------

    n_train = int(
        train_data[
            "n_samples"
        ]
    )


    n_validation = int(
        validation_data[
            "n_samples"
        ]
    )


    # --------------------------------------------------------
    # Cross-check counts against actual array dimensions
    # --------------------------------------------------------

    assert (
        train_data[
            "sequence"
        ].shape[
            0
        ]
        == n_train
    )


    assert (
        train_data[
            "auxiliary"
        ].shape[
            0
        ]
        == n_train
    )


    assert (
        train_data[
            "target_scaled"
        ].shape[
            0
        ]
        == n_train
    )


    assert (
        train_data[
            "target_kw"
        ].shape[
            0
        ]
        == n_train
    )


    assert (
        validation_data[
            "sequence"
        ].shape[
            0
        ]
        == n_validation
    )


    assert (
        validation_data[
            "auxiliary"
        ].shape[
            0
        ]
        == n_validation
    )


    assert (
        validation_data[
            "target_scaled"
        ].shape[
            0
        ]
        == n_validation
    )


    assert (
        validation_data[
            "target_kw"
        ].shape[
            0
        ]
        == n_validation
    )


    # --------------------------------------------------------
    # Canonical sequence dimensions
    # --------------------------------------------------------

    assert (
        train_data[
            "sequence"
        ].shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    assert (
        validation_data[
            "sequence"
        ].shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    # --------------------------------------------------------
    # Frozen feature-family auxiliary width
    # --------------------------------------------------------

    assert (
        train_data[
            "auxiliary"
        ].shape[
            1
        ]
        == expected_auxiliary_dim
    )


    assert (
        validation_data[
            "auxiliary"
        ].shape[
            1
        ]
        == expected_auxiliary_dim
    )


    # --------------------------------------------------------
    # Concatenate TRAIN followed by VALIDATION
    #
    # This preserves chronological population ordering before
    # Keras applies the already-frozen shuffle=True policy
    # during minibatch training.
    # --------------------------------------------------------

    sequence_trainval = np.ascontiguousarray(
        np.concatenate(
            [
                train_data[
                    "sequence"
                ],

                validation_data[
                    "sequence"
                ],
            ],
            axis=0,
        ),
        dtype=np.float32,
    )


    auxiliary_trainval = np.ascontiguousarray(
        np.concatenate(
            [
                train_data[
                    "auxiliary"
                ],

                validation_data[
                    "auxiliary"
                ],
            ],
            axis=0,
        ),
        dtype=np.float32,
    )


    target_scaled_trainval = np.ascontiguousarray(
        np.concatenate(
            [
                train_data[
                    "target_scaled"
                ],

                validation_data[
                    "target_scaled"
                ],
            ],
            axis=0,
        ),
        dtype=np.float32,
    )


    target_kw_trainval = np.ascontiguousarray(
        np.concatenate(
            [
                train_data[
                    "target_kw"
                ],

                validation_data[
                    "target_kw"
                ],
            ],
            axis=0,
        ),
        dtype=np.float32,
    )


    # --------------------------------------------------------
    # Combined population count
    # --------------------------------------------------------

    n_trainval = (
        n_train
        +
        n_validation
    )


    # --------------------------------------------------------
    # Final combined-array checks
    # --------------------------------------------------------

    assert (
        sequence_trainval.shape
        ==
        (
            n_trainval,
            13,
            9,
        )
    )


    assert (
        auxiliary_trainval.shape
        ==
        (
            n_trainval,
            expected_auxiliary_dim,
        )
    )


    assert (
        target_scaled_trainval.shape
        ==
        (
            n_trainval,
        )
    )


    assert (
        target_kw_trainval.shape
        ==
        (
            n_trainval,
        )
    )


    assert np.isfinite(
        sequence_trainval
    ).all()


    assert np.isfinite(
        auxiliary_trainval
    ).all()


    assert np.isfinite(
        target_scaled_trainval
    ).all()


    assert np.isfinite(
        target_kw_trainval
    ).all()


    # --------------------------------------------------------
    # Return final-fitting population
    # --------------------------------------------------------

    return {
        "sequence":
            sequence_trainval,

        "auxiliary":
            auxiliary_trainval,

        "target_scaled":
            target_scaled_trainval,

        "target_kw":
            target_kw_trainval,

        "n_train":
            n_train,

        "n_validation":
            n_validation,

        "n_trainval":
            n_trainval,

        "auxiliary_dim":
            expected_auxiliary_dim,
    }


# ============================================================
# 4. Final frozen refit + held-out TEST helper
# ============================================================
#
# IMPORTANT:
#
# Defining this function does NOT access the TEST set.
#
# TEST is accessed only later when this function is explicitly
# called for a particular horizon × family combination.
# ============================================================

def run_lstm_final_refit_and_test(
    horizon_minutes: int,
    feature_family: str,
    verbose: int = 2,
):
    """
    Run one final frozen LSTM experiment.

    Procedure
    ---------
    1. Retrieve frozen architecture and epoch count.
    2. Assemble TRAIN + VALIDATION.
    3. Retrieve the untouched TEST split.
    4. Reinitialize the model from seed 42.
    5. Train exactly for the frozen epoch count.
    6. Use no validation_data and no early stopping.
    7. Predict TEST once.
    8. Inverse-transform predictions to kW.
    9. Compute TEST MAE, RMSE, and R2.

    Returns
    -------
    result : dict
        Audit metadata and held-out TEST metrics.

    history_df : pandas.DataFrame
        Final TRAIN + VALIDATION loss history.

    test_actual_kw : numpy.ndarray
        Held-out true target values in kW.

    test_prediction_kw : numpy.ndarray
        Held-out predictions in kW.
    """

    # --------------------------------------------------------
    # A. Normalize inputs
    # --------------------------------------------------------

    horizon_minutes = int(
        horizon_minutes
    )

    feature_family = str(
        feature_family
    )


    experiment_key = (
        horizon_minutes,
        feature_family,
    )


    # --------------------------------------------------------
    # B. Verify this is one of the frozen 15 experiments
    # --------------------------------------------------------

    assert experiment_key in (
        LSTM_FROZEN_BEST_EPOCHS
    )


    # --------------------------------------------------------
    # C. Retrieve frozen epoch count
    # --------------------------------------------------------

    frozen_best_epoch = int(
        LSTM_FROZEN_BEST_EPOCHS[
            experiment_key
        ]
    )


    assert (
        1
        <= frozen_best_epoch
        <= DEEP_MAX_EPOCHS
    )


    # --------------------------------------------------------
    # D. Assemble final TRAIN + VALIDATION population
    # --------------------------------------------------------

    trainval_data = (
        get_final_refit_training_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,
        )
    )


    auxiliary_dim = int(
        trainval_data[
            "auxiliary_dim"
        ]
    )


    # --------------------------------------------------------
    # E. Retrieve held-out TEST split
    #
    # This is the first TEST access inside this function.
    # --------------------------------------------------------

    test_data = get_deep_split_data(
        horizon_minutes=
            horizon_minutes,

        feature_family=
            feature_family,

        split_name=
            "test",
    )


    # --------------------------------------------------------
    # F. Verify TEST structure
    # --------------------------------------------------------

    required_test_keys = {
        "sequence",
        "auxiliary",
        "target_scaled",
        "target_kw",
        "n_samples",
    }


    assert required_test_keys.issubset(
        test_data.keys()
    )


    n_test = int(
        test_data[
            "n_samples"
        ]
    )


    assert (
        n_test
        == 26_496
    )


    assert (
        test_data[
            "sequence"
        ].shape
        ==
        (
            n_test,
            13,
            9,
        )
    )


    assert (
        test_data[
            "auxiliary"
        ].shape
        ==
        (
            n_test,
            auxiliary_dim,
        )
    )


    assert (
        test_data[
            "target_scaled"
        ].shape
        ==
        (
            n_test,
        )
    )


    assert (
        test_data[
            "target_kw"
        ].shape
        ==
        (
            n_test,
        )
    )


    # --------------------------------------------------------
    # G. Reset TensorFlow/Keras state and reproducibility seed
    # --------------------------------------------------------

    reset_deep_model_state()


    # --------------------------------------------------------
    # H. Build a fresh frozen LSTM model
    #
    # Exact confirmed factory API.
    # --------------------------------------------------------

    model = build_deep_forecasting_model(

        model_name=
            "LSTM",

        auxiliary_dim=
            auxiliary_dim,

        candidate_config=
            LSTM_FINAL_CANDIDATE_CONFIG,

        seed=
            42,
    )


    # --------------------------------------------------------
    # I. Verify expected parameter count
    # --------------------------------------------------------

    expected_parameter_count = int(
        lstm_epoch_selection_results[
            experiment_key
        ][
            "trainable_parameter_count"
        ]
    )


    assert (
        model.count_params()
        == expected_parameter_count
    )


    # --------------------------------------------------------
    # J. Final TRAIN + VALIDATION refit
    #
    # Frozen protocol:
    #
    #   epochs          = selected before TEST
    #   shuffle         = True
    #   validation_data = NONE
    #   callbacks       = NONE
    #   early stopping  = NONE
    # --------------------------------------------------------

    final_fit_start = (
        time.perf_counter()
    )


    final_history = model.fit(

        x=[
            trainval_data[
                "sequence"
            ],

            trainval_data[
                "auxiliary"
            ],
        ],

        y=
            trainval_data[
                "target_scaled"
            ],

        epochs=
            frozen_best_epoch,

        batch_size=
            DEEP_BATCH_SIZE,

        shuffle=
            True,

        verbose=
            verbose,
    )


    final_fit_seconds = (
        time.perf_counter()
        -
        final_fit_start
    )


    # --------------------------------------------------------
    # K. Verify final fitting history
    # --------------------------------------------------------

    history_df = pd.DataFrame(
        final_history.history
    )


    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                history_df
            )
            + 1,
            dtype=int,
        ),
    )


    assert (
        len(
            history_df
        )
        == frozen_best_epoch
    )


    # There must be no validation-based model selection in
    # this final refit.
    assert (
        "val_loss"
        not in history_df.columns
    )


    # --------------------------------------------------------
    # L. Generate held-out TEST predictions
    # --------------------------------------------------------

    test_prediction_scaled = (
        model.predict(
            [
                test_data[
                    "sequence"
                ],

                test_data[
                    "auxiliary"
                ],
            ],

            batch_size=
                DEEP_BATCH_SIZE,

            verbose=0,
        )
        .reshape(
            -1
        )
    )


    assert (
        test_prediction_scaled.shape
        ==
        (
            n_test,
        )
    )


    assert np.isfinite(
        test_prediction_scaled
    ).all()


    # --------------------------------------------------------
    # M. Convert predictions back to physical kW
    #
    # Exact confirmed helper API.
    # --------------------------------------------------------

    test_prediction_kw = (
        inverse_transform_target_to_kw(

            scaled_values=
                test_prediction_scaled,

            horizon_minutes=
                horizon_minutes,
        )
    )


    test_prediction_kw = np.asarray(
        test_prediction_kw,
        dtype=np.float64,
    ).reshape(
        -1
    )


    test_actual_kw = np.asarray(
        test_data[
            "target_kw"
        ],
        dtype=np.float64,
    ).reshape(
        -1
    )


    assert (
        test_prediction_kw.shape
        ==
        test_actual_kw.shape
        ==
        (
            n_test,
        )
    )


    assert np.isfinite(
        test_prediction_kw
    ).all()


    assert np.isfinite(
        test_actual_kw
    ).all()


    # --------------------------------------------------------
    # N. Compute physical-unit TEST metrics
    #
    # Exact confirmed helper API.
    #
    # Positional arguments are used intentionally.
    # --------------------------------------------------------

    test_metrics = (
        compute_regression_metrics_kw(
            test_actual_kw,
            test_prediction_kw,
        )
    )


    assert set(
        test_metrics.keys()
    ) >= {
        "MAE_kW",
        "RMSE_kW",
        "R2",
    }


    # --------------------------------------------------------
    # O. Build final audit record
    # --------------------------------------------------------

    result = {
        "model":
            "LSTM",

        "candidate_id":
            "RNN_C1",

        "horizon_minutes":
            horizon_minutes,

        "feature_family":
            feature_family,

        "auxiliary_dim":
            auxiliary_dim,

        "trainable_parameter_count":
            int(
                model.count_params()
            ),

        "n_train":
            int(
                trainval_data[
                    "n_train"
                ]
            ),

        "n_validation":
            int(
                trainval_data[
                    "n_validation"
                ]
            ),

        "n_trainval":
            int(
                trainval_data[
                    "n_trainval"
                ]
            ),

        "n_test":
            n_test,

        "frozen_best_epoch":
            frozen_best_epoch,

        "final_epochs_trained":
            int(
                len(
                    history_df
                )
            ),

        "test_MAE_kW":
            float(
                test_metrics[
                    "MAE_kW"
                ]
            ),

        "test_RMSE_kW":
            float(
                test_metrics[
                    "RMSE_kW"
                ]
            ),

        "test_R2":
            float(
                test_metrics[
                    "R2"
                ]
            ),

        "final_fit_seconds":
            float(
                final_fit_seconds
            ),

        "prediction_clipping":
            False,

        "scaler_policy":
            "original_train_only_scalers",

        "refit_population":
            "train_plus_validation",
    }


    # --------------------------------------------------------
    # P. Release TensorFlow model after copying outputs
    # --------------------------------------------------------

    del model
    del final_history


    keras.backend.clear_session()

    gc.collect()


    # --------------------------------------------------------
    # Q. Return copied results
    # --------------------------------------------------------

    return (
        result,
        history_df,
        test_actual_kw,
        test_prediction_kw,
    )


# ============================================================
# 5. NON-TEST smoke checks
# ============================================================
#
# We now test:
#
#   TRAIN + VALIDATION assembly
#   frozen architecture construction
#
# We do NOT call run_lstm_final_refit_and_test().
# ============================================================


# ------------------------------------------------------------
# 5A. Expected split populations
# ------------------------------------------------------------

EXPECTED_LSTM_SPLIT_COUNTS = {
    15: {
        "train": 87_261,
        "validation": 17_568,
        "trainval": 104_829,
    },

    30: {
        "train": 87_258,
        "validation": 17_568,
        "trainval": 104_826,
    },

    60: {
        "train": 87_252,
        "validation": 17_568,
        "trainval": 104_820,
    },
}


# ------------------------------------------------------------
# 5B. Verify TRAIN + VALIDATION assembly for every horizon
#     using F0 only.
# ------------------------------------------------------------

trainval_smoke_rows = []


for horizon_minutes in (
    LSTM_FROZEN_HORIZONS
):

    smoke_data = (
        get_final_refit_training_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                "F0",
        )
    )


    expected_counts = (
        EXPECTED_LSTM_SPLIT_COUNTS[
            horizon_minutes
        ]
    )


    assert (
        smoke_data[
            "n_train"
        ]
        == expected_counts[
            "train"
        ]
    )


    assert (
        smoke_data[
            "n_validation"
        ]
        == expected_counts[
            "validation"
        ]
    )


    assert (
        smoke_data[
            "n_trainval"
        ]
        == expected_counts[
            "trainval"
        ]
    )


    assert (
        smoke_data[
            "auxiliary_dim"
        ]
        == 1
    )


    trainval_smoke_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "n_train":
                smoke_data[
                    "n_train"
                ],

            "n_validation":
                smoke_data[
                    "n_validation"
                ],

            "n_trainval":
                smoke_data[
                    "n_trainval"
                ],

            "sequence_shape":
                str(
                    smoke_data[
                        "sequence"
                    ].shape
                ),

            "auxiliary_shape":
                str(
                    smoke_data[
                        "auxiliary"
                    ].shape
                ),
        }
    )


trainval_smoke_table = pd.DataFrame(
    trainval_smoke_rows
)


# ------------------------------------------------------------
# 5C. Verify all five auxiliary widths by constructing fresh
#     UNTRAINED models only.
# ------------------------------------------------------------

expected_parameter_counts = {
    "F0": 6_497,
    "F1": 6_977,
    "F2": 7_137,
    "F3": 7_425,
    "F4": 10_529,
}


model_factory_smoke_rows = []


for feature_family in (
    LSTM_FROZEN_FEATURE_FAMILIES
):

    auxiliary_dim = (
        LSTM_EXPECTED_AUX_DIMS[
            feature_family
        ]
    )


    reset_deep_model_state()


    smoke_model = (
        build_deep_forecasting_model(

            model_name=
                "LSTM",

            auxiliary_dim=
                auxiliary_dim,

            candidate_config=
                LSTM_FINAL_CANDIDATE_CONFIG,

            seed=
                42,
        )
    )


    parameter_count = int(
        smoke_model.count_params()
    )


    assert (
        parameter_count
        ==
        expected_parameter_counts[
            feature_family
        ]
    )


    model_factory_smoke_rows.append(
        {
            "feature_family":
                feature_family,

            "auxiliary_dim":
                auxiliary_dim,

            "parameter_count":
                parameter_count,
        }
    )


    del smoke_model

    keras.backend.clear_session()

    gc.collect()


model_factory_smoke_table = pd.DataFrame(
    model_factory_smoke_rows
)


# ------------------------------------------------------------
# 6. Display smoke-test results
# ------------------------------------------------------------

print(
    "Final-refit / TEST machinery setup"
)

print(
    "----------------------------------"
)


print(
    "\nTRAIN + VALIDATION assembly:"
)


display(
    trainval_smoke_table
)


print(
    "\nFrozen model-factory verification:"
)


display(
    model_factory_smoke_table
)


# ------------------------------------------------------------
# 7. Final methodological status
# ------------------------------------------------------------

print(
    "\nFrozen final-refit protocol:"
)

print(
    "  Architecture: LSTM RNN_C1"
)

print(
    "  Epochs: frozen separately for all 15 experiments"
)

print(
    "  Scaling: original TRAIN-only scalers"
)

print(
    "  Final fitting population: TRAIN + VALIDATION"
)

print(
    "  Early stopping during final refit: disabled"
)

print(
    "  Final-refit validation_data: none"
)

print(
    "  Prediction clipping: disabled"
)


print(
    "\nHeld-out TEST evaluation:"
)

print(
    "  NOT PERFORMED"
)


print(
    "\nStep 42E final-refit/test machinery setup: PASSED"
)

Final-refit / TEST machinery setup
----------------------------------

TRAIN + VALIDATION assembly:


,horizon_minutes,n_train,n_validation,n_trainval,sequence_shape,auxiliary_shape
0,15,87261,17568,104829,"(104829, 13, 9)","(104829, 1)"
1,30,87258,17568,104826,"(104826, 13, 9)","(104826, 1)"
2,60,87252,17568,104820,"(104820, 13, 9)","(104820, 1)"



Frozen model-factory verification:


,feature_family,auxiliary_dim,parameter_count
0,F0,1,6497
1,F1,16,6977
2,F2,21,7137
3,F3,30,7425
4,F4,127,10529



Frozen final-refit protocol:
  Architecture: LSTM RNN_C1
  Epochs: frozen separately for all 15 experiments
  Scaling: original TRAIN-only scalers
  Final fitting population: TRAIN + VALIDATION
  Early stopping during final refit: disabled
  Final-refit validation_data: none
  Prediction clipping: disabled

Held-out TEST evaluation:
  NOT PERFORMED

Step 42E final-refit/test machinery setup: PASSED


In [57]:
# ============================================================
# Step 43: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F0
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       7
#
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#
#       87,261 + 17,568
#       = 104,829 samples
#
#
# Held-out TEST:
#
#       26,496 samples
#
#
# IMPORTANT
# ---------
#
# Architecture, feature definitions, scaling policy, and epoch
# count were frozen BEFORE this held-out evaluation.
#
# No early stopping.
# No validation_data during final refit.
# No prediction clipping.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_ARCHITECTURE[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            15,
            "F0",
        )
    ]
    == 7
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F0"
    ]
    == 1
)


# ------------------------------------------------------------
# 2. Initialize final-evaluation containers
#
# These containers will eventually hold all 15 final LSTM
# experiments.
# ------------------------------------------------------------

lstm_final_test_results = {}

lstm_final_test_histories = {}

lstm_final_test_actuals = {}

lstm_final_test_predictions = {}


# ------------------------------------------------------------
# 3. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_15_f0_result,
    lstm_final_15_f0_history,
    lstm_final_15_f0_actual_kw,
    lstm_final_15_f0_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        15,

    feature_family=
        "F0",

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_15_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_15_f0_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_final_15_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_final_15_f0_result[
        "auxiliary_dim"
    ]
    == 1
)


assert (
    lstm_final_15_f0_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


# ------------------------------------------------------------
# 5. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f0_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_final_15_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_15_f0_result[
        "n_trainval"
    ]
    == 104_829
)


assert (
    lstm_final_15_f0_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 6. Frozen-epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f0_result[
        "frozen_best_epoch"
    ]
    == 7
)


assert (
    lstm_final_15_f0_result[
        "final_epochs_trained"
    ]
    == 7
)


assert (
    len(
        lstm_final_15_f0_history
    )
    == 7
)


assert (
    "val_loss"
    not in
    lstm_final_15_f0_history.columns
)


# ------------------------------------------------------------
# 7. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_15_f0_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_15_f0_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Metric checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_15_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 9. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f0_actual_kw.shape
    ==
    lstm_final_15_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_15_f0_actual_kw
).all()


assert np.isfinite(
    lstm_final_15_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Store the successful final result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        15,
        "F0",
    )
] = (
    lstm_final_15_f0_result.copy()
)


lstm_final_test_histories[
    (
        15,
        "F0",
    )
] = (
    lstm_final_15_f0_history.copy()
)


lstm_final_test_actuals[
    (
        15,
        "F0",
    )
] = (
    lstm_final_15_f0_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        15,
        "F0",
    )
] = (
    lstm_final_15_f0_prediction_kw.copy()
)


assert set(
    lstm_final_test_results.keys()
) == {
    (
        15,
        "F0",
    )
}


# ------------------------------------------------------------
# 11. Display final TEST metrics
# ------------------------------------------------------------

lstm_final_15_f0_result_df = pd.DataFrame(
    [
        lstm_final_15_f0_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_15_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Display final TRAIN + VALIDATION loss trajectory
#
# There must be exactly seven training epochs and no val_loss.
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_15_f0_history
)


# ------------------------------------------------------------
# 13. Audit the raw, UNCLIPPED predictions
#
# Negative predictions are possible because the output layer
# is unconstrained and our frozen protocol specifies no
# post-hoc clipping.
# ------------------------------------------------------------

lstm_final_15_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_15_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_15_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_15_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_15_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_15_f0_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_15_f0_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_15_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 14. Construct persisted prediction table
# ------------------------------------------------------------

lstm_final_15_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_15_f0_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_15_f0_prediction_kw,
    }
)


assert (
    lstm_final_15_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 15. Persist this final experiment immediately
# ------------------------------------------------------------

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


lstm_15_f0_result_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F0_result.csv"
)


lstm_15_f0_history_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F0_history.csv"
)


lstm_15_f0_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F0_predictions.csv"
)


lstm_final_15_f0_result_df.to_csv(
    lstm_15_f0_result_path,
    index=False,
)


lstm_final_15_f0_history.to_csv(
    lstm_15_f0_history_path,
    index=False,
)


lstm_final_15_f0_prediction_table.to_csv(
    lstm_15_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 16. Confirm artifacts exist
# ------------------------------------------------------------

for output_path in [
    lstm_15_f0_result_path,
    lstm_15_f0_history_path,
    lstm_15_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 17. Report persisted artifacts
# ------------------------------------------------------------

print(
    "\nSaved final 15-min F0 artifacts:"
)


print(
    " ",
    lstm_15_f0_result_path,
)


print(
    " ",
    lstm_15_f0_history_path,
)


print(
    " ",
    lstm_15_f0_predictions_path,
)


# ------------------------------------------------------------
# 18. Final evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0: YES"
)

print(
    "  Remaining combinations: 14"
)


print(
    "\nLSTM 15-min F0 final held-out evaluation: PASSED"
)

Epoch 1/7


2026-09-21 17:42:55.821621: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:42:55.821926: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.1680
Epoch 2/7
410/410 - 4s - 10ms/step - loss: 0.1336
Epoch 3/7
410/410 - 4s - 10ms/step - loss: 0.1377
Epoch 4/7
410/410 - 4s - 10ms/step - loss: 0.1441
Epoch 5/7
410/410 - 4s - 10ms/step - loss: 0.1431
Epoch 6/7
410/410 - 4s - 10ms/step - loss: 0.1512
Epoch 7/7
410/410 - 4s - 10ms/step - loss: 0.1530


2026-09-21 17:43:26.776658: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:43:26.777129: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


Final TRAIN + VALIDATION fitting history:


,epoch,loss
0,1,0.168025
1,2,0.133572
2,3,0.137674
3,4,0.144097
4,5,0.143140
5,6,0.151234
6,7,0.153037



Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.367111,5.903061,4090,0



Saved final 15-min F0 artifacts:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_final_15min_F0_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_final_15min_F0_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_final_15min_F0_predictions.csv

Final LSTM held-out evaluations completed:
  15-min F0: YES
  Remaining combinations: 14

LSTM 15-min F0 final held-out evaluation: PASSED


In [58]:
# ============================================================
# Step 44: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F1
#
# F1 = F0 + Gregorian calendar/time features
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       1
#
# Auxiliary width:
#       16
#
# Trainable parameters:
#       6,977
#
#
# IMPORTANT
# ---------
#
# No architecture, epoch, feature, scaling, or training-policy
# decision is changed using TEST performance.
#
# This cell evaluates ONLY:
#
#       15-minute F1
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            15,
            "F1",
        )
    ]
    == 1
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F1"
    ]
    == 16
)


assert (
    (
        15,
        "F0",
    )
    in lstm_final_test_results
)


# Prevent accidental duplicate final evaluation in this
# notebook session.
assert (
    (
        15,
        "F1",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_15_f1_result,
    lstm_final_15_f1_history,
    lstm_final_15_f1_actual_kw,
    lstm_final_15_f1_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        15,

    feature_family=
        "F1",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_15_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_15_f1_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_final_15_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_final_15_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_final_15_f1_result[
        "trainable_parameter_count"
    ]
    == 6_977
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f1_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_final_15_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_15_f1_result[
        "n_trainval"
    ]
    == 104_829
)


assert (
    lstm_final_15_f1_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f1_result[
        "frozen_best_epoch"
    ]
    == 1
)


assert (
    lstm_final_15_f1_result[
        "final_epochs_trained"
    ]
    == 1
)


assert (
    len(
        lstm_final_15_f1_history
    )
    == 1
)


assert (
    "val_loss"
    not in
    lstm_final_15_f1_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_15_f1_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_15_f1_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_15_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f1_actual_kw.shape
    ==
    lstm_final_15_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_15_f1_actual_kw
).all()


assert np.isfinite(
    lstm_final_15_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm F0 and F1 use the identical TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
    lstm_final_15_f1_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F1 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        15,
        "F1",
    )
] = (
    lstm_final_15_f1_result.copy()
)


lstm_final_test_histories[
    (
        15,
        "F1",
    )
] = (
    lstm_final_15_f1_history.copy()
)


lstm_final_test_actuals[
    (
        15,
        "F1",
    )
] = (
    lstm_final_15_f1_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        15,
        "F1",
    )
] = (
    lstm_final_15_f1_prediction_kw.copy()
)


assert set(
    lstm_final_test_results.keys()
) == {
    (
        15,
        "F0",
    ),
    (
        15,
        "F1",
    ),
}


# ------------------------------------------------------------
# 11. Display final F1 TEST metrics
# ------------------------------------------------------------

lstm_final_15_f1_result_df = pd.DataFrame(
    [
        lstm_final_15_f1_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_15_f1_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Final fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_15_f1_history
)


# ------------------------------------------------------------
# 13. Test-only F0 -> F1 descriptive comparison
#
# Positive percentage:
#       F1 has lower TEST MAE.
#
# Negative percentage:
#       F1 has higher TEST MAE.
#
# This is descriptive evaluation only.
# It is NOT used for model tuning.
# ------------------------------------------------------------

f0_test_mae_15 = (
    lstm_final_test_results[
        (
            15,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


f1_test_mae_15 = (
    lstm_final_15_f1_result[
        "test_MAE_kW"
    ]
)


f0_to_f1_test_improvement_pct_15 = (
    (
        f0_test_mae_15
        -
        f1_test_mae_15
    )
    /
    f0_test_mae_15
    * 100.0
)


print(
    "\n15-min held-out TEST F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_test_mae_15:.6f} kW"
)

print(
    f"  F1 MAE: {f1_test_mae_15:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_test_improvement_pct_15:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 14. Unclipped prediction audit
# ------------------------------------------------------------

lstm_final_15_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_15_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_15_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_15_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_15_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_15_f1_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_15_f1_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_15_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_15_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_15_f1_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_15_f1_prediction_kw,
    }
)


assert (
    lstm_final_15_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final F1 experiment
# ------------------------------------------------------------

lstm_15_f1_result_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F1_result.csv"
)


lstm_15_f1_history_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F1_history.csv"
)


lstm_15_f1_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F1_predictions.csv"
)


lstm_final_15_f1_result_df.to_csv(
    lstm_15_f1_result_path,
    index=False,
)


lstm_final_15_f1_history.to_csv(
    lstm_15_f1_history_path,
    index=False,
)


lstm_final_15_f1_prediction_table.to_csv(
    lstm_15_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_15_f1_result_path,
    lstm_15_f1_history_path,
    lstm_15_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Current 15-minute final TEST progression
# ------------------------------------------------------------

lstm_15min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)


print(
    "\nCurrent 15-minute final TEST progression:"
)


display(
    lstm_15min_final_progress
)


# ------------------------------------------------------------
# 19. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0: YES"
)

print(
    "  15-min F1: YES"
)

print(
    "  Remaining combinations: 13"
)


print(
    "\nLSTM 15-min F1 final held-out evaluation: PASSED"
)

2026-09-21 17:45:05.027746: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:45:05.028241: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1652


2026-09-21 17:45:10.862558: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:45:10.862840: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds
0,LSTM,RNN_C1,15,F1,16,6977,104829,26496,1,1,0.294215,0.622662,0.857704,5.8761


...


15-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.298274 kW
  F1 MAE: 0.294215 kW
  Relative MAE change: +1.3610%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.168082,5.801661,5705,0



Current 15-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,7,0.298274,0.620531,0.858677
1,F1,1,0.294215,0.622662,0.857704



Final LSTM held-out evaluations completed:
  15-min F0: YES
  15-min F1: YES
  Remaining combinations: 13

LSTM 15-min F1 final held-out evaluation: PASSED


In [59]:
# ============================================================
# Step 45: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F2
#
# F2 = F1 + independent solar-astronomy features
#
#       solar_elevation_deg
#       solar_cos_zenith
#       solar_azimuth_sin
#       solar_azimuth_cos
#       solar_daylight
#
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       2
#
# Auxiliary width:
#       21
#
# Trainable parameters:
#       7,137
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       15-minute F2
#
# No architecture, epoch, feature, scaling, or training-policy
# decision is changed using TEST performance.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            15,
            "F2",
        )
    ]
    == 2
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F2"
    ]
    == 21
)


# F0 and F1 must already have been evaluated successfully.
assert (
    (
        15,
        "F0",
    )
    in lstm_final_test_results
)


assert (
    (
        15,
        "F1",
    )
    in lstm_final_test_results
)


# Prevent accidental duplicate evaluation.
assert (
    (
        15,
        "F2",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_15_f2_result,
    lstm_final_15_f2_history,
    lstm_final_15_f2_actual_kw,
    lstm_final_15_f2_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        15,

    feature_family=
        "F2",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f2_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_15_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_15_f2_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_final_15_f2_result[
        "feature_family"
    ]
    == "F2"
)


assert (
    lstm_final_15_f2_result[
        "auxiliary_dim"
    ]
    == 21
)


assert (
    lstm_final_15_f2_result[
        "trainable_parameter_count"
    ]
    == 7_137
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f2_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_final_15_f2_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_15_f2_result[
        "n_trainval"
    ]
    == 104_829
)


assert (
    lstm_final_15_f2_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f2_result[
        "frozen_best_epoch"
    ]
    == 2
)


assert (
    lstm_final_15_f2_result[
        "final_epochs_trained"
    ]
    == 2
)


assert (
    len(
        lstm_final_15_f2_history
    )
    == 2
)


assert (
    "val_loss"
    not in
    lstm_final_15_f2_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_15_f2_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_15_f2_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_15_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f2_actual_kw.shape
    ==
    lstm_final_15_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_15_f2_actual_kw
).all()


assert np.isfinite(
    lstm_final_15_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical TEST target population
#
# F0, F1, and F2 at the same horizon must all be evaluated
# against exactly the same target observations.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
    lstm_final_15_f2_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F1",
        )
    ],
    lstm_final_15_f2_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F2 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        15,
        "F2",
    )
] = (
    lstm_final_15_f2_result.copy()
)


lstm_final_test_histories[
    (
        15,
        "F2",
    )
] = (
    lstm_final_15_f2_history.copy()
)


lstm_final_test_actuals[
    (
        15,
        "F2",
    )
] = (
    lstm_final_15_f2_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        15,
        "F2",
    )
] = (
    lstm_final_15_f2_prediction_kw.copy()
)


assert set(
    lstm_final_test_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
}


# ------------------------------------------------------------
# 11. Display final F2 TEST metrics
# ------------------------------------------------------------

lstm_final_15_f2_result_df = pd.DataFrame(
    [
        lstm_final_15_f2_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_15_f2_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Display final fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_15_f2_history
)


# ------------------------------------------------------------
# 13. Test-only F1 -> F2 descriptive comparison
#
# Positive percentage:
#       F2 has lower TEST MAE.
#
# Negative percentage:
#       F2 has higher TEST MAE.
#
# This comparison is descriptive only and is NOT used for
# tuning or retraining.
# ------------------------------------------------------------

f1_test_mae_15 = (
    lstm_final_test_results[
        (
            15,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


f2_test_mae_15 = (
    lstm_final_15_f2_result[
        "test_MAE_kW"
    ]
)


f1_to_f2_test_improvement_pct_15 = (
    (
        f1_test_mae_15
        -
        f2_test_mae_15
    )
    /
    f1_test_mae_15
    * 100.0
)


print(
    "\n15-min held-out TEST F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_test_mae_15:.6f} kW"
)

print(
    f"  F2 MAE: {f2_test_mae_15:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_test_improvement_pct_15:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 14. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_15_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_15_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_15_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_15_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_15_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_15_f2_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_15_f2_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_15_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_15_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_15_f2_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_15_f2_prediction_kw,
    }
)


assert (
    lstm_final_15_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final F2 experiment immediately
# ------------------------------------------------------------

lstm_15_f2_result_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F2_result.csv"
)


lstm_15_f2_history_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F2_history.csv"
)


lstm_15_f2_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F2_predictions.csv"
)


lstm_final_15_f2_result_df.to_csv(
    lstm_15_f2_result_path,
    index=False,
)


lstm_final_15_f2_history.to_csv(
    lstm_15_f2_history_path,
    index=False,
)


lstm_final_15_f2_prediction_table.to_csv(
    lstm_15_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_15_f2_result_path,
    lstm_15_f2_history_path,
    lstm_15_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Current 15-minute final TEST progression
# ------------------------------------------------------------

lstm_15min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)


print(
    "\nCurrent 15-minute final TEST progression:"
)


display(
    lstm_15min_final_progress
)


# ------------------------------------------------------------
# 19. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0: YES"
)

print(
    "  15-min F1: YES"
)

print(
    "  15-min F2: YES"
)

print(
    "  Remaining combinations: 12"
)


print(
    "\nLSTM 15-min F2 final held-out evaluation: PASSED"
)

Epoch 1/2


2026-09-21 17:48:43.719437: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:48:43.719734: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1833
Epoch 2/2
410/410 - 5s - 11ms/step - loss: 0.1394


2026-09-21 17:48:54.132083: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:48:54.132440: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


15-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.294215 kW
  F2 MAE: 0.298937 kW
  Relative MAE change: -1.6050%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.324689,5.829429,8353,0



Current 15-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,7,0.298274,0.620531,0.858677
1,F1,1,0.294215,0.622662,0.857704
2,F2,2,0.298937,0.622725,0.857676



Final LSTM held-out evaluations completed:
  15-min F0: YES
  15-min F1: YES
  15-min F2: YES
  Remaining combinations: 12

LSTM 15-min F2 final held-out evaluation: PASSED


In [60]:
# ============================================================
# Step 46: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F3
#
# F3 = F2 + continuous lunar / Sun-Moon astronomy
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       2
#
# Auxiliary width:
#       30
#
# Trainable parameters:
#       7,425
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       15-minute F3
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            15,
            "F3",
        )
    ]
    == 2
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F3"
    ]
    == 30
)


# Previous 15-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            15,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate final evaluation.
assert (
    (
        15,
        "F3",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_15_f3_result,
    lstm_final_15_f3_history,
    lstm_final_15_f3_actual_kw,
    lstm_final_15_f3_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        15,

    feature_family=
        "F3",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_15_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_15_f3_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_final_15_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_final_15_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_final_15_f3_result[
        "trainable_parameter_count"
    ]
    == 7_425
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f3_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_final_15_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_15_f3_result[
        "n_trainval"
    ]
    == 104_829
)


assert (
    lstm_final_15_f3_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f3_result[
        "frozen_best_epoch"
    ]
    == 2
)


assert (
    lstm_final_15_f3_result[
        "final_epochs_trained"
    ]
    == 2
)


assert (
    len(
        lstm_final_15_f3_history
    )
    == 2
)


assert (
    "val_loss"
    not in
    lstm_final_15_f3_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_15_f3_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_15_f3_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_15_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f3_actual_kw.shape
    ==
    lstm_final_15_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_15_f3_actual_kw
).all()


assert np.isfinite(
    lstm_final_15_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical held-out target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
    lstm_final_15_f3_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F2",
        )
    ],
    lstm_final_15_f3_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F3 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        15,
        "F3",
    )
] = (
    lstm_final_15_f3_result.copy()
)


lstm_final_test_histories[
    (
        15,
        "F3",
    )
] = (
    lstm_final_15_f3_history.copy()
)


lstm_final_test_actuals[
    (
        15,
        "F3",
    )
] = (
    lstm_final_15_f3_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        15,
        "F3",
    )
] = (
    lstm_final_15_f3_prediction_kw.copy()
)


assert set(
    lstm_final_test_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
}


# ------------------------------------------------------------
# 11. Display final F3 TEST metrics
# ------------------------------------------------------------

lstm_final_15_f3_result_df = pd.DataFrame(
    [
        lstm_final_15_f3_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_15_f3_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Display final fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_15_f3_history
)


# ------------------------------------------------------------
# 13. F2 -> F3 descriptive TEST comparison
#
# Positive percentage:
#       F3 has lower TEST MAE.
#
# Negative percentage:
#       F3 has higher TEST MAE.
# ------------------------------------------------------------

f2_test_mae_15 = (
    lstm_final_test_results[
        (
            15,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


f3_test_mae_15 = (
    lstm_final_15_f3_result[
        "test_MAE_kW"
    ]
)


f2_to_f3_test_improvement_pct_15 = (
    (
        f2_test_mae_15
        -
        f3_test_mae_15
    )
    /
    f2_test_mae_15
    * 100.0
)


print(
    "\n15-min held-out TEST F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_test_mae_15:.6f} kW"
)

print(
    f"  F3 MAE: {f3_test_mae_15:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_test_improvement_pct_15:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 14. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_15_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_15_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_15_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_15_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_15_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_15_f3_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_15_f3_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_15_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_15_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_15_f3_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_15_f3_prediction_kw,
    }
)


assert (
    lstm_final_15_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final F3 experiment
# ------------------------------------------------------------

lstm_15_f3_result_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F3_result.csv"
)


lstm_15_f3_history_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F3_history.csv"
)


lstm_15_f3_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F3_predictions.csv"
)


lstm_final_15_f3_result_df.to_csv(
    lstm_15_f3_result_path,
    index=False,
)


lstm_final_15_f3_history.to_csv(
    lstm_15_f3_history_path,
    index=False,
)


lstm_final_15_f3_prediction_table.to_csv(
    lstm_15_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_15_f3_result_path,
    lstm_15_f3_history_path,
    lstm_15_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Current 15-minute final TEST progression
# ------------------------------------------------------------

lstm_15min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\nCurrent 15-minute final TEST progression:"
)


display(
    lstm_15min_final_progress
)


# ------------------------------------------------------------
# 19. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0: YES"
)

print(
    "  15-min F1: YES"
)

print(
    "  15-min F2: YES"
)

print(
    "  15-min F3: YES"
)

print(
    "  Remaining combinations: 11"
)


print(
    "\nLSTM 15-min F3 final held-out evaluation: PASSED"
)

Epoch 1/2


2026-09-21 17:50:31.499968: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:50:31.500250: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.1876
Epoch 2/2
410/410 - 4s - 10ms/step - loss: 0.1429


2026-09-21 17:50:41.751624: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:50:41.751916: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


15-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.298937 kW
  F3 MAE: 0.326384 kW
  Relative MAE change: -9.1815%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.378269,5.994712,7796,0



Current 15-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,7,0.298274,0.620531,0.858677
1,F1,1,0.294215,0.622662,0.857704
2,F2,2,0.298937,0.622725,0.857676
3,F3,2,0.326384,0.633885,0.852529



Final LSTM held-out evaluations completed:
  15-min F0: YES
  15-min F1: YES
  15-min F2: YES
  15-min F3: YES
  Remaining combinations: 11

LSTM 15-min F3 final held-out evaluation: PASSED


In [61]:
# ============================================================
# Step 47: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       15 minutes
#
# Feature family:
#       F4
#
# F4 = F3 + canonical Panchang one-hot features
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 Panchang indicators
#
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       7
#
# Auxiliary width:
#       127
#
# Trainable parameters:
#       10,529
#
#
# PRIMARY SCIENTIFIC COMPARISON
# -----------------------------
#
#       F3 -> F4
#
# This tests whether the Panchang representation provides
# predictive information beyond the continuous astronomical
# quantities already present in F3.
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       15-minute F4
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            15,
            "F4",
        )
    ]
    == 7
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F4"
    ]
    == 127
)


# Previous 15-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            15,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate final evaluation.
assert (
    (
        15,
        "F4",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_15_f4_result,
    lstm_final_15_f4_history,
    lstm_final_15_f4_actual_kw,
    lstm_final_15_f4_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        15,

    feature_family=
        "F4",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_15_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_15_f4_result[
        "horizon_minutes"
    ]
    == 15
)


assert (
    lstm_final_15_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_final_15_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_final_15_f4_result[
        "trainable_parameter_count"
    ]
    == 10_529
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f4_result[
        "n_train"
    ]
    == 87_261
)


assert (
    lstm_final_15_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_15_f4_result[
        "n_trainval"
    ]
    == 104_829
)


assert (
    lstm_final_15_f4_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f4_result[
        "frozen_best_epoch"
    ]
    == 7
)


assert (
    lstm_final_15_f4_result[
        "final_epochs_trained"
    ]
    == 7
)


assert (
    len(
        lstm_final_15_f4_history
    )
    == 7
)


assert (
    "val_loss"
    not in
    lstm_final_15_f4_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_15_f4_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_15_f4_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_15_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_15_f4_actual_kw.shape
    ==
    lstm_final_15_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_15_f4_actual_kw
).all()


assert np.isfinite(
    lstm_final_15_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical held-out target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            15,
            "F3",
        )
    ],
    lstm_final_15_f4_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F4 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        15,
        "F4",
    )
] = (
    lstm_final_15_f4_result.copy()
)


lstm_final_test_histories[
    (
        15,
        "F4",
    )
] = (
    lstm_final_15_f4_history.copy()
)


lstm_final_test_actuals[
    (
        15,
        "F4",
    )
] = (
    lstm_final_15_f4_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        15,
        "F4",
    )
] = (
    lstm_final_15_f4_prediction_kw.copy()
)


assert set(
    lstm_final_test_results.keys()
) == {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),
}


# ------------------------------------------------------------
# 11. Display final F4 TEST metrics
# ------------------------------------------------------------

lstm_final_15_f4_result_df = pd.DataFrame(
    [
        lstm_final_15_f4_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_15_f4_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Display final fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_15_f4_history
)


# ------------------------------------------------------------
# 13. KEY F3 -> F4 descriptive TEST comparison
#
# Positive percentage:
#       F4 has lower TEST MAE.
#
# Negative percentage:
#       F4 has higher TEST MAE.
#
# This is the central Panchang increment comparison.
# ------------------------------------------------------------

f3_test_mae_15 = (
    lstm_final_test_results[
        (
            15,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


f4_test_mae_15 = (
    lstm_final_15_f4_result[
        "test_MAE_kW"
    ]
)


f3_to_f4_test_improvement_pct_15 = (
    (
        f3_test_mae_15
        -
        f4_test_mae_15
    )
    /
    f3_test_mae_15
    * 100.0
)


print(
    "\n15-min held-out TEST F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_test_mae_15:.6f} kW"
)

print(
    f"  F4 MAE: {f4_test_mae_15:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_test_improvement_pct_15:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 14. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_15_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_15_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_15_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_15_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_15_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_15_f4_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_15_f4_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_15_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_15_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_15_f4_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_15_f4_prediction_kw,
    }
)


assert (
    lstm_final_15_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final F4 experiment
# ------------------------------------------------------------

lstm_15_f4_result_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F4_result.csv"
)


lstm_15_f4_history_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F4_history.csv"
)


lstm_15_f4_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_F4_predictions.csv"
)


lstm_final_15_f4_result_df.to_csv(
    lstm_15_f4_result_path,
    index=False,
)


lstm_final_15_f4_history.to_csv(
    lstm_15_f4_history_path,
    index=False,
)


lstm_final_15_f4_prediction_table.to_csv(
    lstm_15_f4_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_15_f4_result_path,
    lstm_15_f4_history_path,
    lstm_15_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Complete 15-minute final TEST progression
# ------------------------------------------------------------

lstm_15min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        15,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 15-minute final TEST progression:"
)


display(
    lstm_15min_final_progress
)


# ------------------------------------------------------------
# 19. Persist compact 15-minute summary
# ------------------------------------------------------------

lstm_15min_final_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_15min_summary.csv"
)


lstm_15min_final_progress.to_csv(
    lstm_15min_final_summary_path,
    index=False,
)


assert (
    lstm_15min_final_summary_path.exists()
)


assert (
    lstm_15min_final_summary_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  Remaining combinations: 10"
)


print(
    "\n15-minute F3 -> F4 TEST MAE change:"
)

print(
    f"  {f3_to_f4_test_improvement_pct_15:+.4f}%"
)


print(
    "\nLSTM 15-min F4 final held-out evaluation: PASSED"
)

Epoch 1/7


2026-09-21 17:53:21.426591: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:53:21.426917: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1814
Epoch 2/7
410/410 - 4s - 10ms/step - loss: 0.1400
Epoch 3/7
410/410 - 4s - 10ms/step - loss: 0.1406
Epoch 4/7
410/410 - 4s - 10ms/step - loss: 0.1391
Epoch 5/7
410/410 - 4s - 10ms/step - loss: 0.1426
Epoch 6/7
410/410 - 4s - 10ms/step - loss: 0.1615
Epoch 7/7
410/410 - 4s - 10ms/step - loss: 0.1590


2026-09-21 17:53:52.240040: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:53:52.240327: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


15-min held-out TEST F3 -> F4 comparison:
  F3 MAE: 0.326384 kW
  F4 MAE: 0.329271 kW
  Relative MAE change: -0.8845%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.753491,5.855189,6904,0



Complete 15-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,7,0.298274,0.620531,0.858677
1,F1,1,0.294215,0.622662,0.857704
2,F2,2,0.298937,0.622725,0.857676
3,F3,2,0.326384,0.633885,0.852529
4,F4,7,0.329271,0.636115,0.851489



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  Remaining combinations: 10

15-minute F3 -> F4 TEST MAE change:
  -0.8845%

LSTM 15-min F4 final held-out evaluation: PASSED


In [62]:
# ============================================================
# Step 48: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F0
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       10
#
# Auxiliary width:
#       1
#
# Trainable parameters:
#       6,497
#
#
# Final fitting population:
#
#       TRAIN      = 87,258
#       VALIDATION = 17,568
#       -------------------
#       TRAIN+VAL  = 104,826
#
#
# Held-out TEST:
#
#       26,496 samples
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       30-minute F0
#
# No architecture, epoch, feature, scaling, or training-policy
# decision is changed using TEST performance.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            30,
            "F0",
        )
    ]
    == 10
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F0"
    ]
    == 1
)


assert (
    LSTM_FROZEN_ARCHITECTURE[
        "candidate_id"
    ]
    == "RNN_C1"
)


# ------------------------------------------------------------
# 2. Confirm the complete 15-minute grid already exists
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    assert (
        (
            15,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate evaluation.
assert (
    (
        30,
        "F0",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 3. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_30_f0_result,
    lstm_final_30_f0_history,
    lstm_final_30_f0_actual_kw,
    lstm_final_30_f0_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        30,

    feature_family=
        "F0",

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_30_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_30_f0_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_final_30_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_final_30_f0_result[
        "auxiliary_dim"
    ]
    == 1
)


assert (
    lstm_final_30_f0_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


# ------------------------------------------------------------
# 5. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f0_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_final_30_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_30_f0_result[
        "n_trainval"
    ]
    == 104_826
)


assert (
    lstm_final_30_f0_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 6. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f0_result[
        "frozen_best_epoch"
    ]
    == 10
)


assert (
    lstm_final_30_f0_result[
        "final_epochs_trained"
    ]
    == 10
)


assert (
    len(
        lstm_final_30_f0_history
    )
    == 10
)


assert (
    "val_loss"
    not in
    lstm_final_30_f0_history.columns
)


# ------------------------------------------------------------
# 7. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_30_f0_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_30_f0_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_30_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 9. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f0_actual_kw.shape
    ==
    lstm_final_30_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_30_f0_actual_kw
).all()


assert np.isfinite(
    lstm_final_30_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Store successful final F0 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        30,
        "F0",
    )
] = (
    lstm_final_30_f0_result.copy()
)


lstm_final_test_histories[
    (
        30,
        "F0",
    )
] = (
    lstm_final_30_f0_history.copy()
)


lstm_final_test_actuals[
    (
        30,
        "F0",
    )
] = (
    lstm_final_30_f0_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        30,
        "F0",
    )
] = (
    lstm_final_30_f0_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact evaluation-state keys
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


# ------------------------------------------------------------
# 12. Display final TEST metrics
# ------------------------------------------------------------

lstm_final_30_f0_result_df = pd.DataFrame(
    [
        lstm_final_30_f0_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_30_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_30_f0_history
)


# ------------------------------------------------------------
# 14. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_30_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_30_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_30_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_30_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_30_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_30_f0_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_30_f0_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_30_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_30_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_30_f0_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_30_f0_prediction_kw,
    }
)


assert (
    lstm_final_30_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final 30-minute F0 experiment
# ------------------------------------------------------------

lstm_30_f0_result_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F0_result.csv"
)


lstm_30_f0_history_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F0_history.csv"
)


lstm_30_f0_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F0_predictions.csv"
)


lstm_final_30_f0_result_df.to_csv(
    lstm_30_f0_result_path,
    index=False,
)


lstm_final_30_f0_history.to_csv(
    lstm_30_f0_history_path,
    index=False,
)


lstm_final_30_f0_prediction_table.to_csv(
    lstm_30_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_30_f0_result_path,
    lstm_30_f0_history_path,
    lstm_30_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Display current cross-horizon F0 status
#
# This is descriptive only.
# Different horizons are different forecasting tasks, so this
# is not treated as a feature-ablation comparison.
# ------------------------------------------------------------

lstm_f0_cross_horizon_progress = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_R2"
                ],
        }

        for horizon_minutes in [
            15,
            30,
        ]
    ]
)


print(
    "\nCurrent cross-horizon F0 status:"
)


display(
    lstm_f0_cross_horizon_progress
)


# ------------------------------------------------------------
# 19. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0: YES"
)

print(
    "  Remaining combinations: 9"
)


print(
    "\nLSTM 30-min F0 final held-out evaluation: PASSED"
)

Epoch 1/10


2026-09-21 17:57:05.980879: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:57:05.981404: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1995
Epoch 2/10
410/410 - 4s - 10ms/step - loss: 0.1709
Epoch 3/10
410/410 - 4s - 10ms/step - loss: 0.1744
Epoch 4/10
410/410 - 4s - 10ms/step - loss: 0.1755
Epoch 5/10
410/410 - 4s - 10ms/step - loss: 0.1851
Epoch 6/10
410/410 - 4s - 11ms/step - loss: 0.1871
Epoch 7/10
410/410 - 4s - 11ms/step - loss: 0.1855
Epoch 8/10
410/410 - 4s - 10ms/step - loss: 0.2123
Epoch 9/10
410/410 - 4s - 10ms/step - loss: 0.2158
Epoch 10/10
410/410 - 4s - 10ms/step - loss: 0.2489


2026-09-21 17:57:50.092142: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:57:50.092414: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.873182,5.435921,6258,0



Current cross-horizon F0 status:


,horizon_minutes,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,15,7,0.298274,0.620531,0.858677
1,30,10,0.459965,0.780223,0.776579



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0: YES
  Remaining combinations: 9

LSTM 30-min F0 final held-out evaluation: PASSED


In [63]:
# ============================================================
# Step 49: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F1
#
# F1 = F0 + Gregorian calendar/time features
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       1
#
# Auxiliary width:
#       16
#
# Trainable parameters:
#       6,977
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       30-minute F1
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            30,
            "F1",
        )
    ]
    == 1
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F1"
    ]
    == 16
)


assert (
    (
        30,
        "F0",
    )
    in lstm_final_test_results
)


# Prevent accidental duplicate final evaluation.
assert (
    (
        30,
        "F1",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_30_f1_result,
    lstm_final_30_f1_history,
    lstm_final_30_f1_actual_kw,
    lstm_final_30_f1_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        30,

    feature_family=
        "F1",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_30_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_30_f1_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_final_30_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_final_30_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_final_30_f1_result[
        "trainable_parameter_count"
    ]
    == 6_977
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f1_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_final_30_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_30_f1_result[
        "n_trainval"
    ]
    == 104_826
)


assert (
    lstm_final_30_f1_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f1_result[
        "frozen_best_epoch"
    ]
    == 1
)


assert (
    lstm_final_30_f1_result[
        "final_epochs_trained"
    ]
    == 1
)


assert (
    len(
        lstm_final_30_f1_history
    )
    == 1
)


assert (
    "val_loss"
    not in lstm_final_30_f1_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_30_f1_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_30_f1_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_30_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f1_actual_kw.shape
    ==
    lstm_final_30_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_30_f1_actual_kw
).all()


assert np.isfinite(
    lstm_final_30_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 30-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F0",
        )
    ],
    lstm_final_30_f1_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F1 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        30,
        "F1",
    )
] = (
    lstm_final_30_f1_result.copy()
)


lstm_final_test_histories[
    (
        30,
        "F1",
    )
] = (
    lstm_final_30_f1_history.copy()
)


lstm_final_test_actuals[
    (
        30,
        "F1",
    )
] = (
    lstm_final_30_f1_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        30,
        "F1",
    )
] = (
    lstm_final_30_f1_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


# ------------------------------------------------------------
# 12. Display final F1 TEST metrics
# ------------------------------------------------------------

lstm_final_30_f1_result_df = pd.DataFrame(
    [
        lstm_final_30_f1_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_30_f1_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_30_f1_history
)


# ------------------------------------------------------------
# 14. Descriptive F0 -> F1 TEST comparison
#
# Positive:
#       F1 has lower TEST MAE.
#
# Negative:
#       F1 has higher TEST MAE.
# ------------------------------------------------------------

f0_test_mae_30 = (
    lstm_final_test_results[
        (
            30,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


f1_test_mae_30 = (
    lstm_final_30_f1_result[
        "test_MAE_kW"
    ]
)


f0_to_f1_test_improvement_pct_30 = (
    (
        f0_test_mae_30
        -
        f1_test_mae_30
    )
    /
    f0_test_mae_30
    * 100.0
)


print(
    "\n30-min held-out TEST F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_test_mae_30:.6f} kW"
)

print(
    f"  F1 MAE: {f1_test_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_test_improvement_pct_30:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_30_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_30_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_30_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_30_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_30_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_30_f1_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_30_f1_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_30_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_30_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_30_f1_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_30_f1_prediction_kw,
    }
)


assert (
    lstm_final_30_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 30-minute F1 experiment
# ------------------------------------------------------------

lstm_30_f1_result_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F1_result.csv"
)


lstm_30_f1_history_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F1_history.csv"
)


lstm_30_f1_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F1_predictions.csv"
)


lstm_final_30_f1_result_df.to_csv(
    lstm_30_f1_result_path,
    index=False,
)


lstm_final_30_f1_history.to_csv(
    lstm_30_f1_history_path,
    index=False,
)


lstm_final_30_f1_prediction_table.to_csv(
    lstm_30_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_30_f1_result_path,
    lstm_30_f1_history_path,
    lstm_30_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 30-minute final TEST progression
# ------------------------------------------------------------

lstm_30min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)


print(
    "\nCurrent 30-minute final TEST progression:"
)


display(
    lstm_30min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F1: COMPLETE"
)

print(
    "  Remaining combinations: 8"
)


print(
    "\nLSTM 30-min F1 final held-out evaluation: PASSED"
)

2026-09-21 17:59:32.429197: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 17:59:32.429518: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1974


2026-09-21 17:59:38.004548: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 17:59:38.004848: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds
0,LSTM,RNN_C1,30,F1,16,6977,104826,26496,1,1,0.373356,0.741601,0.798151,5.606634


...


30-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.459965 kW
  F1 MAE: 0.373356 kW
  Relative MAE change: +18.8295%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.335781,5.993421,4918,0



Current 30-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.459965,0.780223,0.776579
1,F1,1,0.373356,0.741601,0.798151



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F1: COMPLETE
  Remaining combinations: 8

LSTM 30-min F1 final held-out evaluation: PASSED


In [64]:
# ============================================================
# Step 50: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F2
#
# F2 = F1 + independent solar-astronomy features
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       6
#
# Auxiliary width:
#       21
#
# Trainable parameters:
#       7,137
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       30-minute F2
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            30,
            "F2",
        )
    ]
    == 6
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F2"
    ]
    == 21
)


assert (
    (
        30,
        "F0",
    )
    in lstm_final_test_results
)


assert (
    (
        30,
        "F1",
    )
    in lstm_final_test_results
)


# Prevent accidental duplicate final evaluation.
assert (
    (
        30,
        "F2",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_30_f2_result,
    lstm_final_30_f2_history,
    lstm_final_30_f2_actual_kw,
    lstm_final_30_f2_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        30,

    feature_family=
        "F2",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f2_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_30_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_30_f2_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_final_30_f2_result[
        "feature_family"
    ]
    == "F2"
)


assert (
    lstm_final_30_f2_result[
        "auxiliary_dim"
    ]
    == 21
)


assert (
    lstm_final_30_f2_result[
        "trainable_parameter_count"
    ]
    == 7_137
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f2_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_final_30_f2_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_30_f2_result[
        "n_trainval"
    ]
    == 104_826
)


assert (
    lstm_final_30_f2_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f2_result[
        "frozen_best_epoch"
    ]
    == 6
)


assert (
    lstm_final_30_f2_result[
        "final_epochs_trained"
    ]
    == 6
)


assert (
    len(
        lstm_final_30_f2_history
    )
    == 6
)


assert (
    "val_loss"
    not in lstm_final_30_f2_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_30_f2_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_30_f2_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_30_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f2_actual_kw.shape
    ==
    lstm_final_30_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_30_f2_actual_kw
).all()


assert np.isfinite(
    lstm_final_30_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 30-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F0",
        )
    ],
    lstm_final_30_f2_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F1",
        )
    ],
    lstm_final_30_f2_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F2 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        30,
        "F2",
    )
] = (
    lstm_final_30_f2_result.copy()
)


lstm_final_test_histories[
    (
        30,
        "F2",
    )
] = (
    lstm_final_30_f2_history.copy()
)


lstm_final_test_actuals[
    (
        30,
        "F2",
    )
] = (
    lstm_final_30_f2_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        30,
        "F2",
    )
] = (
    lstm_final_30_f2_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


# ------------------------------------------------------------
# 12. Display final F2 TEST metrics
# ------------------------------------------------------------

lstm_final_30_f2_result_df = pd.DataFrame(
    [
        lstm_final_30_f2_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_30_f2_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_30_f2_history
)


# ------------------------------------------------------------
# 14. Descriptive F1 -> F2 TEST comparison
#
# Positive:
#       F2 has lower TEST MAE.
#
# Negative:
#       F2 has higher TEST MAE.
# ------------------------------------------------------------

f1_test_mae_30 = (
    lstm_final_test_results[
        (
            30,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


f2_test_mae_30 = (
    lstm_final_30_f2_result[
        "test_MAE_kW"
    ]
)


f1_to_f2_test_improvement_pct_30 = (
    (
        f1_test_mae_30
        -
        f2_test_mae_30
    )
    /
    f1_test_mae_30
    * 100.0
)


print(
    "\n30-min held-out TEST F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_test_mae_30:.6f} kW"
)

print(
    f"  F2 MAE: {f2_test_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_test_improvement_pct_30:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_30_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_30_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_30_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_30_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_30_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_30_f2_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_30_f2_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_30_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_30_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_30_f2_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_30_f2_prediction_kw,
    }
)


assert (
    lstm_final_30_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 30-minute F2 experiment
# ------------------------------------------------------------

lstm_30_f2_result_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F2_result.csv"
)


lstm_30_f2_history_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F2_history.csv"
)


lstm_30_f2_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F2_predictions.csv"
)


lstm_final_30_f2_result_df.to_csv(
    lstm_30_f2_result_path,
    index=False,
)


lstm_final_30_f2_history.to_csv(
    lstm_30_f2_history_path,
    index=False,
)


lstm_final_30_f2_prediction_table.to_csv(
    lstm_30_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_30_f2_result_path,
    lstm_30_f2_history_path,
    lstm_30_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 30-minute final TEST progression
# ------------------------------------------------------------

lstm_30min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)


print(
    "\nCurrent 30-minute final TEST progression:"
)


display(
    lstm_30min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F2: COMPLETE"
)

print(
    "  Remaining combinations: 7"
)


print(
    "\nLSTM 30-min F2 final held-out evaluation: PASSED"
)

Epoch 1/6


2026-09-21 18:02:46.455353: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:02:46.456367: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 5s - 13ms/step - loss: 0.2139
Epoch 2/6
410/410 - 4s - 11ms/step - loss: 0.1691
Epoch 3/6
410/410 - 4s - 11ms/step - loss: 0.1694
Epoch 4/6
410/410 - 4s - 10ms/step - loss: 0.1717
Epoch 5/6
410/410 - 4s - 10ms/step - loss: 0.1732
Epoch 6/6
410/410 - 4s - 10ms/step - loss: 0.1755


2026-09-21 18:03:12.498446: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:03:12.498703: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


30-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.373356 kW
  F2 MAE: 0.357165 kW
  Relative MAE change: +4.3365%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.541299,6.081386,4922,0



Current 30-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.459965,0.780223,0.776579
1,F1,1,0.373356,0.741601,0.798151
2,F2,6,0.357165,0.723139,0.808075



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F2: COMPLETE
  Remaining combinations: 7

LSTM 30-min F2 final held-out evaluation: PASSED


In [65]:
# ============================================================
# Step 51: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F3
#
# F3 = F2 + continuous lunar / Sun-Moon astronomy
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       5
#
# Auxiliary width:
#       30
#
# Trainable parameters:
#       7,425
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       30-minute F3
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            30,
            "F3",
        )
    ]
    == 5
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F3"
    ]
    == 30
)


for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            30,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate evaluation.
assert (
    (
        30,
        "F3",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_30_f3_result,
    lstm_final_30_f3_history,
    lstm_final_30_f3_actual_kw,
    lstm_final_30_f3_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        30,

    feature_family=
        "F3",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_30_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_30_f3_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_final_30_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_final_30_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_final_30_f3_result[
        "trainable_parameter_count"
    ]
    == 7_425
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f3_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_final_30_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_30_f3_result[
        "n_trainval"
    ]
    == 104_826
)


assert (
    lstm_final_30_f3_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f3_result[
        "frozen_best_epoch"
    ]
    == 5
)


assert (
    lstm_final_30_f3_result[
        "final_epochs_trained"
    ]
    == 5
)


assert (
    len(
        lstm_final_30_f3_history
    )
    == 5
)


assert (
    "val_loss"
    not in
    lstm_final_30_f3_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_30_f3_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_30_f3_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_30_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f3_actual_kw.shape
    ==
    lstm_final_30_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_30_f3_actual_kw
).all()


assert np.isfinite(
    lstm_final_30_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 30-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F0",
        )
    ],
    lstm_final_30_f3_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F2",
        )
    ],
    lstm_final_30_f3_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F3 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        30,
        "F3",
    )
] = (
    lstm_final_30_f3_result.copy()
)


lstm_final_test_histories[
    (
        30,
        "F3",
    )
] = (
    lstm_final_30_f3_history.copy()
)


lstm_final_test_actuals[
    (
        30,
        "F3",
    )
] = (
    lstm_final_30_f3_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        30,
        "F3",
    )
] = (
    lstm_final_30_f3_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


# ------------------------------------------------------------
# 12. Display final F3 TEST metrics
# ------------------------------------------------------------

lstm_final_30_f3_result_df = pd.DataFrame(
    [
        lstm_final_30_f3_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_30_f3_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_30_f3_history
)


# ------------------------------------------------------------
# 14. Descriptive F2 -> F3 TEST comparison
#
# Positive:
#       F3 has lower TEST MAE.
#
# Negative:
#       F3 has higher TEST MAE.
# ------------------------------------------------------------

f2_test_mae_30 = (
    lstm_final_test_results[
        (
            30,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


f3_test_mae_30 = (
    lstm_final_30_f3_result[
        "test_MAE_kW"
    ]
)


f2_to_f3_test_improvement_pct_30 = (
    (
        f2_test_mae_30
        -
        f3_test_mae_30
    )
    /
    f2_test_mae_30
    * 100.0
)


print(
    "\n30-min held-out TEST F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_test_mae_30:.6f} kW"
)

print(
    f"  F3 MAE: {f3_test_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_test_improvement_pct_30:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_30_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_30_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_30_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_30_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_30_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_30_f3_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_30_f3_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_30_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_30_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_30_f3_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_30_f3_prediction_kw,
    }
)


assert (
    lstm_final_30_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 30-minute F3 experiment
# ------------------------------------------------------------

lstm_30_f3_result_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F3_result.csv"
)


lstm_30_f3_history_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F3_history.csv"
)


lstm_30_f3_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F3_predictions.csv"
)


lstm_final_30_f3_result_df.to_csv(
    lstm_30_f3_result_path,
    index=False,
)


lstm_final_30_f3_history.to_csv(
    lstm_30_f3_history_path,
    index=False,
)


lstm_final_30_f3_prediction_table.to_csv(
    lstm_30_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_30_f3_result_path,
    lstm_30_f3_history_path,
    lstm_30_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 30-minute final TEST progression
# ------------------------------------------------------------

lstm_30min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\nCurrent 30-minute final TEST progression:"
)


display(
    lstm_30min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F3: COMPLETE"
)

print(
    "  Remaining combinations: 6"
)


print(
    "\nLSTM 30-min F3 final held-out evaluation: PASSED"
)

Epoch 1/5


2026-09-21 18:05:21.651188: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:05:21.651515: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2193
Epoch 2/5
410/410 - 5s - 11ms/step - loss: 0.1734
Epoch 3/5
410/410 - 5s - 12ms/step - loss: 0.1777
Epoch 4/5
410/410 - 4s - 11ms/step - loss: 0.1827
Epoch 5/5
410/410 - 5s - 11ms/step - loss: 0.2057


2026-09-21 18:05:46.209537: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:05:46.209869: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


30-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.357165 kW
  F3 MAE: 0.390874 kW
  Relative MAE change: -9.4379%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.604449,5.643751,7380,0



Current 30-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.459965,0.780223,0.776579
1,F1,1,0.373356,0.741601,0.798151
2,F2,6,0.357165,0.723139,0.808075
3,F3,5,0.390874,0.741480,0.798217



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F3: COMPLETE
  Remaining combinations: 6

LSTM 30-min F3 final held-out evaluation: PASSED


In [66]:
# ============================================================
# Step 52: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       30 minutes
#
# Feature family:
#       F4
#
# F4 = F3 + canonical Panchang one-hot features
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 Panchang indicators
#
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       1
#
# Auxiliary width:
#       127
#
# Trainable parameters:
#       10,529
#
#
# PRIMARY SCIENTIFIC COMPARISON
# -----------------------------
#
#       F3 -> F4
#
# This tests whether Panchang representation adds predictive
# information beyond the continuous astronomical controls
# already included in F3.
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       30-minute F4
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            30,
            "F4",
        )
    ]
    == 1
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F4"
    ]
    == 127
)


# Previous 30-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            30,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate evaluation.
assert (
    (
        30,
        "F4",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_30_f4_result,
    lstm_final_30_f4_history,
    lstm_final_30_f4_actual_kw,
    lstm_final_30_f4_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        30,

    feature_family=
        "F4",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_30_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_30_f4_result[
        "horizon_minutes"
    ]
    == 30
)


assert (
    lstm_final_30_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_final_30_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_final_30_f4_result[
        "trainable_parameter_count"
    ]
    == 10_529
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f4_result[
        "n_train"
    ]
    == 87_258
)


assert (
    lstm_final_30_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_30_f4_result[
        "n_trainval"
    ]
    == 104_826
)


assert (
    lstm_final_30_f4_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f4_result[
        "frozen_best_epoch"
    ]
    == 1
)


assert (
    lstm_final_30_f4_result[
        "final_epochs_trained"
    ]
    == 1
)


assert (
    len(
        lstm_final_30_f4_history
    )
    == 1
)


assert (
    "val_loss"
    not in
    lstm_final_30_f4_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_30_f4_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_30_f4_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_30_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_30_f4_actual_kw.shape
    ==
    lstm_final_30_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_30_f4_actual_kw
).all()


assert np.isfinite(
    lstm_final_30_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 30-minute TEST target population
#
# This is especially important for the paired F3 -> F4
# comparison that we will later bootstrap.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            30,
            "F3",
        )
    ],
    lstm_final_30_f4_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F4 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        30,
        "F4",
    )
] = (
    lstm_final_30_f4_result.copy()
)


lstm_final_test_histories[
    (
        30,
        "F4",
    )
] = (
    lstm_final_30_f4_history.copy()
)


lstm_final_test_actuals[
    (
        30,
        "F4",
    )
] = (
    lstm_final_30_f4_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        30,
        "F4",
    )
] = (
    lstm_final_30_f4_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


# ------------------------------------------------------------
# 12. Display final F4 TEST metrics
# ------------------------------------------------------------

lstm_final_30_f4_result_df = pd.DataFrame(
    [
        lstm_final_30_f4_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_30_f4_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_30_f4_history
)


# ------------------------------------------------------------
# 14. KEY F3 -> F4 descriptive TEST comparison
#
# Positive percentage:
#       F4 has lower TEST MAE.
#
# Negative percentage:
#       F4 has higher TEST MAE.
#
# This is the central Panchang-increment comparison.
# ------------------------------------------------------------

f3_test_mae_30 = (
    lstm_final_test_results[
        (
            30,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


f4_test_mae_30 = (
    lstm_final_30_f4_result[
        "test_MAE_kW"
    ]
)


f3_to_f4_test_improvement_pct_30 = (
    (
        f3_test_mae_30
        -
        f4_test_mae_30
    )
    /
    f3_test_mae_30
    * 100.0
)


print(
    "\n30-min held-out TEST F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_test_mae_30:.6f} kW"
)

print(
    f"  F4 MAE: {f4_test_mae_30:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_test_improvement_pct_30:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_30_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_30_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_30_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_30_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_30_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_30_f4_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_30_f4_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_30_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_30_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_30_f4_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_30_f4_prediction_kw,
    }
)


assert (
    lstm_final_30_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 30-minute F4 experiment
# ------------------------------------------------------------

lstm_30_f4_result_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F4_result.csv"
)


lstm_30_f4_history_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F4_history.csv"
)


lstm_30_f4_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_F4_predictions.csv"
)


lstm_final_30_f4_result_df.to_csv(
    lstm_30_f4_result_path,
    index=False,
)


lstm_final_30_f4_history.to_csv(
    lstm_30_f4_history_path,
    index=False,
)


lstm_final_30_f4_prediction_table.to_csv(
    lstm_30_f4_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_30_f4_result_path,
    lstm_30_f4_history_path,
    lstm_30_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Complete 30-minute final TEST progression
# ------------------------------------------------------------

lstm_30min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        30,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 30-minute final TEST progression:"
)


display(
    lstm_30min_final_progress
)


# ------------------------------------------------------------
# 20. Persist compact 30-minute summary
# ------------------------------------------------------------

lstm_30min_final_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_30min_summary.csv"
)


lstm_30min_final_progress.to_csv(
    lstm_30min_final_summary_path,
    index=False,
)


assert (
    lstm_30min_final_summary_path.exists()
)


assert (
    lstm_30min_final_summary_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 21. Current cross-horizon F3 -> F4 TEST summary
#
# Only 15 and 30 minutes are available at this point.
# The 60-minute result will be added later.
# ------------------------------------------------------------

lstm_f3_f4_partial_test_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "F3_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F3",
                    )
                ][
                    "test_MAE_kW"
                ],

            "F4_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F4",
                    )
                ][
                    "test_MAE_kW"
                ],
        }

        for horizon_minutes in [
            15,
            30,
        ]
    ]
)


lstm_f3_f4_partial_test_summary[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        lstm_f3_f4_partial_test_summary[
            "F3_MAE_kW"
        ]
        -
        lstm_f3_f4_partial_test_summary[
            "F4_MAE_kW"
        ]
    )
    /
    lstm_f3_f4_partial_test_summary[
        "F3_MAE_kW"
    ]
    * 100.0
)


print(
    "\nCurrent cross-horizon F3 -> F4 TEST summary:"
)


display(
    lstm_f3_f4_partial_test_summary
)


# ------------------------------------------------------------
# 22. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0-F4: NOT YET EVALUATED"
)

print(
    "  Remaining combinations: 5"
)


print(
    "\n30-minute F3 -> F4 TEST MAE change:"
)

print(
    f"  {f3_to_f4_test_improvement_pct_30:+.4f}%"
)


print(
    "\nLSTM 30-min F4 final held-out evaluation: PASSED"
)

2026-09-21 18:10:08.665404: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:10:08.665697: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.2135


2026-09-21 18:10:14.556289: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:10:14.556548: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds
0,LSTM,RNN_C1,30,F4,127,10529,104826,26496,1,1,0.368213,0.731963,0.803363,5.943593


...


Complete 30-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.459965,0.780223,0.776579
1,F1,1,0.373356,0.741601,0.798151
2,F2,6,0.357165,0.723139,0.808075
3,F3,5,0.390874,0.741480,0.798217
4,F4,1,0.368213,0.731963,0.803363



Current cross-horizon F3 -> F4 TEST summary:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.326384,0.329271,-0.884545
1,30,0.390874,0.368213,5.797532



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0-F4: NOT YET EVALUATED
  Remaining combinations: 5

30-minute F3 -> F4 TEST MAE change:
  +5.7975%

LSTM 30-min F4 final held-out evaluation: PASSED


In [67]:
# ============================================================
# Step 53: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F0
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       10
#
# Auxiliary width:
#       1
#
# Trainable parameters:
#       6,497
#
#
# Final fitting population:
#
#       TRAIN      = 87,252
#       VALIDATION = 17,568
#       -------------------
#       TRAIN+VAL  = 104,820
#
#
# Held-out TEST:
#
#       26,496 samples
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       60-minute F0
#
# No architecture, epoch, feature, scaling, or training-policy
# decision is changed using TEST performance.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            60,
            "F0",
        )
    ]
    == 10
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F0"
    ]
    == 1
)


assert (
    LSTM_FROZEN_ARCHITECTURE[
        "candidate_id"
    ]
    == "RNN_C1"
)


# ------------------------------------------------------------
# 2. Confirm that 15-min and 30-min grids are complete
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        assert (
            (
                horizon_minutes,
                feature_family,
            )
            in lstm_final_test_results
        )


# Prevent accidental duplicate evaluation.
assert (
    (
        60,
        "F0",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 3. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_60_f0_result,
    lstm_final_60_f0_history,
    lstm_final_60_f0_actual_kw,
    lstm_final_60_f0_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        60,

    feature_family=
        "F0",

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f0_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_60_f0_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_60_f0_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_final_60_f0_result[
        "feature_family"
    ]
    == "F0"
)


assert (
    lstm_final_60_f0_result[
        "auxiliary_dim"
    ]
    == 1
)


assert (
    lstm_final_60_f0_result[
        "trainable_parameter_count"
    ]
    == 6_497
)


# ------------------------------------------------------------
# 5. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f0_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_final_60_f0_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_60_f0_result[
        "n_trainval"
    ]
    == 104_820
)


assert (
    lstm_final_60_f0_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 6. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f0_result[
        "frozen_best_epoch"
    ]
    == 10
)


assert (
    lstm_final_60_f0_result[
        "final_epochs_trained"
    ]
    == 10
)


assert (
    len(
        lstm_final_60_f0_history
    )
    == 10
)


assert (
    "val_loss"
    not in
    lstm_final_60_f0_history.columns
)


# ------------------------------------------------------------
# 7. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_60_f0_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_60_f0_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_60_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 9. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f0_actual_kw.shape
    ==
    lstm_final_60_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_60_f0_actual_kw
).all()


assert np.isfinite(
    lstm_final_60_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Store successful final F0 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        60,
        "F0",
    )
] = (
    lstm_final_60_f0_result.copy()
)


lstm_final_test_histories[
    (
        60,
        "F0",
    )
] = (
    lstm_final_60_f0_history.copy()
)


lstm_final_test_actuals[
    (
        60,
        "F0",
    )
] = (
    lstm_final_60_f0_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        60,
        "F0",
    )
] = (
    lstm_final_60_f0_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


assert (
    len(
        lstm_final_test_results
    )
    == 11
)


# ------------------------------------------------------------
# 12. Display final F0 TEST metrics
# ------------------------------------------------------------

lstm_final_60_f0_result_df = pd.DataFrame(
    [
        lstm_final_60_f0_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_60_f0_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_60_f0_history
)


# ------------------------------------------------------------
# 14. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_60_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_60_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_60_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_60_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_60_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_60_f0_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_60_f0_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_60_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Construct prediction table
# ------------------------------------------------------------

lstm_final_60_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_60_f0_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_60_f0_prediction_kw,
    }
)


assert (
    lstm_final_60_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist final 60-minute F0 experiment immediately
# ------------------------------------------------------------

lstm_60_f0_result_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F0_result.csv"
)


lstm_60_f0_history_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F0_history.csv"
)


lstm_60_f0_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F0_predictions.csv"
)


lstm_final_60_f0_result_df.to_csv(
    lstm_60_f0_result_path,
    index=False,
)


lstm_final_60_f0_history.to_csv(
    lstm_60_f0_history_path,
    index=False,
)


lstm_final_60_f0_prediction_table.to_csv(
    lstm_60_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_60_f0_result_path,
    lstm_60_f0_history_path,
    lstm_60_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 18. Cross-horizon F0 summary
#
# Descriptive only:
# horizons are different forecasting tasks.
# ------------------------------------------------------------

lstm_f0_cross_horizon_progress = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F0",
                    )
                ][
                    "test_R2"
                ],
        }

        for horizon_minutes in [
            15,
            30,
            60,
        ]
    ]
)


print(
    "\nComplete cross-horizon F0 status:"
)


display(
    lstm_f0_cross_horizon_progress
)


# ------------------------------------------------------------
# 19. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0: YES"
)

print(
    "  Remaining combinations: 4"
)


print(
    "\nLSTM 60-min F0 final held-out evaluation: PASSED"
)

Epoch 1/10


2026-09-21 18:14:15.414258: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:14:15.414564: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2540
Epoch 2/10
410/410 - 4s - 10ms/step - loss: 0.2177
Epoch 3/10
410/410 - 4s - 10ms/step - loss: 0.2185
Epoch 4/10
410/410 - 4s - 10ms/step - loss: 0.2175
Epoch 5/10
410/410 - 4s - 11ms/step - loss: 0.2168
Epoch 6/10
410/410 - 4s - 10ms/step - loss: 0.2138
Epoch 7/10
410/410 - 4s - 10ms/step - loss: 0.2115
Epoch 8/10
410/410 - 4s - 11ms/step - loss: 0.2182
Epoch 9/10
410/410 - 4s - 10ms/step - loss: 0.2306
Epoch 10/10
410/410 - 4s - 10ms/step - loss: 0.2262


2026-09-21 18:14:59.657844: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:14:59.658132: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.289377,6.079421,4807,0



Complete cross-horizon F0 status:


,horizon_minutes,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,15,7,0.298274,0.620531,0.858677
1,30,10,0.459965,0.780223,0.776579
2,60,10,0.464594,0.863792,0.726155



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0: YES
  Remaining combinations: 4

LSTM 60-min F0 final held-out evaluation: PASSED


In [68]:
# ============================================================
# Step 54: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F1
#
# F1 = F0 + Gregorian calendar/time features
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       4
#
# Auxiliary width:
#       16
#
# Trainable parameters:
#       6,977
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       60-minute F1
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            60,
            "F1",
        )
    ]
    == 4
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F1"
    ]
    == 16
)


assert (
    (
        60,
        "F0",
    )
    in lstm_final_test_results
)


# Prevent accidental duplicate evaluation.
assert (
    (
        60,
        "F1",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_60_f1_result,
    lstm_final_60_f1_history,
    lstm_final_60_f1_actual_kw,
    lstm_final_60_f1_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        60,

    feature_family=
        "F1",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f1_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_60_f1_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_60_f1_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_final_60_f1_result[
        "feature_family"
    ]
    == "F1"
)


assert (
    lstm_final_60_f1_result[
        "auxiliary_dim"
    ]
    == 16
)


assert (
    lstm_final_60_f1_result[
        "trainable_parameter_count"
    ]
    == 6_977
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f1_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_final_60_f1_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_60_f1_result[
        "n_trainval"
    ]
    == 104_820
)


assert (
    lstm_final_60_f1_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f1_result[
        "frozen_best_epoch"
    ]
    == 4
)


assert (
    lstm_final_60_f1_result[
        "final_epochs_trained"
    ]
    == 4
)


assert (
    len(
        lstm_final_60_f1_history
    )
    == 4
)


assert (
    "val_loss"
    not in
    lstm_final_60_f1_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_60_f1_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_60_f1_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_60_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f1_actual_kw.shape
    ==
    lstm_final_60_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_60_f1_actual_kw
).all()


assert np.isfinite(
    lstm_final_60_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 60-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
    lstm_final_60_f1_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F1 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        60,
        "F1",
    )
] = (
    lstm_final_60_f1_result.copy()
)


lstm_final_test_histories[
    (
        60,
        "F1",
    )
] = (
    lstm_final_60_f1_history.copy()
)


lstm_final_test_actuals[
    (
        60,
        "F1",
    )
] = (
    lstm_final_60_f1_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        60,
        "F1",
    )
] = (
    lstm_final_60_f1_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


assert (
    len(
        lstm_final_test_results
    )
    == 12
)


# ------------------------------------------------------------
# 12. Display final F1 TEST metrics
# ------------------------------------------------------------

lstm_final_60_f1_result_df = pd.DataFrame(
    [
        lstm_final_60_f1_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_60_f1_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_60_f1_history
)


# ------------------------------------------------------------
# 14. Descriptive F0 -> F1 TEST comparison
#
# Positive:
#       F1 has lower TEST MAE.
#
# Negative:
#       F1 has higher TEST MAE.
# ------------------------------------------------------------

f0_test_mae_60 = (
    lstm_final_test_results[
        (
            60,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


f1_test_mae_60 = (
    lstm_final_60_f1_result[
        "test_MAE_kW"
    ]
)


f0_to_f1_test_improvement_pct_60 = (
    (
        f0_test_mae_60
        -
        f1_test_mae_60
    )
    /
    f0_test_mae_60
    * 100.0
)


print(
    "\n60-min held-out TEST F0 -> F1 comparison:"
)

print(
    f"  F0 MAE: {f0_test_mae_60:.6f} kW"
)

print(
    f"  F1 MAE: {f1_test_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f0_to_f1_test_improvement_pct_60:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_60_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_60_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_60_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_60_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_60_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_60_f1_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_60_f1_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_60_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_60_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_60_f1_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_60_f1_prediction_kw,
    }
)


assert (
    lstm_final_60_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 60-minute F1 experiment
# ------------------------------------------------------------

lstm_60_f1_result_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F1_result.csv"
)


lstm_60_f1_history_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F1_history.csv"
)


lstm_60_f1_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F1_predictions.csv"
)


lstm_final_60_f1_result_df.to_csv(
    lstm_60_f1_result_path,
    index=False,
)


lstm_final_60_f1_history.to_csv(
    lstm_60_f1_history_path,
    index=False,
)


lstm_final_60_f1_prediction_table.to_csv(
    lstm_60_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_60_f1_result_path,
    lstm_60_f1_history_path,
    lstm_60_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 60-minute final TEST progression
# ------------------------------------------------------------

lstm_60min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)


print(
    "\nCurrent 60-minute final TEST progression:"
)


display(
    lstm_60min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0-F1: COMPLETE"
)

print(
    "  Remaining combinations: 3"
)


print(
    "\nLSTM 60-min F1 final held-out evaluation: PASSED"
)

Epoch 1/4


2026-09-21 18:18:39.158712: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:18:39.159081: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 5s - 13ms/step - loss: 0.2502
Epoch 2/4
410/410 - 4s - 11ms/step - loss: 0.2258
Epoch 3/4
410/410 - 4s - 10ms/step - loss: 0.2229
Epoch 4/4
410/410 - 4s - 10ms/step - loss: 0.2165


2026-09-21 18:18:57.431015: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:18:57.431270: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


60-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.464594 kW
  F1 MAE: 0.466409 kW
  Relative MAE change: -0.3906%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.295578,5.757237,4214,0



Current 60-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.464594,0.863792,0.726155
1,F1,4,0.466409,0.865278,0.725212



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0-F1: COMPLETE
  Remaining combinations: 3

LSTM 60-min F1 final held-out evaluation: PASSED


In [69]:
# ============================================================
# Step 55: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F2
#
# F2 = F1 + independent solar-astronomy features
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       1
#
# Auxiliary width:
#       21
#
# Trainable parameters:
#       7,137
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       60-minute F2
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            60,
            "F2",
        )
    ]
    == 1
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F2"
    ]
    == 21
)


# Previous 60-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
]:

    assert (
        (
            60,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate evaluation.
assert (
    (
        60,
        "F2",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_60_f2_result,
    lstm_final_60_f2_history,
    lstm_final_60_f2_actual_kw,
    lstm_final_60_f2_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        60,

    feature_family=
        "F2",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f2_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_60_f2_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_60_f2_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_final_60_f2_result[
        "feature_family"
    ]
    == "F2"
)


assert (
    lstm_final_60_f2_result[
        "auxiliary_dim"
    ]
    == 21
)


assert (
    lstm_final_60_f2_result[
        "trainable_parameter_count"
    ]
    == 7_137
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f2_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_final_60_f2_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_60_f2_result[
        "n_trainval"
    ]
    == 104_820
)


assert (
    lstm_final_60_f2_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f2_result[
        "frozen_best_epoch"
    ]
    == 1
)


assert (
    lstm_final_60_f2_result[
        "final_epochs_trained"
    ]
    == 1
)


assert (
    len(
        lstm_final_60_f2_history
    )
    == 1
)


assert (
    "val_loss"
    not in
    lstm_final_60_f2_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_60_f2_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_60_f2_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_60_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f2_actual_kw.shape
    ==
    lstm_final_60_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_60_f2_actual_kw
).all()


assert np.isfinite(
    lstm_final_60_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 60-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
    lstm_final_60_f2_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F1",
        )
    ],
    lstm_final_60_f2_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F2 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        60,
        "F2",
    )
] = (
    lstm_final_60_f2_result.copy()
)


lstm_final_test_histories[
    (
        60,
        "F2",
    )
] = (
    lstm_final_60_f2_history.copy()
)


lstm_final_test_actuals[
    (
        60,
        "F2",
    )
] = (
    lstm_final_60_f2_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        60,
        "F2",
    )
] = (
    lstm_final_60_f2_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
    (60, "F2"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


assert (
    len(
        lstm_final_test_results
    )
    == 13
)


# ------------------------------------------------------------
# 12. Display final F2 TEST metrics
# ------------------------------------------------------------

lstm_final_60_f2_result_df = pd.DataFrame(
    [
        lstm_final_60_f2_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_60_f2_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_60_f2_history
)


# ------------------------------------------------------------
# 14. Descriptive F1 -> F2 TEST comparison
#
# Positive:
#       F2 has lower TEST MAE.
#
# Negative:
#       F2 has higher TEST MAE.
# ------------------------------------------------------------

f1_test_mae_60 = (
    lstm_final_test_results[
        (
            60,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


f2_test_mae_60 = (
    lstm_final_60_f2_result[
        "test_MAE_kW"
    ]
)


f1_to_f2_test_improvement_pct_60 = (
    (
        f1_test_mae_60
        -
        f2_test_mae_60
    )
    /
    f1_test_mae_60
    * 100.0
)


print(
    "\n60-min held-out TEST F1 -> F2 comparison:"
)

print(
    f"  F1 MAE: {f1_test_mae_60:.6f} kW"
)

print(
    f"  F2 MAE: {f2_test_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f1_to_f2_test_improvement_pct_60:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_60_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_60_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_60_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_60_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_60_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_60_f2_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_60_f2_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_60_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_60_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_60_f2_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_60_f2_prediction_kw,
    }
)


assert (
    lstm_final_60_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 60-minute F2 experiment
# ------------------------------------------------------------

lstm_60_f2_result_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F2_result.csv"
)


lstm_60_f2_history_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F2_history.csv"
)


lstm_60_f2_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F2_predictions.csv"
)


lstm_final_60_f2_result_df.to_csv(
    lstm_60_f2_result_path,
    index=False,
)


lstm_final_60_f2_history.to_csv(
    lstm_60_f2_history_path,
    index=False,
)


lstm_final_60_f2_prediction_table.to_csv(
    lstm_60_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_60_f2_result_path,
    lstm_60_f2_history_path,
    lstm_60_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 60-minute final TEST progression
# ------------------------------------------------------------

lstm_60min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)


print(
    "\nCurrent 60-minute final TEST progression:"
)


display(
    lstm_60min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0-F2: COMPLETE"
)

print(
    "  Remaining combinations: 2"
)


print(
    "\nLSTM 60-min F2 final held-out evaluation: PASSED"
)

2026-09-21 18:24:17.915363: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:24:17.915645: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 13ms/step - loss: 0.2662


2026-09-21 18:24:23.469751: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:24:23.470187: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds
0,LSTM,RNN_C1,60,F2,21,7137,104820,26496,1,1,0.477248,0.861871,0.727371,5.592073


...


60-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.466409 kW
  F2 MAE: 0.477248 kW
  Relative MAE change: -2.3239%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.356502,6.07432,6690,0



Current 60-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.464594,0.863792,0.726155
1,F1,4,0.466409,0.865278,0.725212
2,F2,1,0.477248,0.861871,0.727371



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0-F2: COMPLETE
  Remaining combinations: 2

LSTM 60-min F2 final held-out evaluation: PASSED


In [70]:
 # ============================================================
# Step 56: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F3
#
# F3 = F2 + continuous lunar / Sun-Moon astronomy
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       3
#
# Auxiliary width:
#       30
#
# Trainable parameters:
#       7,425
#
#
# IMPORTANT
# ---------
#
# This cell evaluates ONLY:
#
#       60-minute F3
#
# No architecture, epoch, feature, scaling, or other
# model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            60,
            "F3",
        )
    ]
    == 3
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F3"
    ]
    == 30
)


# Previous 60-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            60,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate final evaluation.
assert (
    (
        60,
        "F3",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_60_f3_result,
    lstm_final_60_f3_history,
    lstm_final_60_f3_actual_kw,
    lstm_final_60_f3_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        60,

    feature_family=
        "F3",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f3_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_60_f3_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_60_f3_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_final_60_f3_result[
        "feature_family"
    ]
    == "F3"
)


assert (
    lstm_final_60_f3_result[
        "auxiliary_dim"
    ]
    == 30
)


assert (
    lstm_final_60_f3_result[
        "trainable_parameter_count"
    ]
    == 7_425
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f3_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_final_60_f3_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_60_f3_result[
        "n_trainval"
    ]
    == 104_820
)


assert (
    lstm_final_60_f3_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f3_result[
        "frozen_best_epoch"
    ]
    == 3
)


assert (
    lstm_final_60_f3_result[
        "final_epochs_trained"
    ]
    == 3
)


assert (
    len(
        lstm_final_60_f3_history
    )
    == 3
)


assert (
    "val_loss"
    not in
    lstm_final_60_f3_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_60_f3_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_60_f3_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_60_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f3_actual_kw.shape
    ==
    lstm_final_60_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_60_f3_actual_kw
).all()


assert np.isfinite(
    lstm_final_60_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 60-minute TEST target population
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
    lstm_final_60_f3_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F2",
        )
    ],
    lstm_final_60_f3_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F3 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        60,
        "F3",
    )
] = (
    lstm_final_60_f3_result.copy()
)


lstm_final_test_histories[
    (
        60,
        "F3",
    )
] = (
    lstm_final_60_f3_history.copy()
)


lstm_final_test_actuals[
    (
        60,
        "F3",
    )
] = (
    lstm_final_60_f3_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        60,
        "F3",
    )
] = (
    lstm_final_60_f3_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify exact completed grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
    (60, "F2"),
    (60, "F3"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


assert (
    len(
        lstm_final_test_results
    )
    == 14
)


# ------------------------------------------------------------
# 12. Display final F3 TEST metrics
# ------------------------------------------------------------

lstm_final_60_f3_result_df = pd.DataFrame(
    [
        lstm_final_60_f3_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_60_f3_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_60_f3_history
)


# ------------------------------------------------------------
# 14. Descriptive F2 -> F3 TEST comparison
#
# Positive percentage:
#       F3 has lower TEST MAE.
#
# Negative percentage:
#       F3 has higher TEST MAE.
#
# This remains descriptive TEST analysis only.
# ------------------------------------------------------------

f2_test_mae_60 = (
    lstm_final_test_results[
        (
            60,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


f3_test_mae_60 = (
    lstm_final_60_f3_result[
        "test_MAE_kW"
    ]
)


f2_to_f3_test_improvement_pct_60 = (
    (
        f2_test_mae_60
        -
        f3_test_mae_60
    )
    /
    f2_test_mae_60
    * 100.0
)


print(
    "\n60-min held-out TEST F2 -> F3 comparison:"
)

print(
    f"  F2 MAE: {f2_test_mae_60:.6f} kW"
)

print(
    f"  F3 MAE: {f3_test_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f2_to_f3_test_improvement_pct_60:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_60_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_60_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_60_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_60_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_60_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_60_f3_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_60_f3_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_60_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_60_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_60_f3_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_60_f3_prediction_kw,
    }
)


assert (
    lstm_final_60_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 60-minute F3 experiment
# ------------------------------------------------------------

lstm_60_f3_result_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F3_result.csv"
)


lstm_60_f3_history_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F3_history.csv"
)


lstm_60_f3_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F3_predictions.csv"
)


lstm_final_60_f3_result_df.to_csv(
    lstm_60_f3_result_path,
    index=False,
)


lstm_final_60_f3_history.to_csv(
    lstm_60_f3_history_path,
    index=False,
)


lstm_final_60_f3_prediction_table.to_csv(
    lstm_60_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_60_f3_result_path,
    lstm_60_f3_history_path,
    lstm_60_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Current 60-minute final TEST progression
# ------------------------------------------------------------

lstm_60min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)


print(
    "\nCurrent 60-minute final TEST progression:"
)


display(
    lstm_60min_final_progress
)


# ------------------------------------------------------------
# 20. Evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0-F3: COMPLETE"
)

print(
    "  Remaining combinations: 1"
)


print(
    "\nLSTM 60-min F3 final held-out evaluation: PASSED"
)

Epoch 1/3


2026-09-21 18:32:13.510171: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:32:13.510530: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.2713
Epoch 2/3
410/410 - 4s - 10ms/step - loss: 0.2273
Epoch 3/3
410/410 - 4s - 10ms/step - loss: 0.2401


2026-09-21 18:32:27.566761: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:32:27.567021: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


60-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.477248 kW
  F3 MAE: 0.482546 kW
  Relative MAE change: -1.1103%
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.338147,5.821065,5187,0



Current 60-minute final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,10,0.464594,0.863792,0.726155
1,F1,4,0.466409,0.865278,0.725212
2,F2,1,0.477248,0.861871,0.727371
3,F3,3,0.482546,0.879914,0.715838



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0-F3: COMPLETE
  Remaining combinations: 1

LSTM 60-min F3 final held-out evaluation: PASSED


In [71]:
# ============================================================
# Step 57: Final held-out TEST evaluation
#
# Model:
#       LSTM
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F4
#
# F4 = F3 + canonical Panchang one-hot features
#
#       30 Tithi
#        2 Paksha
#       11 Karana
#       27 Nakshatra
#       27 Yoga
#       ----------
#       97 Panchang indicators
#
#
# Frozen architecture:
#       RNN_C1
#
# Frozen best epoch:
#       2
#
# Auxiliary width:
#       127
#
# Trainable parameters:
#       10,529
#
#
# PRIMARY SCIENTIFIC COMPARISON
# -----------------------------
#
#       F3 -> F4
#
# This tests whether Panchang representation adds predictive
# information beyond the continuous astronomical controls
# already included in F3.
#
#
# IMPORTANT
# ---------
#
# This is the FINAL LSTM held-out experiment.
#
# After this cell:
#
#       15-minute F0-F4 = complete
#       30-minute F0-F4 = complete
#       60-minute F0-F4 = complete
#
# Total final experiments = 15
#
# No model-selection decision is changed using TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Reconfirm frozen experiment
# ------------------------------------------------------------

assert (
    LSTM_FROZEN_BEST_EPOCHS[
        (
            60,
            "F4",
        )
    ]
    == 2
)


assert (
    LSTM_EXPECTED_AUX_DIMS[
        "F4"
    ]
    == 127
)


# Previous 60-minute experiments must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            60,
            previous_family,
        )
        in lstm_final_test_results
    )


# Prevent accidental duplicate final evaluation.
assert (
    (
        60,
        "F4",
    )
    not in lstm_final_test_results
)


# ------------------------------------------------------------
# 2. Run final TRAIN + VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    lstm_final_60_f4_result,
    lstm_final_60_f4_history,
    lstm_final_60_f4_actual_kw,
    lstm_final_60_f4_prediction_kw,
) = run_lstm_final_refit_and_test(

    horizon_minutes=
        60,

    feature_family=
        "F4",

    verbose=
        2,
)


# ------------------------------------------------------------
# 3. Strong protocol checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f4_result[
        "model"
    ]
    == "LSTM"
)


assert (
    lstm_final_60_f4_result[
        "candidate_id"
    ]
    == "RNN_C1"
)


assert (
    lstm_final_60_f4_result[
        "horizon_minutes"
    ]
    == 60
)


assert (
    lstm_final_60_f4_result[
        "feature_family"
    ]
    == "F4"
)


assert (
    lstm_final_60_f4_result[
        "auxiliary_dim"
    ]
    == 127
)


assert (
    lstm_final_60_f4_result[
        "trainable_parameter_count"
    ]
    == 10_529
)


# ------------------------------------------------------------
# 4. Population checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f4_result[
        "n_train"
    ]
    == 87_252
)


assert (
    lstm_final_60_f4_result[
        "n_validation"
    ]
    == 17_568
)


assert (
    lstm_final_60_f4_result[
        "n_trainval"
    ]
    == 104_820
)


assert (
    lstm_final_60_f4_result[
        "n_test"
    ]
    == 26_496
)


# ------------------------------------------------------------
# 5. Frozen epoch checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f4_result[
        "frozen_best_epoch"
    ]
    == 2
)


assert (
    lstm_final_60_f4_result[
        "final_epochs_trained"
    ]
    == 2
)


assert (
    len(
        lstm_final_60_f4_history
    )
    == 2
)


assert (
    "val_loss"
    not in
    lstm_final_60_f4_history.columns
)


# ------------------------------------------------------------
# 6. Frozen-policy checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    lstm_final_60_f4_result[
        "scaler_policy"
    ]
    == "original_train_only_scalers"
)


assert (
    lstm_final_60_f4_result[
        "refit_population"
    ]
    == "train_plus_validation"
)


# ------------------------------------------------------------
# 7. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        lstm_final_60_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 8. Prediction checks
# ------------------------------------------------------------

assert (
    lstm_final_60_f4_actual_kw.shape
    ==
    lstm_final_60_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    lstm_final_60_f4_actual_kw
).all()


assert np.isfinite(
    lstm_final_60_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 9. Confirm identical 60-minute TEST target population
#
# This exact equality is important because F3 -> F4 will later
# be evaluated using paired daily-block bootstrap resampling.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F3",
        )
    ],
    lstm_final_60_f4_actual_kw,
)


np.testing.assert_array_equal(
    lstm_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
    lstm_final_60_f4_actual_kw,
)


# ------------------------------------------------------------
# 10. Store successful final F4 result
# ------------------------------------------------------------

lstm_final_test_results[
    (
        60,
        "F4",
    )
] = (
    lstm_final_60_f4_result.copy()
)


lstm_final_test_histories[
    (
        60,
        "F4",
    )
] = (
    lstm_final_60_f4_history.copy()
)


lstm_final_test_actuals[
    (
        60,
        "F4",
    )
] = (
    lstm_final_60_f4_actual_kw.copy()
)


lstm_final_test_predictions[
    (
        60,
        "F4",
    )
] = (
    lstm_final_60_f4_prediction_kw.copy()
)


# ------------------------------------------------------------
# 11. Verify COMPLETE 15-experiment grid
# ------------------------------------------------------------

expected_completed_keys = {
    (15, "F0"),
    (15, "F1"),
    (15, "F2"),
    (15, "F3"),
    (15, "F4"),

    (30, "F0"),
    (30, "F1"),
    (30, "F2"),
    (30, "F3"),
    (30, "F4"),

    (60, "F0"),
    (60, "F1"),
    (60, "F2"),
    (60, "F3"),
    (60, "F4"),
}


assert set(
    lstm_final_test_results.keys()
) == expected_completed_keys


assert (
    len(
        lstm_final_test_results
    )
    == 15
)


assert (
    len(
        lstm_final_test_histories
    )
    == 15
)


assert (
    len(
        lstm_final_test_actuals
    )
    == 15
)


assert (
    len(
        lstm_final_test_predictions
    )
    == 15
)


# ------------------------------------------------------------
# 12. Display final F4 TEST metrics
# ------------------------------------------------------------

lstm_final_60_f4_result_df = pd.DataFrame(
    [
        lstm_final_60_f4_result
    ]
)


print(
    "\nFINAL HELD-OUT TEST RESULT"
)

print(
    "--------------------------"
)


display(
    lstm_final_60_f4_result_df[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Display final TRAIN + VALIDATION fitting history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    lstm_final_60_f4_history
)


# ------------------------------------------------------------
# 14. KEY F3 -> F4 descriptive TEST comparison
#
# Positive percentage:
#       F4 has lower TEST MAE.
#
# Negative percentage:
#       F4 has higher TEST MAE.
#
# This is the central Panchang-increment comparison.
# ------------------------------------------------------------

f3_test_mae_60 = (
    lstm_final_test_results[
        (
            60,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


f4_test_mae_60 = (
    lstm_final_60_f4_result[
        "test_MAE_kW"
    ]
)


f3_to_f4_test_improvement_pct_60 = (
    (
        f3_test_mae_60
        -
        f4_test_mae_60
    )
    /
    f3_test_mae_60
    * 100.0
)


print(
    "\n60-min held-out TEST F3 -> F4 comparison:"
)

print(
    f"  F3 MAE: {f3_test_mae_60:.6f} kW"
)

print(
    f"  F4 MAE: {f4_test_mae_60:.6f} kW"
)

print(
    "  Relative MAE change:",
    f"{f3_to_f4_test_improvement_pct_60:+.4f}%",
)

print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Audit raw UNCLIPPED predictions
# ------------------------------------------------------------

lstm_final_60_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                lstm_final_60_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                lstm_final_60_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                lstm_final_60_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                lstm_final_60_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                lstm_final_60_f4_prediction_kw
                < 0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                lstm_final_60_f4_prediction_kw
                > 10.0
            )
        ),
}


print(
    "\nUnclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            lstm_final_60_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. Construct prediction table
# ------------------------------------------------------------

lstm_final_60_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            lstm_final_60_f4_actual_kw,

        "predicted_P_Solar_kW":
            lstm_final_60_f4_prediction_kw,
    }
)


assert (
    lstm_final_60_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 17. Persist final 60-minute F4 experiment
# ------------------------------------------------------------

lstm_60_f4_result_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F4_result.csv"
)


lstm_60_f4_history_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F4_history.csv"
)


lstm_60_f4_predictions_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_F4_predictions.csv"
)


lstm_final_60_f4_result_df.to_csv(
    lstm_60_f4_result_path,
    index=False,
)


lstm_final_60_f4_history.to_csv(
    lstm_60_f4_history_path,
    index=False,
)


lstm_final_60_f4_prediction_table.to_csv(
    lstm_60_f4_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 18. Verify persisted F4 artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_60_f4_result_path,
    lstm_60_f4_history_path,
    lstm_60_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 19. Complete 60-minute final TEST progression
# ------------------------------------------------------------

lstm_60min_final_progress = pd.DataFrame(
    [
        {
            "feature_family":
                feature_family,

            "frozen_best_epoch":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "frozen_best_epoch"
                ],

            "test_MAE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_MAE_kW"
                ],

            "test_RMSE_kW":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_RMSE_kW"
                ],

            "test_R2":
                lstm_final_test_results[
                    (
                        60,
                        feature_family,
                    )
                ][
                    "test_R2"
                ],
        }

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)


print(
    "\nComplete 60-minute final TEST progression:"
)


display(
    lstm_60min_final_progress
)


# ------------------------------------------------------------
# 20. Persist compact 60-minute summary
# ------------------------------------------------------------

lstm_60min_final_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_60min_summary.csv"
)


lstm_60min_final_progress.to_csv(
    lstm_60min_final_summary_path,
    index=False,
)


assert (
    lstm_60min_final_summary_path.exists()
)


assert (
    lstm_60min_final_summary_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 21. Build COMPLETE 15-experiment final TEST table
# ------------------------------------------------------------

lstm_final_complete_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        result = (
            lstm_final_test_results[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
        )

        lstm_final_complete_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "frozen_best_epoch":
                    result[
                        "frozen_best_epoch"
                    ],

                "auxiliary_dim":
                    result[
                        "auxiliary_dim"
                    ],

                "trainable_parameter_count":
                    result[
                        "trainable_parameter_count"
                    ],

                "test_MAE_kW":
                    result[
                        "test_MAE_kW"
                    ],

                "test_RMSE_kW":
                    result[
                        "test_RMSE_kW"
                    ],

                "test_R2":
                    result[
                        "test_R2"
                    ],
            }
        )


lstm_final_complete_test_table = pd.DataFrame(
    lstm_final_complete_rows
)


assert (
    lstm_final_complete_test_table.shape
    ==
    (
        15,
        8,
    )
)


print(
    "\nCOMPLETE LSTM FINAL HELD-OUT TEST TABLE:"
)


display(
    lstm_final_complete_test_table
)


# ------------------------------------------------------------
# 22. Persist COMPLETE 15-experiment final TEST table
# ------------------------------------------------------------

lstm_final_complete_test_table_path = (
    RESULTS_DIR
    /
    "lstm_final_complete_test_metrics.csv"
)


lstm_final_complete_test_table.to_csv(
    lstm_final_complete_test_table_path,
    index=False,
)


assert (
    lstm_final_complete_test_table_path.exists()
)


assert (
    lstm_final_complete_test_table_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 23. Build complete cross-horizon F3 -> F4 summary
#
# Positive percentage:
#       F4 lower MAE than F3.
#
# Negative percentage:
#       F4 higher MAE than F3.
# ------------------------------------------------------------

lstm_f3_f4_final_test_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon_minutes,

            "F3_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F3",
                    )
                ][
                    "test_MAE_kW"
                ],

            "F4_MAE_kW":
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F4",
                    )
                ][
                    "test_MAE_kW"
                ],
        }

        for horizon_minutes in [
            15,
            30,
            60,
        ]
    ]
)


lstm_f3_f4_final_test_summary[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        lstm_f3_f4_final_test_summary[
            "F3_MAE_kW"
        ]
        -
        lstm_f3_f4_final_test_summary[
            "F4_MAE_kW"
        ]
    )
    /
    lstm_f3_f4_final_test_summary[
        "F3_MAE_kW"
    ]
    * 100.0
)


print(
    "\nComplete cross-horizon F3 -> F4 TEST summary:"
)


display(
    lstm_f3_f4_final_test_summary
)


# ------------------------------------------------------------
# 24. Persist F3 -> F4 summary
# ------------------------------------------------------------

lstm_f3_f4_final_test_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_test_summary.csv"
)


lstm_f3_f4_final_test_summary.to_csv(
    lstm_f3_f4_final_test_summary_path,
    index=False,
)


assert (
    lstm_f3_f4_final_test_summary_path.exists()
)


assert (
    lstm_f3_f4_final_test_summary_path.stat().st_size
    > 0
)


# ------------------------------------------------------------
# 25. Final evaluation-state checkpoint
# ------------------------------------------------------------

print(
    "\nFinal LSTM held-out evaluations completed:"
)

print(
    "  15-min F0-F4: COMPLETE"
)

print(
    "  30-min F0-F4: COMPLETE"
)

print(
    "  60-min F0-F4: COMPLETE"
)

print(
    "  Total completed final experiments: 15 / 15"
)

print(
    "  Remaining combinations: 0"
)


print(
    "\n60-minute F3 -> F4 TEST MAE change:"
)

print(
    f"  {f3_to_f4_test_improvement_pct_60:+.4f}%"
)


print(
    "\nALL LSTM FINAL HELD-OUT EVALUATIONS: COMPLETE"
)

Epoch 1/2


2026-09-21 18:38:31.498645: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 18:38:31.498992: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.2638
Epoch 2/2
410/410 - 4s - 10ms/step - loss: 0.2219


2026-09-21 18:38:41.356660: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 18:38:41.356949: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


FINAL HELD-OUT TEST RESULT
--------------------------


...


COMPLETE LSTM FINAL HELD-OUT TEST TABLE:


,horizon_minutes,feature_family,frozen_best_epoch,auxiliary_dim,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,7,1,6497,0.298274,0.620531,0.858677
1,15,F1,1,16,6977,0.294215,0.622662,0.857704
2,15,F2,2,21,7137,0.298937,0.622725,0.857676
3,15,F3,2,30,7425,0.326384,0.633885,0.852529
4,15,F4,7,127,10529,0.329271,0.636115,0.851489
5,30,F0,10,1,6497,0.459965,0.780223,0.776579
6,30,F1,1,16,6977,0.373356,0.741601,0.798151
7,30,F2,6,21,7137,0.357165,0.723139,0.808075
8,30,F3,5,30,7425,0.390874,0.741480,0.798217
9,30,F4,1,127,10529,0.368213,0.731963,0.803363



Complete cross-horizon F3 -> F4 TEST summary:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.326384,0.329271,-0.884545
1,30,0.390874,0.368213,5.797532
2,60,0.482546,0.489504,-1.441883



Final LSTM held-out evaluations completed:
  15-min F0-F4: COMPLETE
  30-min F0-F4: COMPLETE
  60-min F0-F4: COMPLETE
  Total completed final experiments: 15 / 15
  Remaining combinations: 0

60-minute F3 -> F4 TEST MAE change:
  -1.4419%

ALL LSTM FINAL HELD-OUT EVALUATIONS: COMPLETE


In [73]:
# ============================================================
# Step 58B: Correct canonical TEST timestamp/daylight audit
#
# CORRECTION
# ----------
#
# deep_scaled_data[horizon] stores the split masks directly as:
#
#       train_mask
#       validation_mask
#       test_mask
#
# rather than inside a nested "masks" dictionary.
#
#
# PURPOSE
# -------
#
# Establish the exact mapping:
#
#       held-out prediction row
#           ->
#       canonical valid_time
#           ->
#       solar_daylight
#           ->
#       UTC day
#
# before computing daylight-only metrics or bootstrap results.
#
#
# This cell performs:
#
#       NO training
#       NO retraining
#       NO model selection
#       NO bootstrap
#
# It is purely an alignment/provenance audit.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm complete final LSTM state
# ------------------------------------------------------------

assert len(
    lstm_final_test_results
) == 15

assert len(
    lstm_final_test_predictions
) == 15

assert len(
    lstm_final_test_actuals
) == 15


print(
    "Final LSTM grid confirmed: 15 / 15 experiments."
)


# ------------------------------------------------------------
# 2. Audit canonical split masks
#
# Verify that the direct masks have exactly the same length as
# each canonical base modelling table.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "DIRECT SPLIT-MASK AUDIT"
)

print(
    "============================================================"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )

    scaled_object = (
        deep_scaled_data[
            horizon_minutes
        ]
    )


    print(
        f"\nHorizon: {horizon_minutes} minutes"
    )


    print(
        "Canonical table rows:",
        len(
            base_table
        ),
    )


    for mask_name in [
        "train_mask",
        "validation_mask",
        "test_mask",
    ]:

        assert (
            mask_name
            in scaled_object
        )


        mask = np.asarray(
            scaled_object[
                mask_name
            ]
        )


        print(
            f"  {mask_name}:"
        )

        print(
            "    shape =",
            mask.shape,
        )

        print(
            "    dtype =",
            mask.dtype,
        )

        print(
            "    True count =",
            int(
                mask.sum()
            ),
        )


        assert (
            mask.shape
            ==
            (
                len(
                    base_table
                ),
            )
        )


        assert (
            mask.dtype
            ==
            bool
        )


    # --------------------------------------------------------
    # Expected canonical split sizes
    # --------------------------------------------------------

    expected_counts = {
        15: {
            "train_mask": 87_261,
            "validation_mask": 17_568,
            "test_mask": 26_496,
        },

        30: {
            "train_mask": 87_258,
            "validation_mask": 17_568,
            "test_mask": 26_496,
        },

        60: {
            "train_mask": 87_252,
            "validation_mask": 17_568,
            "test_mask": 26_496,
        },
    }


    for mask_name, expected_count in (
        expected_counts[
            horizon_minutes
        ].items()
    ):

        mask = np.asarray(
            scaled_object[
                mask_name
            ]
        )


        assert (
            int(
                mask.sum()
            )
            ==
            expected_count
        )


# ------------------------------------------------------------
# 3. Cross-check mask membership against the explicit
#    'split' column in deep_base_modelling_tables
#
# This ensures that the modelling table and scaled-data masks
# refer to exactly the same rows.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "MASK <-> SPLIT COLUMN CONSISTENCY"
)

print(
    "============================================================"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    split_column = (
        base_table[
            "split"
        ]
        .astype(
            str
        )
        .to_numpy()
    )


    train_mask = np.asarray(
        deep_scaled_data[
            horizon_minutes
        ][
            "train_mask"
        ]
    )


    validation_mask = np.asarray(
        deep_scaled_data[
            horizon_minutes
        ][
            "validation_mask"
        ]
    )


    test_mask = np.asarray(
        deep_scaled_data[
            horizon_minutes
        ][
            "test_mask"
        ]
    )


    np.testing.assert_array_equal(
        train_mask,
        split_column
        ==
        "train",
    )


    np.testing.assert_array_equal(
        validation_mask,
        split_column
        ==
        "validation",
    )


    np.testing.assert_array_equal(
        test_mask,
        split_column
        ==
        "test",
    )


    print(
        f"{horizon_minutes}-min: "
        "mask and explicit split column agree exactly."
    )


# ------------------------------------------------------------
# 4. Extract canonical held-out TEST metadata
#
# IMPORTANT:
#
# Predictions are ordered exactly according to the TEST mask,
# so this table will later provide the valid timestamps for
# each of the 26,496 prediction rows.
# ------------------------------------------------------------

deep_test_metadata = {}


print(
    "\n============================================================"
)

print(
    "CANONICAL HELD-OUT TEST METADATA"
)

print(
    "============================================================"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    test_mask = np.asarray(
        deep_scaled_data[
            horizon_minutes
        ][
            "test_mask"
        ]
    )


    test_metadata = (
        base_table
        .loc[
            test_mask,
            [
                "valid_time",
                "target_P_Solar_kW",
                "split",
            ],
        ]
        .copy()
    )


    # Store issue time explicitly rather than leaving it only
    # in the DataFrame index.
    test_metadata.insert(
        0,
        "issue_time",
        test_metadata.index,
    )


    test_metadata = (
        test_metadata
        .reset_index(
            drop=True
        )
    )


    assert (
        test_metadata.shape
        ==
        (
            26_496,
            4,
        )
    )


    assert (
        test_metadata[
            "split"
        ]
        .eq(
            "test"
        )
        .all()
    )


    # Valid time must be timezone-aware UTC.
    assert isinstance(
        test_metadata[
            "valid_time"
        ].dtype,
        pd.DatetimeTZDtype,
    )


    assert (
        str(
            test_metadata[
                "valid_time"
            ]
            .dt
            .tz
        )
        ==
        "UTC"
    )


    # --------------------------------------------------------
    # Confirm target alignment against saved/in-memory final
    # predictions for every feature family at this horizon.
    # --------------------------------------------------------

    metadata_target = (
        test_metadata[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )


    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        np.testing.assert_array_equal(
            metadata_target,
            np.asarray(
                lstm_final_test_actuals[
                    (
                        horizon_minutes,
                        feature_family,
                    )
                ],
                dtype=float,
            ),
        )


    deep_test_metadata[
        horizon_minutes
    ] = (
        test_metadata
    )


    print(
        f"\n{horizon_minutes}-min TEST metadata:"
    )

    print(
        "  rows:",
        len(
            test_metadata
        ),
    )

    print(
        "  first issue_time:",
        test_metadata[
            "issue_time"
        ].iloc[
            0
        ],
    )

    print(
        "  last issue_time:",
        test_metadata[
            "issue_time"
        ].iloc[
            -1
        ],
    )

    print(
        "  first valid_time:",
        test_metadata[
            "valid_time"
        ].iloc[
            0
        ],
    )

    print(
        "  last valid_time:",
        test_metadata[
            "valid_time"
        ].iloc[
            -1
        ],
    )

    print(
        "  unique UTC valid-date blocks:",
        test_metadata[
            "valid_time"
        ]
        .dt
        .normalize()
        .nunique(),
    )


# ------------------------------------------------------------
# 5. Search existing notebook DataFrames for solar_daylight
#
# We intentionally reuse the already-computed F2 solar feature
# rather than recomputing solar geometry.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "EXISTING DATAFRAMES CONTAINING solar_daylight"
)

print(
    "============================================================"
)


solar_daylight_dataframe_names = []


for variable_name, variable_value in list(
    globals().items()
):

    if not isinstance(
        variable_value,
        pd.DataFrame,
    ):

        continue


    if (
        "solar_daylight"
        not in
        variable_value.columns
    ):

        continue


    solar_daylight_dataframe_names.append(
        variable_name
    )


    print(
        f"\nObject: {variable_name}"
    )

    print(
        "  shape:",
        variable_value.shape,
    )

    print(
        "  index type:",
        type(
            variable_value.index
        ),
    )

    print(
        "  index name:",
        variable_value.index.name,
    )


    if len(
        variable_value
    ) > 0:

        print(
            "  index start:",
            variable_value.index[
                0
            ],
        )

        print(
            "  index end:",
            variable_value.index[
                -1
            ],
        )


    print(
        "  solar_daylight unique values:",
        np.sort(
            variable_value[
                "solar_daylight"
            ]
            .dropna()
            .unique()
        ),
    )


    print(
        "  solar_daylight count:",
        int(
            (
                variable_value[
                    "solar_daylight"
                ]
                ==
                1
            )
            .sum()
        ),
    )


print(
    "\nNumber of matching DataFrames:",
    len(
        solar_daylight_dataframe_names
    ),
)


print(
    "Matching object names:",
    solar_daylight_dataframe_names,
)


# ------------------------------------------------------------
# 6. Also inspect dictionary objects that may contain an F2
#    DataFrame under horizon-specific or family-specific keys.
#
# We print metadata only.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "DICTIONARY OBJECTS WITH solar_daylight DATAFRAMES"
)

print(
    "============================================================"
)


solar_daylight_dictionary_hits = []


for variable_name, variable_value in list(
    globals().items()
):

    if not isinstance(
        variable_value,
        dict,
    ):

        continue


    for key, nested_value in (
        variable_value.items()
    ):

        if not isinstance(
            nested_value,
            pd.DataFrame,
        ):

            continue


        if (
            "solar_daylight"
            not in
            nested_value.columns
        ):

            continue


        solar_daylight_dictionary_hits.append(
            (
                variable_name,
                key,
            )
        )


        print(
            f"\n{variable_name}[{repr(key)}]"
        )

        print(
            "  shape:",
            nested_value.shape,
        )

        print(
            "  index type:",
            type(
                nested_value.index
            ),
        )

        print(
            "  index name:",
            nested_value.index.name,
        )


        if len(
            nested_value
        ) > 0:

            print(
                "  index start:",
                nested_value.index[
                    0
                ],
            )

            print(
                "  index end:",
                nested_value.index[
                    -1
                ],
            )


        print(
            "  daylight count:",
            int(
                (
                    nested_value[
                        "solar_daylight"
                    ]
                    ==
                    1
                )
                .sum()
            ),
        )


print(
    "\nDictionary hits:",
    solar_daylight_dictionary_hits,
)


# ------------------------------------------------------------
# 7. Audit checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "STEP 58B ALIGNMENT AUDIT COMPLETE"
)

print(
    "============================================================"
)

print(
    "Canonical TEST valid-time metadata has been established."
)

print(
    "No model was trained or modified."
)

print(
    "No daylight metric or bootstrap result has been computed yet."
)

Final LSTM grid confirmed: 15 / 15 experiments.

DIRECT SPLIT-MASK AUDIT

Horizon: 15 minutes
Canonical table rows: 131325
  train_mask:
    shape = (131325,)
    dtype = bool
    True count = 87261
  validation_mask:
    shape = (131325,)
    dtype = bool
    True count = 17568
  test_mask:
    shape = (131325,)
    dtype = bool
    True count = 26496

Horizon: 30 minutes
Canonical table rows: 131322
  train_mask:
    shape = (131322,)
    dtype = bool
    True count = 87258
  validation_mask:
    shape = (131322,)
    dtype = bool
    True count = 17568
  test_mask:
    shape = (131322,)
    dtype = bool
    True count = 26496

Horizon: 60 minutes
Canonical table rows: 131316
  train_mask:
    shape = (131316,)
    dtype = bool
    True count = 87252
  validation_mask:
    shape = (131316,)
    dtype = bool
    True count = 17568
  test_mask:
    shape = (131316,)
    dtype = bool
    True count = 26496

MASK <-> SPLIT COLUMN CONSISTENCY
15-min: mask and explicit split column agree e

In [74]:
# ============================================================
# Step 59: Canonical TEST alignment and daylight-only metrics
#
# PURPOSE
# -------
#
# We now have:
#
#       - exact held-out valid timestamps
#       - exact horizon-specific TEST masks
#       - the existing canonical solar_daylight feature
#       - final predictions for all 15 LSTM experiments
#
# This cell:
#
#       1. proves exact row alignment between:
#
#              deep_base_modelling_tables[h]
#              deep_auxiliary_raw[h]
#
#       2. constructs canonical TEST metadata containing:
#
#              issue_time
#              valid_time
#              UTC day
#              solar_daylight
#              actual target
#
#       3. recomputes ALL-TIME metrics as an alignment check
#
#       4. computes DAYLIGHT-ONLY:
#
#              MAE
#              RMSE
#              R²
#
#          for all:
#
#              3 horizons × 5 feature families = 15 models
#
#       5. produces the daylight F3 -> F4 summary
#
#
# IMPORTANT
# ---------
#
# NO models are trained or retrained.
#
# NO clipping is applied.
#
# NO parameters are changed.
#
# These are post-hoc evaluations of the already-frozen
# held-out TEST predictions.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm complete final state
# ------------------------------------------------------------

HORIZONS = [
    15,
    30,
    60,
]


FEATURE_FAMILIES = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]


assert (
    len(
        lstm_final_test_results
    )
    == 15
)


assert (
    len(
        lstm_final_test_predictions
    )
    == 15
)


assert (
    len(
        lstm_final_test_actuals
    )
    == 15
)


# ------------------------------------------------------------
# 2. Construct exact horizon-specific TEST alignment tables
#
# deep_auxiliary_raw[h] is indexed by issue_time, exactly like
# deep_base_modelling_tables[h].
#
# Its deterministic features, including solar_daylight, were
# evaluated at the forecast valid time.
# ------------------------------------------------------------

lstm_test_alignment = {}


for horizon_minutes in HORIZONS:

    base_table = (
        deep_base_modelling_tables[
            horizon_minutes
        ]
    )


    auxiliary_table_h = (
        deep_auxiliary_raw[
            horizon_minutes
        ]
    )


    # --------------------------------------------------------
    # Exact full-population alignment checks
    # --------------------------------------------------------

    assert (
        len(
            base_table
        )
        ==
        len(
            auxiliary_table_h
        )
    )


    assert (
        base_table.index.equals(
            auxiliary_table_h.index
        )
    )


    assert (
        "solar_daylight"
        in auxiliary_table_h.columns
    )


    test_mask = np.asarray(
        deep_scaled_data[
            horizon_minutes
        ][
            "test_mask"
        ]
    )


    assert (
        test_mask.dtype
        ==
        bool
    )


    assert (
        int(
            test_mask.sum()
        )
        ==
        26_496
    )


    # --------------------------------------------------------
    # Select exact TEST rows from both canonical sources
    # --------------------------------------------------------

    test_base = (
        base_table
        .loc[
            test_mask
        ]
        .copy()
    )


    test_auxiliary = (
        auxiliary_table_h
        .loc[
            test_mask
        ]
        .copy()
    )


    # TEST row order must remain exactly identical.
    assert (
        test_base.index.equals(
            test_auxiliary.index
        )
    )


    # --------------------------------------------------------
    # Build canonical metadata table
    # --------------------------------------------------------

    alignment = pd.DataFrame(
        {
            "issue_time":
                test_base.index,

            "valid_time":
                test_base[
                    "valid_time"
                ].to_numpy(),

            "actual_P_Solar_kW":
                test_base[
                    "target_P_Solar_kW"
                ]
                .to_numpy(
                    dtype=float
                ),

            "solar_daylight":
                test_auxiliary[
                    "solar_daylight"
                ]
                .to_numpy(),
        }
    )


    # --------------------------------------------------------
    # Canonicalize daylight to Boolean
    # --------------------------------------------------------

    daylight_raw = (
        alignment[
            "solar_daylight"
        ]
        .to_numpy()
    )


    unique_daylight_values = np.unique(
        daylight_raw
    )


    assert set(
        unique_daylight_values.tolist()
    ).issubset(
        {
            0,
            1,
            0.0,
            1.0,
        }
    )


    alignment[
        "solar_daylight"
    ] = (
        alignment[
            "solar_daylight"
        ]
        .astype(
            int
        )
    )


    alignment[
        "is_daylight"
    ] = (
        alignment[
            "solar_daylight"
        ]
        .eq(
            1
        )
    )


    # --------------------------------------------------------
    # Add UTC day block identifier using VALID time
    #
    # This is the day definition that will later be used for
    # paired daily-block bootstrap resampling.
    # --------------------------------------------------------

    alignment[
        "utc_valid_day"
    ] = (
        pd.to_datetime(
            alignment[
                "valid_time"
            ],
            utc=True,
        )
        .dt
        .normalize()
    )


    # --------------------------------------------------------
    # Structural checks
    # --------------------------------------------------------

    assert (
        alignment.shape
        ==
        (
            26_496,
            6,
        )
    )


    assert (
        alignment[
            "valid_time"
        ]
        .isna()
        .sum()
        ==
        0
    )


    assert (
        alignment[
            "actual_P_Solar_kW"
        ]
        .isna()
        .sum()
        ==
        0
    )


    assert (
        alignment[
            "solar_daylight"
        ]
        .isna()
        .sum()
        ==
        0
    )


    assert (
        alignment[
            "utc_valid_day"
        ]
        .nunique()
        ==
        92
    )


    # --------------------------------------------------------
    # Confirm TEST target is exactly the target already used
    # by every final feature-family prediction at this horizon.
    # --------------------------------------------------------

    canonical_actual = (
        alignment[
            "actual_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )


    for feature_family in FEATURE_FAMILIES:

        np.testing.assert_array_equal(
            canonical_actual,
            np.asarray(
                lstm_final_test_actuals[
                    (
                        horizon_minutes,
                        feature_family,
                    )
                ],
                dtype=float,
            ),
        )


    lstm_test_alignment[
        horizon_minutes
    ] = (
        alignment
    )


# ------------------------------------------------------------
# 3. Display canonical alignment summary
# ------------------------------------------------------------

alignment_summary_rows = []


for horizon_minutes in HORIZONS:

    alignment = (
        lstm_test_alignment[
            horizon_minutes
        ]
    )


    n_total = len(
        alignment
    )


    n_daylight = int(
        alignment[
            "is_daylight"
        ]
        .sum()
    )


    alignment_summary_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "n_test_total":
                n_total,

            "n_test_daylight":
                n_daylight,

            "daylight_fraction":
                (
                    n_daylight
                    /
                    n_total
                ),

            "first_valid_time":
                alignment[
                    "valid_time"
                ].iloc[
                    0
                ],

            "last_valid_time":
                alignment[
                    "valid_time"
                ].iloc[
                    -1
                ],

            "utc_day_blocks":
                alignment[
                    "utc_valid_day"
                ]
                .nunique(),
        }
    )


lstm_test_alignment_summary = pd.DataFrame(
    alignment_summary_rows
)


print(
    "\nCANONICAL HELD-OUT TEST ALIGNMENT SUMMARY"
)

print(
    "=========================================="
)


display(
    lstm_test_alignment_summary
)


# ------------------------------------------------------------
# 4. Compute ALL-TIME and DAYLIGHT-ONLY metrics
#
# We recompute all-time metrics here first so that we can
# verify that timestamp alignment has not changed row order.
# ------------------------------------------------------------

daylight_metric_rows = []


for horizon_minutes in HORIZONS:

    alignment = (
        lstm_test_alignment[
            horizon_minutes
        ]
    )


    actual_kw = (
        alignment[
            "actual_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )


    daylight_mask = (
        alignment[
            "is_daylight"
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert (
        daylight_mask.shape
        ==
        (
            26_496,
        )
    )


    assert (
        daylight_mask.sum()
        > 0
    )


    for feature_family in FEATURE_FAMILIES:

        prediction_kw = np.asarray(
            lstm_final_test_predictions[
                (
                    horizon_minutes,
                    feature_family,
                )
            ],
            dtype=float,
        )


        assert (
            prediction_kw.shape
            ==
            (
                26_496,
            )
        )


        # ----------------------------------------------------
        # Recompute ALL-TIME metrics
        # ----------------------------------------------------

        all_metrics = (
            compute_regression_metrics_kw(
                actual_kw,
                prediction_kw,
            )
        )


        # ----------------------------------------------------
        # Verify recomputed metrics against the already-saved
        # final held-out metrics.
        # ----------------------------------------------------

        stored_result = (
            lstm_final_test_results[
                (
                    horizon_minutes,
                    feature_family,
                )
            ]
        )


        np.testing.assert_allclose(
            all_metrics[
                "MAE_kW"
            ],
            stored_result[
                "test_MAE_kW"
            ],
            rtol=0.0,
            atol=1e-10,
        )


        np.testing.assert_allclose(
            all_metrics[
                "RMSE_kW"
            ],
            stored_result[
                "test_RMSE_kW"
            ],
            rtol=0.0,
            atol=1e-10,
        )


        np.testing.assert_allclose(
            all_metrics[
                "R2"
            ],
            stored_result[
                "test_R2"
            ],
            rtol=0.0,
            atol=1e-10,
        )


        # ----------------------------------------------------
        # Compute DAYLIGHT-ONLY metrics
        # ----------------------------------------------------

        daylight_actual_kw = (
            actual_kw[
                daylight_mask
            ]
        )


        daylight_prediction_kw = (
            prediction_kw[
                daylight_mask
            ]
        )


        daylight_metrics = (
            compute_regression_metrics_kw(
                daylight_actual_kw,
                daylight_prediction_kw,
            )
        )


        daylight_metric_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "feature_family":
                    feature_family,

                "n_test_total":
                    int(
                        len(
                            actual_kw
                        )
                    ),

                "n_test_daylight":
                    int(
                        daylight_mask.sum()
                    ),

                "all_MAE_kW":
                    float(
                        all_metrics[
                            "MAE_kW"
                        ]
                    ),

                "all_RMSE_kW":
                    float(
                        all_metrics[
                            "RMSE_kW"
                        ]
                    ),

                "all_R2":
                    float(
                        all_metrics[
                            "R2"
                        ]
                    ),

                "daylight_MAE_kW":
                    float(
                        daylight_metrics[
                            "MAE_kW"
                        ]
                    ),

                "daylight_RMSE_kW":
                    float(
                        daylight_metrics[
                            "RMSE_kW"
                        ]
                    ),

                "daylight_R2":
                    float(
                        daylight_metrics[
                            "R2"
                        ]
                    ),
            }
        )


# ------------------------------------------------------------
# 5. Build complete daylight metric table
# ------------------------------------------------------------

lstm_final_daylight_metrics = pd.DataFrame(
    daylight_metric_rows
)


assert (
    lstm_final_daylight_metrics.shape
    ==
    (
        15,
        10,
    )
)


print(
    "\nCOMPLETE LSTM DAYLIGHT-ONLY TEST METRICS"
)

print(
    "========================================"
)


display(
    lstm_final_daylight_metrics
)


# ------------------------------------------------------------
# 6. Build key F3 -> F4 daylight comparison
#
# Positive percentage:
#
#       F4 has LOWER daylight MAE than F3.
#
# Negative percentage:
#
#       F4 has HIGHER daylight MAE than F3.
# ------------------------------------------------------------

f3_f4_daylight_rows = []


for horizon_minutes in HORIZONS:

    horizon_metrics = (
        lstm_final_daylight_metrics[
            lstm_final_daylight_metrics[
                "horizon_minutes"
            ]
            .eq(
                horizon_minutes
            )
        ]
    )


    f3_row = (
        horizon_metrics[
            horizon_metrics[
                "feature_family"
            ]
            .eq(
                "F3"
            )
        ]
        .iloc[
            0
        ]
    )


    f4_row = (
        horizon_metrics[
            horizon_metrics[
                "feature_family"
            ]
            .eq(
                "F4"
            )
        ]
        .iloc[
            0
        ]
    )


    f3_daylight_mae = float(
        f3_row[
            "daylight_MAE_kW"
        ]
    )


    f4_daylight_mae = float(
        f4_row[
            "daylight_MAE_kW"
        ]
    )


    improvement_pct = (
        (
            f3_daylight_mae
            -
            f4_daylight_mae
        )
        /
        f3_daylight_mae
        * 100.0
    )


    f3_f4_daylight_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "n_daylight":
                int(
                    f3_row[
                        "n_test_daylight"
                    ]
                ),

            "F3_daylight_MAE_kW":
                f3_daylight_mae,

            "F4_daylight_MAE_kW":
                f4_daylight_mae,

            "F3_to_F4_daylight_MAE_improvement_pct":
                improvement_pct,
        }
    )


lstm_f3_f4_daylight_summary = pd.DataFrame(
    f3_f4_daylight_rows
)


assert (
    lstm_f3_f4_daylight_summary.shape
    ==
    (
        3,
        5,
    )
)


print(
    "\nF3 -> F4 DAYLIGHT-ONLY TEST SUMMARY"
)

print(
    "===================================="
)


display(
    lstm_f3_f4_daylight_summary
)


# ------------------------------------------------------------
# 7. Persist canonical TEST alignment tables
#
# These are important for later paired daily-block bootstrap
# reproducibility.
# ------------------------------------------------------------

for horizon_minutes in HORIZONS:

    alignment_path = (
        RESULTS_DIR
        /
        (
            f"lstm_final_{horizon_minutes}min_"
            "test_alignment.csv"
        )
    )


    lstm_test_alignment[
        horizon_minutes
    ].to_csv(
        alignment_path,
        index=False,
    )


    assert (
        alignment_path.exists()
    )


    assert (
        alignment_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 8. Persist daylight metric outputs
# ------------------------------------------------------------

lstm_final_daylight_metrics_path = (
    RESULTS_DIR
    /
    "lstm_final_daylight_metrics.csv"
)


lstm_f3_f4_daylight_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_daylight_summary.csv"
)


lstm_test_alignment_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_test_alignment_summary.csv"
)


lstm_final_daylight_metrics.to_csv(
    lstm_final_daylight_metrics_path,
    index=False,
)


lstm_f3_f4_daylight_summary.to_csv(
    lstm_f3_f4_daylight_summary_path,
    index=False,
)


lstm_test_alignment_summary.to_csv(
    lstm_test_alignment_summary_path,
    index=False,
)


for output_path in [
    lstm_final_daylight_metrics_path,
    lstm_f3_f4_daylight_summary_path,
    lstm_test_alignment_summary_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        > 0
    )


# ------------------------------------------------------------
# 9. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "DAYLIGHT-ONLY LSTM EVALUATION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "All 15 all-time metrics reproduced exactly."
)

print(
    "All 15 daylight-only metric sets computed."
)

print(
    "Canonical valid-time/daylight alignment persisted."
)

print(
    "No model was retrained or modified."
)

print(
    "Next stage: paired daily-block bootstrap for F3 vs F4."
)


CANONICAL HELD-OUT TEST ALIGNMENT SUMMARY


,horizon_minutes,n_test_total,n_test_daylight,daylight_fraction,first_valid_time,last_valid_time,utc_day_blocks
0,15,26496,17851,0.673724,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,92
1,30,26496,17851,0.673724,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,92
2,60,26496,17851,0.673724,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00,92



COMPLETE LSTM DAYLIGHT-ONLY TEST METRICS


,horizon_minutes,feature_family,n_test_total,n_test_daylight,all_MAE_kW,all_RMSE_kW,all_R2,daylight_MAE_kW,daylight_RMSE_kW,daylight_R2
0,15,F0,26496,17851,0.298274,0.620531,0.858677,0.429695,0.755401,0.808820
1,15,F1,26496,17851,0.294215,0.622662,0.857704,0.421801,0.758086,0.807459
2,15,F2,26496,17851,0.298937,0.622725,0.857676,0.421090,0.757713,0.807648
3,15,F3,26496,17851,0.326384,0.633885,0.852529,0.443346,0.768750,0.802004
4,15,F4,26496,17851,0.329271,0.636115,0.851489,0.437743,0.768179,0.802298
5,30,F0,26496,17851,0.459965,0.780223,0.776579,0.608879,0.941527,0.703003
6,30,F1,26496,17851,0.373356,0.741601,0.798151,0.539278,0.903102,0.726750
7,30,F2,26496,17851,0.357165,0.723139,0.808075,0.514593,0.880434,0.740295
8,30,F3,26496,17851,0.390874,0.741480,0.798217,0.543590,0.900879,0.728093
9,30,F4,26496,17851,0.368213,0.731963,0.803363,0.520896,0.890644,0.734237



F3 -> F4 DAYLIGHT-ONLY TEST SUMMARY


,horizon_minutes,n_daylight,F3_daylight_MAE_kW,F4_daylight_MAE_kW,F3_to_F4_daylight_MAE_improvement_pct
0,15,17851,0.443346,0.437743,1.263797
1,30,17851,0.543590,0.520896,4.174850
2,60,17851,0.689057,0.665287,3.449669



DAYLIGHT-ONLY LSTM EVALUATION: COMPLETE
All 15 all-time metrics reproduced exactly.
All 15 daylight-only metric sets computed.
Canonical valid-time/daylight alignment persisted.
No model was retrained or modified.
Next stage: paired daily-block bootstrap for F3 vs F4.


In [75]:
# ============================================================
# Step 60: Paired daily-block bootstrap for LSTM F3 vs F4
#
# SCIENTIFIC QUESTION
# -------------------
#
# Does adding the Panchang representation (F4) improve
# held-out prediction error relative to the continuous
# astronomical control model (F3)?
#
#
# We evaluate this separately for:
#
#       15 minutes
#       30 minutes
#       60 minutes
#
# and for two TEST populations:
#
#       1. all held-out observations
#       2. daylight-only observations
#
#
# BOOTSTRAP PROTOCOL
# ------------------
#
# Resampling unit:
#       UTC valid-time DAY
#
# Number of unique TEST days:
#       92
#
# Bootstrap repetitions:
#       10,000
#
# Random seed:
#       42
#
# Confidence interval:
#       95% percentile interval
#
#
# IMPORTANT:
#
# The bootstrap is PAIRED.
#
# For every resampled day, the same observations are used for
# both F3 and F4.
#
# We therefore bootstrap the difference:
#
#       MAE(F3) - MAE(F4)
#
#
# INTERPRETATION
# --------------
#
# Positive difference:
#
#       F4 has LOWER MAE than F3
#       -> Panchang increment is favorable
#
# Negative difference:
#
#       F4 has HIGHER MAE than F3
#       -> Panchang increment is unfavorable
#
#
# The MAE for each bootstrap replicate is calculated after
# pooling all observations from its resampled days.
#
# Therefore this is a ROW-WEIGHTED pooled MAE bootstrap,
# not an unweighted average of daily MAEs.
#
#
# NO MODEL RETRAINING OCCURS IN THIS CELL.
#
# The bootstrap quantifies variation in the already-frozen
# held-out prediction comparison across TEST days.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen bootstrap configuration
# ------------------------------------------------------------

BOOTSTRAP_REPETITIONS = 10_000

BOOTSTRAP_SEED = 42

BOOTSTRAP_CONFIDENCE_LEVEL = 0.95


HORIZONS = [
    15,
    30,
    60,
]


BOOTSTRAP_POPULATIONS = [
    "all",
    "daylight",
]


# ------------------------------------------------------------
# 2. Confirm required final predictions exist
# ------------------------------------------------------------

for horizon_minutes in HORIZONS:

    for feature_family in [
        "F3",
        "F4",
    ]:

        assert (
            (
                horizon_minutes,
                feature_family,
            )
            in lstm_final_test_predictions
        )


        assert (
            (
                horizon_minutes,
                feature_family,
            )
            in lstm_final_test_actuals
        )


# ------------------------------------------------------------
# 3. Confirm canonical TEST days are identical across horizons
#
# All horizons already showed:
#
#       2019-06-01 -> 2019-08-31
#
# with exactly 92 UTC valid-time day blocks.
# ------------------------------------------------------------

canonical_test_days = pd.DatetimeIndex(
    lstm_test_alignment[
        15
    ][
        "utc_valid_day"
    ]
    .drop_duplicates()
    .sort_values()
)


assert (
    len(
        canonical_test_days
    )
    == 92
)


for horizon_minutes in HORIZONS:

    horizon_days = pd.DatetimeIndex(
        lstm_test_alignment[
            horizon_minutes
        ][
            "utc_valid_day"
        ]
        .drop_duplicates()
        .sort_values()
    )


    assert (
        horizon_days.equals(
            canonical_test_days
        )
    )


print(
    "Canonical TEST day blocks:",
    len(
        canonical_test_days
    ),
)


print(
    "First TEST day:",
    canonical_test_days[
        0
    ],
)


print(
    "Last TEST day:",
    canonical_test_days[
        -1
    ],
)


# ------------------------------------------------------------
# 4. Generate ONE deterministic day-resampling matrix
#
# Shape:
#
#       10,000 bootstrap repetitions
#       ×
#       92 sampled day positions
#
# Each row samples 92 days WITH replacement.
#
# The SAME resampled-day matrix is used for:
#
#       all horizons
#       all-time population
#       daylight population
#
# This gives common bootstrap resamples across comparisons.
# ------------------------------------------------------------

bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_day_indices = (
    bootstrap_rng.integers(
        low=0,
        high=len(
            canonical_test_days
        ),
        size=(
            BOOTSTRAP_REPETITIONS,
            len(
                canonical_test_days
            ),
        ),
    )
)


assert (
    bootstrap_day_indices.shape
    ==
    (
        10_000,
        92,
    )
)


# ------------------------------------------------------------
# 5. Containers for:
#
#       daily sufficient statistics
#       bootstrap summaries
#       bootstrap replicate distributions
# ------------------------------------------------------------

lstm_f3_f4_daily_error_tables = {}

bootstrap_summary_rows = []

bootstrap_distribution_frames = []


# ------------------------------------------------------------
# 6. Process each horizon
# ------------------------------------------------------------

for horizon_minutes in HORIZONS:

    alignment = (
        lstm_test_alignment[
            horizon_minutes
        ]
        .copy()
    )


    assert (
        len(
            alignment
        )
        ==
        26_496
    )


    actual_kw = (
        alignment[
            "actual_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )


    f3_prediction_kw = np.asarray(
        lstm_final_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ],
        dtype=float,
    )


    f4_prediction_kw = np.asarray(
        lstm_final_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ],
        dtype=float,
    )


    assert (
        actual_kw.shape
        ==
        f3_prediction_kw.shape
        ==
        f4_prediction_kw.shape
        ==
        (
            26_496,
        )
    )


    # --------------------------------------------------------
    # Paired absolute error vectors
    # --------------------------------------------------------

    f3_absolute_error = np.abs(
        actual_kw
        -
        f3_prediction_kw
    )


    f4_absolute_error = np.abs(
        actual_kw
        -
        f4_prediction_kw
    )


    # Positive row-wise difference means F4 made the smaller
    # absolute error for that observation.
    absolute_error_difference = (
        f3_absolute_error
        -
        f4_absolute_error
    )


    # --------------------------------------------------------
    # Evaluate ALL and DAYLIGHT populations independently
    # --------------------------------------------------------

    for population_name in BOOTSTRAP_POPULATIONS:

        if (
            population_name
            ==
            "all"
        ):

            population_mask = np.ones(
                len(
                    alignment
                ),
                dtype=bool,
            )


        elif (
            population_name
            ==
            "daylight"
        ):

            population_mask = (
                alignment[
                    "is_daylight"
                ]
                .to_numpy(
                    dtype=bool
                )
            )


        else:

            raise ValueError(
                population_name
            )


        # ----------------------------------------------------
        # Create row-level paired error table
        # ----------------------------------------------------

        population_error_table = pd.DataFrame(
            {
                "utc_valid_day":
                    alignment.loc[
                        population_mask,
                        "utc_valid_day",
                    ]
                    .to_numpy(),

                "f3_absolute_error_kW":
                    f3_absolute_error[
                        population_mask
                    ],

                "f4_absolute_error_kW":
                    f4_absolute_error[
                        population_mask
                    ],

                "f3_minus_f4_absolute_error_kW":
                    absolute_error_difference[
                        population_mask
                    ],
            }
        )


        # ----------------------------------------------------
        # Aggregate sufficient statistics by UTC day
        #
        # Keeping sums + counts allows each bootstrap replicate
        # to reconstruct POOLED row-weighted MAE exactly.
        # ----------------------------------------------------

        daily_errors = (
            population_error_table
            .groupby(
                "utc_valid_day",
                sort=True,
            )
            .agg(
                n_rows=(
                    "f3_absolute_error_kW",
                    "size",
                ),

                f3_absolute_error_sum_kW=(
                    "f3_absolute_error_kW",
                    "sum",
                ),

                f4_absolute_error_sum_kW=(
                    "f4_absolute_error_kW",
                    "sum",
                ),

                f3_minus_f4_absolute_error_sum_kW=(
                    "f3_minus_f4_absolute_error_kW",
                    "sum",
                ),
            )
            .reindex(
                canonical_test_days
            )
        )


        daily_errors.index.name = (
            "utc_valid_day"
        )


        # ----------------------------------------------------
        # All 92 days must contribute observations.
        #
        # For daylight this confirms there are daylight
        # observations on every TEST day.
        # ----------------------------------------------------

        assert (
            len(
                daily_errors
            )
            ==
            92
        )


        assert (
            daily_errors[
                "n_rows"
            ]
            .notna()
            .all()
        )


        assert (
            (
                daily_errors[
                    "n_rows"
                ]
                >
                0
            )
            .all()
        )


        # ----------------------------------------------------
        # Save daily table in memory
        # ----------------------------------------------------

        lstm_f3_f4_daily_error_tables[
            (
                horizon_minutes,
                population_name,
            )
        ] = (
            daily_errors.copy()
        )


        # ----------------------------------------------------
        # Extract daily arrays
        # ----------------------------------------------------

        daily_n_rows = (
            daily_errors[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        daily_f3_error_sum = (
            daily_errors[
                "f3_absolute_error_sum_kW"
            ]
            .to_numpy(
                dtype=float
            )
        )


        daily_f4_error_sum = (
            daily_errors[
                "f4_absolute_error_sum_kW"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # ----------------------------------------------------
        # 7. Compute observed pooled point estimates
        # ----------------------------------------------------

        observed_n = int(
            daily_n_rows.sum()
        )


        observed_f3_mae = (
            daily_f3_error_sum.sum()
            /
            observed_n
        )


        observed_f4_mae = (
            daily_f4_error_sum.sum()
            /
            observed_n
        )


        observed_delta_mae = (
            observed_f3_mae
            -
            observed_f4_mae
        )


        observed_relative_improvement_pct = (
            observed_delta_mae
            /
            observed_f3_mae
            * 100.0
        )


        # ----------------------------------------------------
        # Cross-check observed point estimates against the
        # results already calculated earlier.
        # ----------------------------------------------------

        if (
            population_name
            ==
            "all"
        ):

            stored_f3_mae = (
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F3",
                    )
                ][
                    "test_MAE_kW"
                ]
            )


            stored_f4_mae = (
                lstm_final_test_results[
                    (
                        horizon_minutes,
                        "F4",
                    )
                ][
                    "test_MAE_kW"
                ]
            )


        else:

            daylight_horizon_rows = (
                lstm_final_daylight_metrics[
                    lstm_final_daylight_metrics[
                        "horizon_minutes"
                    ]
                    .eq(
                        horizon_minutes
                    )
                ]
            )


            stored_f3_mae = float(
                daylight_horizon_rows.loc[
                    daylight_horizon_rows[
                        "feature_family"
                    ]
                    .eq(
                        "F3"
                    ),
                    "daylight_MAE_kW",
                ]
                .iloc[
                    0
                ]
            )


            stored_f4_mae = float(
                daylight_horizon_rows.loc[
                    daylight_horizon_rows[
                        "feature_family"
                    ]
                    .eq(
                        "F4"
                    ),
                    "daylight_MAE_kW",
                ]
                .iloc[
                    0
                ]
            )


        np.testing.assert_allclose(
            observed_f3_mae,
            stored_f3_mae,
            rtol=0.0,
            atol=1e-10,
        )


        np.testing.assert_allclose(
            observed_f4_mae,
            stored_f4_mae,
            rtol=0.0,
            atol=1e-10,
        )


        # ----------------------------------------------------
        # 8. Bootstrap pooled row-weighted MAEs
        #
        # For each bootstrap replicate:
        #
        #   - select 92 UTC days with replacement
        #   - sum the errors over the selected day blocks
        #   - sum their row counts
        #   - divide total error by total rows
        #
        # This exactly represents concatenating the sampled
        # day blocks and computing MAE over the pooled rows.
        # ----------------------------------------------------

        bootstrap_row_counts = (
            daily_n_rows[
                bootstrap_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_error_sums = (
            daily_f3_error_sum[
                bootstrap_day_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f4_error_sums = (
            daily_f4_error_sum[
                bootstrap_day_indices
            ]
            .sum(
                axis=1
            )
        )


        assert (
            bootstrap_row_counts.shape
            ==
            (
                BOOTSTRAP_REPETITIONS,
            )
        )


        assert (
            (
                bootstrap_row_counts
                >
                0
            )
            .all()
        )


        bootstrap_f3_mae = (
            bootstrap_f3_error_sums
            /
            bootstrap_row_counts
        )


        bootstrap_f4_mae = (
            bootstrap_f4_error_sums
            /
            bootstrap_row_counts
        )


        bootstrap_delta_mae = (
            bootstrap_f3_mae
            -
            bootstrap_f4_mae
        )


        bootstrap_relative_improvement_pct = (
            bootstrap_delta_mae
            /
            bootstrap_f3_mae
            * 100.0
        )


        assert np.isfinite(
            bootstrap_delta_mae
        ).all()


        assert np.isfinite(
            bootstrap_relative_improvement_pct
        ).all()


        # ----------------------------------------------------
        # 9. Percentile confidence intervals
        # ----------------------------------------------------

        alpha = (
            1.0
            -
            BOOTSTRAP_CONFIDENCE_LEVEL
        )


        lower_percentile = (
            100.0
            *
            alpha
            /
            2.0
        )


        upper_percentile = (
            100.0
            *
            (
                1.0
                -
                alpha
                /
                2.0
            )
        )


        delta_ci_lower, delta_ci_upper = (
            np.percentile(
                bootstrap_delta_mae,
                [
                    lower_percentile,
                    upper_percentile,
                ],
            )
        )


        (
            relative_ci_lower,
            relative_ci_upper,
        ) = (
            np.percentile(
                bootstrap_relative_improvement_pct,
                [
                    lower_percentile,
                    upper_percentile,
                ],
            )
        )


        ci_excludes_zero = bool(
            (
                delta_ci_lower
                >
                0.0
            )
            or
            (
                delta_ci_upper
                <
                0.0
            )
        )


        # ----------------------------------------------------
        # Fraction of bootstrap replicates where F4 has the
        # lower pooled MAE.
        #
        # This is descriptive bootstrap information.
        # It is NOT being labelled as a p-value.
        # ----------------------------------------------------

        bootstrap_fraction_f4_better = float(
            np.mean(
                bootstrap_delta_mae
                >
                0.0
            )
        )


        # ----------------------------------------------------
        # 10. Store summary
        # ----------------------------------------------------

        bootstrap_summary_rows.append(
            {
                "model":
                    "LSTM",

                "horizon_minutes":
                    horizon_minutes,

                "population":
                    population_name,

                "n_test_rows":
                    observed_n,

                "n_utc_day_blocks":
                    92,

                "bootstrap_repetitions":
                    BOOTSTRAP_REPETITIONS,

                "bootstrap_seed":
                    BOOTSTRAP_SEED,

                "F3_MAE_kW":
                    observed_f3_mae,

                "F4_MAE_kW":
                    observed_f4_mae,

                "F3_minus_F4_MAE_kW":
                    observed_delta_mae,

                "F3_to_F4_MAE_improvement_pct":
                    observed_relative_improvement_pct,

                "delta_MAE_95CI_lower_kW":
                    delta_ci_lower,

                "delta_MAE_95CI_upper_kW":
                    delta_ci_upper,

                "relative_improvement_95CI_lower_pct":
                    relative_ci_lower,

                "relative_improvement_95CI_upper_pct":
                    relative_ci_upper,

                "bootstrap_fraction_F4_lower_MAE":
                    bootstrap_fraction_f4_better,

                "delta_MAE_CI_excludes_zero":
                    ci_excludes_zero,
            }
        )


        # ----------------------------------------------------
        # 11. Preserve all 10,000 bootstrap replicates
        #
        # This makes the statistical result fully auditable.
        # ----------------------------------------------------

        bootstrap_distribution_frames.append(
            pd.DataFrame(
                {
                    "bootstrap_iteration":
                        np.arange(
                            1,
                            BOOTSTRAP_REPETITIONS
                            +
                            1,
                            dtype=int,
                        ),

                    "horizon_minutes":
                        horizon_minutes,

                    "population":
                        population_name,

                    "bootstrap_n_rows":
                        bootstrap_row_counts,

                    "F3_MAE_kW":
                        bootstrap_f3_mae,

                    "F4_MAE_kW":
                        bootstrap_f4_mae,

                    "F3_minus_F4_MAE_kW":
                        bootstrap_delta_mae,

                    "F3_to_F4_MAE_improvement_pct":
                        bootstrap_relative_improvement_pct,
                }
            )
        )


# ------------------------------------------------------------
# 12. Assemble bootstrap summary
# ------------------------------------------------------------

lstm_f3_f4_bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)


assert (
    lstm_f3_f4_bootstrap_summary.shape
    ==
    (
        6,
        17,
    )
)


print(
    "\nPAIRED DAILY-BLOCK BOOTSTRAP SUMMARY"
)

print(
    "===================================="
)


display(
    lstm_f3_f4_bootstrap_summary
)


# ------------------------------------------------------------
# 13. Compact scientific summary
# ------------------------------------------------------------

lstm_f3_f4_bootstrap_compact = (
    lstm_f3_f4_bootstrap_summary[
        [
            "horizon_minutes",
            "population",
            "F3_MAE_kW",
            "F4_MAE_kW",
            "F3_to_F4_MAE_improvement_pct",
            "delta_MAE_95CI_lower_kW",
            "delta_MAE_95CI_upper_kW",
            "relative_improvement_95CI_lower_pct",
            "relative_improvement_95CI_upper_pct",
            "delta_MAE_CI_excludes_zero",
        ]
    ]
    .copy()
)


print(
    "\nCOMPACT F3 -> F4 BOOTSTRAP RESULT"
)

print(
    "================================="
)


display(
    lstm_f3_f4_bootstrap_compact
)


# ------------------------------------------------------------
# 14. Assemble all replicate distributions
# ------------------------------------------------------------

lstm_f3_f4_bootstrap_distributions = (
    pd.concat(
        bootstrap_distribution_frames,
        ignore_index=True,
    )
)


assert (
    lstm_f3_f4_bootstrap_distributions.shape[
        0
    ]
    ==
    (
        3
        *
        2
        *
        10_000
    )
)


assert (
    lstm_f3_f4_bootstrap_distributions.shape[
        0
    ]
    ==
    60_000
)


# ------------------------------------------------------------
# 15. Assemble daily sufficient-statistic table
#
# Useful for independent reproduction of the bootstrap.
# ------------------------------------------------------------

daily_table_frames = []


for (
    horizon_minutes,
    population_name,
), daily_table in (
    lstm_f3_f4_daily_error_tables.items()
):

    daily_output = (
        daily_table
        .reset_index()
        .copy()
    )


    daily_output.insert(
        0,
        "population",
        population_name,
    )


    daily_output.insert(
        0,
        "horizon_minutes",
        horizon_minutes,
    )


    daily_table_frames.append(
        daily_output
    )


lstm_f3_f4_daily_error_summary = (
    pd.concat(
        daily_table_frames,
        ignore_index=True,
    )
)


assert (
    lstm_f3_f4_daily_error_summary.shape[
        0
    ]
    ==
    (
        3
        *
        2
        *
        92
    )
)


assert (
    lstm_f3_f4_daily_error_summary.shape[
        0
    ]
    ==
    552
)


# ------------------------------------------------------------
# 16. Persist bootstrap outputs
# ------------------------------------------------------------

bootstrap_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_daily_block_bootstrap_summary.csv"
)


bootstrap_compact_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_daily_block_bootstrap_compact.csv"
)


bootstrap_distributions_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_daily_block_bootstrap_distributions.csv"
)


daily_error_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_f3_f4_daily_error_summary.csv"
)


lstm_f3_f4_bootstrap_summary.to_csv(
    bootstrap_summary_path,
    index=False,
)


lstm_f3_f4_bootstrap_compact.to_csv(
    bootstrap_compact_path,
    index=False,
)


lstm_f3_f4_bootstrap_distributions.to_csv(
    bootstrap_distributions_path,
    index=False,
)


lstm_f3_f4_daily_error_summary.to_csv(
    daily_error_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bootstrap_summary_path,
    bootstrap_compact_path,
    bootstrap_distributions_path,
    daily_error_summary_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Final protocol checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE"
)

print(
    "============================================================"
)


print(
    f"UTC day blocks: {len(canonical_test_days)}"
)

print(
    f"Bootstrap repetitions: {BOOTSTRAP_REPETITIONS:,}"
)

print(
    f"Seed: {BOOTSTRAP_SEED}"
)

print(
    "Confidence interval: 95% percentile"
)

print(
    "Statistic: pooled row-weighted MAE(F3) - MAE(F4)"
)

print(
    "Positive difference means F4 has lower MAE."
)

print(
    "No model was retrained."
)

Canonical TEST day blocks: 92
First TEST day: 2019-06-01 00:00:00+00:00
Last TEST day: 2019-08-31 00:00:00+00:00

PAIRED DAILY-BLOCK BOOTSTRAP SUMMARY


,model,horizon_minutes,population,n_test_rows,n_utc_day_blocks,bootstrap_repetitions,bootstrap_seed,F3_MAE_kW,F4_MAE_kW,F3_minus_F4_MAE_kW,F3_to_F4_MAE_improvement_pct,delta_MAE_95CI_lower_kW,delta_MAE_95CI_upper_kW,relative_improvement_95CI_lower_pct,relative_improvement_95CI_upper_pct,bootstrap_fraction_F4_lower_MAE,delta_MAE_CI_excludes_zero
0,LSTM,15,all,26496,92,10000,42,0.326384,0.329271,-0.002887,-0.884545,-0.008553,0.002621,-2.591356,0.800759,0.1567,False
1,LSTM,15,daylight,17851,92,10000,42,0.443346,0.437743,0.005603,1.263797,0.000183,0.011044,0.040153,2.557423,0.9786,True
2,LSTM,30,all,26496,92,10000,42,0.390874,0.368213,0.022661,5.797532,0.013652,0.031041,3.552636,7.939762,1.0000,True
3,LSTM,30,daylight,17851,92,10000,42,0.543590,0.520896,0.022694,4.174850,0.010634,0.033911,1.978721,6.174997,0.9997,True
4,LSTM,60,all,26496,92,10000,42,0.482546,0.489504,-0.006958,-1.441883,-0.016544,0.002763,-3.451026,0.566087,0.0766,False
5,LSTM,60,daylight,17851,92,10000,42,0.689057,0.665287,0.023770,3.449669,0.011491,0.036503,1.664418,5.347754,0.9999,True



COMPACT F3 -> F4 BOOTSTRAP RESULT


,horizon_minutes,population,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct,delta_MAE_95CI_lower_kW,delta_MAE_95CI_upper_kW,relative_improvement_95CI_lower_pct,relative_improvement_95CI_upper_pct,delta_MAE_CI_excludes_zero
0,15,all,0.326384,0.329271,-0.884545,-0.008553,0.002621,-2.591356,0.800759,False
1,15,daylight,0.443346,0.437743,1.263797,0.000183,0.011044,0.040153,2.557423,True
2,30,all,0.390874,0.368213,5.797532,0.013652,0.031041,3.552636,7.939762,True
3,30,daylight,0.543590,0.520896,4.174850,0.010634,0.033911,1.978721,6.174997,True
4,60,all,0.482546,0.489504,-1.441883,-0.016544,0.002763,-3.451026,0.566087,False
5,60,daylight,0.689057,0.665287,3.449669,0.011491,0.036503,1.664418,5.347754,True



PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE
UTC day blocks: 92
Bootstrap repetitions: 10,000
Seed: 42
Confidence interval: 95% percentile
Statistic: pooled row-weighted MAE(F3) - MAE(F4)
Positive difference means F4 has lower MAE.
No model was retrained.


In [76]:
# ============================================================
# Step 61: Final LSTM experiment consolidation
#
# PURPOSE
# -------
#
# The complete LSTM experiment is now finished:
#
#       - architecture selection
#       - validation-based epoch selection
#       - frozen train+validation refit
#       - held-out TEST evaluation
#       - all-time metrics
#       - daylight-only metrics
#       - paired daily-block bootstrap
#
# This cell creates ONE compact final scientific summary for
# the central F3 -> F4 comparison.
#
# NO training or additional statistical testing occurs here.
# ============================================================


# ------------------------------------------------------------
# 1. Extract all-time bootstrap results
# ------------------------------------------------------------

lstm_bootstrap_all = (
    lstm_f3_f4_bootstrap_summary[
        lstm_f3_f4_bootstrap_summary[
            "population"
        ]
        .eq(
            "all"
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Extract daylight bootstrap results
# ------------------------------------------------------------

lstm_bootstrap_daylight = (
    lstm_f3_f4_bootstrap_summary[
        lstm_f3_f4_bootstrap_summary[
            "population"
        ]
        .eq(
            "daylight"
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Index both tables by horizon for exact alignment
# ------------------------------------------------------------

lstm_bootstrap_all = (
    lstm_bootstrap_all
    .set_index(
        "horizon_minutes"
    )
    .sort_index()
)


lstm_bootstrap_daylight = (
    lstm_bootstrap_daylight
    .set_index(
        "horizon_minutes"
    )
    .sort_index()
)


assert (
    lstm_bootstrap_all.index.tolist()
    ==
    [
        15,
        30,
        60,
    ]
)


assert (
    lstm_bootstrap_daylight.index.tolist()
    ==
    [
        15,
        30,
        60,
    ]
)


# ------------------------------------------------------------
# 4. Build compact final LSTM F3 -> F4 scientific table
# ------------------------------------------------------------

lstm_final_scientific_summary = pd.DataFrame(
    {
        "horizon_minutes":
            [
                15,
                30,
                60,
            ],

        # ----------------------------------------------------
        # ALL-TIME comparison
        # ----------------------------------------------------

        "all_F3_MAE_kW":
            lstm_bootstrap_all[
                "F3_MAE_kW"
            ]
            .to_numpy(),

        "all_F4_MAE_kW":
            lstm_bootstrap_all[
                "F4_MAE_kW"
            ]
            .to_numpy(),

        "all_F3_to_F4_improvement_pct":
            lstm_bootstrap_all[
                "F3_to_F4_MAE_improvement_pct"
            ]
            .to_numpy(),

        "all_relative_95CI_lower_pct":
            lstm_bootstrap_all[
                "relative_improvement_95CI_lower_pct"
            ]
            .to_numpy(),

        "all_relative_95CI_upper_pct":
            lstm_bootstrap_all[
                "relative_improvement_95CI_upper_pct"
            ]
            .to_numpy(),

        "all_delta_CI_excludes_zero":
            lstm_bootstrap_all[
                "delta_MAE_CI_excludes_zero"
            ]
            .to_numpy(
                dtype=bool
            ),

        # ----------------------------------------------------
        # DAYLIGHT comparison
        # ----------------------------------------------------

        "daylight_F3_MAE_kW":
            lstm_bootstrap_daylight[
                "F3_MAE_kW"
            ]
            .to_numpy(),

        "daylight_F4_MAE_kW":
            lstm_bootstrap_daylight[
                "F4_MAE_kW"
            ]
            .to_numpy(),

        "daylight_F3_to_F4_improvement_pct":
            lstm_bootstrap_daylight[
                "F3_to_F4_MAE_improvement_pct"
            ]
            .to_numpy(),

        "daylight_relative_95CI_lower_pct":
            lstm_bootstrap_daylight[
                "relative_improvement_95CI_lower_pct"
            ]
            .to_numpy(),

        "daylight_relative_95CI_upper_pct":
            lstm_bootstrap_daylight[
                "relative_improvement_95CI_upper_pct"
            ]
            .to_numpy(),

        "daylight_delta_CI_excludes_zero":
            lstm_bootstrap_daylight[
                "delta_MAE_CI_excludes_zero"
            ]
            .to_numpy(
                dtype=bool
            ),
    }
)


assert (
    lstm_final_scientific_summary.shape
    ==
    (
        3,
        13,
    )
)


# ------------------------------------------------------------
# 5. Add concise evidence labels
#
# These labels are descriptive summaries of whether the
# bootstrap percentile interval excludes zero.
#
# They are NOT additional hypothesis tests.
# ------------------------------------------------------------

def summarize_bootstrap_direction(
    improvement_pct,
    ci_lower_pct,
    ci_upper_pct,
):

    if (
        ci_lower_pct
        >
        0.0
    ):

        return (
            "F4 lower MAE; CI above zero"
        )


    if (
        ci_upper_pct
        <
        0.0
    ):

        return (
            "F4 higher MAE; CI below zero"
        )


    if (
        improvement_pct
        >
        0.0
    ):

        return (
            "F4 lower MAE point estimate; CI crosses zero"
        )


    return (
        "F4 higher MAE point estimate; CI crosses zero"
    )


lstm_final_scientific_summary[
    "all_bootstrap_interpretation"
] = [
    summarize_bootstrap_direction(
        row[
            "all_F3_to_F4_improvement_pct"
        ],
        row[
            "all_relative_95CI_lower_pct"
        ],
        row[
            "all_relative_95CI_upper_pct"
        ],
    )

    for _, row in (
        lstm_final_scientific_summary
        .iterrows()
    )
]


lstm_final_scientific_summary[
    "daylight_bootstrap_interpretation"
] = [
    summarize_bootstrap_direction(
        row[
            "daylight_F3_to_F4_improvement_pct"
        ],
        row[
            "daylight_relative_95CI_lower_pct"
        ],
        row[
            "daylight_relative_95CI_upper_pct"
        ],
    )

    for _, row in (
        lstm_final_scientific_summary
        .iterrows()
    )
]


assert (
    lstm_final_scientific_summary.shape
    ==
    (
        3,
        15,
    )
)


# ------------------------------------------------------------
# 6. Display final scientific summary
# ------------------------------------------------------------

print(
    "\nFINAL LSTM F3 -> F4 SCIENTIFIC SUMMARY"
)

print(
    "======================================="
)


display(
    lstm_final_scientific_summary
)


# ------------------------------------------------------------
# 7. Display concise interpretation table
# ------------------------------------------------------------

lstm_final_interpretation_table = (
    lstm_final_scientific_summary[
        [
            "horizon_minutes",

            "all_F3_to_F4_improvement_pct",
            "all_relative_95CI_lower_pct",
            "all_relative_95CI_upper_pct",
            "all_bootstrap_interpretation",

            "daylight_F3_to_F4_improvement_pct",
            "daylight_relative_95CI_lower_pct",
            "daylight_relative_95CI_upper_pct",
            "daylight_bootstrap_interpretation",
        ]
    ]
    .copy()
)


print(
    "\nCOMPACT LSTM INTERPRETATION TABLE"
)

print(
    "================================="
)


display(
    lstm_final_interpretation_table
)


# ------------------------------------------------------------
# 8. Persist final LSTM scientific summary
# ------------------------------------------------------------

lstm_final_scientific_summary_path = (
    RESULTS_DIR
    /
    "lstm_final_scientific_summary.csv"
)


lstm_final_interpretation_table_path = (
    RESULTS_DIR
    /
    "lstm_final_interpretation_table.csv"
)


lstm_final_scientific_summary.to_csv(
    lstm_final_scientific_summary_path,
    index=False,
)


lstm_final_interpretation_table.to_csv(
    lstm_final_interpretation_table_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Verify artifacts
# ------------------------------------------------------------

for output_path in [
    lstm_final_scientific_summary_path,
    lstm_final_interpretation_table_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 10. Final LSTM checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "LSTM EXPERIMENT: FULLY CONSOLIDATED"
)

print(
    "============================================================"
)


print(
    "Architecture selection: COMPLETE"
)

print(
    "Epoch selection: COMPLETE"
)

print(
    "Final held-out evaluations: 15 / 15 COMPLETE"
)

print(
    "Daylight-only evaluation: COMPLETE"
)

print(
    "Paired daily-block bootstrap: COMPLETE"
)

print(
    "Final scientific summary: PERSISTED"
)

print(
    "\nNo further LSTM model fitting is required."
)


FINAL LSTM F3 -> F4 SCIENTIFIC SUMMARY


,horizon_minutes,all_F3_MAE_kW,all_F4_MAE_kW,all_F3_to_F4_improvement_pct,all_relative_95CI_lower_pct,all_relative_95CI_upper_pct,all_delta_CI_excludes_zero,daylight_F3_MAE_kW,daylight_F4_MAE_kW,daylight_F3_to_F4_improvement_pct,daylight_relative_95CI_lower_pct,daylight_relative_95CI_upper_pct,daylight_delta_CI_excludes_zero,all_bootstrap_interpretation,daylight_bootstrap_interpretation
0,15,0.326384,0.329271,-0.884545,-2.591356,0.800759,False,0.443346,0.437743,1.263797,0.040153,2.557423,True,F4 higher MAE point estimate; CI crosses zero,F4 lower MAE; CI above zero
1,30,0.390874,0.368213,5.797532,3.552636,7.939762,True,0.543590,0.520896,4.174850,1.978721,6.174997,True,F4 lower MAE; CI above zero,F4 lower MAE; CI above zero
2,60,0.482546,0.489504,-1.441883,-3.451026,0.566087,False,0.689057,0.665287,3.449669,1.664418,5.347754,True,F4 higher MAE point estimate; CI crosses zero,F4 lower MAE; CI above zero



COMPACT LSTM INTERPRETATION TABLE


,horizon_minutes,all_F3_to_F4_improvement_pct,all_relative_95CI_lower_pct,all_relative_95CI_upper_pct,all_bootstrap_interpretation,daylight_F3_to_F4_improvement_pct,daylight_relative_95CI_lower_pct,daylight_relative_95CI_upper_pct,daylight_bootstrap_interpretation
0,15,-0.884545,-2.591356,0.800759,F4 higher MAE point estimate; CI crosses zero,1.263797,0.040153,2.557423,F4 lower MAE; CI above zero
1,30,5.797532,3.552636,7.939762,F4 lower MAE; CI above zero,4.174850,1.978721,6.174997,F4 lower MAE; CI above zero
2,60,-1.441883,-3.451026,0.566087,F4 higher MAE point estimate; CI crosses zero,3.449669,1.664418,5.347754,F4 lower MAE; CI above zero



LSTM EXPERIMENT: FULLY CONSOLIDATED
Architecture selection: COMPLETE
Epoch selection: COMPLETE
Final held-out evaluations: 15 / 15 COMPLETE
Daylight-only evaluation: COMPLETE
Paired daily-block bootstrap: COMPLETE
Final scientific summary: PERSISTED

No further LSTM model fitting is required.


In [78]:
# ============================================================
# Step 62: GRU architecture-selection preflight
#
# PURPOSE
# -------
#
# Begin the second deep-learning model family:
#
#       GRU
#
# Architecture selection follows the SAME frozen protocol used
# for LSTM:
#
#       horizon = 60 minutes
#       feature family = F0
#       selection metric = validation MAE
#
# Candidate architecture family:
#
#       RNN_C1
#       RNN_C2
#       RNN_C3
#       RNN_C4
#
# No TEST data is accessed in this cell.
# No model is trained in this cell.
#
# We first inspect the exact helper APIs and locate the existing
# recurrent candidate configuration object already created
# earlier in this notebook.
# ============================================================


import inspect


# ------------------------------------------------------------
# 1. Confirm LSTM section is complete
# ------------------------------------------------------------

assert (
    len(
        lstm_final_test_results
    )
    == 15
)


assert (
    lstm_final_scientific_summary.shape
    ==
    (
        3,
        15,
    )
)


print(
    "LSTM section confirmed complete."
)


# ------------------------------------------------------------
# 2. Reconfirm GRU architecture-selection protocol
# ------------------------------------------------------------

GRU_ARCHITECTURE_SELECTION_HORIZON = 60

GRU_ARCHITECTURE_SELECTION_FAMILY = "F0"


print(
    "\nGRU architecture-selection protocol:"
)

print(
    "  model: GRU"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F0"
)

print(
    "  ranking: validation MAE -> validation RMSE -> fewer params"
)

print(
    "  held-out TEST: NOT USED"
)


# ------------------------------------------------------------
# 3. Inspect reusable helper signatures
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "HELPER SIGNATURES"
)

print(
    "============================================================"
)


print(
    "\nbuild_deep_forecasting_model:"
)

print(
    inspect.signature(
        build_deep_forecasting_model
    )
)


print(
    "\nrun_validation_candidate:"
)

print(
    inspect.signature(
        run_validation_candidate
    )
)


print(
    "\nget_deep_split_data:"
)

print(
    inspect.signature(
        get_deep_split_data
    )
)


# ------------------------------------------------------------
# 4. Verify 60-minute F0 TRAIN / VALIDATION populations
#
# Architecture selection must use only these two populations.
# ------------------------------------------------------------

gru_arch_train_data = (
    get_deep_split_data(
        60,
        "F0",
        "train",
    )
)


gru_arch_validation_data = (
    get_deep_split_data(
        60,
        "F0",
        "validation",
    )
)


assert (
    gru_arch_train_data[
        "n_samples"
    ]
    == 87_252
)


assert (
    gru_arch_validation_data[
        "n_samples"
    ]
    == 17_568
)


assert (
    gru_arch_train_data[
        "sequence"
    ].shape
    ==
    (
        87_252,
        13,
        9,
    )
)


assert (
    gru_arch_validation_data[
        "sequence"
    ].shape
    ==
    (
        17_568,
        13,
        9,
    )
)


assert (
    gru_arch_train_data[
        "auxiliary"
    ].shape
    ==
    (
        87_252,
        1,
    )
)


assert (
    gru_arch_validation_data[
        "auxiliary"
    ].shape
    ==
    (
        17_568,
        1,
    )
)


print(
    "\n60-min F0 architecture-selection populations:"
)

print(
    "  TRAIN:",
    gru_arch_train_data[
        "n_samples"
    ],
)

print(
    "  VALIDATION:",
    gru_arch_validation_data[
        "n_samples"
    ],
)

print(
    "  sequence shape:",
    gru_arch_train_data[
        "sequence"
    ].shape[
        1:
    ],
)

print(
    "  auxiliary width:",
    gru_arch_train_data[
        "auxiliary"
    ].shape[
        1
    ],
)


# ------------------------------------------------------------
# 5. Locate the recurrent candidate definitions already present
#    in notebook globals.
#
# We do not redefine them yet. We first identify the canonical
# object that was used for the LSTM architecture search.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "RECURRENT CANDIDATE CONFIGURATION OBJECTS"
)

print(
    "============================================================"
)


recurrent_candidate_hits = []


for variable_name, variable_value in list(
    globals().items()
):

    name_lower = (
        variable_name.lower()
    )


    if not any(
        token in name_lower

        for token in [
            "candidate",
            "recurrent",
            "architecture",
        ]
    ):

        continue


    candidate_ids_found = []


    if isinstance(
        variable_value,
        dict,
    ):

        candidate_ids_found = [
            candidate_id

            for candidate_id in [
                "RNN_C1",
                "RNN_C2",
                "RNN_C3",
                "RNN_C4",
            ]

            if candidate_id
            in variable_value
        ]


    elif isinstance(
        variable_value,
        (
            list,
            tuple,
        ),
    ):

        for item in variable_value:

            if isinstance(
                item,
                dict,
            ):

                possible_id = (
                    item.get(
                        "candidate_id"
                    )
                )


                if possible_id in [
                    "RNN_C1",
                    "RNN_C2",
                    "RNN_C3",
                    "RNN_C4",
                ]:

                    candidate_ids_found.append(
                        possible_id
                    )


    if candidate_ids_found:

        recurrent_candidate_hits.append(
            variable_name
        )


        print(
            f"\nObject: {variable_name}"
        )

        print(
            "  type:",
            type(
                variable_value
            ),
        )

        print(
            "  candidate IDs found:",
            candidate_ids_found,
        )


        # Safe compact preview.
        if isinstance(
            variable_value,
            dict,
        ):

            for candidate_id in [
                "RNN_C1",
                "RNN_C2",
                "RNN_C3",
                "RNN_C4",
            ]:

                if (
                    candidate_id
                    in variable_value
                ):

                    print(
                        f"  {candidate_id}:",
                        variable_value[
                            candidate_id
                        ],
                    )


        elif isinstance(
            variable_value,
            (
                list,
                tuple,
            ),
        ):

            for item in variable_value:

                if isinstance(
                    item,
                    dict,
                ):

                    if (
                        item.get(
                            "candidate_id"
                        )
                        in
                        [
                            "RNN_C1",
                            "RNN_C2",
                            "RNN_C3",
                            "RNN_C4",
                        ]
                    ):

                        print(
                            " ",
                            item,
                        )


print(
    "\nCandidate-definition object hits:",
    recurrent_candidate_hits,
)


# ------------------------------------------------------------
# 6. Confirm the model factory accepts GRU
#
# Use the canonical RNN_C1 configuration already defined in
# RECURRENT_ARCHITECTURE_CANDIDATES.
#
# Build only an UNTRAINED smoke-test model.
#
# This does NOT train the model and does NOT access TEST data.
# ------------------------------------------------------------

gru_smoke_config = next(
    dict(candidate_config)

    for candidate_config
    in RECURRENT_ARCHITECTURE_CANDIDATES

    if (
        candidate_config[
            "candidate_id"
        ]
        ==
        "RNN_C1"
    )
)


assert (
    gru_smoke_config
    ==
    {
        "candidate_id":
            "RNN_C1",

        "units":
            32,

        "n_recurrent_layers":
            1,

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    }
)


gru_smoke_model = (
    build_deep_forecasting_model(
        model_name="GRU",
        auxiliary_dim=1,
        candidate_config=gru_smoke_config,
        seed=42,
    )
)


gru_smoke_parameter_count = (
    gru_smoke_model.count_params()
)


print(
    "\nGRU RNN_C1 untrained smoke-test parameter count:",
    gru_smoke_parameter_count,
)


assert (
    gru_smoke_parameter_count
    >
    0
)


# Clean up the untrained smoke-test model.
del gru_smoke_model

tf.keras.backend.clear_session()


# ------------------------------------------------------------
# 7. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU ARCHITECTURE-SELECTION PREFLIGHT: PASSED"
)

print(
    "============================================================"
)


print(
    "No model was trained."
)

print(
    "No held-out TEST data was accessed."
)

print(
    "Next step: run RNN_C1-RNN_C4 on 60-min F0 validation only."
)

LSTM section confirmed complete.

GRU architecture-selection protocol:
  model: GRU
  horizon: 60 minutes
  feature family: F0
  ranking: validation MAE -> validation RMSE -> fewer params
  held-out TEST: NOT USED

HELPER SIGNATURES

build_deep_forecasting_model:
(model_name: str, auxiliary_dim: int, candidate_config: dict, seed: int = 42)

run_validation_candidate:
(model_name: str, candidate_config: dict, horizon_minutes: int, feature_family: str, verbose: int = 0)

get_deep_split_data:
(horizon_minutes: int, feature_family: str, split_name: str)

60-min F0 architecture-selection populations:
  TRAIN: 87252
  VALIDATION: 17568
  sequence shape: (13, 9)
  auxiliary width: 1

RECURRENT CANDIDATE CONFIGURATION OBJECTS

Object: RECURRENT_ARCHITECTURE_CANDIDATES
  type: <class 'tuple'>
  candidate IDs found: ['RNN_C1', 'RNN_C2', 'RNN_C3', 'RNN_C4']
  {'candidate_id': 'RNN_C1', 'units': 32, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}
  {'candidate_id': 'RNN_C2', 'uni

In [79]:
# ============================================================
# Step 63: GRU architecture selection
#
# MODEL FAMILY
# ------------
#
#       GRU
#
#
# SELECTION POPULATION
# --------------------
#
# Horizon:
#       60 minutes
#
# Feature family:
#       F0
#
# TRAIN:
#       87,252 samples
#
# VALIDATION:
#       17,568 samples
#
#
# CANDIDATES
# ----------
#
#       RNN_C1
#       RNN_C2
#       RNN_C3
#       RNN_C4
#
#
# SELECTION RULE
# --------------
#
# Mechanical ranking:
#
#       1. lowest validation MAE
#       2. lowest validation RMSE
#       3. fewer trainable parameters
#
#
# IMPORTANT
# ---------
#
# Held-out TEST data is NOT used.
#
# Each candidate:
#
#       - starts from a fresh seeded model
#       - uses the frozen training protocol
#       - uses EarlyStopping on validation loss
#       - restores the best validation weights
#
# After this cell, ONE GRU architecture is frozen for all:
#
#       15 / 30 / 60 minute horizons
#       F0 / F1 / F2 / F3 / F4 families
# ============================================================


# ------------------------------------------------------------
# 1. Confirm canonical GRU candidate set
# ------------------------------------------------------------

assert (
    len(
        RECURRENT_ARCHITECTURE_CANDIDATES
    )
    == 4
)


gru_candidate_ids = [
    candidate_config[
        "candidate_id"
    ]

    for candidate_config
    in RECURRENT_ARCHITECTURE_CANDIDATES
]


assert (
    gru_candidate_ids
    ==
    [
        "RNN_C1",
        "RNN_C2",
        "RNN_C3",
        "RNN_C4",
    ]
)


print(
    "GRU architecture candidates:",
    gru_candidate_ids,
)


# ------------------------------------------------------------
# 2. Confirm TEST remains outside architecture selection
# ------------------------------------------------------------

gru_arch_train = (
    get_deep_split_data(
        60,
        "F0",
        "train",
    )
)


gru_arch_validation = (
    get_deep_split_data(
        60,
        "F0",
        "validation",
    )
)


assert (
    gru_arch_train[
        "n_samples"
    ]
    == 87_252
)


assert (
    gru_arch_validation[
        "n_samples"
    ]
    == 17_568
)


print(
    "\nArchitecture-selection population:"
)

print(
    "  TRAIN:",
    gru_arch_train[
        "n_samples"
    ],
)

print(
    "  VALIDATION:",
    gru_arch_validation[
        "n_samples"
    ],
)

print(
    "  TEST: NOT ACCESSED"
)


# ------------------------------------------------------------
# 3. Run the four GRU candidates
#
# We keep both:
#
#       result dictionary
#       training-history DataFrame
#
# for every candidate.
# ------------------------------------------------------------

gru_architecture_results = []

gru_architecture_selection_history = {}


for candidate_config in (
    RECURRENT_ARCHITECTURE_CANDIDATES
):

    candidate_id = (
        candidate_config[
            "candidate_id"
        ]
    )


    print(
        "\n============================================================"
    )

    print(
        f"GRU architecture candidate: {candidate_id}"
    )

    print(
        "============================================================"
    )


    candidate_output = (
        run_validation_candidate(
            model_name=
                "GRU",

            candidate_config=
                dict(
                    candidate_config
                ),

            horizon_minutes=
                60,

            feature_family=
                "F0",

            verbose=
                2,
        )
    )


    # --------------------------------------------------------
    # Canonical helper currently returns:
    #
    #       (result_dict, history_df)
    #
    # Keep a defensive check so notebook failure is immediate
    # if the helper API is accidentally changed later.
    # --------------------------------------------------------

    assert isinstance(
        candidate_output,
        tuple,
    )


    assert (
        len(
            candidate_output
        )
        ==
        2
    )


    (
        candidate_result,
        candidate_history,
    ) = candidate_output


    assert isinstance(
        candidate_result,
        dict,
    )


    assert isinstance(
        candidate_history,
        pd.DataFrame,
    )


    # --------------------------------------------------------
    # Basic identity checks
    # --------------------------------------------------------

    assert (
        candidate_result[
            "model"
        ]
        ==
        "GRU"
    )


    assert (
        candidate_result[
            "candidate_id"
        ]
        ==
        candidate_id
    )


    assert (
        candidate_result[
            "horizon_minutes"
        ]
        ==
        60
    )


    assert (
        candidate_result[
            "feature_family"
        ]
        ==
        "F0"
    )


    # --------------------------------------------------------
    # Store result and history
    # --------------------------------------------------------

    gru_architecture_results.append(
        candidate_result.copy()
    )


    gru_architecture_selection_history[
        candidate_id
    ] = (
        candidate_history.copy()
    )


    # --------------------------------------------------------
    # Compact candidate result
    # --------------------------------------------------------

    print(
        "\nCandidate result:"
    )


    display(
        pd.DataFrame(
            [
                candidate_result
            ]
        )
    )


# ------------------------------------------------------------
# 4. Verify all four candidates completed exactly once
# ------------------------------------------------------------

assert (
    len(
        gru_architecture_results
    )
    ==
    4
)


assert set(
    gru_architecture_selection_history.keys()
) == {
    "RNN_C1",
    "RNN_C2",
    "RNN_C3",
    "RNN_C4",
}


completed_candidate_ids = [
    result[
        "candidate_id"
    ]

    for result
    in gru_architecture_results
]


assert (
    completed_candidate_ids
    ==
    [
        "RNN_C1",
        "RNN_C2",
        "RNN_C3",
        "RNN_C4",
    ]
)


# ------------------------------------------------------------
# 5. Inspect result schema
#
# This also protects us from silently using the wrong column
# names in the mechanical ranking.
# ------------------------------------------------------------

gru_architecture_results_df = pd.DataFrame(
    gru_architecture_results
)


print(
    "\nGRU architecture-result columns:"
)

print(
    list(
        gru_architecture_results_df.columns
    )
)


# ------------------------------------------------------------
# 6. Confirm required ranking columns exist
# ------------------------------------------------------------

required_gru_ranking_columns = [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "trainable_parameter_count",
]


for column_name in (
    required_gru_ranking_columns
):

    assert (
        column_name
        in gru_architecture_results_df.columns
    )


# ------------------------------------------------------------
# 7. Mechanical architecture ranking
#
# Ranking rule frozen before looking at GRU results:
#
#       validation MAE
#           ->
#       validation RMSE
#           ->
#       fewer parameters
# ------------------------------------------------------------

gru_architecture_results_ranked = (
    gru_architecture_results_df
    .sort_values(
        by=[
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "trainable_parameter_count",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind=
            "mergesort",
    )
    .reset_index(
        drop=True
    )
)


gru_architecture_results_ranked.insert(
    0,
    "selection_rank",
    np.arange(
        1,
        len(
            gru_architecture_results_ranked
        )
        +
        1,
        dtype=int,
    ),
)


print(
    "\n============================================================"
)

print(
    "GRU ARCHITECTURE RANKING"
)

print(
    "============================================================"
)


display(
    gru_architecture_results_ranked
)


# ------------------------------------------------------------
# 8. Freeze selected GRU architecture
# ------------------------------------------------------------

gru_selected_architecture_result = (
    gru_architecture_results_ranked
    .iloc[
        0
    ]
    .to_dict()
)


gru_selected_candidate_id = (
    gru_selected_architecture_result[
        "candidate_id"
    ]
)


gru_selected_candidate_config = next(
    dict(
        candidate_config
    )

    for candidate_config
    in RECURRENT_ARCHITECTURE_CANDIDATES

    if (
        candidate_config[
            "candidate_id"
        ]
        ==
        gru_selected_candidate_id
    )
)


# ------------------------------------------------------------
# 9. Verify frozen configuration is one of the declared
#    candidates and contains all required fields
# ------------------------------------------------------------

assert (
    gru_selected_candidate_id
    in
    {
        "RNN_C1",
        "RNN_C2",
        "RNN_C3",
        "RNN_C4",
    }
)


for required_key in [
    "candidate_id",
    "units",
    "n_recurrent_layers",
    "dropout",
    "fusion_dense_units",
]:

    assert (
        required_key
        in gru_selected_candidate_config
    )


# ------------------------------------------------------------
# 10. Display selected architecture
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "FROZEN GRU ARCHITECTURE"
)

print(
    "============================================================"
)


print(
    "Selected candidate:",
    gru_selected_candidate_id,
)


print(
    "Configuration:"
)


print(
    gru_selected_candidate_config
)


print(
    "\nSelection metrics:"
)


print(
    "  Validation MAE:",
    f"{gru_selected_architecture_result['validation_MAE_kW']:.6f}",
    "kW",
)


print(
    "  Validation RMSE:",
    f"{gru_selected_architecture_result['validation_RMSE_kW']:.6f}",
    "kW",
)


print(
    "  Trainable parameters:",
    int(
        gru_selected_architecture_result[
            "trainable_parameter_count"
        ]
    ),
)


# ------------------------------------------------------------
# 11. Persist architecture-search results immediately
# ------------------------------------------------------------

gru_architecture_results_path = (
    RESULTS_DIR
    /
    "gru_architecture_selection_60min_F0.csv"
)


gru_architecture_results_ranked.to_csv(
    gru_architecture_results_path,
    index=False,
)


assert (
    gru_architecture_results_path.exists()
)


assert (
    gru_architecture_results_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 12. Persist each candidate training history
# ------------------------------------------------------------

for candidate_id, history_df in (
    gru_architecture_selection_history.items()
):

    history_path = (
        RESULTS_DIR
        /
        (
            "gru_architecture_selection_"
            f"{candidate_id}_history.csv"
        )
    )


    history_df.to_csv(
        history_path,
        index=False,
    )


    assert (
        history_path.exists()
    )


    assert (
        history_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 13. Persist selected architecture configuration
# ------------------------------------------------------------

gru_selected_architecture_path = (
    RESULTS_DIR
    /
    "gru_selected_architecture.json"
)


with open(
    gru_selected_architecture_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        {
            "model":
                "GRU",

            "selection_horizon_minutes":
                60,

            "selection_feature_family":
                "F0",

            "selection_rule":
                (
                    "validation_MAE_kW -> "
                    "validation_RMSE_kW -> "
                    "fewer_parameters"
                ),

            "selected_candidate_id":
                gru_selected_candidate_id,

            "selected_candidate_config":
                gru_selected_candidate_config,

            "selected_validation_MAE_kW":
                float(
                    gru_selected_architecture_result[
                        "validation_MAE_kW"
                    ]
                ),

            "selected_validation_RMSE_kW":
                float(
                    gru_selected_architecture_result[
                        "validation_RMSE_kW"
                    ]
                ),

            "selected_trainable_parameter_count":
                int(
                    gru_selected_architecture_result[
                        "trainable_parameter_count"
                    ]
                ),

            "test_used_for_selection":
                False,
        },
        file,
        indent=2,
    )


assert (
    gru_selected_architecture_path.exists()
)


assert (
    gru_selected_architecture_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 14. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU ARCHITECTURE SELECTION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Candidates evaluated: 4 / 4"
)

print(
    "Selection population: 60-min F0 TRAIN + VALIDATION only"
)

print(
    "Held-out TEST used: NO"
)

print(
    "Frozen candidate:",
    gru_selected_candidate_id,
)

print(
    "\nNext stage: GRU validation-based epoch selection "
    "for all 15 horizon × feature-family combinations."
)

GRU architecture candidates: ['RNN_C1', 'RNN_C2', 'RNN_C3', 'RNN_C4']

Architecture-selection population:
  TRAIN: 87252
  VALIDATION: 17568
  TEST: NOT ACCESSED

GRU architecture candidate: RNN_C1
Epoch 1/80


2026-09-21 20:07:02.416719: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 20:07:02.417359: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 7s - 19ms/step - loss: 0.2486 - val_loss: 0.3044
Epoch 2/80
341/341 - 4s - 12ms/step - loss: 0.2108 - val_loss: 0.3112
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.2153 - val_loss: 0.3057
Epoch 4/80
341/341 - 4s - 12ms/step - loss: 0.2203 - val_loss: 0.3187
Epoch 5/80
341/341 - 4s - 12ms/step - loss: 0.2212 - val_loss: 0.3208
Epoch 6/80
341/341 - 4s - 12ms/step - loss: 0.2352 - val_loss: 0.3512
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.2347 - val_loss: 0.3264
Epoch 8/80
341/341 - 4s - 10ms/step - loss: 0.2480 - val_loss: 0.3402
Epoch 9/80
341/341 - 4s - 10ms/step - loss: 0.2549 - val_loss: 0.3364


2026-09-21 20:07:39.781057: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 20:07:39.781304: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


Candidate result:


...


Candidate result:


,model,candidate_id,horizon_minutes,feature_family,n_train,n_validation,auxiliary_dim,trainable_parameter_count,epochs_ran,best_epoch,callback_best_val_loss_scaled,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds,config_units,config_n_recurrent_layers,config_dropout,config_fusion_dense_units
0,GRU,RNN_C4,60,F0,87252,17568,1,41505,17,9,0.30458,0.483646,0.874722,0.7964,100.626474,64,2,0.2,32



GRU architecture-result columns:
['model', 'candidate_id', 'horizon_minutes', 'feature_family', 'n_train', 'n_validation', 'auxiliary_dim', 'trainable_parameter_count', 'epochs_ran', 'best_epoch', 'callback_best_val_loss_scaled', 'validation_MAE_kW', 'validation_RMSE_kW', 'validation_R2', 'fit_seconds', 'config_units', 'config_n_recurrent_layers', 'config_dropout', 'config_fusion_dense_units']

GRU ARCHITECTURE RANKING


,selection_rank,model,candidate_id,horizon_minutes,feature_family,n_train,n_validation,auxiliary_dim,trainable_parameter_count,epochs_ran,best_epoch,callback_best_val_loss_scaled,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds,config_units,config_n_recurrent_layers,config_dropout,config_fusion_dense_units
0,1,GRU,RNN_C2,60,F0,87252,17568,1,16545,11,3,0.288566,0.458217,0.848638,0.808361,40.478717,64,1,0.0,32
1,2,GRU,RNN_C3,60,F0,87252,17568,1,16545,9,1,0.303947,0.482641,0.878507,0.794634,35.409694,64,1,0.2,32
2,3,GRU,RNN_C1,60,F0,87252,17568,1,5249,9,1,0.304359,0.483296,0.871889,0.797717,37.481876,32,1,0.0,32
3,4,GRU,RNN_C4,60,F0,87252,17568,1,41505,17,9,0.304580,0.483646,0.874722,0.796400,100.626474,64,2,0.2,32



FROZEN GRU ARCHITECTURE
Selected candidate: RNN_C2
Configuration:
{'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

Selection metrics:
  Validation MAE: 0.458217 kW
  Validation RMSE: 0.848638 kW
  Trainable parameters: 16545

GRU ARCHITECTURE SELECTION: COMPLETE
Candidates evaluated: 4 / 4
Selection population: 60-min F0 TRAIN + VALIDATION only
Held-out TEST used: NO
Frozen candidate: RNN_C2

Next stage: GRU validation-based epoch selection for all 15 horizon × feature-family combinations.


In [80]:
# ============================================================
# Step 64: GRU validation-based epoch selection
#
# PURPOSE
# -------
#
# The GRU architecture has now been selected and frozen:
#
#       candidate_id = RNN_C2
#       units = 64
#       recurrent layers = 1
#       dropout = 0.0
#       fusion dense units = 32
#
#
# We now determine the frozen training epoch count separately
# for every:
#
#       horizon × feature-family
#
# combination:
#
#       15 min: F0 F1 F2 F3 F4
#       30 min: F0 F1 F2 F3 F4
#       60 min: F0 F1 F2 F3 F4
#
#
# PROTOCOL
# --------
#
# Each experiment uses:
#
#       TRAIN only for fitting
#       VALIDATION only for early stopping / epoch selection
#
# Held-out TEST data is NOT accessed.
#
#
# IMPORTANT
# ---------
#
# The already-completed:
#
#       60-minute F0 / GRU RNN_C2
#
# architecture-selection run is reused here.
#
# It is NOT retrained.
#
# After this cell, the best epoch for every GRU experiment is
# frozen before any GRU final TEST evaluation begins.
# ============================================================


# ------------------------------------------------------------
# 1. Freeze selected GRU architecture configuration
# ------------------------------------------------------------

GRU_SELECTED_CANDIDATE_ID = (
    gru_selected_candidate_id
)


GRU_SELECTED_CANDIDATE_CONFIG = dict(
    gru_selected_candidate_config
)


assert (
    GRU_SELECTED_CANDIDATE_ID
    ==
    "RNN_C2"
)


assert (
    GRU_SELECTED_CANDIDATE_CONFIG
    ==
    {
        "candidate_id":
            "RNN_C2",

        "units":
            64,

        "n_recurrent_layers":
            1,

        "dropout":
            0.0,

        "fusion_dense_units":
            32,
    }
)


print(
    "Frozen GRU architecture:"
)

print(
    GRU_SELECTED_CANDIDATE_CONFIG
)


# ------------------------------------------------------------
# 2. Define canonical experiment order
# ------------------------------------------------------------

GRU_HORIZONS = [
    15,
    30,
    60,
]


GRU_FEATURE_FAMILIES = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]


gru_epoch_selection_experiments = [
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes in GRU_HORIZONS

    for feature_family in GRU_FEATURE_FAMILIES
]


assert (
    len(
        gru_epoch_selection_experiments
    )
    ==
    15
)


# ------------------------------------------------------------
# 3. Initialize epoch-selection containers
# ------------------------------------------------------------

gru_epoch_selection_results = {}

gru_epoch_selection_histories = {}


# ------------------------------------------------------------
# 4. Reuse the already-completed 60-min F0 RNN_C2 run
#
# This run was used for architecture selection and already
# provides the required validation-selected epoch.
# ------------------------------------------------------------

gru_reused_60_f0_result = dict(
    gru_selected_architecture_result
)


# Remove the ranking-only field if present.
gru_reused_60_f0_result.pop(
    "selection_rank",
    None,
)


assert (
    gru_reused_60_f0_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_reused_60_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_reused_60_f0_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_reused_60_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


assert (
    gru_reused_60_f0_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_reused_60_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_reused_60_f0_result[
        "best_epoch"
    ]
    ==
    3
)


gru_epoch_selection_results[
    (
        60,
        "F0",
    )
] = (
    gru_reused_60_f0_result.copy()
)


gru_epoch_selection_histories[
    (
        60,
        "F0",
    )
] = (
    gru_architecture_selection_history[
        "RNN_C2"
    ]
    .copy()
)


print(
    "\nReused existing architecture-selection run:"
)

print(
    "  60-min F0"
)

print(
    "  best epoch:",
    gru_epoch_selection_results[
        (
            60,
            "F0",
        )
    ][
        "best_epoch"
    ],
)

print(
    "  no retraining performed"
)


# ------------------------------------------------------------
# 5. Run remaining 14 validation experiments
# ------------------------------------------------------------

for (
    horizon_minutes,
    feature_family,
) in gru_epoch_selection_experiments:

    # --------------------------------------------------------
    # 60/F0 already exists from architecture selection.
    # --------------------------------------------------------

    if (
        horizon_minutes
        ==
        60
        and
        feature_family
        ==
        "F0"
    ):

        continue


    print(
        "\n============================================================"
    )

    print(
        "GRU epoch-selection experiment:"
    )

    print(
        f"  horizon = {horizon_minutes} minutes"
    )

    print(
        f"  feature family = {feature_family}"
    )

    print(
        "============================================================"
    )


    (
        experiment_result,
        experiment_history,
    ) = run_validation_candidate(

        model_name=
            "GRU",

        candidate_config=
            dict(
                GRU_SELECTED_CANDIDATE_CONFIG
            ),

        horizon_minutes=
            horizon_minutes,

        feature_family=
            feature_family,

        verbose=
            2,
    )


    # --------------------------------------------------------
    # Identity checks
    # --------------------------------------------------------

    assert (
        experiment_result[
            "model"
        ]
        ==
        "GRU"
    )


    assert (
        experiment_result[
            "candidate_id"
        ]
        ==
        "RNN_C2"
    )


    assert (
        experiment_result[
            "horizon_minutes"
        ]
        ==
        horizon_minutes
    )


    assert (
        experiment_result[
            "feature_family"
        ]
        ==
        feature_family
    )


    # --------------------------------------------------------
    # Validation-only epoch-selection checks
    # --------------------------------------------------------

    assert (
        experiment_result[
            "best_epoch"
        ]
        >=
        1
    )


    assert (
        experiment_result[
            "best_epoch"
        ]
        <=
        experiment_result[
            "epochs_ran"
        ]
    )


    assert (
        experiment_result[
            "epochs_ran"
        ]
        <=
        DEEP_MAX_EPOCHS
    )


    assert np.isfinite(
        experiment_result[
            "validation_MAE_kW"
        ]
    )


    assert np.isfinite(
        experiment_result[
            "validation_RMSE_kW"
        ]
    )


    assert np.isfinite(
        experiment_result[
            "validation_R2"
        ]
    )


    # --------------------------------------------------------
    # Store successful experiment
    # --------------------------------------------------------

    gru_epoch_selection_results[
        (
            horizon_minutes,
            feature_family,
        )
    ] = (
        experiment_result.copy()
    )


    gru_epoch_selection_histories[
        (
            horizon_minutes,
            feature_family,
        )
    ] = (
        experiment_history.copy()
    )


    print(
        "\nSelected epoch:",
        experiment_result[
            "best_epoch"
        ],
    )

    print(
        "Validation MAE:",
        f"{experiment_result['validation_MAE_kW']:.6f}",
        "kW",
    )


# ------------------------------------------------------------
# 6. Confirm all 15 combinations completed
# ------------------------------------------------------------

expected_gru_epoch_keys = set(
    gru_epoch_selection_experiments
)


assert set(
    gru_epoch_selection_results.keys()
) == expected_gru_epoch_keys


assert set(
    gru_epoch_selection_histories.keys()
) == expected_gru_epoch_keys


assert (
    len(
        gru_epoch_selection_results
    )
    ==
    15
)


assert (
    len(
        gru_epoch_selection_histories
    )
    ==
    15
)


# ------------------------------------------------------------
# 7. Build complete validation-selection result table
# ------------------------------------------------------------

gru_epoch_selection_rows = []


for (
    horizon_minutes,
    feature_family,
) in gru_epoch_selection_experiments:

    result = (
        gru_epoch_selection_results[
            (
                horizon_minutes,
                feature_family,
            )
        ]
    )


    gru_epoch_selection_rows.append(
        {
            "model":
                result[
                    "model"
                ],

            "candidate_id":
                result[
                    "candidate_id"
                ],

            "horizon_minutes":
                result[
                    "horizon_minutes"
                ],

            "feature_family":
                result[
                    "feature_family"
                ],

            "n_train":
                result[
                    "n_train"
                ],

            "n_validation":
                result[
                    "n_validation"
                ],

            "auxiliary_dim":
                result[
                    "auxiliary_dim"
                ],

            "trainable_parameter_count":
                result[
                    "trainable_parameter_count"
                ],

            "epochs_ran":
                result[
                    "epochs_ran"
                ],

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "callback_best_val_loss_scaled":
                result[
                    "callback_best_val_loss_scaled"
                ],

            "validation_MAE_kW":
                result[
                    "validation_MAE_kW"
                ],

            "validation_RMSE_kW":
                result[
                    "validation_RMSE_kW"
                ],

            "validation_R2":
                result[
                    "validation_R2"
                ],

            "fit_seconds":
                result[
                    "fit_seconds"
                ],
        }
    )


gru_epoch_selection_table = pd.DataFrame(
    gru_epoch_selection_rows
)


assert (
    gru_epoch_selection_table.shape
    ==
    (
        15,
        15,
    )
)


# ------------------------------------------------------------
# 8. Build frozen best-epoch mapping
# ------------------------------------------------------------

GRU_FROZEN_BEST_EPOCHS = {
    (
        int(
            row[
                "horizon_minutes"
            ]
        ),
        str(
            row[
                "feature_family"
            ]
        ),
    ):
        int(
            row[
                "best_epoch"
            ]
        )

    for _, row in (
        gru_epoch_selection_table
        .iterrows()
    )
}


assert (
    len(
        GRU_FROZEN_BEST_EPOCHS
    )
    ==
    15
)


# ------------------------------------------------------------
# 9. Create compact epoch matrix
# ------------------------------------------------------------

gru_frozen_epoch_matrix = (
    gru_epoch_selection_table
    .pivot(
        index=
            "horizon_minutes",

        columns=
            "feature_family",

        values=
            "best_epoch",
    )
    .reindex(
        index=[
            15,
            30,
            60,
        ],

        columns=[
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ],
    )
    .astype(
        int
    )
)


print(
    "\n============================================================"
)

print(
    "GRU FROZEN BEST-EPOCH MATRIX"
)

print(
    "============================================================"
)


display(
    gru_frozen_epoch_matrix
)


# ------------------------------------------------------------
# 10. Display complete validation table
# ------------------------------------------------------------

print(
    "\nGRU VALIDATION-BASED EPOCH-SELECTION RESULTS"
)

print(
    "============================================"
)


display(
    gru_epoch_selection_table[
        [
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "epochs_ran",
            "best_epoch",
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
            "fit_seconds",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Compute validation F3 -> F4 descriptive comparison
#
# Positive:
#       F4 has lower validation MAE.
#
# Negative:
#       F4 has higher validation MAE.
#
# This is descriptive only.
# ------------------------------------------------------------

gru_validation_f3_f4_rows = []


for horizon_minutes in GRU_HORIZONS:

    f3_mae = (
        gru_epoch_selection_results[
            (
                horizon_minutes,
                "F3",
            )
        ][
            "validation_MAE_kW"
        ]
    )


    f4_mae = (
        gru_epoch_selection_results[
            (
                horizon_minutes,
                "F4",
            )
        ][
            "validation_MAE_kW"
        ]
    )


    improvement_pct = (
        (
            f3_mae
            -
            f4_mae
        )
        /
        f3_mae
        *
        100.0
    )


    gru_validation_f3_f4_rows.append(
        {
            "horizon_minutes":
                horizon_minutes,

            "F3_validation_MAE_kW":
                f3_mae,

            "F4_validation_MAE_kW":
                f4_mae,

            "F3_to_F4_validation_MAE_improvement_pct":
                improvement_pct,
        }
    )


gru_validation_f3_f4_summary = pd.DataFrame(
    gru_validation_f3_f4_rows
)


print(
    "\nGRU VALIDATION F3 -> F4 SUMMARY"
)

print(
    "================================"
)


display(
    gru_validation_f3_f4_summary
)


# ------------------------------------------------------------
# 12. Persist complete epoch-selection table
# ------------------------------------------------------------

gru_epoch_selection_table_path = (
    RESULTS_DIR
    /
    "gru_epoch_selection_results.csv"
)


gru_frozen_epoch_matrix_path = (
    RESULTS_DIR
    /
    "gru_frozen_epoch_selection.csv"
)


gru_validation_f3_f4_summary_path = (
    RESULTS_DIR
    /
    "gru_validation_f3_f4_summary.csv"
)


gru_epoch_selection_table.to_csv(
    gru_epoch_selection_table_path,
    index=False,
)


gru_frozen_epoch_matrix.to_csv(
    gru_frozen_epoch_matrix_path,
    index=True,
)


gru_validation_f3_f4_summary.to_csv(
    gru_validation_f3_f4_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 13. Persist all validation histories
# ------------------------------------------------------------

for (
    horizon_minutes,
    feature_family,
), history_df in (
    gru_epoch_selection_histories.items()
):

    history_path = (
        RESULTS_DIR
        /
        (
            f"gru_epoch_selection_"
            f"{horizon_minutes}min_"
            f"{feature_family}_history.csv"
        )
    )


    history_df.to_csv(
        history_path,
        index=False,
    )


    assert (
        history_path.exists()
    )


    assert (
        history_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 14. Verify persisted summary artifacts
# ------------------------------------------------------------

for output_path in [
    gru_epoch_selection_table_path,
    gru_frozen_epoch_matrix_path,
    gru_validation_f3_f4_summary_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Final protocol checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU VALIDATION-BASED EPOCH SELECTION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen architecture:",
    GRU_SELECTED_CANDIDATE_ID,
)


print(
    "Validation experiments completed: 15 / 15"
)


print(
    "60-min F0 reused from architecture selection: YES"
)


print(
    "Held-out TEST used: NO"
)


print(
    "\nAll 15 GRU epoch counts are now frozen."
)

Frozen GRU architecture:
{'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

Reused existing architecture-selection run:
  60-min F0
  best epoch: 3
  no retraining performed

GRU epoch-selection experiment:
  horizon = 15 minutes
  feature family = F0
Epoch 1/80


2026-09-21 20:15:22.904833: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 20:15:22.905165: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 5s - 16ms/step - loss: 0.1456 - val_loss: 0.1934
Epoch 2/80
341/341 - 4s - 12ms/step - loss: 0.1314 - val_loss: 0.1964
Epoch 3/80
341/341 - 4s - 11ms/step - loss: 0.1255 - val_loss: 0.1800
Epoch 4/80
341/341 - 4s - 11ms/step - loss: 0.1287 - val_loss: 0.1921
Epoch 5/80
341/341 - 4s - 11ms/step - loss: 0.1376 - val_loss: 0.1902
Epoch 6/80
341/341 - 4s - 11ms/step - loss: 0.1290 - val_loss: 0.1960
Epoch 7/80
341/341 - 4s - 11ms/step - loss: 0.1299 - val_loss: 0.1858
Epoch 8/80
341/341 - 4s - 11ms/step - loss: 0.1244 - val_loss: 0.1936
Epoch 9/80
341/341 - 4s - 11ms/step - loss: 0.1230 - val_loss: 0.1933
Epoch 10/80
341/341 - 4s - 11ms/step - loss: 0.1266 - val_loss: 0.1833
Epoch 11/80
341/341 - 4s - 11ms/step - loss: 0.1244 - val_loss: 0.1879


2026-09-21 20:16:05.510753: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 20:16:05.511012: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


Selected epoch: 3
Validation MAE: 0.285771 kW

GRU epoch-selection experiment:
  horizon = 15 minutes
  feature family = F1
Epoch 1/80


...


GRU VALIDATION-BASED EPOCH-SELECTION RESULTS


,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2,fit_seconds
0,15,F0,1,16545,11,3,0.285771,0.641117,0.890627,42.734853
1,15,F1,16,17025,12,4,0.283971,0.629744,0.894472,46.049460
2,15,F2,21,17185,15,7,0.280915,0.639890,0.891045,59.310043
3,15,F3,30,17473,10,2,0.297938,0.646145,0.888904,38.718935
4,15,F4,127,20577,12,4,0.293352,0.640418,0.890865,46.294188
5,30,F0,1,16545,12,4,0.360006,0.733881,0.856686,46.809522
6,30,F1,16,17025,19,11,0.360505,0.732664,0.857161,72.852065
7,30,F2,21,17185,10,2,0.360275,0.730244,0.858103,38.815570
8,30,F3,30,17473,13,5,0.375805,0.748993,0.850723,49.277869
9,30,F4,127,20577,12,4,0.370649,0.738923,0.854710,46.081424



GRU VALIDATION F3 -> F4 SUMMARY


,horizon_minutes,F3_validation_MAE_kW,F4_validation_MAE_kW,F3_to_F4_validation_MAE_improvement_pct
0,15,0.297938,0.293352,1.539284
1,30,0.375805,0.370649,1.372076
2,60,0.457618,0.486494,-6.310108



GRU VALIDATION-BASED EPOCH SELECTION: COMPLETE
Frozen architecture: RNN_C2
Validation experiments completed: 15 / 15
60-min F0 reused from architecture selection: YES
Held-out TEST used: NO

All 15 GRU epoch counts are now frozen.


In [81]:
# ============================================================
# Step 65: Freeze GRU final-refit / held-out TEST protocol
#
# PURPOSE
# -------
#
# GRU architecture selection is complete.
#
# GRU validation-based epoch selection is complete.
#
# Therefore all model-selection decisions are now frozen
# BEFORE any GRU held-out TEST evaluation.
#
#
# FROZEN GRU ARCHITECTURE
# -----------------------
#
#       candidate_id = RNN_C2
#       units = 64
#       recurrent layers = 1
#       dropout = 0.0
#       fusion dense units = 32
#
#
# FINAL REFIT PROTOCOL
# --------------------
#
# For each horizon × feature-family experiment:
#
#   1. initialize a fresh GRU using seed = 42
#
#   2. use the ORIGINAL train-only scalers already fitted
#      earlier in the notebook
#
#   3. concatenate:
#
#           TRAIN + VALIDATION
#
#      in chronological order
#
#   4. fit for EXACTLY the frozen best epoch count selected
#      from the original TRAIN / VALIDATION experiment
#
#   5. use:
#
#           batch_size = 256
#           shuffle = True
#           NO validation_data
#           NO callbacks
#           NO early stopping
#
#   6. evaluate the held-out TEST set exactly once
#
#   7. perform NO prediction clipping
#
#
# IMPORTANT
# ---------
#
# This cell DEFINES the final-evaluation helper.
#
# It does NOT call the helper.
#
# Therefore no new GRU TEST prediction is produced here.
# ============================================================


import gc
import json
import time


# ------------------------------------------------------------
# 1. Freeze architecture explicitly
# ------------------------------------------------------------

GRU_FINAL_CANDIDATE_CONFIG = {
    "candidate_id":
        "RNN_C2",

    "units":
        64,

    "n_recurrent_layers":
        1,

    "dropout":
        0.0,

    "fusion_dense_units":
        32,
}


assert (
    GRU_SELECTED_CANDIDATE_CONFIG
    ==
    GRU_FINAL_CANDIDATE_CONFIG
)


# ------------------------------------------------------------
# 2. Freeze the 15 validation-selected epoch counts explicitly
#
# These values must NEVER be changed after viewing TEST
# performance.
# ------------------------------------------------------------

GRU_FROZEN_EPOCHS_EXPECTED = {
    (15, "F0"): 3,
    (15, "F1"): 4,
    (15, "F2"): 7,
    (15, "F3"): 2,
    (15, "F4"): 4,

    (30, "F0"): 4,
    (30, "F1"): 11,
    (30, "F2"): 2,
    (30, "F3"): 5,
    (30, "F4"): 4,

    (60, "F0"): 3,
    (60, "F1"): 13,
    (60, "F2"): 3,
    (60, "F3"): 5,
    (60, "F4"): 1,
}


assert (
    GRU_FROZEN_BEST_EPOCHS
    ==
    GRU_FROZEN_EPOCHS_EXPECTED
)


# Use a separate immutable-by-convention final mapping.
GRU_FINAL_FROZEN_EPOCHS = dict(
    GRU_FROZEN_EPOCHS_EXPECTED
)


assert (
    len(
        GRU_FINAL_FROZEN_EPOCHS
    )
    ==
    15
)


# ------------------------------------------------------------
# 3. Freeze expected auxiliary widths
#
# These are identical to the controlled feature-family widths
# used throughout the deep-learning experiment.
# ------------------------------------------------------------

GRU_EXPECTED_AUX_DIMS = {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


# ------------------------------------------------------------
# 4. Freeze expected parameter counts for RNN_C2
#
# Only the auxiliary-input width changes across F0-F4.
# The recurrent architecture itself remains fixed.
# ------------------------------------------------------------

GRU_EXPECTED_PARAMETER_COUNTS = {
    "F0": 16_545,
    "F1": 17_025,
    "F2": 17_185,
    "F3": 17_473,
    "F4": 20_577,
}


# ------------------------------------------------------------
# 5. Verify parameter counts against validation-selection
#    results already obtained.
# ------------------------------------------------------------

for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    observed_counts = (
        gru_epoch_selection_table.loc[
            gru_epoch_selection_table[
                "feature_family"
            ]
            .eq(
                feature_family
            ),
            "trainable_parameter_count",
        ]
        .unique()
    )


    assert (
        len(
            observed_counts
        )
        ==
        1
    )


    assert (
        int(
            observed_counts[
                0
            ]
        )
        ==
        GRU_EXPECTED_PARAMETER_COUNTS[
            feature_family
        ]
    )


# ------------------------------------------------------------
# 6. Freeze expected population sizes
#
# Final refit population:
#
#       TRAIN + VALIDATION
#
# Held-out TEST remains separate.
# ------------------------------------------------------------

GRU_EXPECTED_POPULATIONS = {
    15: {
        "n_train": 87_261,
        "n_validation": 17_568,
        "n_trainval": 104_829,
        "n_test": 26_496,
    },

    30: {
        "n_train": 87_258,
        "n_validation": 17_568,
        "n_trainval": 104_826,
        "n_test": 26_496,
    },

    60: {
        "n_train": 87_252,
        "n_validation": 17_568,
        "n_trainval": 104_820,
        "n_test": 26_496,
    },
}


# ------------------------------------------------------------
# 7. Helper:
#
#    retrieve chronological TRAIN + VALIDATION data
#
# IMPORTANT:
#
# No scaler is refitted.
#
# get_deep_split_data() returns arrays already transformed
# using the original TRAIN-only scalers.
# ------------------------------------------------------------

def get_gru_final_refit_training_data(
    horizon_minutes: int,
    feature_family: str,
):

    assert (
        horizon_minutes
        in
        [
            15,
            30,
            60,
        ]
    )


    assert (
        feature_family
        in
        GRU_EXPECTED_AUX_DIMS
    )


    train_data = (
        get_deep_split_data(
            horizon_minutes,
            feature_family,
            "train",
        )
    )


    validation_data = (
        get_deep_split_data(
            horizon_minutes,
            feature_family,
            "validation",
        )
    )


    expected_population = (
        GRU_EXPECTED_POPULATIONS[
            horizon_minutes
        ]
    )


    # --------------------------------------------------------
    # Population checks
    # --------------------------------------------------------

    assert (
        train_data[
            "n_samples"
        ]
        ==
        expected_population[
            "n_train"
        ]
    )


    assert (
        validation_data[
            "n_samples"
        ]
        ==
        expected_population[
            "n_validation"
        ]
    )


    # --------------------------------------------------------
    # Input-width checks
    # --------------------------------------------------------

    expected_aux_dim = (
        GRU_EXPECTED_AUX_DIMS[
            feature_family
        ]
    )


    assert (
        train_data[
            "sequence"
        ].shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    assert (
        validation_data[
            "sequence"
        ].shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    assert (
        train_data[
            "auxiliary"
        ].shape[
            1
        ]
        ==
        expected_aux_dim
    )


    assert (
        validation_data[
            "auxiliary"
        ].shape[
            1
        ]
        ==
        expected_aux_dim
    )


    # --------------------------------------------------------
    # Chronological concatenation:
    #
    #       TRAIN first
    #       VALIDATION second
    # --------------------------------------------------------

    trainval_sequence = np.concatenate(
        [
            train_data[
                "sequence"
            ],
            validation_data[
                "sequence"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_auxiliary = np.concatenate(
        [
            train_data[
                "auxiliary"
            ],
            validation_data[
                "auxiliary"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_target_scaled = np.concatenate(
        [
            train_data[
                "target_scaled"
            ],
            validation_data[
                "target_scaled"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    # --------------------------------------------------------
    # Final population checks
    # --------------------------------------------------------

    expected_n_trainval = (
        expected_population[
            "n_trainval"
        ]
    )


    assert (
        trainval_sequence.shape[
            0
        ]
        ==
        expected_n_trainval
    )


    assert (
        trainval_auxiliary.shape[
            0
        ]
        ==
        expected_n_trainval
    )


    assert (
        trainval_target_scaled.shape[
            0
        ]
        ==
        expected_n_trainval
    )


    return {
        "sequence":
            trainval_sequence,

        "auxiliary":
            trainval_auxiliary,

        "target_scaled":
            trainval_target_scaled,

        "n_train":
            train_data[
                "n_samples"
            ],

        "n_validation":
            validation_data[
                "n_samples"
            ],

        "n_trainval":
            expected_n_trainval,
    }


# ------------------------------------------------------------
# 8. Canonical GRU final-refit + held-out TEST helper
#
# This helper is defined now but NOT executed in this cell.
# ------------------------------------------------------------

def run_gru_final_refit_and_test(
    horizon_minutes: int,
    feature_family: str,
    verbose: int = 2,
):

    # --------------------------------------------------------
    # Validate requested experiment
    # --------------------------------------------------------

    experiment_key = (
        horizon_minutes,
        feature_family,
    )


    assert (
        experiment_key
        in
        GRU_FINAL_FROZEN_EPOCHS
    )


    frozen_best_epoch = int(
        GRU_FINAL_FROZEN_EPOCHS[
            experiment_key
        ]
    )


    expected_aux_dim = (
        GRU_EXPECTED_AUX_DIMS[
            feature_family
        ]
    )


    expected_parameter_count = (
        GRU_EXPECTED_PARAMETER_COUNTS[
            feature_family
        ]
    )


    expected_population = (
        GRU_EXPECTED_POPULATIONS[
            horizon_minutes
        ]
    )


    # --------------------------------------------------------
    # Retrieve TRAIN + VALIDATION
    # --------------------------------------------------------

    trainval_data = (
        get_gru_final_refit_training_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,
        )
    )


    # --------------------------------------------------------
    # Retrieve held-out TEST
    #
    # This is the first point at which this helper touches the
    # GRU TEST population.
    # --------------------------------------------------------

    test_data = (
        get_deep_split_data(
            horizon_minutes,
            feature_family,
            "test",
        )
    )


    assert (
        test_data[
            "n_samples"
        ]
        ==
        expected_population[
            "n_test"
        ]
    )


    assert (
        test_data[
            "sequence"
        ].shape
        ==
        (
            expected_population[
                "n_test"
            ],
            13,
            9,
        )
    )


    assert (
        test_data[
            "auxiliary"
        ].shape
        ==
        (
            expected_population[
                "n_test"
            ],
            expected_aux_dim,
        )
    )


    # --------------------------------------------------------
    # Fresh deterministic model initialization
    # --------------------------------------------------------

    tf.keras.backend.clear_session()


    final_model = (
        build_deep_forecasting_model(
            model_name=
                "GRU",

            auxiliary_dim=
                expected_aux_dim,

            candidate_config=
                dict(
                    GRU_FINAL_CANDIDATE_CONFIG
                ),

            seed=
                42,
        )
    )


    # --------------------------------------------------------
    # Architecture / parameter-count audit
    # --------------------------------------------------------

    actual_parameter_count = int(
        final_model.count_params()
    )


    assert (
        actual_parameter_count
        ==
        expected_parameter_count
    )


    # --------------------------------------------------------
    # Final TRAIN + VALIDATION refit
    #
    # EXACT frozen epoch count.
    #
    # Deliberately:
    #
    #       NO validation_data
    #       NO callbacks
    #       NO EarlyStopping
    # --------------------------------------------------------

    fit_start_time = (
        time.perf_counter()
    )


    final_history_object = (
        final_model.fit(
            [
                trainval_data[
                    "sequence"
                ],
                trainval_data[
                    "auxiliary"
                ],
            ],

            trainval_data[
                "target_scaled"
            ],

            epochs=
                frozen_best_epoch,

            batch_size=
                256,

            shuffle=
                True,

            verbose=
                verbose,
        )
    )


    final_fit_seconds = (
        time.perf_counter()
        -
        fit_start_time
    )


    # --------------------------------------------------------
    # Convert final fitting history to auditable DataFrame
    # --------------------------------------------------------

    final_history = pd.DataFrame(
        final_history_object.history
    )


    final_history.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                final_history
            )
            +
            1,
            dtype=int,
        ),
    )


    assert (
        len(
            final_history
        )
        ==
        frozen_best_epoch
    )


    assert (
        "loss"
        in
        final_history.columns
    )


    # Critical audit:
    #
    # final refit MUST NOT contain validation loss because
    # validation has been merged into the fitting population.
    assert (
        "val_loss"
        not in
        final_history.columns
    )


    # --------------------------------------------------------
    # Single held-out TEST prediction
    #
    # No clipping.
    # --------------------------------------------------------

    test_prediction_scaled = (
        final_model.predict(
            [
                test_data[
                    "sequence"
                ],
                test_data[
                    "auxiliary"
                ],
            ],
            verbose=0,
        )
        .reshape(
            -1
        )
    )


    # --------------------------------------------------------
    # Convert predictions back to physical kW units using the
    # ORIGINAL train-only target scaler.
    # --------------------------------------------------------

    test_prediction_kw = (
        inverse_transform_target_to_kw(
            scaled_values=
                test_prediction_scaled,

            horizon_minutes=
                horizon_minutes,
        )
        .reshape(
            -1
        )
    )


    test_actual_kw = (
        np.asarray(
            test_data[
                "target_kw"
            ],
            dtype=float,
        )
        .reshape(
            -1
        )
    )


    assert (
        test_actual_kw.shape
        ==
        test_prediction_kw.shape
        ==
        (
            expected_population[
                "n_test"
            ],
        )
    )


    # --------------------------------------------------------
    # Held-out regression metrics
    # --------------------------------------------------------

    test_metrics = (
        compute_regression_metrics_kw(
            test_actual_kw,
            test_prediction_kw,
        )
    )


    # --------------------------------------------------------
    # Final auditable result row
    # --------------------------------------------------------

    final_result = {
        "model":
            "GRU",

        "candidate_id":
            GRU_FINAL_CANDIDATE_CONFIG[
                "candidate_id"
            ],

        "horizon_minutes":
            horizon_minutes,

        "feature_family":
            feature_family,

        "auxiliary_dim":
            expected_aux_dim,

        "trainable_parameter_count":
            actual_parameter_count,

        "n_train":
            expected_population[
                "n_train"
            ],

        "n_validation":
            expected_population[
                "n_validation"
            ],

        "n_trainval":
            expected_population[
                "n_trainval"
            ],

        "n_test":
            expected_population[
                "n_test"
            ],

        "frozen_best_epoch":
            frozen_best_epoch,

        "final_epochs_trained":
            len(
                final_history
            ),

        "test_MAE_kW":
            float(
                test_metrics[
                    "MAE_kW"
                ]
            ),

        "test_RMSE_kW":
            float(
                test_metrics[
                    "RMSE_kW"
                ]
            ),

        "test_R2":
            float(
                test_metrics[
                    "R2"
                ]
            ),

        "final_fit_seconds":
            float(
                final_fit_seconds
            ),

        "prediction_clipping":
            False,

        "scaler_policy":
            "original_train_only_scalers",

        "refit_population":
            "train_plus_validation",
    }


    # --------------------------------------------------------
    # Cleanup
    #
    # Predictions / targets / result / history remain returned.
    # The Keras model itself is not retained in memory.
    # --------------------------------------------------------

    del final_model
    del final_history_object

    tf.keras.backend.clear_session()

    gc.collect()


    return (
        final_result,
        final_history,
        test_actual_kw,
        test_prediction_kw,
    )


# ------------------------------------------------------------
# 9. Preflight TRAIN + VALIDATION populations only
#
# This verifies the refit-data helper without executing a TEST
# evaluation.
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        preflight_trainval = (
            get_gru_final_refit_training_data(
                horizon_minutes=
                    horizon_minutes,

                feature_family=
                    feature_family,
            )
        )


        assert (
            preflight_trainval[
                "n_trainval"
            ]
            ==
            GRU_EXPECTED_POPULATIONS[
                horizon_minutes
            ][
                "n_trainval"
            ]
        )


        assert (
            preflight_trainval[
                "auxiliary"
            ].shape[
                1
            ]
            ==
            GRU_EXPECTED_AUX_DIMS[
                feature_family
            ]
        )


# ------------------------------------------------------------
# 10. Persist the frozen GRU final protocol BEFORE TEST use
# ------------------------------------------------------------

gru_frozen_protocol_path = (
    RESULTS_DIR
    /
    "gru_frozen_protocol.json"
)


gru_frozen_protocol_payload = {
    "model":
        "GRU",

    "architecture":
        GRU_FINAL_CANDIDATE_CONFIG,

    "architecture_selection":
        {
            "horizon_minutes": 60,
            "feature_family": "F0",
            "test_used": False,
        },

    "frozen_best_epochs":
        {
            str(
                horizon_minutes
            ):
                {
                    feature_family:
                        int(
                            GRU_FINAL_FROZEN_EPOCHS[
                                (
                                    horizon_minutes,
                                    feature_family,
                                )
                            ]
                        )

                    for feature_family in [
                        "F0",
                        "F1",
                        "F2",
                        "F3",
                        "F4",
                    ]
                }

            for horizon_minutes in [
                15,
                30,
                60,
            ]
        },

    "auxiliary_dimensions":
        GRU_EXPECTED_AUX_DIMS,

    "expected_parameter_counts":
        GRU_EXPECTED_PARAMETER_COUNTS,

    "final_refit":
        {
            "population":
                "train_plus_validation",

            "scaler_policy":
                "original_train_only_scalers",

            "batch_size":
                256,

            "shuffle":
                True,

            "callbacks":
                None,

            "validation_during_final_refit":
                False,

            "prediction_clipping":
                False,

            "seed":
                42,
        },

    "test_evaluations_completed_at_freeze":
        0,
}


with open(
    gru_frozen_protocol_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        gru_frozen_protocol_payload,
        file,
        indent=2,
    )


assert (
    gru_frozen_protocol_path.exists()
)


assert (
    gru_frozen_protocol_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 11. Initialize EMPTY final GRU TEST containers
#
# Their emptiness is part of the protocol checkpoint.
# ------------------------------------------------------------

gru_final_test_results = {}

gru_final_test_histories = {}

gru_final_test_actuals = {}

gru_final_test_predictions = {}


assert (
    len(
        gru_final_test_results
    )
    ==
    0
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    0
)


# ------------------------------------------------------------
# 12. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU FINAL-REFIT PROTOCOL: FROZEN"
)

print(
    "============================================================"
)


print(
    "Architecture:",
    GRU_FINAL_CANDIDATE_CONFIG,
)


print(
    "\nFrozen epoch matrix:"
)


display(
    pd.DataFrame(
        {
            feature_family: [
                GRU_FINAL_FROZEN_EPOCHS[
                    (
                        horizon_minutes,
                        feature_family,
                    )
                ]

                for horizon_minutes in [
                    15,
                    30,
                    60,
                ]
            ]

            for feature_family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        },
        index=[
            15,
            30,
            60,
        ],
    )
    .rename_axis(
        "horizon_minutes"
    )
)


print(
    "Final-refit helper defined successfully."
)

print(
    "TRAIN + VALIDATION preflight: PASSED"
)

print(
    "GRU held-out TEST evaluations completed: 0 / 15"
)

print(
    "No GRU TEST prediction was produced in this cell."
)

print(
    "\nNext experiment: 15-minute F0 final refit + TEST."
)


GRU FINAL-REFIT PROTOCOL: FROZEN
Architecture: {'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

Frozen epoch matrix:


,F0,F1,F2,F3,F4
horizon_minutes,,,,,
15,3,4,7,2,4
30,4,11,2,5,4
60,3,13,3,5,1


Final-refit helper defined successfully.
TRAIN + VALIDATION preflight: PASSED
GRU held-out TEST evaluations completed: 0 / 15
No GRU TEST prediction was produced in this cell.

Next experiment: 15-minute F0 final refit + TEST.


In [82]:
# ============================================================
# Step 66: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 15 minutes
#       feature family = F0
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       3
#
# Auxiliary width:
#       1
#
# Expected parameters:
#       16,545
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,261 + 17,568 = 104,829
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# This is the FIRST GRU held-out TEST evaluation.
#
# No tuning will occur after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 15

GRU_TEST_FEATURE_FAMILY = "F0"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Pre-test protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    3
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    1
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16_545
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_261
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# This experiment must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running FIRST GRU held-out TEST experiment:"
)

print(
    "  horizon: 15 minutes"
)

print(
    "  feature family: F0"
)

print(
    "  frozen epoch: 3"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 3. Run canonical final refit + TEST
# ------------------------------------------------------------

(
    gru_15_f0_result,
    gru_15_f0_history,
    gru_15_f0_actual_kw,
    gru_15_f0_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Audit returned result
# ------------------------------------------------------------

assert (
    gru_15_f0_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_15_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_15_f0_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    gru_15_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


assert (
    gru_15_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    gru_15_f0_result[
        "trainable_parameter_count"
    ]
    ==
    16_545
)


assert (
    gru_15_f0_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    gru_15_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_15_f0_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    gru_15_f0_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_15_f0_result[
        "frozen_best_epoch"
    ]
    ==
    3
)


assert (
    gru_15_f0_result[
        "final_epochs_trained"
    ]
    ==
    3
)


assert (
    gru_15_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_15_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_15_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 5. Final fitting-history audit
#
# Exactly 3 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_15_f0_history
    )
    ==
    3
)


assert (
    gru_15_f0_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
    ]
)


assert (
    "loss"
    in
    gru_15_f0_history.columns
)


assert (
    "val_loss"
    not in
    gru_15_f0_history.columns
)


# ------------------------------------------------------------
# 6. Prediction/target audit
# ------------------------------------------------------------

assert (
    gru_15_f0_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_15_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_15_f0_actual_kw
).all()


assert np.isfinite(
    gru_15_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 7. Recompute metrics independently
#
# This checks the metrics stored by the helper.
# ------------------------------------------------------------

gru_15_f0_metric_check = (
    compute_regression_metrics_kw(
        gru_15_f0_actual_kw,
        gru_15_f0_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_15_f0_metric_check[
        "MAE_kW"
    ],
    gru_15_f0_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f0_metric_check[
        "RMSE_kW"
    ],
    gru_15_f0_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f0_metric_check[
        "R2"
    ],
    gru_15_f0_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 8. Store in canonical final GRU containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_15_f0_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_15_f0_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_15_f0_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_15_f0_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    1
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    1
)


# ------------------------------------------------------------
# 9. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_15_f0_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_15_f0_history
)


# ------------------------------------------------------------
# 10. Unclipped prediction audit
#
# Negative predictions are NOT modified.
# This is purely descriptive.
# ------------------------------------------------------------

gru_15_f0_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_15_f0_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_15_f0_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_15_f0_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_15_f0_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_15_f0_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_15_f0_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_15_f0_prediction_audit
)


# ------------------------------------------------------------
# 11. Persist result
# ------------------------------------------------------------

gru_15_f0_result_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F0_result.csv"
)


pd.DataFrame(
    [
        gru_15_f0_result
    ]
).to_csv(
    gru_15_f0_result_path,
    index=False,
)


# ------------------------------------------------------------
# 12. Persist final-refit history
# ------------------------------------------------------------

gru_15_f0_history_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F0_history.csv"
)


gru_15_f0_history.to_csv(
    gru_15_f0_history_path,
    index=False,
)


# ------------------------------------------------------------
# 13. Persist row-level held-out predictions
#
# Timestamp alignment will be attached later using the same
# canonical valid-time alignment already established for LSTM.
# ------------------------------------------------------------

gru_15_f0_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F0_predictions.csv"
)


gru_15_f0_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_15_f0_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_15_f0_actual_kw,

        "predicted_P_Solar_kW":
            gru_15_f0_prediction_kw,
    }
)


gru_15_f0_predictions_table.to_csv(
    gru_15_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 14. Verify artifacts
# ------------------------------------------------------------

for output_path in [
    gru_15_f0_result_path,
    gru_15_f0_history_path,
    gru_15_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 15-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_15_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_15_f0_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_15_f0_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_15_f0_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_15_f0_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

Running FIRST GRU held-out TEST experiment:
  horizon: 15 minutes
  feature family: F0
  frozen epoch: 3
  architecture: RNN_C2
Epoch 1/3


2026-09-21 20:57:25.704971: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 20:57:25.705735: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1512
Epoch 2/3
410/410 - 4s - 11ms/step - loss: 0.1389
Epoch 3/3
410/410 - 4s - 10ms/step - loss: 0.1425

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F0,1,16545,87261,17568,104829,26496,3,3,0.33298,0.639402,0.84995,14.215378,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


,epoch,loss
0,1,0.151192
1,2,0.138886
2,3,0.142522



Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.400584,6.214066,4740,0



GRU 15-MIN F0 FINAL TEST: COMPLETE
Frozen epoch used: 3
Final epochs trained: 3
TEST MAE: 0.332980 kW
TEST RMSE: 0.639402 kW
TEST R²: 0.849950

GRU final TEST evaluations completed: 1 / 15
Remaining: 14
No prediction clipping was applied.


In [83]:
# ============================================================
# Step 67: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 15 minutes
#       feature family = F1
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       4
#
# Auxiliary width:
#       16
#
# Expected parameters:
#       17,025
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,261 + 17,568 = 104,829
#
# Held-out TEST:
#       26,496
#
#
# This is the SECOND GRU held-out TEST evaluation.
#
# No tuning occurs after observing F0 or F1 TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 15

GRU_TEST_FEATURE_FAMILY = "F1"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    4
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_025
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_261
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F0 must already exist.
assert (
    (
        15,
        "F0",
    )
    in
    gru_final_test_results
)


# F1 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 15 minutes"
)

print(
    "  feature family: F1"
)

print(
    "  frozen epoch: 4"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 3. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_15_f1_result,
    gru_15_f1_history,
    gru_15_f1_actual_kw,
    gru_15_f1_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Audit returned result
# ------------------------------------------------------------

assert (
    gru_15_f1_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_15_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_15_f1_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    gru_15_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


assert (
    gru_15_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    gru_15_f1_result[
        "trainable_parameter_count"
    ]
    ==
    17_025
)


assert (
    gru_15_f1_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    gru_15_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_15_f1_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    gru_15_f1_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_15_f1_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    gru_15_f1_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    gru_15_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_15_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_15_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 5. Final-refit history audit
#
# Exactly 4 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_15_f1_history
    )
    ==
    4
)


assert (
    gru_15_f1_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
    ]
)


assert (
    "loss"
    in
    gru_15_f1_history.columns
)


assert (
    "val_loss"
    not in
    gru_15_f1_history.columns
)


# ------------------------------------------------------------
# 6. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_15_f1_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_15_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_15_f1_actual_kw
).all()


assert np.isfinite(
    gru_15_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 7. Critical same-horizon target-population check
#
# F0 and F1 must be evaluated against the exact same target
# vector in the exact same row order.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    gru_15_f1_actual_kw,
    gru_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
)


# ------------------------------------------------------------
# 8. Independently recompute metrics
# ------------------------------------------------------------

gru_15_f1_metric_check = (
    compute_regression_metrics_kw(
        gru_15_f1_actual_kw,
        gru_15_f1_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_15_f1_metric_check[
        "MAE_kW"
    ],
    gru_15_f1_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f1_metric_check[
        "RMSE_kW"
    ],
    gru_15_f1_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f1_metric_check[
        "R2"
    ],
    gru_15_f1_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 9. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_15_f1_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_15_f1_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_15_f1_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_15_f1_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    2
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    2
)


# ------------------------------------------------------------
# 10. Display final held-out result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_15_f1_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_15_f1_history
)


# ------------------------------------------------------------
# 11. Descriptive F0 -> F1 TEST comparison
#
# Positive percentage:
#
#       F1 has lower MAE than F0.
#
# Negative percentage:
#
#       F1 has higher MAE than F0.
#
# This is descriptive TEST reporting only.
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_15_f0_mae = float(
    gru_final_test_results[
        (
            15,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_15_f1_mae = float(
    gru_15_f1_result[
        "test_MAE_kW"
    ]
)


gru_15_f0_to_f1_improvement_pct = (
    (
        gru_15_f0_mae
        -
        gru_15_f1_mae
    )
    /
    gru_15_f0_mae
    *
    100.0
)


print(
    "\n15-min held-out TEST F0 -> F1 comparison:"
)


print(
    "  F0 MAE:",
    f"{gru_15_f0_mae:.6f}",
    "kW",
)


print(
    "  F1 MAE:",
    f"{gru_15_f1_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_15_f0_to_f1_improvement_pct:+.4f}%",
)


print(
    "  Positive means F1 has lower TEST MAE."
)


# ------------------------------------------------------------
# 12. Unclipped prediction audit
# ------------------------------------------------------------

gru_15_f1_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_15_f1_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_15_f1_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_15_f1_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_15_f1_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_15_f1_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_15_f1_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_15_f1_prediction_audit
)


# ------------------------------------------------------------
# 13. Persist result
# ------------------------------------------------------------

gru_15_f1_result_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F1_result.csv"
)


pd.DataFrame(
    [
        gru_15_f1_result
    ]
).to_csv(
    gru_15_f1_result_path,
    index=False,
)


# ------------------------------------------------------------
# 14. Persist final-refit history
# ------------------------------------------------------------

gru_15_f1_history_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F1_history.csv"
)


gru_15_f1_history.to_csv(
    gru_15_f1_history_path,
    index=False,
)


# ------------------------------------------------------------
# 15. Persist row-level predictions
# ------------------------------------------------------------

gru_15_f1_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F1_predictions.csv"
)


gru_15_f1_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_15_f1_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_15_f1_actual_kw,

        "predicted_P_Solar_kW":
            gru_15_f1_prediction_kw,
    }
)


gru_15_f1_predictions_table.to_csv(
    gru_15_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 16. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    gru_15_f1_result_path,
    gru_15_f1_history_path,
    gru_15_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Current 15-minute GRU progression
# ------------------------------------------------------------

gru_15min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                15,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 15-minute GRU final TEST progression:"
)


display(
    gru_15min_partial_summary
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 15-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_15_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_15_f1_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_15_f1_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_15_f1_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_15_f1_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 15-minute F2."
)

Running GRU held-out TEST experiment:
  horizon: 15 minutes
  feature family: F1
  frozen epoch: 4
  architecture: RNN_C2
Epoch 1/4


2026-09-21 20:58:53.055336: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 20:58:53.055633: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.1747
Epoch 2/4
410/410 - 5s - 12ms/step - loss: 0.1325
Epoch 3/4
410/410 - 4s - 11ms/step - loss: 0.1316
Epoch 4/4
410/410 - 4s - 10ms/step - loss: 0.1342

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F1,16,17025,87261,17568,104829,26496,4,4,0.278635,0.601394,0.867259,20.056457,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


15-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.332980 kW
  F1 MAE: 0.278635 kW
  Relative MAE change: +16.3208%
  Positive means F1 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.176314,5.861999,6087,0



Current 15-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.332980,0.639402,0.849950
1,F1,4,0.278635,0.601394,0.867259



GRU 15-MIN F1 FINAL TEST: COMPLETE
Frozen epoch used: 4
Final epochs trained: 4
TEST MAE: 0.278635 kW
TEST RMSE: 0.601394 kW
TEST R²: 0.867259

GRU final TEST evaluations completed: 2 / 15
Remaining: 13
No prediction clipping was applied.

Next experiment: 15-minute F2.


In [84]:
# ============================================================
# Step 68: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 15 minutes
#       feature family = F2
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       7
#
# Auxiliary width:
#       21
#
# Expected parameters:
#       17,185
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,261 + 17,568 = 104,829
#
# Held-out TEST:
#       26,496
#
#
# No tuning is performed after observing F0/F1/F2 TEST results.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 15

GRU_TEST_FEATURE_FAMILY = "F2"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    7
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    21
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_185
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F0 and F1 must already exist.
assert (
    (
        15,
        "F0",
    )
    in
    gru_final_test_results
)


assert (
    (
        15,
        "F1",
    )
    in
    gru_final_test_results
)


# F2 must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 15 minutes"
)

print(
    "  feature family: F2"
)

print(
    "  frozen epoch: 7"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 3. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_15_f2_result,
    gru_15_f2_history,
    gru_15_f2_actual_kw,
    gru_15_f2_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_15_f2_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_15_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_15_f2_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    gru_15_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


assert (
    gru_15_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    gru_15_f2_result[
        "trainable_parameter_count"
    ]
    ==
    17_185
)


assert (
    gru_15_f2_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    gru_15_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_15_f2_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    gru_15_f2_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_15_f2_result[
        "frozen_best_epoch"
    ]
    ==
    7
)


assert (
    gru_15_f2_result[
        "final_epochs_trained"
    ]
    ==
    7
)


assert (
    gru_15_f2_result[
        "prediction_clipping"
    ]
    is False
)


# ------------------------------------------------------------
# 5. Final-refit history audit
#
# Exactly 7 epochs.
# No validation loss is allowed.
# ------------------------------------------------------------

assert (
    len(
        gru_15_f2_history
    )
    ==
    7
)


assert (
    gru_15_f2_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
    ]
)


assert (
    "loss"
    in
    gru_15_f2_history.columns
)


assert (
    "val_loss"
    not in
    gru_15_f2_history.columns
)


# ------------------------------------------------------------
# 6. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_15_f2_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_15_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_15_f2_actual_kw
).all()


assert np.isfinite(
    gru_15_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 7. Same-horizon target-population checks
#
# F0, F1, and F2 must all use exactly the same held-out
# target vector in exactly the same row order.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    gru_15_f2_actual_kw,
    gru_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
)


np.testing.assert_array_equal(
    gru_15_f2_actual_kw,
    gru_final_test_actuals[
        (
            15,
            "F1",
        )
    ],
)


# ------------------------------------------------------------
# 8. Independently recompute metrics
# ------------------------------------------------------------

gru_15_f2_metric_check = (
    compute_regression_metrics_kw(
        gru_15_f2_actual_kw,
        gru_15_f2_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_15_f2_metric_check[
        "MAE_kW"
    ],
    gru_15_f2_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f2_metric_check[
        "RMSE_kW"
    ],
    gru_15_f2_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f2_metric_check[
        "R2"
    ],
    gru_15_f2_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 9. Store in canonical GRU containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_15_f2_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_15_f2_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_15_f2_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_15_f2_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    3
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    3
)


# ------------------------------------------------------------
# 10. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_15_f2_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_15_f2_history
)


# ------------------------------------------------------------
# 11. Descriptive F1 -> F2 TEST comparison
#
# Positive:
#       F2 has lower MAE than F1.
#
# Negative:
#       F2 has higher MAE than F1.
#
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_15_f1_mae = float(
    gru_final_test_results[
        (
            15,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_15_f2_mae = float(
    gru_15_f2_result[
        "test_MAE_kW"
    ]
)


gru_15_f1_to_f2_improvement_pct = (
    (
        gru_15_f1_mae
        -
        gru_15_f2_mae
    )
    /
    gru_15_f1_mae
    *
    100.0
)


print(
    "\n15-min held-out TEST F1 -> F2 comparison:"
)


print(
    "  F1 MAE:",
    f"{gru_15_f1_mae:.6f}",
    "kW",
)


print(
    "  F2 MAE:",
    f"{gru_15_f2_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_15_f1_to_f2_improvement_pct:+.4f}%",
)


print(
    "  Positive means F2 has lower TEST MAE."
)


# ------------------------------------------------------------
# 12. Unclipped prediction audit
# ------------------------------------------------------------

gru_15_f2_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_15_f2_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_15_f2_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_15_f2_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_15_f2_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_15_f2_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_15_f2_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_15_f2_prediction_audit
)


# ------------------------------------------------------------
# 13. Persist result, history, and predictions
# ------------------------------------------------------------

gru_15_f2_result_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F2_result.csv"
)


gru_15_f2_history_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F2_history.csv"
)


gru_15_f2_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F2_predictions.csv"
)


pd.DataFrame(
    [
        gru_15_f2_result
    ]
).to_csv(
    gru_15_f2_result_path,
    index=False,
)


gru_15_f2_history.to_csv(
    gru_15_f2_history_path,
    index=False,
)


gru_15_f2_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_15_f2_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_15_f2_actual_kw,

        "predicted_P_Solar_kW":
            gru_15_f2_prediction_kw,
    }
)


gru_15_f2_predictions_table.to_csv(
    gru_15_f2_predictions_path,
    index=False,
)


for output_path in [
    gru_15_f2_result_path,
    gru_15_f2_history_path,
    gru_15_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 14. Current 15-minute GRU progression
# ------------------------------------------------------------

gru_15min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                15,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 15-minute GRU final TEST progression:"
)


display(
    gru_15min_partial_summary
)


# ------------------------------------------------------------
# 15. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 15-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_15_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_15_f2_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_15_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_15_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_15_f2_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 15-minute F3."
)

Running GRU held-out TEST experiment:
  horizon: 15 minutes
  feature family: F2
  frozen epoch: 7
  architecture: RNN_C2
Epoch 1/7


2026-09-21 21:00:46.748035: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:00:46.748456: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1758
Epoch 2/7
410/410 - 5s - 11ms/step - loss: 0.1397
Epoch 3/7
410/410 - 4s - 10ms/step - loss: 0.1431
Epoch 4/7
410/410 - 4s - 10ms/step - loss: 0.1440
Epoch 5/7
410/410 - 4s - 10ms/step - loss: 0.1448
Epoch 6/7
410/410 - 4s - 10ms/step - loss: 0.1507
Epoch 7/7
410/410 - 4s - 10ms/step - loss: 0.1431

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F2,21,17185,87261,17568,104829,26496,7,7,0.299681,0.604346,0.865953,31.73226,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


15-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.278635 kW
  F2 MAE: 0.299681 kW
  Relative MAE change: -7.5533%
  Positive means F2 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.340877,5.879192,4159,0



Current 15-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.332980,0.639402,0.849950
1,F1,4,0.278635,0.601394,0.867259
2,F2,7,0.299681,0.604346,0.865953



GRU 15-MIN F2 FINAL TEST: COMPLETE
Frozen epoch used: 7
Final epochs trained: 7
TEST MAE: 0.299681 kW
TEST RMSE: 0.604346 kW
TEST R²: 0.865953

GRU final TEST evaluations completed: 3 / 15
Remaining: 12
No prediction clipping was applied.

Next experiment: 15-minute F3.


In [85]:
# ============================================================
# Step 69: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 15 minutes
#       feature family = F3
#
#
# F3 represents:
#
#       F2
#       +
#       continuous lunar / Sun-Moon astronomical features
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       2
#
# Auxiliary width:
#       30
#
# Expected parameters:
#       17,473
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,261 + 17,568 = 104,829
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# F3 is the direct astronomical-control model against which
# F4 (Panchang representation) will subsequently be compared.
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 15

GRU_TEST_FEATURE_FAMILY = "F3"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    2
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    30
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_473
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_261
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# Previous 15-minute families must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            15,
            previous_family,
        )
        in
        gru_final_test_results
    )


# F3 must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 15 minutes"
)

print(
    "  feature family: F3"
)

print(
    "  frozen epoch: 2"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 3. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_15_f3_result,
    gru_15_f3_history,
    gru_15_f3_actual_kw,
    gru_15_f3_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_15_f3_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_15_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_15_f3_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    gru_15_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


assert (
    gru_15_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    gru_15_f3_result[
        "trainable_parameter_count"
    ]
    ==
    17_473
)


assert (
    gru_15_f3_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    gru_15_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_15_f3_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    gru_15_f3_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_15_f3_result[
        "frozen_best_epoch"
    ]
    ==
    2
)


assert (
    gru_15_f3_result[
        "final_epochs_trained"
    ]
    ==
    2
)


assert (
    gru_15_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_15_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_15_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 5. Final-refit history audit
#
# Exactly 2 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_15_f3_history
    )
    ==
    2
)


assert (
    gru_15_f3_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
    ]
)


assert (
    "loss"
    in
    gru_15_f3_history.columns
)


assert (
    "val_loss"
    not in
    gru_15_f3_history.columns
)


# ------------------------------------------------------------
# 6. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_15_f3_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_15_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_15_f3_actual_kw
).all()


assert np.isfinite(
    gru_15_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 7. Same-horizon target-population checks
#
# Every 15-minute family must be evaluated against the exact
# same held-out target vector in the same row order.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    np.testing.assert_array_equal(
        gru_15_f3_actual_kw,
        gru_final_test_actuals[
            (
                15,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 8. Independently recompute metrics
# ------------------------------------------------------------

gru_15_f3_metric_check = (
    compute_regression_metrics_kw(
        gru_15_f3_actual_kw,
        gru_15_f3_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_15_f3_metric_check[
        "MAE_kW"
    ],
    gru_15_f3_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f3_metric_check[
        "RMSE_kW"
    ],
    gru_15_f3_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f3_metric_check[
        "R2"
    ],
    gru_15_f3_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 9. Store in canonical GRU containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_15_f3_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_15_f3_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_15_f3_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_15_f3_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    4
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    4
)


# ------------------------------------------------------------
# 10. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_15_f3_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_15_f3_history
)


# ------------------------------------------------------------
# 11. Descriptive F2 -> F3 TEST comparison
#
# Positive:
#       F3 has lower MAE than F2.
#
# Negative:
#       F3 has higher MAE than F2.
#
# No tuning follows from this result.
# ------------------------------------------------------------

gru_15_f2_mae = float(
    gru_final_test_results[
        (
            15,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_15_f3_mae = float(
    gru_15_f3_result[
        "test_MAE_kW"
    ]
)


gru_15_f2_to_f3_improvement_pct = (
    (
        gru_15_f2_mae
        -
        gru_15_f3_mae
    )
    /
    gru_15_f2_mae
    *
    100.0
)


print(
    "\n15-min held-out TEST F2 -> F3 comparison:"
)


print(
    "  F2 MAE:",
    f"{gru_15_f2_mae:.6f}",
    "kW",
)


print(
    "  F3 MAE:",
    f"{gru_15_f3_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_15_f2_to_f3_improvement_pct:+.4f}%",
)


print(
    "  Positive means F3 has lower TEST MAE."
)


# ------------------------------------------------------------
# 12. Unclipped prediction audit
# ------------------------------------------------------------

gru_15_f3_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_15_f3_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_15_f3_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_15_f3_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_15_f3_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_15_f3_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_15_f3_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_15_f3_prediction_audit
)


# ------------------------------------------------------------
# 13. Persist result, history, and row-level predictions
# ------------------------------------------------------------

gru_15_f3_result_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F3_result.csv"
)


gru_15_f3_history_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F3_history.csv"
)


gru_15_f3_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F3_predictions.csv"
)


pd.DataFrame(
    [
        gru_15_f3_result
    ]
).to_csv(
    gru_15_f3_result_path,
    index=False,
)


gru_15_f3_history.to_csv(
    gru_15_f3_history_path,
    index=False,
)


gru_15_f3_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_15_f3_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_15_f3_actual_kw,

        "predicted_P_Solar_kW":
            gru_15_f3_prediction_kw,
    }
)


gru_15_f3_predictions_table.to_csv(
    gru_15_f3_predictions_path,
    index=False,
)


for output_path in [
    gru_15_f3_result_path,
    gru_15_f3_history_path,
    gru_15_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 14. Current 15-minute GRU progression
# ------------------------------------------------------------

gru_15min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                15,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 15-minute GRU final TEST progression:"
)


display(
    gru_15min_partial_summary
)


# ------------------------------------------------------------
# 15. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 15-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_15_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_15_f3_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_15_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_15_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_15_f3_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 15-minute F4."
)

Running GRU held-out TEST experiment:
  horizon: 15 minutes
  feature family: F3
  frozen epoch: 2
  architecture: RNN_C2
Epoch 1/2


2026-09-21 21:02:36.452419: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:02:36.452720: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.1737
Epoch 2/2
410/410 - 5s - 11ms/step - loss: 0.1412

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F3,30,17473,87261,17568,104829,26496,2,2,0.291426,0.612331,0.862387,10.310117,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


15-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.299681 kW
  F3 MAE: 0.291426 kW
  Relative MAE change: +2.7545%
  Positive means F3 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.158125,5.832924,5554,0



Current 15-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.332980,0.639402,0.849950
1,F1,4,0.278635,0.601394,0.867259
2,F2,7,0.299681,0.604346,0.865953
3,F3,2,0.291426,0.612331,0.862387



GRU 15-MIN F3 FINAL TEST: COMPLETE
Frozen epoch used: 2
Final epochs trained: 2
TEST MAE: 0.291426 kW
TEST RMSE: 0.612331 kW
TEST R²: 0.862387

GRU final TEST evaluations completed: 4 / 15
Remaining: 11
No prediction clipping was applied.

Next experiment: 15-minute F4.


In [86]:
# ============================================================
# Step 70: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 15 minutes
#       feature family = F4
#
#
# CENTRAL ABLATION
# ----------------
#
#       F3 = historical/Gregorian/solar
#            + continuous lunar / Sun-Moon astronomy
#
#       F4 = F3
#            + Panchang categorical representation
#
#
# Therefore:
#
#       F3 -> F4
#
# is the key test of whether the Panchang representation adds
# information beyond the underlying continuous astronomy.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       4
#
# Auxiliary width:
#       127
#
# Expected parameters:
#       20,577
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,261 + 17,568 = 104,829
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# The 15-minute F4 settings were frozen from validation before
# any GRU held-out TEST evaluation began.
#
# No tuning occurs after this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 15

GRU_TEST_FEATURE_FAMILY = "F4"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    4
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    127
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    20_577
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_261
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 3. F0-F3 must already exist.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            15,
            previous_family,
        )
        in
        gru_final_test_results
    )


# F4 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 15 minutes"
)

print(
    "  feature family: F4"
)

print(
    "  frozen epoch: 4"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_15_f4_result,
    gru_15_f4_history,
    gru_15_f4_actual_kw,
    gru_15_f4_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_15_f4_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_15_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_15_f4_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    gru_15_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


assert (
    gru_15_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    gru_15_f4_result[
        "trainable_parameter_count"
    ]
    ==
    20_577
)


assert (
    gru_15_f4_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    gru_15_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_15_f4_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    gru_15_f4_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_15_f4_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    gru_15_f4_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    gru_15_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_15_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_15_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 6. Final-refit history audit
#
# Exactly 4 epochs.
# No validation loss is allowed.
# ------------------------------------------------------------

assert (
    len(
        gru_15_f4_history
    )
    ==
    4
)


assert (
    gru_15_f4_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
    ]
)


assert (
    "loss"
    in
    gru_15_f4_history.columns
)


assert (
    "val_loss"
    not in
    gru_15_f4_history.columns
)


# ------------------------------------------------------------
# 7. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_15_f4_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_15_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_15_f4_actual_kw
).all()


assert np.isfinite(
    gru_15_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Exact same held-out population across F0-F4
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    np.testing.assert_array_equal(
        gru_15_f4_actual_kw,
        gru_final_test_actuals[
            (
                15,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 9. Independently recompute metrics
# ------------------------------------------------------------

gru_15_f4_metric_check = (
    compute_regression_metrics_kw(
        gru_15_f4_actual_kw,
        gru_15_f4_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_15_f4_metric_check[
        "MAE_kW"
    ],
    gru_15_f4_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f4_metric_check[
        "RMSE_kW"
    ],
    gru_15_f4_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_15_f4_metric_check[
        "R2"
    ],
    gru_15_f4_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 10. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_15_f4_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_15_f4_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_15_f4_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_15_f4_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    5
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    5
)


# ------------------------------------------------------------
# 11. Display F4 held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_15_f4_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_15_f4_history
)


# ------------------------------------------------------------
# 12. CENTRAL 15-minute F3 -> F4 TEST comparison
#
# Positive:
#
#       F4 has LOWER MAE than F3.
#
# Negative:
#
#       F4 has HIGHER MAE than F3.
#
# This is descriptive held-out TEST reporting only.
#
# Statistical stability will be assessed later using the same
# paired daily-block bootstrap protocol used for LSTM.
# ------------------------------------------------------------

gru_15_f3_mae = float(
    gru_final_test_results[
        (
            15,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_15_f4_mae = float(
    gru_15_f4_result[
        "test_MAE_kW"
    ]
)


gru_15_f3_to_f4_improvement_pct = (
    (
        gru_15_f3_mae
        -
        gru_15_f4_mae
    )
    /
    gru_15_f3_mae
    *
    100.0
)


print(
    "\n15-min held-out TEST F3 -> F4 comparison:"
)


print(
    "  F3 MAE:",
    f"{gru_15_f3_mae:.6f}",
    "kW",
)


print(
    "  F4 MAE:",
    f"{gru_15_f4_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_15_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "  Positive means F4 has lower TEST MAE."
)


print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 13. Unclipped prediction audit
# ------------------------------------------------------------

gru_15_f4_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_15_f4_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_15_f4_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_15_f4_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_15_f4_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_15_f4_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_15_f4_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_15_f4_prediction_audit
)


# ------------------------------------------------------------
# 14. Persist F4 result, history, and predictions
# ------------------------------------------------------------

gru_15_f4_result_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F4_result.csv"
)


gru_15_f4_history_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F4_history.csv"
)


gru_15_f4_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_15min_F4_predictions.csv"
)


pd.DataFrame(
    [
        gru_15_f4_result
    ]
).to_csv(
    gru_15_f4_result_path,
    index=False,
)


gru_15_f4_history.to_csv(
    gru_15_f4_history_path,
    index=False,
)


gru_15_f4_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_15_f4_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_15_f4_actual_kw,

        "predicted_P_Solar_kW":
            gru_15_f4_prediction_kw,
    }
)


gru_15_f4_predictions_table.to_csv(
    gru_15_f4_predictions_path,
    index=False,
)


for output_path in [
    gru_15_f4_result_path,
    gru_15_f4_history_path,
    gru_15_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Complete 15-minute GRU summary
# ------------------------------------------------------------

gru_final_15min_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                15,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "auxiliary_dim",
        "trainable_parameter_count",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


assert (
    gru_final_15min_summary.shape
    ==
    (
        5,
        7,
    )
)


print(
    "\nCOMPLETE 15-minute GRU final TEST progression:"
)


display(
    gru_final_15min_summary
)


# ------------------------------------------------------------
# 16. Persist complete 15-minute summary
# ------------------------------------------------------------

gru_final_15min_summary_path = (
    RESULTS_DIR
    /
    "gru_final_15min_summary.csv"
)


gru_final_15min_summary.to_csv(
    gru_final_15min_summary_path,
    index=False,
)


assert (
    gru_final_15min_summary_path.exists()
)


assert (
    gru_final_15min_summary_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 17. Completion checks
# ------------------------------------------------------------

for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    assert (
        (
            15,
            feature_family,
        )
        in
        gru_final_test_results
    )


assert (
    len(
        gru_final_test_results
    )
    ==
    5
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 15-MINUTE FINAL TEST SERIES: COMPLETE"
)

print(
    "============================================================"
)


print(
    "15-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "Frozen epoch used for F4:",
    gru_15_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained for F4:",
    gru_15_f4_result[
        "final_epochs_trained"
    ],
)


print(
    "15-min F3 -> F4 TEST MAE change:",
    f"{gru_15_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext horizon: 30 minutes."
)

Running GRU held-out TEST experiment:
  horizon: 15 minutes
  feature family: F4
  frozen epoch: 4
  architecture: RNN_C2
Epoch 1/4


2026-09-21 21:03:54.752139: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:03:54.753182: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.1722
Epoch 2/4
410/410 - 4s - 10ms/step - loss: 0.1411
Epoch 3/4
410/410 - 5s - 11ms/step - loss: 0.1441
Epoch 4/4
410/410 - 4s - 10ms/step - loss: 0.1528

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F4,127,20577,87261,17568,104829,26496,4,4,0.313546,0.614217,0.861538,19.080512,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


15-min held-out TEST F3 -> F4 comparison:
  F3 MAE: 0.291426 kW
  F4 MAE: 0.313546 kW
  Relative MAE change: -7.5900%
  Positive means F4 has lower TEST MAE.
  Descriptive TEST comparison only — no tuning.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.370052,5.924759,9491,0



COMPLETE 15-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,auxiliary_dim,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,1,16545,0.332980,0.639402,0.849950
1,F1,4,16,17025,0.278635,0.601394,0.867259
2,F2,7,21,17185,0.299681,0.604346,0.865953
3,F3,2,30,17473,0.291426,0.612331,0.862387
4,F4,4,127,20577,0.313546,0.614217,0.861538



GRU 15-MINUTE FINAL TEST SERIES: COMPLETE
15-min F0-F4 evaluations: 5 / 5 COMPLETE
Frozen epoch used for F4: 4
Final epochs trained for F4: 4
15-min F3 -> F4 TEST MAE change: -7.5900%

GRU final TEST evaluations completed: 5 / 15
Remaining: 10
No prediction clipping was applied.

Next horizon: 30 minutes.


In [87]:
# ============================================================
# Step 71: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 30 minutes
#       feature family = F0
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       4
#
# Auxiliary width:
#       1
#
# Expected parameters:
#       16,545
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,258 + 17,568 = 104,826
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# The complete 15-minute GRU series is already frozen and
# complete.
#
# This cell begins the independent 30-minute held-out series.
#
# No tuning occurs after observing TEST performance.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 30

GRU_TEST_FEATURE_FAMILY = "F0"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm previous GRU final state
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    5
)


for completed_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    assert (
        (
            15,
            completed_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    4
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    1
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16_545
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_258
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# This 30-minute experiment must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 30 minutes"
)

print(
    "  feature family: F0"
)

print(
    "  frozen epoch: 4"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_30_f0_result,
    gru_30_f0_history,
    gru_30_f0_actual_kw,
    gru_30_f0_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_30_f0_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_30_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_30_f0_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    gru_30_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


assert (
    gru_30_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    gru_30_f0_result[
        "trainable_parameter_count"
    ]
    ==
    16_545
)


assert (
    gru_30_f0_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    gru_30_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_30_f0_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    gru_30_f0_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_30_f0_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    gru_30_f0_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    gru_30_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_30_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_30_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 6. Final-refit history audit
#
# Exactly 4 epochs.
# No validation loss is allowed.
# ------------------------------------------------------------

assert (
    len(
        gru_30_f0_history
    )
    ==
    4
)


assert (
    gru_30_f0_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
    ]
)


assert (
    "loss"
    in
    gru_30_f0_history.columns
)


assert (
    "val_loss"
    not in
    gru_30_f0_history.columns
)


# ------------------------------------------------------------
# 7. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_30_f0_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_30_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_30_f0_actual_kw
).all()


assert np.isfinite(
    gru_30_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Independently recompute metrics
# ------------------------------------------------------------

gru_30_f0_metric_check = (
    compute_regression_metrics_kw(
        gru_30_f0_actual_kw,
        gru_30_f0_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_30_f0_metric_check[
        "MAE_kW"
    ],
    gru_30_f0_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f0_metric_check[
        "RMSE_kW"
    ],
    gru_30_f0_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f0_metric_check[
        "R2"
    ],
    gru_30_f0_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 9. Store in canonical GRU containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_30_f0_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_30_f0_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_30_f0_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_30_f0_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    6
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    6
)


# ------------------------------------------------------------
# 10. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_30_f0_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_30_f0_history
)


# ------------------------------------------------------------
# 11. Unclipped prediction audit
#
# Negative predictions remain unchanged.
# ------------------------------------------------------------

gru_30_f0_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_30_f0_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_30_f0_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_30_f0_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_30_f0_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_30_f0_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_30_f0_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_30_f0_prediction_audit
)


# ------------------------------------------------------------
# 12. Persist result
# ------------------------------------------------------------

gru_30_f0_result_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F0_result.csv"
)


pd.DataFrame(
    [
        gru_30_f0_result
    ]
).to_csv(
    gru_30_f0_result_path,
    index=False,
)


# ------------------------------------------------------------
# 13. Persist final-refit history
# ------------------------------------------------------------

gru_30_f0_history_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F0_history.csv"
)


gru_30_f0_history.to_csv(
    gru_30_f0_history_path,
    index=False,
)


# ------------------------------------------------------------
# 14. Persist row-level predictions
# ------------------------------------------------------------

gru_30_f0_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F0_predictions.csv"
)


gru_30_f0_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_30_f0_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_30_f0_actual_kw,

        "predicted_P_Solar_kW":
            gru_30_f0_prediction_kw,
    }
)


gru_30_f0_predictions_table.to_csv(
    gru_30_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 15. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    gru_30_f0_result_path,
    gru_30_f0_history_path,
    gru_30_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 16. Current GRU completion state
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 30-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_30_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_30_f0_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_30_f0_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_30_f0_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_30_f0_result['test_R2']:.6f}",
)


print(
    "\n15-min F0-F4: COMPLETE"
)

print(
    "30-min F0: COMPLETE"
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 30-minute F1."
)

Running GRU held-out TEST experiment:
  horizon: 30 minutes
  feature family: F0
  frozen epoch: 4
  architecture: RNN_C2
Epoch 1/4


2026-09-21 21:05:12.282176: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:05:12.282524: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 16ms/step - loss: 0.1868
Epoch 2/4
410/410 - 5s - 11ms/step - loss: 0.1684
Epoch 3/4
410/410 - 4s - 11ms/step - loss: 0.1674
Epoch 4/4
410/410 - 4s - 11ms/step - loss: 0.1664

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,30,F0,1,16545,87258,17568,104826,26496,4,4,0.39043,0.737658,0.800292,19.728615,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


,epoch,loss
0,1,0.186791
1,2,0.168379
2,3,0.167365
3,4,0.166407



Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.264354,6.152026,2050,0



GRU 30-MIN F0 FINAL TEST: COMPLETE
Frozen epoch used: 4
Final epochs trained: 4
TEST MAE: 0.390430 kW
TEST RMSE: 0.737658 kW
TEST R²: 0.800292

15-min F0-F4: COMPLETE
30-min F0: COMPLETE

GRU final TEST evaluations completed: 6 / 15
Remaining: 9
No prediction clipping was applied.

Next experiment: 30-minute F1.


In [88]:
# ============================================================
# Step 72: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 30 minutes
#       feature family = F1
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       11
#
# Auxiliary width:
#       16
#
# Expected parameters:
#       17,025
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,258 + 17,568 = 104,826
#
# Held-out TEST:
#       26,496
#
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 30

GRU_TEST_FEATURE_FAMILY = "F1"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    11
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_025
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_258
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# 30-minute F0 must already exist.
assert (
    (
        30,
        "F0",
    )
    in
    gru_final_test_results
)


# 30-minute F1 must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 30 minutes"
)

print(
    "  feature family: F1"
)

print(
    "  frozen epoch: 11"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 3. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_30_f1_result,
    gru_30_f1_history,
    gru_30_f1_actual_kw,
    gru_30_f1_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 4. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_30_f1_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_30_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_30_f1_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    gru_30_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


assert (
    gru_30_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    gru_30_f1_result[
        "trainable_parameter_count"
    ]
    ==
    17_025
)


assert (
    gru_30_f1_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    gru_30_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_30_f1_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    gru_30_f1_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_30_f1_result[
        "frozen_best_epoch"
    ]
    ==
    11
)


assert (
    gru_30_f1_result[
        "final_epochs_trained"
    ]
    ==
    11
)


assert (
    gru_30_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_30_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_30_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 5. Final-refit history audit
#
# Exactly 11 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_30_f1_history
    )
    ==
    11
)


assert (
    gru_30_f1_history[
        "epoch"
    ]
    .tolist()
    ==
    list(
        range(
            1,
            12,
        )
    )
)


assert (
    "loss"
    in
    gru_30_f1_history.columns
)


assert (
    "val_loss"
    not in
    gru_30_f1_history.columns
)


# ------------------------------------------------------------
# 6. Prediction / target checks
# ------------------------------------------------------------

assert (
    gru_30_f1_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_30_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_30_f1_actual_kw
).all()


assert np.isfinite(
    gru_30_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 7. Same-horizon target-population check
#
# 30-minute F0 and F1 must use exactly the same held-out
# target vector in exactly the same row order.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    gru_30_f1_actual_kw,
    gru_final_test_actuals[
        (
            30,
            "F0",
        )
    ],
)


# ------------------------------------------------------------
# 8. Independently recompute metrics
# ------------------------------------------------------------

gru_30_f1_metric_check = (
    compute_regression_metrics_kw(
        gru_30_f1_actual_kw,
        gru_30_f1_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_30_f1_metric_check[
        "MAE_kW"
    ],
    gru_30_f1_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f1_metric_check[
        "RMSE_kW"
    ],
    gru_30_f1_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f1_metric_check[
        "R2"
    ],
    gru_30_f1_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 9. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_30_f1_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_30_f1_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_30_f1_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_30_f1_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    7
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    7
)


# ------------------------------------------------------------
# 10. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_30_f1_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_30_f1_history
)


# ------------------------------------------------------------
# 11. Descriptive F0 -> F1 comparison
#
# Positive:
#       F1 has lower TEST MAE than F0.
#
# Negative:
#       F1 has higher TEST MAE than F0.
#
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_30_f0_mae = float(
    gru_final_test_results[
        (
            30,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_30_f1_mae = float(
    gru_30_f1_result[
        "test_MAE_kW"
    ]
)


gru_30_f0_to_f1_improvement_pct = (
    (
        gru_30_f0_mae
        -
        gru_30_f1_mae
    )
    /
    gru_30_f0_mae
    *
    100.0
)


print(
    "\n30-min held-out TEST F0 -> F1 comparison:"
)


print(
    "  F0 MAE:",
    f"{gru_30_f0_mae:.6f}",
    "kW",
)


print(
    "  F1 MAE:",
    f"{gru_30_f1_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_30_f0_to_f1_improvement_pct:+.4f}%",
)


print(
    "  Positive means F1 has lower TEST MAE."
)


# ------------------------------------------------------------
# 12. Unclipped prediction audit
# ------------------------------------------------------------

gru_30_f1_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_30_f1_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_30_f1_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_30_f1_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_30_f1_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_30_f1_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_30_f1_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_30_f1_prediction_audit
)


# ------------------------------------------------------------
# 13. Persist result, history, and row-level predictions
# ------------------------------------------------------------

gru_30_f1_result_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F1_result.csv"
)


gru_30_f1_history_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F1_history.csv"
)


gru_30_f1_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F1_predictions.csv"
)


pd.DataFrame(
    [
        gru_30_f1_result
    ]
).to_csv(
    gru_30_f1_result_path,
    index=False,
)


gru_30_f1_history.to_csv(
    gru_30_f1_history_path,
    index=False,
)


gru_30_f1_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_30_f1_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_30_f1_actual_kw,

        "predicted_P_Solar_kW":
            gru_30_f1_prediction_kw,
    }
)


gru_30_f1_predictions_table.to_csv(
    gru_30_f1_predictions_path,
    index=False,
)


for output_path in [
    gru_30_f1_result_path,
    gru_30_f1_history_path,
    gru_30_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 14. Current 30-minute GRU progression
# ------------------------------------------------------------

gru_30min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                30,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 30-minute GRU final TEST progression:"
)


display(
    gru_30min_partial_summary
)


# ------------------------------------------------------------
# 15. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 30-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_30_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_30_f1_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_30_f1_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_30_f1_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_30_f1_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 30-minute F2."
)

Running GRU held-out TEST experiment:
  horizon: 30 minutes
  feature family: F1
  frozen epoch: 11
  architecture: RNN_C2
Epoch 1/11


2026-09-21 21:06:45.344120: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:06:45.344430: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2126
Epoch 2/11
410/410 - 4s - 11ms/step - loss: 0.1691
Epoch 3/11
410/410 - 4s - 10ms/step - loss: 0.1718
Epoch 4/11
410/410 - 5s - 11ms/step - loss: 0.1723
Epoch 5/11
410/410 - 4s - 10ms/step - loss: 0.1751
Epoch 6/11
410/410 - 4s - 10ms/step - loss: 0.1731
Epoch 7/11
410/410 - 4s - 10ms/step - loss: 0.1764
Epoch 8/11
410/410 - 4s - 11ms/step - loss: 0.1791
Epoch 9/11
410/410 - 4s - 11ms/step - loss: 0.1751
Epoch 10/11
410/410 - 4s - 10ms/step - loss: 0.1740
Epoch 11/11
410/410 - 4s - 10ms/step - loss: 0.1770

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,30,F1,16,17025,87258,17568,104826,26496,11,11,0.364036,0.702368,0.818943,49.494695,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


30-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.390430 kW
  F1 MAE: 0.364036 kW
  Relative MAE change: +6.7601%
  Positive means F1 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.297612,5.995966,5691,0



Current 30-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,4,0.390430,0.737658,0.800292
1,F1,11,0.364036,0.702368,0.818943



GRU 30-MIN F1 FINAL TEST: COMPLETE
Frozen epoch used: 11
Final epochs trained: 11
TEST MAE: 0.364036 kW
TEST RMSE: 0.702368 kW
TEST R²: 0.818943

GRU final TEST evaluations completed: 7 / 15
Remaining: 8
No prediction clipping was applied.

Next experiment: 30-minute F2.


In [89]:
# ============================================================
# Step 73: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 30 minutes
#       feature family = F2
#
#
# F2 adds the independently computed solar-astronomy features
# to F1.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       2
#
# Auxiliary width:
#       21
#
# Expected parameters:
#       17,185
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,258 + 17,568 = 104,826
#
# Held-out TEST:
#       26,496
#
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 30

GRU_TEST_FEATURE_FAMILY = "F2"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current GRU completion state
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    7
)


assert (
    (
        30,
        "F0",
    )
    in
    gru_final_test_results
)


assert (
    (
        30,
        "F1",
    )
    in
    gru_final_test_results
)


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    2
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    21
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_185
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_258
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F2 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 30 minutes"
)

print(
    "  feature family: F2"
)

print(
    "  frozen epoch: 2"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_30_f2_result,
    gru_30_f2_history,
    gru_30_f2_actual_kw,
    gru_30_f2_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_30_f2_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_30_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_30_f2_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    gru_30_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


assert (
    gru_30_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    gru_30_f2_result[
        "trainable_parameter_count"
    ]
    ==
    17_185
)


assert (
    gru_30_f2_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    gru_30_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_30_f2_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    gru_30_f2_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_30_f2_result[
        "frozen_best_epoch"
    ]
    ==
    2
)


assert (
    gru_30_f2_result[
        "final_epochs_trained"
    ]
    ==
    2
)


assert (
    gru_30_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_30_f2_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_30_f2_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 6. Final-refit history audit
#
# Exactly 2 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_30_f2_history
    )
    ==
    2
)


assert (
    gru_30_f2_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
    ]
)


assert (
    "loss"
    in
    gru_30_f2_history.columns
)


assert (
    "val_loss"
    not in
    gru_30_f2_history.columns
)


# ------------------------------------------------------------
# 7. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_30_f2_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_30_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_30_f2_actual_kw
).all()


assert np.isfinite(
    gru_30_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Same-horizon target-population audit
#
# F0, F1 and F2 must all be evaluated on exactly the same
# 30-minute held-out target vector in the same row order.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
]:

    np.testing.assert_array_equal(
        gru_30_f2_actual_kw,
        gru_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 9. Independently recompute metrics
# ------------------------------------------------------------

gru_30_f2_metric_check = (
    compute_regression_metrics_kw(
        gru_30_f2_actual_kw,
        gru_30_f2_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_30_f2_metric_check[
        "MAE_kW"
    ],
    gru_30_f2_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f2_metric_check[
        "RMSE_kW"
    ],
    gru_30_f2_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f2_metric_check[
        "R2"
    ],
    gru_30_f2_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 10. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_30_f2_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_30_f2_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_30_f2_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_30_f2_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    8
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    8
)


# ------------------------------------------------------------
# 11. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_30_f2_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_30_f2_history
)


# ------------------------------------------------------------
# 12. Descriptive F1 -> F2 TEST comparison
#
# Positive:
#       F2 has lower TEST MAE than F1.
#
# Negative:
#       F2 has higher TEST MAE than F1.
#
# This remains descriptive only.
# ------------------------------------------------------------

gru_30_f1_mae = float(
    gru_final_test_results[
        (
            30,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_30_f2_mae = float(
    gru_30_f2_result[
        "test_MAE_kW"
    ]
)


gru_30_f1_to_f2_improvement_pct = (
    (
        gru_30_f1_mae
        -
        gru_30_f2_mae
    )
    /
    gru_30_f1_mae
    *
    100.0
)


print(
    "\n30-min held-out TEST F1 -> F2 comparison:"
)


print(
    "  F1 MAE:",
    f"{gru_30_f1_mae:.6f}",
    "kW",
)


print(
    "  F2 MAE:",
    f"{gru_30_f2_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_30_f1_to_f2_improvement_pct:+.4f}%",
)


print(
    "  Positive means F2 has lower TEST MAE."
)


# ------------------------------------------------------------
# 13. Unclipped prediction audit
# ------------------------------------------------------------

gru_30_f2_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_30_f2_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_30_f2_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_30_f2_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_30_f2_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_30_f2_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_30_f2_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_30_f2_prediction_audit
)


# ------------------------------------------------------------
# 14. Persist result, history, and row-level predictions
# ------------------------------------------------------------

gru_30_f2_result_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F2_result.csv"
)


gru_30_f2_history_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F2_history.csv"
)


gru_30_f2_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F2_predictions.csv"
)


pd.DataFrame(
    [
        gru_30_f2_result
    ]
).to_csv(
    gru_30_f2_result_path,
    index=False,
)


gru_30_f2_history.to_csv(
    gru_30_f2_history_path,
    index=False,
)


gru_30_f2_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_30_f2_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_30_f2_actual_kw,

        "predicted_P_Solar_kW":
            gru_30_f2_prediction_kw,
    }
)


gru_30_f2_predictions_table.to_csv(
    gru_30_f2_predictions_path,
    index=False,
)


for output_path in [
    gru_30_f2_result_path,
    gru_30_f2_history_path,
    gru_30_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Current 30-minute progression
# ------------------------------------------------------------

gru_30min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                30,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 30-minute GRU final TEST progression:"
)


display(
    gru_30min_partial_summary
)


# ------------------------------------------------------------
# 16. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 30-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_30_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_30_f2_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_30_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_30_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_30_f2_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 30-minute F3."
)

Running GRU held-out TEST experiment:
  horizon: 30 minutes
  feature family: F2
  frozen epoch: 2
  architecture: RNN_C2
Epoch 1/2


2026-09-21 21:08:33.027045: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:08:33.027337: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 7s - 17ms/step - loss: 0.2132
Epoch 2/2
410/410 - 5s - 11ms/step - loss: 0.1763

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,30,F2,21,17185,87258,17568,104826,26496,2,2,0.396951,0.72397,0.807634,11.431251,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


30-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.364036 kW
  F2 MAE: 0.396951 kW
  Relative MAE change: -9.0415%
  Positive means F2 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.435637,6.056424,9925,0



Current 30-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,4,0.390430,0.737658,0.800292
1,F1,11,0.364036,0.702368,0.818943
2,F2,2,0.396951,0.723970,0.807634



GRU 30-MIN F2 FINAL TEST: COMPLETE
Frozen epoch used: 2
Final epochs trained: 2
TEST MAE: 0.396951 kW
TEST RMSE: 0.723970 kW
TEST R²: 0.807634

GRU final TEST evaluations completed: 8 / 15
Remaining: 7
No prediction clipping was applied.

Next experiment: 30-minute F3.


In [90]:
# ============================================================
# Step 74: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 30 minutes
#       feature family = F3
#
#
# F3 adds continuous lunar / Sun-Moon astronomy to F2.
#
# This is the direct control model for the later F3 -> F4
# Panchang comparison.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       5
#
# Auxiliary width:
#       30
#
# Expected parameters:
#       17,473
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,258 + 17,568 = 104,826
#
# Held-out TEST:
#       26,496
#
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 30

GRU_TEST_FEATURE_FAMILY = "F3"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm previous 30-minute experiments
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    8
)


for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            30,
            previous_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    5
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    30
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_473
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_258
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F3 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 30 minutes"
)

print(
    "  feature family: F3"
)

print(
    "  frozen epoch: 5"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_30_f3_result,
    gru_30_f3_history,
    gru_30_f3_actual_kw,
    gru_30_f3_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_30_f3_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_30_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_30_f3_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    gru_30_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


assert (
    gru_30_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    gru_30_f3_result[
        "trainable_parameter_count"
    ]
    ==
    17_473
)


assert (
    gru_30_f3_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    gru_30_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_30_f3_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    gru_30_f3_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_30_f3_result[
        "frozen_best_epoch"
    ]
    ==
    5
)


assert (
    gru_30_f3_result[
        "final_epochs_trained"
    ]
    ==
    5
)


assert (
    gru_30_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_30_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_30_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 6. Final-refit history audit
#
# Exactly 5 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_30_f3_history
    )
    ==
    5
)


assert (
    gru_30_f3_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
        5,
    ]
)


assert (
    "loss"
    in
    gru_30_f3_history.columns
)


assert (
    "val_loss"
    not in
    gru_30_f3_history.columns
)


# ------------------------------------------------------------
# 7. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_30_f3_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_30_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_30_f3_actual_kw
).all()


assert np.isfinite(
    gru_30_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Same-horizon target-population audit
#
# F0-F3 must all use the same 30-minute held-out target vector
# in exactly the same row order.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    np.testing.assert_array_equal(
        gru_30_f3_actual_kw,
        gru_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 9. Independently recompute metrics
# ------------------------------------------------------------

gru_30_f3_metric_check = (
    compute_regression_metrics_kw(
        gru_30_f3_actual_kw,
        gru_30_f3_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_30_f3_metric_check[
        "MAE_kW"
    ],
    gru_30_f3_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f3_metric_check[
        "RMSE_kW"
    ],
    gru_30_f3_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f3_metric_check[
        "R2"
    ],
    gru_30_f3_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 10. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_30_f3_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_30_f3_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_30_f3_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_30_f3_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    9
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    9
)


# ------------------------------------------------------------
# 11. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_30_f3_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_30_f3_history
)


# ------------------------------------------------------------
# 12. Descriptive F2 -> F3 TEST comparison
#
# Positive:
#       F3 has lower TEST MAE than F2.
#
# Negative:
#       F3 has higher TEST MAE than F2.
#
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_30_f2_mae = float(
    gru_final_test_results[
        (
            30,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_30_f3_mae = float(
    gru_30_f3_result[
        "test_MAE_kW"
    ]
)


gru_30_f2_to_f3_improvement_pct = (
    (
        gru_30_f2_mae
        -
        gru_30_f3_mae
    )
    /
    gru_30_f2_mae
    *
    100.0
)


print(
    "\n30-min held-out TEST F2 -> F3 comparison:"
)


print(
    "  F2 MAE:",
    f"{gru_30_f2_mae:.6f}",
    "kW",
)


print(
    "  F3 MAE:",
    f"{gru_30_f3_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_30_f2_to_f3_improvement_pct:+.4f}%",
)


print(
    "  Positive means F3 has lower TEST MAE."
)


# ------------------------------------------------------------
# 13. Unclipped prediction audit
# ------------------------------------------------------------

gru_30_f3_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_30_f3_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_30_f3_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_30_f3_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_30_f3_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_30_f3_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_30_f3_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_30_f3_prediction_audit
)


# ------------------------------------------------------------
# 14. Persist result, history, and row-level predictions
# ------------------------------------------------------------

gru_30_f3_result_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F3_result.csv"
)


gru_30_f3_history_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F3_history.csv"
)


gru_30_f3_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F3_predictions.csv"
)


pd.DataFrame(
    [
        gru_30_f3_result
    ]
).to_csv(
    gru_30_f3_result_path,
    index=False,
)


gru_30_f3_history.to_csv(
    gru_30_f3_history_path,
    index=False,
)


gru_30_f3_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_30_f3_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_30_f3_actual_kw,

        "predicted_P_Solar_kW":
            gru_30_f3_prediction_kw,
    }
)


gru_30_f3_predictions_table.to_csv(
    gru_30_f3_predictions_path,
    index=False,
)


for output_path in [
    gru_30_f3_result_path,
    gru_30_f3_history_path,
    gru_30_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Current 30-minute GRU progression
# ------------------------------------------------------------

gru_30min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                30,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 30-minute GRU final TEST progression:"
)


display(
    gru_30min_partial_summary
)


# ------------------------------------------------------------
# 16. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 30-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_30_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_30_f3_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_30_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_30_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_30_f3_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)

print(
    "\nNext experiment: 30-minute F4."
)

Running GRU held-out TEST experiment:
  horizon: 30 minutes
  feature family: F3
  frozen epoch: 5
  architecture: RNN_C2
Epoch 1/5


2026-09-21 21:09:57.642026: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:09:57.642297: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 14ms/step - loss: 0.2111
Epoch 2/5
410/410 - 4s - 11ms/step - loss: 0.1820
Epoch 3/5
410/410 - 4s - 10ms/step - loss: 0.1834
Epoch 4/5
410/410 - 4s - 10ms/step - loss: 0.1902
Epoch 5/5
410/410 - 4s - 10ms/step - loss: 0.1916

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,30,F3,30,17473,87258,17568,104826,26496,5,5,0.391312,0.7271,0.805967,22.56205,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


30-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.396951 kW
  F3 MAE: 0.391312 kW
  Relative MAE change: +1.4206%
  Positive means F3 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.663954,5.880756,7130,0



Current 30-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,4,0.390430,0.737658,0.800292
1,F1,11,0.364036,0.702368,0.818943
2,F2,2,0.396951,0.723970,0.807634
3,F3,5,0.391312,0.727100,0.805967



GRU 30-MIN F3 FINAL TEST: COMPLETE
Frozen epoch used: 5
Final epochs trained: 5
TEST MAE: 0.391312 kW
TEST RMSE: 0.727100 kW
TEST R²: 0.805967

GRU final TEST evaluations completed: 9 / 15
Remaining: 6
No prediction clipping was applied.

Next experiment: 30-minute F4.


In [91]:
# ============================================================
# Step 75: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 30 minutes
#       feature family = F4
#
#
# CENTRAL ABLATION
# ----------------
#
#       F3 = F2
#            + continuous lunar / Sun-Moon astronomy
#
#       F4 = F3
#            + Panchang categorical representation
#
#
# Therefore:
#
#       F3 -> F4
#
# directly tests whether Panchang representation adds
# predictive information beyond the continuous astronomical
# controls already present in F3.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       4
#
# Auxiliary width:
#       127
#
# Expected parameters:
#       20,577
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,258 + 17,568 = 104,826
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# No tuning occurs after observing this result.
#
# Statistical stability of F3 -> F4 will be examined later
# using the paired UTC daily-block bootstrap.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 30

GRU_TEST_FEATURE_FAMILY = "F4"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current GRU completion state
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    9
)


for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            30,
            previous_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    4
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    127
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    20_577
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_258
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F4 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 30 minutes"
)

print(
    "  feature family: F4"
)

print(
    "  frozen epoch: 4"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_30_f4_result,
    gru_30_f4_history,
    gru_30_f4_actual_kw,
    gru_30_f4_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result audit
# ------------------------------------------------------------

assert (
    gru_30_f4_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_30_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_30_f4_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    gru_30_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


assert (
    gru_30_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    gru_30_f4_result[
        "trainable_parameter_count"
    ]
    ==
    20_577
)


assert (
    gru_30_f4_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    gru_30_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_30_f4_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    gru_30_f4_result[
        "n_test"
    ]
    ==
    26_496
)


assert (
    gru_30_f4_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    gru_30_f4_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    gru_30_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_30_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_30_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 6. Final-refit history audit
#
# Exactly 4 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_30_f4_history
    )
    ==
    4
)


assert (
    gru_30_f4_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
    ]
)


assert (
    "loss"
    in
    gru_30_f4_history.columns
)


assert (
    "val_loss"
    not in
    gru_30_f4_history.columns
)


# ------------------------------------------------------------
# 7. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_30_f4_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_30_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_30_f4_actual_kw
).all()


assert np.isfinite(
    gru_30_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 8. Exact same held-out population across 30-min F0-F4
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    np.testing.assert_array_equal(
        gru_30_f4_actual_kw,
        gru_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 9. Independently recompute held-out metrics
# ------------------------------------------------------------

gru_30_f4_metric_check = (
    compute_regression_metrics_kw(
        gru_30_f4_actual_kw,
        gru_30_f4_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_30_f4_metric_check[
        "MAE_kW"
    ],
    gru_30_f4_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f4_metric_check[
        "RMSE_kW"
    ],
    gru_30_f4_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_30_f4_metric_check[
        "R2"
    ],
    gru_30_f4_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 10. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_30_f4_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_30_f4_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_30_f4_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_30_f4_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    10
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    10
)


# ------------------------------------------------------------
# 11. Display F4 held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_30_f4_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_30_f4_history
)


# ------------------------------------------------------------
# 12. CENTRAL 30-minute F3 -> F4 comparison
#
# Positive:
#
#       F4 has LOWER TEST MAE than F3.
#
# Negative:
#
#       F4 has HIGHER TEST MAE than F3.
#
# This is descriptive held-out TEST reporting only.
# ------------------------------------------------------------

gru_30_f3_mae = float(
    gru_final_test_results[
        (
            30,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_30_f4_mae = float(
    gru_30_f4_result[
        "test_MAE_kW"
    ]
)


gru_30_f3_to_f4_improvement_pct = (
    (
        gru_30_f3_mae
        -
        gru_30_f4_mae
    )
    /
    gru_30_f3_mae
    *
    100.0
)


print(
    "\n30-min held-out TEST F3 -> F4 comparison:"
)


print(
    "  F3 MAE:",
    f"{gru_30_f3_mae:.6f}",
    "kW",
)


print(
    "  F4 MAE:",
    f"{gru_30_f4_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_30_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "  Positive means F4 has lower TEST MAE."
)


print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 13. Unclipped prediction audit
# ------------------------------------------------------------

gru_30_f4_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_30_f4_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_30_f4_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_30_f4_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_30_f4_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_30_f4_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_30_f4_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_30_f4_prediction_audit
)


# ------------------------------------------------------------
# 14. Persist result, history, and row-level predictions
# ------------------------------------------------------------

gru_30_f4_result_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F4_result.csv"
)


gru_30_f4_history_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F4_history.csv"
)


gru_30_f4_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_30min_F4_predictions.csv"
)


pd.DataFrame(
    [
        gru_30_f4_result
    ]
).to_csv(
    gru_30_f4_result_path,
    index=False,
)


gru_30_f4_history.to_csv(
    gru_30_f4_history_path,
    index=False,
)


gru_30_f4_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_30_f4_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_30_f4_actual_kw,

        "predicted_P_Solar_kW":
            gru_30_f4_prediction_kw,
    }
)


gru_30_f4_predictions_table.to_csv(
    gru_30_f4_predictions_path,
    index=False,
)


for output_path in [
    gru_30_f4_result_path,
    gru_30_f4_history_path,
    gru_30_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 15. Complete 30-minute GRU summary
# ------------------------------------------------------------

gru_final_30min_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                30,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "auxiliary_dim",
        "trainable_parameter_count",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


assert (
    gru_final_30min_summary.shape
    ==
    (
        5,
        7,
    )
)


print(
    "\nCOMPLETE 30-minute GRU final TEST progression:"
)


display(
    gru_final_30min_summary
)


# ------------------------------------------------------------
# 16. Show current F3 -> F4 GRU comparison across completed
#     horizons only: 15 and 30 minutes.
# ------------------------------------------------------------

gru_completed_f3_f4_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                15,

            "F3_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            15,
                            "F3",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F4_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            15,
                            "F4",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),
        },

        {
            "horizon_minutes":
                30,

            "F3_test_MAE_kW":
                gru_30_f3_mae,

            "F4_test_MAE_kW":
                gru_30_f4_mae,
        },
    ]
)


gru_completed_f3_f4_summary[
    "F3_to_F4_test_MAE_improvement_pct"
] = (
    (
        gru_completed_f3_f4_summary[
            "F3_test_MAE_kW"
        ]
        -
        gru_completed_f3_f4_summary[
            "F4_test_MAE_kW"
        ]
    )
    /
    gru_completed_f3_f4_summary[
        "F3_test_MAE_kW"
    ]
    *
    100.0
)


print(
    "\nGRU F3 -> F4 held-out comparison so far:"
)


display(
    gru_completed_f3_f4_summary
)


# ------------------------------------------------------------
# 17. Persist complete 30-minute summary
# ------------------------------------------------------------

gru_final_30min_summary_path = (
    RESULTS_DIR
    /
    "gru_final_30min_summary.csv"
)


gru_final_30min_summary.to_csv(
    gru_final_30min_summary_path,
    index=False,
)


assert (
    gru_final_30min_summary_path.exists()
)


assert (
    gru_final_30min_summary_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 18. Completion audit
# ------------------------------------------------------------

for feature_family in [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]:

    assert (
        (
            30,
            feature_family,
        )
        in
        gru_final_test_results
    )


assert (
    len(
        gru_final_test_results
    )
    ==
    10
)


# ------------------------------------------------------------
# 19. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 30-MINUTE FINAL TEST SERIES: COMPLETE"
)

print(
    "============================================================"
)


print(
    "15-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "30-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "Frozen epoch used for 30-min F4:",
    gru_30_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained for 30-min F4:",
    gru_30_f4_result[
        "final_epochs_trained"
    ],
)


print(
    "30-min F3 -> F4 TEST MAE change:",
    f"{gru_30_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext horizon: 60 minutes."
)

Running GRU held-out TEST experiment:
  horizon: 30 minutes
  feature family: F4
  frozen epoch: 4
  architecture: RNN_C2
Epoch 1/4


2026-09-21 21:11:47.867655: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:11:47.867968: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2085
Epoch 2/4
410/410 - 4s - 11ms/step - loss: 0.1762
Epoch 3/4
410/410 - 4s - 10ms/step - loss: 0.1880
Epoch 4/4
410/410 - 4s - 10ms/step - loss: 0.1868

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,30,F4,127,20577,87258,17568,104826,26496,4,4,0.412697,0.751714,0.792608,18.426495,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


COMPLETE 30-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,auxiliary_dim,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,4,1,16545,0.390430,0.737658,0.800292
1,F1,11,16,17025,0.364036,0.702368,0.818943
2,F2,2,21,17185,0.396951,0.723970,0.807634
3,F3,5,30,17473,0.391312,0.727100,0.805967
4,F4,4,127,20577,0.412697,0.751714,0.792608



GRU F3 -> F4 held-out comparison so far:


,horizon_minutes,F3_test_MAE_kW,F4_test_MAE_kW,F3_to_F4_test_MAE_improvement_pct
0,15,0.291426,0.313546,-7.590039
1,30,0.391312,0.412697,-5.464907



GRU 30-MINUTE FINAL TEST SERIES: COMPLETE
15-min F0-F4 evaluations: 5 / 5 COMPLETE
30-min F0-F4 evaluations: 5 / 5 COMPLETE
Frozen epoch used for 30-min F4: 4
Final epochs trained for 30-min F4: 4
30-min F3 -> F4 TEST MAE change: -5.4649%

GRU final TEST evaluations completed: 10 / 15
Remaining: 5
No prediction clipping was applied.

Next horizon: 60 minutes.


In [92]:
# ============================================================
# Step 76: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 60 minutes
#       feature family = F0
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       3
#
# Auxiliary width:
#       1
#
# Expected parameters:
#       16,545
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,252 + 17,568 = 104,820
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# 15-minute series: COMPLETE
# 30-minute series: COMPLETE
#
# This cell begins the final 60-minute GRU series.
#
# No tuning occurs after observing TEST performance.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 60

GRU_TEST_FEATURE_FAMILY = "F0"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current GRU completion state
#
# Exactly 10 final TEST experiments should already exist:
#
#       15-min F0-F4
#       30-min F0-F4
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    10
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    10
)


for completed_horizon in [
    15,
    30,
]:

    for completed_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        assert (
            (
                completed_horizon,
                completed_family,
            )
            in
            gru_final_test_results
        )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    3
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    1
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16_545
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_252
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# 60-minute F0 must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F0"
)

print(
    "  frozen epoch: 3"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final TRAIN + VALIDATION refit
#    followed by held-out TEST prediction
# ------------------------------------------------------------

(
    gru_60_f0_result,
    gru_60_f0_history,
    gru_60_f0_actual_kw,
    gru_60_f0_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result identity audit
# ------------------------------------------------------------

assert (
    gru_60_f0_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_60_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_60_f0_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_60_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


# ------------------------------------------------------------
# 6. Architecture / population audit
# ------------------------------------------------------------

assert (
    gru_60_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    gru_60_f0_result[
        "trainable_parameter_count"
    ]
    ==
    16_545
)


assert (
    gru_60_f0_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_60_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_60_f0_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    gru_60_f0_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen-epoch / final-refit protocol audit
# ------------------------------------------------------------

assert (
    gru_60_f0_result[
        "frozen_best_epoch"
    ]
    ==
    3
)


assert (
    gru_60_f0_result[
        "final_epochs_trained"
    ]
    ==
    3
)


assert (
    gru_60_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_60_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_60_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Final fitting-history audit
#
# Exactly 3 epochs must have been trained.
#
# There must be NO val_loss because validation was merged with
# training for the final refit and no validation_data was
# supplied.
# ------------------------------------------------------------

assert (
    len(
        gru_60_f0_history
    )
    ==
    3
)


assert (
    gru_60_f0_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
    ]
)


assert (
    "loss"
    in
    gru_60_f0_history.columns
)


assert (
    "val_loss"
    not in
    gru_60_f0_history.columns
)


# ------------------------------------------------------------
# 9. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_60_f0_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_60_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_60_f0_actual_kw
).all()


assert np.isfinite(
    gru_60_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Independently recompute held-out metrics
#
# This verifies that the helper's stored metrics correspond
# exactly to the returned prediction vector.
# ------------------------------------------------------------

gru_60_f0_metric_check = (
    compute_regression_metrics_kw(
        gru_60_f0_actual_kw,
        gru_60_f0_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_60_f0_metric_check[
        "MAE_kW"
    ],
    gru_60_f0_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f0_metric_check[
        "RMSE_kW"
    ],
    gru_60_f0_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f0_metric_check[
        "R2"
    ],
    gru_60_f0_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 11. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_60_f0_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_60_f0_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_60_f0_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_60_f0_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    11
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    11
)


# ------------------------------------------------------------
# 12. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_60_f0_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_60_f0_history
)


# ------------------------------------------------------------
# 13. Unclipped prediction audit
#
# Negative predictions are retained exactly as produced.
# No physical clipping is applied.
# ------------------------------------------------------------

gru_60_f0_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_60_f0_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_60_f0_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_60_f0_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_60_f0_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_60_f0_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_60_f0_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_60_f0_prediction_audit
)


# ------------------------------------------------------------
# 14. Persist result
# ------------------------------------------------------------

gru_60_f0_result_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F0_result.csv"
)


pd.DataFrame(
    [
        gru_60_f0_result
    ]
).to_csv(
    gru_60_f0_result_path,
    index=False,
)


# ------------------------------------------------------------
# 15. Persist final-refit history
# ------------------------------------------------------------

gru_60_f0_history_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F0_history.csv"
)


gru_60_f0_history.to_csv(
    gru_60_f0_history_path,
    index=False,
)


# ------------------------------------------------------------
# 16. Persist row-level held-out predictions
# ------------------------------------------------------------

gru_60_f0_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F0_predictions.csv"
)


gru_60_f0_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_60_f0_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_60_f0_actual_kw,

        "predicted_P_Solar_kW":
            gru_60_f0_prediction_kw,
    }
)


gru_60_f0_predictions_table.to_csv(
    gru_60_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    gru_60_f0_result_path,
    gru_60_f0_history_path,
    gru_60_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )


    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Current completion state
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 60-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_60_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_60_f0_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_60_f0_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_60_f0_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_60_f0_result['test_R2']:.6f}",
)


print(
    "\n15-min F0-F4: COMPLETE"
)

print(
    "30-min F0-F4: COMPLETE"
)

print(
    "60-min F0: COMPLETE"
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext experiment: 60-minute F1."
)

Running GRU held-out TEST experiment:
  horizon: 60 minutes
  feature family: F0
  frozen epoch: 3
  architecture: RNN_C2
Epoch 1/3


2026-09-21 21:13:05.810121: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:13:05.810507: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2434
Epoch 2/3
410/410 - 5s - 11ms/step - loss: 0.2245
Epoch 3/3
410/410 - 4s - 11ms/step - loss: 0.2334

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,60,F0,1,16545,87252,17568,104820,26496,3,3,0.503131,0.891575,0.708256,15.477623,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


,epoch,loss
0,1,0.243450
1,2,0.224515
2,3,0.233358



Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.422542,6.13056,7309,0



GRU 60-MIN F0 FINAL TEST: COMPLETE
Frozen epoch used: 3
Final epochs trained: 3
TEST MAE: 0.503131 kW
TEST RMSE: 0.891575 kW
TEST R²: 0.708256

15-min F0-F4: COMPLETE
30-min F0-F4: COMPLETE
60-min F0: COMPLETE

GRU final TEST evaluations completed: 11 / 15
Remaining: 4
No prediction clipping was applied.

Next experiment: 60-minute F1.


In [93]:
# ============================================================
# Step 77: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 60 minutes
#       feature family = F1
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       13
#
# Auxiliary width:
#       16
#
# Expected parameters:
#       17,025
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,252 + 17,568 = 104,820
#
# Held-out TEST:
#       26,496
#
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 60

GRU_TEST_FEATURE_FAMILY = "F1"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current completion state
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    11
)


assert (
    (
        60,
        "F0",
    )
    in
    gru_final_test_results
)


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    13
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    16
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_025
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_252
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F1 must not already exist.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F1"
)

print(
    "  frozen epoch: 13"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final refit + held-out TEST
# ------------------------------------------------------------

(
    gru_60_f1_result,
    gru_60_f1_history,
    gru_60_f1_actual_kw,
    gru_60_f1_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result identity audit
# ------------------------------------------------------------

assert (
    gru_60_f1_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_60_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_60_f1_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_60_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


# ------------------------------------------------------------
# 6. Architecture / population audit
# ------------------------------------------------------------

assert (
    gru_60_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    gru_60_f1_result[
        "trainable_parameter_count"
    ]
    ==
    17_025
)


assert (
    gru_60_f1_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_60_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_60_f1_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    gru_60_f1_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen-epoch / refit-protocol audit
# ------------------------------------------------------------

assert (
    gru_60_f1_result[
        "frozen_best_epoch"
    ]
    ==
    13
)


assert (
    gru_60_f1_result[
        "final_epochs_trained"
    ]
    ==
    13
)


assert (
    gru_60_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_60_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_60_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Final-refit history audit
#
# Exactly 13 epochs.
# No validation loss is permitted.
# ------------------------------------------------------------

assert (
    len(
        gru_60_f1_history
    )
    ==
    13
)


assert (
    gru_60_f1_history[
        "epoch"
    ]
    .tolist()
    ==
    list(
        range(
            1,
            14,
        )
    )
)


assert (
    "loss"
    in
    gru_60_f1_history.columns
)


assert (
    "val_loss"
    not in
    gru_60_f1_history.columns
)


# ------------------------------------------------------------
# 9. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_60_f1_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_60_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_60_f1_actual_kw
).all()


assert np.isfinite(
    gru_60_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Same-horizon held-out population check
#
# F0 and F1 must use exactly the same target vector in exactly
# the same row order.
# ------------------------------------------------------------

np.testing.assert_array_equal(
    gru_60_f1_actual_kw,
    gru_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
)


# ------------------------------------------------------------
# 11. Independently recompute metrics
# ------------------------------------------------------------

gru_60_f1_metric_check = (
    compute_regression_metrics_kw(
        gru_60_f1_actual_kw,
        gru_60_f1_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_60_f1_metric_check[
        "MAE_kW"
    ],
    gru_60_f1_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f1_metric_check[
        "RMSE_kW"
    ],
    gru_60_f1_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f1_metric_check[
        "R2"
    ],
    gru_60_f1_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 12. Store in canonical GRU final containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_60_f1_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_60_f1_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_60_f1_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_60_f1_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    12
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    12
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_60_f1_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_60_f1_history
)


# ------------------------------------------------------------
# 14. Descriptive F0 -> F1 TEST comparison
#
# Positive:
#       F1 has lower MAE than F0.
#
# Negative:
#       F1 has higher MAE than F0.
#
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_60_f0_mae = float(
    gru_final_test_results[
        (
            60,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_60_f1_mae = float(
    gru_60_f1_result[
        "test_MAE_kW"
    ]
)


gru_60_f0_to_f1_improvement_pct = (
    (
        gru_60_f0_mae
        -
        gru_60_f1_mae
    )
    /
    gru_60_f0_mae
    *
    100.0
)


print(
    "\n60-min held-out TEST F0 -> F1 comparison:"
)


print(
    "  F0 MAE:",
    f"{gru_60_f0_mae:.6f}",
    "kW",
)


print(
    "  F1 MAE:",
    f"{gru_60_f1_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_60_f0_to_f1_improvement_pct:+.4f}%",
)


print(
    "  Positive means F1 has lower TEST MAE."
)


# ------------------------------------------------------------
# 15. Unclipped prediction audit
# ------------------------------------------------------------

gru_60_f1_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_60_f1_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_60_f1_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_60_f1_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_60_f1_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_60_f1_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_60_f1_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_60_f1_prediction_audit
)


# ------------------------------------------------------------
# 16. Persist result / history / row-level predictions
# ------------------------------------------------------------

gru_60_f1_result_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F1_result.csv"
)


gru_60_f1_history_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F1_history.csv"
)


gru_60_f1_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F1_predictions.csv"
)


pd.DataFrame(
    [
        gru_60_f1_result
    ]
).to_csv(
    gru_60_f1_result_path,
    index=False,
)


gru_60_f1_history.to_csv(
    gru_60_f1_history_path,
    index=False,
)


gru_60_f1_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_60_f1_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_60_f1_actual_kw,

        "predicted_P_Solar_kW":
            gru_60_f1_prediction_kw,
    }
)


gru_60_f1_predictions_table.to_csv(
    gru_60_f1_predictions_path,
    index=False,
)


for output_path in [
    gru_60_f1_result_path,
    gru_60_f1_history_path,
    gru_60_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Current 60-minute progression
# ------------------------------------------------------------

gru_60min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                60,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 60-minute GRU final TEST progression:"
)


display(
    gru_60min_partial_summary
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 60-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_60_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_60_f1_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_60_f1_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_60_f1_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_60_f1_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext experiment: 60-minute F2."
)

Running GRU held-out TEST experiment:
  horizon: 60 minutes
  feature family: F1
  frozen epoch: 13
  architecture: RNN_C2
Epoch 1/13


2026-09-21 21:14:20.539640: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:14:20.539911: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2684
Epoch 2/13
410/410 - 5s - 11ms/step - loss: 0.2202
Epoch 3/13
410/410 - 4s - 10ms/step - loss: 0.2169
Epoch 4/13
410/410 - 4s - 10ms/step - loss: 0.2210
Epoch 5/13
410/410 - 4s - 10ms/step - loss: 0.2191
Epoch 6/13
410/410 - 4s - 10ms/step - loss: 0.2190
Epoch 7/13
410/410 - 4s - 10ms/step - loss: 0.2188
Epoch 8/13
410/410 - 4s - 10ms/step - loss: 0.2225
Epoch 9/13
410/410 - 4s - 10ms/step - loss: 0.2191
Epoch 10/13
410/410 - 4s - 10ms/step - loss: 0.2183
Epoch 11/13
410/410 - 4s - 10ms/step - loss: 0.2191
Epoch 12/13
410/410 - 4s - 10ms/step - loss: 0.2206
Epoch 13/13
410/410 - 4s - 10ms/step - loss: 0.2167

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,60,F1,16,17025,87252,17568,104820,26496,13,13,0.509348,0.885489,0.712225,57.225346,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


60-min held-out TEST F0 -> F1 comparison:
  F0 MAE: 0.503131 kW
  F1 MAE: 0.509348 kW
  Relative MAE change: -1.2357%
  Positive means F1 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.56578,5.994343,4503,0



Current 60-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.503131,0.891575,0.708256
1,F1,13,0.509348,0.885489,0.712225



GRU 60-MIN F1 FINAL TEST: COMPLETE
Frozen epoch used: 13
Final epochs trained: 13
TEST MAE: 0.509348 kW
TEST RMSE: 0.885489 kW
TEST R²: 0.712225

GRU final TEST evaluations completed: 12 / 15
Remaining: 3
No prediction clipping was applied.

Next experiment: 60-minute F2.


In [94]:
# ============================================================
# Step 78: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 60 minutes
#       feature family = F2
#
#
# F2 extends F1 with deterministic solar-astronomy features.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       3
#
# Auxiliary width:
#       21
#
# Expected parameters:
#       17,185
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,252 + 17,568 = 104,820
#
# Held-out TEST:
#       26,496
#
#
# No tuning occurs after observing this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 60

GRU_TEST_FEATURE_FAMILY = "F2"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current GRU completion state
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    12
)


for previous_family in [
    "F0",
    "F1",
]:

    assert (
        (
            60,
            previous_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    3
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    21
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_185
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_252
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F2 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F2"
)

print(
    "  frozen epoch: 3"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final TRAIN + VALIDATION refit
#    followed by held-out TEST evaluation
# ------------------------------------------------------------

(
    gru_60_f2_result,
    gru_60_f2_history,
    gru_60_f2_actual_kw,
    gru_60_f2_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result identity audit
# ------------------------------------------------------------

assert (
    gru_60_f2_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_60_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_60_f2_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_60_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


# ------------------------------------------------------------
# 6. Architecture / population audit
# ------------------------------------------------------------

assert (
    gru_60_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    gru_60_f2_result[
        "trainable_parameter_count"
    ]
    ==
    17_185
)


assert (
    gru_60_f2_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_60_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_60_f2_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    gru_60_f2_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen-epoch / final-refit protocol audit
# ------------------------------------------------------------

assert (
    gru_60_f2_result[
        "frozen_best_epoch"
    ]
    ==
    3
)


assert (
    gru_60_f2_result[
        "final_epochs_trained"
    ]
    ==
    3
)


assert (
    gru_60_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_60_f2_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_60_f2_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Final-refit history audit
#
# Exactly 3 epochs.
#
# There must be no validation loss because TRAIN and
# VALIDATION were merged for the final fit and no validation
# data were supplied during refitting.
# ------------------------------------------------------------

assert (
    len(
        gru_60_f2_history
    )
    ==
    3
)


assert (
    gru_60_f2_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
    ]
)


assert (
    "loss"
    in
    gru_60_f2_history.columns
)


assert (
    "val_loss"
    not in
    gru_60_f2_history.columns
)


# ------------------------------------------------------------
# 9. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_60_f2_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_60_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_60_f2_actual_kw
).all()


assert np.isfinite(
    gru_60_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Same-horizon held-out population audit
#
# F0, F1, and F2 must all be evaluated on precisely the same
# target observations in precisely the same row order.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
]:

    np.testing.assert_array_equal(
        gru_60_f2_actual_kw,
        gru_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 11. Independently recompute held-out metrics
# ------------------------------------------------------------

gru_60_f2_metric_check = (
    compute_regression_metrics_kw(
        gru_60_f2_actual_kw,
        gru_60_f2_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_60_f2_metric_check[
        "MAE_kW"
    ],
    gru_60_f2_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f2_metric_check[
        "RMSE_kW"
    ],
    gru_60_f2_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f2_metric_check[
        "R2"
    ],
    gru_60_f2_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 12. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_60_f2_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_60_f2_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_60_f2_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_60_f2_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    13
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    13
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_60_f2_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_60_f2_history
)


# ------------------------------------------------------------
# 14. Descriptive F1 -> F2 TEST comparison
#
# Positive:
#       F2 has lower TEST MAE than F1.
#
# Negative:
#       F2 has higher TEST MAE than F1.
#
# No model selection or tuning follows from this value.
# ------------------------------------------------------------

gru_60_f1_mae = float(
    gru_final_test_results[
        (
            60,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_60_f2_mae = float(
    gru_60_f2_result[
        "test_MAE_kW"
    ]
)


gru_60_f1_to_f2_improvement_pct = (
    (
        gru_60_f1_mae
        -
        gru_60_f2_mae
    )
    /
    gru_60_f1_mae
    *
    100.0
)


print(
    "\n60-min held-out TEST F1 -> F2 comparison:"
)


print(
    "  F1 MAE:",
    f"{gru_60_f1_mae:.6f}",
    "kW",
)


print(
    "  F2 MAE:",
    f"{gru_60_f2_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_60_f1_to_f2_improvement_pct:+.4f}%",
)


print(
    "  Positive means F2 has lower TEST MAE."
)


# ------------------------------------------------------------
# 15. Unclipped prediction audit
# ------------------------------------------------------------

gru_60_f2_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_60_f2_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_60_f2_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_60_f2_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_60_f2_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_60_f2_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_60_f2_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_60_f2_prediction_audit
)


# ------------------------------------------------------------
# 16. Persist result, final-refit history, and predictions
# ------------------------------------------------------------

gru_60_f2_result_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F2_result.csv"
)


gru_60_f2_history_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F2_history.csv"
)


gru_60_f2_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F2_predictions.csv"
)


pd.DataFrame(
    [
        gru_60_f2_result
    ]
).to_csv(
    gru_60_f2_result_path,
    index=False,
)


gru_60_f2_history.to_csv(
    gru_60_f2_history_path,
    index=False,
)


gru_60_f2_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_60_f2_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_60_f2_actual_kw,

        "predicted_P_Solar_kW":
            gru_60_f2_prediction_kw,
    }
)


gru_60_f2_predictions_table.to_csv(
    gru_60_f2_predictions_path,
    index=False,
)


for output_path in [
    gru_60_f2_result_path,
    gru_60_f2_history_path,
    gru_60_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Current 60-minute GRU progression
# ------------------------------------------------------------

gru_60min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                60,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 60-minute GRU final TEST progression:"
)


display(
    gru_60min_partial_summary
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 60-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_60_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_60_f2_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_60_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_60_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_60_f2_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext experiment: 60-minute F3."
)

Running GRU held-out TEST experiment:
  horizon: 60 minutes
  feature family: F2
  frozen epoch: 3
  architecture: RNN_C2
Epoch 1/3


2026-09-21 21:16:46.711558: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:16:46.711915: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 16ms/step - loss: 0.2668
Epoch 2/3
410/410 - 4s - 11ms/step - loss: 0.2303
Epoch 3/3
410/410 - 4s - 11ms/step - loss: 0.2283

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,60,F2,21,17185,87252,17568,104820,26496,3,3,0.463652,0.861421,0.727656,15.227522,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


60-min held-out TEST F1 -> F2 comparison:
  F1 MAE: 0.509348 kW
  F2 MAE: 0.463652 kW
  Relative MAE change: +8.9715%
  Positive means F2 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.284022,5.884127,6861,0



Current 60-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.503131,0.891575,0.708256
1,F1,13,0.509348,0.885489,0.712225
2,F2,3,0.463652,0.861421,0.727656



GRU 60-MIN F2 FINAL TEST: COMPLETE
Frozen epoch used: 3
Final epochs trained: 3
TEST MAE: 0.463652 kW
TEST RMSE: 0.861421 kW
TEST R²: 0.727656

GRU final TEST evaluations completed: 13 / 15
Remaining: 2
No prediction clipping was applied.

Next experiment: 60-minute F3.


In [95]:
# ============================================================
# Step 79: GRU final held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 60 minutes
#       feature family = F3
#
#
# F3 extends F2 with continuous lunar / Sun-Moon
# astronomical features.
#
# This is the direct astronomical control model for the final
# 60-minute F3 -> F4 Panchang ablation.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       5
#
# Auxiliary width:
#       30
#
# Expected parameters:
#       17,473
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,252 + 17,568 = 104,820
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# No tuning occurs after observing this result.
#
# After this cell, only 60-minute F4 remains.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 60

GRU_TEST_FEATURE_FAMILY = "F3"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current GRU completion state
#
# We should already have:
#
#       15-min F0-F4 = 5
#       30-min F0-F4 = 5
#       60-min F0-F2 = 3
#
# Total = 13 completed TEST evaluations.
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    13
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    13
)


for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert (
        (
            60,
            previous_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    5
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    30
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    17_473
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_252
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F3 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running GRU held-out TEST experiment:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F3"
)

print(
    "  frozen epoch: 5"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final TRAIN + VALIDATION refit
#    followed by held-out TEST evaluation
# ------------------------------------------------------------

(
    gru_60_f3_result,
    gru_60_f3_history,
    gru_60_f3_actual_kw,
    gru_60_f3_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result identity audit
# ------------------------------------------------------------

assert (
    gru_60_f3_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_60_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_60_f3_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_60_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


# ------------------------------------------------------------
# 6. Architecture / population audit
# ------------------------------------------------------------

assert (
    gru_60_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    gru_60_f3_result[
        "trainable_parameter_count"
    ]
    ==
    17_473
)


assert (
    gru_60_f3_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_60_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_60_f3_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    gru_60_f3_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen-epoch / final-refit protocol audit
# ------------------------------------------------------------

assert (
    gru_60_f3_result[
        "frozen_best_epoch"
    ]
    ==
    5
)


assert (
    gru_60_f3_result[
        "final_epochs_trained"
    ]
    ==
    5
)


assert (
    gru_60_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_60_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_60_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Final-refit history audit
#
# Exactly 5 epochs must have been trained.
#
# No validation loss is permitted because TRAIN and
# VALIDATION were merged before the final fit.
# ------------------------------------------------------------

assert (
    len(
        gru_60_f3_history
    )
    ==
    5
)


assert (
    gru_60_f3_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
        2,
        3,
        4,
        5,
    ]
)


assert (
    "loss"
    in
    gru_60_f3_history.columns
)


assert (
    "val_loss"
    not in
    gru_60_f3_history.columns
)


# ------------------------------------------------------------
# 9. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_60_f3_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_60_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_60_f3_actual_kw
).all()


assert np.isfinite(
    gru_60_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Same-horizon held-out population audit
#
# F0-F3 must all use exactly the same 60-minute target vector
# in exactly the same row order.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    np.testing.assert_array_equal(
        gru_60_f3_actual_kw,
        gru_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 11. Independently recompute held-out metrics
# ------------------------------------------------------------

gru_60_f3_metric_check = (
    compute_regression_metrics_kw(
        gru_60_f3_actual_kw,
        gru_60_f3_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_60_f3_metric_check[
        "MAE_kW"
    ],
    gru_60_f3_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f3_metric_check[
        "RMSE_kW"
    ],
    gru_60_f3_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f3_metric_check[
        "R2"
    ],
    gru_60_f3_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 12. Store in canonical GRU final-result containers
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_60_f3_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_60_f3_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_60_f3_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_60_f3_prediction_kw.copy()
)


assert (
    len(
        gru_final_test_results
    )
    ==
    14
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    14
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_60_f3_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_60_f3_history
)


# ------------------------------------------------------------
# 14. Descriptive F2 -> F3 TEST comparison
#
# Positive:
#       F3 has lower TEST MAE than F2.
#
# Negative:
#       F3 has higher TEST MAE than F2.
#
# No tuning follows from this comparison.
# ------------------------------------------------------------

gru_60_f2_mae = float(
    gru_final_test_results[
        (
            60,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_60_f3_mae = float(
    gru_60_f3_result[
        "test_MAE_kW"
    ]
)


gru_60_f2_to_f3_improvement_pct = (
    (
        gru_60_f2_mae
        -
        gru_60_f3_mae
    )
    /
    gru_60_f2_mae
    *
    100.0
)


print(
    "\n60-min held-out TEST F2 -> F3 comparison:"
)


print(
    "  F2 MAE:",
    f"{gru_60_f2_mae:.6f}",
    "kW",
)


print(
    "  F3 MAE:",
    f"{gru_60_f3_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_60_f2_to_f3_improvement_pct:+.4f}%",
)


print(
    "  Positive means F3 has lower TEST MAE."
)


# ------------------------------------------------------------
# 15. Unclipped prediction audit
# ------------------------------------------------------------

gru_60_f3_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_60_f3_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_60_f3_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_60_f3_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_60_f3_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_60_f3_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_60_f3_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_60_f3_prediction_audit
)


# ------------------------------------------------------------
# 16. Persist result / history / row-level predictions
# ------------------------------------------------------------

gru_60_f3_result_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F3_result.csv"
)


gru_60_f3_history_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F3_history.csv"
)


gru_60_f3_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F3_predictions.csv"
)


pd.DataFrame(
    [
        gru_60_f3_result
    ]
).to_csv(
    gru_60_f3_result_path,
    index=False,
)


gru_60_f3_history.to_csv(
    gru_60_f3_history_path,
    index=False,
)


gru_60_f3_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_60_f3_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_60_f3_actual_kw,

        "predicted_P_Solar_kW":
            gru_60_f3_prediction_kw,
    }
)


gru_60_f3_predictions_table.to_csv(
    gru_60_f3_predictions_path,
    index=False,
)


for output_path in [
    gru_60_f3_result_path,
    gru_60_f3_history_path,
    gru_60_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Current 60-minute GRU progression
# ------------------------------------------------------------

gru_60min_partial_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                60,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


print(
    "\nCurrent 60-minute GRU final TEST progression:"
)


display(
    gru_60min_partial_summary
)


# ------------------------------------------------------------
# 18. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU 60-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch used:",
    gru_60_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained:",
    gru_60_f3_result[
        "final_epochs_trained"
    ],
)


print(
    "TEST MAE:",
    f"{gru_60_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{gru_60_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R²:",
    f"{gru_60_f3_result['test_R2']:.6f}",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext and final GRU TEST experiment: 60-minute F4."
)

Running GRU held-out TEST experiment:
  horizon: 60 minutes
  feature family: F3
  frozen epoch: 5
  architecture: RNN_C2
Epoch 1/5


2026-09-21 21:18:05.001764: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:18:05.002076: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2649
Epoch 2/5
410/410 - 4s - 11ms/step - loss: 0.2294
Epoch 3/5
410/410 - 4s - 11ms/step - loss: 0.2293
Epoch 4/5
410/410 - 4s - 10ms/step - loss: 0.2343
Epoch 5/5
410/410 - 4s - 10ms/step - loss: 0.2366

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,60,F3,30,17473,87252,17568,104820,26496,5,5,0.446411,0.817761,0.754564,23.57112,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


60-min held-out TEST F2 -> F3 comparison:
  F2 MAE: 0.463652 kW
  F3 MAE: 0.446411 kW
  Relative MAE change: +3.7184%
  Positive means F3 has lower TEST MAE.

Unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.496778,5.976195,7762,0



Current 60-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,0.503131,0.891575,0.708256
1,F1,13,0.509348,0.885489,0.712225
2,F2,3,0.463652,0.861421,0.727656
3,F3,5,0.446411,0.817761,0.754564



GRU 60-MIN F3 FINAL TEST: COMPLETE
Frozen epoch used: 5
Final epochs trained: 5
TEST MAE: 0.446411 kW
TEST RMSE: 0.817761 kW
TEST R²: 0.754564

GRU final TEST evaluations completed: 14 / 15
Remaining: 1
No prediction clipping was applied.

Next and final GRU TEST experiment: 60-minute F4.


In [96]:
# ============================================================
# Step 80: FINAL GRU held-out TEST evaluation
#
# Experiment:
#
#       model          = GRU
#       horizon        = 60 minutes
#       feature family = F4
#
#
# CENTRAL ABLATION
# ----------------
#
#       F3 = continuous lunar / Sun-Moon astronomy control
#
#       F4 = F3
#            + Panchang categorical representation
#
#
# Therefore:
#
#       F3 -> F4
#
# is the central test of whether the Panchang representation
# adds predictive information beyond continuous astronomy.
#
#
# FROZEN BEFORE TEST
# ------------------
#
# Architecture:
#       RNN_C2
#
# Frozen epoch:
#       1
#
# Auxiliary width:
#       127
#
# Expected parameters:
#       20,577
#
# Final fitting population:
#
#       TRAIN + VALIDATION
#       87,252 + 17,568 = 104,820
#
# Held-out TEST:
#       26,496
#
#
# IMPORTANT
# ---------
#
# This is the 15th and FINAL GRU held-out TEST experiment.
#
# No tuning follows from this result.
# ============================================================


# ------------------------------------------------------------
# 1. Experiment identity
# ------------------------------------------------------------

GRU_TEST_HORIZON = 60

GRU_TEST_FEATURE_FAMILY = "F4"

GRU_TEST_KEY = (
    GRU_TEST_HORIZON,
    GRU_TEST_FEATURE_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current state
#
# Exactly 14 GRU held-out TEST evaluations must already exist.
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    14
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    14
)


# All prior 60-minute families must already exist.
for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert (
        (
            60,
            previous_family,
        )
        in
        gru_final_test_results
    )


# ------------------------------------------------------------
# 3. Frozen-protocol assertions
# ------------------------------------------------------------

assert (
    GRU_FINAL_FROZEN_EPOCHS[
        GRU_TEST_KEY
    ]
    ==
    1
)


assert (
    GRU_EXPECTED_AUX_DIMS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    127
)


assert (
    GRU_EXPECTED_PARAMETER_COUNTS[
        GRU_TEST_FEATURE_FAMILY
    ]
    ==
    20_577
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_train"
    ]
    ==
    87_252
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_validation"
    ]
    ==
    17_568
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    GRU_EXPECTED_POPULATIONS[
        GRU_TEST_HORIZON
    ][
        "n_test"
    ]
    ==
    26_496
)


# F4 must not already have been evaluated.
assert (
    GRU_TEST_KEY
    not in
    gru_final_test_results
)


assert (
    GRU_TEST_KEY
    not in
    gru_final_test_predictions
)


print(
    "Running FINAL GRU held-out TEST experiment:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F4"
)

print(
    "  frozen epoch: 1"
)

print(
    "  architecture: RNN_C2"
)


# ------------------------------------------------------------
# 4. Canonical final TRAIN + VALIDATION refit
#    followed by held-out TEST evaluation
# ------------------------------------------------------------

(
    gru_60_f4_result,
    gru_60_f4_history,
    gru_60_f4_actual_kw,
    gru_60_f4_prediction_kw,
) = run_gru_final_refit_and_test(
    horizon_minutes=
        GRU_TEST_HORIZON,

    feature_family=
        GRU_TEST_FEATURE_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Returned-result identity audit
# ------------------------------------------------------------

assert (
    gru_60_f4_result[
        "model"
    ]
    ==
    "GRU"
)


assert (
    gru_60_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    gru_60_f4_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    gru_60_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


# ------------------------------------------------------------
# 6. Architecture / population audit
# ------------------------------------------------------------

assert (
    gru_60_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    gru_60_f4_result[
        "trainable_parameter_count"
    ]
    ==
    20_577
)


assert (
    gru_60_f4_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    gru_60_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    gru_60_f4_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    gru_60_f4_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen-epoch / final-refit protocol audit
#
# This model must train for EXACTLY ONE epoch.
# ------------------------------------------------------------

assert (
    gru_60_f4_result[
        "frozen_best_epoch"
    ]
    ==
    1
)


assert (
    gru_60_f4_result[
        "final_epochs_trained"
    ]
    ==
    1
)


assert (
    gru_60_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    gru_60_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    gru_60_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 8. Final-refit history audit
#
# Exactly ONE epoch.
#
# No validation loss is allowed.
# ------------------------------------------------------------

assert (
    len(
        gru_60_f4_history
    )
    ==
    1
)


assert (
    gru_60_f4_history[
        "epoch"
    ]
    .tolist()
    ==
    [
        1,
    ]
)


assert (
    "loss"
    in
    gru_60_f4_history.columns
)


assert (
    "val_loss"
    not in
    gru_60_f4_history.columns
)


# ------------------------------------------------------------
# 9. Prediction / target audit
# ------------------------------------------------------------

assert (
    gru_60_f4_actual_kw.shape
    ==
    (
        26_496,
    )
)


assert (
    gru_60_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    gru_60_f4_actual_kw
).all()


assert np.isfinite(
    gru_60_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 10. Exact same 60-minute held-out population across F0-F4
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    np.testing.assert_array_equal(
        gru_60_f4_actual_kw,
        gru_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
    )


# ------------------------------------------------------------
# 11. Independently recompute held-out metrics
# ------------------------------------------------------------

gru_60_f4_metric_check = (
    compute_regression_metrics_kw(
        gru_60_f4_actual_kw,
        gru_60_f4_prediction_kw,
    )
)


np.testing.assert_allclose(
    gru_60_f4_metric_check[
        "MAE_kW"
    ],
    gru_60_f4_result[
        "test_MAE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f4_metric_check[
        "RMSE_kW"
    ],
    gru_60_f4_result[
        "test_RMSE_kW"
    ],
    rtol=0.0,
    atol=1e-10,
)


np.testing.assert_allclose(
    gru_60_f4_metric_check[
        "R2"
    ],
    gru_60_f4_result[
        "test_R2"
    ],
    rtol=0.0,
    atol=1e-10,
)


# ------------------------------------------------------------
# 12. Store final GRU TEST result
# ------------------------------------------------------------

gru_final_test_results[
    GRU_TEST_KEY
] = (
    gru_60_f4_result.copy()
)


gru_final_test_histories[
    GRU_TEST_KEY
] = (
    gru_60_f4_history.copy()
)


gru_final_test_actuals[
    GRU_TEST_KEY
] = (
    gru_60_f4_actual_kw.copy()
)


gru_final_test_predictions[
    GRU_TEST_KEY
] = (
    gru_60_f4_prediction_kw.copy()
)


# All 15 experiments must now exist.
assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_histories
    )
    ==
    15
)


assert (
    len(
        gru_final_test_actuals
    )
    ==
    15
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    15
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

print(
    "\nFINAL GRU HELD-OUT TEST RESULT"
)

print(
    "------------------------------"
)


display(
    pd.DataFrame(
        [
            gru_60_f4_result
        ]
    )
)


print(
    "\nFinal TRAIN + VALIDATION fitting history:"
)


display(
    gru_60_f4_history
)


# ------------------------------------------------------------
# 14. CENTRAL 60-minute F3 -> F4 TEST comparison
#
# Positive:
#
#       F4 has lower TEST MAE than F3.
#
# Negative:
#
#       F4 has higher TEST MAE than F3.
#
# Descriptive held-out point estimate only.
# ------------------------------------------------------------

gru_60_f3_mae = float(
    gru_final_test_results[
        (
            60,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


gru_60_f4_mae = float(
    gru_60_f4_result[
        "test_MAE_kW"
    ]
)


gru_60_f3_to_f4_improvement_pct = (
    (
        gru_60_f3_mae
        -
        gru_60_f4_mae
    )
    /
    gru_60_f3_mae
    *
    100.0
)


print(
    "\n60-min held-out TEST F3 -> F4 comparison:"
)


print(
    "  F3 MAE:",
    f"{gru_60_f3_mae:.6f}",
    "kW",
)


print(
    "  F4 MAE:",
    f"{gru_60_f4_mae:.6f}",
    "kW",
)


print(
    "  Relative MAE change:",
    f"{gru_60_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "  Positive means F4 has lower TEST MAE."
)


print(
    "  Descriptive TEST comparison only — no tuning."
)


# ------------------------------------------------------------
# 15. Unclipped prediction audit
# ------------------------------------------------------------

gru_60_f4_prediction_audit = pd.DataFrame(
    [
        {
            "actual_min_kW":
                float(
                    np.min(
                        gru_60_f4_actual_kw
                    )
                ),

            "actual_max_kW":
                float(
                    np.max(
                        gru_60_f4_actual_kw
                    )
                ),

            "prediction_min_kW":
                float(
                    np.min(
                        gru_60_f4_prediction_kw
                    )
                ),

            "prediction_max_kW":
                float(
                    np.max(
                        gru_60_f4_prediction_kw
                    )
                ),

            "prediction_below_zero_count":
                int(
                    np.sum(
                        gru_60_f4_prediction_kw
                        <
                        0.0
                    )
                ),

            "prediction_above_10kW_count":
                int(
                    np.sum(
                        gru_60_f4_prediction_kw
                        >
                        10.0
                    )
                ),
        }
    ]
)


print(
    "\nUnclipped prediction audit:"
)


display(
    gru_60_f4_prediction_audit
)


# ------------------------------------------------------------
# 16. Persist F4 result, history, and row-level predictions
# ------------------------------------------------------------

gru_60_f4_result_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F4_result.csv"
)


gru_60_f4_history_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F4_history.csv"
)


gru_60_f4_predictions_path = (
    RESULTS_DIR
    /
    "gru_final_60min_F4_predictions.csv"
)


pd.DataFrame(
    [
        gru_60_f4_result
    ]
).to_csv(
    gru_60_f4_result_path,
    index=False,
)


gru_60_f4_history.to_csv(
    gru_60_f4_history_path,
    index=False,
)


gru_60_f4_predictions_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                len(
                    gru_60_f4_actual_kw
                ),
                dtype=int,
            ),

        "actual_P_Solar_kW":
            gru_60_f4_actual_kw,

        "predicted_P_Solar_kW":
            gru_60_f4_prediction_kw,
    }
)


gru_60_f4_predictions_table.to_csv(
    gru_60_f4_predictions_path,
    index=False,
)


for output_path in [
    gru_60_f4_result_path,
    gru_60_f4_history_path,
    gru_60_f4_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Complete 60-minute GRU summary
# ------------------------------------------------------------

gru_final_60min_summary = pd.DataFrame(
    [
        gru_final_test_results[
            (
                60,
                feature_family,
            )
        ]

        for feature_family in [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ]
    ]
)[
    [
        "feature_family",
        "frozen_best_epoch",
        "auxiliary_dim",
        "trainable_parameter_count",
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
    ]
]


assert (
    gru_final_60min_summary.shape
    ==
    (
        5,
        7,
    )
)


print(
    "\nCOMPLETE 60-minute GRU final TEST progression:"
)


display(
    gru_final_60min_summary
)


# ------------------------------------------------------------
# 18. Persist complete 60-minute summary
# ------------------------------------------------------------

gru_final_60min_summary_path = (
    RESULTS_DIR
    /
    "gru_final_60min_summary.csv"
)


gru_final_60min_summary.to_csv(
    gru_final_60min_summary_path,
    index=False,
)


assert (
    gru_final_60min_summary_path.exists()
)


assert (
    gru_final_60min_summary_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 19. F3 -> F4 point-estimate summary across ALL horizons
# ------------------------------------------------------------

gru_f3_f4_test_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon,

            "F3_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F3",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F4_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F4",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),
        }

        for horizon in [
            15,
            30,
            60,
        ]
    ]
)


gru_f3_f4_test_summary[
    "F3_to_F4_test_MAE_improvement_pct"
] = (
    (
        gru_f3_f4_test_summary[
            "F3_test_MAE_kW"
        ]
        -
        gru_f3_f4_test_summary[
            "F4_test_MAE_kW"
        ]
    )
    /
    gru_f3_f4_test_summary[
        "F3_test_MAE_kW"
    ]
    *
    100.0
)


print(
    "\nGRU F3 -> F4 held-out TEST point estimates:"
)


display(
    gru_f3_f4_test_summary
)


# ------------------------------------------------------------
# 20. Persist F3 -> F4 point-estimate summary
# ------------------------------------------------------------

gru_f3_f4_test_summary_path = (
    RESULTS_DIR
    /
    "gru_f3_f4_test_summary.csv"
)


gru_f3_f4_test_summary.to_csv(
    gru_f3_f4_test_summary_path,
    index=False,
)


assert (
    gru_f3_f4_test_summary_path.exists()
)


assert (
    gru_f3_f4_test_summary_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 21. Final completeness audit
#
# Every horizon × family combination must now exist.
# ------------------------------------------------------------

expected_gru_test_keys = {
    (
        horizon,
        feature_family,
    )

    for horizon in [
        15,
        30,
        60,
    ]

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]
}


assert (
    set(
        gru_final_test_results.keys()
    )
    ==
    expected_gru_test_keys
)


assert (
    set(
        gru_final_test_histories.keys()
    )
    ==
    expected_gru_test_keys
)


assert (
    set(
        gru_final_test_actuals.keys()
    )
    ==
    expected_gru_test_keys
)


assert (
    set(
        gru_final_test_predictions.keys()
    )
    ==
    expected_gru_test_keys
)


# ------------------------------------------------------------
# 22. FINAL GRU TEST checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU FINAL HELD-OUT TEST SERIES: COMPLETE"
)

print(
    "============================================================"
)


print(
    "15-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "30-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "60-min F0-F4 evaluations: 5 / 5 COMPLETE"
)


print(
    "\nFrozen epoch used for 60-min F4:",
    gru_60_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "Final epochs trained for 60-min F4:",
    gru_60_f4_result[
        "final_epochs_trained"
    ],
)


print(
    "60-min F3 -> F4 TEST MAE change:",
    f"{gru_60_f3_to_f4_improvement_pct:+.4f}%",
)


print(
    "\nGRU final TEST evaluations completed:",
    len(
        gru_final_test_results
    ),
    "/ 15",
)


print(
    "Remaining:",
    15
    -
    len(
        gru_final_test_results
    ),
)


print(
    "No prediction clipping was applied."
)


print(
    "\nNext step after verification:"
)

print(
    "Consolidate all GRU TEST metrics, then perform the"
)

print(
    "predefined daylight-only and paired daily-block"
)

print(
    "bootstrap F3 -> F4 analyses."
)

Running FINAL GRU held-out TEST experiment:
  horizon: 60 minutes
  feature family: F4
  frozen epoch: 1
  architecture: RNN_C2


2026-09-21 21:19:48.751884: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:19:48.752174: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 6s - 15ms/step - loss: 0.2622

FINAL GRU HELD-OUT TEST RESULT
------------------------------


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,60,F4,127,20577,87252,17568,104820,26496,1,1,0.466455,0.835076,0.74406,6.18676,False,original_train_only_scalers,train_plus_validation



Final TRAIN + VALIDATION fitting history:


...


COMPLETE 60-minute GRU final TEST progression:


,feature_family,frozen_best_epoch,auxiliary_dim,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,3,1,16545,0.503131,0.891575,0.708256
1,F1,13,16,17025,0.509348,0.885489,0.712225
2,F2,3,21,17185,0.463652,0.861421,0.727656
3,F3,5,30,17473,0.446411,0.817761,0.754564
4,F4,1,127,20577,0.466455,0.835076,0.744060



GRU F3 -> F4 held-out TEST point estimates:


,horizon_minutes,F3_test_MAE_kW,F4_test_MAE_kW,F3_to_F4_test_MAE_improvement_pct
0,15,0.291426,0.313546,-7.590039
1,30,0.391312,0.412697,-5.464907
2,60,0.446411,0.466455,-4.489917



GRU FINAL HELD-OUT TEST SERIES: COMPLETE
15-min F0-F4 evaluations: 5 / 5 COMPLETE
30-min F0-F4 evaluations: 5 / 5 COMPLETE
60-min F0-F4 evaluations: 5 / 5 COMPLETE

Frozen epoch used for 60-min F4: 1
Final epochs trained for 60-min F4: 1
60-min F3 -> F4 TEST MAE change: -4.4899%

GRU final TEST evaluations completed: 15 / 15
Remaining: 0
No prediction clipping was applied.

Next step after verification:
Consolidate all GRU TEST metrics, then perform the
predefined daylight-only and paired daily-block
bootstrap F3 -> F4 analyses.


In [97]:
# ============================================================
# Step 81: Consolidate the COMPLETE GRU held-out TEST results
#
# PURPOSE
# -------
#
# All GRU model fitting and held-out TEST evaluation is now
# complete.
#
# We have:
#
#       3 forecasting horizons
#           ×
#       5 feature families
#
#       = 15 final held-out TEST evaluations
#
#
# This cell performs NO MODEL TRAINING.
# This cell performs NO TEST-SET TUNING.
#
# It only:
#
#   1. verifies completeness,
#   2. verifies the frozen protocol metadata,
#   3. verifies target alignment within each horizon,
#   4. consolidates all 15 TEST results,
#   5. constructs the central F3 -> F4 comparison table,
#   6. persists the consolidated GRU outputs.
#
#
# After this cell is verified, we will move to the predefined
# daylight-only analysis.
# ============================================================


# ------------------------------------------------------------
# 1. Canonical experiment ordering
# ------------------------------------------------------------

GRU_FINAL_HORIZONS = [
    15,
    30,
    60,
]


GRU_FINAL_FEATURE_FAMILIES = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]


GRU_EXPECTED_FINAL_KEYS = [
    (
        horizon,
        feature_family,
    )

    for horizon in GRU_FINAL_HORIZONS

    for feature_family in GRU_FINAL_FEATURE_FAMILIES
]


assert (
    len(
        GRU_EXPECTED_FINAL_KEYS
    )
    ==
    15
)


# ------------------------------------------------------------
# 2. Global completeness audit
#
# Every canonical container must contain exactly the same
# 15 horizon × feature-family combinations.
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_histories
    )
    ==
    15
)


assert (
    len(
        gru_final_test_actuals
    )
    ==
    15
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    15
)


assert (
    set(
        gru_final_test_results.keys()
    )
    ==
    set(
        GRU_EXPECTED_FINAL_KEYS
    )
)


assert (
    set(
        gru_final_test_histories.keys()
    )
    ==
    set(
        GRU_EXPECTED_FINAL_KEYS
    )
)


assert (
    set(
        gru_final_test_actuals.keys()
    )
    ==
    set(
        GRU_EXPECTED_FINAL_KEYS
    )
)


assert (
    set(
        gru_final_test_predictions.keys()
    )
    ==
    set(
        GRU_EXPECTED_FINAL_KEYS
    )
)


print(
    "GRU final TEST completeness audit: PASSED"
)

print(
    "  expected experiments: 15"
)

print(
    "  stored experiments:",
    len(
        gru_final_test_results
    ),
)


# ------------------------------------------------------------
# 3. Frozen epoch schedule audit
#
# These epoch counts came from validation BEFORE held-out TEST
# evaluation and must match exactly.
# ------------------------------------------------------------

gru_expected_frozen_epochs = {
    (15, "F0"): 3,
    (15, "F1"): 4,
    (15, "F2"): 7,
    (15, "F3"): 2,
    (15, "F4"): 4,

    (30, "F0"): 4,
    (30, "F1"): 11,
    (30, "F2"): 2,
    (30, "F3"): 5,
    (30, "F4"): 4,

    (60, "F0"): 3,
    (60, "F1"): 13,
    (60, "F2"): 3,
    (60, "F3"): 5,
    (60, "F4"): 1,
}


assert (
    GRU_FINAL_FROZEN_EPOCHS
    ==
    gru_expected_frozen_epochs
)


# ------------------------------------------------------------
# 4. Expected fixed architecture metadata
# ------------------------------------------------------------

gru_expected_auxiliary_dims = {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


gru_expected_parameter_counts = {
    "F0": 16_545,
    "F1": 17_025,
    "F2": 17_185,
    "F3": 17_473,
    "F4": 20_577,
}


gru_expected_populations = {
    15: {
        "n_train": 87_261,
        "n_validation": 17_568,
        "n_trainval": 104_829,
        "n_test": 26_496,
    },

    30: {
        "n_train": 87_258,
        "n_validation": 17_568,
        "n_trainval": 104_826,
        "n_test": 26_496,
    },

    60: {
        "n_train": 87_252,
        "n_validation": 17_568,
        "n_trainval": 104_820,
        "n_test": 26_496,
    },
}


# ------------------------------------------------------------
# 5. Per-experiment protocol audit
#
# Verify that every stored TEST result still carries exactly
# the metadata required by the frozen protocol.
# ------------------------------------------------------------

for (
    horizon,
    feature_family,
) in GRU_EXPECTED_FINAL_KEYS:

    result = gru_final_test_results[
        (
            horizon,
            feature_family,
        )
    ]

    history = gru_final_test_histories[
        (
            horizon,
            feature_family,
        )
    ]

    actual_kw = gru_final_test_actuals[
        (
            horizon,
            feature_family,
        )
    ]

    prediction_kw = gru_final_test_predictions[
        (
            horizon,
            feature_family,
        )
    ]


    # --------------------------------------------------------
    # Model / architecture identity
    # --------------------------------------------------------

    assert (
        result[
            "model"
        ]
        ==
        "GRU"
    )


    assert (
        result[
            "candidate_id"
        ]
        ==
        "RNN_C2"
    )


    assert (
        result[
            "horizon_minutes"
        ]
        ==
        horizon
    )


    assert (
        result[
            "feature_family"
        ]
        ==
        feature_family
    )


    # --------------------------------------------------------
    # Auxiliary width and trainable parameters
    # --------------------------------------------------------

    assert (
        result[
            "auxiliary_dim"
        ]
        ==
        gru_expected_auxiliary_dims[
            feature_family
        ]
    )


    assert (
        result[
            "trainable_parameter_count"
        ]
        ==
        gru_expected_parameter_counts[
            feature_family
        ]
    )


    # --------------------------------------------------------
    # Population sizes
    # --------------------------------------------------------

    assert (
        result[
            "n_train"
        ]
        ==
        gru_expected_populations[
            horizon
        ][
            "n_train"
        ]
    )


    assert (
        result[
            "n_validation"
        ]
        ==
        gru_expected_populations[
            horizon
        ][
            "n_validation"
        ]
    )


    assert (
        result[
            "n_trainval"
        ]
        ==
        gru_expected_populations[
            horizon
        ][
            "n_trainval"
        ]
    )


    assert (
        result[
            "n_test"
        ]
        ==
        gru_expected_populations[
            horizon
        ][
            "n_test"
        ]
    )


    # --------------------------------------------------------
    # Frozen epoch / exact final training length
    # --------------------------------------------------------

    expected_epoch = (
        gru_expected_frozen_epochs[
            (
                horizon,
                feature_family,
            )
        ]
    )


    assert (
        result[
            "frozen_best_epoch"
        ]
        ==
        expected_epoch
    )


    assert (
        result[
            "final_epochs_trained"
        ]
        ==
        expected_epoch
    )


    assert (
        len(
            history
        )
        ==
        expected_epoch
    )


    assert (
        history[
            "epoch"
        ]
        .tolist()
        ==
        list(
            range(
                1,
                expected_epoch + 1,
            )
        )
    )


    # Final refit must not contain validation monitoring.
    assert (
        "val_loss"
        not in
        history.columns
    )


    # --------------------------------------------------------
    # Frozen preprocessing / evaluation policy
    # --------------------------------------------------------

    assert (
        result[
            "prediction_clipping"
        ]
        is False
    )


    assert (
        result[
            "scaler_policy"
        ]
        ==
        "original_train_only_scalers"
    )


    assert (
        result[
            "refit_population"
        ]
        ==
        "train_plus_validation"
    )


    # --------------------------------------------------------
    # Prediction / actual shape and finite-value checks
    # --------------------------------------------------------

    assert (
        actual_kw.shape
        ==
        (
            26_496,
        )
    )


    assert (
        prediction_kw.shape
        ==
        (
            26_496,
        )
    )


    assert np.isfinite(
        actual_kw
    ).all()


    assert np.isfinite(
        prediction_kw
    ).all()


print(
    "Per-experiment frozen-protocol audit: PASSED"
)


# ------------------------------------------------------------
# 6. Exact target alignment audit within each horizon
#
# For a valid feature-family ablation, F0-F4 must be tested
# against precisely the same target observations and row order
# for each horizon.
# ------------------------------------------------------------

for horizon in GRU_FINAL_HORIZONS:

    reference_actual_kw = (
        gru_final_test_actuals[
            (
                horizon,
                "F0",
            )
        ]
    )


    for feature_family in [
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        np.testing.assert_array_equal(
            reference_actual_kw,

            gru_final_test_actuals[
                (
                    horizon,
                    feature_family,
                )
            ],
        )


print(
    "Within-horizon TEST target alignment audit: PASSED"
)


# ------------------------------------------------------------
# 7. Recompute every metric directly from stored predictions
#
# This protects the consolidated table against accidental
# mismatch between a stored result dictionary and its stored
# prediction vector.
# ------------------------------------------------------------

for (
    horizon,
    feature_family,
) in GRU_EXPECTED_FINAL_KEYS:

    result = gru_final_test_results[
        (
            horizon,
            feature_family,
        )
    ]

    actual_kw = gru_final_test_actuals[
        (
            horizon,
            feature_family,
        )
    ]

    prediction_kw = gru_final_test_predictions[
        (
            horizon,
            feature_family,
        )
    ]


    recomputed_metrics = (
        compute_regression_metrics_kw(
            actual_kw,
            prediction_kw,
        )
    )


    np.testing.assert_allclose(
        recomputed_metrics[
            "MAE_kW"
        ],
        result[
            "test_MAE_kW"
        ],
        rtol=0.0,
        atol=1e-10,
    )


    np.testing.assert_allclose(
        recomputed_metrics[
            "RMSE_kW"
        ],
        result[
            "test_RMSE_kW"
        ],
        rtol=0.0,
        atol=1e-10,
    )


    np.testing.assert_allclose(
        recomputed_metrics[
            "R2"
        ],
        result[
            "test_R2"
        ],
        rtol=0.0,
        atol=1e-10,
    )


print(
    "Stored-prediction metric recomputation audit: PASSED"
)


# ------------------------------------------------------------
# 8. Build the canonical 15-row GRU TEST results table
# ------------------------------------------------------------

gru_final_test_results_table = pd.DataFrame(
    [
        gru_final_test_results[
            (
                horizon,
                feature_family,
            )
        ]

        for horizon in GRU_FINAL_HORIZONS

        for feature_family in GRU_FINAL_FEATURE_FAMILIES
    ]
)


# Retain a clean, publication-friendly ordering.
gru_final_test_results_table = (
    gru_final_test_results_table[
        [
            "model",
            "candidate_id",
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "n_train",
            "n_validation",
            "n_trainval",
            "n_test",
            "frozen_best_epoch",
            "final_epochs_trained",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
            "final_fit_seconds",
            "prediction_clipping",
            "scaler_policy",
            "refit_population",
        ]
    ]
    .reset_index(
        drop=True
    )
)


assert (
    gru_final_test_results_table.shape
    ==
    (
        15,
        19,
    )
)


print(
    "\n============================================================"
)

print(
    "COMPLETE GRU HELD-OUT TEST RESULTS"
)

print(
    "============================================================"
)


display(
    gru_final_test_results_table
)


# ------------------------------------------------------------
# 9. Compact metric table
#
# This is easier to inspect than the full protocol table.
# ------------------------------------------------------------

gru_final_test_metrics_compact = (
    gru_final_test_results_table[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
    .copy()
)


print(
    "\nCompact GRU held-out TEST metric table:"
)


display(
    gru_final_test_metrics_compact
)


# ------------------------------------------------------------
# 10. Central F3 -> F4 held-out TEST comparison
#
# Definition:
#
#       relative improvement (%)
#
#         = (MAE_F3 - MAE_F4)
#           -------------------
#                 MAE_F3
#           × 100
#
#
# Positive:
#       F4 has lower MAE.
#
# Negative:
#       F4 has higher MAE.
# ------------------------------------------------------------

gru_final_f3_f4_test_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon,

            "F3_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F3",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F4_test_MAE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F4",
                        )
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F3_test_RMSE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F3",
                        )
                    ][
                        "test_RMSE_kW"
                    ]
                ),

            "F4_test_RMSE_kW":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F4",
                        )
                    ][
                        "test_RMSE_kW"
                    ]
                ),

            "F3_test_R2":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F3",
                        )
                    ][
                        "test_R2"
                    ]
                ),

            "F4_test_R2":
                float(
                    gru_final_test_results[
                        (
                            horizon,
                            "F4",
                        )
                    ][
                        "test_R2"
                    ]
                ),
        }

        for horizon in GRU_FINAL_HORIZONS
    ]
)


gru_final_f3_f4_test_summary[
    "F3_to_F4_MAE_difference_kW"
] = (
    gru_final_f3_f4_test_summary[
        "F3_test_MAE_kW"
    ]
    -
    gru_final_f3_f4_test_summary[
        "F4_test_MAE_kW"
    ]
)


gru_final_f3_f4_test_summary[
    "F3_to_F4_MAE_improvement_pct"
] = (
    (
        gru_final_f3_f4_test_summary[
            "F3_test_MAE_kW"
        ]
        -
        gru_final_f3_f4_test_summary[
            "F4_test_MAE_kW"
        ]
    )
    /
    gru_final_f3_f4_test_summary[
        "F3_test_MAE_kW"
    ]
    *
    100.0
)


# Positive difference means F4 has lower absolute error.
np.testing.assert_allclose(
    gru_final_f3_f4_test_summary[
        "F3_to_F4_MAE_difference_kW"
    ]
    .to_numpy(),
    np.array(
        [
            0.291426
            -
            0.313546,

            0.391312
            -
            0.412697,

            0.446411
            -
            0.466455,
        ]
    ),
    rtol=0.0,
    atol=5e-6,
)


print(
    "\n============================================================"
)

print(
    "GRU CENTRAL F3 -> F4 HELD-OUT TEST COMPARISON"
)

print(
    "============================================================"
)


display(
    gru_final_f3_f4_test_summary
)


# ------------------------------------------------------------
# 11. Direction-only diagnostic
#
# This is not a statistical test.
#
# It simply records whether the observed all-row point estimate
# has F4 below or above F3 in MAE.
# ------------------------------------------------------------

gru_final_f3_f4_test_summary[
    "F4_lower_MAE_than_F3"
] = (
    gru_final_f3_f4_test_summary[
        "F4_test_MAE_kW"
    ]
    <
    gru_final_f3_f4_test_summary[
        "F3_test_MAE_kW"
    ]
)


print(
    "\nObserved all-row F3 -> F4 MAE direction:"
)


display(
    gru_final_f3_f4_test_summary[
        [
            "horizon_minutes",
            "F3_test_MAE_kW",
            "F4_test_MAE_kW",
            "F3_to_F4_MAE_improvement_pct",
            "F4_lower_MAE_than_F3",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Persist consolidated GRU results
# ------------------------------------------------------------

gru_final_test_results_table_path = (
    RESULTS_DIR
    /
    "gru_final_test_results_all.csv"
)


gru_final_test_metrics_compact_path = (
    RESULTS_DIR
    /
    "gru_final_test_metrics_compact.csv"
)


gru_final_f3_f4_test_summary_path = (
    RESULTS_DIR
    /
    "gru_final_f3_f4_test_summary.csv"
)


gru_final_test_results_table.to_csv(
    gru_final_test_results_table_path,
    index=False,
)


gru_final_test_metrics_compact.to_csv(
    gru_final_test_metrics_compact_path,
    index=False,
)


gru_final_f3_f4_test_summary.to_csv(
    gru_final_f3_f4_test_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 13. Verify persisted consolidation artifacts
# ------------------------------------------------------------

for output_path in [
    gru_final_test_results_table_path,
    gru_final_test_metrics_compact_path,
    gru_final_f3_f4_test_summary_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 14. Final consolidation checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU FINAL TEST CONSOLIDATION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Experiments consolidated:",
    len(
        gru_final_test_results_table
    ),
)


print(
    "Expected experiments: 15"
)


print(
    "\nAll 15 frozen-epoch final refits: VERIFIED"
)


print(
    "All within-horizon target vectors: VERIFIED"
)


print(
    "All stored metrics against predictions: VERIFIED"
)


print(
    "No prediction clipping: VERIFIED"
)


print(
    "\nAll-row F3 -> F4 MAE changes:"
)


for row in (
    gru_final_f3_f4_test_summary
    .itertuples(
        index=False
    )
):

    print(
        f"  {int(row.horizon_minutes):>2d}-min:",
        f"{row.F3_to_F4_MAE_improvement_pct:+.4f}%"
    )


print(
    "\nArtifacts written:"
)


print(
    " ",
    gru_final_test_results_table_path
)


print(
    " ",
    gru_final_test_metrics_compact_path
)


print(
    " ",
    gru_final_f3_f4_test_summary_path
)


print(
    "\nNo further GRU model fitting is required."
)


print(
    "Next step: predefined daylight-only F3 -> F4 analysis."
)

GRU final TEST completeness audit: PASSED
  expected experiments: 15
  stored experiments: 15
Per-experiment frozen-protocol audit: PASSED
Within-horizon TEST target alignment audit: PASSED
Stored-prediction metric recomputation audit: PASSED

COMPLETE GRU HELD-OUT TEST RESULTS


,model,candidate_id,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,n_train,n_validation,n_trainval,n_test,frozen_best_epoch,final_epochs_trained,test_MAE_kW,test_RMSE_kW,test_R2,final_fit_seconds,prediction_clipping,scaler_policy,refit_population
0,GRU,RNN_C2,15,F0,1,16545,87261,17568,104829,26496,3,3,0.332980,0.639402,0.849950,14.215378,False,original_train_only_scalers,train_plus_validation
1,GRU,RNN_C2,15,F1,16,17025,87261,17568,104829,26496,4,4,0.278635,0.601394,0.867259,20.056457,False,original_train_only_scalers,train_plus_validation
2,GRU,RNN_C2,15,F2,21,17185,87261,17568,104829,26496,7,7,0.299681,0.604346,0.865953,31.732260,False,original_train_only_scalers,train_plus_validation
3,GRU,RNN_C2,15,F3,30,17473,87261,17568,104829,26496,2,2,0.291426,0.612331,0.862387,10.310117,False,original_train_only_scalers,train_plus_validation
4,GRU,RNN_C2,15,F4,127,20577,87261,17568,104829,26496,4,4,0.313546,0.614217,0.861538,19.080512,False,original_train_only_scalers,train_plus_validation
5,GRU,RNN_C2,30,F0,1,16545,87258,17568,104826,26496,4,4,0.390430,0.737658,0.800292,19.728615,False,original_train_only_scalers,train_plus_validation
6,GRU,RNN_C2,30,F1,16,17025,87258,17568,104826,26496,11,11,0.364036,0.702368,0.818943,49.494695,False,original_train_only_scalers,train_plus_validation
7,GRU,RNN_C2,30,F2,21,17185,87258,17568,104826,26496,2,2,0.396951,0.723970,0.807634,11.431251,False,original_train_only_scalers,train_plus_validation
8,GRU,RNN_C2,30,F3,30,17473,87258,17568,104826,26496,5,5,0.391312,0.727100,0.805967,22.562050,False,original_train_only_scalers,train_plus_validation
9,GRU,RNN_C2,30,F4,127,20577,87258,17568,104826,26496,4,4,0.412697,0.751714,0.792608,18.426495,False,original_train_only_scalers,train_plus_validation



Compact GRU held-out TEST metric table:


,horizon_minutes,feature_family,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,3,0.332980,0.639402,0.849950
1,15,F1,4,0.278635,0.601394,0.867259
2,15,F2,7,0.299681,0.604346,0.865953
3,15,F3,2,0.291426,0.612331,0.862387
4,15,F4,4,0.313546,0.614217,0.861538
5,30,F0,4,0.390430,0.737658,0.800292
6,30,F1,11,0.364036,0.702368,0.818943
7,30,F2,2,0.396951,0.723970,0.807634
8,30,F3,5,0.391312,0.727100,0.805967
9,30,F4,4,0.412697,0.751714,0.792608



GRU CENTRAL F3 -> F4 HELD-OUT TEST COMPARISON


,horizon_minutes,F3_test_MAE_kW,F4_test_MAE_kW,F3_test_RMSE_kW,F4_test_RMSE_kW,F3_test_R2,F4_test_R2,F3_to_F4_MAE_difference_kW,F3_to_F4_MAE_improvement_pct
0,15,0.291426,0.313546,0.612331,0.614217,0.862387,0.861538,-0.022119,-7.590039
1,30,0.391312,0.412697,0.727100,0.751714,0.805967,0.792608,-0.021385,-5.464907
2,60,0.446411,0.466455,0.817761,0.835076,0.754564,0.744060,-0.020043,-4.489917



Observed all-row F3 -> F4 MAE direction:


,horizon_minutes,F3_test_MAE_kW,F4_test_MAE_kW,F3_to_F4_MAE_improvement_pct,F4_lower_MAE_than_F3
0,15,0.291426,0.313546,-7.590039,False
1,30,0.391312,0.412697,-5.464907,False
2,60,0.446411,0.466455,-4.489917,False



GRU FINAL TEST CONSOLIDATION: COMPLETE
Experiments consolidated: 15
Expected experiments: 15

All 15 frozen-epoch final refits: VERIFIED
All within-horizon target vectors: VERIFIED
All stored metrics against predictions: VERIFIED
No prediction clipping: VERIFIED

All-row F3 -> F4 MAE changes:
  15-min: -7.5900%
  30-min: -5.4649%
  60-min: -4.4899%

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_final_test_results_all.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_final_test_metrics_compact.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_final_f3_f4_test_summary.csv

No further GRU model fitting is required.
Next step: predefined daylight-only F3 -> F4 analysis.


In [98]:
# ============================================================
# Step 82: GRU daylight-only held-out TEST analysis
#
# PURPOSE
# -------
#
# Evaluate the central:
#
#       F3 -> F4
#
# comparison using ONLY daylight forecast-valid timestamps.
#
#
# IMPORTANT
# ---------
#
# NO model is trained in this cell.
# NO prediction is regenerated.
# NO hyperparameter is changed.
# NO clipping is applied.
#
# We reuse:
#
#   1. the already-frozen GRU TEST predictions, and
#   2. the SAME horizon-specific daylight masks previously
#      established during the LSTM analysis.
#
#
# Daylight definition:
#
#       forecast-valid solar elevation > 0 degrees
#
#
# Previously established expected TEST populations:
#
#       all TEST rows = 26,496
#       daylight rows = 17,851
#
# for each of:
#
#       15, 30, and 60 minutes.
#
#
# This lets us determine whether the all-row F3 -> F4 behavior
# changes when nighttime / zero-generation periods are removed.
# ============================================================


# ------------------------------------------------------------
# 1. Analysis constants
# ------------------------------------------------------------

GRU_DAYLIGHT_HORIZONS = [
    15,
    30,
    60,
]


GRU_DAYLIGHT_FEATURE_FAMILIES = [
    "F3",
    "F4",
]


GRU_EXPECTED_TEST_ROWS = 26_496

GRU_EXPECTED_DAYLIGHT_ROWS = 17_851


# ------------------------------------------------------------
# 2. Confirm that GRU final TEST evaluation is complete
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_actuals
    )
    ==
    15
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    15
)


for horizon in GRU_DAYLIGHT_HORIZONS:

    for feature_family in (
        GRU_DAYLIGHT_FEATURE_FAMILIES
    ):

        assert (
            (
                horizon,
                feature_family,
            )
            in
            gru_final_test_results
        )


print(
    "GRU final TEST predictions available: VERIFIED"
)


# ------------------------------------------------------------
# 3. Reuse the canonical LSTM TEST-alignment object
#
# The daylight masks were already established and audited in
# the earlier LSTM analysis.
#
# We deliberately reuse those masks rather than reconstructing
# daylight independently for GRU. This guarantees that LSTM
# and GRU daylight evaluations use exactly the same TEST rows.
#
#
# The helper below only normalizes the already-existing
# `lstm_test_alignment` object into one boolean mask per
# horizon. It does NOT recompute astronomy.
# ------------------------------------------------------------

assert (
    "lstm_test_alignment"
    in
    globals()
), (
    "Expected the previously created `lstm_test_alignment` "
    "object from the LSTM daylight analysis."
)


def extract_existing_daylight_mask(
    alignment_object,
    horizon_minutes,
):
    """
    Extract the previously established daylight mask for one
    forecast horizon without recalculating solar astronomy.

    Supported representations:
        - dictionary keyed by horizon
        - DataFrame containing horizon + daylight columns
        - horizon entry as DataFrame / Series / dictionary /
          boolean ndarray
    """

    # --------------------------------------------------------
    # Case A:
    #     top-level dictionary keyed by horizon
    # --------------------------------------------------------

    if isinstance(
        alignment_object,
        dict,
    ):

        if horizon_minutes in alignment_object:

            horizon_alignment = (
                alignment_object[
                    horizon_minutes
                ]
            )

        elif str(
            horizon_minutes
        ) in alignment_object:

            horizon_alignment = (
                alignment_object[
                    str(
                        horizon_minutes
                    )
                ]
            )

        else:

            raise KeyError(
                "Could not find horizon "
                f"{horizon_minutes} in lstm_test_alignment. "
                f"Available keys: "
                f"{list(alignment_object.keys())}"
            )


    # --------------------------------------------------------
    # Case B:
    #     one DataFrame holding all horizons
    # --------------------------------------------------------

    elif isinstance(
        alignment_object,
        pd.DataFrame,
    ):

        horizon_column_candidates = [
            "horizon_minutes",
            "horizon",
            "forecast_horizon_minutes",
        ]


        horizon_column = next(
            (
                column
                for column
                in horizon_column_candidates

                if column
                in alignment_object.columns
            ),
            None,
        )


        if horizon_column is None:

            # A single-horizon DataFrame would be ambiguous
            # here because three horizons are required.
            raise KeyError(
                "lstm_test_alignment is a DataFrame, but no "
                "recognized horizon column was found. "
                f"Columns: {list(alignment_object.columns)}"
            )


        horizon_alignment = (
            alignment_object.loc[
                alignment_object[
                    horizon_column
                ]
                ==
                horizon_minutes
            ]
            .copy()
        )


    else:

        raise TypeError(
            "Unexpected type for lstm_test_alignment: "
            f"{type(alignment_object)}"
        )


    # --------------------------------------------------------
    # Normalize the horizon-specific object to a boolean mask.
    # --------------------------------------------------------

    daylight_key_candidates = [
        "daylight",
        "daylight_mask",
        "is_daylight",
    ]


    # DataFrame representation
    if isinstance(
        horizon_alignment,
        pd.DataFrame,
    ):

        daylight_column = next(
            (
                column
                for column
                in daylight_key_candidates

                if column
                in horizon_alignment.columns
            ),
            None,
        )


        if daylight_column is None:

            raise KeyError(
                "No recognized daylight column found for "
                f"{horizon_minutes}-minute alignment. "
                f"Columns: {list(horizon_alignment.columns)}"
            )


        daylight_mask = (
            horizon_alignment[
                daylight_column
            ]
            .to_numpy(
                dtype=bool
            )
        )


    # Series representation
    elif isinstance(
        horizon_alignment,
        pd.Series,
    ):

        daylight_mask = (
            horizon_alignment
            .to_numpy(
                dtype=bool
            )
        )


    # Dictionary representation
    elif isinstance(
        horizon_alignment,
        dict,
    ):

        daylight_key = next(
            (
                key
                for key
                in daylight_key_candidates

                if key
                in horizon_alignment
            ),
            None,
        )


        if daylight_key is None:

            raise KeyError(
                "No recognized daylight key found for "
                f"{horizon_minutes}-minute alignment. "
                f"Keys: {list(horizon_alignment.keys())}"
            )


        daylight_mask = np.asarray(
            horizon_alignment[
                daylight_key
            ],
            dtype=bool,
        )


    # Raw array representation
    else:

        daylight_mask = np.asarray(
            horizon_alignment
        )


        if daylight_mask.dtype != bool:

            raise TypeError(
                "Raw horizon alignment is not boolean for "
                f"{horizon_minutes} minutes. "
                f"dtype={daylight_mask.dtype}"
            )


    daylight_mask = np.asarray(
        daylight_mask,
        dtype=bool,
    ).reshape(-1)


    return daylight_mask


# ------------------------------------------------------------
# 4. Extract and strongly audit the existing daylight masks
# ------------------------------------------------------------

gru_daylight_masks = {}


for horizon in GRU_DAYLIGHT_HORIZONS:

    daylight_mask = (
        extract_existing_daylight_mask(
            alignment_object=
                lstm_test_alignment,

            horizon_minutes=
                horizon,
        )
    )


    # Same complete held-out TEST length.
    assert (
        daylight_mask.shape
        ==
        (
            GRU_EXPECTED_TEST_ROWS,
        )
    )


    # Same canonical daylight population previously established
    # during the LSTM analysis.
    assert (
        int(
            daylight_mask.sum()
        )
        ==
        GRU_EXPECTED_DAYLIGHT_ROWS
    )


    gru_daylight_masks[
        horizon
    ] = daylight_mask.copy()


print(
    "\nCanonical daylight-mask audit:"
)


gru_daylight_mask_audit = pd.DataFrame(
    [
        {
            "horizon_minutes":
                horizon,

            "n_test_rows":
                len(
                    gru_daylight_masks[
                        horizon
                    ]
                ),

            "n_daylight_rows":
                int(
                    gru_daylight_masks[
                        horizon
                    ]
                    .sum()
                ),

            "n_non_daylight_rows":
                int(
                    (
                        ~gru_daylight_masks[
                            horizon
                        ]
                    )
                    .sum()
                ),
        }

        for horizon in GRU_DAYLIGHT_HORIZONS
    ]
)


display(
    gru_daylight_mask_audit
)


# ------------------------------------------------------------
# 5. Storage for daylight-only metrics
# ------------------------------------------------------------

gru_daylight_metric_rows = []


# ------------------------------------------------------------
# 6. Calculate daylight-only F3 and F4 metrics
#
# Again:
#
#       these are the EXISTING TEST predictions,
#
# simply evaluated on the predefined daylight subset.
# ------------------------------------------------------------

for horizon in GRU_DAYLIGHT_HORIZONS:

    daylight_mask = (
        gru_daylight_masks[
            horizon
        ]
    )


    # --------------------------------------------------------
    # The held-out target must be identical between F3 and F4.
    # --------------------------------------------------------

    actual_f3_kw = (
        gru_final_test_actuals[
            (
                horizon,
                "F3",
            )
        ]
    )


    actual_f4_kw = (
        gru_final_test_actuals[
            (
                horizon,
                "F4",
            )
        ]
    )


    np.testing.assert_array_equal(
        actual_f3_kw,
        actual_f4_kw,
    )


    # --------------------------------------------------------
    # Daylight target subset
    # --------------------------------------------------------

    actual_daylight_kw = (
        actual_f3_kw[
            daylight_mask
        ]
    )


    assert (
        actual_daylight_kw.shape
        ==
        (
            GRU_EXPECTED_DAYLIGHT_ROWS,
        )
    )


    # --------------------------------------------------------
    # Evaluate F3 and F4 separately.
    # --------------------------------------------------------

    for feature_family in [
        "F3",
        "F4",
    ]:

        prediction_kw = (
            gru_final_test_predictions[
                (
                    horizon,
                    feature_family,
                )
            ]
        )


        assert (
            prediction_kw.shape
            ==
            (
                GRU_EXPECTED_TEST_ROWS,
            )
        )


        prediction_daylight_kw = (
            prediction_kw[
                daylight_mask
            ]
        )


        assert (
            prediction_daylight_kw.shape
            ==
            (
                GRU_EXPECTED_DAYLIGHT_ROWS,
            )
        )


        assert np.isfinite(
            prediction_daylight_kw
        ).all()


        daylight_metrics = (
            compute_regression_metrics_kw(
                actual_daylight_kw,
                prediction_daylight_kw,
            )
        )


        gru_daylight_metric_rows.append(
            {
                "model":
                    "GRU",

                "horizon_minutes":
                    horizon,

                "feature_family":
                    feature_family,

                "evaluation_scope":
                    "daylight",

                "daylight_definition":
                    (
                        "forecast-valid solar "
                        "elevation > 0 degrees"
                    ),

                "n_test_rows":
                    GRU_EXPECTED_TEST_ROWS,

                "n_evaluation_rows":
                    GRU_EXPECTED_DAYLIGHT_ROWS,

                "MAE_kW":
                    float(
                        daylight_metrics[
                            "MAE_kW"
                        ]
                    ),

                "RMSE_kW":
                    float(
                        daylight_metrics[
                            "RMSE_kW"
                        ]
                    ),

                "R2":
                    float(
                        daylight_metrics[
                            "R2"
                        ]
                    ),

                "prediction_clipping":
                    False,
            }
        )


# ------------------------------------------------------------
# 7. Canonical daylight metric table
# ------------------------------------------------------------

gru_daylight_metrics = pd.DataFrame(
    gru_daylight_metric_rows
)


assert (
    gru_daylight_metrics.shape
    ==
    (
        6,
        11,
    )
)


assert (
    gru_daylight_metrics[
        "n_evaluation_rows"
    ]
    .eq(
        GRU_EXPECTED_DAYLIGHT_ROWS
    )
    .all()
)


print(
    "\n============================================================"
)

print(
    "GRU DAYLIGHT-ONLY HELD-OUT TEST METRICS"
)

print(
    "============================================================"
)


display(
    gru_daylight_metrics
)


# ------------------------------------------------------------
# 8. Build central daylight-only F3 -> F4 comparison
#
# Positive improvement:
#
#       F4 has LOWER daylight MAE than F3.
#
# Negative improvement:
#
#       F4 has HIGHER daylight MAE than F3.
# ------------------------------------------------------------

gru_daylight_f3_f4_rows = []


for horizon in GRU_DAYLIGHT_HORIZONS:

    horizon_metrics = (
        gru_daylight_metrics.loc[
            gru_daylight_metrics[
                "horizon_minutes"
            ]
            ==
            horizon
        ]
    )


    f3_row = (
        horizon_metrics.loc[
            horizon_metrics[
                "feature_family"
            ]
            ==
            "F3"
        ]
        .iloc[
            0
        ]
    )


    f4_row = (
        horizon_metrics.loc[
            horizon_metrics[
                "feature_family"
            ]
            ==
            "F4"
        ]
        .iloc[
            0
        ]
    )


    daylight_mae_difference_kw = (
        float(
            f3_row[
                "MAE_kW"
            ]
        )
        -
        float(
            f4_row[
                "MAE_kW"
            ]
        )
    )


    daylight_mae_improvement_pct = (
        daylight_mae_difference_kw
        /
        float(
            f3_row[
                "MAE_kW"
            ]
        )
        *
        100.0
    )


    # Existing all-row point estimate for comparison.
    all_row_summary = (
        gru_final_f3_f4_test_summary.loc[
            gru_final_f3_f4_test_summary[
                "horizon_minutes"
            ]
            ==
            horizon
        ]
        .iloc[
            0
        ]
    )


    gru_daylight_f3_f4_rows.append(
        {
            "horizon_minutes":
                horizon,

            "n_daylight_rows":
                GRU_EXPECTED_DAYLIGHT_ROWS,

            "F3_daylight_MAE_kW":
                float(
                    f3_row[
                        "MAE_kW"
                    ]
                ),

            "F4_daylight_MAE_kW":
                float(
                    f4_row[
                        "MAE_kW"
                    ]
                ),

            "F3_to_F4_daylight_MAE_difference_kW":
                daylight_mae_difference_kw,

            "F3_to_F4_daylight_MAE_improvement_pct":
                daylight_mae_improvement_pct,

            "F3_daylight_RMSE_kW":
                float(
                    f3_row[
                        "RMSE_kW"
                    ]
                ),

            "F4_daylight_RMSE_kW":
                float(
                    f4_row[
                        "RMSE_kW"
                    ]
                ),

            "F3_daylight_R2":
                float(
                    f3_row[
                        "R2"
                    ]
                ),

            "F4_daylight_R2":
                float(
                    f4_row[
                        "R2"
                    ]
                ),

            "F3_to_F4_all_row_MAE_improvement_pct":
                float(
                    all_row_summary[
                        "F3_to_F4_MAE_improvement_pct"
                    ]
                ),

            "F4_lower_daylight_MAE_than_F3":
                bool(
                    float(
                        f4_row[
                            "MAE_kW"
                        ]
                    )
                    <
                    float(
                        f3_row[
                            "MAE_kW"
                        ]
                    )
                ),
        }
    )


gru_daylight_f3_f4_summary = pd.DataFrame(
    gru_daylight_f3_f4_rows
)


assert (
    gru_daylight_f3_f4_summary.shape
    ==
    (
        3,
        12,
    )
)


print(
    "\n============================================================"
)

print(
    "GRU CENTRAL F3 -> F4 DAYLIGHT-ONLY COMPARISON"
)

print(
    "============================================================"
)


display(
    gru_daylight_f3_f4_summary
)


# ------------------------------------------------------------
# 9. Compact all-row vs daylight comparison
#
# This is especially important scientifically because PV
# forecasts contain many nighttime observations where the
# target is zero or near zero.
# ------------------------------------------------------------

gru_all_vs_daylight_f3_f4 = (
    gru_daylight_f3_f4_summary[
        [
            "horizon_minutes",
            "F3_to_F4_all_row_MAE_improvement_pct",
            "F3_to_F4_daylight_MAE_improvement_pct",
            "F4_lower_daylight_MAE_than_F3",
        ]
    ]
    .copy()
)


print(
    "\nAll-row versus daylight-only F3 -> F4 MAE changes:"
)


display(
    gru_all_vs_daylight_f3_f4
)


# ------------------------------------------------------------
# 10. Persist daylight-only outputs
# ------------------------------------------------------------

gru_daylight_metrics_path = (
    RESULTS_DIR
    /
    "gru_daylight_metrics_F3_F4.csv"
)


gru_daylight_f3_f4_summary_path = (
    RESULTS_DIR
    /
    "gru_daylight_f3_f4_summary.csv"
)


gru_all_vs_daylight_f3_f4_path = (
    RESULTS_DIR
    /
    "gru_all_vs_daylight_f3_f4.csv"
)


gru_daylight_metrics.to_csv(
    gru_daylight_metrics_path,
    index=False,
)


gru_daylight_f3_f4_summary.to_csv(
    gru_daylight_f3_f4_summary_path,
    index=False,
)


gru_all_vs_daylight_f3_f4.to_csv(
    gru_all_vs_daylight_f3_f4_path,
    index=False,
)


for output_path in [
    gru_daylight_metrics_path,
    gru_daylight_f3_f4_summary_path,
    gru_all_vs_daylight_f3_f4_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 11. Final daylight-analysis checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU DAYLIGHT-ONLY ANALYSIS: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Daylight definition:"
)

print(
    "  forecast-valid solar elevation > 0 degrees"
)


print(
    "\nDaylight rows per horizon:",
    GRU_EXPECTED_DAYLIGHT_ROWS,
    "/",
    GRU_EXPECTED_TEST_ROWS,
)


print(
    "\nF3 -> F4 daylight MAE changes:"
)


for row in (
    gru_daylight_f3_f4_summary
    .itertuples(
        index=False
    )
):

    print(
        f"  {int(row.horizon_minutes):>2d}-min:",
        (
            f"{row.F3_to_F4_daylight_MAE_improvement_pct:+.4f}%"
        ),
    )


print(
    "\nNo retraining was performed."
)

print(
    "No prediction clipping was applied."
)


print(
    "\nArtifacts written:"
)


print(
    " ",
    gru_daylight_metrics_path
)


print(
    " ",
    gru_daylight_f3_f4_summary_path
)


print(
    " ",
    gru_all_vs_daylight_f3_f4_path
)


print(
    "\nNext step after verification:"
)

print(
    "Paired 10,000-replicate UTC daily-block bootstrap for"
)

print(
    "GRU F3 -> F4 on both all-row and daylight scopes."
)

GRU final TEST predictions available: VERIFIED

Canonical daylight-mask audit:


,horizon_minutes,n_test_rows,n_daylight_rows,n_non_daylight_rows
0,15,26496,17851,8645
1,30,26496,17851,8645
2,60,26496,17851,8645



GRU DAYLIGHT-ONLY HELD-OUT TEST METRICS


,model,horizon_minutes,feature_family,evaluation_scope,daylight_definition,n_test_rows,n_evaluation_rows,MAE_kW,RMSE_kW,R2,prediction_clipping
0,GRU,15,F3,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.411546,0.745051,0.814023,False
1,GRU,15,F4,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.416634,0.744165,0.814465,False
2,GRU,30,F3,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.533328,0.880402,0.740314,False
3,GRU,30,F4,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.579026,0.913870,0.720195,False
4,GRU,60,F3,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.620729,0.993085,0.669585,False
5,GRU,60,F4,daylight,forecast-valid solar elevation > 0 degrees,26496,17851,0.652406,1.014898,0.654911,False



GRU CENTRAL F3 -> F4 DAYLIGHT-ONLY COMPARISON


,horizon_minutes,n_daylight_rows,F3_daylight_MAE_kW,F4_daylight_MAE_kW,F3_to_F4_daylight_MAE_difference_kW,F3_to_F4_daylight_MAE_improvement_pct,F3_daylight_RMSE_kW,F4_daylight_RMSE_kW,F3_daylight_R2,F4_daylight_R2,F3_to_F4_all_row_MAE_improvement_pct,F4_lower_daylight_MAE_than_F3
0,15,17851,0.411546,0.416634,-0.005088,-1.236311,0.745051,0.744165,0.814023,0.814465,-7.590039,False
1,30,17851,0.533328,0.579026,-0.045698,-8.568435,0.880402,0.913870,0.740314,0.720195,-5.464907,False
2,60,17851,0.620729,0.652406,-0.031678,-5.103283,0.993085,1.014898,0.669585,0.654911,-4.489917,False



All-row versus daylight-only F3 -> F4 MAE changes:


,horizon_minutes,F3_to_F4_all_row_MAE_improvement_pct,F3_to_F4_daylight_MAE_improvement_pct,F4_lower_daylight_MAE_than_F3
0,15,-7.590039,-1.236311,False
1,30,-5.464907,-8.568435,False
2,60,-4.489917,-5.103283,False



GRU DAYLIGHT-ONLY ANALYSIS: COMPLETE
Daylight definition:
  forecast-valid solar elevation > 0 degrees

Daylight rows per horizon: 17851 / 26496

F3 -> F4 daylight MAE changes:
  15-min: -1.2363%
  30-min: -8.5684%
  60-min: -5.1033%

No retraining was performed.
No prediction clipping was applied.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_daylight_metrics_F3_F4.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_daylight_f3_f4_summary.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_all_vs_daylight_f3_f4.csv

Next step after verification:
Paired 10,000-replicate UTC daily-block bootstrap for
GRU F3 -> F4 on both all-row and daylight scopes.


In [99]:
# ============================================================
# Step 83: Paired UTC daily-block bootstrap for GRU F3 vs F4
#
# PURPOSE
# -------
#
# Quantify uncertainty around the held-out TEST comparison:
#
#       F3 -> F4
#
# for:
#
#       horizons:
#           15, 30, 60 minutes
#
#       evaluation scopes:
#           all TEST rows
#           daylight-only TEST rows
#
#
# FROZEN BOOTSTRAP PROTOCOL
# -------------------------
#
# Bootstrap unit:
#       forecast-valid UTC calendar day
#
# Number of TEST day blocks:
#       92
#
# Number of bootstrap replicates:
#       10,000
#
# Random seed:
#       42
#
# Paired comparison:
#
#       MAE_F3 - MAE_F4
#
# Interpretation:
#
#       positive  -> F4 has lower MAE
#       negative  -> F4 has higher MAE
#
#
# VERY IMPORTANT
# --------------
#
# Each sampled day is used simultaneously for F3 and F4.
#
# We DO NOT independently bootstrap F3 and F4 because doing so
# would destroy the paired structure of the experiment.
#
#
# The bootstrap statistic is based on POOLED ROW-WEIGHTED MAE:
#
#       sum of absolute errors across sampled daily blocks
#       ---------------------------------------------------
#       total number of rows across sampled daily blocks
#
#
# This means days with different numbers of daylight rows retain
# their correct contribution to the daylight-only MAE.
#
#
# NO MODEL TRAINING occurs in this cell.
# NO predictions are regenerated.
# NO clipping is applied.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen bootstrap settings
# ------------------------------------------------------------

GRU_BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]


GRU_BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


GRU_N_BOOTSTRAP = 10_000

GRU_BOOTSTRAP_RANDOM_SEED = 42

GRU_EXPECTED_TEST_ROWS = 26_496

GRU_EXPECTED_DAYLIGHT_ROWS = 17_851

GRU_EXPECTED_TEST_DAYS = 92


# ------------------------------------------------------------
# 2. Preconditions
#
# Final GRU results and daylight analysis must already exist.
# ------------------------------------------------------------

assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_predictions
    )
    ==
    15
)


assert (
    "gru_daylight_masks"
    in
    globals()
)


assert (
    "gru_daylight_f3_f4_summary"
    in
    globals()
)


assert (
    set(
        gru_daylight_masks.keys()
    )
    ==
    set(
        GRU_BOOTSTRAP_HORIZONS
    )
)


# ------------------------------------------------------------
# 3. Helper:
#    extract the EXISTING forecast-valid timestamps from the
#    previously constructed LSTM TEST-alignment object.
#
# We are NOT recreating timestamps or astronomy.
#
# This keeps the GRU bootstrap aligned with the same held-out
# rows already audited during the LSTM / GRU analyses.
# ------------------------------------------------------------

def extract_existing_valid_time(
    alignment_object,
    horizon_minutes,
):
    """
    Extract the already-established TEST forecast-valid timestamps
    for one horizon from `lstm_test_alignment`.

    The helper supports the same general alignment representations
    handled in the preceding daylight-analysis cell.
    """

    # --------------------------------------------------------
    # Obtain the horizon-specific alignment object.
    # --------------------------------------------------------

    if isinstance(
        alignment_object,
        dict,
    ):

        if horizon_minutes in alignment_object:

            horizon_alignment = (
                alignment_object[
                    horizon_minutes
                ]
            )

        elif str(
            horizon_minutes
        ) in alignment_object:

            horizon_alignment = (
                alignment_object[
                    str(
                        horizon_minutes
                    )
                ]
            )

        else:

            raise KeyError(
                "Could not find horizon "
                f"{horizon_minutes} in lstm_test_alignment. "
                f"Available keys: "
                f"{list(alignment_object.keys())}"
            )


    elif isinstance(
        alignment_object,
        pd.DataFrame,
    ):

        horizon_column_candidates = [
            "horizon_minutes",
            "horizon",
            "forecast_horizon_minutes",
        ]


        horizon_column = next(
            (
                column
                for column
                in horizon_column_candidates

                if column
                in alignment_object.columns
            ),
            None,
        )


        if horizon_column is None:

            raise KeyError(
                "lstm_test_alignment is a DataFrame, but no "
                "recognized horizon column was found. "
                f"Columns: {list(alignment_object.columns)}"
            )


        horizon_alignment = (
            alignment_object.loc[
                alignment_object[
                    horizon_column
                ]
                ==
                horizon_minutes
            ]
            .copy()
        )


    else:

        raise TypeError(
            "Unexpected type for lstm_test_alignment: "
            f"{type(alignment_object)}"
        )


    # --------------------------------------------------------
    # Extract valid-time values.
    # --------------------------------------------------------

    valid_time_candidates = [
        "valid_time",
        "valid_time_utc",
        "forecast_valid_time",
        "forecast_valid_time_utc",
    ]


    if isinstance(
        horizon_alignment,
        pd.DataFrame,
    ):

        valid_time_column = next(
            (
                column
                for column
                in valid_time_candidates

                if column
                in horizon_alignment.columns
            ),
            None,
        )


        if valid_time_column is None:

            raise KeyError(
                "No recognized valid-time column found for "
                f"{horizon_minutes}-minute alignment. "
                f"Columns: {list(horizon_alignment.columns)}"
            )


        valid_time = (
            horizon_alignment[
                valid_time_column
            ]
        )


    elif isinstance(
        horizon_alignment,
        dict,
    ):

        valid_time_key = next(
            (
                key
                for key
                in valid_time_candidates

                if key
                in horizon_alignment
            ),
            None,
        )


        if valid_time_key is None:

            raise KeyError(
                "No recognized valid-time key found for "
                f"{horizon_minutes}-minute alignment. "
                f"Keys: {list(horizon_alignment.keys())}"
            )


        valid_time = (
            horizon_alignment[
                valid_time_key
            ]
        )


    else:

        raise TypeError(
            "Horizon-specific TEST alignment does not contain "
            "an accessible valid-time field. "
            f"Type: {type(horizon_alignment)}"
        )


    # Normalize explicitly to UTC.
    valid_time = pd.DatetimeIndex(
        pd.to_datetime(
            valid_time,
            utc=True,
        )
    )


    return valid_time


# ------------------------------------------------------------
# 4. Extract and audit TEST valid timestamps
# ------------------------------------------------------------

gru_test_valid_times = {}


for horizon in GRU_BOOTSTRAP_HORIZONS:

    valid_time = (
        extract_existing_valid_time(
            alignment_object=
                lstm_test_alignment,

            horizon_minutes=
                horizon,
        )
    )


    assert (
        len(
            valid_time
        )
        ==
        GRU_EXPECTED_TEST_ROWS
    )


    assert (
        valid_time.min()
        ==
        pd.Timestamp(
            "2019-06-01 00:00:00",
            tz="UTC",
        )
    )


    assert (
        valid_time.max()
        ==
        pd.Timestamp(
            "2019-08-31 23:55:00",
            tz="UTC",
        )
    )


    assert (
        valid_time.is_monotonic_increasing
    )


    assert (
        not
        valid_time.has_duplicates
    )


    # 92 UTC calendar days:
    # June + July + August 2019.
    valid_dates = (
        valid_time.floor(
            "D"
        )
    )


    assert (
        valid_dates.nunique()
        ==
        GRU_EXPECTED_TEST_DAYS
    )


    gru_test_valid_times[
        horizon
    ] = valid_time


print(
    "GRU TEST valid-time alignment audit: PASSED"
)


# ------------------------------------------------------------
# 5. Strongly verify GRU predictions against timestamp vectors
# ------------------------------------------------------------

for horizon in GRU_BOOTSTRAP_HORIZONS:

    valid_time = (
        gru_test_valid_times[
            horizon
        ]
    )


    daylight_mask = (
        gru_daylight_masks[
            horizon
        ]
    )


    assert (
        len(
            daylight_mask
        )
        ==
        len(
            valid_time
        )
        ==
        GRU_EXPECTED_TEST_ROWS
    )


    assert (
        int(
            daylight_mask.sum()
        )
        ==
        GRU_EXPECTED_DAYLIGHT_ROWS
    )


    # F3 and F4 must use exactly the same target.
    np.testing.assert_array_equal(
        gru_final_test_actuals[
            (
                horizon,
                "F3",
            )
        ],

        gru_final_test_actuals[
            (
                horizon,
                "F4",
            )
        ],
    )


print(
    "GRU bootstrap row-alignment audit: PASSED"
)


# ------------------------------------------------------------
# 6. Initialize reproducible bootstrap RNG
#
# One RNG is used sequentially across the complete bootstrap
# analysis, matching the frozen experimental convention.
# ------------------------------------------------------------

gru_bootstrap_rng = (
    np.random.default_rng(
        GRU_BOOTSTRAP_RANDOM_SEED
    )
)


# ------------------------------------------------------------
# 7. Bootstrap output containers
# ------------------------------------------------------------

gru_bootstrap_summary_rows = []

gru_bootstrap_distribution_rows = []

gru_bootstrap_daily_block_audit_rows = []


# ------------------------------------------------------------
# 8. Run paired daily-block bootstrap
# ------------------------------------------------------------

for horizon in GRU_BOOTSTRAP_HORIZONS:

    # --------------------------------------------------------
    # Retrieve fixed held-out TEST vectors.
    # --------------------------------------------------------

    valid_time = (
        gru_test_valid_times[
            horizon
        ]
    )


    actual_kw = (
        gru_final_test_actuals[
            (
                horizon,
                "F3",
            )
        ]
    )


    prediction_f3_kw = (
        gru_final_test_predictions[
            (
                horizon,
                "F3",
            )
        ]
    )


    prediction_f4_kw = (
        gru_final_test_predictions[
            (
                horizon,
                "F4",
            )
        ]
    )


    daylight_mask = (
        gru_daylight_masks[
            horizon
        ]
    )


    # --------------------------------------------------------
    # Build one row-aligned held-out comparison table.
    # --------------------------------------------------------

    df_gru_bootstrap_base = pd.DataFrame(
        {
            "valid_time":
                valid_time,

            "actual_kW":
                actual_kw,

            "prediction_F3_kW":
                prediction_f3_kw,

            "prediction_F4_kW":
                prediction_f4_kw,

            "daylight":
                daylight_mask,
        }
    )


    assert (
        len(
            df_gru_bootstrap_base
        )
        ==
        GRU_EXPECTED_TEST_ROWS
    )


    # Paired absolute errors.
    df_gru_bootstrap_base[
        "abs_error_F3"
    ] = np.abs(
        df_gru_bootstrap_base[
            "actual_kW"
        ]
        -
        df_gru_bootstrap_base[
            "prediction_F3_kW"
        ]
    )


    df_gru_bootstrap_base[
        "abs_error_F4"
    ] = np.abs(
        df_gru_bootstrap_base[
            "actual_kW"
        ]
        -
        df_gru_bootstrap_base[
            "prediction_F4_kW"
        ]
    )


    # UTC daily block identifier.
    df_gru_bootstrap_base[
        "valid_date"
    ] = (
        df_gru_bootstrap_base[
            "valid_time"
        ]
        .dt.floor(
            "D"
        )
    )


    # --------------------------------------------------------
    # Analyze BOTH frozen scopes.
    # --------------------------------------------------------

    for scope_name in GRU_BOOTSTRAP_SCOPES:

        if scope_name == "all":

            df_scope = (
                df_gru_bootstrap_base
                .copy()
            )


            expected_scope_rows = (
                GRU_EXPECTED_TEST_ROWS
            )


        elif scope_name == "daylight":

            df_scope = (
                df_gru_bootstrap_base.loc[
                    df_gru_bootstrap_base[
                        "daylight"
                    ]
                ]
                .copy()
            )


            expected_scope_rows = (
                GRU_EXPECTED_DAYLIGHT_ROWS
            )


        else:

            raise ValueError(
                f"Unexpected bootstrap scope: {scope_name}"
            )


        assert (
            len(
                df_scope
            )
            ==
            expected_scope_rows
        )


        # ----------------------------------------------------
        # 9. Collapse each UTC calendar day to sufficient
        #    statistics.
        #
        # Retain:
        #
        #       n_rows
        #       sum |error F3|
        #       sum |error F4|
        #
        # Sampling these daily aggregates with replacement is
        # mathematically equivalent to concatenating all rows
        # from the sampled days and recomputing pooled MAE.
        # ----------------------------------------------------

        daily_blocks = (
            df_scope
            .groupby(
                "valid_date",
                sort=True,
            )
            .agg(
                n_rows=(
                    "actual_kW",
                    "size",
                ),

                abs_error_F3_sum=(
                    "abs_error_F3",
                    "sum",
                ),

                abs_error_F4_sum=(
                    "abs_error_F4",
                    "sum",
                ),
            )
        )


        assert (
            len(
                daily_blocks
            )
            ==
            GRU_EXPECTED_TEST_DAYS
        )


        assert (
            int(
                daily_blocks[
                    "n_rows"
                ]
                .sum()
            )
            ==
            expected_scope_rows
        )


        assert (
            daily_blocks[
                "n_rows"
            ]
            .gt(
                0
            )
            .all()
        )


        # Record daily-block population diagnostics.
        gru_bootstrap_daily_block_audit_rows.append(
            {
                "horizon_minutes":
                    horizon,

                "scope":
                    scope_name,

                "n_rows":
                    len(
                        df_scope
                    ),

                "n_daily_blocks":
                    len(
                        daily_blocks
                    ),

                "min_rows_per_block":
                    int(
                        daily_blocks[
                            "n_rows"
                        ]
                        .min()
                    ),

                "max_rows_per_block":
                    int(
                        daily_blocks[
                            "n_rows"
                        ]
                        .max()
                    ),
            }
        )


        # ----------------------------------------------------
        # 10. Observed statistic
        # ----------------------------------------------------

        observed_f3_mae = float(
            df_scope[
                "abs_error_F3"
            ]
            .mean()
        )


        observed_f4_mae = float(
            df_scope[
                "abs_error_F4"
            ]
            .mean()
        )


        observed_mae_difference = (
            observed_f3_mae
            -
            observed_f4_mae
        )


        observed_relative_improvement_pct = (
            observed_mae_difference
            /
            observed_f3_mae
            *
            100.0
        )


        # ----------------------------------------------------
        # 11. Convert daily sufficient statistics to arrays
        # ----------------------------------------------------

        block_n_rows = (
            daily_blocks[
                "n_rows"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        block_f3_error_sum = (
            daily_blocks[
                "abs_error_F3_sum"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        block_f4_error_sum = (
            daily_blocks[
                "abs_error_F4_sum"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        n_blocks = len(
            daily_blocks
        )


        assert (
            n_blocks
            ==
            GRU_EXPECTED_TEST_DAYS
        )


        # ----------------------------------------------------
        # 12. Draw paired daily blocks
        #
        # Shape:
        #
        #       10,000 x 92
        #
        # Each row is one bootstrap replicate containing
        # 92 UTC days sampled WITH replacement.
        # ----------------------------------------------------

        sampled_block_indices = (
            gru_bootstrap_rng.integers(
                low=0,
                high=n_blocks,
                size=(
                    GRU_N_BOOTSTRAP,
                    n_blocks,
                ),
            )
        )


        # ----------------------------------------------------
        # 13. Calculate pooled row-weighted MAE per replicate
        # ----------------------------------------------------

        bootstrap_n_rows = (
            block_n_rows[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_error_sum = (
            block_f3_error_sum[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f4_error_sum = (
            block_f4_error_sum[
                sampled_block_indices
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_mae = (
            bootstrap_f3_error_sum
            /
            bootstrap_n_rows
        )


        bootstrap_f4_mae = (
            bootstrap_f4_error_sum
            /
            bootstrap_n_rows
        )


        # Positive means F4 has lower MAE.
        bootstrap_mae_difference = (
            bootstrap_f3_mae
            -
            bootstrap_f4_mae
        )


        bootstrap_relative_improvement_pct = (
            bootstrap_mae_difference
            /
            bootstrap_f3_mae
            *
            100.0
        )


        assert np.isfinite(
            bootstrap_f3_mae
        ).all()


        assert np.isfinite(
            bootstrap_f4_mae
        ).all()


        assert np.isfinite(
            bootstrap_mae_difference
        ).all()


        assert np.isfinite(
            bootstrap_relative_improvement_pct
        ).all()


        # ----------------------------------------------------
        # 14. 95% percentile confidence intervals
        # ----------------------------------------------------

        (
            mae_difference_ci_lower,
            mae_difference_ci_upper,
        ) = np.percentile(
            bootstrap_mae_difference,
            [
                2.5,
                97.5,
            ],
        )


        (
            relative_ci_lower,
            relative_ci_upper,
        ) = np.percentile(
            bootstrap_relative_improvement_pct,
            [
                2.5,
                97.5,
            ],
        )


        # ----------------------------------------------------
        # 15. Directional bootstrap diagnostic
        #
        # This is NOT being presented as a formal p-value.
        #
        # It is simply the proportion of bootstrap replicates
        # in which F4 has lower pooled MAE than F3.
        # ----------------------------------------------------

        bootstrap_fraction_f4_better = float(
            np.mean(
                bootstrap_mae_difference
                >
                0.0
            )
        )


        # ----------------------------------------------------
        # 16. Store summary
        # ----------------------------------------------------

        gru_bootstrap_summary_rows.append(
            {
                "model":
                    "GRU",

                "horizon_minutes":
                    horizon,

                "scope":
                    scope_name,

                "n_rows":
                    expected_scope_rows,

                "n_daily_blocks":
                    n_blocks,

                "n_bootstrap":
                    GRU_N_BOOTSTRAP,

                "bootstrap_seed":
                    GRU_BOOTSTRAP_RANDOM_SEED,

                "observed_F3_MAE_kW":
                    observed_f3_mae,

                "observed_F4_MAE_kW":
                    observed_f4_mae,

                "observed_F3_minus_F4_MAE_kW":
                    observed_mae_difference,

                "observed_relative_improvement_pct":
                    observed_relative_improvement_pct,

                "MAE_difference_CI95_lower_kW":
                    float(
                        mae_difference_ci_lower
                    ),

                "MAE_difference_CI95_upper_kW":
                    float(
                        mae_difference_ci_upper
                    ),

                "relative_improvement_CI95_lower_pct":
                    float(
                        relative_ci_lower
                    ),

                "relative_improvement_CI95_upper_pct":
                    float(
                        relative_ci_upper
                    ),

                "bootstrap_fraction_F4_lower_MAE":
                    bootstrap_fraction_f4_better,

                "paired":
                    True,

                "block_unit":
                    "forecast-valid UTC calendar day",

                "model_retraining":
                    False,

                "prediction_clipping":
                    False,
            }
        )


        # ----------------------------------------------------
        # 17. Store complete bootstrap distribution
        #
        # We preserve this for later plots / model comparison.
        # ----------------------------------------------------

        for bootstrap_index in range(
            GRU_N_BOOTSTRAP
        ):

            gru_bootstrap_distribution_rows.append(
                {
                    "model":
                        "GRU",

                    "horizon_minutes":
                        horizon,

                    "scope":
                        scope_name,

                    "bootstrap_replicate":
                        bootstrap_index + 1,

                    "sampled_total_rows":
                        int(
                            bootstrap_n_rows[
                                bootstrap_index
                            ]
                        ),

                    "F3_MAE_kW":
                        float(
                            bootstrap_f3_mae[
                                bootstrap_index
                            ]
                        ),

                    "F4_MAE_kW":
                        float(
                            bootstrap_f4_mae[
                                bootstrap_index
                            ]
                        ),

                    "F3_minus_F4_MAE_kW":
                        float(
                            bootstrap_mae_difference[
                                bootstrap_index
                            ]
                        ),

                    "relative_improvement_pct":
                        float(
                            bootstrap_relative_improvement_pct[
                                bootstrap_index
                            ]
                        ),
                }
            )


# ------------------------------------------------------------
# 18. Construct canonical bootstrap outputs
# ------------------------------------------------------------

gru_bootstrap_summary = pd.DataFrame(
    gru_bootstrap_summary_rows
)


gru_bootstrap_distributions = pd.DataFrame(
    gru_bootstrap_distribution_rows
)


gru_bootstrap_daily_block_audit = pd.DataFrame(
    gru_bootstrap_daily_block_audit_rows
)


# 3 horizons × 2 scopes.
assert (
    gru_bootstrap_summary.shape[
        0
    ]
    ==
    6
)


# 6 analyses × 10,000 replicates.
assert (
    gru_bootstrap_distributions.shape[
        0
    ]
    ==
    60_000
)


assert (
    gru_bootstrap_daily_block_audit.shape[
        0
    ]
    ==
    6
)


# Every analysis must contain exactly 92 UTC day blocks.
assert (
    gru_bootstrap_summary[
        "n_daily_blocks"
    ]
    .eq(
        GRU_EXPECTED_TEST_DAYS
    )
    .all()
)


# ------------------------------------------------------------
# 19. Cross-check observed statistics against earlier GRU
#     all-row and daylight point estimates
# ------------------------------------------------------------

for row in (
    gru_bootstrap_summary
    .itertuples(
        index=False
    )
):

    horizon = int(
        row.horizon_minutes
    )


    if row.scope == "all":

        previous_row = (
            gru_final_f3_f4_test_summary.loc[
                gru_final_f3_f4_test_summary[
                    "horizon_minutes"
                ]
                ==
                horizon
            ]
            .iloc[
                0
            ]
        )


        expected_relative_pct = float(
            previous_row[
                "F3_to_F4_MAE_improvement_pct"
            ]
        )


    elif row.scope == "daylight":

        previous_row = (
            gru_daylight_f3_f4_summary.loc[
                gru_daylight_f3_f4_summary[
                    "horizon_minutes"
                ]
                ==
                horizon
            ]
            .iloc[
                0
            ]
        )


        expected_relative_pct = float(
            previous_row[
                "F3_to_F4_daylight_MAE_improvement_pct"
            ]
        )


    else:

        raise ValueError(
            row.scope
        )


    np.testing.assert_allclose(
        row.observed_relative_improvement_pct,
        expected_relative_pct,
        rtol=0.0,
        atol=1e-10,
    )


print(
    "Observed-statistic cross-check: PASSED"
)


# ------------------------------------------------------------
# 20. Display daily-block population audit
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU DAILY-BLOCK POPULATION AUDIT"
)

print(
    "============================================================"
)


display(
    gru_bootstrap_daily_block_audit
)


# ------------------------------------------------------------
# 21. Display bootstrap summary
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU PAIRED DAILY-BLOCK BOOTSTRAP SUMMARY"
)

print(
    "============================================================"
)


display(
    gru_bootstrap_summary
)


# ------------------------------------------------------------
# 22. Compact scientific summary
#
# The relative-improvement CI is particularly easy to compare
# with the earlier LSTM analysis.
# ------------------------------------------------------------

gru_bootstrap_compact_summary = (
    gru_bootstrap_summary[
        [
            "horizon_minutes",
            "scope",
            "n_rows",
            "observed_F3_MAE_kW",
            "observed_F4_MAE_kW",
            "observed_relative_improvement_pct",
            "relative_improvement_CI95_lower_pct",
            "relative_improvement_CI95_upper_pct",
            "bootstrap_fraction_F4_lower_MAE",
        ]
    ]
    .copy()
)


print(
    "\nCompact GRU F3 -> F4 bootstrap summary:"
)


display(
    gru_bootstrap_compact_summary
)


# ------------------------------------------------------------
# 23. CI direction diagnostic
#
# Categories:
#
#       CI entirely > 0:
#           bootstrap interval favors lower F4 MAE
#
#       CI entirely < 0:
#           bootstrap interval favors lower F3 MAE
#
#       otherwise:
#           interval crosses zero
#
# This is descriptive bootstrap interpretation, not a claim
# of universal model superiority.
# ------------------------------------------------------------

def classify_gru_bootstrap_ci_direction(
    lower,
    upper,
):

    if lower > 0.0:

        return "CI entirely above zero"

    if upper < 0.0:

        return "CI entirely below zero"

    return "CI crosses zero"


gru_bootstrap_compact_summary[
    "CI_direction"
] = [
    classify_gru_bootstrap_ci_direction(
        lower,
        upper,
    )

    for lower, upper in zip(
        gru_bootstrap_compact_summary[
            "relative_improvement_CI95_lower_pct"
        ],

        gru_bootstrap_compact_summary[
            "relative_improvement_CI95_upper_pct"
        ],
    )
]


print(
    "\nBootstrap CI direction diagnostic:"
)


display(
    gru_bootstrap_compact_summary[
        [
            "horizon_minutes",
            "scope",
            "observed_relative_improvement_pct",
            "relative_improvement_CI95_lower_pct",
            "relative_improvement_CI95_upper_pct",
            "CI_direction",
        ]
    ]
)


# ------------------------------------------------------------
# 24. Persist bootstrap artifacts
# ------------------------------------------------------------

gru_bootstrap_summary_path = (
    RESULTS_DIR
    /
    "gru_f3_f4_daily_block_bootstrap_summary.csv"
)


gru_bootstrap_compact_summary_path = (
    RESULTS_DIR
    /
    "gru_f3_f4_daily_block_bootstrap_compact.csv"
)


gru_bootstrap_daily_block_audit_path = (
    RESULTS_DIR
    /
    "gru_f3_f4_daily_block_audit.csv"
)


gru_bootstrap_distributions_path = (
    RESULTS_DIR
    /
    "gru_f3_f4_daily_block_bootstrap_distributions.csv.gz"
)


gru_bootstrap_summary.to_csv(
    gru_bootstrap_summary_path,
    index=False,
)


gru_bootstrap_compact_summary.to_csv(
    gru_bootstrap_compact_summary_path,
    index=False,
)


gru_bootstrap_daily_block_audit.to_csv(
    gru_bootstrap_daily_block_audit_path,
    index=False,
)


gru_bootstrap_distributions.to_csv(
    gru_bootstrap_distributions_path,
    index=False,
    compression="gzip",
)


# ------------------------------------------------------------
# 25. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    gru_bootstrap_summary_path,
    gru_bootstrap_compact_summary_path,
    gru_bootstrap_daily_block_audit_path,
    gru_bootstrap_distributions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 26. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "GRU PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Bootstrap replicates per comparison:",
    GRU_N_BOOTSTRAP,
)


print(
    "UTC daily blocks per comparison:",
    GRU_EXPECTED_TEST_DAYS,
)


print(
    "Random seed:",
    GRU_BOOTSTRAP_RANDOM_SEED,
)


print(
    "\nRelative F3 -> F4 MAE estimates and 95% CIs:"
)


for row in (
    gru_bootstrap_compact_summary
    .itertuples(
        index=False
    )
):

    print(
        (
            f"  {int(row.horizon_minutes):>2d}-min "
            f"{row.scope:<8s}: "
            f"{row.observed_relative_improvement_pct:+.4f}% "
            f"["
            f"{row.relative_improvement_CI95_lower_pct:+.4f}%, "
            f"{row.relative_improvement_CI95_upper_pct:+.4f}%"
            f"]"
        )
    )


print(
    "\nInterpretation of sign:"
)

print(
    "  positive = F4 lower MAE"
)

print(
    "  negative = F4 higher MAE"
)


print(
    "\nNo retraining was performed."
)

print(
    "No prediction clipping was applied."
)


print(
    "\nArtifacts written:"
)


print(
    " ",
    gru_bootstrap_summary_path
)


print(
    " ",
    gru_bootstrap_compact_summary_path
)


print(
    " ",
    gru_bootstrap_daily_block_audit_path
)


print(
    " ",
    gru_bootstrap_distributions_path
)


print(
    "\nNext step after verification:"
)

print(
    "Consolidate the completed GRU analysis and compare its"
)

print(
    "F3 -> F4 behavior with the completed LSTM results before"
)

print(
    "moving to BiLSTM."
)

GRU TEST valid-time alignment audit: PASSED
GRU bootstrap row-alignment audit: PASSED
Observed-statistic cross-check: PASSED

GRU DAILY-BLOCK POPULATION AUDIT


,horizon_minutes,scope,n_rows,n_daily_blocks,min_rows_per_block,max_rows_per_block
0,15,all,26496,92,288,288
1,15,daylight,17851,92,165,207
2,30,all,26496,92,288,288
3,30,daylight,17851,92,165,207
4,60,all,26496,92,288,288
5,60,daylight,17851,92,165,207



GRU PAIRED DAILY-BLOCK BOOTSTRAP SUMMARY


,model,horizon_minutes,scope,n_rows,n_daily_blocks,n_bootstrap,bootstrap_seed,observed_F3_MAE_kW,observed_F4_MAE_kW,observed_F3_minus_F4_MAE_kW,observed_relative_improvement_pct,MAE_difference_CI95_lower_kW,MAE_difference_CI95_upper_kW,relative_improvement_CI95_lower_pct,relative_improvement_CI95_upper_pct,bootstrap_fraction_F4_lower_MAE,paired,block_unit,model_retraining,prediction_clipping
0,GRU,15,all,26496,92,10000,42,0.291426,0.313546,-0.022119,-7.590039,-0.026917,-0.017263,-9.358493,-5.883449,0.0000,True,forecast-valid UTC calendar day,False,False
1,GRU,15,daylight,17851,92,10000,42,0.411546,0.416634,-0.005088,-1.236311,-0.009546,-0.000521,-2.286129,-0.131478,0.0145,True,forecast-valid UTC calendar day,False,False
2,GRU,30,all,26496,92,10000,42,0.391312,0.412697,-0.021385,-5.464907,-0.030153,-0.012562,-7.888403,-3.164433,0.0000,True,forecast-valid UTC calendar day,False,False
3,GRU,30,daylight,17851,92,10000,42,0.533328,0.579026,-0.045698,-8.568435,-0.054964,-0.036566,-10.803010,-6.560699,0.0000,True,forecast-valid UTC calendar day,False,False
4,GRU,60,all,26496,92,10000,42,0.446411,0.466455,-0.020043,-4.489917,-0.030918,-0.008901,-7.025055,-1.989763,0.0000,True,forecast-valid UTC calendar day,False,False
5,GRU,60,daylight,17851,92,10000,42,0.620729,0.652406,-0.031678,-5.103283,-0.044649,-0.018694,-7.317425,-2.975631,0.0000,True,forecast-valid UTC calendar day,False,False



Compact GRU F3 -> F4 bootstrap summary:


,horizon_minutes,scope,n_rows,observed_F3_MAE_kW,observed_F4_MAE_kW,observed_relative_improvement_pct,relative_improvement_CI95_lower_pct,relative_improvement_CI95_upper_pct,bootstrap_fraction_F4_lower_MAE
0,15,all,26496,0.291426,0.313546,-7.590039,-9.358493,-5.883449,0.0000
1,15,daylight,17851,0.411546,0.416634,-1.236311,-2.286129,-0.131478,0.0145
2,30,all,26496,0.391312,0.412697,-5.464907,-7.888403,-3.164433,0.0000
3,30,daylight,17851,0.533328,0.579026,-8.568435,-10.803010,-6.560699,0.0000
4,60,all,26496,0.446411,0.466455,-4.489917,-7.025055,-1.989763,0.0000
5,60,daylight,17851,0.620729,0.652406,-5.103283,-7.317425,-2.975631,0.0000



Bootstrap CI direction diagnostic:


,horizon_minutes,scope,observed_relative_improvement_pct,relative_improvement_CI95_lower_pct,relative_improvement_CI95_upper_pct,CI_direction
0,15,all,-7.590039,-9.358493,-5.883449,CI entirely below zero
1,15,daylight,-1.236311,-2.286129,-0.131478,CI entirely below zero
2,30,all,-5.464907,-7.888403,-3.164433,CI entirely below zero
3,30,daylight,-8.568435,-10.803010,-6.560699,CI entirely below zero
4,60,all,-4.489917,-7.025055,-1.989763,CI entirely below zero
5,60,daylight,-5.103283,-7.317425,-2.975631,CI entirely below zero



GRU PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE
Bootstrap replicates per comparison: 10000
UTC daily blocks per comparison: 92
Random seed: 42

Relative F3 -> F4 MAE estimates and 95% CIs:
  15-min all     : -7.5900% [-9.3585%, -5.8834%]
  15-min daylight: -1.2363% [-2.2861%, -0.1315%]
  30-min all     : -5.4649% [-7.8884%, -3.1644%]
  30-min daylight: -8.5684% [-10.8030%, -6.5607%]
  60-min all     : -4.4899% [-7.0251%, -1.9898%]
  60-min daylight: -5.1033% [-7.3174%, -2.9756%]

Interpretation of sign:
  positive = F4 lower MAE
  negative = F4 higher MAE

No retraining was performed.
No prediction clipping was applied.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_f3_f4_daily_block_bootstrap_summary.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/gru_f3_f4_daily_block_bootstrap_compact.csv
  /Use

In [100]:
# ============================================================
# Step 84: Cross-model F3 -> F4 evidence consolidation
#
# Models completed so far:
#
#       1. LSTM
#       2. GRU
#
#
# PURPOSE
# -------
#
# Compare the central Panchang ablation:
#
#       F3 -> F4
#
# across the two completed recurrent model classes.
#
#
# We compare:
#
#   - all-row held-out MAE change,
#   - daylight-only held-out MAE change,
#   - paired daily-block bootstrap 95% CI,
#   - direction of the bootstrap interval.
#
#
# IMPORTANT
# ---------
#
# This cell:
#
#   - performs NO training,
#   - generates NO new predictions,
#   - performs NO hyperparameter selection,
#   - performs NO clipping.
#
#
# Sign convention everywhere:
#
#       positive % = F4 has LOWER MAE than F3
#       negative % = F4 has HIGHER MAE than F3
#
#
# The purpose is NOT to declare a model-general conclusion yet.
#
# BiLSTM and TCN are still pending.
# ============================================================


# ------------------------------------------------------------
# 1. Horizons and scopes
# ------------------------------------------------------------

COMPLETED_MODEL_HORIZONS = [
    15,
    30,
    60,
]


COMPARISON_SCOPES = [
    "all",
    "daylight",
]


# ------------------------------------------------------------
# 2. Verify completed LSTM and GRU final prediction stores
# ------------------------------------------------------------

assert (
    "lstm_final_test_results"
    in
    globals()
)


assert (
    "gru_final_test_results"
    in
    globals()
)


assert (
    len(
        lstm_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


for horizon in COMPLETED_MODEL_HORIZONS:

    for feature_family in [
        "F3",
        "F4",
    ]:

        assert (
            (
                horizon,
                feature_family,
            )
            in
            lstm_final_test_results
        )


        assert (
            (
                horizon,
                feature_family,
            )
            in
            gru_final_test_results
        )


print(
    "LSTM and GRU final TEST stores: VERIFIED"
)


# ------------------------------------------------------------
# 3. Reconstruct all-row F3 -> F4 point estimates directly
#    from the canonical final TEST result stores.
#
# We do this instead of copying displayed numbers.
# ------------------------------------------------------------

cross_model_all_row_rows = []


for model_name, result_store in [
    (
        "LSTM",
        lstm_final_test_results,
    ),
    (
        "GRU",
        gru_final_test_results,
    ),
]:

    for horizon in COMPLETED_MODEL_HORIZONS:

        f3_mae = float(
            result_store[
                (
                    horizon,
                    "F3",
                )
            ][
                "test_MAE_kW"
            ]
        )


        f4_mae = float(
            result_store[
                (
                    horizon,
                    "F4",
                )
            ][
                "test_MAE_kW"
            ]
        )


        relative_improvement_pct = (
            (
                f3_mae
                -
                f4_mae
            )
            /
            f3_mae
            *
            100.0
        )


        cross_model_all_row_rows.append(
            {
                "model":
                    model_name,

                "horizon_minutes":
                    horizon,

                "scope":
                    "all",

                "F3_MAE_kW":
                    f3_mae,

                "F4_MAE_kW":
                    f4_mae,

                "F3_minus_F4_MAE_kW":
                    (
                        f3_mae
                        -
                        f4_mae
                    ),

                "relative_improvement_pct":
                    relative_improvement_pct,
            }
        )


cross_model_all_row = pd.DataFrame(
    cross_model_all_row_rows
)


assert (
    cross_model_all_row.shape
    ==
    (
        6,
        7,
    )
)


# ------------------------------------------------------------
# 4. Audit the reconstructed GRU all-row values against the
#    completed GRU consolidation table.
# ------------------------------------------------------------

for horizon in COMPLETED_MODEL_HORIZONS:

    reconstructed_value = float(
        cross_model_all_row.loc[
            (
                cross_model_all_row[
                    "model"
                ]
                ==
                "GRU"
            )
            &
            (
                cross_model_all_row[
                    "horizon_minutes"
                ]
                ==
                horizon
            ),
            "relative_improvement_pct",
        ]
        .iloc[
            0
        ]
    )


    stored_value = float(
        gru_final_f3_f4_test_summary.loc[
            gru_final_f3_f4_test_summary[
                "horizon_minutes"
            ]
            ==
            horizon,
            "F3_to_F4_MAE_improvement_pct",
        ]
        .iloc[
            0
        ]
    )


    np.testing.assert_allclose(
        reconstructed_value,
        stored_value,
        rtol=0.0,
        atol=1e-10,
    )


print(
    "All-row point-estimate reconstruction: PASSED"
)


# ------------------------------------------------------------
# 5. LSTM daylight point estimates
#
# These are the already completed and previously audited LSTM
# daylight results.
#
# They are recorded here explicitly so this cross-model table
# remains self-contained and reproducible.
# ------------------------------------------------------------

lstm_daylight_reference = pd.DataFrame(
    [
        {
            "model":
                "LSTM",

            "horizon_minutes":
                15,

            "scope":
                "daylight",

            "relative_improvement_pct":
                1.263797,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                30,

            "scope":
                "daylight",

            "relative_improvement_pct":
                4.174850,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                60,

            "scope":
                "daylight",

            "relative_improvement_pct":
                3.449669,
        },
    ]
)


# ------------------------------------------------------------
# 6. GRU daylight point estimates come directly from Cell 86.
# ------------------------------------------------------------

gru_daylight_reference = (
    gru_daylight_f3_f4_summary[
        [
            "horizon_minutes",
            "F3_to_F4_daylight_MAE_improvement_pct",
        ]
    ]
    .copy()
)


gru_daylight_reference.insert(
    0,
    "model",
    "GRU",
)


gru_daylight_reference[
    "scope"
] = "daylight"


gru_daylight_reference = (
    gru_daylight_reference.rename(
        columns={
            (
                "F3_to_F4_daylight_"
                "MAE_improvement_pct"
            ):
                "relative_improvement_pct",
        }
    )
)


gru_daylight_reference = (
    gru_daylight_reference[
        [
            "model",
            "horizon_minutes",
            "scope",
            "relative_improvement_pct",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Combine daylight point estimates
# ------------------------------------------------------------

cross_model_daylight = pd.concat(
    [
        lstm_daylight_reference,
        gru_daylight_reference,
    ],
    ignore_index=True,
)


assert (
    cross_model_daylight.shape
    ==
    (
        6,
        4,
    )
)


# ------------------------------------------------------------
# 8. LSTM paired daily-block bootstrap reference
#
# These are the previously completed LSTM 10,000-replicate,
# 92-day-block, seed-42 percentile confidence intervals.
#
# Sign convention:
#
#       positive = F4 lower MAE
#       negative = F4 higher MAE
# ------------------------------------------------------------

lstm_bootstrap_reference = pd.DataFrame(
    [
        {
            "model":
                "LSTM",

            "horizon_minutes":
                15,

            "scope":
                "all",

            "relative_CI95_lower_pct":
                -2.591356,

            "relative_CI95_upper_pct":
                0.800759,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                15,

            "scope":
                "daylight",

            "relative_CI95_lower_pct":
                0.040153,

            "relative_CI95_upper_pct":
                2.557423,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                30,

            "scope":
                "all",

            "relative_CI95_lower_pct":
                3.552636,

            "relative_CI95_upper_pct":
                7.939762,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                30,

            "scope":
                "daylight",

            "relative_CI95_lower_pct":
                1.978721,

            "relative_CI95_upper_pct":
                6.174997,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                60,

            "scope":
                "all",

            "relative_CI95_lower_pct":
                -3.451026,

            "relative_CI95_upper_pct":
                0.566087,
        },

        {
            "model":
                "LSTM",

            "horizon_minutes":
                60,

            "scope":
                "daylight",

            "relative_CI95_lower_pct":
                1.664418,

            "relative_CI95_upper_pct":
                5.347754,
        },
    ]
)


# ------------------------------------------------------------
# 9. GRU bootstrap values come directly from Cell 87.
# ------------------------------------------------------------

gru_bootstrap_reference = (
    gru_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "relative_improvement_CI95_lower_pct",
            "relative_improvement_CI95_upper_pct",
        ]
    ]
    .copy()
)


gru_bootstrap_reference = (
    gru_bootstrap_reference.rename(
        columns={
            "relative_improvement_CI95_lower_pct":
                "relative_CI95_lower_pct",

            "relative_improvement_CI95_upper_pct":
                "relative_CI95_upper_pct",
        }
    )
)


# ------------------------------------------------------------
# 10. Assemble point estimates for BOTH models and BOTH scopes
# ------------------------------------------------------------

cross_model_point_rows = []


# LSTM all-row values directly reconstructed from final results.
for row in (
    cross_model_all_row.loc[
        cross_model_all_row[
            "model"
        ]
        ==
        "LSTM"
    ]
    .itertuples(
        index=False
    )
):

    cross_model_point_rows.append(
        {
            "model":
                "LSTM",

            "horizon_minutes":
                int(
                    row.horizon_minutes
                ),

            "scope":
                "all",

            "relative_improvement_pct":
                float(
                    row.relative_improvement_pct
                ),
        }
    )


# GRU all-row values directly reconstructed from final results.
for row in (
    cross_model_all_row.loc[
        cross_model_all_row[
            "model"
        ]
        ==
        "GRU"
    ]
    .itertuples(
        index=False
    )
):

    cross_model_point_rows.append(
        {
            "model":
                "GRU",

            "horizon_minutes":
                int(
                    row.horizon_minutes
                ),

            "scope":
                "all",

            "relative_improvement_pct":
                float(
                    row.relative_improvement_pct
                ),
        }
    )


# Add daylight values.
for row in (
    cross_model_daylight
    .itertuples(
        index=False
    )
):

    cross_model_point_rows.append(
        {
            "model":
                str(
                    row.model
                ),

            "horizon_minutes":
                int(
                    row.horizon_minutes
                ),

            "scope":
                str(
                    row.scope
                ),

            "relative_improvement_pct":
                float(
                    row.relative_improvement_pct
                ),
        }
    )


cross_model_points = pd.DataFrame(
    cross_model_point_rows
)


assert (
    cross_model_points.shape
    ==
    (
        12,
        4,
    )
)


# ------------------------------------------------------------
# 11. Combine bootstrap intervals for both model classes
# ------------------------------------------------------------

cross_model_bootstrap = pd.concat(
    [
        lstm_bootstrap_reference,
        gru_bootstrap_reference,
    ],
    ignore_index=True,
)


assert (
    cross_model_bootstrap.shape
    ==
    (
        12,
        5,
    )
)


# ------------------------------------------------------------
# 12. Merge point estimates with bootstrap intervals
# ------------------------------------------------------------

lstm_gru_f3_f4_comparison = (
    cross_model_points.merge(
        cross_model_bootstrap,
        on=[
            "model",
            "horizon_minutes",
            "scope",
        ],
        how="inner",
        validate="one_to_one",
    )
)


assert (
    lstm_gru_f3_f4_comparison.shape
    ==
    (
        12,
        6,
    )
)


# Canonical ordering.
lstm_gru_f3_f4_comparison[
    "model"
] = pd.Categorical(
    lstm_gru_f3_f4_comparison[
        "model"
    ],
    categories=[
        "LSTM",
        "GRU",
    ],
    ordered=True,
)


lstm_gru_f3_f4_comparison[
    "scope"
] = pd.Categorical(
    lstm_gru_f3_f4_comparison[
        "scope"
    ],
    categories=[
        "all",
        "daylight",
    ],
    ordered=True,
)


lstm_gru_f3_f4_comparison = (
    lstm_gru_f3_f4_comparison.sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 13. Bootstrap CI-direction classification
# ------------------------------------------------------------

def classify_cross_model_ci(
    lower,
    upper,
):

    if lower > 0.0:

        return "F4 lower MAE across CI"

    if upper < 0.0:

        return "F4 higher MAE across CI"

    return "CI crosses zero"


lstm_gru_f3_f4_comparison[
    "CI_direction"
] = [
    classify_cross_model_ci(
        lower,
        upper,
    )

    for lower, upper in zip(
        lstm_gru_f3_f4_comparison[
            "relative_CI95_lower_pct"
        ],

        lstm_gru_f3_f4_comparison[
            "relative_CI95_upper_pct"
        ],
    )
]


# ------------------------------------------------------------
# 14. Point-estimate direction
# ------------------------------------------------------------

lstm_gru_f3_f4_comparison[
    "point_estimate_direction"
] = np.where(
    lstm_gru_f3_f4_comparison[
        "relative_improvement_pct"
    ]
    >
    0.0,

    "F4 lower MAE",

    np.where(
        lstm_gru_f3_f4_comparison[
            "relative_improvement_pct"
        ]
        <
        0.0,

        "F4 higher MAE",

        "equal MAE",
    ),
)


# ------------------------------------------------------------
# 15. Display the complete cross-model comparison
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "LSTM vs GRU: F3 -> F4 HELD-OUT EVIDENCE"
)

print(
    "============================================================"
)


display(
    lstm_gru_f3_f4_comparison
)


# ------------------------------------------------------------
# 16. Pivoted presentation
#
# Makes the architecture disagreement especially easy to see.
# ------------------------------------------------------------

lstm_gru_f3_f4_pivot = (
    lstm_gru_f3_f4_comparison.pivot(
        index=[
            "horizon_minutes",
            "scope",
        ],
        columns="model",
        values="relative_improvement_pct",
    )
    .reset_index()
)


lstm_gru_f3_f4_pivot.columns.name = None


print(
    "\nF3 -> F4 relative MAE change by model:"
)


display(
    lstm_gru_f3_f4_pivot
)


# ------------------------------------------------------------
# 17. Determine whether LSTM and GRU point-estimate signs agree
#     at each horizon × scope.
#
# This is descriptive only.
# ------------------------------------------------------------

lstm_gru_f3_f4_pivot[
    "same_sign"
] = (
    np.sign(
        lstm_gru_f3_f4_pivot[
            "LSTM"
        ]
    )
    ==
    np.sign(
        lstm_gru_f3_f4_pivot[
            "GRU"
        ]
    )
)


print(
    "\nPoint-estimate sign agreement:"
)


display(
    lstm_gru_f3_f4_pivot
)


# ------------------------------------------------------------
# 18. Expected qualitative pattern audit
#
# Based on the completed, frozen analyses:
#
# LSTM:
#   - 15 all:       interval crosses zero
#   - 15 daylight:  interval above zero
#   - 30 all:       interval above zero
#   - 30 daylight:  interval above zero
#   - 60 all:       interval crosses zero
#   - 60 daylight:  interval above zero
#
# GRU:
#   - every interval is entirely below zero
#
# These assertions protect against accidental table-merging or
# sign-convention mistakes.
# ------------------------------------------------------------

expected_lstm_ci_directions = {
    (15, "all"):
        "CI crosses zero",

    (15, "daylight"):
        "F4 lower MAE across CI",

    (30, "all"):
        "F4 lower MAE across CI",

    (30, "daylight"):
        "F4 lower MAE across CI",

    (60, "all"):
        "CI crosses zero",

    (60, "daylight"):
        "F4 lower MAE across CI",
}


expected_gru_ci_direction = (
    "F4 higher MAE across CI"
)


for (
    horizon,
    scope,
), expected_direction in (
    expected_lstm_ci_directions.items()
):

    observed_direction = (
        lstm_gru_f3_f4_comparison.loc[
            (
                lstm_gru_f3_f4_comparison[
                    "model"
                ]
                ==
                "LSTM"
            )
            &
            (
                lstm_gru_f3_f4_comparison[
                    "horizon_minutes"
                ]
                ==
                horizon
            )
            &
            (
                lstm_gru_f3_f4_comparison[
                    "scope"
                ]
                ==
                scope
            ),
            "CI_direction",
        ]
        .iloc[
            0
        ]
    )


    assert (
        observed_direction
        ==
        expected_direction
    )


gru_ci_directions = (
    lstm_gru_f3_f4_comparison.loc[
        lstm_gru_f3_f4_comparison[
            "model"
        ]
        ==
        "GRU",
        "CI_direction",
    ]
    .tolist()
)


assert all(
    direction
    ==
    expected_gru_ci_direction

    for direction
    in gru_ci_directions
)


print(
    "Cross-model CI-direction audit: PASSED"
)


# ------------------------------------------------------------
# 19. Persist cross-model comparison
# ------------------------------------------------------------

lstm_gru_f3_f4_comparison_path = (
    RESULTS_DIR
    /
    "lstm_gru_f3_f4_comparison.csv"
)


lstm_gru_f3_f4_pivot_path = (
    RESULTS_DIR
    /
    "lstm_gru_f3_f4_point_estimate_pivot.csv"
)


lstm_gru_f3_f4_comparison.to_csv(
    lstm_gru_f3_f4_comparison_path,
    index=False,
)


lstm_gru_f3_f4_pivot.to_csv(
    lstm_gru_f3_f4_pivot_path,
    index=False,
)


for output_path in [
    lstm_gru_f3_f4_comparison_path,
    lstm_gru_f3_f4_pivot_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 20. Final scientific checkpoint
#
# We deliberately keep this interpretation cautious.
#
# Two model classes are not enough to claim a model-general
# Panchang effect.
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "LSTM + GRU CROSS-MODEL CONSOLIDATION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "\nLSTM F3 -> F4 behavior:"
)

print(
    "  Daylight point estimates favor F4 at 15, 30, and 60 min."
)

print(
    "  Daylight bootstrap CIs are above zero at all 3 horizons."
)

print(
    "  All-row evidence is mixed:"
)

print(
    "    15-min CI crosses zero;"
)

print(
    "    30-min CI is above zero;"
)

print(
    "    60-min CI crosses zero."
)


print(
    "\nGRU F3 -> F4 behavior:"
)

print(
    "  F4 has higher MAE at every horizon in both scopes."
)

print(
    "  All 6 GRU bootstrap CIs are entirely below zero."
)


print(
    "\nCurrent scientific implication:"
)

print(
    "  The F3 -> F4 effect is architecture-dependent so far."
)

print(
    "  Therefore a model-general Panchang benefit is NOT yet"
)

print(
    "  supported by the completed LSTM + GRU evidence."
)


print(
    "\nNo retraining was performed."
)

print(
    "No prediction clipping was applied."
)


print(
    "\nArtifacts written:"
)


print(
    " ",
    lstm_gru_f3_f4_comparison_path
)


print(
    " ",
    lstm_gru_f3_f4_pivot_path
)


print(
    "\nNext step after verification:"
)

print(
    "Begin BiLSTM with the same leakage-safe sequence protocol,"
)

print(
    "starting from architecture-selection preflight."
)

LSTM and GRU final TEST stores: VERIFIED
All-row point-estimate reconstruction: PASSED

LSTM vs GRU: F3 -> F4 HELD-OUT EVIDENCE


,model,horizon_minutes,scope,relative_improvement_pct,relative_CI95_lower_pct,relative_CI95_upper_pct,CI_direction,point_estimate_direction
0,LSTM,15,all,-0.884545,-2.591356,0.800759,CI crosses zero,F4 higher MAE
1,LSTM,15,daylight,1.263797,0.040153,2.557423,F4 lower MAE across CI,F4 lower MAE
2,LSTM,30,all,5.797532,3.552636,7.939762,F4 lower MAE across CI,F4 lower MAE
3,LSTM,30,daylight,4.174850,1.978721,6.174997,F4 lower MAE across CI,F4 lower MAE
4,LSTM,60,all,-1.441883,-3.451026,0.566087,CI crosses zero,F4 higher MAE
5,LSTM,60,daylight,3.449669,1.664418,5.347754,F4 lower MAE across CI,F4 lower MAE
6,GRU,15,all,-7.590039,-9.358493,-5.883449,F4 higher MAE across CI,F4 higher MAE
7,GRU,15,daylight,-1.236311,-2.286129,-0.131478,F4 higher MAE across CI,F4 higher MAE
8,GRU,30,all,-5.464907,-7.888403,-3.164433,F4 higher MAE across CI,F4 higher MAE
9,GRU,30,daylight,-8.568435,-10.803010,-6.560699,F4 higher MAE across CI,F4 higher MAE



F3 -> F4 relative MAE change by model:


,horizon_minutes,scope,LSTM,GRU
0,15,all,-0.884545,-7.590039
1,15,daylight,1.263797,-1.236311
2,30,all,5.797532,-5.464907
3,30,daylight,4.174850,-8.568435
4,60,all,-1.441883,-4.489917
5,60,daylight,3.449669,-5.103283



Point-estimate sign agreement:


,horizon_minutes,scope,LSTM,GRU,same_sign
0,15,all,-0.884545,-7.590039,True
1,15,daylight,1.263797,-1.236311,False
2,30,all,5.797532,-5.464907,False
3,30,daylight,4.174850,-8.568435,False
4,60,all,-1.441883,-4.489917,True
5,60,daylight,3.449669,-5.103283,False


Cross-model CI-direction audit: PASSED

LSTM + GRU CROSS-MODEL CONSOLIDATION: COMPLETE

LSTM F3 -> F4 behavior:
  Daylight point estimates favor F4 at 15, 30, and 60 min.
  Daylight bootstrap CIs are above zero at all 3 horizons.
  All-row evidence is mixed:
    15-min CI crosses zero;
    30-min CI is above zero;
    60-min CI crosses zero.

GRU F3 -> F4 behavior:
  F4 has higher MAE at every horizon in both scopes.
  All 6 GRU bootstrap CIs are entirely below zero.

Current scientific implication:
  The F3 -> F4 effect is architecture-dependent so far.
  Therefore a model-general Panchang benefit is NOT yet
  supported by the completed LSTM + GRU evidence.

No retraining was performed.
No prediction clipping was applied.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_gru_f3_f4_comparison.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Pro

In [101]:
# ============================================================
# Step 85: BiLSTM preflight and leakage-safety audit
#
# PURPOSE
# -------
#
# Begin the third deep-learning model class:
#
#       Bidirectional LSTM (BiLSTM)
#
#
# This cell performs:
#
#   1. protocol / completion checks,
#   2. sequence-shape checks,
#   3. leakage-safety documentation,
#   4. recurrent architecture-candidate checks,
#   5. one untrained BiLSTM smoke-build,
#   6. parameter-count audit.
#
#
# IMPORTANT
# ---------
#
# NO model is trained in this cell.
# NO validation result is generated.
# NO held-out TEST data are evaluated.
#
#
# WHY BiLSTM IS LEAKAGE-SAFE HERE
# -------------------------------
#
# Each sequence contains ONLY historical information:
#
#       [t-60, t-55, ..., t-5, t]
#
# relative to the forecast issue time t.
#
# The backward LSTM direction traverses this SAME historical
# window in reverse order:
#
#       [t, t-5, ..., t-60]
#
# It does NOT access:
#
#       t+5, t+10, ..., or the forecast target time.
#
# Therefore bidirectional processing is confined entirely to
# observations already available at issue time.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm previous model families are complete
# ------------------------------------------------------------

assert (
    "lstm_final_test_results"
    in
    globals()
)


assert (
    "gru_final_test_results"
    in
    globals()
)


assert (
    len(
        lstm_final_test_results
    )
    ==
    15
)


assert (
    len(
        gru_final_test_results
    )
    ==
    15
)


print(
    "Completed model families before BiLSTM:"
)

print(
    "  LSTM: 15 / 15 final TEST evaluations"
)

print(
    "  GRU : 15 / 15 final TEST evaluations"
)


# ------------------------------------------------------------
# 2. Freeze BiLSTM model-selection protocol
#
# Exactly the same architecture candidate family used for the
# preceding recurrent models is reused.
#
# Architecture selection occurs ONLY on:
#
#       horizon = 60 minutes
#       family  = F0
#
# Ranking:
#
#       1. lowest validation MAE
#       2. lowest validation RMSE
#       3. fewer trainable parameters
#
# TEST is not involved.
# ------------------------------------------------------------

BILSTM_ARCHITECTURE_SELECTION_HORIZON = 60

BILSTM_ARCHITECTURE_SELECTION_FAMILY = "F0"


BILSTM_ARCHITECTURE_SELECTION_RANKING = [
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "trainable_parameter_count",
]


assert (
    BILSTM_ARCHITECTURE_SELECTION_HORIZON
    ==
    60
)


assert (
    BILSTM_ARCHITECTURE_SELECTION_FAMILY
    ==
    "F0"
)


# ------------------------------------------------------------
# 3. Verify recurrent architecture candidates
#
# These must remain the same candidate definitions already used
# for LSTM / GRU architecture selection.
# ------------------------------------------------------------

expected_recurrent_candidate_ids = [
    "RNN_C1",
    "RNN_C2",
    "RNN_C3",
    "RNN_C4",
]


actual_recurrent_candidate_ids = [
    candidate[
        "candidate_id"
    ]

    for candidate in (
        RECURRENT_ARCHITECTURE_CANDIDATES
    )
]


assert (
    actual_recurrent_candidate_ids
    ==
    expected_recurrent_candidate_ids
)


# Expected candidate definitions.
expected_recurrent_candidate_configs = {
    "RNN_C1": {
        "units": 32,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    },

    "RNN_C2": {
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    },

    "RNN_C3": {
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.2,
        "fusion_dense_units": 32,
    },

    "RNN_C4": {
        "units": 64,
        "n_recurrent_layers": 2,
        "dropout": 0.2,
        "fusion_dense_units": 32,
    },
}


for candidate in (
    RECURRENT_ARCHITECTURE_CANDIDATES
):

    candidate_id = candidate[
        "candidate_id"
    ]


    expected_config = (
        expected_recurrent_candidate_configs[
            candidate_id
        ]
    )


    for parameter_name, expected_value in (
        expected_config.items()
    ):

        assert (
            candidate[
                parameter_name
            ]
            ==
            expected_value
        )


print(
    "\nBiLSTM recurrent candidate definitions: VERIFIED"
)


# ------------------------------------------------------------
# 4. Retrieve ONLY TRAIN data for the architecture-selection
#    configuration.
#
# No TEST data are requested here.
# ------------------------------------------------------------

bilstm_preflight_train_data = (
    get_deep_split_data(
        horizon_minutes=
            BILSTM_ARCHITECTURE_SELECTION_HORIZON,

        feature_family=
            BILSTM_ARCHITECTURE_SELECTION_FAMILY,

        split_name=
            "train",
    )
)


# ------------------------------------------------------------
# 5. Inspect canonical data shapes
# ------------------------------------------------------------

bilstm_preflight_sequence = (
    bilstm_preflight_train_data[
        "sequence"
    ]
)


bilstm_preflight_auxiliary = (
    bilstm_preflight_train_data[
        "auxiliary"
    ]
)


bilstm_preflight_target_scaled = (
    bilstm_preflight_train_data[
        "target_scaled"
    ]
)


bilstm_preflight_target_kw = (
    bilstm_preflight_train_data[
        "target_kw"
    ]
)


# ------------------------------------------------------------
# 6. Sequence-shape audit
#
# Expected:
#
#       rows      = 87,252
#       timesteps = 13
#       channels  = 9
#
# The 13 timesteps correspond to the historical 60-minute
# observation window sampled at 5-minute resolution:
#
#       t-60, t-55, ..., t-5, t
# ------------------------------------------------------------

assert (
    bilstm_preflight_sequence.shape
    ==
    (
        87_252,
        13,
        9,
    )
)


assert (
    bilstm_preflight_auxiliary.shape
    ==
    (
        87_252,
        1,
    )
)


assert (
    bilstm_preflight_target_scaled.shape[
        0
    ]
    ==
    87_252
)


assert (
    bilstm_preflight_target_kw.shape[
        0
    ]
    ==
    87_252
)


assert np.isfinite(
    bilstm_preflight_sequence
).all()


assert np.isfinite(
    bilstm_preflight_auxiliary
).all()


assert np.isfinite(
    bilstm_preflight_target_scaled
).all()


assert np.isfinite(
    bilstm_preflight_target_kw
).all()


print(
    "\nBiLSTM TRAIN input audit:"
)

print(
    "  sequence shape:",
    bilstm_preflight_sequence.shape,
)

print(
    "  auxiliary shape:",
    bilstm_preflight_auxiliary.shape,
)

print(
    "  target_scaled rows:",
    bilstm_preflight_target_scaled.shape[
        0
    ],
)

print(
    "  target_kw rows:",
    bilstm_preflight_target_kw.shape[
        0
    ],
)


# ------------------------------------------------------------
# 7. Explicit leakage-safety declaration
#
# The BiLSTM is bidirectional WITHIN the historical sequence.
#
# It is NOT bidirectional across future chronological samples.
#
# Therefore:
#
#       backward traversal endpoint = t
#
# and never:
#
#       t + forecast horizon
# ------------------------------------------------------------

BILSTM_SEQUENCE_PROTOCOL = {
    "n_timesteps":
        13,

    "n_sequence_channels":
        9,

    "history_window_minutes":
        60,

    "sequence_endpoint":
        "forecast issue time t",

    "contains_future_observations":
        False,

    "bidirectional_scope":
        "within historical sequence only",

    "future_same_time_GHI_or_POA":
        False,

    "test_used_for_architecture_selection":
        False,
}


assert (
    BILSTM_SEQUENCE_PROTOCOL[
        "contains_future_observations"
    ]
    is False
)


assert (
    BILSTM_SEQUENCE_PROTOCOL[
        "future_same_time_GHI_or_POA"
    ]
    is False
)


assert (
    BILSTM_SEQUENCE_PROTOCOL[
        "test_used_for_architecture_selection"
    ]
    is False
)


print(
    "\nBiLSTM leakage-safety protocol:"
)


display(
    pd.DataFrame(
        [
            BILSTM_SEQUENCE_PROTOCOL
        ]
    )
)


# ------------------------------------------------------------
# 8. Smoke-build the smallest candidate
#
# This checks that the common model builder correctly supports:
#
#       model_name = "BiLSTM"
#
# No fitting occurs.
# ------------------------------------------------------------

BILSTM_SMOKE_CANDIDATE = (
    RECURRENT_ARCHITECTURE_CANDIDATES[
        0
    ]
)


assert (
    BILSTM_SMOKE_CANDIDATE[
        "candidate_id"
    ]
    ==
    "RNN_C1"
)


tf.keras.backend.clear_session()


bilstm_smoke_model = (
    build_deep_forecasting_model(
        model_name=
            "BiLSTM",

        auxiliary_dim=
            1,

        candidate_config=
            BILSTM_SMOKE_CANDIDATE,

        seed=
            42,
    )
)


# ------------------------------------------------------------
# 9. Smoke-model parameter audit
#
# For RNN_C1:
#
#       units                  = 32
#       recurrent layers       = 1
#       sequence channels      = 9
#       BiLSTM directions      = 2
#
#
# One 32-unit LSTM:
#
#       4 * units * (input_dim + units + 1)
#
#       = 4 * 32 * (9 + 32 + 1)
#       = 5,376
#
#
# Bidirectional recurrent block:
#
#       2 * 5,376 = 10,752
#
#
# Concatenated BiLSTM output:
#
#       32 + 32 = 64
#
# plus F0 auxiliary width:
#
#       64 + 1 = 65
#
#
# Fusion Dense(32):
#
#       65 * 32 + 32 = 2,112
#
#
# Output Dense(1):
#
#       32 * 1 + 1 = 33
#
#
# Total:
#
#       10,752 + 2,112 + 33
#       = 12,897
# ------------------------------------------------------------

BILSTM_RNN_C1_F0_EXPECTED_PARAMS = (
    12_897
)


bilstm_smoke_parameter_count = int(
    bilstm_smoke_model.count_params()
)


assert (
    bilstm_smoke_parameter_count
    ==
    BILSTM_RNN_C1_F0_EXPECTED_PARAMS
)


print(
    "\nBiLSTM smoke build:"
)

print(
    "  candidate:",
    BILSTM_SMOKE_CANDIDATE[
        "candidate_id"
    ],
)

print(
    "  trainable parameter count:",
    bilstm_smoke_parameter_count,
)

print(
    "  expected parameter count:",
    BILSTM_RNN_C1_F0_EXPECTED_PARAMS,
)


# ------------------------------------------------------------
# 10. Verify model input structure
#
# Expected:
#
#       historical sequence input:
#           (None, 13, 9)
#
#       auxiliary input:
#           (None, 1)
# ------------------------------------------------------------

bilstm_smoke_input_shapes = [
    tuple(
        input_tensor.shape
    )

    for input_tensor in (
        bilstm_smoke_model.inputs
    )
]


assert (
    bilstm_smoke_input_shapes[
        0
    ][
        1:
    ]
    ==
    (
        13,
        9,
    )
)


assert (
    bilstm_smoke_input_shapes[
        1
    ][
        1:
    ]
    ==
    (
        1,
    )
)


print(
    "  input shapes:",
    bilstm_smoke_input_shapes,
)


# ------------------------------------------------------------
# 11. Confirm scalar regression output
# ------------------------------------------------------------

assert (
    tuple(
        bilstm_smoke_model.output.shape
    )[
        1:
    ]
    ==
    (
        1,
    )
)


print(
    "  output shape:",
    tuple(
        bilstm_smoke_model.output.shape
    ),
)


# ------------------------------------------------------------
# 12. Clean up smoke model
#
# The architecture-selection cell will create fresh models
# under the deterministic seed protocol.
# ------------------------------------------------------------

del bilstm_smoke_model


tf.keras.backend.clear_session()


# ------------------------------------------------------------
# 13. Preflight checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM PREFLIGHT: PASSED"
)

print(
    "============================================================"
)


print(
    "Architecture-selection configuration:"
)

print(
    "  horizon: 60 minutes"
)

print(
    "  feature family: F0"
)

print(
    "  candidate count:",
    len(
        RECURRENT_ARCHITECTURE_CANDIDATES
    ),
)


print(
    "\nHistorical sequence:"
)

print(
    "  13 timesteps"
)

print(
    "  9 observed channels"
)

print(
    "  endpoint = forecast issue time t"
)

print(
    "  no future observations"
)


print(
    "\nBiLSTM interpretation:"
)

print(
    "  forward and backward passes operate only inside the"
)

print(
    "  already-observed historical sequence."
)


print(
    "\nSmoke-build:"
)

print(
    "  RNN_C1 parameter count:",
    BILSTM_RNN_C1_F0_EXPECTED_PARAMS,
)


print(
    "\nNo fitting was performed."
)

print(
    "No validation metric was inspected."
)

print(
    "Held-out TEST was not used."
)


print(
    "\nNext step after verification:"
)

print(
    "BiLSTM architecture selection on 60-minute F0 using"
)

print(
    "RNN_C1 through RNN_C4 and validation-only ranking."
)

Completed model families before BiLSTM:
  LSTM: 15 / 15 final TEST evaluations
  GRU : 15 / 15 final TEST evaluations

BiLSTM recurrent candidate definitions: VERIFIED

BiLSTM TRAIN input audit:
  sequence shape: (87252, 13, 9)
  auxiliary shape: (87252, 1)
  target_scaled rows: 87252
  target_kw rows: 87252

BiLSTM leakage-safety protocol:


,n_timesteps,n_sequence_channels,history_window_minutes,sequence_endpoint,contains_future_observations,bidirectional_scope,future_same_time_GHI_or_POA,test_used_for_architecture_selection
0,13,9,60,forecast issue time t,False,within historical sequence only,False,False



BiLSTM smoke build:
  candidate: RNN_C1
  trainable parameter count: 12897
  expected parameter count: 12897
  input shapes: [(None, 13, 9), (None, 1)]
  output shape: (None, 1)

BiLSTM PREFLIGHT: PASSED
Architecture-selection configuration:
  horizon: 60 minutes
  feature family: F0
  candidate count: 4

Historical sequence:
  13 timesteps
  9 observed channels
  endpoint = forecast issue time t
  no future observations

BiLSTM interpretation:
  forward and backward passes operate only inside the
  already-observed historical sequence.

Smoke-build:
  RNN_C1 parameter count: 12897

No fitting was performed.
No validation metric was inspected.
Held-out TEST was not used.

Next step after verification:
BiLSTM architecture selection on 60-minute F0 using
RNN_C1 through RNN_C4 and validation-only ranking.


In [102]:
# ============================================================
# Step 86: BiLSTM architecture selection
#
# Selection configuration:
#
#       model          = BiLSTM
#       horizon        = 60 minutes
#       feature family = F0
#
#
# Candidate architectures:
#
#       RNN_C1
#       RNN_C2
#       RNN_C3
#       RNN_C4
#
#
# VALIDATION-ONLY ranking:
#
#       1. lowest validation MAE
#       2. lowest validation RMSE
#       3. fewer trainable parameters
#
#
# IMPORTANT
# ---------
#
# Held-out TEST is NOT used in this cell.
#
# The architecture selected here will be frozen for every
# later BiLSTM:
#
#       horizon × feature-family
#
# experiment.
#
# Epoch counts are NOT frozen globally here.
# They will later be selected separately from validation for
# each of the 15 horizon × feature-family experiments.
# ============================================================


import json


# ------------------------------------------------------------
# 1. Frozen architecture-selection identity
# ------------------------------------------------------------

BILSTM_ARCH_SELECTION_HORIZON = 60

BILSTM_ARCH_SELECTION_FAMILY = "F0"

BILSTM_ARCH_SELECTION_MODEL = "BiLSTM"


assert (
    BILSTM_ARCH_SELECTION_HORIZON
    ==
    60
)


assert (
    BILSTM_ARCH_SELECTION_FAMILY
    ==
    "F0"
)


# ------------------------------------------------------------
# 2. Confirm no BiLSTM final TEST state exists yet
#
# We are still strictly in model-selection territory.
# ------------------------------------------------------------

assert (
    "bilstm_final_test_results"
    not in
    globals()
), (
    "BiLSTM final TEST containers already exist unexpectedly."
)


# ------------------------------------------------------------
# 3. Expected trainable parameter counts
#
# These are deterministic consequences of the candidate
# architectures and the F0 auxiliary width of 1.
#
#
# RNN_C1:
#       BiLSTM(32) × 1
#       = 12,897
#
#
# RNN_C2 / RNN_C3:
#       BiLSTM(64) × 1
#       = 42,081
#
#
# RNN_C4:
#       stacked BiLSTM(64) × 2
#       = 140,897
#
#
# Dropout changes no trainable parameter count, therefore
# RNN_C2 and RNN_C3 must have identical counts.
# ------------------------------------------------------------

BILSTM_EXPECTED_ARCH_SELECTION_PARAMS = {
    "RNN_C1": 12_897,
    "RNN_C2": 42_081,
    "RNN_C3": 42_081,
    "RNN_C4": 140_897,
}


# ------------------------------------------------------------
# 4. Verify candidate definitions once more
# ------------------------------------------------------------

assert (
    len(
        RECURRENT_ARCHITECTURE_CANDIDATES
    )
    ==
    4
)


assert (
    [
        candidate[
            "candidate_id"
        ]

        for candidate in (
            RECURRENT_ARCHITECTURE_CANDIDATES
        )
    ]
    ==
    [
        "RNN_C1",
        "RNN_C2",
        "RNN_C3",
        "RNN_C4",
    ]
)


# ------------------------------------------------------------
# 5. Storage
# ------------------------------------------------------------

bilstm_architecture_selection_results = []

bilstm_architecture_selection_histories = {}


# ------------------------------------------------------------
# 6. Run all four candidates
#
# The existing canonical helper:
#
#       run_validation_candidate(...)
#
# uses the frozen training protocol:
#
#       batch size      = 256
#       max epochs      = 80
#       Adam LR         = 1e-3
#       clipnorm        = 1.0
#       loss            = standardized MAE
#       patience        = 8
#       min_delta       = 1e-4
#       restore best    = True
#       shuffle         = True
#       seed            = 42
#
#
# Only TRAIN and VALIDATION are involved.
# ------------------------------------------------------------

for candidate_config in (
    RECURRENT_ARCHITECTURE_CANDIDATES
):

    candidate_id = (
        candidate_config[
            "candidate_id"
        ]
    )


    print(
        "\n============================================================"
    )

    print(
        "BiLSTM architecture candidate:",
        candidate_id,
    )

    print(
        "============================================================"
    )


    (
        candidate_result,
        candidate_history,
    ) = run_validation_candidate(
        model_name=
            BILSTM_ARCH_SELECTION_MODEL,

        candidate_config=
            candidate_config,

        horizon_minutes=
            BILSTM_ARCH_SELECTION_HORIZON,

        feature_family=
            BILSTM_ARCH_SELECTION_FAMILY,

        verbose=
            2,
    )


    # --------------------------------------------------------
    # Basic result identity checks
    # --------------------------------------------------------

    assert (
        candidate_result[
            "model"
        ]
        ==
        "BiLSTM"
    )


    assert (
        candidate_result[
            "candidate_id"
        ]
        ==
        candidate_id
    )


    assert (
        candidate_result[
            "horizon_minutes"
        ]
        ==
        60
    )


    assert (
        candidate_result[
            "feature_family"
        ]
        ==
        "F0"
    )


    assert (
        candidate_result[
            "auxiliary_dim"
        ]
        ==
        1
    )


    # --------------------------------------------------------
    # Parameter-count audit
    # --------------------------------------------------------

    expected_parameter_count = (
        BILSTM_EXPECTED_ARCH_SELECTION_PARAMS[
            candidate_id
        ]
    )


    assert (
        candidate_result[
            "trainable_parameter_count"
        ]
        ==
        expected_parameter_count
    )


    # --------------------------------------------------------
    # Validation metric sanity checks
    # --------------------------------------------------------

    for metric_name in [
        "validation_MAE_kW",
        "validation_RMSE_kW",
        "validation_R2",
    ]:

        assert np.isfinite(
            candidate_result[
                metric_name
            ]
        )


    # --------------------------------------------------------
    # Early-stopping / history sanity checks
    # --------------------------------------------------------

    assert (
        candidate_result[
            "epochs_ran"
        ]
        ==
        len(
            candidate_history
        )
    )


    assert (
        1
        <=
        candidate_result[
            "best_epoch"
        ]
        <=
        candidate_result[
            "epochs_ran"
        ]
    )


    assert (
        candidate_result[
            "epochs_ran"
        ]
        <=
        80
    )


    assert (
        "loss"
        in
        candidate_history.columns
    )


    assert (
        "val_loss"
        in
        candidate_history.columns
    )


    # --------------------------------------------------------
    # Save canonical result and history
    # --------------------------------------------------------

    bilstm_architecture_selection_results.append(
        candidate_result.copy()
    )


    bilstm_architecture_selection_histories[
        candidate_id
    ] = (
        candidate_history.copy()
    )


# ------------------------------------------------------------
# 7. Construct architecture-selection table
# ------------------------------------------------------------

bilstm_architecture_selection_table = (
    pd.DataFrame(
        bilstm_architecture_selection_results
    )
)


assert (
    len(
        bilstm_architecture_selection_table
    )
    ==
    4
)


# ------------------------------------------------------------
# 8. Confirm one result exists for every candidate
# ------------------------------------------------------------

assert (
    set(
        bilstm_architecture_selection_table[
            "candidate_id"
        ]
    )
    ==
    {
        "RNN_C1",
        "RNN_C2",
        "RNN_C3",
        "RNN_C4",
    }
)


# ------------------------------------------------------------
# 9. Rank candidates using the PREDECLARED rule
#
# Primary:
#       validation MAE ascending
#
# Secondary:
#       validation RMSE ascending
#
# Tertiary:
#       parameter count ascending
#
# Stable sort is used only to make any theoretical exact tie
# deterministic.
# ------------------------------------------------------------

bilstm_architecture_selection_table = (
    bilstm_architecture_selection_table
    .sort_values(
        by=[
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "trainable_parameter_count",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


bilstm_architecture_selection_table.insert(
    0,
    "selection_rank",
    np.arange(
        1,
        len(
            bilstm_architecture_selection_table
        )
        +
        1,
        dtype=int,
    ),
)


# ------------------------------------------------------------
# 10. Select the rank-1 candidate
# ------------------------------------------------------------

bilstm_selected_candidate_id = str(
    bilstm_architecture_selection_table.loc[
        0,
        "candidate_id",
    ]
)


bilstm_selected_architecture = next(
    candidate.copy()

    for candidate in (
        RECURRENT_ARCHITECTURE_CANDIDATES
    )

    if candidate[
        "candidate_id"
    ]
    ==
    bilstm_selected_candidate_id
)


# ------------------------------------------------------------
# 11. Strong selected-architecture checks
# ------------------------------------------------------------

assert (
    bilstm_selected_architecture[
        "candidate_id"
    ]
    ==
    bilstm_selected_candidate_id
)


assert (
    bilstm_selected_candidate_id
    in
    BILSTM_EXPECTED_ARCH_SELECTION_PARAMS
)


selected_parameter_count = int(
    bilstm_architecture_selection_table.loc[
        0,
        "trainable_parameter_count",
    ]
)


assert (
    selected_parameter_count
    ==
    BILSTM_EXPECTED_ARCH_SELECTION_PARAMS[
        bilstm_selected_candidate_id
    ]
)


# ------------------------------------------------------------
# 12. Display ranked architecture-selection table
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM ARCHITECTURE SELECTION RESULTS"
)

print(
    "============================================================"
)


display_columns = [
    "selection_rank",
    "candidate_id",
    "auxiliary_dim",
    "trainable_parameter_count",
    "epochs_ran",
    "best_epoch",
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
]


display(
    bilstm_architecture_selection_table[
        display_columns
    ]
)


# ------------------------------------------------------------
# 13. Display selected architecture
# ------------------------------------------------------------

print(
    "\nSelected BiLSTM architecture:"
)


display(
    pd.DataFrame(
        [
            bilstm_selected_architecture
        ]
    )
)


print(
    "\nSelection basis:"
)

print(
    "  1. minimum validation MAE"
)

print(
    "  2. minimum validation RMSE"
)

print(
    "  3. fewer trainable parameters"
)


print(
    "\nSelected candidate:",
    bilstm_selected_candidate_id,
)


print(
    "Selected validation MAE:",
    (
        f"{bilstm_architecture_selection_table.loc[0, 'validation_MAE_kW']:.6f}"
    ),
    "kW",
)


print(
    "Selected validation RMSE:",
    (
        f"{bilstm_architecture_selection_table.loc[0, 'validation_RMSE_kW']:.6f}"
    ),
    "kW",
)


print(
    "Selected best epoch:",
    int(
        bilstm_architecture_selection_table.loc[
            0,
            "best_epoch",
        ]
    ),
)


print(
    "Selected parameter count:",
    selected_parameter_count,
)


# ------------------------------------------------------------
# 14. Persist architecture-selection result table
# ------------------------------------------------------------

bilstm_architecture_selection_path = (
    RESULTS_DIR
    /
    "bilstm_architecture_selection_60min_F0.csv"
)


bilstm_architecture_selection_table.to_csv(
    bilstm_architecture_selection_path,
    index=False,
)


# ------------------------------------------------------------
# 15. Persist each candidate's training history
# ------------------------------------------------------------

bilstm_architecture_history_paths = {}


for candidate_id, candidate_history in (
    bilstm_architecture_selection_histories.items()
):

    history_path = (
        RESULTS_DIR
        /
        (
            "bilstm_architecture_selection_"
            f"{candidate_id}_60min_F0_history.csv"
        )
    )


    candidate_history.to_csv(
        history_path,
        index=False,
    )


    bilstm_architecture_history_paths[
        candidate_id
    ] = history_path


# ------------------------------------------------------------
# 16. Freeze selected architecture metadata
#
# This file records the architecture selection BEFORE any
# BiLSTM final held-out TEST evaluation.
# ------------------------------------------------------------

bilstm_selected_architecture_record = {
    "model":
        "BiLSTM",

    "selection_horizon_minutes":
        60,

    "selection_feature_family":
        "F0",

    "selection_dataset":
        "validation",

    "test_used_for_selection":
        False,

    "ranking_rule":
        [
            "validation_MAE_kW ascending",
            "validation_RMSE_kW ascending",
            "trainable_parameter_count ascending",
        ],

    "selected_candidate":
        bilstm_selected_architecture,

    "selected_trainable_parameter_count":
        selected_parameter_count,

    "selected_best_epoch_on_60min_F0_validation":
        int(
            bilstm_architecture_selection_table.loc[
                0,
                "best_epoch",
            ]
        ),

    "selected_validation_MAE_kW":
        float(
            bilstm_architecture_selection_table.loc[
                0,
                "validation_MAE_kW",
            ]
        ),

    "selected_validation_RMSE_kW":
        float(
            bilstm_architecture_selection_table.loc[
                0,
                "validation_RMSE_kW",
            ]
        ),

    "selected_validation_R2":
        float(
            bilstm_architecture_selection_table.loc[
                0,
                "validation_R2",
            ]
        ),

    "sequence_protocol":
        BILSTM_SEQUENCE_PROTOCOL,
}


bilstm_selected_architecture_path = (
    RESULTS_DIR
    /
    "bilstm_selected_architecture.json"
)


with open(
    bilstm_selected_architecture_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        bilstm_selected_architecture_record,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

assert (
    bilstm_architecture_selection_path.exists()
)


assert (
    bilstm_architecture_selection_path.stat().st_size
    >
    0
)


assert (
    bilstm_selected_architecture_path.exists()
)


assert (
    bilstm_selected_architecture_path.stat().st_size
    >
    0
)


for history_path in (
    bilstm_architecture_history_paths.values()
):

    assert (
        history_path.exists()
    )

    assert (
        history_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Clean TensorFlow state after architecture selection
# ------------------------------------------------------------

tf.keras.backend.clear_session()


# ------------------------------------------------------------
# 19. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM ARCHITECTURE SELECTION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Selection horizon:",
    BILSTM_ARCH_SELECTION_HORIZON,
    "minutes",
)


print(
    "Selection family:",
    BILSTM_ARCH_SELECTION_FAMILY,
)


print(
    "Candidates evaluated:",
    len(
        bilstm_architecture_selection_table
    ),
)


print(
    "\nFrozen BiLSTM architecture:"
)

print(
    bilstm_selected_architecture
)


print(
    "\nTEST evaluations performed: 0"
)


print(
    "Held-out TEST was not used."
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_architecture_selection_path
)

print(
    " ",
    bilstm_selected_architecture_path
)


print(
    "\nNext step after verification:"
)

print(
    "Use the frozen BiLSTM architecture to perform validation-"
)

print(
    "only epoch selection for all 15 horizon × feature-family"
)

print(
    "experiments."
)


BiLSTM architecture candidate: RNN_C1
Epoch 1/80


2026-09-21 21:33:00.652362: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 21:33:00.653037: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 32s - 94ms/step - loss: 0.2290 - val_loss: 0.3184
Epoch 2/80
341/341 - 29s - 85ms/step - loss: 0.2036 - val_loss: 0.2971
Epoch 3/80
341/341 - 30s - 88ms/step - loss: 0.1972 - val_loss: 0.2941
Epoch 4/80
341/341 - 29s - 84ms/step - loss: 0.1962 - val_loss: 0.3059
Epoch 5/80
341/341 - 28s - 83ms/step - loss: 0.2044 - val_loss: 0.3193
Epoch 6/80
341/341 - 29s - 86ms/step - loss: 0.2257 - val_loss: 0.3227
Epoch 7/80
341/341 - 29s - 84ms/step - loss: 0.2300 - val_loss: 0.3261
Epoch 8/80
341/341 - 29s - 84ms/step - loss: 0.2346 - val_loss: 0.3326
Epoch 9/80
341/341 - 29s - 84ms/step - loss: 0.2586 - val_loss: 0.3772
Epoch 10/80
341/341 - 30s - 87ms/step - loss: 0.3014 - val_loss: 0.3771
Epoch 11/80
341/341 - 30s - 87ms/step - loss: 0.2932 - val_loss: 0.4818


2026-09-21 21:38:23.398078: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-21 21:38:23.398345: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


BiLSTM architecture candidate: RNN_C2
Epoch 1/80


...


BiLSTM ARCHITECTURE SELECTION RESULTS


,selection_rank,candidate_id,auxiliary_dim,trainable_parameter_count,epochs_ran,best_epoch,validation_MAE_kW,validation_RMSE_kW,validation_R2
0,1,RNN_C2,1,42081,14,6,0.464612,0.871569,0.797865
1,2,RNN_C1,1,12897,11,3,0.466951,0.880175,0.793854
2,3,RNN_C4,1,140897,16,8,0.468511,0.861965,0.802295
3,4,RNN_C3,1,42081,13,5,0.472260,0.862978,0.801831



Selected BiLSTM architecture:


,candidate_id,units,n_recurrent_layers,dropout,fusion_dense_units
0,RNN_C2,64,1,0.0,32



Selection basis:
  1. minimum validation MAE
  2. minimum validation RMSE
  3. fewer trainable parameters

Selected candidate: RNN_C2
Selected validation MAE: 0.464612 kW
Selected validation RMSE: 0.871569 kW
Selected best epoch: 6
Selected parameter count: 42081

BiLSTM ARCHITECTURE SELECTION: COMPLETE
Selection horizon: 60 minutes
Selection family: F0
Candidates evaluated: 4

Frozen BiLSTM architecture:
{'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

TEST evaluations performed: 0
Held-out TEST was not used.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_architecture_selection_60min_F0.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_selected_architecture.json

Next step after verification:
Use the frozen BiLSTM architecture

In [103]:
# ============================================================
# Step 87: BiLSTM validation-only epoch selection
#
# PURPOSE
# -------
#
# The BiLSTM architecture is now FROZEN:
#
#       candidate = RNN_C2
#       units = 64
#       recurrent layers = 1
#       dropout = 0.0
#       fusion Dense units = 32
#
#
# We now determine the number of training epochs separately for
# every:
#
#       forecast horizon × feature family
#
# giving:
#
#       3 horizons × 5 feature families = 15 experiments
#
#
# IMPORTANT EXPERIMENTAL RULE
# ---------------------------
#
# Epoch selection uses TRAIN + VALIDATION only.
#
# Held-out TEST is NOT used.
#
#
# NOTE ABOUT 60-min F0
# --------------------
#
# The selected BiLSTM architecture RNN_C2 was already trained
# under this exact validation protocol during architecture
# selection on 60-minute F0.
#
# Therefore we REUSE that completed validation run rather than
# training the exact same model a second time.
#
# Remaining new training runs:
#
#       14
#
#
# The cell also writes incremental checkpoint files after every
# completed experiment because BiLSTM training is relatively
# expensive on this machine.
# ============================================================


# ------------------------------------------------------------
# 1. Freeze selected architecture
# ------------------------------------------------------------

BILSTM_FROZEN_CANDIDATE_CONFIG = (
    bilstm_selected_architecture.copy()
)


assert (
    BILSTM_FROZEN_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


print(
    "Frozen BiLSTM architecture:",
    BILSTM_FROZEN_CANDIDATE_CONFIG,
)


# ------------------------------------------------------------
# 2. Full validation experiment grid
# ------------------------------------------------------------

BILSTM_EPOCH_SELECTION_HORIZONS = [
    15,
    30,
    60,
]


BILSTM_EPOCH_SELECTION_FAMILIES = [
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
]


BILSTM_EXPECTED_EXPERIMENT_KEYS = {
    (
        horizon,
        feature_family,
    )

    for horizon in (
        BILSTM_EPOCH_SELECTION_HORIZONS
    )

    for feature_family in (
        BILSTM_EPOCH_SELECTION_FAMILIES
    )
}


assert (
    len(
        BILSTM_EXPECTED_EXPERIMENT_KEYS
    )
    ==
    15
)


# ------------------------------------------------------------
# 3. Expected auxiliary widths
# ------------------------------------------------------------

BILSTM_EXPECTED_AUXILIARY_DIMS = {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


# ------------------------------------------------------------
# 4. Expected trainable parameter counts
#
# For frozen BiLSTM RNN_C2:
#
# Each direction:
#
#   4 * 64 * (9 + 64 + 1)
#   = 18,944
#
# Two directions:
#
#   37,888
#
# Concatenated recurrent output width:
#
#   128
#
# Total parameter count as a function of auxiliary width d:
#
#   37,888
#   + Dense((128 + d) -> 32)
#   + Dense(32 -> 1)
#
#   = 42,049 + 32d
#
# Therefore:
#
#   F0   d=1    -> 42,081
#   F1   d=16   -> 42,561
#   F2   d=21   -> 42,721
#   F3   d=30   -> 43,009
#   F4   d=127  -> 46,113
# ------------------------------------------------------------

BILSTM_EXPECTED_PARAMETER_COUNTS = {
    "F0": 42_081,
    "F1": 42_561,
    "F2": 42_721,
    "F3": 43_009,
    "F4": 46_113,
}


# ------------------------------------------------------------
# 5. Expected chronological TRAIN / VALIDATION populations
# ------------------------------------------------------------

BILSTM_EXPECTED_POPULATIONS = {
    15: {
        "train": 87_261,
        "validation": 17_568,
    },

    30: {
        "train": 87_258,
        "validation": 17_568,
    },

    60: {
        "train": 87_252,
        "validation": 17_568,
    },
}


# ------------------------------------------------------------
# 6. Verify canonical split populations BEFORE training
# ------------------------------------------------------------

population_audit_rows = []


for horizon in (
    BILSTM_EPOCH_SELECTION_HORIZONS
):

    for feature_family in (
        BILSTM_EPOCH_SELECTION_FAMILIES
    ):

        train_data = (
            get_deep_split_data(
                horizon_minutes=
                    horizon,

                feature_family=
                    feature_family,

                split_name=
                    "train",
            )
        )


        validation_data = (
            get_deep_split_data(
                horizon_minutes=
                    horizon,

                feature_family=
                    feature_family,

                split_name=
                    "validation",
            )
        )


        n_train = (
            train_data[
                "sequence"
            ]
            .shape[
                0
            ]
        )


        n_validation = (
            validation_data[
                "sequence"
            ]
            .shape[
                0
            ]
        )


        auxiliary_dim = (
            train_data[
                "auxiliary"
            ]
            .shape[
                1
            ]
        )


        assert (
            n_train
            ==
            BILSTM_EXPECTED_POPULATIONS[
                horizon
            ][
                "train"
            ]
        )


        assert (
            n_validation
            ==
            BILSTM_EXPECTED_POPULATIONS[
                horizon
            ][
                "validation"
            ]
        )


        assert (
            auxiliary_dim
            ==
            BILSTM_EXPECTED_AUXILIARY_DIMS[
                feature_family
            ]
        )


        assert (
            validation_data[
                "auxiliary"
            ]
            .shape[
                1
            ]
            ==
            auxiliary_dim
        )


        population_audit_rows.append(
            {
                "horizon_minutes":
                    horizon,

                "feature_family":
                    feature_family,

                "n_train":
                    n_train,

                "n_validation":
                    n_validation,

                "auxiliary_dim":
                    auxiliary_dim,
            }
        )


bilstm_epoch_selection_population_audit = (
    pd.DataFrame(
        population_audit_rows
    )
)


assert (
    bilstm_epoch_selection_population_audit.shape
    ==
    (
        15,
        5,
    )
)


print(
    "\nBiLSTM TRAIN / VALIDATION population audit: PASSED"
)


display(
    bilstm_epoch_selection_population_audit
)


# ------------------------------------------------------------
# 7. Checkpoint paths
#
# Results are saved incrementally after every experiment.
# ------------------------------------------------------------

bilstm_epoch_selection_partial_path = (
    RESULTS_DIR
    /
    "bilstm_epoch_selection_results_partial.csv"
)


# ------------------------------------------------------------
# 8. Initialize / resume result storage
#
# If this cell has already completed some BiLSTM experiments
# during the current notebook session or a partial CSV exists,
# those completed experiments are reused.
#
# This does NOT involve TEST data.
# ------------------------------------------------------------

if (
    bilstm_epoch_selection_partial_path.exists()
):

    existing_partial_df = pd.read_csv(
        bilstm_epoch_selection_partial_path
    )


    # Keep only rows belonging to the currently frozen
    # BiLSTM configuration.
    existing_partial_df = (
        existing_partial_df.loc[
            (
                existing_partial_df[
                    "model"
                ]
                ==
                "BiLSTM"
            )
            &
            (
                existing_partial_df[
                    "candidate_id"
                ]
                ==
                "RNN_C2"
            )
        ]
        .copy()
    )


    bilstm_epoch_selection_results = {
        (
            int(
                row[
                    "horizon_minutes"
                ]
            ),
            str(
                row[
                    "feature_family"
                ]
            ),
        ):
            row.to_dict()

        for _, row in (
            existing_partial_df.iterrows()
        )
    }


    print(
        "\nLoaded",
        len(
            bilstm_epoch_selection_results
        ),
        "previously completed BiLSTM validation experiments."
    )


else:

    bilstm_epoch_selection_results = {}


# History container for experiments completed in this session.
if (
    "bilstm_epoch_selection_histories"
    not in
    globals()
):

    bilstm_epoch_selection_histories = {}


# ------------------------------------------------------------
# 9. Reuse selected 60-minute F0 architecture-selection run
#
# This is the exact RNN_C2 validation run that selected the
# BiLSTM architecture.
# ------------------------------------------------------------

reuse_key = (
    60,
    "F0",
)


if (
    reuse_key
    not in
    bilstm_epoch_selection_results
):

    selected_60_f0_result = next(
        result.copy()

        for result in (
            bilstm_architecture_selection_results
        )

        if (
            result[
                "candidate_id"
            ]
            ==
            "RNN_C2"
        )
    )


    assert (
        selected_60_f0_result[
            "model"
        ]
        ==
        "BiLSTM"
    )


    assert (
        selected_60_f0_result[
            "horizon_minutes"
        ]
        ==
        60
    )


    assert (
        selected_60_f0_result[
            "feature_family"
        ]
        ==
        "F0"
    )


    assert (
        selected_60_f0_result[
            "best_epoch"
        ]
        ==
        6
    )


    assert (
        selected_60_f0_result[
            "trainable_parameter_count"
        ]
        ==
        BILSTM_EXPECTED_PARAMETER_COUNTS[
            "F0"
        ]
    )


    selected_60_f0_result[
        "epoch_selection_source"
    ] = (
        "reused_architecture_selection_run"
    )


    bilstm_epoch_selection_results[
        reuse_key
    ] = selected_60_f0_result


    # Preserve the corresponding history under the canonical
    # epoch-selection naming scheme.
    reused_history = (
        bilstm_architecture_selection_histories[
            "RNN_C2"
        ]
        .copy()
    )


    bilstm_epoch_selection_histories[
        reuse_key
    ] = reused_history


    reused_history_path = (
        RESULTS_DIR
        /
        "bilstm_epoch_selection_60min_F0_history.csv"
    )


    reused_history.to_csv(
        reused_history_path,
        index=False,
    )


    print(
        "\nReused completed 60-min F0 validation run:",
        "best epoch =",
        selected_60_f0_result[
            "best_epoch"
        ],
    )


# ------------------------------------------------------------
# 10. Helper to persist partial results safely
# ------------------------------------------------------------

def save_bilstm_epoch_selection_checkpoint():

    checkpoint_df = pd.DataFrame(
        list(
            bilstm_epoch_selection_results.values()
        )
    )


    if (
        len(
            checkpoint_df
        )
        >
        0
    ):

        checkpoint_df = (
            checkpoint_df.sort_values(
                [
                    "horizon_minutes",
                    "feature_family",
                ]
            )
            .reset_index(
                drop=True
            )
        )


    checkpoint_df.to_csv(
        bilstm_epoch_selection_partial_path,
        index=False,
    )


# Save the reused 60/F0 result immediately.
save_bilstm_epoch_selection_checkpoint()


# ------------------------------------------------------------
# 11. Run remaining validation-only experiments
# ------------------------------------------------------------

for horizon in (
    BILSTM_EPOCH_SELECTION_HORIZONS
):

    for feature_family in (
        BILSTM_EPOCH_SELECTION_FAMILIES
    ):

        experiment_key = (
            horizon,
            feature_family,
        )


        # ----------------------------------------------------
        # Skip already-completed experiments.
        # ----------------------------------------------------

        if (
            experiment_key
            in
            bilstm_epoch_selection_results
        ):

            print(
                "\nSkipping already completed:",
                f"{horizon}-min {feature_family}",
            )

            continue


        print(
            "\n============================================================"
        )

        print(
            "BiLSTM validation epoch selection:"
        )

        print(
            "  horizon:",
            horizon,
            "minutes",
        )

        print(
            "  feature family:",
            feature_family,
        )

        print(
            "============================================================"
        )


        # ----------------------------------------------------
        # Train with VALIDATION early stopping.
        #
        # No TEST data are used.
        # ----------------------------------------------------

        (
            validation_result,
            validation_history,
        ) = run_validation_candidate(
            model_name=
                "BiLSTM",

            candidate_config=
                BILSTM_FROZEN_CANDIDATE_CONFIG,

            horizon_minutes=
                horizon,

            feature_family=
                feature_family,

            verbose=
                2,
        )


        # ----------------------------------------------------
        # 12. Identity audits
        # ----------------------------------------------------

        assert (
            validation_result[
                "model"
            ]
            ==
            "BiLSTM"
        )


        assert (
            validation_result[
                "candidate_id"
            ]
            ==
            "RNN_C2"
        )


        assert (
            validation_result[
                "horizon_minutes"
            ]
            ==
            horizon
        )


        assert (
            validation_result[
                "feature_family"
            ]
            ==
            feature_family
        )


        assert (
            validation_result[
                "auxiliary_dim"
            ]
            ==
            BILSTM_EXPECTED_AUXILIARY_DIMS[
                feature_family
            ]
        )


        assert (
            validation_result[
                "trainable_parameter_count"
            ]
            ==
            BILSTM_EXPECTED_PARAMETER_COUNTS[
                feature_family
            ]
        )


        # ----------------------------------------------------
        # 13. Metric / early-stopping audits
        # ----------------------------------------------------

        for metric_name in [
            "validation_MAE_kW",
            "validation_RMSE_kW",
            "validation_R2",
        ]:

            assert np.isfinite(
                validation_result[
                    metric_name
                ]
            )


        assert (
            1
            <=
            validation_result[
                "best_epoch"
            ]
            <=
            validation_result[
                "epochs_ran"
            ]
            <=
            80
        )


        assert (
            len(
                validation_history
            )
            ==
            validation_result[
                "epochs_ran"
            ]
        )


        assert (
            "loss"
            in
            validation_history.columns
        )


        assert (
            "val_loss"
            in
            validation_history.columns
        )


        # Mark this as a newly executed epoch-selection run.
        validation_result[
            "epoch_selection_source"
        ] = (
            "dedicated_validation_run"
        )


        # ----------------------------------------------------
        # 14. Store in memory
        # ----------------------------------------------------

        bilstm_epoch_selection_results[
            experiment_key
        ] = (
            validation_result.copy()
        )


        bilstm_epoch_selection_histories[
            experiment_key
        ] = (
            validation_history.copy()
        )


        # ----------------------------------------------------
        # 15. Persist this experiment's history immediately
        # ----------------------------------------------------

        history_path = (
            RESULTS_DIR
            /
            (
                "bilstm_epoch_selection_"
                f"{horizon}min_"
                f"{feature_family}_history.csv"
            )
        )


        validation_history.to_csv(
            history_path,
            index=False,
        )


        # ----------------------------------------------------
        # 16. Persist cumulative result checkpoint immediately
        # ----------------------------------------------------

        save_bilstm_epoch_selection_checkpoint()


        print(
            "\nCompleted:",
            f"{horizon}-min {feature_family}",
        )


        print(
            "  epochs ran:",
            validation_result[
                "epochs_ran"
            ],
        )


        print(
            "  best epoch:",
            validation_result[
                "best_epoch"
            ],
        )


        print(
            "  validation MAE:",
            (
                f"{validation_result['validation_MAE_kW']:.6f}"
            ),
            "kW",
        )


# ------------------------------------------------------------
# 17. Verify complete 15-experiment grid
# ------------------------------------------------------------

assert (
    set(
        bilstm_epoch_selection_results.keys()
    )
    ==
    BILSTM_EXPECTED_EXPERIMENT_KEYS
)


assert (
    len(
        bilstm_epoch_selection_results
    )
    ==
    15
)


# ------------------------------------------------------------
# 18. Construct final canonical epoch-selection table
# ------------------------------------------------------------

bilstm_epoch_selection_table = (
    pd.DataFrame(
        list(
            bilstm_epoch_selection_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    bilstm_epoch_selection_table.shape[
        0
    ]
    ==
    15
)


# ------------------------------------------------------------
# 19. Final per-row protocol audits
# ------------------------------------------------------------

for row in (
    bilstm_epoch_selection_table.itertuples(
        index=False
    )
):

    assert (
        row.candidate_id
        ==
        "RNN_C2"
    )


    assert (
        int(
            row.auxiliary_dim
        )
        ==
        BILSTM_EXPECTED_AUXILIARY_DIMS[
            row.feature_family
        ]
    )


    assert (
        int(
            row.trainable_parameter_count
        )
        ==
        BILSTM_EXPECTED_PARAMETER_COUNTS[
            row.feature_family
        ]
    )


    assert (
        1
        <=
        int(
            row.best_epoch
        )
        <=
        int(
            row.epochs_ran
        )
        <=
        80
    )


# ------------------------------------------------------------
# 20. Freeze best epoch for every horizon × family
# ------------------------------------------------------------

BILSTM_FROZEN_EPOCHS = {
    (
        int(
            row.horizon_minutes
        ),
        str(
            row.feature_family
        ),
    ):
        int(
            row.best_epoch
        )

    for row in (
        bilstm_epoch_selection_table.itertuples(
            index=False
        )
    )
}


assert (
    len(
        BILSTM_FROZEN_EPOCHS
    )
    ==
    15
)


# ------------------------------------------------------------
# 21. Human-readable frozen epoch matrix
# ------------------------------------------------------------

bilstm_frozen_epoch_selection = (
    bilstm_epoch_selection_table[
        [
            "horizon_minutes",
            "feature_family",
            "best_epoch",
        ]
    ]
    .rename(
        columns={
            "best_epoch":
                "frozen_best_epoch",
        }
    )
    .pivot(
        index=
            "horizon_minutes",

        columns=
            "feature_family",

        values=
            "frozen_best_epoch",
    )
)


bilstm_frozen_epoch_selection = (
    bilstm_frozen_epoch_selection.loc[
        [
            15,
            30,
            60,
        ],
        [
            "F0",
            "F1",
            "F2",
            "F3",
            "F4",
        ],
    ]
)


# ------------------------------------------------------------
# 22. Validation-stage F3 -> F4 comparison
#
# This is descriptive validation information only.
#
# Positive relative change:
#       F4 has lower validation MAE.
#
# Negative relative change:
#       F4 has higher validation MAE.
# ------------------------------------------------------------

bilstm_validation_f3_f4_rows = []


for horizon in (
    BILSTM_EPOCH_SELECTION_HORIZONS
):

    f3_row = (
        bilstm_epoch_selection_table.loc[
            (
                bilstm_epoch_selection_table[
                    "horizon_minutes"
                ]
                ==
                horizon
            )
            &
            (
                bilstm_epoch_selection_table[
                    "feature_family"
                ]
                ==
                "F3"
            )
        ]
        .iloc[
            0
        ]
    )


    f4_row = (
        bilstm_epoch_selection_table.loc[
            (
                bilstm_epoch_selection_table[
                    "horizon_minutes"
                ]
                ==
                horizon
            )
            &
            (
                bilstm_epoch_selection_table[
                    "feature_family"
                ]
                ==
                "F4"
            )
        ]
        .iloc[
            0
        ]
    )


    f3_mae = float(
        f3_row[
            "validation_MAE_kW"
        ]
    )


    f4_mae = float(
        f4_row[
            "validation_MAE_kW"
        ]
    )


    mae_difference = (
        f3_mae
        -
        f4_mae
    )


    relative_improvement_pct = (
        mae_difference
        /
        f3_mae
        *
        100.0
    )


    bilstm_validation_f3_f4_rows.append(
        {
            "horizon_minutes":
                horizon,

            "F3_validation_MAE_kW":
                f3_mae,

            "F4_validation_MAE_kW":
                f4_mae,

            "F3_to_F4_validation_MAE_difference_kW":
                mae_difference,

            "F3_to_F4_validation_MAE_improvement_pct":
                relative_improvement_pct,

            "F4_lower_validation_MAE_than_F3":
                bool(
                    f4_mae
                    <
                    f3_mae
                ),
        }
    )


bilstm_validation_f3_f4_summary = (
    pd.DataFrame(
        bilstm_validation_f3_f4_rows
    )
)


assert (
    bilstm_validation_f3_f4_summary.shape
    ==
    (
        3,
        6,
    )
)


# ------------------------------------------------------------
# 23. Display complete validation results
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM VALIDATION EPOCH-SELECTION RESULTS"
)

print(
    "============================================================"
)


display_columns = [
    "horizon_minutes",
    "feature_family",
    "auxiliary_dim",
    "trainable_parameter_count",
    "epochs_ran",
    "best_epoch",
    "validation_MAE_kW",
    "validation_RMSE_kW",
    "validation_R2",
    "epoch_selection_source",
]


display(
    bilstm_epoch_selection_table[
        display_columns
    ]
)


# ------------------------------------------------------------
# 24. Display frozen epoch matrix
# ------------------------------------------------------------

print(
    "\nFrozen BiLSTM epochs:"
)


display(
    bilstm_frozen_epoch_selection
)


# ------------------------------------------------------------
# 25. Display validation F3 -> F4 comparison
# ------------------------------------------------------------

print(
    "\nValidation-only F3 -> F4 comparison:"
)


display(
    bilstm_validation_f3_f4_summary
)


# ------------------------------------------------------------
# 26. Persist canonical final artifacts
# ------------------------------------------------------------

bilstm_epoch_selection_results_path = (
    RESULTS_DIR
    /
    "bilstm_epoch_selection_results.csv"
)


bilstm_frozen_epoch_selection_path = (
    RESULTS_DIR
    /
    "bilstm_frozen_epoch_selection.csv"
)


bilstm_validation_f3_f4_summary_path = (
    RESULTS_DIR
    /
    "bilstm_validation_f3_f4_summary.csv"
)


bilstm_epoch_selection_table.to_csv(
    bilstm_epoch_selection_results_path,
    index=False,
)


bilstm_frozen_epoch_selection.to_csv(
    bilstm_frozen_epoch_selection_path,
)


bilstm_validation_f3_f4_summary.to_csv(
    bilstm_validation_f3_f4_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 27. Verify final artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_epoch_selection_results_path,
    bilstm_frozen_epoch_selection_path,
    bilstm_validation_f3_f4_summary_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 28. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM VALIDATION EPOCH SELECTION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen architecture:",
    BILSTM_FROZEN_CANDIDATE_CONFIG,
)


print(
    "Completed validation experiments:",
    len(
        bilstm_epoch_selection_results
    ),
    "/ 15",
)


print(
    "\nFrozen best epochs:"
)


for horizon in (
    BILSTM_EPOCH_SELECTION_HORIZONS
):

    values = [
        (
            feature_family,
            BILSTM_FROZEN_EPOCHS[
                (
                    horizon,
                    feature_family,
                )
            ],
        )

        for feature_family in (
            BILSTM_EPOCH_SELECTION_FAMILIES
        )
    ]


    print(
        f"  {horizon}-min:",
        values,
    )


print(
    "\nValidation F3 -> F4 MAE changes:"
)


for row in (
    bilstm_validation_f3_f4_summary
    .itertuples(
        index=False
    )
):

    print(
        (
            f"  {int(row.horizon_minutes):>2d}-min: "
            f"{row.F3_to_F4_validation_MAE_improvement_pct:+.4f}%"
        )
    )


print(
    "\nHeld-out TEST evaluations performed: 0"
)


print(
    "TEST was not used for epoch selection."
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_epoch_selection_results_path,
)

print(
    " ",
    bilstm_frozen_epoch_selection_path,
)

print(
    " ",
    bilstm_validation_f3_f4_summary_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Freeze the BiLSTM final-refit protocol, then begin the"
)

print(
    "15 final TRAIN+VALIDATION refits and held-out TEST runs."
)

Frozen BiLSTM architecture: {'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

BiLSTM TRAIN / VALIDATION population audit: PASSED


,horizon_minutes,feature_family,n_train,n_validation,auxiliary_dim
0,15,F0,87261,17568,1
1,15,F1,87261,17568,16
2,15,F2,87261,17568,21
3,15,F3,87261,17568,30
4,15,F4,87261,17568,127
5,30,F0,87258,17568,1
6,30,F1,87258,17568,16
7,30,F2,87258,17568,21
8,30,F3,87258,17568,30
9,30,F4,87258,17568,127



Reused completed 60-min F0 validation run: best epoch = 6

BiLSTM validation epoch selection:
  horizon: 15 minutes
  feature family: F0
Epoch 1/80


2026-09-21 22:28:46.915947: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-21 22:28:46.916239: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

341/341 - 34s - 98ms/step - loss: 0.1454 - val_loss: 0.1918
Epoch 2/80
341/341 - 31s - 89ms/step - loss: 0.1231 - val_loss: 0.1873
Epoch 3/80
341/341 - 30s - 89ms/step - loss: 0.1198 - val_loss: 0.1849
Epoch 4/80
341/341 - 30s - 89ms/step - loss: 0.1215 - val_loss: 0.1780
Epoch 5/80
341/341 - 30s - 88ms/step - loss: 0.1208 - val_loss: 0.1776
Epoch 6/80
341/341 - 30s - 89ms/step - loss: 0.1209 - val_loss: 0.1746
Epoch 7/80
341/341 - 30s - 89ms/step - loss: 0.1176 - val_loss: 0.1800
Epoch 8/80
341/341 - 30s - 89ms/step - loss: 0.1158 - val_loss: 0.1805
Epoch 9/80
341/341 - 31s - 91ms/step - loss: 0.1213 - val_loss: 0.1873
Epoch 10/80
341/341 - 30s - 88ms/step - loss: 0.1197 - val_loss: 0.1909
Epoch 11/80
341/341 - 30s - 88ms/step - loss: 0.1234 - val_loss: 0.2019
Epoch 12/80
341/341 - 30s - 89ms/step - loss: 0.1256 - val_loss: 0.1914
Epoch 13/80
341/341 - 30s - 88ms/step - loss: 0.1261 - val_loss: 0.1898
Epoch 14/80
341/341 - 30s - 88ms/step - loss: 0.1313 - val_loss: 0.2137


...


Frozen BiLSTM epochs:


feature_family,F0,F1,F2,F3,F4
horizon_minutes,,,,,
15,6,5,2,4,2
30,16,7,3,17,3
60,6,6,6,5,4



Validation-only F3 -> F4 comparison:


,horizon_minutes,F3_validation_MAE_kW,F4_validation_MAE_kW,F3_to_F4_validation_MAE_difference_kW,F3_to_F4_validation_MAE_improvement_pct,F4_lower_validation_MAE_than_F3
0,15,0.295260,0.296404,-0.001144,-0.387568,False
1,30,0.369176,0.360481,0.008695,2.355340,True
2,60,0.470686,0.461001,0.009685,2.057580,True



BiLSTM VALIDATION EPOCH SELECTION: COMPLETE
Frozen architecture: {'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}
Completed validation experiments: 15 / 15

Frozen best epochs:
  15-min: [('F0', 6), ('F1', 5), ('F2', 2), ('F3', 4), ('F4', 2)]
  30-min: [('F0', 16), ('F1', 7), ('F2', 3), ('F3', 17), ('F4', 3)]
  60-min: [('F0', 6), ('F1', 6), ('F2', 6), ('F3', 5), ('F4', 4)]

Validation F3 -> F4 MAE changes:
  15-min: -0.3876%
  30-min: +2.3553%
  60-min: +2.0576%

Held-out TEST evaluations performed: 0
TEST was not used for epoch selection.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_epoch_selection_results.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_frozen_epoch_selection.csv
  /Users/siddharthgupta/Downloads/NSUT/Seme

In [104]:
# ============================================================
# Step 88: Freeze BiLSTM final-refit / held-out TEST protocol
#
# PURPOSE
# -------
#
# Architecture selection is complete.
# Validation-based epoch selection is complete.
#
# We now freeze the procedure that will be used for each of the
# 15 BiLSTM horizon × feature-family final experiments.
#
#
# FINAL REFIT PROCEDURE
# ---------------------
#
# For each experiment:
#
#   1. Start from a FRESH BiLSTM model.
#
#   2. Use the already-frozen architecture:
#
#          RNN_C2
#          64 units
#          1 BiLSTM layer
#          dropout = 0
#          fusion Dense(32)
#
#   3. Concatenate:
#
#          TRAIN
#          then VALIDATION
#
#      into one final-refit population.
#
#   4. Keep the ORIGINAL TRAIN-ONLY scalers.
#
#      We DO NOT refit scalers on TRAIN+VALIDATION.
#
#   5. Train for EXACTLY the previously selected number of
#      epochs.
#
#      No EarlyStopping.
#      No callbacks.
#      No validation_data.
#
#   6. Evaluate the held-out TEST set exactly once.
#
#   7. Do NOT clip predictions.
#
#
# IMPORTANT
# ---------
#
# This cell performs:
#
#       NO final fitting
#       NO TEST prediction
#       NO TEST metric inspection
#
# It only freezes and audits the procedure.
# ============================================================


import json
import time


# ------------------------------------------------------------
# 1. Frozen BiLSTM architecture
# ------------------------------------------------------------

BILSTM_FINAL_CANDIDATE_CONFIG = {
    "candidate_id": "RNN_C2",
    "units": 64,
    "n_recurrent_layers": 1,
    "dropout": 0.0,
    "fusion_dense_units": 32,
}


assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    BILSTM_FROZEN_CANDIDATE_CONFIG
)


# ------------------------------------------------------------
# 2. Freeze exact epoch counts
#
# These values come exclusively from TRAIN / VALIDATION.
# ------------------------------------------------------------

BILSTM_FINAL_FROZEN_EPOCHS = {
    # 15-minute horizon
    (15, "F0"): 6,
    (15, "F1"): 5,
    (15, "F2"): 2,
    (15, "F3"): 4,
    (15, "F4"): 2,

    # 30-minute horizon
    (30, "F0"): 16,
    (30, "F1"): 7,
    (30, "F2"): 3,
    (30, "F3"): 17,
    (30, "F4"): 3,

    # 60-minute horizon
    (60, "F0"): 6,
    (60, "F1"): 6,
    (60, "F2"): 6,
    (60, "F3"): 5,
    (60, "F4"): 4,
}


assert (
    BILSTM_FINAL_FROZEN_EPOCHS
    ==
    BILSTM_FROZEN_EPOCHS
)


assert (
    len(
        BILSTM_FINAL_FROZEN_EPOCHS
    )
    ==
    15
)


# ------------------------------------------------------------
# 3. Expected auxiliary dimensions
# ------------------------------------------------------------

BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS = {
    "F0": 1,
    "F1": 16,
    "F2": 21,
    "F3": 30,
    "F4": 127,
}


# ------------------------------------------------------------
# 4. Expected parameter counts
#
# Frozen RNN_C2 BiLSTM:
#
#   F0 -> 42,081
#   F1 -> 42,561
#   F2 -> 42,721
#   F3 -> 43,009
#   F4 -> 46,113
# ------------------------------------------------------------

BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS = {
    "F0": 42_081,
    "F1": 42_561,
    "F2": 42_721,
    "F3": 43_009,
    "F4": 46_113,
}


# ------------------------------------------------------------
# 5. Expected chronological populations
# ------------------------------------------------------------

BILSTM_FINAL_EXPECTED_POPULATIONS = {
    15: {
        "train": 87_261,
        "validation": 17_568,
        "trainval": 104_829,
        "test": 26_496,
    },

    30: {
        "train": 87_258,
        "validation": 17_568,
        "trainval": 104_826,
        "test": 26_496,
    },

    60: {
        "train": 87_252,
        "validation": 17_568,
        "trainval": 104_820,
        "test": 26_496,
    },
}


# ------------------------------------------------------------
# 6. Verify frozen epoch table against Cell 91
# ------------------------------------------------------------

for (
    horizon,
    feature_family,
), frozen_epoch in (
    BILSTM_FINAL_FROZEN_EPOCHS.items()
):

    epoch_selection_row = (
        bilstm_epoch_selection_table.loc[
            (
                bilstm_epoch_selection_table[
                    "horizon_minutes"
                ]
                ==
                horizon
            )
            &
            (
                bilstm_epoch_selection_table[
                    "feature_family"
                ]
                ==
                feature_family
            )
        ]
    )


    assert (
        len(
            epoch_selection_row
        )
        ==
        1
    )


    assert (
        int(
            epoch_selection_row[
                "best_epoch"
            ]
            .iloc[
                0
            ]
        )
        ==
        frozen_epoch
    )


print(
    "BiLSTM frozen-epoch audit: PASSED"
)


# ------------------------------------------------------------
# 7. Helper:
#    construct TRAIN + VALIDATION final-refit population
#
# IMPORTANT:
#
# We concatenate arrays that have ALREADY been transformed
# using the original TRAIN-only preprocessing/scalers.
#
# No scaler is fitted or modified here.
#
# Ordering is:
#
#       TRAIN rows first
#       VALIDATION rows second
#
# which also preserves chronology.
# ------------------------------------------------------------

def get_bilstm_final_refit_training_data(
    horizon_minutes,
    feature_family,
):

    # --------------------------------------------------------
    # Retrieve canonical TRAIN and VALIDATION splits.
    # --------------------------------------------------------

    train_data = (
        get_deep_split_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,

            split_name=
                "train",
        )
    )


    validation_data = (
        get_deep_split_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,

            split_name=
                "validation",
        )
    )


    # --------------------------------------------------------
    # Population checks
    # --------------------------------------------------------

    expected = (
        BILSTM_FINAL_EXPECTED_POPULATIONS[
            horizon_minutes
        ]
    )


    assert (
        train_data[
            "sequence"
        ]
        .shape[
            0
        ]
        ==
        expected[
            "train"
        ]
    )


    assert (
        validation_data[
            "sequence"
        ]
        .shape[
            0
        ]
        ==
        expected[
            "validation"
        ]
    )


    # --------------------------------------------------------
    # Sequence-shape checks
    # --------------------------------------------------------

    assert (
        train_data[
            "sequence"
        ]
        .shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    assert (
        validation_data[
            "sequence"
        ]
        .shape[
            1:
        ]
        ==
        (
            13,
            9,
        )
    )


    # --------------------------------------------------------
    # Auxiliary-width checks
    # --------------------------------------------------------

    expected_auxiliary_dim = (
        BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
            feature_family
        ]
    )


    assert (
        train_data[
            "auxiliary"
        ]
        .shape[
            1
        ]
        ==
        expected_auxiliary_dim
    )


    assert (
        validation_data[
            "auxiliary"
        ]
        .shape[
            1
        ]
        ==
        expected_auxiliary_dim
    )


    # --------------------------------------------------------
    # Concatenate TRAIN then VALIDATION.
    # --------------------------------------------------------

    trainval_sequence = np.concatenate(
        [
            train_data[
                "sequence"
            ],
            validation_data[
                "sequence"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_auxiliary = np.concatenate(
        [
            train_data[
                "auxiliary"
            ],
            validation_data[
                "auxiliary"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_target_scaled = np.concatenate(
        [
            train_data[
                "target_scaled"
            ],
            validation_data[
                "target_scaled"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_target_kw = np.concatenate(
        [
            train_data[
                "target_kw"
            ],
            validation_data[
                "target_kw"
            ],
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    # --------------------------------------------------------
    # Final population checks
    # --------------------------------------------------------

    assert (
        trainval_sequence.shape[
            0
        ]
        ==
        expected[
            "trainval"
        ]
    )


    assert (
        trainval_auxiliary.shape[
            0
        ]
        ==
        expected[
            "trainval"
        ]
    )


    assert (
        trainval_target_scaled.shape[
            0
        ]
        ==
        expected[
            "trainval"
        ]
    )


    assert (
        trainval_target_kw.shape[
            0
        ]
        ==
        expected[
            "trainval"
        ]
    )


    # --------------------------------------------------------
    # Finite-value checks
    # --------------------------------------------------------

    assert np.isfinite(
        trainval_sequence
    ).all()


    assert np.isfinite(
        trainval_auxiliary
    ).all()


    assert np.isfinite(
        trainval_target_scaled
    ).all()


    assert np.isfinite(
        trainval_target_kw
    ).all()


    return {
        "sequence":
            trainval_sequence,

        "auxiliary":
            trainval_auxiliary,

        "target_scaled":
            trainval_target_scaled,

        "target_kw":
            trainval_target_kw,
    }


# ------------------------------------------------------------
# 8. Final-refit + TEST helper
#
# IMPORTANT:
#
# This function is DEFINED here but NOT CALLED in this cell.
#
# We will call it one experiment at a time in later cells.
# ------------------------------------------------------------

def run_bilstm_final_refit_and_test(
    horizon_minutes,
    feature_family,
    verbose=2,
):

    # --------------------------------------------------------
    # Frozen experiment metadata
    # --------------------------------------------------------

    experiment_key = (
        horizon_minutes,
        feature_family,
    )


    assert (
        experiment_key
        in
        BILSTM_FINAL_FROZEN_EPOCHS
    )


    frozen_best_epoch = (
        BILSTM_FINAL_FROZEN_EPOCHS[
            experiment_key
        ]
    )


    expected_auxiliary_dim = (
        BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
            feature_family
        ]
    )


    expected_parameter_count = (
        BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
            feature_family
        ]
    )


    expected_population = (
        BILSTM_FINAL_EXPECTED_POPULATIONS[
            horizon_minutes
        ]
    )


    # --------------------------------------------------------
    # Retrieve final TRAIN+VALIDATION population.
    # --------------------------------------------------------

    trainval_data = (
        get_bilstm_final_refit_training_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,
        )
    )


    # --------------------------------------------------------
    # Retrieve held-out TEST split.
    #
    # This is the FIRST stage at which TEST is used for the
    # given final BiLSTM experiment.
    # --------------------------------------------------------

    test_data = (
        get_deep_split_data(
            horizon_minutes=
                horizon_minutes,

            feature_family=
                feature_family,

            split_name=
                "test",
        )
    )


    # --------------------------------------------------------
    # TEST population / shape audits
    # --------------------------------------------------------

    assert (
        test_data[
            "sequence"
        ]
        .shape
        ==
        (
            expected_population[
                "test"
            ],
            13,
            9,
        )
    )


    assert (
        test_data[
            "auxiliary"
        ]
        .shape
        ==
        (
            expected_population[
                "test"
            ],
            expected_auxiliary_dim,
        )
    )


    assert (
        test_data[
            "target_kw"
        ]
        .shape[
            0
        ]
        ==
        expected_population[
            "test"
        ]
    )


    # --------------------------------------------------------
    # Fresh model.
    #
    # Clear previous TensorFlow model state before creating the
    # final experiment.
    # --------------------------------------------------------

    tf.keras.backend.clear_session()


    final_model = (
        build_deep_forecasting_model(
            model_name=
                "BiLSTM",

            auxiliary_dim=
                expected_auxiliary_dim,

            candidate_config=
                BILSTM_FINAL_CANDIDATE_CONFIG,

            seed=
                42,
        )
    )


    # --------------------------------------------------------
    # Exact architecture / parameter audit
    # --------------------------------------------------------

    trainable_parameter_count = int(
        final_model.count_params()
    )


    assert (
        trainable_parameter_count
        ==
        expected_parameter_count
    )


    # --------------------------------------------------------
    # FINAL REFIT
    #
    # Exactly the frozen number of epochs.
    #
    # No:
    #
    #       validation_data
    #       callbacks
    #       EarlyStopping
    #
    # shuffle=True is retained from the declared protocol.
    #
    # Individual samples remain internally ordered 13-step
    # historical sequences; only sample order changes.
    # --------------------------------------------------------

    fit_start_time = (
        time.perf_counter()
    )


    final_fit_history_object = (
        final_model.fit(
            [
                trainval_data[
                    "sequence"
                ],
                trainval_data[
                    "auxiliary"
                ],
            ],

            trainval_data[
                "target_scaled"
            ],

            epochs=
                frozen_best_epoch,

            batch_size=
                256,

            shuffle=
                True,

            verbose=
                verbose,
        )
    )


    final_fit_seconds = float(
        time.perf_counter()
        -
        fit_start_time
    )


    # --------------------------------------------------------
    # Convert history to canonical DataFrame
    # --------------------------------------------------------

    final_history = pd.DataFrame(
        final_fit_history_object.history
    )


    final_history.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                final_history
            )
            +
            1,
            dtype=int,
        ),
    )


    assert (
        len(
            final_history
        )
        ==
        frozen_best_epoch
    )


    assert (
        "loss"
        in
        final_history.columns
    )


    assert (
        "val_loss"
        not in
        final_history.columns
    )


    # --------------------------------------------------------
    # Held-out TEST prediction
    # --------------------------------------------------------

    test_prediction_scaled = (
        final_model.predict(
            [
                test_data[
                    "sequence"
                ],
                test_data[
                    "auxiliary"
                ],
            ],

            batch_size=
                256,

            verbose=
                0,
        )
        .reshape(
            -1
        )
    )


    assert (
        test_prediction_scaled.shape
        ==
        (
            expected_population[
                "test"
            ],
        )
    )


    assert np.isfinite(
        test_prediction_scaled
    ).all()


    # --------------------------------------------------------
    # Convert predictions back to physical kW.
    #
    # IMPORTANT:
    #
    # Uses the original TRAIN-only target scaler.
    # --------------------------------------------------------

    test_prediction_kw = (
        inverse_transform_target_to_kw(
            scaled_values=
                test_prediction_scaled,

            horizon_minutes=
                horizon_minutes,
        )
    )


    test_prediction_kw = np.asarray(
        test_prediction_kw
    ).reshape(
        -1
    )


    test_actual_kw = np.asarray(
        test_data[
            "target_kw"
        ]
    ).reshape(
        -1
    )


    assert (
        test_actual_kw.shape
        ==
        test_prediction_kw.shape
        ==
        (
            expected_population[
                "test"
            ],
        )
    )


    assert np.isfinite(
        test_actual_kw
    ).all()


    assert np.isfinite(
        test_prediction_kw
    ).all()


    # --------------------------------------------------------
    # TEST metrics
    #
    # Raw predictions are used.
    # No clipping.
    # --------------------------------------------------------

    test_metrics = (
        compute_regression_metrics_kw(
            test_actual_kw,
            test_prediction_kw,
        )
    )


    # --------------------------------------------------------
    # Canonical final result record
    # --------------------------------------------------------

    final_result = {
        "model":
            "BiLSTM",

        "candidate_id":
            BILSTM_FINAL_CANDIDATE_CONFIG[
                "candidate_id"
            ],

        "horizon_minutes":
            horizon_minutes,

        "feature_family":
            feature_family,

        "auxiliary_dim":
            expected_auxiliary_dim,

        "trainable_parameter_count":
            trainable_parameter_count,

        "n_train":
            expected_population[
                "train"
            ],

        "n_validation":
            expected_population[
                "validation"
            ],

        "n_trainval":
            expected_population[
                "trainval"
            ],

        "n_test":
            expected_population[
                "test"
            ],

        "frozen_best_epoch":
            frozen_best_epoch,

        "final_epochs_trained":
            len(
                final_history
            ),

        "test_MAE_kW":
            float(
                test_metrics[
                    "MAE_kW"
                ]
            ),

        "test_RMSE_kW":
            float(
                test_metrics[
                    "RMSE_kW"
                ]
            ),

        "test_R2":
            float(
                test_metrics[
                    "R2"
                ]
            ),

        "final_fit_seconds":
            final_fit_seconds,

        "prediction_clipping":
            False,

        "scaler_policy":
            "original_train_only_scalers",

        "refit_population":
            "train_plus_validation",
    }


    # --------------------------------------------------------
    # Final result sanity checks
    # --------------------------------------------------------

    assert (
        final_result[
            "frozen_best_epoch"
        ]
        ==
        final_result[
            "final_epochs_trained"
        ]
    )


    assert (
        final_result[
            "prediction_clipping"
        ]
        is False
    )


    assert (
        final_result[
            "scaler_policy"
        ]
        ==
        "original_train_only_scalers"
    )


    assert (
        final_result[
            "refit_population"
        ]
        ==
        "train_plus_validation"
    )


    for metric_name in [
        "test_MAE_kW",
        "test_RMSE_kW",
        "test_R2",
        "final_fit_seconds",
    ]:

        assert np.isfinite(
            final_result[
                metric_name
            ]
        )


    # --------------------------------------------------------
    # Remove model from memory before returning.
    #
    # Predictions / metrics / history are retained.
    # --------------------------------------------------------

    del final_model

    tf.keras.backend.clear_session()


    return (
        final_result,
        final_history,
        test_actual_kw,
        test_prediction_kw,
    )


# ------------------------------------------------------------
# 9. Initialize canonical final TEST stores
#
# These remain EMPTY after this cell.
# ------------------------------------------------------------

bilstm_final_test_results = {}

bilstm_final_test_histories = {}

bilstm_final_test_actuals = {}

bilstm_final_test_predictions = {}


assert (
    len(
        bilstm_final_test_results
    )
    ==
    0
)


# ------------------------------------------------------------
# 10. Dry-run TRAIN+VALIDATION data audit
#
# We verify all 15 combinations WITHOUT fitting a model and
# WITHOUT loading TEST.
# ------------------------------------------------------------

bilstm_final_refit_data_audit_rows = []


for horizon in [
    15,
    30,
    60,
]:

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]:

        trainval_data = (
            get_bilstm_final_refit_training_data(
                horizon_minutes=
                    horizon,

                feature_family=
                    feature_family,
            )
        )


        expected_population = (
            BILSTM_FINAL_EXPECTED_POPULATIONS[
                horizon
            ]
        )


        expected_auxiliary_dim = (
            BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
                feature_family
            ]
        )


        assert (
            trainval_data[
                "sequence"
            ]
            .shape
            ==
            (
                expected_population[
                    "trainval"
                ],
                13,
                9,
            )
        )


        assert (
            trainval_data[
                "auxiliary"
            ]
            .shape
            ==
            (
                expected_population[
                    "trainval"
                ],
                expected_auxiliary_dim,
            )
        )


        bilstm_final_refit_data_audit_rows.append(
            {
                "horizon_minutes":
                    horizon,

                "feature_family":
                    feature_family,

                "n_trainval":
                    trainval_data[
                        "sequence"
                    ]
                    .shape[
                        0
                    ],

                "auxiliary_dim":
                    trainval_data[
                        "auxiliary"
                    ]
                    .shape[
                        1
                    ],

                "frozen_best_epoch":
                    BILSTM_FINAL_FROZEN_EPOCHS[
                        (
                            horizon,
                            feature_family,
                        )
                    ],

                "expected_parameter_count":
                    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
                        feature_family
                    ],
            }
        )


bilstm_final_refit_data_audit = pd.DataFrame(
    bilstm_final_refit_data_audit_rows
)


assert (
    bilstm_final_refit_data_audit.shape
    ==
    (
        15,
        6,
    )
)


print(
    "\nBiLSTM final-refit TRAIN+VALIDATION audit:"
)


display(
    bilstm_final_refit_data_audit
)


# ------------------------------------------------------------
# 11. Persist frozen protocol BEFORE TEST evaluation
# ------------------------------------------------------------

bilstm_frozen_protocol_record = {
    "model":
        "BiLSTM",

    "candidate_config":
        BILSTM_FINAL_CANDIDATE_CONFIG,

    "architecture_selection":
        {
            "horizon_minutes":
                60,

            "feature_family":
                "F0",

            "selection_dataset":
                "validation",

            "test_used":
                False,
        },

    "frozen_epochs":
        {
            (
                f"{horizon}_{feature_family}"
            ):
                epoch

            for (
                horizon,
                feature_family,
            ), epoch in (
                BILSTM_FINAL_FROZEN_EPOCHS.items()
            )
        },

    "batch_size":
        256,

    "shuffle":
        True,

    "seed":
        42,

    "final_refit_population":
        "train_plus_validation",

    "scaler_policy":
        "original_train_only_scalers",

    "final_refit_callbacks":
        False,

    "final_refit_validation_data":
        False,

    "prediction_clipping":
        False,

    "sequence_timesteps":
        13,

    "sequence_channels":
        9,

    "sequence_endpoint":
        "forecast issue time t",

    "bidirectional_scope":
        "historical sequence only",

    "contains_future_observations":
        False,

    "test_rows_per_horizon":
        26_496,

    "test_evaluations_completed_at_protocol_freeze":
        0,
}


bilstm_frozen_protocol_path = (
    RESULTS_DIR
    /
    "bilstm_frozen_protocol.json"
)


with open(
    bilstm_frozen_protocol_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        bilstm_frozen_protocol_record,
        file,
        indent=2,
    )


assert (
    bilstm_frozen_protocol_path.exists()
)


assert (
    bilstm_frozen_protocol_path.stat().st_size
    >
    0
)


# ------------------------------------------------------------
# 12. Final protocol checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL-REFIT PROTOCOL: FROZEN"
)

print(
    "============================================================"
)


print(
    "Architecture:",
    BILSTM_FINAL_CANDIDATE_CONFIG,
)


print(
    "\nFrozen epoch matrix:"
)


display(
    bilstm_frozen_epoch_selection
)


print(
    "Final refit:"
)

print(
    "  population: TRAIN + VALIDATION"
)

print(
    "  scalers: original TRAIN-only scalers"
)

print(
    "  callbacks: none"
)

print(
    "  validation during final fit: none"
)

print(
    "  shuffle: True"
)

print(
    "  batch size: 256"
)

print(
    "  seed: 42"
)


print(
    "\nPrediction policy:"
)

print(
    "  clipping: False"
)


print(
    "\nBiLSTM leakage policy:"
)

print(
    "  bidirectional traversal is restricted to"
)

print(
    "  the historical sequence ending at issue time t."
)


print(
    "\nHeld-out TEST evaluations performed: 0"
)


print(
    "\nCanonical result stores initialized and empty."
)


print(
    "\nFrozen protocol artifact:"
)

print(
    " ",
    bilstm_frozen_protocol_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run the first final BiLSTM experiment:"
)

print(
    "  15-minute horizon, F0."
)

BiLSTM frozen-epoch audit: PASSED

BiLSTM final-refit TRAIN+VALIDATION audit:


,horizon_minutes,feature_family,n_trainval,auxiliary_dim,frozen_best_epoch,expected_parameter_count
0,15,F0,104829,1,6,42081
1,15,F1,104829,16,5,42561
2,15,F2,104829,21,2,42721
3,15,F3,104829,30,4,43009
4,15,F4,104829,127,2,46113
5,30,F0,104826,1,16,42081
6,30,F1,104826,16,7,42561
7,30,F2,104826,21,3,42721
8,30,F3,104826,30,17,43009
9,30,F4,104826,127,3,46113



BiLSTM FINAL-REFIT PROTOCOL: FROZEN
Architecture: {'candidate_id': 'RNN_C2', 'units': 64, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

Frozen epoch matrix:


feature_family,F0,F1,F2,F3,F4
horizon_minutes,,,,,
15,6,5,2,4,2
30,16,7,3,17,3
60,6,6,6,5,4


Final refit:
  population: TRAIN + VALIDATION
  scalers: original TRAIN-only scalers
  callbacks: none
  validation during final fit: none
  shuffle: True
  batch size: 256
  seed: 42

Prediction policy:
  clipping: False

BiLSTM leakage policy:
  bidirectional traversal is restricted to
  the historical sequence ending at issue time t.

Held-out TEST evaluations performed: 0

Canonical result stores initialized and empty.

Frozen protocol artifact:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_frozen_protocol.json

Next step after verification:
Run the first final BiLSTM experiment:
  15-minute horizon, F0.


In [105]:
# ============================================================
# Step 89: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 15 minutes
#       feature family = F0
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 6
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 6 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 completely held-out rows
#
#
# Prediction clipping:
#
#       NONE
#
#
# This is the FIRST final held-out TEST evaluation for BiLSTM.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_15_F0_HORIZON = 15

BILSTM_FINAL_15_F0_FAMILY = "F0"

BILSTM_FINAL_15_F0_KEY = (
    BILSTM_FINAL_15_F0_HORIZON,
    BILSTM_FINAL_15_F0_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm the final TEST store is still empty
#
# Before this experiment, no BiLSTM TEST result should exist.
# ------------------------------------------------------------

assert (
    len(
        bilstm_final_test_results
    )
    ==
    0
)


assert (
    len(
        bilstm_final_test_histories
    )
    ==
    0
)


assert (
    len(
        bilstm_final_test_actuals
    )
    ==
    0
)


assert (
    len(
        bilstm_final_test_predictions
    )
    ==
    0
)


# ------------------------------------------------------------
# 3. Confirm frozen protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_15_F0_KEY
    ]
    ==
    6
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F0"
    ]
    ==
    1
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F0"
    ]
    ==
    42_081
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
#
# The helper defined in Cell 92:
#
#   - creates a fresh BiLSTM,
#   - fits TRAIN+VALIDATION for exactly 6 epochs,
#   - uses no validation during final fitting,
#   - predicts TEST once,
#   - computes raw unclipped TEST metrics.
# ------------------------------------------------------------

(
    bilstm_final_15_f0_result,
    bilstm_final_15_f0_history,
    bilstm_final_15_f0_actual_kw,
    bilstm_final_15_f0_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_15_F0_HORIZON,

    feature_family=
        BILSTM_FINAL_15_F0_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f0_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_15_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_15_f0_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    bilstm_final_15_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


# ------------------------------------------------------------
# 6. Frozen architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    bilstm_final_15_f0_result[
        "trainable_parameter_count"
    ]
    ==
    42_081
)


assert (
    bilstm_final_15_f0_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    bilstm_final_15_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_15_f0_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    bilstm_final_15_f0_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f0_result[
        "frozen_best_epoch"
    ]
    ==
    6
)


assert (
    bilstm_final_15_f0_result[
        "final_epochs_trained"
    ]
    ==
    6
)


assert (
    len(
        bilstm_final_15_f0_history
    )
    ==
    6
)


assert (
    "val_loss"
    not in
    bilstm_final_15_f0_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_15_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_15_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_15_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f0_actual_kw.shape
    ==
    bilstm_final_15_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_15_f0_actual_kw
).all()


assert np.isfinite(
    bilstm_final_15_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Store successful experiment in canonical BiLSTM stores
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_15_F0_KEY
] = (
    bilstm_final_15_f0_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_15_F0_KEY
] = (
    bilstm_final_15_f0_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_15_F0_KEY
] = (
    bilstm_final_15_f0_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_15_F0_KEY
] = (
    bilstm_final_15_f0_prediction_kw.copy()
)


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (
            15,
            "F0",
        )
    }
)


# ------------------------------------------------------------
# 12. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_15_f0_result_df = pd.DataFrame(
    [
        bilstm_final_15_f0_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 15-MIN F0"
)

print(
    "============================================================"
)


display(
    bilstm_final_15_f0_result_df
)


# ------------------------------------------------------------
# 13. Display final-refit history
#
# There should be exactly 6 training-loss values and no
# validation loss because validation was incorporated into the
# final TRAIN+VALIDATION refit population.
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_15_f0_history
)


# ------------------------------------------------------------
# 14. Raw UNCLIPPED prediction audit
#
# Solar PV power is physically nonnegative, but we deliberately
# preserve raw model outputs in the primary evaluation.
#
# This avoids introducing a post-hoc clipping advantage.
# ------------------------------------------------------------

bilstm_final_15_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_15_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_15_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_15_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_15_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_15_f0_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_15_f0_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_15_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Build prediction table
#
# Row position is sufficient here because the canonical
# valid-time alignment has already been audited independently.
# ------------------------------------------------------------

bilstm_final_15_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_15_f0_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_15_f0_prediction_kw,
    }
)


assert (
    bilstm_final_15_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist experiment artifacts immediately
# ------------------------------------------------------------

bilstm_15_f0_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F0_result.csv"
)


bilstm_15_f0_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F0_history.csv"
)


bilstm_15_f0_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F0_predictions.csv"
)


bilstm_final_15_f0_result_df.to_csv(
    bilstm_15_f0_result_path,
    index=False,
)


bilstm_final_15_f0_history.to_csv(
    bilstm_15_f0_history_path,
    index=False,
)


bilstm_final_15_f0_prediction_table.to_csv(
    bilstm_15_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_15_f0_result_path,
    bilstm_15_f0_history_path,
    bilstm_15_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Current BiLSTM final-TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = pd.DataFrame(
    [
        result

        for result in (
            bilstm_final_test_results.values()
        )
    ]
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 19. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 15-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_15_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    (
        f"{bilstm_final_15_f0_result['test_MAE_kW']:.6f}"
    ),
    "kW",
)


print(
    "TEST RMSE:",
    (
        f"{bilstm_final_15_f0_result['test_RMSE_kW']:.6f}"
    ),
    "kW",
)


print(
    "TEST R2:",
    (
        f"{bilstm_final_15_f0_result['test_R2']:.6f}"
    ),
)


print(
    "\nPrediction clipping:",
    bilstm_final_15_f0_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_15_f0_result_path
)

print(
    " ",
    bilstm_15_f0_history_path
)

print(
    " ",
    bilstm_15_f0_predictions_path
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 15-minute F1."
)

Epoch 1/6


2026-09-22 11:15:44.309240: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:15:44.309903: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 85ms/step - loss: 0.1520
Epoch 2/6
410/410 - 32s - 78ms/step - loss: 0.1315
Epoch 3/6
410/410 - 31s - 76ms/step - loss: 0.1303
Epoch 4/6
410/410 - 31s - 76ms/step - loss: 0.1342
Epoch 5/6
410/410 - 31s - 76ms/step - loss: 0.1326
Epoch 6/6
410/410 - 31s - 76ms/step - loss: 0.1279


2026-09-22 11:18:55.484038: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:18:55.484310: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


BiLSTM FINAL HELD-OUT TEST — 15-MIN F0


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.102283,5.808852,2823,0



Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736



BiLSTM 15-MIN F0 FINAL TEST: COMPLETE
Frozen epoch: 6
TEST MAE: 0.271462 kW
TEST RMSE: 0.595756 kW
TEST R2: 0.869736

Prediction clipping: False

Completed BiLSTM final TEST experiments: 1 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F0_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F0_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F0_predictions.csv

Next step after verification:
Run final BiLSTM 15-minute F1.


In [106]:
# ============================================================
# Step 90: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 15 minutes
#       feature family = F1
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 5
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 5 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
#
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_15_F1_HORIZON = 15

BILSTM_FINAL_15_F1_FAMILY = "F1"

BILSTM_FINAL_15_F1_KEY = (
    BILSTM_FINAL_15_F1_HORIZON,
    BILSTM_FINAL_15_F1_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Only 15-min F0 should have been evaluated so far.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (
            15,
            "F0",
        )
    }
)


assert (
    BILSTM_FINAL_15_F1_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen protocol for 15-min F1
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_15_F1_KEY
    ]
    ==
    5
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F1"
    ]
    ==
    16
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F1"
    ]
    ==
    42_561
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_15_f1_result,
    bilstm_final_15_f1_history,
    bilstm_final_15_f1_actual_kw,
    bilstm_final_15_f1_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_15_F1_HORIZON,

    feature_family=
        BILSTM_FINAL_15_F1_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f1_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_15_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_15_f1_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    bilstm_final_15_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    bilstm_final_15_f1_result[
        "trainable_parameter_count"
    ]
    ==
    42_561
)


assert (
    bilstm_final_15_f1_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    bilstm_final_15_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_15_f1_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    bilstm_final_15_f1_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f1_result[
        "frozen_best_epoch"
    ]
    ==
    5
)


assert (
    bilstm_final_15_f1_result[
        "final_epochs_trained"
    ]
    ==
    5
)


assert (
    len(
        bilstm_final_15_f1_history
    )
    ==
    5
)


assert (
    "val_loss"
    not in
    bilstm_final_15_f1_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_15_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_15_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_15_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f1_actual_kw.shape
    ==
    bilstm_final_15_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_15_f1_actual_kw
).all()


assert np.isfinite(
    bilstm_final_15_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment audit
#
# F0 and F1 at the same 15-minute horizon must predict the
# exact same target vector.
# ------------------------------------------------------------

assert np.array_equal(
    bilstm_final_test_actuals[
        (
            15,
            "F0",
        )
    ],
    bilstm_final_15_f1_actual_kw,
)


print(
    "15-min F0/F1 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_15_F1_KEY
] = (
    bilstm_final_15_f1_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_15_F1_KEY
] = (
    bilstm_final_15_f1_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_15_F1_KEY
] = (
    bilstm_final_15_f1_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_15_F1_KEY
] = (
    bilstm_final_15_f1_prediction_kw.copy()
)


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (
            15,
            "F0",
        ),
        (
            15,
            "F1",
        ),
    }
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_15_f1_result_df = pd.DataFrame(
    [
        bilstm_final_15_f1_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 15-MIN F1"
)

print(
    "============================================================"
)


display(
    bilstm_final_15_f1_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_15_f1_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_15_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_15_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_15_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_15_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_15_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_15_f1_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_15_f1_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_15_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F0 -> F1 TEST comparison
#
# Positive percentage means F1 has LOWER TEST MAE than F0.
# This is descriptive only; no model choice is made from TEST.
# ------------------------------------------------------------

bilstm_15_f0_mae = float(
    bilstm_final_test_results[
        (
            15,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f1_mae = float(
    bilstm_final_15_f1_result[
        "test_MAE_kW"
    ]
)


bilstm_15_f0_to_f1_improvement_pct = (
    (
        bilstm_15_f0_mae
        -
        bilstm_15_f1_mae
    )
    /
    bilstm_15_f0_mae
    *
    100.0
)


print(
    "\n15-min F0 -> F1 TEST MAE change:"
)


print(
    (
        f"  F0 MAE: {bilstm_15_f0_mae:.6f} kW"
    )
)


print(
    (
        f"  F1 MAE: {bilstm_15_f1_mae:.6f} kW"
    )
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f0_to_f1_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. Build prediction table
# ------------------------------------------------------------

bilstm_final_15_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_15_f1_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_15_f1_prediction_kw,
    }
)


assert (
    bilstm_final_15_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 18. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_15_f1_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F1_result.csv"
)


bilstm_15_f1_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F1_history.csv"
)


bilstm_15_f1_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F1_predictions.csv"
)


bilstm_final_15_f1_result_df.to_csv(
    bilstm_15_f1_result_path,
    index=False,
)


bilstm_final_15_f1_history.to_csv(
    bilstm_15_f1_history_path,
    index=False,
)


bilstm_final_15_f1_prediction_table.to_csv(
    bilstm_15_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 19. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_15_f1_result_path,
    bilstm_15_f1_history_path,
    bilstm_15_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 20. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 15-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_15_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    (
        f"{bilstm_final_15_f1_result['test_MAE_kW']:.6f}"
    ),
    "kW",
)


print(
    "TEST RMSE:",
    (
        f"{bilstm_final_15_f1_result['test_RMSE_kW']:.6f}"
    ),
    "kW",
)


print(
    "TEST R2:",
    (
        f"{bilstm_final_15_f1_result['test_R2']:.6f}"
    ),
)


print(
    "\nPrediction clipping:",
    bilstm_final_15_f1_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_15_f1_result_path
)

print(
    " ",
    bilstm_15_f1_history_path
)

print(
    " ",
    bilstm_15_f1_predictions_path
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 15-minute F2."
)

Epoch 1/5


2026-09-22 11:20:49.522722: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:20:49.523320: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 83ms/step - loss: 0.1820
Epoch 2/5
410/410 - 31s - 75ms/step - loss: 0.1437
Epoch 3/5
410/410 - 31s - 77ms/step - loss: 0.1444
Epoch 4/5
410/410 - 31s - 75ms/step - loss: 0.1478
Epoch 5/5
410/410 - 32s - 77ms/step - loss: 0.1453


2026-09-22 11:23:28.260980: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:23:28.261368: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

15-min F0/F1 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 15-MIN F1


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.322575,6.011927,7743,0



15-min F0 -> F1 TEST MAE change:
  F0 MAE: 0.271462 kW
  F1 MAE: 0.310092 kW
  Relative improvement: -14.230077%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320



BiLSTM 15-MIN F1 FINAL TEST: COMPLETE
Frozen epoch: 5
TEST MAE: 0.310092 kW
TEST RMSE: 0.623503 kW
TEST R2: 0.857320

Prediction clipping: False

Completed BiLSTM final TEST experiments: 2 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F1_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F1_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F1_predictions.csv

Next step after verification:
Run final BiLSTM 15-minute F2.


In [107]:
# ============================================================
# Step 91: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 15 minutes
#       feature family = F2
#
#
# INTERPRETATION OF F2
# --------------------
#
# F2 contains:
#
#       F0 historical / meteorological information
#       +
#       F1 conventional Gregorian-time information
#       +
#       solar-astronomy information
#
# Therefore this experiment begins the astronomical part of
# our feature-family ablation.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 2
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 2 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_15_F2_HORIZON = 15

BILSTM_FINAL_15_F2_FAMILY = "F2"

BILSTM_FINAL_15_F2_KEY = (
    BILSTM_FINAL_15_F2_HORIZON,
    BILSTM_FINAL_15_F2_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Only 15-min F0 and F1 should exist so far.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
    }
)


assert (
    BILSTM_FINAL_15_F2_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen F2 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_15_F2_KEY
    ]
    ==
    2
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F2"
    ]
    ==
    21
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F2"
    ]
    ==
    42_721
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_15_f2_result,
    bilstm_final_15_f2_history,
    bilstm_final_15_f2_actual_kw,
    bilstm_final_15_f2_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_15_F2_HORIZON,

    feature_family=
        BILSTM_FINAL_15_F2_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f2_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_15_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_15_f2_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    bilstm_final_15_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    bilstm_final_15_f2_result[
        "trainable_parameter_count"
    ]
    ==
    42_721
)


assert (
    bilstm_final_15_f2_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    bilstm_final_15_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_15_f2_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    bilstm_final_15_f2_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f2_result[
        "frozen_best_epoch"
    ]
    ==
    2
)


assert (
    bilstm_final_15_f2_result[
        "final_epochs_trained"
    ]
    ==
    2
)


assert (
    len(
        bilstm_final_15_f2_history
    )
    ==
    2
)


assert (
    "val_loss"
    not in
    bilstm_final_15_f2_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_15_f2_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_15_f2_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_15_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f2_actual_kw.shape
    ==
    bilstm_final_15_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_15_f2_actual_kw
).all()


assert np.isfinite(
    bilstm_final_15_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment audit
#
# F0, F1 and F2 must forecast exactly the same 15-minute
# target vector.
# ------------------------------------------------------------

assert np.array_equal(
    bilstm_final_test_actuals[
        (15, "F0")
    ],
    bilstm_final_15_f2_actual_kw,
)


assert np.array_equal(
    bilstm_final_test_actuals[
        (15, "F1")
    ],
    bilstm_final_15_f2_actual_kw,
)


print(
    "15-min F0/F1/F2 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_15_F2_KEY
] = (
    bilstm_final_15_f2_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_15_F2_KEY
] = (
    bilstm_final_15_f2_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_15_F2_KEY
] = (
    bilstm_final_15_f2_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_15_F2_KEY
] = (
    bilstm_final_15_f2_prediction_kw.copy()
)


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
    }
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_15_f2_result_df = pd.DataFrame(
    [
        bilstm_final_15_f2_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 15-MIN F2"
)

print(
    "============================================================"
)


display(
    bilstm_final_15_f2_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_15_f2_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_15_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_15_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_15_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_15_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_15_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_15_f2_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_15_f2_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_15_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F1 -> F2 TEST comparison
#
# This isolates the addition of the SOLAR-ASTRONOMY layer.
#
# Positive percentage:
#
#       F2 has lower TEST MAE than F1.
#
# Negative percentage:
#
#       F2 has higher TEST MAE than F1.
#
# Descriptive only.
# No model choice is made from TEST.
# ------------------------------------------------------------

bilstm_15_f1_mae = float(
    bilstm_final_test_results[
        (15, "F1")
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f2_mae = float(
    bilstm_final_15_f2_result[
        "test_MAE_kW"
    ]
)


bilstm_15_f1_to_f2_improvement_pct = (
    (
        bilstm_15_f1_mae
        -
        bilstm_15_f2_mae
    )
    /
    bilstm_15_f1_mae
    *
    100.0
)


print(
    "\n15-min F1 -> F2 TEST MAE change "
    "(addition of solar astronomy):"
)


print(
    f"  F1 MAE: {bilstm_15_f1_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_15_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f1_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. Also retain F0 -> F2 comparison
#
# This shows F2 relative to the pure baseline.
# ------------------------------------------------------------

bilstm_15_f0_mae = float(
    bilstm_final_test_results[
        (15, "F0")
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f0_to_f2_improvement_pct = (
    (
        bilstm_15_f0_mae
        -
        bilstm_15_f2_mae
    )
    /
    bilstm_15_f0_mae
    *
    100.0
)


print(
    "\n15-min F0 -> F2 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_15_f0_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_15_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f0_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_15_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_15_f2_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_15_f2_prediction_kw,
    }
)


assert (
    bilstm_final_15_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_15_f2_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F2_result.csv"
)


bilstm_15_f2_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F2_history.csv"
)


bilstm_15_f2_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F2_predictions.csv"
)


bilstm_final_15_f2_result_df.to_csv(
    bilstm_15_f2_result_path,
    index=False,
)


bilstm_final_15_f2_history.to_csv(
    bilstm_15_f2_history_path,
    index=False,
)


bilstm_final_15_f2_prediction_table.to_csv(
    bilstm_15_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_15_f2_result_path,
    bilstm_15_f2_history_path,
    bilstm_15_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 15-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_15_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_15_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_15_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_15_f2_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_15_f2_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_15_f2_result_path
)

print(
    " ",
    bilstm_15_f2_history_path
)

print(
    " ",
    bilstm_15_f2_predictions_path
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 15-minute F3 "
    "(continuous Sun-Moon astronomy)."
)

Epoch 1/2


2026-09-22 11:24:50.612082: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:24:50.612448: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 84ms/step - loss: 0.1770
Epoch 2/2
410/410 - 31s - 76ms/step - loss: 0.1403


2026-09-22 11:25:56.071994: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:25:56.072268: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

15-min F0/F1/F2 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 15-MIN F2


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.114098,5.917549,4519,0



15-min F1 -> F2 TEST MAE change (addition of solar astronomy):
  F1 MAE: 0.310092 kW
  F2 MAE: 0.285021 kW
  Relative improvement: +8.084883%

15-min F0 -> F2 TEST MAE change:
  F0 MAE: 0.271462 kW
  F2 MAE: 0.285021 kW
  Relative improvement: -4.994710%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475



BiLSTM 15-MIN F2 FINAL TEST: COMPLETE
Frozen epoch: 2
TEST MAE: 0.285021 kW
TEST RMSE: 0.616572 kW
TEST R2: 0.860475

Prediction clipping: False

Completed BiLSTM final TEST experiments: 3 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F2_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F2_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F2_predictions.csv

Next step after verification:
Run final BiLSTM 15-minute F3 (continuous Sun-Moon astronomy).


In [108]:
# ============================================================
# Step 92: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 15 minutes
#       feature family = F3
#
#
# SCIENTIFIC ROLE OF F3
# ---------------------
#
# F3 contains everything already available in F2:
#
#       historical / meteorological information
#       Gregorian temporal information
#       solar astronomy
#
# and additionally introduces:
#
#       continuous lunar / Sun-Moon astronomical information
#
#
# Therefore the comparison:
#
#       F2 -> F3
#
# asks:
#
#       Does continuous lunar / Sun-Moon astronomy add
#       predictive information beyond solar astronomy?
#
#
# This is distinct from the later:
#
#       F3 -> F4
#
# comparison, which asks whether Panchang categorical
# representation adds information beyond the underlying
# continuous astronomy.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 4
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 4 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_15_F3_HORIZON = 15

BILSTM_FINAL_15_F3_FAMILY = "F3"

BILSTM_FINAL_15_F3_KEY = (
    BILSTM_FINAL_15_F3_HORIZON,
    BILSTM_FINAL_15_F3_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# F0, F1 and F2 should be the only completed experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
    }
)


assert (
    BILSTM_FINAL_15_F3_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen F3 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_15_F3_KEY
    ]
    ==
    4
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F3"
    ]
    ==
    30
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F3"
    ]
    ==
    43_009
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_15_f3_result,
    bilstm_final_15_f3_history,
    bilstm_final_15_f3_actual_kw,
    bilstm_final_15_f3_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_15_F3_HORIZON,

    feature_family=
        BILSTM_FINAL_15_F3_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f3_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_15_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_15_f3_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    bilstm_final_15_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    bilstm_final_15_f3_result[
        "trainable_parameter_count"
    ]
    ==
    43_009
)


assert (
    bilstm_final_15_f3_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    bilstm_final_15_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_15_f3_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    bilstm_final_15_f3_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f3_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    bilstm_final_15_f3_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    len(
        bilstm_final_15_f3_history
    )
    ==
    4
)


assert (
    "val_loss"
    not in
    bilstm_final_15_f3_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_15_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_15_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_15_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f3_actual_kw.shape
    ==
    bilstm_final_15_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_15_f3_actual_kw
).all()


assert np.isfinite(
    bilstm_final_15_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# All feature families at the same horizon must predict the
# exact same target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                15,
                previous_family,
            )
        ],
        bilstm_final_15_f3_actual_kw,
    )


print(
    "15-min F0/F1/F2/F3 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_15_F3_KEY
] = (
    bilstm_final_15_f3_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_15_F3_KEY
] = (
    bilstm_final_15_f3_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_15_F3_KEY
] = (
    bilstm_final_15_f3_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_15_F3_KEY
] = (
    bilstm_final_15_f3_prediction_kw.copy()
)


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
    }
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_15_f3_result_df = pd.DataFrame(
    [
        bilstm_final_15_f3_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 15-MIN F3"
)

print(
    "============================================================"
)


display(
    bilstm_final_15_f3_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_15_f3_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_15_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_15_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_15_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_15_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_15_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_15_f3_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_15_f3_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_15_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F2 -> F3 TEST comparison
#
# THIS IS AN IMPORTANT SCIENTIFIC COMPARISON.
#
# F2:
#       solar astronomy
#
# F3:
#       solar astronomy
#       +
#       continuous lunar / Sun-Moon astronomy
#
#
# Positive percentage:
#
#       F3 has LOWER TEST MAE than F2.
#
# Negative percentage:
#
#       F3 has HIGHER TEST MAE than F2.
#
# Descriptive only.
# ------------------------------------------------------------

bilstm_15_f2_mae = float(
    bilstm_final_test_results[
        (
            15,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f3_mae = float(
    bilstm_final_15_f3_result[
        "test_MAE_kW"
    ]
)


bilstm_15_f2_to_f3_improvement_pct = (
    (
        bilstm_15_f2_mae
        -
        bilstm_15_f3_mae
    )
    /
    bilstm_15_f2_mae
    *
    100.0
)


print(
    "\n15-min F2 -> F3 TEST MAE change "
    "(addition of continuous lunar / Sun-Moon astronomy):"
)


print(
    f"  F2 MAE: {bilstm_15_f2_mae:.6f} kW"
)


print(
    f"  F3 MAE: {bilstm_15_f3_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f2_to_f3_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. F0 -> F3 reference comparison
#
# Useful later for the complete ablation table.
# ------------------------------------------------------------

bilstm_15_f0_mae = float(
    bilstm_final_test_results[
        (
            15,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f0_to_f3_improvement_pct = (
    (
        bilstm_15_f0_mae
        -
        bilstm_15_f3_mae
    )
    /
    bilstm_15_f0_mae
    *
    100.0
)


print(
    "\n15-min F0 -> F3 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_15_f0_mae:.6f} kW"
)


print(
    f"  F3 MAE: {bilstm_15_f3_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f0_to_f3_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_15_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_15_f3_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_15_f3_prediction_kw,
    }
)


assert (
    bilstm_final_15_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_15_f3_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F3_result.csv"
)


bilstm_15_f3_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F3_history.csv"
)


bilstm_15_f3_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F3_predictions.csv"
)


bilstm_final_15_f3_result_df.to_csv(
    bilstm_15_f3_result_path,
    index=False,
)


bilstm_final_15_f3_history.to_csv(
    bilstm_15_f3_history_path,
    index=False,
)


bilstm_final_15_f3_prediction_table.to_csv(
    bilstm_15_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_15_f3_result_path,
    bilstm_15_f3_history_path,
    bilstm_15_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 15-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_15_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_15_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_15_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_15_f3_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_15_f3_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_15_f3_result_path,
)

print(
    " ",
    bilstm_15_f3_history_path,
)

print(
    " ",
    bilstm_15_f3_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 15-minute F4."
)

print(
    "That will give the first BiLSTM held-out F3 -> F4"
)

print(
    "Panchang comparison."
)

Epoch 1/4


2026-09-22 11:29:08.993127: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:29:08.993801: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 83ms/step - loss: 0.1849
Epoch 2/4
410/410 - 31s - 76ms/step - loss: 0.1491
Epoch 3/4
410/410 - 31s - 76ms/step - loss: 0.1504
Epoch 4/4
410/410 - 31s - 76ms/step - loss: 0.1507


2026-09-22 11:31:16.270788: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:31:16.271896: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

15-min F0/F1/F2/F3 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 15-MIN F3


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.313235,5.780952,8477,0



15-min F2 -> F3 TEST MAE change (addition of continuous lunar / Sun-Moon astronomy):
  F2 MAE: 0.285021 kW
  F3 MAE: 0.308417 kW
  Relative improvement: -8.208453%

15-min F0 -> F3 TEST MAE change:
  F0 MAE: 0.271462 kW
  F3 MAE: 0.308417 kW
  Relative improvement: -13.613151%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525



BiLSTM 15-MIN F3 FINAL TEST: COMPLETE
Frozen epoch: 4
TEST MAE: 0.308417 kW
TEST RMSE: 0.612025 kW
TEST R2: 0.862525

Prediction clipping: False

Completed BiLSTM final TEST experiments: 4 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F3_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F3_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_15min_F3_predictions.csv

Next step after verification:
Run final BiLSTM 15-minute F4.
That will give the first BiLSTM held-out F3 -> F4
Panchang comparison.


In [109]:
# ============================================================
# Step 93: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 15 minutes
#       feature family = F4
#
#
# SCIENTIFIC ROLE OF F4
# ---------------------
#
# F4 contains EVERYTHING in F3:
#
#       history / meteorology
#       Gregorian temporal features
#       solar astronomy
#       continuous lunar / Sun-Moon astronomy
#
# plus:
#
#       Panchang categorical features
#
#       - Tithi
#       - Paksha
#       - Karana
#       - Nakshatra
#       - Yoga
#
#
# Therefore:
#
#       F3 -> F4
#
# is the CORE novelty comparison.
#
# It asks:
#
#       After continuous solar/lunar astronomy is already
#       supplied, does the Panchang representation provide
#       additional predictive information?
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 2
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 2 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_15_F4_HORIZON = 15

BILSTM_FINAL_15_F4_FAMILY = "F4"

BILSTM_FINAL_15_F4_KEY = (
    BILSTM_FINAL_15_F4_HORIZON,
    BILSTM_FINAL_15_F4_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# F0-F3 at 15 minutes must already be complete.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
    }
)


assert (
    BILSTM_FINAL_15_F4_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen F4 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_15_F4_KEY
    ]
    ==
    2
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F4"
    ]
    ==
    127
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F4"
    ]
    ==
    46_113
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_15_f4_result,
    bilstm_final_15_f4_history,
    bilstm_final_15_f4_actual_kw,
    bilstm_final_15_f4_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_15_F4_HORIZON,

    feature_family=
        BILSTM_FINAL_15_F4_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f4_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_15_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_15_f4_result[
        "horizon_minutes"
    ]
    ==
    15
)


assert (
    bilstm_final_15_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    bilstm_final_15_f4_result[
        "trainable_parameter_count"
    ]
    ==
    46_113
)


assert (
    bilstm_final_15_f4_result[
        "n_train"
    ]
    ==
    87_261
)


assert (
    bilstm_final_15_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_15_f4_result[
        "n_trainval"
    ]
    ==
    104_829
)


assert (
    bilstm_final_15_f4_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f4_result[
        "frozen_best_epoch"
    ]
    ==
    2
)


assert (
    bilstm_final_15_f4_result[
        "final_epochs_trained"
    ]
    ==
    2
)


assert (
    len(
        bilstm_final_15_f4_history
    )
    ==
    2
)


assert (
    "val_loss"
    not in
    bilstm_final_15_f4_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_15_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_15_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_15_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_15_f4_actual_kw.shape
    ==
    bilstm_final_15_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_15_f4_actual_kw
).all()


assert np.isfinite(
    bilstm_final_15_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Complete 15-minute target alignment audit
#
# F0-F4 must all forecast exactly the same target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                15,
                previous_family,
            )
        ],
        bilstm_final_15_f4_actual_kw,
    )


print(
    "15-min F0/F1/F2/F3/F4 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_15_F4_KEY
] = (
    bilstm_final_15_f4_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_15_F4_KEY
] = (
    bilstm_final_15_f4_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_15_F4_KEY
] = (
    bilstm_final_15_f4_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_15_F4_KEY
] = (
    bilstm_final_15_f4_prediction_kw.copy()
)


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
    }
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_15_f4_result_df = pd.DataFrame(
    [
        bilstm_final_15_f4_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 15-MIN F4"
)

print(
    "============================================================"
)


display(
    bilstm_final_15_f4_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_15_f4_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_15_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_15_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_15_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_15_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_15_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_15_f4_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_15_f4_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_15_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. CORE F3 -> F4 TEST comparison
#
# Positive percentage:
#
#       F4 has LOWER TEST MAE than F3.
#
# Negative percentage:
#
#       F4 has HIGHER TEST MAE than F3.
#
#
# This is the central Panchang increment.
# ------------------------------------------------------------

bilstm_15_f3_mae = float(
    bilstm_final_test_results[
        (
            15,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f4_mae = float(
    bilstm_final_15_f4_result[
        "test_MAE_kW"
    ]
)


bilstm_15_f3_to_f4_improvement_pct = (
    (
        bilstm_15_f3_mae
        -
        bilstm_15_f4_mae
    )
    /
    bilstm_15_f3_mae
    *
    100.0
)


print(
    "\n============================================================"
)

print(
    "CORE 15-MIN F3 -> F4 PANCHANG COMPARISON"
)

print(
    "============================================================"
)


print(
    f"F3 TEST MAE: {bilstm_15_f3_mae:.6f} kW"
)


print(
    f"F4 TEST MAE: {bilstm_15_f4_mae:.6f} kW"
)


print(
    (
        "F3 -> F4 relative MAE improvement: "
        f"{bilstm_15_f3_to_f4_improvement_pct:+.6f}%"
    )
)


print(
    "Positive = F4 lower MAE than F3."
)


# ------------------------------------------------------------
# 17. F2 -> F4 descriptive comparison
#
# This is NOT the main novelty test because F4 differs from F2
# by both:
#
#       continuous lunar astronomy
#       +
#       Panchang representation
#
# It is retained only for the complete ablation.
# ------------------------------------------------------------

bilstm_15_f2_mae = float(
    bilstm_final_test_results[
        (
            15,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_15_f2_to_f4_improvement_pct = (
    (
        bilstm_15_f2_mae
        -
        bilstm_15_f4_mae
    )
    /
    bilstm_15_f2_mae
    *
    100.0
)


print(
    "\n15-min F2 -> F4 TEST MAE change:"
)


print(
    f"  F2 MAE: {bilstm_15_f2_mae:.6f} kW"
)


print(
    f"  F4 MAE: {bilstm_15_f4_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_15_f2_to_f4_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Complete 15-minute BiLSTM ablation table
# ------------------------------------------------------------

bilstm_15min_ablation_table = (
    pd.DataFrame(
        [
            bilstm_final_test_results[
                (
                    15,
                    feature_family,
                )
            ]

            for feature_family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        ]
    )
    [
        [
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "frozen_best_epoch",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


print(
    "\nComplete BiLSTM 15-minute ablation:"
)


display(
    bilstm_15min_ablation_table
)


# ------------------------------------------------------------
# 19. Build prediction table
# ------------------------------------------------------------

bilstm_final_15_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_15_f4_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_15_f4_prediction_kw,
    }
)


assert (
    bilstm_final_15_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 20. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_15_f4_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F4_result.csv"
)


bilstm_15_f4_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F4_history.csv"
)


bilstm_15_f4_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_F4_predictions.csv"
)


bilstm_15min_ablation_path = (
    RESULTS_DIR
    /
    "bilstm_final_15min_ablation.csv"
)


bilstm_final_15_f4_result_df.to_csv(
    bilstm_15_f4_result_path,
    index=False,
)


bilstm_final_15_f4_history.to_csv(
    bilstm_15_f4_history_path,
    index=False,
)


bilstm_final_15_f4_prediction_table.to_csv(
    bilstm_15_f4_predictions_path,
    index=False,
)


bilstm_15min_ablation_table.to_csv(
    bilstm_15min_ablation_path,
    index=False,
)


# ------------------------------------------------------------
# 21. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_15_f4_result_path,
    bilstm_15_f4_history_path,
    bilstm_15_f4_predictions_path,
    bilstm_15min_ablation_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 22. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 23. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 15-MIN F4 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_15_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_15_f4_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_15_f4_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_15_f4_result['test_R2']:.6f}",
)


print(
    "\nCORE F3 -> F4 TEST MAE improvement:",
    f"{bilstm_15_f3_to_f4_improvement_pct:+.6f}%",
)


print(
    "\nPrediction clipping:",
    bilstm_final_15_f4_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nNext step after verification:"
)

print(
    "Begin the 30-minute BiLSTM final TEST experiments,"
)

print(
    "starting with F0."
)

Epoch 1/2


2026-09-22 11:33:13.287202: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:33:13.287518: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 33s - 81ms/step - loss: 0.1797
Epoch 2/2
410/410 - 32s - 79ms/step - loss: 0.1434


2026-09-22 11:34:18.777101: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:34:18.777400: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

15-min F0/F1/F2/F3/F4 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 15-MIN F4


...


CORE 15-MIN F3 -> F4 PANCHANG COMPARISON
F3 TEST MAE: 0.308417 kW
F4 TEST MAE: 0.303657 kW
F3 -> F4 relative MAE improvement: +1.543316%
Positive = F4 lower MAE than F3.

15-min F2 -> F4 TEST MAE change:
  F2 MAE: 0.285021 kW
  F4 MAE: 0.303657 kW
  Relative improvement: -6.538454%

Complete BiLSTM 15-minute ablation:


,feature_family,auxiliary_dim,trainable_parameter_count,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,F0,1,42081,6,0.271462,0.595756,0.869736
1,F1,16,42561,5,0.310092,0.623503,0.857320
2,F2,21,42721,2,0.285021,0.616572,0.860475
3,F3,30,43009,4,0.308417,0.612025,0.862525
4,F4,127,46113,2,0.303657,0.616196,0.860644



Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644



BiLSTM 15-MIN F4 FINAL TEST: COMPLETE
Frozen epoch: 2
TEST MAE: 0.303657 kW
TEST RMSE: 0.616196 kW
TEST R2: 0.860644

CORE F3 -> F4 TEST MAE improvement: +1.543316%

Prediction clipping: False

Completed BiLSTM final TEST experiments: 5 / 15

Next step after verification:
Begin the 30-minute BiLSTM final TEST experiments,
starting with F0.


In [110]:
# ============================================================
# Step 94: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 30 minutes
#       feature family = F0
#
#
# This begins the complete 30-minute BiLSTM ablation.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 16
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 16 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_30_F0_HORIZON = 30

BILSTM_FINAL_30_F0_FAMILY = "F0"

BILSTM_FINAL_30_F0_KEY = (
    BILSTM_FINAL_30_F0_HORIZON,
    BILSTM_FINAL_30_F0_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# All five 15-minute experiments must already be complete.
# No 30-minute experiment should exist yet.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
    }
)


assert (
    BILSTM_FINAL_30_F0_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 30-min F0 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_30_F0_KEY
    ]
    ==
    16
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F0"
    ]
    ==
    1
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F0"
    ]
    ==
    42_081
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_30_f0_result,
    bilstm_final_30_f0_history,
    bilstm_final_30_f0_actual_kw,
    bilstm_final_30_f0_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_30_F0_HORIZON,

    feature_family=
        BILSTM_FINAL_30_F0_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f0_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_30_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_30_f0_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    bilstm_final_30_f0_result[
        "trainable_parameter_count"
    ]
    ==
    42_081
)


assert (
    bilstm_final_30_f0_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    bilstm_final_30_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_30_f0_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    bilstm_final_30_f0_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f0_result[
        "frozen_best_epoch"
    ]
    ==
    16
)


assert (
    bilstm_final_30_f0_result[
        "final_epochs_trained"
    ]
    ==
    16
)


assert (
    len(
        bilstm_final_30_f0_history
    )
    ==
    16
)


assert (
    "val_loss"
    not in
    bilstm_final_30_f0_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_30_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_30_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_30_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f0_actual_kw.shape
    ==
    bilstm_final_30_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_30_f0_actual_kw
).all()


assert np.isfinite(
    bilstm_final_30_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_30_F0_KEY
] = (
    bilstm_final_30_f0_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_30_F0_KEY
] = (
    bilstm_final_30_f0_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_30_F0_KEY
] = (
    bilstm_final_30_f0_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_30_F0_KEY
] = (
    bilstm_final_30_f0_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    6
)


# ------------------------------------------------------------
# 12. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_30_f0_result_df = pd.DataFrame(
    [
        bilstm_final_30_f0_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 30-MIN F0"
)

print(
    "============================================================"
)


display(
    bilstm_final_30_f0_result_df
)


# ------------------------------------------------------------
# 13. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_30_f0_history
)


# ------------------------------------------------------------
# 14. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_30_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_30_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_30_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_30_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_30_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_30_f0_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_30_f0_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_30_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Build prediction table
# ------------------------------------------------------------

bilstm_final_30_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_30_f0_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_30_f0_prediction_kw,
    }
)


assert (
    bilstm_final_30_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_30_f0_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F0_result.csv"
)


bilstm_30_f0_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F0_history.csv"
)


bilstm_30_f0_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F0_predictions.csv"
)


bilstm_final_30_f0_result_df.to_csv(
    bilstm_30_f0_result_path,
    index=False,
)


bilstm_final_30_f0_history.to_csv(
    bilstm_30_f0_history_path,
    index=False,
)


bilstm_final_30_f0_prediction_table.to_csv(
    bilstm_30_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_30_f0_result_path,
    bilstm_30_f0_history_path,
    bilstm_30_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 19. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 30-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_30_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_30_f0_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_30_f0_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_30_f0_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_30_f0_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_30_f0_result_path,
)

print(
    " ",
    bilstm_30_f0_history_path,
)

print(
    " ",
    bilstm_30_f0_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 30-minute F1."
)

Epoch 1/16


2026-09-22 11:35:44.095234: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 11:35:44.095590: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 85ms/step - loss: 0.1876
Epoch 2/16
410/410 - 31s - 76ms/step - loss: 0.1660
Epoch 3/16
410/410 - 31s - 76ms/step - loss: 0.1652
Epoch 4/16
410/410 - 31s - 76ms/step - loss: 0.1634
Epoch 5/16
410/410 - 31s - 76ms/step - loss: 0.1667
Epoch 6/16
410/410 - 31s - 76ms/step - loss: 0.1802
Epoch 7/16
410/410 - 31s - 76ms/step - loss: 0.1806
Epoch 8/16
410/410 - 31s - 76ms/step - loss: 0.1867
Epoch 9/16
410/410 - 31s - 76ms/step - loss: 0.1888
Epoch 10/16
410/410 - 31s - 76ms/step - loss: 0.2077
Epoch 11/16
410/410 - 31s - 76ms/step - loss: 0.2171
Epoch 12/16
410/410 - 31s - 76ms/step - loss: 0.2069
Epoch 13/16
410/410 - 31s - 76ms/step - loss: 0.2282
Epoch 14/16
410/410 - 31s - 76ms/step - loss: 0.2369
Epoch 15/16
410/410 - 31s - 76ms/step - loss: 0.2404
Epoch 16/16
410/410 - 31s - 76ms/step - loss: 0.2600


2026-09-22 11:44:06.469441: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 11:44:06.469711: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


BiLSTM FINAL HELD-OUT TEST — 30-MIN F0


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-1.975688,6.432861,3154,0



Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205



BiLSTM 30-MIN F0 FINAL TEST: COMPLETE
Frozen epoch: 16
TEST MAE: 0.479948 kW
TEST RMSE: 0.789551 kW
TEST R2: 0.771205

Prediction clipping: False

Completed BiLSTM final TEST experiments: 6 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F0_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F0_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F0_predictions.csv

Next step after verification:
Run final BiLSTM 30-minute F1.


In [111]:
# ============================================================
# Step 95: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 30 minutes
#       feature family = F1
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 7
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 7 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_30_F1_HORIZON = 30

BILSTM_FINAL_30_F1_FAMILY = "F1"

BILSTM_FINAL_30_F1_KEY = (
    BILSTM_FINAL_30_F1_HORIZON,
    BILSTM_FINAL_30_F1_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Five 15-minute experiments + 30-minute F0
# should be complete.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
        (30, "F0"),
    }
)


assert (
    BILSTM_FINAL_30_F1_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 30-min F1 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_30_F1_KEY
    ]
    ==
    7
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F1"
    ]
    ==
    16
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F1"
    ]
    ==
    42_561
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_30_f1_result,
    bilstm_final_30_f1_history,
    bilstm_final_30_f1_actual_kw,
    bilstm_final_30_f1_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_30_F1_HORIZON,

    feature_family=
        BILSTM_FINAL_30_F1_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f1_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_30_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_30_f1_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    bilstm_final_30_f1_result[
        "trainable_parameter_count"
    ]
    ==
    42_561
)


assert (
    bilstm_final_30_f1_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    bilstm_final_30_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_30_f1_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    bilstm_final_30_f1_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f1_result[
        "frozen_best_epoch"
    ]
    ==
    7
)


assert (
    bilstm_final_30_f1_result[
        "final_epochs_trained"
    ]
    ==
    7
)


assert (
    len(
        bilstm_final_30_f1_history
    )
    ==
    7
)


assert (
    "val_loss"
    not in
    bilstm_final_30_f1_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_30_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_30_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_30_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f1_actual_kw.shape
    ==
    bilstm_final_30_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_30_f1_actual_kw
).all()


assert np.isfinite(
    bilstm_final_30_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# 30-min F0 and F1 must predict the exact same target vector.
# ------------------------------------------------------------

assert np.array_equal(
    bilstm_final_test_actuals[
        (
            30,
            "F0",
        )
    ],
    bilstm_final_30_f1_actual_kw,
)


print(
    "30-min F0/F1 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_30_F1_KEY
] = (
    bilstm_final_30_f1_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_30_F1_KEY
] = (
    bilstm_final_30_f1_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_30_F1_KEY
] = (
    bilstm_final_30_f1_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_30_F1_KEY
] = (
    bilstm_final_30_f1_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    7
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_30_f1_result_df = pd.DataFrame(
    [
        bilstm_final_30_f1_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 30-MIN F1"
)

print(
    "============================================================"
)


display(
    bilstm_final_30_f1_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_30_f1_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_30_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_30_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_30_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_30_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_30_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_30_f1_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_30_f1_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_30_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F0 -> F1 TEST comparison
#
# Positive percentage:
#
#       F1 has LOWER TEST MAE than F0.
#
# Negative percentage:
#
#       F1 has HIGHER TEST MAE than F0.
#
# This is descriptive only.
# ------------------------------------------------------------

bilstm_30_f0_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f1_mae = float(
    bilstm_final_30_f1_result[
        "test_MAE_kW"
    ]
)


bilstm_30_f0_to_f1_improvement_pct = (
    (
        bilstm_30_f0_mae
        -
        bilstm_30_f1_mae
    )
    /
    bilstm_30_f0_mae
    *
    100.0
)


print(
    "\n30-min F0 -> F1 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_30_f0_mae:.6f} kW"
)


print(
    f"  F1 MAE: {bilstm_30_f1_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_30_f0_to_f1_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. Build prediction table
# ------------------------------------------------------------

bilstm_final_30_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_30_f1_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_30_f1_prediction_kw,
    }
)


assert (
    bilstm_final_30_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 18. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_30_f1_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F1_result.csv"
)


bilstm_30_f1_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F1_history.csv"
)


bilstm_30_f1_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F1_predictions.csv"
)


bilstm_final_30_f1_result_df.to_csv(
    bilstm_30_f1_result_path,
    index=False,
)


bilstm_final_30_f1_history.to_csv(
    bilstm_30_f1_history_path,
    index=False,
)


bilstm_final_30_f1_prediction_table.to_csv(
    bilstm_30_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 19. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_30_f1_result_path,
    bilstm_30_f1_history_path,
    bilstm_30_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 20. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 30-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_30_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_30_f1_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_30_f1_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_30_f1_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_30_f1_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_30_f1_result_path,
)

print(
    " ",
    bilstm_30_f1_history_path,
)

print(
    " ",
    bilstm_30_f1_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 30-minute F2."
)

Epoch 1/7


2026-09-22 12:09:40.657989: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:09:40.658393: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 36s - 87ms/step - loss: 0.2140
Epoch 2/7
410/410 - 31s - 76ms/step - loss: 0.1843
Epoch 3/7
410/410 - 31s - 75ms/step - loss: 0.1906
Epoch 4/7
410/410 - 31s - 75ms/step - loss: 0.1791
Epoch 5/7
410/410 - 31s - 75ms/step - loss: 0.1763
Epoch 6/7
410/410 - 31s - 75ms/step - loss: 0.1882
Epoch 7/7
410/410 - 31s - 75ms/step - loss: 0.1915


2026-09-22 12:13:21.627554: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:13:21.627833: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

30-min F0/F1 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 30-MIN F1


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.295621,5.887377,3329,0



30-min F0 -> F1 TEST MAE change:
  F0 MAE: 0.479948 kW
  F1 MAE: 0.390785 kW
  Relative improvement: +18.577687%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312



BiLSTM 30-MIN F1 FINAL TEST: COMPLETE
Frozen epoch: 7
TEST MAE: 0.390785 kW
TEST RMSE: 0.715113 kW
TEST R2: 0.812312

Prediction clipping: False

Completed BiLSTM final TEST experiments: 7 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F1_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F1_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F1_predictions.csv

Next step after verification:
Run final BiLSTM 30-minute F2.


In [112]:
# ============================================================
# Step 96: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 30 minutes
#       feature family = F2
#
#
# SCIENTIFIC ROLE
# ---------------
#
# F2 adds the SOLAR-ASTRONOMY feature layer on top of F1.
#
# Therefore:
#
#       F1 -> F2
#
# asks whether solar-position / solar-geometry information adds
# predictive value beyond conventional temporal information.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 3
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 3 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_30_F2_HORIZON = 30

BILSTM_FINAL_30_F2_FAMILY = "F2"

BILSTM_FINAL_30_F2_KEY = (
    BILSTM_FINAL_30_F2_HORIZON,
    BILSTM_FINAL_30_F2_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed so far:
#
#       15-min: F0-F4
#       30-min: F0-F1
#
# Total = 7 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
        (30, "F0"),
        (30, "F1"),
    }
)


assert (
    BILSTM_FINAL_30_F2_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 30-min F2 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_30_F2_KEY
    ]
    ==
    3
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F2"
    ]
    ==
    21
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F2"
    ]
    ==
    42_721
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_30_f2_result,
    bilstm_final_30_f2_history,
    bilstm_final_30_f2_actual_kw,
    bilstm_final_30_f2_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_30_F2_HORIZON,

    feature_family=
        BILSTM_FINAL_30_F2_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f2_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_30_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_30_f2_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    bilstm_final_30_f2_result[
        "trainable_parameter_count"
    ]
    ==
    42_721
)


assert (
    bilstm_final_30_f2_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    bilstm_final_30_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_30_f2_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    bilstm_final_30_f2_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f2_result[
        "frozen_best_epoch"
    ]
    ==
    3
)


assert (
    bilstm_final_30_f2_result[
        "final_epochs_trained"
    ]
    ==
    3
)


assert (
    len(
        bilstm_final_30_f2_history
    )
    ==
    3
)


assert (
    "val_loss"
    not in
    bilstm_final_30_f2_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_30_f2_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_30_f2_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_30_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f2_actual_kw.shape
    ==
    bilstm_final_30_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_30_f2_actual_kw
).all()


assert np.isfinite(
    bilstm_final_30_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# F0, F1 and F2 must all forecast exactly the same
# 30-minute target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
        bilstm_final_30_f2_actual_kw,
    )


print(
    "30-min F0/F1/F2 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_30_F2_KEY
] = (
    bilstm_final_30_f2_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_30_F2_KEY
] = (
    bilstm_final_30_f2_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_30_F2_KEY
] = (
    bilstm_final_30_f2_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_30_F2_KEY
] = (
    bilstm_final_30_f2_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    8
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_30_f2_result_df = pd.DataFrame(
    [
        bilstm_final_30_f2_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 30-MIN F2"
)

print(
    "============================================================"
)


display(
    bilstm_final_30_f2_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_30_f2_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_30_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_30_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_30_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_30_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_30_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_30_f2_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_30_f2_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_30_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F1 -> F2 TEST comparison
#
# This isolates the solar-astronomy increment.
#
# Positive:
#       F2 has lower MAE than F1.
#
# Negative:
#       F2 has higher MAE than F1.
# ------------------------------------------------------------

bilstm_30_f1_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f2_mae = float(
    bilstm_final_30_f2_result[
        "test_MAE_kW"
    ]
)


bilstm_30_f1_to_f2_improvement_pct = (
    (
        bilstm_30_f1_mae
        -
        bilstm_30_f2_mae
    )
    /
    bilstm_30_f1_mae
    *
    100.0
)


print(
    "\n30-min F1 -> F2 TEST MAE change "
    "(addition of solar astronomy):"
)


print(
    f"  F1 MAE: {bilstm_30_f1_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_30_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_30_f1_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. F0 -> F2 descriptive comparison
# ------------------------------------------------------------

bilstm_30_f0_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f0_to_f2_improvement_pct = (
    (
        bilstm_30_f0_mae
        -
        bilstm_30_f2_mae
    )
    /
    bilstm_30_f0_mae
    *
    100.0
)


print(
    "\n30-min F0 -> F2 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_30_f0_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_30_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_30_f0_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_30_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_30_f2_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_30_f2_prediction_kw,
    }
)


assert (
    bilstm_final_30_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_30_f2_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F2_result.csv"
)


bilstm_30_f2_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F2_history.csv"
)


bilstm_30_f2_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F2_predictions.csv"
)


bilstm_final_30_f2_result_df.to_csv(
    bilstm_30_f2_result_path,
    index=False,
)


bilstm_final_30_f2_history.to_csv(
    bilstm_30_f2_history_path,
    index=False,
)


bilstm_final_30_f2_prediction_table.to_csv(
    bilstm_30_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_30_f2_result_path,
    bilstm_30_f2_history_path,
    bilstm_30_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 30-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_30_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_30_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_30_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_30_f2_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_30_f2_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_30_f2_result_path,
)

print(
    " ",
    bilstm_30_f2_history_path,
)

print(
    " ",
    bilstm_30_f2_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 30-minute F3 "
    "(continuous Sun-Moon astronomy)."
)

Epoch 1/3


2026-09-22 12:15:59.081845: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:15:59.086732: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 84ms/step - loss: 0.2097
Epoch 2/3
410/410 - 31s - 76ms/step - loss: 0.1738
Epoch 3/3
410/410 - 31s - 76ms/step - loss: 0.1745


2026-09-22 12:17:35.773929: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:17:35.774286: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

30-min F0/F1/F2 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 30-MIN F2


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.391061,5.934324,2850,0



30-min F1 -> F2 TEST MAE change (addition of solar astronomy):
  F1 MAE: 0.390785 kW
  F2 MAE: 0.358851 kW
  Relative improvement: +8.171885%

30-min F0 -> F2 TEST MAE change:
  F0 MAE: 0.479948 kW
  F2 MAE: 0.358851 kW
  Relative improvement: +25.231425%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160



BiLSTM 30-MIN F2 FINAL TEST: COMPLETE
Frozen epoch: 3
TEST MAE: 0.358851 kW
TEST RMSE: 0.707744 kW
TEST R2: 0.816160

Prediction clipping: False

Completed BiLSTM final TEST experiments: 8 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F2_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F2_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F2_predictions.csv

Next step after verification:
Run final BiLSTM 30-minute F3 (continuous Sun-Moon astronomy).


In [113]:
# ============================================================
# Step 97: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 30 minutes
#       feature family = F3
#
#
# SCIENTIFIC ROLE OF F3
# ---------------------
#
# F3 contains everything in F2:
#
#       history / meteorology
#       Gregorian temporal information
#       solar astronomy
#
# plus:
#
#       continuous lunar / Sun-Moon astronomical information
#
#
# Therefore:
#
#       F2 -> F3
#
# asks whether continuous lunar / Sun-Moon astronomy adds
# predictive information beyond the solar-astronomy layer.
#
#
# After this experiment, F4 will add the Panchang categories.
#
# Therefore the later:
#
#       F3 -> F4
#
# comparison remains our CORE novelty test.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 17
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 17 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_30_F3_HORIZON = 30

BILSTM_FINAL_30_F3_FAMILY = "F3"

BILSTM_FINAL_30_F3_KEY = (
    BILSTM_FINAL_30_F3_HORIZON,
    BILSTM_FINAL_30_F3_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed so far:
#
#       15-min: F0-F4
#       30-min: F0-F2
#
# Total = 8 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
    }
)


assert (
    BILSTM_FINAL_30_F3_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 30-min F3 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_30_F3_KEY
    ]
    ==
    17
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F3"
    ]
    ==
    30
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F3"
    ]
    ==
    43_009
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_30_f3_result,
    bilstm_final_30_f3_history,
    bilstm_final_30_f3_actual_kw,
    bilstm_final_30_f3_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_30_F3_HORIZON,

    feature_family=
        BILSTM_FINAL_30_F3_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f3_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_30_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_30_f3_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f3_result[
        "trainable_parameter_count"
    ]
    ==
    43_009
)


assert (
    bilstm_final_30_f3_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    bilstm_final_30_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_30_f3_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    bilstm_final_30_f3_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f3_result[
        "frozen_best_epoch"
    ]
    ==
    17
)


assert (
    bilstm_final_30_f3_result[
        "final_epochs_trained"
    ]
    ==
    17
)


assert (
    len(
        bilstm_final_30_f3_history
    )
    ==
    17
)


assert (
    "val_loss"
    not in
    bilstm_final_30_f3_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_30_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_30_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_30_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f3_actual_kw.shape
    ==
    bilstm_final_30_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_30_f3_actual_kw
).all()


assert np.isfinite(
    bilstm_final_30_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# F0-F3 at the 30-minute horizon must predict the exact same
# held-out target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
        bilstm_final_30_f3_actual_kw,
    )


print(
    "30-min F0/F1/F2/F3 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_30_F3_KEY
] = (
    bilstm_final_30_f3_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_30_F3_KEY
] = (
    bilstm_final_30_f3_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_30_F3_KEY
] = (
    bilstm_final_30_f3_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_30_F3_KEY
] = (
    bilstm_final_30_f3_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    9
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_30_f3_result_df = pd.DataFrame(
    [
        bilstm_final_30_f3_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 30-MIN F3"
)

print(
    "============================================================"
)


display(
    bilstm_final_30_f3_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
#
# NOTE:
#
# We do NOT alter training even if the TRAIN+VALIDATION loss
# begins increasing before epoch 17.
#
# Epoch 17 was chosen by the earlier validation-only protocol,
# before held-out TEST evaluation.
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_30_f3_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_30_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_30_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_30_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_30_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_30_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_30_f3_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_30_f3_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_30_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. IMPORTANT F2 -> F3 TEST comparison
#
# F2:
#
#       solar astronomy
#
# F3:
#
#       solar astronomy
#       +
#       continuous lunar / Sun-Moon astronomy
#
#
# Positive percentage:
#
#       F3 has LOWER TEST MAE than F2.
#
# Negative percentage:
#
#       F3 has HIGHER TEST MAE than F2.
#
# This is descriptive held-out performance.
# ------------------------------------------------------------

bilstm_30_f2_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f3_mae = float(
    bilstm_final_30_f3_result[
        "test_MAE_kW"
    ]
)


bilstm_30_f2_to_f3_improvement_pct = (
    (
        bilstm_30_f2_mae
        -
        bilstm_30_f3_mae
    )
    /
    bilstm_30_f2_mae
    *
    100.0
)


print(
    "\n============================================================"
)

print(
    "30-MIN F2 -> F3 CONTINUOUS LUNISOLAR COMPARISON"
)

print(
    "============================================================"
)


print(
    f"F2 TEST MAE: {bilstm_30_f2_mae:.6f} kW"
)


print(
    f"F3 TEST MAE: {bilstm_30_f3_mae:.6f} kW"
)


print(
    (
        "F2 -> F3 relative MAE improvement: "
        f"{bilstm_30_f2_to_f3_improvement_pct:+.6f}%"
    )
)


print(
    "Positive = F3 lower MAE than F2."
)


# ------------------------------------------------------------
# 17. F0 -> F3 reference comparison
# ------------------------------------------------------------

bilstm_30_f0_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f0_to_f3_improvement_pct = (
    (
        bilstm_30_f0_mae
        -
        bilstm_30_f3_mae
    )
    /
    bilstm_30_f0_mae
    *
    100.0
)


print(
    "\n30-min F0 -> F3 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_30_f0_mae:.6f} kW"
)


print(
    f"  F3 MAE: {bilstm_30_f3_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_30_f0_to_f3_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_30_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_30_f3_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_30_f3_prediction_kw,
    }
)


assert (
    bilstm_final_30_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_30_f3_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F3_result.csv"
)


bilstm_30_f3_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F3_history.csv"
)


bilstm_30_f3_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F3_predictions.csv"
)


bilstm_final_30_f3_result_df.to_csv(
    bilstm_30_f3_result_path,
    index=False,
)


bilstm_final_30_f3_history.to_csv(
    bilstm_30_f3_history_path,
    index=False,
)


bilstm_final_30_f3_prediction_table.to_csv(
    bilstm_30_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_30_f3_result_path,
    bilstm_30_f3_history_path,
    bilstm_30_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 30-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_30_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_30_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_30_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_30_f3_result['test_R2']:.6f}",
)


print(
    "\nF2 -> F3 TEST MAE improvement:",
    f"{bilstm_30_f2_to_f3_improvement_pct:+.6f}%",
)


print(
    "\nPrediction clipping:",
    bilstm_final_30_f3_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_30_f3_result_path,
)

print(
    " ",
    bilstm_30_f3_history_path,
)

print(
    " ",
    bilstm_30_f3_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 30-minute F4."
)

print(
    "That will produce the second BiLSTM held-out"
)

print(
    "F3 -> F4 Panchang comparison."
)

Epoch 1/17


2026-09-22 12:23:39.917854: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:23:39.918249: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 84ms/step - loss: 0.2161
Epoch 2/17
410/410 - 31s - 75ms/step - loss: 0.1790
Epoch 3/17
410/410 - 31s - 75ms/step - loss: 0.1798
Epoch 4/17
410/410 - 31s - 75ms/step - loss: 0.1790
Epoch 5/17
410/410 - 31s - 75ms/step - loss: 0.1912
Epoch 6/17
410/410 - 31s - 75ms/step - loss: 0.1908
Epoch 7/17
410/410 - 31s - 76ms/step - loss: 0.2035
Epoch 8/17
410/410 - 31s - 76ms/step - loss: 0.2003
Epoch 9/17
410/410 - 31s - 75ms/step - loss: 0.2032
Epoch 10/17
410/410 - 31s - 75ms/step - loss: 0.1996
Epoch 11/17
410/410 - 32s - 77ms/step - loss: 0.2090
Epoch 12/17
410/410 - 31s - 76ms/step - loss: 0.2041
Epoch 13/17
410/410 - 31s - 76ms/step - loss: 0.2082
Epoch 14/17
410/410 - 31s - 76ms/step - loss: 0.2278
Epoch 15/17
410/410 - 31s - 75ms/step - loss: 0.2233
Epoch 16/17
410/410 - 31s - 77ms/step - loss: 0.2125
Epoch 17/17
410/410 - 31s - 76ms/step - loss: 0.2190


2026-09-22 12:32:30.847012: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:32:30.847314: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

30-min F0/F1/F2/F3 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 30-MIN F3


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.728233,5.633871,5888,0



30-MIN F2 -> F3 CONTINUOUS LUNISOLAR COMPARISON
F2 TEST MAE: 0.358851 kW
F3 TEST MAE: 0.429713 kW
F2 -> F3 relative MAE improvement: -19.747141%
Positive = F3 lower MAE than F2.

30-min F0 -> F3 TEST MAE change:
  F0 MAE: 0.479948 kW
  F3 MAE: 0.429713 kW
  Relative improvement: +10.466769%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178



BiLSTM 30-MIN F3 FINAL TEST: COMPLETE
Frozen epoch: 17
TEST MAE: 0.429713 kW
TEST RMSE: 0.722947 kW
TEST R2: 0.808178

F2 -> F3 TEST MAE improvement: -19.747141%

Prediction clipping: False

Completed BiLSTM final TEST experiments: 9 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F3_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F3_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_30min_F3_predictions.csv

Next step after verification:
Run final BiLSTM 30-minute F4.
That will produce the second BiLSTM held-out
F3 -> F4 Panchang comparison.


In [114]:
# ============================================================
# Step 98: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 30 minutes
#       feature family = F4
#
#
# SCIENTIFIC ROLE OF F4
# ---------------------
#
# F4 contains everything in F3:
#
#       history / meteorology
#       Gregorian temporal information
#       solar astronomy
#       continuous lunar / Sun-Moon astronomy
#
# plus the Panchang categorical representation:
#
#       Tithi
#       Paksha
#       Karana
#       Nakshatra
#       Yoga
#
#
# Therefore:
#
#       F3 -> F4
#
# is our CORE novelty comparison.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 3
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 3 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_30_F4_HORIZON = 30

BILSTM_FINAL_30_F4_FAMILY = "F4"

BILSTM_FINAL_30_F4_KEY = (
    BILSTM_FINAL_30_F4_HORIZON,
    BILSTM_FINAL_30_F4_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed so far:
#
#       15-min: F0-F4
#       30-min: F0-F3
#
# Total = 9 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),
        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
    }
)


assert (
    BILSTM_FINAL_30_F4_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 30-min F4 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_30_F4_KEY
    ]
    ==
    3
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F4"
    ]
    ==
    127
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F4"
    ]
    ==
    46_113
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_30_f4_result,
    bilstm_final_30_f4_history,
    bilstm_final_30_f4_actual_kw,
    bilstm_final_30_f4_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_30_F4_HORIZON,

    feature_family=
        BILSTM_FINAL_30_F4_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f4_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_30_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_30_f4_result[
        "horizon_minutes"
    ]
    ==
    30
)


assert (
    bilstm_final_30_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    bilstm_final_30_f4_result[
        "trainable_parameter_count"
    ]
    ==
    46_113
)


assert (
    bilstm_final_30_f4_result[
        "n_train"
    ]
    ==
    87_258
)


assert (
    bilstm_final_30_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_30_f4_result[
        "n_trainval"
    ]
    ==
    104_826
)


assert (
    bilstm_final_30_f4_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f4_result[
        "frozen_best_epoch"
    ]
    ==
    3
)


assert (
    bilstm_final_30_f4_result[
        "final_epochs_trained"
    ]
    ==
    3
)


assert (
    len(
        bilstm_final_30_f4_history
    )
    ==
    3
)


assert (
    "val_loss"
    not in
    bilstm_final_30_f4_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_30_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_30_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_30_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_30_f4_actual_kw.shape
    ==
    bilstm_final_30_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_30_f4_actual_kw
).all()


assert np.isfinite(
    bilstm_final_30_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Complete 30-minute target alignment
#
# F0-F4 must forecast the exact same target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                30,
                previous_family,
            )
        ],
        bilstm_final_30_f4_actual_kw,
    )


print(
    "30-min F0/F1/F2/F3/F4 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_30_F4_KEY
] = (
    bilstm_final_30_f4_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_30_F4_KEY
] = (
    bilstm_final_30_f4_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_30_F4_KEY
] = (
    bilstm_final_30_f4_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_30_F4_KEY
] = (
    bilstm_final_30_f4_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    10
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_30_f4_result_df = pd.DataFrame(
    [
        bilstm_final_30_f4_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 30-MIN F4"
)

print(
    "============================================================"
)


display(
    bilstm_final_30_f4_result_df
)


# ------------------------------------------------------------
# 14. Final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_30_f4_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_30_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_30_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_30_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_30_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_30_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_30_f4_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_30_f4_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_30_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. CORE F3 -> F4 TEST comparison
#
# Positive:
#
#       F4 has LOWER MAE than F3.
#
# Negative:
#
#       F4 has HIGHER MAE than F3.
#
# This is the central Panchang increment.
# ------------------------------------------------------------

bilstm_30_f3_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f4_mae = float(
    bilstm_final_30_f4_result[
        "test_MAE_kW"
    ]
)


bilstm_30_f3_to_f4_improvement_pct = (
    (
        bilstm_30_f3_mae
        -
        bilstm_30_f4_mae
    )
    /
    bilstm_30_f3_mae
    *
    100.0
)


print(
    "\n============================================================"
)

print(
    "CORE 30-MIN F3 -> F4 PANCHANG COMPARISON"
)

print(
    "============================================================"
)


print(
    f"F3 TEST MAE: {bilstm_30_f3_mae:.6f} kW"
)


print(
    f"F4 TEST MAE: {bilstm_30_f4_mae:.6f} kW"
)


print(
    (
        "F3 -> F4 relative MAE improvement: "
        f"{bilstm_30_f3_to_f4_improvement_pct:+.6f}%"
    )
)


print(
    "Positive = F4 lower MAE than F3."
)


# ------------------------------------------------------------
# 17. F2 -> F4 descriptive comparison
#
# This is useful for the complete ablation, but is NOT the
# primary novelty test because both lunar astronomy and
# Panchang differ relative to F2.
# ------------------------------------------------------------

bilstm_30_f2_mae = float(
    bilstm_final_test_results[
        (
            30,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_30_f2_to_f4_improvement_pct = (
    (
        bilstm_30_f2_mae
        -
        bilstm_30_f4_mae
    )
    /
    bilstm_30_f2_mae
    *
    100.0
)


print(
    "\n30-min F2 -> F4 TEST MAE change:"
)


print(
    f"  F2 MAE: {bilstm_30_f2_mae:.6f} kW"
)


print(
    f"  F4 MAE: {bilstm_30_f4_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_30_f2_to_f4_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Complete 30-minute BiLSTM ablation table
# ------------------------------------------------------------

bilstm_30min_ablation_table = (
    pd.DataFrame(
        [
            bilstm_final_test_results[
                (
                    30,
                    feature_family,
                )
            ]

            for feature_family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        ]
    )
    [
        [
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "frozen_best_epoch",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


print(
    "\nComplete BiLSTM 30-minute ablation:"
)


display(
    bilstm_30min_ablation_table
)


# ------------------------------------------------------------
# 19. Compact BiLSTM F3 -> F4 status so far
#
# After this cell we will have:
#
#       15-minute comparison
#       30-minute comparison
#
# 60-minute remains untouched.
# ------------------------------------------------------------

bilstm_f3_f4_partial_summary = pd.DataFrame(
    [
        {
            "horizon_minutes":
                15,

            "F3_MAE_kW":
                float(
                    bilstm_final_test_results[
                        (15, "F3")
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F4_MAE_kW":
                float(
                    bilstm_final_test_results[
                        (15, "F4")
                    ][
                        "test_MAE_kW"
                    ]
                ),

            "F3_to_F4_MAE_improvement_pct":
                float(
                    bilstm_15_f3_to_f4_improvement_pct
                ),
        },

        {
            "horizon_minutes":
                30,

            "F3_MAE_kW":
                bilstm_30_f3_mae,

            "F4_MAE_kW":
                bilstm_30_f4_mae,

            "F3_to_F4_MAE_improvement_pct":
                bilstm_30_f3_to_f4_improvement_pct,
        },
    ]
)


print(
    "\nBiLSTM F3 -> F4 held-out TEST status so far:"
)


display(
    bilstm_f3_f4_partial_summary
)


# ------------------------------------------------------------
# 20. Build prediction table
# ------------------------------------------------------------

bilstm_final_30_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_30_f4_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_30_f4_prediction_kw,
    }
)


assert (
    bilstm_final_30_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 21. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_30_f4_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F4_result.csv"
)


bilstm_30_f4_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F4_history.csv"
)


bilstm_30_f4_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_F4_predictions.csv"
)


bilstm_30min_ablation_path = (
    RESULTS_DIR
    /
    "bilstm_final_30min_ablation.csv"
)


bilstm_f3_f4_partial_summary_path = (
    RESULTS_DIR
    /
    "bilstm_f3_f4_partial_test_summary.csv"
)


bilstm_final_30_f4_result_df.to_csv(
    bilstm_30_f4_result_path,
    index=False,
)


bilstm_final_30_f4_history.to_csv(
    bilstm_30_f4_history_path,
    index=False,
)


bilstm_final_30_f4_prediction_table.to_csv(
    bilstm_30_f4_predictions_path,
    index=False,
)


bilstm_30min_ablation_table.to_csv(
    bilstm_30min_ablation_path,
    index=False,
)


bilstm_f3_f4_partial_summary.to_csv(
    bilstm_f3_f4_partial_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 22. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_30_f4_result_path,
    bilstm_30_f4_history_path,
    bilstm_30_f4_predictions_path,
    bilstm_30min_ablation_path,
    bilstm_f3_f4_partial_summary_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 23. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 24. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 30-MIN F4 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_30_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_30_f4_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_30_f4_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_30_f4_result['test_R2']:.6f}",
)


print(
    "\nCORE F3 -> F4 TEST MAE improvement:",
    f"{bilstm_30_f3_to_f4_improvement_pct:+.6f}%",
)


print(
    "\nPrediction clipping:",
    bilstm_final_30_f4_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nNext step after verification:"
)

print(
    "Begin the 60-minute BiLSTM final TEST block,"
)

print(
    "starting with F0."
)

Epoch 1/3


2026-09-22 12:34:09.056161: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:34:09.056470: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 85ms/step - loss: 0.2106
Epoch 2/3
410/410 - 31s - 77ms/step - loss: 0.1748
Epoch 3/3
410/410 - 31s - 76ms/step - loss: 0.1865


2026-09-22 12:35:46.532674: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:35:46.532990: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

30-min F0/F1/F2/F3/F4 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 30-MIN F4


...


BiLSTM F3 -> F4 held-out TEST status so far:


,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct
0,15,0.308417,0.303657,1.543316
1,30,0.429713,0.392481,8.664420



Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178
9,30,F4,3,46113,0.392481,0.734626,0.801930



BiLSTM 30-MIN F4 FINAL TEST: COMPLETE
Frozen epoch: 3
TEST MAE: 0.392481 kW
TEST RMSE: 0.734626 kW
TEST R2: 0.801930

CORE F3 -> F4 TEST MAE improvement: +8.664420%

Prediction clipping: False

Completed BiLSTM final TEST experiments: 10 / 15

Next step after verification:
Begin the 60-minute BiLSTM final TEST block,
starting with F0.


In [115]:
# ============================================================
# Step 99: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 60 minutes
#       feature family = F0
#
#
# This begins the FINAL horizon block for BiLSTM.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 6
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 6 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_60_F0_HORIZON = 60

BILSTM_FINAL_60_F0_FAMILY = "F0"

BILSTM_FINAL_60_F0_KEY = (
    BILSTM_FINAL_60_F0_HORIZON,
    BILSTM_FINAL_60_F0_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed so far:
#
#       15-min: F0-F4
#       30-min: F0-F4
#
# Total = 10 experiments.
#
# No 60-minute TEST experiment should exist yet.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),

        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
        (30, "F4"),
    }
)


assert (
    BILSTM_FINAL_60_F0_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 60-min F0 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_60_F0_KEY
    ]
    ==
    6
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F0"
    ]
    ==
    1
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F0"
    ]
    ==
    42_081
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_60_f0_result,
    bilstm_final_60_f0_history,
    bilstm_final_60_f0_actual_kw,
    bilstm_final_60_f0_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_60_F0_HORIZON,

    feature_family=
        BILSTM_FINAL_60_F0_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f0_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_60_f0_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_60_f0_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    bilstm_final_60_f0_result[
        "feature_family"
    ]
    ==
    "F0"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f0_result[
        "auxiliary_dim"
    ]
    ==
    1
)


assert (
    bilstm_final_60_f0_result[
        "trainable_parameter_count"
    ]
    ==
    42_081
)


assert (
    bilstm_final_60_f0_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    bilstm_final_60_f0_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_60_f0_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    bilstm_final_60_f0_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f0_result[
        "frozen_best_epoch"
    ]
    ==
    6
)


assert (
    bilstm_final_60_f0_result[
        "final_epochs_trained"
    ]
    ==
    6
)


assert (
    len(
        bilstm_final_60_f0_history
    )
    ==
    6
)


assert (
    "val_loss"
    not in
    bilstm_final_60_f0_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f0_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_60_f0_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_60_f0_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_60_f0_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f0_actual_kw.shape
    ==
    bilstm_final_60_f0_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_60_f0_actual_kw
).all()


assert np.isfinite(
    bilstm_final_60_f0_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_60_F0_KEY
] = (
    bilstm_final_60_f0_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_60_F0_KEY
] = (
    bilstm_final_60_f0_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_60_F0_KEY
] = (
    bilstm_final_60_f0_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_60_F0_KEY
] = (
    bilstm_final_60_f0_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    11
)


# ------------------------------------------------------------
# 12. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_60_f0_result_df = pd.DataFrame(
    [
        bilstm_final_60_f0_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 60-MIN F0"
)

print(
    "============================================================"
)


display(
    bilstm_final_60_f0_result_df
)


# ------------------------------------------------------------
# 13. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_60_f0_history
)


# ------------------------------------------------------------
# 14. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_60_f0_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_60_f0_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_60_f0_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_60_f0_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_60_f0_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_60_f0_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_60_f0_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_60_f0_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 15. Build prediction table
# ------------------------------------------------------------

bilstm_final_60_f0_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_60_f0_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_60_f0_prediction_kw,
    }
)


assert (
    bilstm_final_60_f0_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 16. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_60_f0_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F0_result.csv"
)


bilstm_60_f0_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F0_history.csv"
)


bilstm_60_f0_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F0_predictions.csv"
)


bilstm_final_60_f0_result_df.to_csv(
    bilstm_60_f0_result_path,
    index=False,
)


bilstm_final_60_f0_history.to_csv(
    bilstm_60_f0_history_path,
    index=False,
)


bilstm_final_60_f0_prediction_table.to_csv(
    bilstm_60_f0_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 17. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_60_f0_result_path,
    bilstm_60_f0_history_path,
    bilstm_60_f0_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 18. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 19. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 60-MIN F0 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_60_f0_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_60_f0_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_60_f0_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_60_f0_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_60_f0_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_60_f0_result_path,
)

print(
    " ",
    bilstm_60_f0_history_path,
)

print(
    " ",
    bilstm_60_f0_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 60-minute F1."
)

Epoch 1/6


2026-09-22 12:43:01.531279: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:43:01.531663: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 82ms/step - loss: 0.2388
Epoch 2/6
410/410 - 31s - 77ms/step - loss: 0.2186
Epoch 3/6
410/410 - 31s - 76ms/step - loss: 0.2167
Epoch 4/6
410/410 - 31s - 76ms/step - loss: 0.2147
Epoch 5/6
410/410 - 31s - 76ms/step - loss: 0.2308
Epoch 6/6
410/410 - 31s - 76ms/step - loss: 0.2293


2026-09-22 12:46:11.248433: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:46:11.248747: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


BiLSTM FINAL HELD-OUT TEST — 60-MIN F0


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.286217,6.368788,304,0



Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178
9,30,F4,3,46113,0.392481,0.734626,0.801930



BiLSTM 60-MIN F0 FINAL TEST: COMPLETE
Frozen epoch: 6
TEST MAE: 0.508764 kW
TEST RMSE: 0.895735 kW
TEST R2: 0.705527

Prediction clipping: False

Completed BiLSTM final TEST experiments: 11 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F0_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F0_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F0_predictions.csv

Next step after verification:
Run final BiLSTM 60-minute F1.


In [116]:
# ============================================================
# Step 100: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 60 minutes
#       feature family = F1
#
#
# SCIENTIFIC ROLE
# ---------------
#
# F1 augments the basic F0 forecasting information with the
# conventional Gregorian / temporal feature representation.
#
# Therefore:
#
#       F0 -> F1
#
# describes the effect of adding the conventional calendar /
# temporal representation at the 60-minute horizon.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 6
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 6 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_60_F1_HORIZON = 60

BILSTM_FINAL_60_F1_FAMILY = "F1"

BILSTM_FINAL_60_F1_KEY = (
    BILSTM_FINAL_60_F1_HORIZON,
    BILSTM_FINAL_60_F1_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed so far:
#
#       15-min: F0-F4
#       30-min: F0-F4
#       60-min: F0
#
# Total = 11 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),

        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
        (30, "F4"),

        (60, "F0"),
    }
)


assert (
    BILSTM_FINAL_60_F1_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 60-min F1 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_60_F1_KEY
    ]
    ==
    6
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F1"
    ]
    ==
    16
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F1"
    ]
    ==
    42_561
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_60_f1_result,
    bilstm_final_60_f1_history,
    bilstm_final_60_f1_actual_kw,
    bilstm_final_60_f1_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_60_F1_HORIZON,

    feature_family=
        BILSTM_FINAL_60_F1_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f1_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_60_f1_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_60_f1_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    bilstm_final_60_f1_result[
        "feature_family"
    ]
    ==
    "F1"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f1_result[
        "auxiliary_dim"
    ]
    ==
    16
)


assert (
    bilstm_final_60_f1_result[
        "trainable_parameter_count"
    ]
    ==
    42_561
)


assert (
    bilstm_final_60_f1_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    bilstm_final_60_f1_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_60_f1_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    bilstm_final_60_f1_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f1_result[
        "frozen_best_epoch"
    ]
    ==
    6
)


assert (
    bilstm_final_60_f1_result[
        "final_epochs_trained"
    ]
    ==
    6
)


assert (
    len(
        bilstm_final_60_f1_history
    )
    ==
    6
)


assert (
    "val_loss"
    not in
    bilstm_final_60_f1_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f1_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_60_f1_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_60_f1_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_60_f1_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f1_actual_kw.shape
    ==
    bilstm_final_60_f1_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_60_f1_actual_kw
).all()


assert np.isfinite(
    bilstm_final_60_f1_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# F0 and F1 at 60 minutes must forecast exactly the same
# held-out target vector.
# ------------------------------------------------------------

assert np.array_equal(
    bilstm_final_test_actuals[
        (
            60,
            "F0",
        )
    ],
    bilstm_final_60_f1_actual_kw,
)


print(
    "60-min F0/F1 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_60_F1_KEY
] = (
    bilstm_final_60_f1_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_60_F1_KEY
] = (
    bilstm_final_60_f1_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_60_F1_KEY
] = (
    bilstm_final_60_f1_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_60_F1_KEY
] = (
    bilstm_final_60_f1_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    12
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_60_f1_result_df = pd.DataFrame(
    [
        bilstm_final_60_f1_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 60-MIN F1"
)

print(
    "============================================================"
)


display(
    bilstm_final_60_f1_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_60_f1_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_60_f1_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_60_f1_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_60_f1_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_60_f1_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_60_f1_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_60_f1_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_60_f1_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_60_f1_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F0 -> F1 TEST comparison
#
# Positive percentage:
#
#       F1 has LOWER TEST MAE than F0.
#
# Negative percentage:
#
#       F1 has HIGHER TEST MAE than F0.
#
# Descriptive only.
# ------------------------------------------------------------

bilstm_60_f0_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f1_mae = float(
    bilstm_final_60_f1_result[
        "test_MAE_kW"
    ]
)


bilstm_60_f0_to_f1_improvement_pct = (
    (
        bilstm_60_f0_mae
        -
        bilstm_60_f1_mae
    )
    /
    bilstm_60_f0_mae
    *
    100.0
)


print(
    "\n60-min F0 -> F1 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_60_f0_mae:.6f} kW"
)


print(
    f"  F1 MAE: {bilstm_60_f1_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_60_f0_to_f1_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. Build prediction table
# ------------------------------------------------------------

bilstm_final_60_f1_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_60_f1_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_60_f1_prediction_kw,
    }
)


assert (
    bilstm_final_60_f1_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 18. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_60_f1_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F1_result.csv"
)


bilstm_60_f1_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F1_history.csv"
)


bilstm_60_f1_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F1_predictions.csv"
)


bilstm_final_60_f1_result_df.to_csv(
    bilstm_60_f1_result_path,
    index=False,
)


bilstm_final_60_f1_history.to_csv(
    bilstm_60_f1_history_path,
    index=False,
)


bilstm_final_60_f1_prediction_table.to_csv(
    bilstm_60_f1_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 19. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_60_f1_result_path,
    bilstm_60_f1_history_path,
    bilstm_60_f1_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 20. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 60-MIN F1 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_60_f1_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_60_f1_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_60_f1_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_60_f1_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_60_f1_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_60_f1_result_path,
)

print(
    " ",
    bilstm_60_f1_history_path,
)

print(
    " ",
    bilstm_60_f1_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 60-minute F2."
)

Epoch 1/6


2026-09-22 12:48:13.991792: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:48:13.992095: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 83ms/step - loss: 0.2665
Epoch 2/6
410/410 - 31s - 76ms/step - loss: 0.2308
Epoch 3/6
410/410 - 31s - 76ms/step - loss: 0.2225
Epoch 4/6
410/410 - 31s - 76ms/step - loss: 0.2304
Epoch 5/6
410/410 - 31s - 76ms/step - loss: 0.2271
Epoch 6/6
410/410 - 31s - 76ms/step - loss: 0.2337


2026-09-22 12:51:24.011618: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 12:51:24.011900: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

60-min F0/F1 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 60-MIN F1


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.470577,5.899073,6757,0



60-min F0 -> F1 TEST MAE change:
  F0 MAE: 0.508764 kW
  F1 MAE: 0.479160 kW
  Relative improvement: +5.818823%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178
9,30,F4,3,46113,0.392481,0.734626,0.801930



BiLSTM 60-MIN F1 FINAL TEST: COMPLETE
Frozen epoch: 6
TEST MAE: 0.479160 kW
TEST RMSE: 0.861954 kW
TEST R2: 0.727319

Prediction clipping: False

Completed BiLSTM final TEST experiments: 12 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F1_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F1_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F1_predictions.csv

Next step after verification:
Run final BiLSTM 60-minute F2.


In [117]:
# ============================================================
# Step 101: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 60 minutes
#       feature family = F2
#
#
# SCIENTIFIC ROLE
# ---------------
#
# F2 adds the SOLAR-ASTRONOMY layer on top of F1.
#
# Therefore:
#
#       F1 -> F2
#
# asks whether solar-position / solar-geometry information
# improves 60-minute forecasting beyond the conventional
# Gregorian / temporal representation.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 6
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 6 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_60_F2_HORIZON = 60

BILSTM_FINAL_60_F2_FAMILY = "F2"

BILSTM_FINAL_60_F2_KEY = (
    BILSTM_FINAL_60_F2_HORIZON,
    BILSTM_FINAL_60_F2_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed:
#
#       15-min: F0-F4
#       30-min: F0-F4
#       60-min: F0-F1
#
# Total = 12 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),

        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
        (30, "F4"),

        (60, "F0"),
        (60, "F1"),
    }
)


assert (
    BILSTM_FINAL_60_F2_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 60-min F2 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_60_F2_KEY
    ]
    ==
    6
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F2"
    ]
    ==
    21
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F2"
    ]
    ==
    42_721
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_60_f2_result,
    bilstm_final_60_f2_history,
    bilstm_final_60_f2_actual_kw,
    bilstm_final_60_f2_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_60_F2_HORIZON,

    feature_family=
        BILSTM_FINAL_60_F2_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f2_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_60_f2_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_60_f2_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    bilstm_final_60_f2_result[
        "feature_family"
    ]
    ==
    "F2"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f2_result[
        "auxiliary_dim"
    ]
    ==
    21
)


assert (
    bilstm_final_60_f2_result[
        "trainable_parameter_count"
    ]
    ==
    42_721
)


assert (
    bilstm_final_60_f2_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    bilstm_final_60_f2_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_60_f2_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    bilstm_final_60_f2_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f2_result[
        "frozen_best_epoch"
    ]
    ==
    6
)


assert (
    bilstm_final_60_f2_result[
        "final_epochs_trained"
    ]
    ==
    6
)


assert (
    len(
        bilstm_final_60_f2_history
    )
    ==
    6
)


assert (
    "val_loss"
    not in
    bilstm_final_60_f2_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f2_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_60_f2_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_60_f2_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_60_f2_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f2_actual_kw.shape
    ==
    bilstm_final_60_f2_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_60_f2_actual_kw
).all()


assert np.isfinite(
    bilstm_final_60_f2_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# F0, F1 and F2 at the 60-minute horizon must forecast
# exactly the same held-out target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
        bilstm_final_60_f2_actual_kw,
    )


print(
    "60-min F0/F1/F2 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_60_F2_KEY
] = (
    bilstm_final_60_f2_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_60_F2_KEY
] = (
    bilstm_final_60_f2_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_60_F2_KEY
] = (
    bilstm_final_60_f2_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_60_F2_KEY
] = (
    bilstm_final_60_f2_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    13
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_60_f2_result_df = pd.DataFrame(
    [
        bilstm_final_60_f2_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 60-MIN F2"
)

print(
    "============================================================"
)


display(
    bilstm_final_60_f2_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_60_f2_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_60_f2_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_60_f2_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_60_f2_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_60_f2_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_60_f2_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_60_f2_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_60_f2_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_60_f2_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. F1 -> F2 TEST comparison
#
# This isolates the solar-astronomy increment.
#
# Positive:
#
#       F2 has LOWER TEST MAE than F1.
#
# Negative:
#
#       F2 has HIGHER TEST MAE than F1.
# ------------------------------------------------------------

bilstm_60_f1_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F1",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f2_mae = float(
    bilstm_final_60_f2_result[
        "test_MAE_kW"
    ]
)


bilstm_60_f1_to_f2_improvement_pct = (
    (
        bilstm_60_f1_mae
        -
        bilstm_60_f2_mae
    )
    /
    bilstm_60_f1_mae
    *
    100.0
)


print(
    "\n60-min F1 -> F2 TEST MAE change "
    "(addition of solar astronomy):"
)


print(
    f"  F1 MAE: {bilstm_60_f1_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_60_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_60_f1_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 17. F0 -> F2 reference comparison
# ------------------------------------------------------------

bilstm_60_f0_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f0_to_f2_improvement_pct = (
    (
        bilstm_60_f0_mae
        -
        bilstm_60_f2_mae
    )
    /
    bilstm_60_f0_mae
    *
    100.0
)


print(
    "\n60-min F0 -> F2 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_60_f0_mae:.6f} kW"
)


print(
    f"  F2 MAE: {bilstm_60_f2_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_60_f0_to_f2_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_60_f2_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_60_f2_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_60_f2_prediction_kw,
    }
)


assert (
    bilstm_final_60_f2_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_60_f2_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F2_result.csv"
)


bilstm_60_f2_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F2_history.csv"
)


bilstm_60_f2_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F2_predictions.csv"
)


bilstm_final_60_f2_result_df.to_csv(
    bilstm_60_f2_result_path,
    index=False,
)


bilstm_final_60_f2_history.to_csv(
    bilstm_60_f2_history_path,
    index=False,
)


bilstm_final_60_f2_prediction_table.to_csv(
    bilstm_60_f2_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_60_f2_result_path,
    bilstm_60_f2_history_path,
    bilstm_60_f2_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 60-MIN F2 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_60_f2_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_60_f2_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_60_f2_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_60_f2_result['test_R2']:.6f}",
)


print(
    "\nPrediction clipping:",
    bilstm_final_60_f2_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_60_f2_result_path,
)

print(
    " ",
    bilstm_60_f2_history_path,
)

print(
    " ",
    bilstm_60_f2_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run final BiLSTM 60-minute F3 "
    "(continuous Sun-Moon astronomy)."
)

Epoch 1/6


2026-09-22 12:56:53.635300: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 12:56:53.635665: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 34s - 84ms/step - loss: 0.2640
Epoch 2/6
410/410 - 31s - 76ms/step - loss: 0.2267
Epoch 3/6
410/410 - 31s - 76ms/step - loss: 0.2278
Epoch 4/6
410/410 - 31s - 76ms/step - loss: 0.2232
Epoch 5/6
410/410 - 31s - 75ms/step - loss: 0.2324
Epoch 6/6
410/410 - 31s - 77ms/step - loss: 0.2355


2026-09-22 13:00:03.661100: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 13:00:03.661396: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

60-min F0/F1/F2 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 60-MIN F2


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.585505,6.208127,1062,0



60-min F1 -> F2 TEST MAE change (addition of solar astronomy):
  F1 MAE: 0.479160 kW
  F2 MAE: 0.507799 kW
  Relative improvement: -5.977025%

60-min F0 -> F2 TEST MAE change:
  F0 MAE: 0.508764 kW
  F2 MAE: 0.507799 kW
  Relative improvement: +0.189590%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178
9,30,F4,3,46113,0.392481,0.734626,0.801930



BiLSTM 60-MIN F2 FINAL TEST: COMPLETE
Frozen epoch: 6
TEST MAE: 0.507799 kW
TEST RMSE: 0.855739 kW
TEST R2: 0.731238

Prediction clipping: False

Completed BiLSTM final TEST experiments: 13 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F2_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F2_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F2_predictions.csv

Next step after verification:
Run final BiLSTM 60-minute F3 (continuous Sun-Moon astronomy).


In [118]:
# ============================================================
# Step 102: Final BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 60 minutes
#       feature family = F3
#
#
# SCIENTIFIC ROLE OF F3
# ---------------------
#
# F3 contains everything already available in F2:
#
#       history / meteorological information
#       Gregorian temporal information
#       solar astronomy
#
# and additionally introduces:
#
#       continuous lunar / Sun-Moon astronomical information
#
#
# Therefore:
#
#       F2 -> F3
#
# asks whether continuous lunisolar astronomy adds predictive
# information beyond the solar-astronomy layer.
#
#
# The NEXT and FINAL BiLSTM experiment will be F4:
#
#       F3 -> F4
#
# which is our core Panchang comparison.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 5
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 5 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_60_F3_HORIZON = 60

BILSTM_FINAL_60_F3_FAMILY = "F3"

BILSTM_FINAL_60_F3_KEY = (
    BILSTM_FINAL_60_F3_HORIZON,
    BILSTM_FINAL_60_F3_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed:
#
#       15-min: F0-F4
#       30-min: F0-F4
#       60-min: F0-F2
#
# Total = 13 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),

        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
        (30, "F4"),

        (60, "F0"),
        (60, "F1"),
        (60, "F2"),
    }
)


assert (
    BILSTM_FINAL_60_F3_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 60-min F3 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_60_F3_KEY
    ]
    ==
    5
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F3"
    ]
    ==
    30
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F3"
    ]
    ==
    43_009
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_60_f3_result,
    bilstm_final_60_f3_history,
    bilstm_final_60_f3_actual_kw,
    bilstm_final_60_f3_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_60_F3_HORIZON,

    feature_family=
        BILSTM_FINAL_60_F3_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f3_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_60_f3_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_60_f3_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    bilstm_final_60_f3_result[
        "feature_family"
    ]
    ==
    "F3"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f3_result[
        "auxiliary_dim"
    ]
    ==
    30
)


assert (
    bilstm_final_60_f3_result[
        "trainable_parameter_count"
    ]
    ==
    43_009
)


assert (
    bilstm_final_60_f3_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    bilstm_final_60_f3_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_60_f3_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    bilstm_final_60_f3_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f3_result[
        "frozen_best_epoch"
    ]
    ==
    5
)


assert (
    bilstm_final_60_f3_result[
        "final_epochs_trained"
    ]
    ==
    5
)


assert (
    len(
        bilstm_final_60_f3_history
    )
    ==
    5
)


assert (
    "val_loss"
    not in
    bilstm_final_60_f3_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f3_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_60_f3_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_60_f3_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_60_f3_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f3_actual_kw.shape
    ==
    bilstm_final_60_f3_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_60_f3_actual_kw
).all()


assert np.isfinite(
    bilstm_final_60_f3_prediction_kw
).all()


# ------------------------------------------------------------
# 11. Same-horizon target alignment
#
# F0-F3 at the 60-minute horizon must forecast exactly the
# same held-out target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
        bilstm_final_60_f3_actual_kw,
    )


print(
    "60-min F0/F1/F2/F3 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_60_F3_KEY
] = (
    bilstm_final_60_f3_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_60_F3_KEY
] = (
    bilstm_final_60_f3_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_60_F3_KEY
] = (
    bilstm_final_60_f3_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_60_F3_KEY
] = (
    bilstm_final_60_f3_prediction_kw.copy()
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    14
)


# ------------------------------------------------------------
# 13. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_60_f3_result_df = pd.DataFrame(
    [
        bilstm_final_60_f3_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 60-MIN F3"
)

print(
    "============================================================"
)


display(
    bilstm_final_60_f3_result_df
)


# ------------------------------------------------------------
# 14. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_60_f3_history
)


# ------------------------------------------------------------
# 15. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_60_f3_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_60_f3_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_60_f3_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_60_f3_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_60_f3_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_60_f3_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_60_f3_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_60_f3_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 16. IMPORTANT F2 -> F3 TEST comparison
#
# F2:
#
#       solar astronomy
#
# F3:
#
#       solar astronomy
#       +
#       continuous lunar / Sun-Moon astronomy
#
#
# Positive percentage:
#
#       F3 has LOWER TEST MAE than F2.
#
# Negative percentage:
#
#       F3 has HIGHER TEST MAE than F2.
#
#
# This is descriptive held-out TEST performance.
# ------------------------------------------------------------

bilstm_60_f2_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f3_mae = float(
    bilstm_final_60_f3_result[
        "test_MAE_kW"
    ]
)


bilstm_60_f2_to_f3_improvement_pct = (
    (
        bilstm_60_f2_mae
        -
        bilstm_60_f3_mae
    )
    /
    bilstm_60_f2_mae
    *
    100.0
)


print(
    "\n============================================================"
)

print(
    "60-MIN F2 -> F3 CONTINUOUS LUNISOLAR COMPARISON"
)

print(
    "============================================================"
)


print(
    f"F2 TEST MAE: {bilstm_60_f2_mae:.6f} kW"
)


print(
    f"F3 TEST MAE: {bilstm_60_f3_mae:.6f} kW"
)


print(
    (
        "F2 -> F3 relative MAE improvement: "
        f"{bilstm_60_f2_to_f3_improvement_pct:+.6f}%"
    )
)


print(
    "Positive = F3 lower MAE than F2."
)


# ------------------------------------------------------------
# 17. F0 -> F3 reference comparison
# ------------------------------------------------------------

bilstm_60_f0_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F0",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f0_to_f3_improvement_pct = (
    (
        bilstm_60_f0_mae
        -
        bilstm_60_f3_mae
    )
    /
    bilstm_60_f0_mae
    *
    100.0
)


print(
    "\n60-min F0 -> F3 TEST MAE change:"
)


print(
    f"  F0 MAE: {bilstm_60_f0_mae:.6f} kW"
)


print(
    f"  F3 MAE: {bilstm_60_f3_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_60_f0_to_f3_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 18. Build prediction table
# ------------------------------------------------------------

bilstm_final_60_f3_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_60_f3_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_60_f3_prediction_kw,
    }
)


assert (
    bilstm_final_60_f3_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 19. Persist experiment artifacts
# ------------------------------------------------------------

bilstm_60_f3_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F3_result.csv"
)


bilstm_60_f3_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F3_history.csv"
)


bilstm_60_f3_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F3_predictions.csv"
)


bilstm_final_60_f3_result_df.to_csv(
    bilstm_60_f3_result_path,
    index=False,
)


bilstm_final_60_f3_history.to_csv(
    bilstm_60_f3_history_path,
    index=False,
)


bilstm_final_60_f3_prediction_table.to_csv(
    bilstm_60_f3_predictions_path,
    index=False,
)


# ------------------------------------------------------------
# 20. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_60_f3_result_path,
    bilstm_60_f3_history_path,
    bilstm_60_f3_predictions_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 21. Current BiLSTM final TEST progress
# ------------------------------------------------------------

bilstm_final_test_progress = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCurrent BiLSTM final TEST progress:"
)


display(
    bilstm_final_test_progress[
        [
            "horizon_minutes",
            "feature_family",
            "frozen_best_epoch",
            "trainable_parameter_count",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 22. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM 60-MIN F3 FINAL TEST: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Frozen epoch:",
    bilstm_final_60_f3_result[
        "frozen_best_epoch"
    ],
)


print(
    "TEST MAE:",
    f"{bilstm_final_60_f3_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "TEST RMSE:",
    f"{bilstm_final_60_f3_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "TEST R2:",
    f"{bilstm_final_60_f3_result['test_R2']:.6f}",
)


print(
    "\nF2 -> F3 TEST MAE improvement:",
    f"{bilstm_60_f2_to_f3_improvement_pct:+.6f}%",
)


print(
    "\nPrediction clipping:",
    bilstm_final_60_f3_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nArtifacts written:"
)

print(
    " ",
    bilstm_60_f3_result_path,
)

print(
    " ",
    bilstm_60_f3_history_path,
)

print(
    " ",
    bilstm_60_f3_predictions_path,
)


print(
    "\nNext step after verification:"
)

print(
    "Run the FINAL BiLSTM experiment:"
)

print(
    "60-minute F4."
)

print(
    "That will complete all 15 BiLSTM final TEST fits."
)

Epoch 1/5


2026-09-22 13:01:43.095261: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 13:01:43.095610: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 86ms/step - loss: 0.2665
Epoch 2/5
410/410 - 31s - 76ms/step - loss: 0.2258
Epoch 3/5
410/410 - 31s - 76ms/step - loss: 0.2308
Epoch 4/5
410/410 - 31s - 76ms/step - loss: 0.2281
Epoch 5/5
410/410 - 31s - 77ms/step - loss: 0.2296


2026-09-22 13:04:23.352556: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 13:04:23.352843: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

60-min F0/F1/F2/F3 TEST target alignment: PASSED

BiLSTM FINAL HELD-OUT TEST — 60-MIN F3


...


Raw unclipped prediction audit:


,actual_min_kW,actual_max_kW,prediction_min_kW,prediction_max_kW,prediction_below_zero_count,prediction_above_10kW_count
0,0.0,6.505993,-0.355618,5.882925,4168,0



60-MIN F2 -> F3 CONTINUOUS LUNISOLAR COMPARISON
F2 TEST MAE: 0.507799 kW
F3 TEST MAE: 0.477851 kW
F2 -> F3 relative MAE improvement: +5.897705%
Positive = F3 lower MAE than F2.

60-min F0 -> F3 TEST MAE change:
  F0 MAE: 0.508764 kW
  F3 MAE: 0.477851 kW
  Relative improvement: +6.076113%

Current BiLSTM final TEST progress:


,horizon_minutes,feature_family,frozen_best_epoch,trainable_parameter_count,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,6,42081,0.271462,0.595756,0.869736
1,15,F1,5,42561,0.310092,0.623503,0.857320
2,15,F2,2,42721,0.285021,0.616572,0.860475
3,15,F3,4,43009,0.308417,0.612025,0.862525
4,15,F4,2,46113,0.303657,0.616196,0.860644
5,30,F0,16,42081,0.479948,0.789551,0.771205
6,30,F1,7,42561,0.390785,0.715113,0.812312
7,30,F2,3,42721,0.358851,0.707744,0.816160
8,30,F3,17,43009,0.429713,0.722947,0.808178
9,30,F4,3,46113,0.392481,0.734626,0.801930



BiLSTM 60-MIN F3 FINAL TEST: COMPLETE
Frozen epoch: 5
TEST MAE: 0.477851 kW
TEST RMSE: 0.846885 kW
TEST R2: 0.736770

F2 -> F3 TEST MAE improvement: +5.897705%

Prediction clipping: False

Completed BiLSTM final TEST experiments: 14 / 15

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F3_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F3_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F3_predictions.csv

Next step after verification:
Run the FINAL BiLSTM experiment:
60-minute F4.
That will complete all 15 BiLSTM final TEST fits.


In [119]:
# ============================================================
# Step 103: FINAL BiLSTM held-out TEST
#
# Experiment:
#
#       horizon        = 60 minutes
#       feature family = F4
#
#
# THIS IS THE FINAL BiLSTM MODEL FIT.
#
#
# SCIENTIFIC ROLE OF F4
# ---------------------
#
# F4 contains EVERYTHING already supplied to F3:
#
#       history / meteorology
#       Gregorian temporal features
#       solar astronomy
#       continuous lunar / Sun-Moon astronomy
#
# and additionally supplies the categorical Panchang layer:
#
#       Tithi
#       Paksha
#       Karana
#       Nakshatra
#       Yoga
#
#
# Therefore:
#
#       F3 -> F4
#
# is the CORE comparison for the proposed Panchang
# representation.
#
#
# FROZEN BEFORE TEST:
#
#       architecture = RNN_C2
#       units        = 64
#       layers       = 1
#       dropout      = 0.0
#       fusion Dense = 32
#
#       frozen epoch = 4
#
#
# FINAL REFIT:
#
#       TRAIN + VALIDATION
#       exactly 4 epochs
#       original TRAIN-only scalers
#       no callbacks
#       no validation_data
#
#
# TEST:
#
#       26,496 held-out rows
#
#
# Prediction clipping:
#
#       NONE
#
#
# After this cell:
#
#       15 / 15 BiLSTM final TEST experiments complete.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen experiment identity
# ------------------------------------------------------------

BILSTM_FINAL_60_F4_HORIZON = 60

BILSTM_FINAL_60_F4_FAMILY = "F4"

BILSTM_FINAL_60_F4_KEY = (
    BILSTM_FINAL_60_F4_HORIZON,
    BILSTM_FINAL_60_F4_FAMILY,
)


# ------------------------------------------------------------
# 2. Confirm current TEST progress
#
# Completed:
#
#       15-min: F0-F4
#       30-min: F0-F4
#       60-min: F0-F3
#
# Total = 14 experiments.
# ------------------------------------------------------------

assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    {
        (15, "F0"),
        (15, "F1"),
        (15, "F2"),
        (15, "F3"),
        (15, "F4"),

        (30, "F0"),
        (30, "F1"),
        (30, "F2"),
        (30, "F3"),
        (30, "F4"),

        (60, "F0"),
        (60, "F1"),
        (60, "F2"),
        (60, "F3"),
    }
)


assert (
    BILSTM_FINAL_60_F4_KEY
    not in
    bilstm_final_test_results
)


# ------------------------------------------------------------
# 3. Confirm frozen 60-min F4 protocol
# ------------------------------------------------------------

assert (
    BILSTM_FINAL_CANDIDATE_CONFIG
    ==
    {
        "candidate_id": "RNN_C2",
        "units": 64,
        "n_recurrent_layers": 1,
        "dropout": 0.0,
        "fusion_dense_units": 32,
    }
)


assert (
    BILSTM_FINAL_FROZEN_EPOCHS[
        BILSTM_FINAL_60_F4_KEY
    ]
    ==
    4
)


assert (
    BILSTM_FINAL_EXPECTED_AUXILIARY_DIMS[
        "F4"
    ]
    ==
    127
)


assert (
    BILSTM_FINAL_EXPECTED_PARAMETER_COUNTS[
        "F4"
    ]
    ==
    46_113
)


# ------------------------------------------------------------
# 4. Run FINAL TRAIN+VALIDATION refit and held-out TEST
# ------------------------------------------------------------

(
    bilstm_final_60_f4_result,
    bilstm_final_60_f4_history,
    bilstm_final_60_f4_actual_kw,
    bilstm_final_60_f4_prediction_kw,
) = run_bilstm_final_refit_and_test(
    horizon_minutes=
        BILSTM_FINAL_60_F4_HORIZON,

    feature_family=
        BILSTM_FINAL_60_F4_FAMILY,

    verbose=
        2,
)


# ------------------------------------------------------------
# 5. Identity checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f4_result[
        "model"
    ]
    ==
    "BiLSTM"
)


assert (
    bilstm_final_60_f4_result[
        "candidate_id"
    ]
    ==
    "RNN_C2"
)


assert (
    bilstm_final_60_f4_result[
        "horizon_minutes"
    ]
    ==
    60
)


assert (
    bilstm_final_60_f4_result[
        "feature_family"
    ]
    ==
    "F4"
)


# ------------------------------------------------------------
# 6. Architecture / population checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f4_result[
        "auxiliary_dim"
    ]
    ==
    127
)


assert (
    bilstm_final_60_f4_result[
        "trainable_parameter_count"
    ]
    ==
    46_113
)


assert (
    bilstm_final_60_f4_result[
        "n_train"
    ]
    ==
    87_252
)


assert (
    bilstm_final_60_f4_result[
        "n_validation"
    ]
    ==
    17_568
)


assert (
    bilstm_final_60_f4_result[
        "n_trainval"
    ]
    ==
    104_820
)


assert (
    bilstm_final_60_f4_result[
        "n_test"
    ]
    ==
    26_496
)


# ------------------------------------------------------------
# 7. Frozen epoch checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f4_result[
        "frozen_best_epoch"
    ]
    ==
    4
)


assert (
    bilstm_final_60_f4_result[
        "final_epochs_trained"
    ]
    ==
    4
)


assert (
    len(
        bilstm_final_60_f4_history
    )
    ==
    4
)


assert (
    "val_loss"
    not in
    bilstm_final_60_f4_history.columns
)


# ------------------------------------------------------------
# 8. Frozen-policy checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f4_result[
        "prediction_clipping"
    ]
    is False
)


assert (
    bilstm_final_60_f4_result[
        "scaler_policy"
    ]
    ==
    "original_train_only_scalers"
)


assert (
    bilstm_final_60_f4_result[
        "refit_population"
    ]
    ==
    "train_plus_validation"
)


# ------------------------------------------------------------
# 9. Metric sanity checks
# ------------------------------------------------------------

for metric_name in [
    "test_MAE_kW",
    "test_RMSE_kW",
    "test_R2",
    "final_fit_seconds",
]:

    assert np.isfinite(
        bilstm_final_60_f4_result[
            metric_name
        ]
    )


# ------------------------------------------------------------
# 10. Prediction-array checks
# ------------------------------------------------------------

assert (
    bilstm_final_60_f4_actual_kw.shape
    ==
    bilstm_final_60_f4_prediction_kw.shape
    ==
    (
        26_496,
    )
)


assert np.isfinite(
    bilstm_final_60_f4_actual_kw
).all()


assert np.isfinite(
    bilstm_final_60_f4_prediction_kw
).all()


# ------------------------------------------------------------
# 11. COMPLETE 60-minute target-alignment audit
#
# All five feature families must forecast exactly the same
# held-out 60-minute target vector.
# ------------------------------------------------------------

for previous_family in [
    "F0",
    "F1",
    "F2",
    "F3",
]:

    assert np.array_equal(
        bilstm_final_test_actuals[
            (
                60,
                previous_family,
            )
        ],
        bilstm_final_60_f4_actual_kw,
    )


print(
    "60-min F0/F1/F2/F3/F4 TEST target alignment: PASSED"
)


# ------------------------------------------------------------
# 12. Store successful FINAL experiment
# ------------------------------------------------------------

bilstm_final_test_results[
    BILSTM_FINAL_60_F4_KEY
] = (
    bilstm_final_60_f4_result.copy()
)


bilstm_final_test_histories[
    BILSTM_FINAL_60_F4_KEY
] = (
    bilstm_final_60_f4_history.copy()
)


bilstm_final_test_actuals[
    BILSTM_FINAL_60_F4_KEY
] = (
    bilstm_final_60_f4_actual_kw.copy()
)


bilstm_final_test_predictions[
    BILSTM_FINAL_60_F4_KEY
] = (
    bilstm_final_60_f4_prediction_kw.copy()
)


# ------------------------------------------------------------
# 13. Confirm ALL 15 BiLSTM final TEST experiments exist
# ------------------------------------------------------------

expected_bilstm_final_keys = {
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes in [
        15,
        30,
        60,
    ]

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]
}


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    expected_bilstm_final_keys
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    15
)


print(
    "All 15 BiLSTM final TEST experiments: PRESENT"
)


# ------------------------------------------------------------
# 14. Display final held-out TEST result
# ------------------------------------------------------------

bilstm_final_60_f4_result_df = pd.DataFrame(
    [
        bilstm_final_60_f4_result
    ]
)


print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL HELD-OUT TEST — 60-MIN F4"
)

print(
    "============================================================"
)


display(
    bilstm_final_60_f4_result_df
)


# ------------------------------------------------------------
# 15. Display final-refit history
# ------------------------------------------------------------

print(
    "\nFinal TRAIN+VALIDATION refit history:"
)


display(
    bilstm_final_60_f4_history
)


# ------------------------------------------------------------
# 16. Raw UNCLIPPED prediction audit
# ------------------------------------------------------------

bilstm_final_60_f4_prediction_audit = {
    "actual_min_kW":
        float(
            np.min(
                bilstm_final_60_f4_actual_kw
            )
        ),

    "actual_max_kW":
        float(
            np.max(
                bilstm_final_60_f4_actual_kw
            )
        ),

    "prediction_min_kW":
        float(
            np.min(
                bilstm_final_60_f4_prediction_kw
            )
        ),

    "prediction_max_kW":
        float(
            np.max(
                bilstm_final_60_f4_prediction_kw
            )
        ),

    "prediction_below_zero_count":
        int(
            np.sum(
                bilstm_final_60_f4_prediction_kw
                <
                0.0
            )
        ),

    "prediction_above_10kW_count":
        int(
            np.sum(
                bilstm_final_60_f4_prediction_kw
                >
                10.0
            )
        ),
}


print(
    "\nRaw unclipped prediction audit:"
)


display(
    pd.DataFrame(
        [
            bilstm_final_60_f4_prediction_audit
        ]
    )
)


# ------------------------------------------------------------
# 17. CORE 60-minute F3 -> F4 TEST comparison
#
# Positive percentage:
#
#       F4 has LOWER MAE than F3.
#
# Negative percentage:
#
#       F4 has HIGHER MAE than F3.
#
#
# This is the final BiLSTM Panchang increment.
# ------------------------------------------------------------

bilstm_60_f3_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F3",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f4_mae = float(
    bilstm_final_60_f4_result[
        "test_MAE_kW"
    ]
)


bilstm_60_f3_to_f4_improvement_pct = (
    (
        bilstm_60_f3_mae
        -
        bilstm_60_f4_mae
    )
    /
    bilstm_60_f3_mae
    *
    100.0
)


print(
    "\n============================================================"
)

print(
    "CORE 60-MIN F3 -> F4 PANCHANG COMPARISON"
)

print(
    "============================================================"
)


print(
    f"F3 TEST MAE: {bilstm_60_f3_mae:.6f} kW"
)


print(
    f"F4 TEST MAE: {bilstm_60_f4_mae:.6f} kW"
)


print(
    (
        "F3 -> F4 relative MAE improvement: "
        f"{bilstm_60_f3_to_f4_improvement_pct:+.6f}%"
    )
)


print(
    "Positive = F4 lower MAE than F3."
)


# ------------------------------------------------------------
# 18. F2 -> F4 descriptive comparison
#
# NOT the core novelty comparison because F2 -> F4 introduces
# both:
#
#       continuous lunar astronomy
#       Panchang categorical representation
# ------------------------------------------------------------

bilstm_60_f2_mae = float(
    bilstm_final_test_results[
        (
            60,
            "F2",
        )
    ][
        "test_MAE_kW"
    ]
)


bilstm_60_f2_to_f4_improvement_pct = (
    (
        bilstm_60_f2_mae
        -
        bilstm_60_f4_mae
    )
    /
    bilstm_60_f2_mae
    *
    100.0
)


print(
    "\n60-min F2 -> F4 TEST MAE change:"
)


print(
    f"  F2 MAE: {bilstm_60_f2_mae:.6f} kW"
)


print(
    f"  F4 MAE: {bilstm_60_f4_mae:.6f} kW"
)


print(
    (
        "  Relative improvement: "
        f"{bilstm_60_f2_to_f4_improvement_pct:+.6f}%"
    )
)


# ------------------------------------------------------------
# 19. Complete 60-minute BiLSTM ablation table
# ------------------------------------------------------------

bilstm_60min_ablation_table = (
    pd.DataFrame(
        [
            bilstm_final_test_results[
                (
                    60,
                    feature_family,
                )
            ]

            for feature_family in [
                "F0",
                "F1",
                "F2",
                "F3",
                "F4",
            ]
        ]
    )
    [
        [
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "frozen_best_epoch",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


print(
    "\nComplete BiLSTM 60-minute ablation:"
)


display(
    bilstm_60min_ablation_table
)


# ------------------------------------------------------------
# 20. COMPLETE BiLSTM final TEST table
#
# All 3 horizons x all 5 feature families.
# ------------------------------------------------------------

bilstm_complete_final_test_table = (
    pd.DataFrame(
        list(
            bilstm_final_test_results.values()
        )
    )
    .sort_values(
        [
            "horizon_minutes",
            "feature_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    bilstm_complete_final_test_table.shape[0]
    ==
    15
)


print(
    "\n============================================================"
)

print(
    "COMPLETE BiLSTM FINAL HELD-OUT TEST TABLE"
)

print(
    "============================================================"
)


display(
    bilstm_complete_final_test_table[
        [
            "horizon_minutes",
            "feature_family",
            "auxiliary_dim",
            "trainable_parameter_count",
            "frozen_best_epoch",
            "test_MAE_kW",
            "test_RMSE_kW",
            "test_R2",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Build complete BiLSTM F3 -> F4 summary
#
# Recompute directly from the canonical result store rather
# than relying on previously calculated temporary variables.
# ------------------------------------------------------------

bilstm_f3_f4_test_summary_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    f3_mae = float(
        bilstm_final_test_results[
            (
                horizon_minutes,
                "F3",
            )
        ][
            "test_MAE_kW"
        ]
    )


    f4_mae = float(
        bilstm_final_test_results[
            (
                horizon_minutes,
                "F4",
            )
        ][
            "test_MAE_kW"
        ]
    )


    absolute_mae_difference_kw = (
        f3_mae
        -
        f4_mae
    )


    relative_improvement_pct = (
        absolute_mae_difference_kw
        /
        f3_mae
        *
        100.0
    )


    bilstm_f3_f4_test_summary_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "F3_MAE_kW":
                f3_mae,

            "F4_MAE_kW":
                f4_mae,

            "F3_minus_F4_MAE_kW":
                absolute_mae_difference_kw,

            "F3_to_F4_MAE_improvement_pct":
                relative_improvement_pct,

            "F4_lower_MAE_than_F3":
                bool(
                    f4_mae
                    <
                    f3_mae
                ),
        }
    )


bilstm_f3_f4_test_summary = pd.DataFrame(
    bilstm_f3_f4_test_summary_rows
)


print(
    "\n============================================================"
)

print(
    "COMPLETE BiLSTM F3 -> F4 HELD-OUT TEST SUMMARY"
)

print(
    "============================================================"
)


print(
    "Positive improvement = F4 lower MAE than F3."
)


display(
    bilstm_f3_f4_test_summary
)


# ------------------------------------------------------------
# 22. Build final F4 prediction table
# ------------------------------------------------------------

bilstm_final_60_f4_prediction_table = pd.DataFrame(
    {
        "test_row_position":
            np.arange(
                26_496,
                dtype=int,
            ),

        "actual_P_Solar_kW":
            bilstm_final_60_f4_actual_kw,

        "predicted_P_Solar_kW":
            bilstm_final_60_f4_prediction_kw,
    }
)


assert (
    bilstm_final_60_f4_prediction_table.shape
    ==
    (
        26_496,
        3,
    )
)


# ------------------------------------------------------------
# 23. Persist FINAL BiLSTM artifacts
# ------------------------------------------------------------

bilstm_60_f4_result_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F4_result.csv"
)


bilstm_60_f4_history_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F4_history.csv"
)


bilstm_60_f4_predictions_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_F4_predictions.csv"
)


bilstm_60min_ablation_path = (
    RESULTS_DIR
    /
    "bilstm_final_60min_ablation.csv"
)


bilstm_complete_final_test_path = (
    RESULTS_DIR
    /
    "bilstm_complete_final_test_results.csv"
)


bilstm_f3_f4_test_summary_path = (
    RESULTS_DIR
    /
    "bilstm_f3_f4_test_summary.csv"
)


bilstm_final_60_f4_result_df.to_csv(
    bilstm_60_f4_result_path,
    index=False,
)


bilstm_final_60_f4_history.to_csv(
    bilstm_60_f4_history_path,
    index=False,
)


bilstm_final_60_f4_prediction_table.to_csv(
    bilstm_60_f4_predictions_path,
    index=False,
)


bilstm_60min_ablation_table.to_csv(
    bilstm_60min_ablation_path,
    index=False,
)


bilstm_complete_final_test_table.to_csv(
    bilstm_complete_final_test_path,
    index=False,
)


bilstm_f3_f4_test_summary.to_csv(
    bilstm_f3_f4_test_summary_path,
    index=False,
)


# ------------------------------------------------------------
# 24. Verify all newly persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_60_f4_result_path,
    bilstm_60_f4_history_path,
    bilstm_60_f4_predictions_path,
    bilstm_60min_ablation_path,
    bilstm_complete_final_test_path,
    bilstm_f3_f4_test_summary_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 25. FINAL BiLSTM checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM FINAL TEST BLOCK: COMPLETE"
)

print(
    "============================================================"
)


print(
    "60-min F4 frozen epoch:",
    bilstm_final_60_f4_result[
        "frozen_best_epoch"
    ],
)


print(
    "60-min F4 TEST MAE:",
    f"{bilstm_final_60_f4_result['test_MAE_kW']:.6f}",
    "kW",
)


print(
    "60-min F4 TEST RMSE:",
    f"{bilstm_final_60_f4_result['test_RMSE_kW']:.6f}",
    "kW",
)


print(
    "60-min F4 TEST R2:",
    f"{bilstm_final_60_f4_result['test_R2']:.6f}",
)


print(
    "\nCORE 60-min F3 -> F4 MAE improvement:",
    f"{bilstm_60_f3_to_f4_improvement_pct:+.6f}%",
)


print(
    "\nPrediction clipping:",
    bilstm_final_60_f4_result[
        "prediction_clipping"
    ],
)


print(
    "\nCompleted BiLSTM final TEST experiments:",
    len(
        bilstm_final_test_results
    ),
    "/ 15",
)


print(
    "\nALL BiLSTM model fitting is now complete."
)


print(
    "\nNo additional BiLSTM fitting should be performed"
)

print(
    "during the remaining evaluation / inference analysis."
)


print(
    "\nArtifacts written:"
)

for output_path in [
    bilstm_60_f4_result_path,
    bilstm_60_f4_history_path,
    bilstm_60_f4_predictions_path,
    bilstm_60min_ablation_path,
    bilstm_complete_final_test_path,
    bilstm_f3_f4_test_summary_path,
]:

    print(
        " ",
        output_path,
    )


print(
    "\nNext step after verification:"
)

print(
    "BiLSTM daylight F3/F4 evaluation using the canonical"
)

print(
    "forecast-valid solar-elevation > 0 degree mask."
)

Epoch 1/4


2026-09-22 13:06:21.097538: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-22 13:06:21.097874: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

410/410 - 35s - 85ms/step - loss: 0.2624
Epoch 2/4
410/410 - 31s - 76ms/step - loss: 0.2235
Epoch 3/4
410/410 - 31s - 76ms/step - loss: 0.2276
Epoch 4/4
410/410 - 32s - 78ms/step - loss: 0.2382


2026-09-22 13:08:30.673101: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-22 13:08:30.673399: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

60-min F0/F1/F2/F3/F4 TEST target alignment: PASSED
All 15 BiLSTM final TEST experiments: PRESENT

BiLSTM FINAL HELD-OUT TEST — 60-MIN F4


...


COMPLETE BiLSTM FINAL HELD-OUT TEST TABLE


,horizon_minutes,feature_family,auxiliary_dim,trainable_parameter_count,frozen_best_epoch,test_MAE_kW,test_RMSE_kW,test_R2
0,15,F0,1,42081,6,0.271462,0.595756,0.869736
1,15,F1,16,42561,5,0.310092,0.623503,0.857320
2,15,F2,21,42721,2,0.285021,0.616572,0.860475
3,15,F3,30,43009,4,0.308417,0.612025,0.862525
4,15,F4,127,46113,2,0.303657,0.616196,0.860644
5,30,F0,1,42081,16,0.479948,0.789551,0.771205
6,30,F1,16,42561,7,0.390785,0.715113,0.812312
7,30,F2,21,42721,3,0.358851,0.707744,0.816160
8,30,F3,30,43009,17,0.429713,0.722947,0.808178
9,30,F4,127,46113,3,0.392481,0.734626,0.801930



COMPLETE BiLSTM F3 -> F4 HELD-OUT TEST SUMMARY
Positive improvement = F4 lower MAE than F3.


,model,horizon_minutes,F3_MAE_kW,F4_MAE_kW,F3_minus_F4_MAE_kW,F3_to_F4_MAE_improvement_pct,F4_lower_MAE_than_F3
0,BiLSTM,15,0.308417,0.303657,0.004760,1.543316,True
1,BiLSTM,30,0.429713,0.392481,0.037232,8.664420,True
2,BiLSTM,60,0.477851,0.520756,-0.042905,-8.978698,False



BiLSTM FINAL TEST BLOCK: COMPLETE
60-min F4 frozen epoch: 4
60-min F4 TEST MAE: 0.520756 kW
60-min F4 TEST RMSE: 0.862331 kW
60-min F4 TEST R2: 0.727080

CORE 60-min F3 -> F4 MAE improvement: -8.978698%

Prediction clipping: False

Completed BiLSTM final TEST experiments: 15 / 15

ALL BiLSTM model fitting is now complete.

No additional BiLSTM fitting should be performed
during the remaining evaluation / inference analysis.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F4_result.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F4_history.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_final_60min_F4_predictions.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTI

In [120]:
# ============================================================
# Step 104: BiLSTM daylight-only F3/F4 evaluation
#
# IMPORTANT:
#
#       NO MODEL FITTING OCCURS IN THIS CELL.
#
# All 15 BiLSTM final TRAIN+VALIDATION refits and TEST
# predictions are already frozen and complete.
#
#
# PURPOSE
# -------
#
# Evaluate the CORE comparison:
#
#       F3 -> F4
#
# specifically during daylight.
#
#
# CANONICAL DAYLIGHT DEFINITION
# -----------------------------
#
#       forecast-valid solar elevation > 0 degrees
#
#
# This is the SAME daylight definition previously used for:
#
#       LSTM
#       GRU
#
#
# Expected TEST population:
#
#       total rows       = 26,496
#       daylight rows    = 17,851
#       non-daylight     =  8,645
#
#
# The cell intentionally RECOVERS the previously established
# canonical daylight mask from the current notebook state
# rather than constructing a new competing definition.
#
#
# Metrics:
#
#       MAE
#       RMSE
#       R^2
#
#
# Predictions:
#
#       raw
#       unclipped
#
# ============================================================


# ------------------------------------------------------------
# 1. Confirm that all BiLSTM model fitting is finished
# ------------------------------------------------------------

expected_bilstm_final_keys = {
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes in [
        15,
        30,
        60,
    ]

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]
}


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    expected_bilstm_final_keys
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    15
)


assert (
    set(
        bilstm_final_test_predictions.keys()
    )
    ==
    expected_bilstm_final_keys
)


assert (
    set(
        bilstm_final_test_actuals.keys()
    )
    ==
    expected_bilstm_final_keys
)


print(
    "BiLSTM final fitting state: 15 / 15 COMPLETE"
)

print(
    "No model fitting will be performed in this cell."
)


# ------------------------------------------------------------
# 2. Recover the EXISTING canonical daylight mask
#
# Earlier LSTM / GRU evaluation established:
#
#       n_test      = 26,496
#       n_daylight  = 17,851
#
# We search the CURRENT notebook namespace for one-dimensional
# boolean arrays having exactly that population.
#
# Multiple variable names are acceptable IF they all contain
# exactly the same boolean pattern.
#
# This prevents us from silently creating a different daylight
# definition for BiLSTM.
# ------------------------------------------------------------

EXPECTED_TEST_ROWS = 26_496

EXPECTED_DAYLIGHT_ROWS = 17_851

EXPECTED_NON_DAYLIGHT_ROWS = (
    EXPECTED_TEST_ROWS
    -
    EXPECTED_DAYLIGHT_ROWS
)


canonical_daylight_mask_candidates = {}


for variable_name, variable_value in list(
    globals().items()
):

    try:

        candidate_array = np.asarray(
            variable_value
        )

    except Exception:

        continue


    if (
        candidate_array.ndim
        !=
        1
    ):

        continue


    if (
        candidate_array.shape[0]
        !=
        EXPECTED_TEST_ROWS
    ):

        continue


    if (
        candidate_array.dtype
        !=
        np.bool_
    ):

        continue


    if (
        int(
            candidate_array.sum()
        )
        !=
        EXPECTED_DAYLIGHT_ROWS
    ):

        continue


    canonical_daylight_mask_candidates[
        variable_name
    ] = candidate_array.copy()


assert (
    len(
        canonical_daylight_mask_candidates
    )
    >=
    1
), (
    "Could not recover the previously established canonical "
    "daylight mask from the notebook namespace."
)


print(
    "\nCandidate canonical daylight-mask variables found:"
)


for candidate_name in (
    canonical_daylight_mask_candidates.keys()
):

    print(
        " ",
        candidate_name,
    )


# ------------------------------------------------------------
# 3. Verify that all recovered candidates represent the SAME
#    daylight rows
#
# If several earlier cells retained different variable names
# for the same mask, that is fine.
#
# If they differ in actual row membership, stop immediately.
# ------------------------------------------------------------

candidate_daylight_masks = list(
    canonical_daylight_mask_candidates.values()
)


reference_daylight_mask = (
    candidate_daylight_masks[0]
)


for candidate_mask in (
    candidate_daylight_masks[1:]
):

    assert np.array_equal(
        candidate_mask,
        reference_daylight_mask,
    ), (
        "Multiple 17,851-row boolean masks were found but "
        "they are not identical. Daylight mask is ambiguous."
    )


bilstm_canonical_daylight_mask = (
    reference_daylight_mask.copy()
)


assert (
    bilstm_canonical_daylight_mask.dtype
    ==
    np.bool_
)


assert (
    bilstm_canonical_daylight_mask.shape
    ==
    (
        EXPECTED_TEST_ROWS,
    )
)


assert (
    int(
        bilstm_canonical_daylight_mask.sum()
    )
    ==
    EXPECTED_DAYLIGHT_ROWS
)


assert (
    int(
        (
            ~bilstm_canonical_daylight_mask
        ).sum()
    )
    ==
    EXPECTED_NON_DAYLIGHT_ROWS
)


print(
    "\nCanonical daylight mask audit: PASSED"
)

print(
    "  TEST rows:",
    EXPECTED_TEST_ROWS,
)

print(
    "  Daylight rows:",
    int(
        bilstm_canonical_daylight_mask.sum()
    ),
)

print(
    "  Non-daylight rows:",
    int(
        (
            ~bilstm_canonical_daylight_mask
        ).sum()
    ),
)


# ------------------------------------------------------------
# 4. Optional semantic audit
#
# If numeric solar-elevation vectors are still resident in the
# notebook, check whether solar_elevation > 0 reproduces the
# recovered canonical mask exactly.
#
# We do NOT require such a vector to remain in memory because
# the canonical boolean mask has already been established by
# the earlier evaluation cells.
# ------------------------------------------------------------

matching_solar_elevation_sources = []


for variable_name, variable_value in list(
    globals().items()
):

    lower_name = str(
        variable_name
    ).lower()


    if not (
        (
            "solar"
            in
            lower_name
        )
        and
        (
            "elev"
            in
            lower_name
        )
    ):

        continue


    try:

        candidate_array = np.asarray(
            variable_value,
            dtype=float,
        ).reshape(
            -1
        )

    except Exception:

        continue


    if (
        candidate_array.shape[0]
        !=
        EXPECTED_TEST_ROWS
    ):

        continue


    if not np.isfinite(
        candidate_array
    ).all():

        continue


    candidate_mask = (
        candidate_array
        >
        0.0
    )


    if np.array_equal(
        candidate_mask,
        bilstm_canonical_daylight_mask,
    ):

        matching_solar_elevation_sources.append(
            variable_name
        )


if (
    len(
        matching_solar_elevation_sources
    )
    >
    0
):

    print(
        "\nSolar-elevation > 0 semantic audit: PASSED"
    )

    print(
        "Matching solar-elevation source(s):"
    )

    for source_name in (
        matching_solar_elevation_sources
    ):

        print(
            " ",
            source_name,
        )

else:

    print(
        "\nNo retained numeric solar-elevation vector was"
    )

    print(
        "needed for this cell; the previously established"
    )

    print(
        "canonical daylight mask is being reused directly."
    )


# ------------------------------------------------------------
# 5. Local metric helper
#
# Pure NumPy implementation keeps this evaluation independent
# of any fitting/model code.
# ------------------------------------------------------------

def compute_bilstm_evaluation_metrics(
    actual_kw,
    prediction_kw,
):

    actual_kw = np.asarray(
        actual_kw,
        dtype=float,
    ).reshape(
        -1
    )


    prediction_kw = np.asarray(
        prediction_kw,
        dtype=float,
    ).reshape(
        -1
    )


    assert (
        actual_kw.shape
        ==
        prediction_kw.shape
    )


    assert (
        actual_kw.size
        >
        0
    )


    assert np.isfinite(
        actual_kw
    ).all()


    assert np.isfinite(
        prediction_kw
    ).all()


    residual_kw = (
        actual_kw
        -
        prediction_kw
    )


    mae_kw = float(
        np.mean(
            np.abs(
                residual_kw
            )
        )
    )


    rmse_kw = float(
        np.sqrt(
            np.mean(
                residual_kw ** 2
            )
        )
    )


    ss_res = float(
        np.sum(
            residual_kw ** 2
        )
    )


    centered_actual = (
        actual_kw
        -
        np.mean(
            actual_kw
        )
    )


    ss_tot = float(
        np.sum(
            centered_actual ** 2
        )
    )


    assert (
        ss_tot
        >
        0.0
    )


    r2 = float(
        1.0
        -
        (
            ss_res
            /
            ss_tot
        )
    )


    return {
        "MAE_kW":
            mae_kw,

        "RMSE_kW":
            rmse_kw,

        "R2":
            r2,
    }


# ------------------------------------------------------------
# 6. Evaluate F3 and F4 on DAYLIGHT rows
#
# No re-prediction.
#
# We use the already saved raw held-out TEST predictions.
# ------------------------------------------------------------

bilstm_daylight_metrics_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # Same-horizon F3/F4 target vectors must match exactly.
    # --------------------------------------------------------

    f3_actual_kw = np.asarray(
        bilstm_final_test_actuals[
            (
                horizon_minutes,
                "F3",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    f4_actual_kw = np.asarray(
        bilstm_final_test_actuals[
            (
                horizon_minutes,
                "F4",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    assert np.array_equal(
        f3_actual_kw,
        f4_actual_kw,
    )


    assert (
        f3_actual_kw.shape
        ==
        (
            EXPECTED_TEST_ROWS,
        )
    )


    f3_prediction_kw = np.asarray(
        bilstm_final_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    f4_prediction_kw = np.asarray(
        bilstm_final_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    assert (
        f3_prediction_kw.shape
        ==
        f4_prediction_kw.shape
        ==
        (
            EXPECTED_TEST_ROWS,
        )
    )


    # --------------------------------------------------------
    # Apply the canonical forecast-valid daylight mask.
    # --------------------------------------------------------

    daylight_actual_kw = (
        f3_actual_kw[
            bilstm_canonical_daylight_mask
        ]
    )


    daylight_f3_prediction_kw = (
        f3_prediction_kw[
            bilstm_canonical_daylight_mask
        ]
    )


    daylight_f4_prediction_kw = (
        f4_prediction_kw[
            bilstm_canonical_daylight_mask
        ]
    )


    assert (
        daylight_actual_kw.shape
        ==
        daylight_f3_prediction_kw.shape
        ==
        daylight_f4_prediction_kw.shape
        ==
        (
            EXPECTED_DAYLIGHT_ROWS,
        )
    )


    # --------------------------------------------------------
    # Daylight metrics: F3
    # --------------------------------------------------------

    f3_daylight_metrics = (
        compute_bilstm_evaluation_metrics(
            daylight_actual_kw,
            daylight_f3_prediction_kw,
        )
    )


    # --------------------------------------------------------
    # Daylight metrics: F4
    # --------------------------------------------------------

    f4_daylight_metrics = (
        compute_bilstm_evaluation_metrics(
            daylight_actual_kw,
            daylight_f4_prediction_kw,
        )
    )


    bilstm_daylight_metrics_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "scope":
                "daylight",

            "feature_family":
                "F3",

            "daylight_definition":
                "forecast_valid_solar_elevation_gt_0_deg",

            "n_test_total":
                EXPECTED_TEST_ROWS,

            "n_daylight":
                EXPECTED_DAYLIGHT_ROWS,

            "n_non_daylight":
                EXPECTED_NON_DAYLIGHT_ROWS,

            "MAE_kW":
                f3_daylight_metrics[
                    "MAE_kW"
                ],

            "RMSE_kW":
                f3_daylight_metrics[
                    "RMSE_kW"
                ],

            "R2":
                f3_daylight_metrics[
                    "R2"
                ],

            "prediction_clipping":
                False,
        }
    )


    bilstm_daylight_metrics_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "scope":
                "daylight",

            "feature_family":
                "F4",

            "daylight_definition":
                "forecast_valid_solar_elevation_gt_0_deg",

            "n_test_total":
                EXPECTED_TEST_ROWS,

            "n_daylight":
                EXPECTED_DAYLIGHT_ROWS,

            "n_non_daylight":
                EXPECTED_NON_DAYLIGHT_ROWS,

            "MAE_kW":
                f4_daylight_metrics[
                    "MAE_kW"
                ],

            "RMSE_kW":
                f4_daylight_metrics[
                    "RMSE_kW"
                ],

            "R2":
                f4_daylight_metrics[
                    "R2"
                ],

            "prediction_clipping":
                False,
        }
    )


bilstm_daylight_metrics_f3_f4 = pd.DataFrame(
    bilstm_daylight_metrics_rows
)


assert (
    bilstm_daylight_metrics_f3_f4.shape[0]
    ==
    6
)


# ------------------------------------------------------------
# 7. Display raw daylight F3/F4 metrics
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM DAYLIGHT F3/F4 METRICS"
)

print(
    "============================================================"
)


display(
    bilstm_daylight_metrics_f3_f4
)


# ------------------------------------------------------------
# 8. Build one-row-per-horizon daylight F3 -> F4 summary
# ------------------------------------------------------------

bilstm_daylight_f3_f4_summary_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    horizon_metrics = (
        bilstm_daylight_metrics_f3_f4[
            bilstm_daylight_metrics_f3_f4[
                "horizon_minutes"
            ]
            ==
            horizon_minutes
        ]
    )


    f3_row = (
        horizon_metrics[
            horizon_metrics[
                "feature_family"
            ]
            ==
            "F3"
        ]
        .iloc[
            0
        ]
    )


    f4_row = (
        horizon_metrics[
            horizon_metrics[
                "feature_family"
            ]
            ==
            "F4"
        ]
        .iloc[
            0
        ]
    )


    f3_mae = float(
        f3_row[
            "MAE_kW"
        ]
    )


    f4_mae = float(
        f4_row[
            "MAE_kW"
        ]
    )


    f3_minus_f4_mae = (
        f3_mae
        -
        f4_mae
    )


    relative_improvement_pct = (
        f3_minus_f4_mae
        /
        f3_mae
        *
        100.0
    )


    bilstm_daylight_f3_f4_summary_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "scope":
                "daylight",

            "n_daylight":
                EXPECTED_DAYLIGHT_ROWS,

            "F3_MAE_kW":
                f3_mae,

            "F4_MAE_kW":
                f4_mae,

            "F3_minus_F4_MAE_kW":
                f3_minus_f4_mae,

            "F3_to_F4_MAE_improvement_pct":
                relative_improvement_pct,

            "F4_lower_MAE_than_F3":
                bool(
                    f4_mae
                    <
                    f3_mae
                ),
        }
    )


bilstm_daylight_f3_f4_summary = pd.DataFrame(
    bilstm_daylight_f3_f4_summary_rows
)


print(
    "\n============================================================"
)

print(
    "BiLSTM DAYLIGHT F3 -> F4 SUMMARY"
)

print(
    "============================================================"
)


print(
    "Positive improvement = F4 lower daylight MAE than F3."
)


display(
    bilstm_daylight_f3_f4_summary
)


# ------------------------------------------------------------
# 9. Build ALL-ROWS versus DAYLIGHT comparison
#
# This lets us see whether the F3 -> F4 effect changes when
# nighttime / non-daylight rows are excluded.
# ------------------------------------------------------------

bilstm_all_vs_daylight_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # All-rows MAEs from canonical final TEST result store.
    # --------------------------------------------------------

    all_f3_mae = float(
        bilstm_final_test_results[
            (
                horizon_minutes,
                "F3",
            )
        ][
            "test_MAE_kW"
        ]
    )


    all_f4_mae = float(
        bilstm_final_test_results[
            (
                horizon_minutes,
                "F4",
            )
        ][
            "test_MAE_kW"
        ]
    )


    all_improvement_pct = (
        (
            all_f3_mae
            -
            all_f4_mae
        )
        /
        all_f3_mae
        *
        100.0
    )


    # --------------------------------------------------------
    # Daylight MAEs from daylight summary.
    # --------------------------------------------------------

    daylight_row = (
        bilstm_daylight_f3_f4_summary[
            bilstm_daylight_f3_f4_summary[
                "horizon_minutes"
            ]
            ==
            horizon_minutes
        ]
        .iloc[
            0
        ]
    )


    daylight_f3_mae = float(
        daylight_row[
            "F3_MAE_kW"
        ]
    )


    daylight_f4_mae = float(
        daylight_row[
            "F4_MAE_kW"
        ]
    )


    daylight_improvement_pct = float(
        daylight_row[
            "F3_to_F4_MAE_improvement_pct"
        ]
    )


    # --------------------------------------------------------
    # Two rows per horizon:
    #
    #       all
    #       daylight
    # --------------------------------------------------------

    bilstm_all_vs_daylight_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "scope":
                "all",

            "n_rows":
                EXPECTED_TEST_ROWS,

            "F3_MAE_kW":
                all_f3_mae,

            "F4_MAE_kW":
                all_f4_mae,

            "F3_to_F4_MAE_improvement_pct":
                all_improvement_pct,

            "F4_lower_MAE_than_F3":
                bool(
                    all_f4_mae
                    <
                    all_f3_mae
                ),
        }
    )


    bilstm_all_vs_daylight_rows.append(
        {
            "model":
                "BiLSTM",

            "horizon_minutes":
                horizon_minutes,

            "scope":
                "daylight",

            "n_rows":
                EXPECTED_DAYLIGHT_ROWS,

            "F3_MAE_kW":
                daylight_f3_mae,

            "F4_MAE_kW":
                daylight_f4_mae,

            "F3_to_F4_MAE_improvement_pct":
                daylight_improvement_pct,

            "F4_lower_MAE_than_F3":
                bool(
                    daylight_f4_mae
                    <
                    daylight_f3_mae
                ),
        }
    )


bilstm_all_vs_daylight_f3_f4 = pd.DataFrame(
    bilstm_all_vs_daylight_rows
)


assert (
    bilstm_all_vs_daylight_f3_f4.shape[0]
    ==
    6
)


print(
    "\n============================================================"
)

print(
    "BiLSTM ALL-ROWS vs DAYLIGHT F3 -> F4"
)

print(
    "============================================================"
)


display(
    bilstm_all_vs_daylight_f3_f4
)


# ------------------------------------------------------------
# 10. Compact direction audit
#
# No statistical inference yet.
#
# The paired daily-block bootstrap comes NEXT.
# ------------------------------------------------------------

print(
    "\nF3 -> F4 point-estimate direction audit:"
)


for horizon_minutes in [
    15,
    30,
    60,
]:

    all_row = (
        bilstm_all_vs_daylight_f3_f4[
            (
                bilstm_all_vs_daylight_f3_f4[
                    "horizon_minutes"
                ]
                ==
                horizon_minutes
            )
            &
            (
                bilstm_all_vs_daylight_f3_f4[
                    "scope"
                ]
                ==
                "all"
            )
        ]
        .iloc[
            0
        ]
    )


    daylight_row = (
        bilstm_all_vs_daylight_f3_f4[
            (
                bilstm_all_vs_daylight_f3_f4[
                    "horizon_minutes"
                ]
                ==
                horizon_minutes
            )
            &
            (
                bilstm_all_vs_daylight_f3_f4[
                    "scope"
                ]
                ==
                "daylight"
            )
        ]
        .iloc[
            0
        ]
    )


    print(
        (
            f"  {horizon_minutes:>2}-min | "
            f"all: "
            f"{all_row['F3_to_F4_MAE_improvement_pct']:+.6f}%"
            f" | daylight: "
            f"{daylight_row['F3_to_F4_MAE_improvement_pct']:+.6f}%"
        )
    )


# ------------------------------------------------------------
# 11. Persist daylight evaluation artifacts
# ------------------------------------------------------------

bilstm_daylight_metrics_path = (
    RESULTS_DIR
    /
    "bilstm_daylight_metrics_F3_F4.csv"
)


bilstm_daylight_summary_path = (
    RESULTS_DIR
    /
    "bilstm_daylight_f3_f4_summary.csv"
)


bilstm_all_vs_daylight_path = (
    RESULTS_DIR
    /
    "bilstm_all_vs_daylight_f3_f4.csv"
)


bilstm_daylight_metrics_f3_f4.to_csv(
    bilstm_daylight_metrics_path,
    index=False,
)


bilstm_daylight_f3_f4_summary.to_csv(
    bilstm_daylight_summary_path,
    index=False,
)


bilstm_all_vs_daylight_f3_f4.to_csv(
    bilstm_all_vs_daylight_path,
    index=False,
)


# ------------------------------------------------------------
# 12. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_daylight_metrics_path,
    bilstm_daylight_summary_path,
    bilstm_all_vs_daylight_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 13. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM DAYLIGHT F3/F4 EVALUATION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Daylight definition:"
)

print(
    "  forecast-valid solar elevation > 0 degrees"
)


print(
    "\nPopulation:"
)

print(
    "  TEST total:",
    EXPECTED_TEST_ROWS,
)

print(
    "  Daylight:",
    EXPECTED_DAYLIGHT_ROWS,
)

print(
    "  Non-daylight:",
    EXPECTED_NON_DAYLIGHT_ROWS,
)


print(
    "\nNo model fitting performed."
)

print(
    "No prediction clipping performed."
)


print(
    "\nArtifacts written:"
)


for output_path in [
    bilstm_daylight_metrics_path,
    bilstm_daylight_summary_path,
    bilstm_all_vs_daylight_path,
]:

    print(
        " ",
        output_path,
    )


print(
    "\nNext step after verification:"
)

print(
    "Run the paired 10,000-replicate UTC daily-block bootstrap"
)

print(
    "for BiLSTM F3 vs F4 on both:"
)

print(
    "  1. all TEST rows"
)

print(
    "  2. daylight TEST rows"
)

BiLSTM final fitting state: 15 / 15 COMPLETE
No model fitting will be performed in this cell.

Candidate canonical daylight-mask variables found:
  population_mask
  daylight_mask

Canonical daylight mask audit: PASSED
  TEST rows: 26496
  Daylight rows: 17851
  Non-daylight rows: 8645

No retained numeric solar-elevation vector was
needed for this cell; the previously established
canonical daylight mask is being reused directly.

BiLSTM DAYLIGHT F3/F4 METRICS


,model,horizon_minutes,scope,feature_family,daylight_definition,n_test_total,n_daylight,n_non_daylight,MAE_kW,RMSE_kW,R2,prediction_clipping
0,BiLSTM,15,daylight,F3,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.414187,0.742051,0.815518,False
1,BiLSTM,15,daylight,F4,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.421259,0.748874,0.812110,False
2,BiLSTM,30,daylight,F3,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.565345,0.870933,0.745870,False
3,BiLSTM,30,daylight,F4,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.546643,0.892570,0.733086,False
4,BiLSTM,60,daylight,F3,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.649313,1.025221,0.647855,False
5,BiLSTM,60,daylight,F4,forecast_valid_solar_elevation_gt_0_deg,26496,17851,8645,0.693452,1.040654,0.637173,False



BiLSTM DAYLIGHT F3 -> F4 SUMMARY
Positive improvement = F4 lower daylight MAE than F3.


,model,horizon_minutes,scope,n_daylight,F3_MAE_kW,F4_MAE_kW,F3_minus_F4_MAE_kW,F3_to_F4_MAE_improvement_pct,F4_lower_MAE_than_F3
0,BiLSTM,15,daylight,17851,0.414187,0.421259,-0.007072,-1.707391,False
1,BiLSTM,30,daylight,17851,0.565345,0.546643,0.018703,3.308169,True
2,BiLSTM,60,daylight,17851,0.649313,0.693452,-0.044138,-6.797675,False



BiLSTM ALL-ROWS vs DAYLIGHT F3 -> F4


,model,horizon_minutes,scope,n_rows,F3_MAE_kW,F4_MAE_kW,F3_to_F4_MAE_improvement_pct,F4_lower_MAE_than_F3
0,BiLSTM,15,all,26496,0.308417,0.303657,1.543316,True
1,BiLSTM,15,daylight,17851,0.414187,0.421259,-1.707391,False
2,BiLSTM,30,all,26496,0.429713,0.392481,8.664420,True
3,BiLSTM,30,daylight,17851,0.565345,0.546643,3.308169,True
4,BiLSTM,60,all,26496,0.477851,0.520756,-8.978698,False
5,BiLSTM,60,daylight,17851,0.649313,0.693452,-6.797675,False



F3 -> F4 point-estimate direction audit:
  15-min | all: +1.543316% | daylight: -1.707391%
  30-min | all: +8.664420% | daylight: +3.308169%
  60-min | all: -8.978698% | daylight: -6.797675%

BiLSTM DAYLIGHT F3/F4 EVALUATION: COMPLETE
Daylight definition:
  forecast-valid solar elevation > 0 degrees

Population:
  TEST total: 26496
  Daylight: 17851
  Non-daylight: 8645

No model fitting performed.
No prediction clipping performed.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_daylight_metrics_F3_F4.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_daylight_f3_f4_summary.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_all_vs_daylight_f3_f4.csv

Next step after verification:
Run the paired 10,

In [121]:
# ============================================================
# Step 105: BiLSTM paired UTC daily-block bootstrap
#
# IMPORTANT:
#
#       NO MODEL FITTING OCCURS IN THIS CELL.
#
#
# PURPOSE
# -------
#
# Quantify the sampling uncertainty of the CORE comparison:
#
#       F3 -> F4
#
# for BiLSTM at:
#
#       15 minutes
#       30 minutes
#       60 minutes
#
# and for BOTH evaluation scopes:
#
#       1. all held-out TEST rows
#       2. daylight-only TEST rows
#
#
# This deliberately follows the SAME inferential protocol
# already used for the completed LSTM and GRU analyses.
#
#
# BOOTSTRAP PROTOCOL
# ------------------
#
# Resampling unit:
#
#       UTC forecast-valid calendar DAY
#
# Number of TEST days:
#
#       92
#
# TEST cadence:
#
#       5 minutes
#
# Rows per complete calendar day:
#
#       288
#
# Replicates:
#
#       10,000
#
# RNG seed:
#
#       42
#
# One RNG is instantiated ONCE and consumed sequentially
# across:
#
#       15 all
#       15 daylight
#       30 all
#       30 daylight
#       60 all
#       60 daylight
#
#
# PAIRED RESAMPLING
# -----------------
#
# Within a replicate, the EXACT SAME sampled UTC days are
# used for:
#
#       F3
#       F4
#
# This preserves the paired nature of the forecast comparison.
#
#
# STATISTIC
# ---------
#
#       MAE_F3 - MAE_F4
#
# Positive:
#
#       F4 has LOWER MAE than F3.
#
# Negative:
#
#       F4 has HIGHER MAE than F3.
#
#
# Relative improvement:
#
#       100 * (MAE_F3 - MAE_F4) / MAE_F3
#
#
# MAE aggregation:
#
#       pooled ROW-WEIGHTED MAE
#
# This matters especially for daylight because the number of
# daylight rows differs across calendar days.
#
#
# Confidence interval:
#
#       percentile 95% bootstrap interval
#       [2.5th percentile, 97.5th percentile]
#
#
# Predictions:
#
#       raw
#       unclipped
#
# ============================================================


# ------------------------------------------------------------
# 1. Frozen bootstrap protocol
# ------------------------------------------------------------

BILSTM_BOOTSTRAP_REPLICATES = 10_000

BILSTM_BOOTSTRAP_SEED = 42

BILSTM_BOOTSTRAP_CI_LOW_PERCENTILE = 2.5

BILSTM_BOOTSTRAP_CI_HIGH_PERCENTILE = 97.5


EXPECTED_TEST_ROWS = 26_496

EXPECTED_DAYLIGHT_ROWS = 17_851

EXPECTED_NON_DAYLIGHT_ROWS = 8_645

EXPECTED_TEST_DAYS = 92

EXPECTED_ROWS_PER_COMPLETE_DAY = 288


# ------------------------------------------------------------
# 2. Confirm all BiLSTM fitting is already complete
# ------------------------------------------------------------

expected_bilstm_final_keys = {
    (
        horizon_minutes,
        feature_family,
    )

    for horizon_minutes in [
        15,
        30,
        60,
    ]

    for feature_family in [
        "F0",
        "F1",
        "F2",
        "F3",
        "F4",
    ]
}


assert (
    set(
        bilstm_final_test_results.keys()
    )
    ==
    expected_bilstm_final_keys
)


assert (
    len(
        bilstm_final_test_results
    )
    ==
    15
)


print(
    "BiLSTM fitting state: COMPLETE"
)

print(
    "No model fitting will occur in this bootstrap cell."
)


# ------------------------------------------------------------
# 3. Confirm canonical daylight mask from Step 104
# ------------------------------------------------------------

assert (
    bilstm_canonical_daylight_mask.shape
    ==
    (
        EXPECTED_TEST_ROWS,
    )
)


assert (
    bilstm_canonical_daylight_mask.dtype
    ==
    np.bool_
)


assert (
    int(
        bilstm_canonical_daylight_mask.sum()
    )
    ==
    EXPECTED_DAYLIGHT_ROWS
)


assert (
    int(
        (
            ~bilstm_canonical_daylight_mask
        ).sum()
    )
    ==
    EXPECTED_NON_DAYLIGHT_ROWS
)


# ------------------------------------------------------------
# 4. Reconstruct the canonical forecast-valid TEST timeline
#
# Frozen TEST interval:
#
#       [2019-06-01 00:00 UTC,
#        2019-09-01 00:00 UTC)
#
# at 5-minute resolution.
#
# This contains:
#
#       92 days
#       288 rows/day
#       26,496 rows
#
# and matches the forecast-valid TEST population used for all
# three horizons.
# ------------------------------------------------------------

bilstm_test_valid_time_utc = pd.date_range(
    start=
        "2019-06-01 00:00:00",

    end=
        "2019-09-01 00:00:00",

    freq=
        "5min",

    inclusive=
        "left",

    tz=
        "UTC",
)


assert (
    len(
        bilstm_test_valid_time_utc
    )
    ==
    EXPECTED_TEST_ROWS
)


# Calendar-day label for every TEST row.
bilstm_test_utc_day = (
    bilstm_test_valid_time_utc.normalize()
)


bilstm_unique_test_days = pd.DatetimeIndex(
    bilstm_test_utc_day.unique()
)


assert (
    len(
        bilstm_unique_test_days
    )
    ==
    EXPECTED_TEST_DAYS
)


assert (
    bilstm_unique_test_days.is_monotonic_increasing
)


# ------------------------------------------------------------
# 5. Audit complete all-row UTC day blocks
# ------------------------------------------------------------

all_rows_per_day = np.asarray(
    [
        int(
            np.sum(
                bilstm_test_utc_day
                ==
                utc_day
            )
        )

        for utc_day in (
            bilstm_unique_test_days
        )
    ],
    dtype=int,
)


assert (
    all_rows_per_day.shape
    ==
    (
        EXPECTED_TEST_DAYS,
    )
)


assert np.all(
    all_rows_per_day
    ==
    EXPECTED_ROWS_PER_COMPLETE_DAY
)


assert (
    int(
        all_rows_per_day.sum()
    )
    ==
    EXPECTED_TEST_ROWS
)


print(
    "\nAll-row UTC daily-block audit: PASSED"
)

print(
    "  Number of blocks:",
    len(
        bilstm_unique_test_days
    ),
)

print(
    "  Rows per block:",
    int(
        all_rows_per_day.min()
    ),
)


# ------------------------------------------------------------
# 6. Audit daylight rows within each UTC day
#
# Unlike the all-row population, daylight block sizes vary
# naturally through the season.
# ------------------------------------------------------------

daylight_rows_per_day = np.asarray(
    [
        int(
            np.sum(
                (
                    bilstm_test_utc_day
                    ==
                    utc_day
                )
                &
                bilstm_canonical_daylight_mask
            )
        )

        for utc_day in (
            bilstm_unique_test_days
        )
    ],
    dtype=int,
)


assert (
    daylight_rows_per_day.shape
    ==
    (
        EXPECTED_TEST_DAYS,
    )
)


assert (
    int(
        daylight_rows_per_day.sum()
    )
    ==
    EXPECTED_DAYLIGHT_ROWS
)


assert np.all(
    daylight_rows_per_day
    >
    0
)


# These were the daylight day-size bounds established in the
# earlier LSTM / GRU analysis.
assert (
    int(
        daylight_rows_per_day.min()
    )
    ==
    165
)


assert (
    int(
        daylight_rows_per_day.max()
    )
    ==
    207
)


print(
    "\nDaylight UTC daily-block audit: PASSED"
)

print(
    "  Number of daylight-containing blocks:",
    int(
        np.sum(
            daylight_rows_per_day
            >
            0
        )
    ),
)

print(
    "  Minimum daylight rows/day:",
    int(
        daylight_rows_per_day.min()
    ),
)

print(
    "  Maximum daylight rows/day:",
    int(
        daylight_rows_per_day.max()
    ),
)

print(
    "  Total daylight rows:",
    int(
        daylight_rows_per_day.sum()
    ),
)


# ------------------------------------------------------------
# 7. Persist a reproducibility audit of block populations
# ------------------------------------------------------------

bilstm_bootstrap_block_audit = pd.DataFrame(
    {
        "utc_day":
            bilstm_unique_test_days,

        "n_all_rows":
            all_rows_per_day,

        "n_daylight_rows":
            daylight_rows_per_day,
    }
)


# ------------------------------------------------------------
# 8. Create ONE RNG for the complete six-comparison procedure
#
# Do NOT reinitialize this RNG inside the loops.
# ------------------------------------------------------------

bilstm_bootstrap_rng = np.random.default_rng(
    BILSTM_BOOTSTRAP_SEED
)


# ------------------------------------------------------------
# 9. Containers for summary and replicate-level results
# ------------------------------------------------------------

bilstm_bootstrap_summary_rows = []

bilstm_bootstrap_replicate_frames = []


# ------------------------------------------------------------
# 10. Run paired daily-block bootstrap
#
# Loop order is intentionally fixed:
#
#       horizon 15:
#           all
#           daylight
#
#       horizon 30:
#           all
#           daylight
#
#       horizon 60:
#           all
#           daylight
#
# One RNG is consumed sequentially in exactly that order.
# ------------------------------------------------------------

for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # Retrieve canonical held-out target and raw predictions.
    # --------------------------------------------------------

    actual_f3_kw = np.asarray(
        bilstm_final_test_actuals[
            (
                horizon_minutes,
                "F3",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    actual_f4_kw = np.asarray(
        bilstm_final_test_actuals[
            (
                horizon_minutes,
                "F4",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    prediction_f3_kw = np.asarray(
        bilstm_final_test_predictions[
            (
                horizon_minutes,
                "F3",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    prediction_f4_kw = np.asarray(
        bilstm_final_test_predictions[
            (
                horizon_minutes,
                "F4",
            )
        ],
        dtype=float,
    ).reshape(
        -1
    )


    # Same held-out target population is essential for paired
    # model comparison.
    np.testing.assert_array_equal(
        actual_f3_kw,
        actual_f4_kw,
    )


    assert (
        actual_f3_kw.shape
        ==
        prediction_f3_kw.shape
        ==
        prediction_f4_kw.shape
        ==
        (
            EXPECTED_TEST_ROWS,
        )
    )


    # Raw absolute errors.
    absolute_error_f3_kw = np.abs(
        actual_f3_kw
        -
        prediction_f3_kw
    )


    absolute_error_f4_kw = np.abs(
        actual_f3_kw
        -
        prediction_f4_kw
    )


    # --------------------------------------------------------
    # Evaluate both populations.
    # --------------------------------------------------------

    for scope in [
        "all",
        "daylight",
    ]:

        if (
            scope
            ==
            "all"
        ):

            scope_mask = np.ones(
                EXPECTED_TEST_ROWS,
                dtype=bool,
            )

        else:

            scope_mask = (
                bilstm_canonical_daylight_mask.copy()
            )


        # ----------------------------------------------------
        # Build per-day sufficient statistics:
        #
        #       absolute-error sum F3
        #       absolute-error sum F4
        #       row count
        #
        # The bootstrap then resamples these paired daily
        # blocks and computes POOLED row-weighted MAE.
        # ----------------------------------------------------

        daily_f3_absolute_error_sum = []

        daily_f4_absolute_error_sum = []

        daily_row_count = []


        for utc_day in (
            bilstm_unique_test_days
        ):

            day_scope_mask = (
                (
                    bilstm_test_utc_day
                    ==
                    utc_day
                )
                &
                scope_mask
            )


            n_day_scope_rows = int(
                np.sum(
                    day_scope_mask
                )
            )


            assert (
                n_day_scope_rows
                >
                0
            )


            daily_row_count.append(
                n_day_scope_rows
            )


            daily_f3_absolute_error_sum.append(
                float(
                    np.sum(
                        absolute_error_f3_kw[
                            day_scope_mask
                        ]
                    )
                )
            )


            daily_f4_absolute_error_sum.append(
                float(
                    np.sum(
                        absolute_error_f4_kw[
                            day_scope_mask
                        ]
                    )
                )
            )


        daily_row_count = np.asarray(
            daily_row_count,
            dtype=np.int64,
        )


        daily_f3_absolute_error_sum = np.asarray(
            daily_f3_absolute_error_sum,
            dtype=np.float64,
        )


        daily_f4_absolute_error_sum = np.asarray(
            daily_f4_absolute_error_sum,
            dtype=np.float64,
        )


        assert (
            daily_row_count.shape
            ==
            daily_f3_absolute_error_sum.shape
            ==
            daily_f4_absolute_error_sum.shape
            ==
            (
                EXPECTED_TEST_DAYS,
            )
        )


        # ----------------------------------------------------
        # Observed pooled MAE on the original TEST sample.
        # ----------------------------------------------------

        observed_f3_mae_kw = float(
            daily_f3_absolute_error_sum.sum()
            /
            daily_row_count.sum()
        )


        observed_f4_mae_kw = float(
            daily_f4_absolute_error_sum.sum()
            /
            daily_row_count.sum()
        )


        observed_difference_kw = float(
            observed_f3_mae_kw
            -
            observed_f4_mae_kw
        )


        observed_relative_improvement_pct = float(
            observed_difference_kw
            /
            observed_f3_mae_kw
            *
            100.0
        )


        # ----------------------------------------------------
        # Cross-check against the Step-104 canonical summary.
        # ----------------------------------------------------

        canonical_scope_row = (
            bilstm_all_vs_daylight_f3_f4[
                (
                    bilstm_all_vs_daylight_f3_f4[
                        "horizon_minutes"
                    ]
                    ==
                    horizon_minutes
                )
                &
                (
                    bilstm_all_vs_daylight_f3_f4[
                        "scope"
                    ]
                    ==
                    scope
                )
            ]
            .iloc[
                0
            ]
        )


        assert np.isclose(
            observed_f3_mae_kw,
            float(
                canonical_scope_row[
                    "F3_MAE_kW"
                ]
            ),
            rtol=0.0,
            atol=1e-10,
        )


        assert np.isclose(
            observed_f4_mae_kw,
            float(
                canonical_scope_row[
                    "F4_MAE_kW"
                ]
            ),
            rtol=0.0,
            atol=1e-10,
        )


        assert np.isclose(
            observed_relative_improvement_pct,
            float(
                canonical_scope_row[
                    "F3_to_F4_MAE_improvement_pct"
                ]
            ),
            rtol=0.0,
            atol=1e-8,
        )


        # ----------------------------------------------------
        # Draw paired bootstrap samples of UTC DAYS.
        #
        # Shape:
        #
        #       10,000 x 92
        #
        # Every replicate samples 92 calendar-day blocks with
        # replacement from the original 92 days.
        # ----------------------------------------------------

        sampled_day_positions = (
            bilstm_bootstrap_rng.integers(
                low=
                    0,

                high=
                    EXPECTED_TEST_DAYS,

                size=
                    (
                        BILSTM_BOOTSTRAP_REPLICATES,
                        EXPECTED_TEST_DAYS,
                    ),
            )
        )


        # ----------------------------------------------------
        # Pool all rows from the sampled blocks.
        #
        # For daylight, the denominator changes between
        # bootstrap replicates because sampled days contain
        # different numbers of daylight rows.
        # ----------------------------------------------------

        bootstrap_row_counts = (
            daily_row_count[
                sampled_day_positions
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f3_error_sums = (
            daily_f3_absolute_error_sum[
                sampled_day_positions
            ]
            .sum(
                axis=1
            )
        )


        bootstrap_f4_error_sums = (
            daily_f4_absolute_error_sum[
                sampled_day_positions
            ]
            .sum(
                axis=1
            )
        )


        assert np.all(
            bootstrap_row_counts
            >
            0
        )


        bootstrap_f3_mae_kw = (
            bootstrap_f3_error_sums
            /
            bootstrap_row_counts
        )


        bootstrap_f4_mae_kw = (
            bootstrap_f4_error_sums
            /
            bootstrap_row_counts
        )


        # ----------------------------------------------------
        # Paired bootstrap statistic.
        #
        # Positive:
        #
        #       F4 lower MAE.
        # ----------------------------------------------------

        bootstrap_difference_kw = (
            bootstrap_f3_mae_kw
            -
            bootstrap_f4_mae_kw
        )


        bootstrap_relative_improvement_pct = (
            bootstrap_difference_kw
            /
            bootstrap_f3_mae_kw
            *
            100.0
        )


        assert np.isfinite(
            bootstrap_f3_mae_kw
        ).all()


        assert np.isfinite(
            bootstrap_f4_mae_kw
        ).all()


        assert np.isfinite(
            bootstrap_difference_kw
        ).all()


        assert np.isfinite(
            bootstrap_relative_improvement_pct
        ).all()


        # ----------------------------------------------------
        # Percentile 95% confidence intervals
        # ----------------------------------------------------

        absolute_ci_low_kw = float(
            np.percentile(
                bootstrap_difference_kw,
                BILSTM_BOOTSTRAP_CI_LOW_PERCENTILE,
            )
        )


        absolute_ci_high_kw = float(
            np.percentile(
                bootstrap_difference_kw,
                BILSTM_BOOTSTRAP_CI_HIGH_PERCENTILE,
            )
        )


        relative_ci_low_pct = float(
            np.percentile(
                bootstrap_relative_improvement_pct,
                BILSTM_BOOTSTRAP_CI_LOW_PERCENTILE,
            )
        )


        relative_ci_high_pct = float(
            np.percentile(
                bootstrap_relative_improvement_pct,
                BILSTM_BOOTSTRAP_CI_HIGH_PERCENTILE,
            )
        )


        # ----------------------------------------------------
        # Fraction of bootstrap replicates in which F4 has
        # lower MAE than F3.
        #
        # This is a descriptive bootstrap proportion.
        # It is NOT being labeled as a formal p-value.
        # ----------------------------------------------------

        bootstrap_fraction_f4_better = float(
            np.mean(
                bootstrap_difference_kw
                >
                0.0
            )
        )


        # ----------------------------------------------------
        # Simple CI direction label
        # ----------------------------------------------------

        if (
            relative_ci_low_pct
            >
            0.0
        ):

            ci_direction = (
                "entirely_above_zero"
            )

        elif (
            relative_ci_high_pct
            <
            0.0
        ):

            ci_direction = (
                "entirely_below_zero"
            )

        else:

            ci_direction = (
                "crosses_zero"
            )


        # ----------------------------------------------------
        # Summary row
        # ----------------------------------------------------

        bilstm_bootstrap_summary_rows.append(
            {
                "model":
                    "BiLSTM",

                "horizon_minutes":
                    horizon_minutes,

                "scope":
                    scope,

                "bootstrap_unit":
                    "UTC_forecast_valid_calendar_day",

                "n_unique_day_blocks":
                    EXPECTED_TEST_DAYS,

                "bootstrap_replicates":
                    BILSTM_BOOTSTRAP_REPLICATES,

                "bootstrap_seed":
                    BILSTM_BOOTSTRAP_SEED,

                "statistic":
                    "MAE_F3_minus_MAE_F4",

                "positive_means":
                    "F4_lower_MAE",

                "observed_F3_MAE_kW":
                    observed_f3_mae_kw,

                "observed_F4_MAE_kW":
                    observed_f4_mae_kw,

                "observed_F3_minus_F4_MAE_kW":
                    observed_difference_kw,

                "observed_relative_improvement_pct":
                    observed_relative_improvement_pct,

                "absolute_difference_CI95_low_kW":
                    absolute_ci_low_kw,

                "absolute_difference_CI95_high_kW":
                    absolute_ci_high_kw,

                "relative_improvement_CI95_low_pct":
                    relative_ci_low_pct,

                "relative_improvement_CI95_high_pct":
                    relative_ci_high_pct,

                "bootstrap_fraction_F4_lower_MAE":
                    bootstrap_fraction_f4_better,

                "relative_CI_zero_status":
                    ci_direction,
            }
        )


        # ----------------------------------------------------
        # Replicate-level audit table
        #
        # Saving the replicate distribution makes subsequent
        # checks possible without rerunning the bootstrap.
        # ----------------------------------------------------

        bilstm_bootstrap_replicate_frames.append(
            pd.DataFrame(
                {
                    "model":
                        "BiLSTM",

                    "horizon_minutes":
                        horizon_minutes,

                    "scope":
                        scope,

                    "replicate":
                        np.arange(
                            1,
                            BILSTM_BOOTSTRAP_REPLICATES
                            +
                            1,
                            dtype=int,
                        ),

                    "pooled_row_count":
                        bootstrap_row_counts,

                    "F3_MAE_kW":
                        bootstrap_f3_mae_kw,

                    "F4_MAE_kW":
                        bootstrap_f4_mae_kw,

                    "F3_minus_F4_MAE_kW":
                        bootstrap_difference_kw,

                    "F3_to_F4_MAE_improvement_pct":
                        bootstrap_relative_improvement_pct,

                    "F4_lower_MAE_than_F3":
                        (
                            bootstrap_difference_kw
                            >
                            0.0
                        ),
                }
            )
        )


# ------------------------------------------------------------
# 11. Consolidate all six bootstrap comparisons
# ------------------------------------------------------------

bilstm_f3_f4_daily_block_bootstrap = pd.DataFrame(
    bilstm_bootstrap_summary_rows
)


bilstm_f3_f4_bootstrap_replicates = pd.concat(
    bilstm_bootstrap_replicate_frames,
    axis=0,
    ignore_index=True,
)


assert (
    bilstm_f3_f4_daily_block_bootstrap.shape[0]
    ==
    6
)


assert (
    bilstm_f3_f4_bootstrap_replicates.shape[0]
    ==
    (
        6
        *
        BILSTM_BOOTSTRAP_REPLICATES
    )
)


# ------------------------------------------------------------
# 12. Display primary bootstrap results
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM PAIRED UTC DAILY-BLOCK BOOTSTRAP"
)

print(
    "============================================================"
)


print(
    "Statistic: MAE_F3 - MAE_F4"
)

print(
    "Positive = F4 lower MAE than F3."
)

print(
    "95% CI = percentile bootstrap interval."
)


display(
    bilstm_f3_f4_daily_block_bootstrap[
        [
            "horizon_minutes",
            "scope",
            "observed_F3_MAE_kW",
            "observed_F4_MAE_kW",
            "observed_relative_improvement_pct",
            "relative_improvement_CI95_low_pct",
            "relative_improvement_CI95_high_pct",
            "bootstrap_fraction_F4_lower_MAE",
            "relative_CI_zero_status",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Human-readable compact summary
# ------------------------------------------------------------

print(
    "\nCompact BiLSTM F3 -> F4 bootstrap summary:"
)


for _, row in (
    bilstm_f3_f4_daily_block_bootstrap.iterrows()
):

    print(
        (
            f"  {int(row['horizon_minutes']):>2}-min "
            f"{row['scope']:<8} | "
            f"effect "
            f"{row['observed_relative_improvement_pct']:+.6f}%"
            f" | 95% CI "
            f"[{row['relative_improvement_CI95_low_pct']:+.6f}%, "
            f"{row['relative_improvement_CI95_high_pct']:+.6f}%]"
            f" | fraction F4-better "
            f"{row['bootstrap_fraction_F4_lower_MAE']:.4f}"
        )
    )


# ------------------------------------------------------------
# 14. CI direction table
# ------------------------------------------------------------

bilstm_bootstrap_direction_table = (
    bilstm_f3_f4_daily_block_bootstrap[
        [
            "horizon_minutes",
            "scope",
            "observed_relative_improvement_pct",
            "relative_improvement_CI95_low_pct",
            "relative_improvement_CI95_high_pct",
            "relative_CI_zero_status",
        ]
    ]
    .copy()
)


print(
    "\n95% CI direction audit:"
)


display(
    bilstm_bootstrap_direction_table
)


# ------------------------------------------------------------
# 15. Persist bootstrap artifacts
# ------------------------------------------------------------

bilstm_bootstrap_summary_path = (
    RESULTS_DIR
    /
    "bilstm_f3_f4_daily_block_bootstrap.csv"
)


bilstm_bootstrap_replicates_path = (
    RESULTS_DIR
    /
    "bilstm_f3_f4_daily_block_bootstrap_replicates.csv"
)


bilstm_bootstrap_block_audit_path = (
    RESULTS_DIR
    /
    "bilstm_f3_f4_daily_block_bootstrap_block_audit.csv"
)


bilstm_f3_f4_daily_block_bootstrap.to_csv(
    bilstm_bootstrap_summary_path,
    index=False,
)


bilstm_f3_f4_bootstrap_replicates.to_csv(
    bilstm_bootstrap_replicates_path,
    index=False,
)


bilstm_bootstrap_block_audit.to_csv(
    bilstm_bootstrap_block_audit_path,
    index=False,
)


# ------------------------------------------------------------
# 16. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    bilstm_bootstrap_summary_path,
    bilstm_bootstrap_replicates_path,
    bilstm_bootstrap_block_audit_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 17. Final bootstrap checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "BiLSTM PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Bootstrap replicates per comparison:",
    BILSTM_BOOTSTRAP_REPLICATES,
)


print(
    "UTC calendar-day blocks:",
    EXPECTED_TEST_DAYS,
)


print(
    "Seed:",
    BILSTM_BOOTSTRAP_SEED,
)


print(
    "Scopes evaluated:"
)

print(
    "  all"
)

print(
    "  daylight"
)


print(
    "\nNo model fitting performed."
)

print(
    "No prediction clipping performed."
)


print(
    "\nArtifacts written:"
)


for output_path in [
    bilstm_bootstrap_summary_path,
    bilstm_bootstrap_replicates_path,
    bilstm_bootstrap_block_audit_path,
]:

    print(
        " ",
        output_path,
    )


print(
    "\nNext step after verification:"
)

print(
    "Consolidate LSTM + GRU + BiLSTM F3 -> F4 evidence"
)

print(
    "across all horizons and both all/daylight scopes."
)

BiLSTM fitting state: COMPLETE
No model fitting will occur in this bootstrap cell.

All-row UTC daily-block audit: PASSED
  Number of blocks: 92
  Rows per block: 288

Daylight UTC daily-block audit: PASSED
  Number of daylight-containing blocks: 92
  Minimum daylight rows/day: 165
  Maximum daylight rows/day: 207
  Total daylight rows: 17851

BiLSTM PAIRED UTC DAILY-BLOCK BOOTSTRAP
Statistic: MAE_F3 - MAE_F4
Positive = F4 lower MAE than F3.
95% CI = percentile bootstrap interval.


,horizon_minutes,scope,observed_F3_MAE_kW,observed_F4_MAE_kW,observed_relative_improvement_pct,relative_improvement_CI95_low_pct,relative_improvement_CI95_high_pct,bootstrap_fraction_F4_lower_MAE,relative_CI_zero_status
0,15,all,0.308417,0.303657,1.543316,-1.120785,4.145202,0.8713,crosses_zero
1,15,daylight,0.414187,0.421259,-1.707391,-3.845899,0.317928,0.0543,crosses_zero
2,30,all,0.429713,0.392481,8.664420,6.533049,10.831739,1.0000,entirely_above_zero
3,30,daylight,0.565345,0.546643,3.308169,1.464350,5.210904,0.9997,entirely_above_zero
4,60,all,0.477851,0.520756,-8.978698,-13.730808,-4.908481,0.0000,entirely_below_zero
5,60,daylight,0.649313,0.693452,-6.797675,-10.805271,-3.256245,0.0000,entirely_below_zero



Compact BiLSTM F3 -> F4 bootstrap summary:
  15-min all      | effect +1.543316% | 95% CI [-1.120785%, +4.145202%] | fraction F4-better 0.8713
  15-min daylight | effect -1.707391% | 95% CI [-3.845899%, +0.317928%] | fraction F4-better 0.0543
  30-min all      | effect +8.664420% | 95% CI [+6.533049%, +10.831739%] | fraction F4-better 1.0000
  30-min daylight | effect +3.308169% | 95% CI [+1.464350%, +5.210904%] | fraction F4-better 0.9997
  60-min all      | effect -8.978698% | 95% CI [-13.730808%, -4.908481%] | fraction F4-better 0.0000
  60-min daylight | effect -6.797675% | 95% CI [-10.805271%, -3.256245%] | fraction F4-better 0.0000

95% CI direction audit:


,horizon_minutes,scope,observed_relative_improvement_pct,relative_improvement_CI95_low_pct,relative_improvement_CI95_high_pct,relative_CI_zero_status
0,15,all,1.543316,-1.120785,4.145202,crosses_zero
1,15,daylight,-1.707391,-3.845899,0.317928,crosses_zero
2,30,all,8.664420,6.533049,10.831739,entirely_above_zero
3,30,daylight,3.308169,1.464350,5.210904,entirely_above_zero
4,60,all,-8.978698,-13.730808,-4.908481,entirely_below_zero
5,60,daylight,-6.797675,-10.805271,-3.256245,entirely_below_zero



BiLSTM PAIRED DAILY-BLOCK BOOTSTRAP: COMPLETE
Bootstrap replicates per comparison: 10000
UTC calendar-day blocks: 92
Seed: 42
Scopes evaluated:
  all
  daylight

No model fitting performed.
No prediction clipping performed.

Artifacts written:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_f3_f4_daily_block_bootstrap.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_f3_f4_daily_block_bootstrap_replicates.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/bilstm_f3_f4_daily_block_bootstrap_block_audit.csv

Next step after verification:
Consolidate LSTM + GRU + BiLSTM F3 -> F4 evidence
across all horizons and both all/daylight scopes.


In [123]:
# ============================================================
# Step 106:
# Cross-architecture consolidation of F3 -> F4 evidence
#
# Models completed:
#
#       LSTM
#       GRU
#       BiLSTM
#
#
# Horizons:
#
#       15 min
#       30 min
#       60 min
#
#
# Evaluation scopes:
#
#       all TEST rows
#       daylight TEST rows
#
#
# Total comparisons:
#
#       3 models
#       x 3 horizons
#       x 2 scopes
#       = 18 rows
#
#
# IMPORTANT:
#
#       NO MODEL FITTING
#       NO NEW TEST PREDICTIONS
#       NO HYPERPARAMETER TUNING
#
#
# We consolidate the already-completed paired UTC daily-block
# bootstrap results.
#
#
# Interpretation:
#
#       positive F3 -> F4 effect
#           = F4 has LOWER MAE
#
#       negative F3 -> F4 effect
#           = F4 has HIGHER MAE
#
#
# CI status:
#
#       entirely_above_zero
#       entirely_below_zero
#       crosses_zero
#
#
# The purpose is to determine whether the F3 -> F4 effect is:
#
#       architecture-consistent
#
# or
#
#       architecture-dependent
#
#
# TCN has NOT yet been included.
# ============================================================


# ------------------------------------------------------------
# 1. Confirm current BiLSTM bootstrap result
# ------------------------------------------------------------

assert (
    "bilstm_f3_f4_daily_block_bootstrap"
    in
    globals()
)


assert (
    bilstm_f3_f4_daily_block_bootstrap.shape[0]
    ==
    6
)


assert (
    set(
        bilstm_f3_f4_daily_block_bootstrap[
            "horizon_minutes"
        ]
    )
    ==
    {
        15,
        30,
        60,
    }
)


assert (
    set(
        bilstm_f3_f4_daily_block_bootstrap[
            "scope"
        ]
    )
    ==
    {
        "all",
        "daylight",
    }
)


# ------------------------------------------------------------
# 2. Load the previously consolidated LSTM + GRU bootstrap
#    comparison artifact
#
# This artifact was created before BiLSTM fitting began.
#
# Loading the persisted artifact rather than manually typing
# previous numerical results reduces transcription risk.
# ------------------------------------------------------------

lstm_gru_comparison_path = (
    RESULTS_DIR
    /
    "lstm_gru_f3_f4_comparison.csv"
)


assert (
    lstm_gru_comparison_path.exists()
), (
    "Expected prior LSTM+GRU comparison artifact was not found: "
    f"{lstm_gru_comparison_path}"
)


lstm_gru_previous_comparison_raw = pd.read_csv(
    lstm_gru_comparison_path
)


print(
    "Loaded previous LSTM + GRU comparison:"
)

print(
    " ",
    lstm_gru_comparison_path,
)


print(
    "\nColumns in previous comparison artifact:"
)

print(
    list(
        lstm_gru_previous_comparison_raw.columns
    )
)


# ------------------------------------------------------------
# 3. Utility for resolving equivalent column names
#
# Earlier consolidation cells may have used slightly different
# descriptive names. We identify the required scientific
# quantities explicitly rather than assuming one exact spelling.
# ------------------------------------------------------------

def resolve_column_name(
    dataframe,
    candidate_names,
    quantity_description,
):

    for candidate_name in candidate_names:

        if (
            candidate_name
            in
            dataframe.columns
        ):

            return candidate_name


    raise KeyError(
        (
            f"Could not identify column for "
            f"{quantity_description}.\n"
            f"Available columns:\n"
            f"{list(dataframe.columns)}"
        )
    )


# ------------------------------------------------------------
# 4. Resolve columns in prior LSTM + GRU artifact
# ------------------------------------------------------------

previous_model_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        "model",
        "Model",
    ],
    "model",
)


previous_horizon_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        "horizon_minutes",
        "horizon",
        "Horizon_minutes",
    ],
    "forecast horizon",
)


previous_scope_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        "scope",
        "evaluation_scope",
        "population",
    ],
    "evaluation scope",
)


previous_effect_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        "observed_relative_improvement_pct",
        "F3_to_F4_MAE_improvement_pct",
        "relative_improvement_pct",
        "relative_effect_pct",
        "point_estimate_pct",
    ],
    "F3 -> F4 relative MAE point estimate",
)


previous_ci_low_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        # Exact column name used by the existing
        # LSTM + GRU consolidation artifact.
        "relative_CI95_lower_pct",

        # Other accepted historical variants.
        "relative_improvement_CI95_low_pct",
        "relative_CI95_low_pct",
        "CI95_low_pct",
        "ci95_low_pct",
        "ci_low_pct",
    ],
    "95% CI lower bound",
)


previous_ci_high_col = resolve_column_name(
    lstm_gru_previous_comparison_raw,
    [
        # Exact column name used by the existing
        # LSTM + GRU consolidation artifact.
        "relative_CI95_upper_pct",

        # Other accepted historical variants.
        "relative_improvement_CI95_high_pct",
        "relative_CI95_high_pct",
        "CI95_high_pct",
        "ci95_high_pct",
        "ci_high_pct",
    ],
    "95% CI upper bound",
)


print(
    "\nResolved prior-artifact columns:"
)

print(
    "  model:",
    previous_model_col,
)

print(
    "  horizon:",
    previous_horizon_col,
)

print(
    "  scope:",
    previous_scope_col,
)

print(
    "  point estimate:",
    previous_effect_col,
)

print(
    "  CI low:",
    previous_ci_low_col,
)

print(
    "  CI high:",
    previous_ci_high_col,
)


# ------------------------------------------------------------
# 5. Normalize previous LSTM + GRU results
# ------------------------------------------------------------

lstm_gru_normalized = pd.DataFrame(
    {
        "model":
            lstm_gru_previous_comparison_raw[
                previous_model_col
            ].astype(
                str
            ),

        "horizon_minutes":
            lstm_gru_previous_comparison_raw[
                previous_horizon_col
            ].astype(
                int
            ),

        "scope":
            lstm_gru_previous_comparison_raw[
                previous_scope_col
            ].astype(
                str
            ),

        "F3_to_F4_MAE_improvement_pct":
            lstm_gru_previous_comparison_raw[
                previous_effect_col
            ].astype(
                float
            ),

        "CI95_low_pct":
            lstm_gru_previous_comparison_raw[
                previous_ci_low_col
            ].astype(
                float
            ),

        "CI95_high_pct":
            lstm_gru_previous_comparison_raw[
                previous_ci_high_col
            ].astype(
                float
            ),
    }
)


# Normalize capitalization / spacing.
lstm_gru_normalized[
    "model"
] = (
    lstm_gru_normalized[
        "model"
    ]
    .str.strip()
)


lstm_gru_normalized[
    "scope"
] = (
    lstm_gru_normalized[
        "scope"
    ]
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 6. Validate exact previous two-model grid
#
# Expected:
#
#       2 models
#       x 3 horizons
#       x 2 scopes
#       = 12 rows
# ------------------------------------------------------------

assert (
    lstm_gru_normalized.shape[0]
    ==
    12
)


assert (
    set(
        lstm_gru_normalized[
            "model"
        ]
    )
    ==
    {
        "LSTM",
        "GRU",
    }
)


assert (
    set(
        lstm_gru_normalized[
            "horizon_minutes"
        ]
    )
    ==
    {
        15,
        30,
        60,
    }
)


assert (
    set(
        lstm_gru_normalized[
            "scope"
        ]
    )
    ==
    {
        "all",
        "daylight",
    }
)


assert (
    lstm_gru_normalized[
        [
            "F3_to_F4_MAE_improvement_pct",
            "CI95_low_pct",
            "CI95_high_pct",
        ]
    ]
    .apply(
        np.isfinite
    )
    .all()
    .all()
)


# ------------------------------------------------------------
# 7. Normalize current BiLSTM bootstrap result
# ------------------------------------------------------------

bilstm_normalized = pd.DataFrame(
    {
        "model":
            bilstm_f3_f4_daily_block_bootstrap[
                "model"
            ].astype(
                str
            ),

        "horizon_minutes":
            bilstm_f3_f4_daily_block_bootstrap[
                "horizon_minutes"
            ].astype(
                int
            ),

        "scope":
            bilstm_f3_f4_daily_block_bootstrap[
                "scope"
            ].astype(
                str
            ),

        "F3_to_F4_MAE_improvement_pct":
            bilstm_f3_f4_daily_block_bootstrap[
                "observed_relative_improvement_pct"
            ].astype(
                float
            ),

        "CI95_low_pct":
            bilstm_f3_f4_daily_block_bootstrap[
                "relative_improvement_CI95_low_pct"
            ].astype(
                float
            ),

        "CI95_high_pct":
            bilstm_f3_f4_daily_block_bootstrap[
                "relative_improvement_CI95_high_pct"
            ].astype(
                float
            ),
    }
)


assert (
    bilstm_normalized.shape[0]
    ==
    6
)


assert (
    set(
        bilstm_normalized[
            "model"
        ]
    )
    ==
    {
        "BiLSTM",
    }
)


# ------------------------------------------------------------
# 8. Combine all three completed recurrent architectures
# ------------------------------------------------------------

lstm_gru_bilstm_f3_f4_comparison = (
    pd.concat(
        [
            lstm_gru_normalized,
            bilstm_normalized,
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_minutes",
            "scope",
            "model",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 9. Validate the exact 18-row experimental grid
# ------------------------------------------------------------

assert (
    lstm_gru_bilstm_f3_f4_comparison.shape[0]
    ==
    18
)


assert (
    set(
        lstm_gru_bilstm_f3_f4_comparison[
            "model"
        ]
    )
    ==
    {
        "LSTM",
        "GRU",
        "BiLSTM",
    }
)


expected_three_model_grid = {
    (
        model,
        horizon_minutes,
        scope,
    )

    for model in [
        "LSTM",
        "GRU",
        "BiLSTM",
    ]

    for horizon_minutes in [
        15,
        30,
        60,
    ]

    for scope in [
        "all",
        "daylight",
    ]
}


actual_three_model_grid = set(
    zip(
        lstm_gru_bilstm_f3_f4_comparison[
            "model"
        ],

        lstm_gru_bilstm_f3_f4_comparison[
            "horizon_minutes"
        ],

        lstm_gru_bilstm_f3_f4_comparison[
            "scope"
        ],
    )
)


assert (
    actual_three_model_grid
    ==
    expected_three_model_grid
)


# ------------------------------------------------------------
# 10. Add point-estimate direction
#
# Positive:
#
#       F4 lower MAE
#
# Negative:
#
#       F4 higher MAE
# ------------------------------------------------------------

lstm_gru_bilstm_f3_f4_comparison[
    "point_direction"
] = np.where(
    lstm_gru_bilstm_f3_f4_comparison[
        "F3_to_F4_MAE_improvement_pct"
    ]
    >
    0.0,

    "F4_lower_MAE",

    np.where(
        lstm_gru_bilstm_f3_f4_comparison[
            "F3_to_F4_MAE_improvement_pct"
        ]
        <
        0.0,

        "F4_higher_MAE",

        "tie",
    ),
)


# ------------------------------------------------------------
# 11. Recompute CI status uniformly across all models
# ------------------------------------------------------------

def classify_relative_ci(
    ci_low,
    ci_high,
):

    if (
        ci_low
        >
        0.0
    ):

        return (
            "entirely_above_zero"
        )


    if (
        ci_high
        <
        0.0
    ):

        return (
            "entirely_below_zero"
        )


    return (
        "crosses_zero"
    )


lstm_gru_bilstm_f3_f4_comparison[
    "CI_zero_status"
] = [
    classify_relative_ci(
        ci_low,
        ci_high,
    )

    for ci_low, ci_high in zip(
        lstm_gru_bilstm_f3_f4_comparison[
            "CI95_low_pct"
        ],

        lstm_gru_bilstm_f3_f4_comparison[
            "CI95_high_pct"
        ],
    )
]


# ------------------------------------------------------------
# 12. Display the complete 18-row evidence table
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "LSTM + GRU + BiLSTM F3 -> F4 EVIDENCE"
)

print(
    "============================================================"
)


print(
    "Positive effect = F4 lower MAE than F3."
)

print(
    "Negative effect = F4 higher MAE than F3."
)


display(
    lstm_gru_bilstm_f3_f4_comparison
)


# ------------------------------------------------------------
# 13. Point-estimate pivot
#
# Convenient architecture-by-condition view.
# ------------------------------------------------------------

three_model_effect_pivot = (
    lstm_gru_bilstm_f3_f4_comparison
    .pivot(
        index=[
            "horizon_minutes",
            "scope",
        ],
        columns=
            "model",
        values=
            "F3_to_F4_MAE_improvement_pct",
    )
    .reset_index()
)


# Force intuitive model order.
three_model_effect_pivot = (
    three_model_effect_pivot[
        [
            "horizon_minutes",
            "scope",
            "LSTM",
            "GRU",
            "BiLSTM",
        ]
    ]
)


print(
    "\n============================================================"
)

print(
    "THREE-MODEL POINT-ESTIMATE PIVOT"
)

print(
    "============================================================"
)


display(
    three_model_effect_pivot
)


# ------------------------------------------------------------
# 14. CI-status pivot
# ------------------------------------------------------------

three_model_ci_status_pivot = (
    lstm_gru_bilstm_f3_f4_comparison
    .pivot(
        index=[
            "horizon_minutes",
            "scope",
        ],
        columns=
            "model",
        values=
            "CI_zero_status",
    )
    .reset_index()
)


three_model_ci_status_pivot = (
    three_model_ci_status_pivot[
        [
            "horizon_minutes",
            "scope",
            "LSTM",
            "GRU",
            "BiLSTM",
        ]
    ]
)


print(
    "\n============================================================"
)

print(
    "THREE-MODEL 95% CI STATUS"
)

print(
    "============================================================"
)


display(
    three_model_ci_status_pivot
)


# ------------------------------------------------------------
# 15. Build cross-architecture consensus table
#
# This does NOT average architectures.
#
# Instead it asks:
#
#       Are the signs consistent?
#
#       Are all CIs on the same side of zero?
#
# This is more transparent for the current ablation than
# collapsing heterogeneous architectures to one mean number.
# ------------------------------------------------------------

three_model_consensus_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    for scope in [
        "all",
        "daylight",
    ]:

        subset = (
            lstm_gru_bilstm_f3_f4_comparison[
                (
                    lstm_gru_bilstm_f3_f4_comparison[
                        "horizon_minutes"
                    ]
                    ==
                    horizon_minutes
                )
                &
                (
                    lstm_gru_bilstm_f3_f4_comparison[
                        "scope"
                    ]
                    ==
                    scope
                )
            ]
            .copy()
        )


        assert (
            subset.shape[0]
            ==
            3
        )


        n_positive_point = int(
            np.sum(
                subset[
                    "F3_to_F4_MAE_improvement_pct"
                ]
                >
                0.0
            )
        )


        n_negative_point = int(
            np.sum(
                subset[
                    "F3_to_F4_MAE_improvement_pct"
                ]
                <
                0.0
            )
        )


        n_ci_above_zero = int(
            np.sum(
                subset[
                    "CI_zero_status"
                ]
                ==
                "entirely_above_zero"
            )
        )


        n_ci_below_zero = int(
            np.sum(
                subset[
                    "CI_zero_status"
                ]
                ==
                "entirely_below_zero"
            )
        )


        n_ci_crosses_zero = int(
            np.sum(
                subset[
                    "CI_zero_status"
                ]
                ==
                "crosses_zero"
            )
        )


        if (
            n_positive_point
            ==
            3
        ):

            point_consensus = (
                "all_three_F4_lower_MAE"
            )

        elif (
            n_negative_point
            ==
            3
        ):

            point_consensus = (
                "all_three_F4_higher_MAE"
            )

        else:

            point_consensus = (
                "mixed_architecture_directions"
            )


        if (
            n_ci_above_zero
            ==
            3
        ):

            ci_consensus = (
                "all_three_CIs_above_zero"
            )

        elif (
            n_ci_below_zero
            ==
            3
        ):

            ci_consensus = (
                "all_three_CIs_below_zero"
            )

        else:

            ci_consensus = (
                "no_three_model_CI_consensus"
            )


        three_model_consensus_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "scope":
                    scope,

                "n_models":
                    3,

                "n_positive_point_estimates":
                    n_positive_point,

                "n_negative_point_estimates":
                    n_negative_point,

                "point_estimate_consensus":
                    point_consensus,

                "n_CIs_entirely_above_zero":
                    n_ci_above_zero,

                "n_CIs_entirely_below_zero":
                    n_ci_below_zero,

                "n_CIs_crossing_zero":
                    n_ci_crosses_zero,

                "CI_consensus":
                    ci_consensus,
            }
        )


three_model_consensus = pd.DataFrame(
    three_model_consensus_rows
)


assert (
    three_model_consensus.shape[0]
    ==
    6
)


print(
    "\n============================================================"
)

print(
    "THREE-MODEL ARCHITECTURE-CONSISTENCY AUDIT"
)

print(
    "============================================================"
)


display(
    three_model_consensus
)


# ------------------------------------------------------------
# 16. Compact human-readable architecture comparison
# ------------------------------------------------------------

print(
    "\nCompact three-model F3 -> F4 comparison:"
)


for _, row in (
    three_model_effect_pivot.iterrows()
):

    horizon_minutes = int(
        row[
            "horizon_minutes"
        ]
    )


    scope = str(
        row[
            "scope"
        ]
    )


    print(
        (
            f"  {horizon_minutes:>2}-min "
            f"{scope:<8} | "
            f"LSTM {row['LSTM']:+.6f}% | "
            f"GRU {row['GRU']:+.6f}% | "
            f"BiLSTM {row['BiLSTM']:+.6f}%"
        )
    )


# ------------------------------------------------------------
# 17. Overall scientific checkpoint
#
# This is deliberately descriptive.
#
# TCN is still pending, so we do not make a four-architecture
# final conclusion here.
# ------------------------------------------------------------

n_conditions_with_unanimous_point_direction = int(
    np.sum(
        three_model_consensus[
            "point_estimate_consensus"
        ]
        !=
        "mixed_architecture_directions"
    )
)


n_conditions_with_three_model_ci_consensus = int(
    np.sum(
        three_model_consensus[
            "CI_consensus"
        ]
        !=
        "no_three_model_CI_consensus"
    )
)


print(
    "\n============================================================"
)

print(
    "SCIENTIFIC CHECKPOINT AFTER THREE ARCHITECTURES"
)

print(
    "============================================================"
)


print(
    "Conditions examined:",
    len(
        three_model_consensus
    ),
)


print(
    "Conditions with unanimous point-estimate direction:",
    n_conditions_with_unanimous_point_direction,
)


print(
    "Conditions with all three 95% CIs on the same side of zero:",
    n_conditions_with_three_model_ci_consensus,
)


if (
    n_conditions_with_three_model_ci_consensus
    ==
    0
):

    print(
        "\nCurrent evidence does NOT show a three-architecture"
    )

    print(
        "CI-consistent F3 -> F4 effect in any horizon/scope"
    )

    print(
        "condition."
    )


print(
    "\nInterpretation:"
)

print(
    "  The F3 -> F4 effect is currently architecture- and"
)

print(
    "  horizon-dependent."
)


print(
    "  This does not establish a model-general Panchang benefit"
)

print(
    "  or a model-general Panchang detriment."
)


print(
    "  TCN remains to be evaluated under the same frozen"
)

print(
    "  experimental protocol."
)


# ------------------------------------------------------------
# 18. Persist three-model consolidation artifacts
# ------------------------------------------------------------

three_model_comparison_path = (
    RESULTS_DIR
    /
    "lstm_gru_bilstm_f3_f4_comparison.csv"
)


three_model_effect_pivot_path = (
    RESULTS_DIR
    /
    "lstm_gru_bilstm_f3_f4_point_estimate_pivot.csv"
)


three_model_ci_status_pivot_path = (
    RESULTS_DIR
    /
    "lstm_gru_bilstm_f3_f4_ci_status_pivot.csv"
)


three_model_consensus_path = (
    RESULTS_DIR
    /
    "lstm_gru_bilstm_f3_f4_consensus.csv"
)


lstm_gru_bilstm_f3_f4_comparison.to_csv(
    three_model_comparison_path,
    index=False,
)


three_model_effect_pivot.to_csv(
    three_model_effect_pivot_path,
    index=False,
)


three_model_ci_status_pivot.to_csv(
    three_model_ci_status_pivot_path,
    index=False,
)


three_model_consensus.to_csv(
    three_model_consensus_path,
    index=False,
)


# ------------------------------------------------------------
# 19. Verify persisted artifacts
# ------------------------------------------------------------

for output_path in [
    three_model_comparison_path,
    three_model_effect_pivot_path,
    three_model_ci_status_pivot_path,
    three_model_consensus_path,
]:

    assert (
        output_path.exists()
    )

    assert (
        output_path.stat().st_size
        >
        0
    )


# ------------------------------------------------------------
# 20. Final checkpoint
# ------------------------------------------------------------

print(
    "\n============================================================"
)

print(
    "LSTM + GRU + BiLSTM CONSOLIDATION: COMPLETE"
)

print(
    "============================================================"
)


print(
    "Models consolidated:"
)

print(
    "  LSTM"
)

print(
    "  GRU"
)

print(
    "  BiLSTM"
)


print(
    "\nComparisons consolidated:",
    len(
        lstm_gru_bilstm_f3_f4_comparison
    ),
)


print(
    "\nNo model fitting performed."
)

print(
    "No TEST predictions regenerated."
)

print(
    "No hyperparameters modified."
)


print(
    "\nArtifacts written:"
)


for output_path in [
    three_model_comparison_path,
    three_model_effect_pivot_path,
    three_model_ci_status_pivot_path,
    three_model_consensus_path,
]:

    print(
        " ",
        output_path,
    )


print(
    "\nNext step after verification:"
)

print(
    "Begin TCN preflight and causal-architecture verification."
)

Loaded previous LSTM + GRU comparison:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook11_deep_learning_models/lstm_gru_f3_f4_comparison.csv

Columns in previous comparison artifact:
['model', 'horizon_minutes', 'scope', 'relative_improvement_pct', 'relative_CI95_lower_pct', 'relative_CI95_upper_pct', 'CI_direction', 'point_estimate_direction']

Resolved prior-artifact columns:
  model: model
  horizon: horizon_minutes
  scope: scope
  point estimate: relative_improvement_pct
  CI low: relative_CI95_lower_pct
  CI high: relative_CI95_upper_pct

LSTM + GRU + BiLSTM F3 -> F4 EVIDENCE
Positive effect = F4 lower MAE than F3.
Negative effect = F4 higher MAE than F3.


,model,horizon_minutes,scope,F3_to_F4_MAE_improvement_pct,CI95_low_pct,CI95_high_pct,point_direction,CI_zero_status
0,BiLSTM,15,all,1.543316,-1.120785,4.145202,F4_lower_MAE,crosses_zero
1,GRU,15,all,-7.590039,-9.358493,-5.883449,F4_higher_MAE,entirely_below_zero
2,LSTM,15,all,-0.884545,-2.591356,0.800759,F4_higher_MAE,crosses_zero
3,BiLSTM,15,daylight,-1.707391,-3.845899,0.317928,F4_higher_MAE,crosses_zero
4,GRU,15,daylight,-1.236311,-2.286129,-0.131478,F4_higher_MAE,entirely_below_zero
5,LSTM,15,daylight,1.263797,0.040153,2.557423,F4_lower_MAE,entirely_above_zero
6,BiLSTM,30,all,8.664420,6.533049,10.831739,F4_lower_MAE,entirely_above_zero
7,GRU,30,all,-5.464907,-7.888403,-3.164433,F4_higher_MAE,entirely_below_zero
8,LSTM,30,all,5.797532,3.552636,7.939762,F4_lower_MAE,entirely_above_zero
9,BiLSTM,30,daylight,3.308169,1.464350,5.210904,F4_lower_MAE,entirely_above_zero



THREE-MODEL POINT-ESTIMATE PIVOT


model,horizon_minutes,scope,LSTM,GRU,BiLSTM
0,15,all,-0.884545,-7.590039,1.543316
1,15,daylight,1.263797,-1.236311,-1.707391
2,30,all,5.797532,-5.464907,8.664420
3,30,daylight,4.174850,-8.568435,3.308169
4,60,all,-1.441883,-4.489917,-8.978698
5,60,daylight,3.449669,-5.103283,-6.797675



THREE-MODEL 95% CI STATUS


model,horizon_minutes,scope,LSTM,GRU,BiLSTM
0,15,all,crosses_zero,entirely_below_zero,crosses_zero
1,15,daylight,entirely_above_zero,entirely_below_zero,crosses_zero
2,30,all,entirely_above_zero,entirely_below_zero,entirely_above_zero
3,30,daylight,entirely_above_zero,entirely_below_zero,entirely_above_zero
4,60,all,crosses_zero,entirely_below_zero,entirely_below_zero
5,60,daylight,entirely_above_zero,entirely_below_zero,entirely_below_zero



THREE-MODEL ARCHITECTURE-CONSISTENCY AUDIT


,horizon_minutes,scope,n_models,n_positive_point_estimates,n_negative_point_estimates,point_estimate_consensus,n_CIs_entirely_above_zero,n_CIs_entirely_below_zero,n_CIs_crossing_zero,CI_consensus
0,15,all,3,1,2,mixed_architecture_directions,0,1,2,no_three_model_CI_consensus
1,15,daylight,3,1,2,mixed_architecture_directions,1,1,1,no_three_model_CI_consensus
2,30,all,3,2,1,mixed_architecture_directions,2,1,0,no_three_model_CI_consensus
3,30,daylight,3,2,1,mixed_architecture_directions,2,1,0,no_three_model_CI_consensus
4,60,all,3,0,3,all_three_F4_higher_MAE,0,2,1,no_three_model_CI_consensus
5,60,daylight,3,1,2,mixed_architecture_directions,1,2,0,no_three_model_CI_consensus



Compact three-model F3 -> F4 comparison:
  15-min all      | LSTM -0.884545% | GRU -7.590039% | BiLSTM +1.543316%
  15-min daylight | LSTM +1.263797% | GRU -1.236311% | BiLSTM -1.707391%
  30-min all      | LSTM +5.797532% | GRU -5.464907% | BiLSTM +8.664420%
  30-min daylight | LSTM +4.174850% | GRU -8.568435% | BiLSTM +3.308169%
  60-min all      | LSTM -1.441883% | GRU -4.489917% | BiLSTM -8.978698%
  60-min daylight | LSTM +3.449669% | GRU -5.103283% | BiLSTM -6.797675%

SCIENTIFIC CHECKPOINT AFTER THREE ARCHITECTURES
Conditions examined: 6
Conditions with unanimous point-estimate direction: 1
Conditions with all three 95% CIs on the same side of zero: 0

Current evidence does NOT show a three-architecture
CI-consistent F3 -> F4 effect in any horizon/scope
condition.

Interpretation:
  The F3 -> F4 effect is currently architecture- and
  horizon-dependent.
  This does not establish a model-general Panchang benefit
  or a model-general Panchang detriment.
  TCN remains to be evalua